In [ ]:
import pandas as pd 
train = pd.read_parquet("/Users/ruaczcf/Downloads/wruuQNI2/train_solo_track.parquet")
train.head()

In [ ]:
train.describe()

In [ ]:
train.route_id.unique()

In [ ]:
train[train["target_1h"] < 0.8e6]["target_1h"].plot(kind="hist")

In [ ]:
train[train["route_id"] != 791].groupby("route_id")["target_1h"].mean().plot(kind="hist")

In [ ]:
train[["route_id", "timestamp"]].nunique()

In [ ]:
route_history = train[train["route_id"] == 877].copy()
route_history["month"] = route_history["timestamp"].dt.month
route_history.groupby("month")[["target_1h", "status_3"]].agg("sum")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

train = pd.read_parquet("train_solo_track.parquet")
train["timestamp"] = pd.to_datetime(train["timestamp"])

route_history = train[train["route_id"] == 877].copy()
route_history = route_history[
    route_history["timestamp"].between("2025-10-30", "2025-11-10")
].sort_values("timestamp").reset_index(drop=True)

# Заменяем сырой таргет на сглаженный
route_history["target_1h"] = route_history["target_1h"].ewm(span=24).mean()

# route_history.drop(columns=["status_1", "status_2", "status_3"]).plot(x="timestamp", figsize=(18, 5))
route_history[["timestamp", "status_6"]].plot(x="timestamp", figsize=(18, 5))

plt.title("Route 410 — target_1h (EWM span=4) + statuses")
plt.tight_layout()
plt.savefig("route_410_smooth_statuses.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
route_history[["timestamp", "status_1"]].plot(x="timestamp", figsize=(18, 5))

In [ ]:
test = pd.read_parquet("/Users/ruaczcf/Downloads/wruuQNI2/test_solo_track.parquet")
test["route_id"].head()

In [ ]:
train["timestamp"].describe()

In [ ]:
test["timestamp"].describe()

In [ ]:
# ============================================================
# SARIMA per warehouse — Solo Track
# ============================================================
import warnings
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ---------- Config ----------
TRACK = "solo"
TRAIN_DAYS = 14        # последние N дней для обучения (=672 часовых точки)

# Порядки для HOURLY данных (m=24 — суточная сезонность)
SARIMA_ORDER          = (1, 1, 1)
SARIMA_SEASONAL_ORDER = (1, 1, 0, 24)

TRACK_CONFIG = {
    "solo": {"train_path": "train_solo_track.parquet",
             "test_path":  "test_solo_track.parquet",
             "target_col": "target_1h"},
    "team": {"train_path": "train_team_track.parquet",
             "test_path":  "test_team_track.parquet",
             "target_col": "target_2h"},
}
cfg        = TRACK_CONFIG[TRACK]
TARGET_COL = cfg["target_col"]

# ---------- Загрузка ----------
train_df = pd.read_parquet(cfg["train_path"])
test_df  = pd.read_parquet(cfg["test_path"])

train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])

train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

# Отрезаем только последние TRAIN_DAYS (быстрее и актуальнее)
cutoff   = train_df["timestamp"].max() - pd.Timedelta(days=TRAIN_DAYS)
train_df = train_df[train_df["timestamp"] > cutoff].copy()

print(f"Train shape (last {TRAIN_DAYS}d): {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Unique routes: {test_df['route_id'].nunique()}")

# ---------- SARIMA per route ----------
predictions: dict[int, float] = {}
route_ids = test_df["route_id"].unique()

for route_id in tqdm(route_ids, desc="SARIMA"):

    # --- 1. Тренировочный 30-мин ряд ---
    route_train_30 = (
        train_df[train_df["route_id"] == route_id]
        .set_index("timestamp")[TARGET_COL]
        .asfreq("30min")
        .interpolate(method="time")
        .fillna(method="bfill")
        .fillna(method="ffill")
    )

    # --- 2. Ресэмплируем в часовой (быстрее, m=24) ---
    route_train_1h = route_train_30.resample("1h").mean().interpolate(method="time")

    # --- 3. Тестовые строки для этого route ---
    route_test = test_df[test_df["route_id"] == route_id].sort_values("timestamp")

    last_train_ts = route_train_1h.index[-1]
    max_test_ts   = route_test["timestamp"].max().ceil("1h")
    n_hours = int((max_test_ts - last_train_ts) / pd.Timedelta("1h")) + 2

    try:
        model = SARIMAX(
            route_train_1h,
            order=SARIMA_ORDER,
            seasonal_order=SARIMA_SEASONAL_ORDER,
            enforce_stationarity=False,
            enforce_invertibility=False,
        )
        result = model.fit(disp=False, maxiter=200, method="lbfgs")

        # --- 4. Форкаст почасовой ---
        hourly_idx = pd.date_range(
            start=last_train_ts + pd.Timedelta("1h"),
            periods=n_hours,
            freq="1h",
        )
        forecast_1h = pd.Series(result.forecast(steps=n_hours).values, index=hourly_idx)

        # --- 5. Expand до 30-мин через линейную интерполяцию ---
        forecast_30min = (
            forecast_1h
            .resample("30min")
            .interpolate(method="linear")
        )

        # --- 6. Маппинг на test ids ---
        for _, row in route_test.iterrows():
            ts = row["timestamp"]
            pred = (
                float(forecast_30min.get(ts, forecast_30min.iloc[-1]))
            )
            predictions[row["id"]] = max(0.0, pred)

    except Exception as e:
        print(f"\n  ⚠ Route {route_id} failed ({e}), seasonal-mean fallback")
        for _, row in route_test.iterrows():
            h, m = row["timestamp"].hour, row["timestamp"].minute
            mask = (route_train_30.index.hour == h) & (route_train_30.index.minute == m)
            fb   = route_train_30[mask].mean() if mask.any() else route_train_30.mean()
            predictions[row["id"]] = max(0.0, float(fb))

# ---------- Submission ----------
submission = (
    pd.DataFrame(list(predictions.items()), columns=["id", "y_pred"])
    .sort_values("id")
    .reset_index(drop=True)
)
submission.to_csv("submission_sarima.csv", index=False)
print("\n✅ Saved: submission_sarima.csv")
print(submission.head(10).to_string())
print(f"\ny_pred stats:\n{submission['y_pred'].describe().round(2)}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║         GRADIENT BOOSTING (LightGBM) — Team Track           ║
# ║  Log1p-target, lag/rolling features, recursive prediction   ║
# ╚══════════════════════════════════════════════════════════════╝
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

# ─────────────────────────── CONFIG ───────────────────────────
TRACK = "team"   # "solo" | "team"

TRACK_CONFIG = {
    "solo": {"train_path": "train_solo_track.parquet",
             "test_path":  "test_solo_track.parquet",
             "target_col": "target_1h"},
    "team": {"train_path": "train_team_track.parquet",
             "test_path":  "test_team_track.parquet",
             "target_col": "target_2h"},
}
cfg        = TRACK_CONFIG[TRACK]
TARGET_COL = cfg["target_col"]
STATUS_COLS = [f"status_{i}" for i in range(1, 9)]

# Лаги в шагах (1 шаг = 30 мин)
LAG_STEPS = [1, 2, 3, 4, 6, 8, 12, 16, 24, 48, 96, 336]

# Rolling windows (в шагах)
ROLLING_WINDOWS = [4, 8, 24, 48, 96]

# Буффер для рекурсивного предсказания
MAX_LAG    = max(LAG_STEPS) + max(ROLLING_WINDOWS)   # 432 шага = 9 дней
BUFFER_LEN = MAX_LAG + 1                             # 433 строки

VAL_DAYS     = 14      # последние N дней → валидация
RANDOM_STATE = 42

# LightGBM:  objective=regression_l2 в log1p-пространстве
# ≡ минимизация MSLE в оригинальном → хорошо для логнормального таргета
LGB_PARAMS = {
    "objective":         "regression_l2",
    "metric":            "mae",
    "boosting_type":     "gbdt",
    "n_estimators":      3000,
    "learning_rate":     0.03,
    "num_leaves":        127,
    "max_depth":         -1,
    "min_child_samples": 20,
    "subsample":         0.8,
    "subsample_freq":    1,
    "colsample_bytree":  0.7,
    "reg_alpha":         0.05,
    "reg_lambda":        1.0,
    "n_jobs":            -1,
    "random_state":      RANDOM_STATE,
    "verbose":           -1,
}
EARLY_STOPPING = 150

# ─────────────────────────── LOAD ───────────────────────────
train_df = pd.read_parquet(cfg["train_path"])
test_df  = pd.read_parquet(cfg["test_path"])

train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])

train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

# Добавляем office_from_id в тест (статичное свойство маршрута)
route_meta = train_df.groupby("route_id")["office_from_id"].first().reset_index()
test_df    = test_df.merge(route_meta, on="route_id", how="left")

print(f"Train: {train_df.shape}  |  {train_df['timestamp'].min()} → {train_df['timestamp'].max()}")
print(f"Test:  {test_df.shape}   |  {test_df['timestamp'].min()} → {test_df['timestamp'].max()}")


# ──────────────────── FEATURE ENGINEERING ────────────────────
def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    ts = df["timestamp"]
    out = df.copy()
    out["hour"]         = ts.dt.hour.astype(np.int8)
    out["halfhour_idx"] = (ts.dt.hour * 2 + ts.dt.minute // 30).astype(np.int8)  # 0..47
    out["dow"]          = ts.dt.dayofweek.astype(np.int8)
    out["month"]        = ts.dt.month.astype(np.int8)
    out["is_weekend"]   = (ts.dt.dayofweek >= 5).astype(np.int8)
    # Циклическое кодирование (непрерывность между 23→0 и воскресенье→понедельник)
    out["hh_sin"]  = np.sin(2 * np.pi * out["halfhour_idx"] / 48)
    out["hh_cos"]  = np.cos(2 * np.pi * out["halfhour_idx"] / 48)
    out["dow_sin"] = np.sin(2 * np.pi * out["dow"] / 7)
    out["dow_cos"] = np.cos(2 * np.pi * out["dow"] / 7)
    return out


def add_lag_rolling_features(grp: pd.DataFrame) -> pd.DataFrame:
    """Применяется к отдельному маршруту, отсортированному по времени."""
    grp = grp.copy()
    tgt = grp[TARGET_COL]

    # ── Лаги таргета ──
    for lag in LAG_STEPS:
        grp[f"lag_{lag}"] = tgt.shift(lag)

    # ── Rolling stats (shift(1) → не утекаем текущее значение) ──
    shifted = tgt.shift(1)
    for w in ROLLING_WINDOWS:
        mp = max(1, w // 4)
        grp[f"rmean_{w}"] = shifted.rolling(w, min_periods=mp).mean()
        grp[f"rstd_{w}"]  = shifted.rolling(w, min_periods=mp).std()

    grp["rmax_48"] = shifted.rolling(48, min_periods=24).max()
    grp["rmin_48"] = shifted.rolling(48, min_periods=24).min()

    # ── Трендовые фичи ──
    grp["trend_4v8"]    = grp["rmean_4"]  - grp["rmean_8"]    # краткосрочный тренд
    grp["trend_8v48"]   = grp["rmean_8"]  - grp["rmean_48"]   # среднесрочный тренд
    grp["ratio_yest"]   = tgt / (tgt.shift(48) + 1e-6)        # отношение ко вчера

    # ── Лаги статус-фич (только безопасные: 48 и 336 шагов) ──
    # Для тестовых строк status_{i} = NaN, но shift(48/336) → в трейн → OK
    for col in STATUS_COLS:
        if col in grp.columns:
            for lag in [48, 336]:
                grp[f"{col}_lag{lag}"] = grp[col].shift(lag)

    return grp


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    df = add_time_features(df)
    df = (
        df.groupby("route_id", group_keys=False)
          .apply(add_lag_rolling_features)
    )
    return df


# ──────────────────── BUILD TRAIN FEATURES ────────────────────
print("\nBuilding train features...")
train_feat = build_features(train_df)

# Дропаем строки, где lag_336 = NaN (первые 336 строк каждого маршрута)
train_feat = train_feat.dropna(subset=[f"lag_{LAG_STEPS[-1]}"]).copy()
print(f"Train after lag-dropna: {train_feat.shape}")

# Колонки для модели (без таргета и "сырых" статус-фич)
FEATURE_COLS = [
    c for c in train_feat.columns
    if c not in (["timestamp", TARGET_COL] + STATUS_COLS)
]
print(f"Features: {len(FEATURE_COLS)}  →  {FEATURE_COLS[:10]} ...")

# ──────────────────── TRAIN / VAL SPLIT ────────────────────
val_cutoff = train_feat["timestamp"].max() - pd.Timedelta(days=VAL_DAYS)
mask_tr    = train_feat["timestamp"] <= val_cutoff
mask_val   = train_feat["timestamp"] >  val_cutoff

X_tr  = train_feat.loc[mask_tr,  FEATURE_COLS]
y_tr  = np.log1p(train_feat.loc[mask_tr,  TARGET_COL].clip(lower=0))
X_val = train_feat.loc[mask_val, FEATURE_COLS]
y_val = np.log1p(train_feat.loc[mask_val, TARGET_COL].clip(lower=0))

print(f"\nTrain set : {X_tr.shape}")
print(f"Val   set : {X_val.shape}  ({train_feat.loc[mask_val,'timestamp'].min()} → {train_feat.loc[mask_val,'timestamp'].max()})")

# ──────────────────── TRAIN LIGHTGBM ────────────────────
cat_feats = ["route_id", "office_from_id", "dow", "month"]

dtrain = lgb.Dataset(
    X_tr,  label=y_tr,
    categorical_feature=cat_feats,
    free_raw_data=False,
)
dval = lgb.Dataset(
    X_val, label=y_val,
    categorical_feature=cat_feats,
    free_raw_data=False,
    reference=dtrain,
)

print("\nTraining LightGBM...")
callbacks = [
    lgb.early_stopping(EARLY_STOPPING, verbose=True),
    lgb.log_evaluation(period=200),
]
model = lgb.train(
    LGB_PARAMS,
    dtrain,
    num_boost_round=LGB_PARAMS["n_estimators"],
    valid_sets=[dval],
    callbacks=callbacks,
)

# Метрика на валидации в оригинальном пространстве
val_pred = np.expm1(model.predict(X_val)).clip(0)
val_true = np.expm1(y_val)
print(f"\n✅ Val MAE (original space): {mean_absolute_error(val_true, val_pred):.4f}")

# Важность фич
feat_imp = pd.Series(
    model.feature_importance(importance_type="gain"),
    index=FEATURE_COLS
).sort_values(ascending=False)
print(f"\nTop-15 features:\n{feat_imp.head(15).to_string()}")


# ──────────────────── RECURSIVE TEST PREDICTION ────────────────────
# Для каждого маршрута предсказываем тестовые шаги один за другим,
# используя уже предсказанные значения как лаги для следующих шагов.
print("\nRecursive test prediction...")
predictions: dict[int, float] = {}

for route_id in tqdm(sorted(test_df["route_id"].unique())):
    # Берём последние BUFFER_LEN строк трейна для этого маршрута
    route_train = (
        train_df[train_df["route_id"] == route_id]
        .tail(BUFFER_LEN)
        .copy()
    )
    route_test = test_df[test_df["route_id"] == route_id].sort_values("timestamp")

    buffer = route_train.copy()  # будем расширять по одной строке

    for _, test_row in route_test.iterrows():
        # Добавляем тестовую строку с NaN-таргетом в буффер
        new_row = {
            "route_id":     route_id,
            "office_from_id": test_row["office_from_id"],
            "timestamp":    test_row["timestamp"],
            TARGET_COL:     np.nan,
        }
        for col in STATUS_COLS:
            new_row[col] = np.nan  # status-лаги берутся из глубины буфера (lag>=48)

        buffer = pd.concat([buffer, pd.DataFrame([new_row])], ignore_index=True)

        # Фичи только для последней строки буфера
        sub = buffer.tail(BUFFER_LEN)
        sub = add_time_features(sub)
        sub = add_lag_rolling_features(sub)

        feat_row = sub.iloc[[-1]][FEATURE_COLS]
        pred_log = model.predict(feat_row)[0]
        pred     = float(max(0.0, np.expm1(pred_log)))

        # Сохраняем предсказание и заполняем буфер (нужен для следующих лагов)
        buffer.at[buffer.index[-1], TARGET_COL] = pred
        predictions[test_row["id"]] = pred

# ──────────────────── SUBMISSION ────────────────────
submission = (
    pd.DataFrame(list(predictions.items()), columns=["id", "y_pred"])
    .sort_values("id")
    .reset_index(drop=True)
)
submission.to_csv("submission_lgb.csv", index=False)
print(f"\n✅ Saved: submission_lgb.csv  ({len(submission)} rows)")
print(submission.head(10).to_string())
print(f"\ny_pred stats:\n{submission['y_pred'].describe().round(2)}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║        LightGBM Forecasting — WAPE + |Relative Bias|           ║
# ║   Tweedie loss, lag/rolling features, recursive prediction,    ║
# ║   bias calibration post-hoc                                    ║
# ╚══════════════════════════════════════════════════════════════════╝
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None


# ═══════════════════════════════════════════════════════════════════
#  METRIC
# ═══════════════════════════════════════════════════════════════════
def wape_rbias(y_true: np.ndarray, y_pred: np.ndarray) -> tuple[float, float, float]:
    """Returns (total, WAPE, |Relative Bias|)."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None)
    wape  = np.abs(y_pred - y_true).sum() / (y_true.sum() + 1e-9)
    rbias = np.abs(y_pred.sum() / (y_true.sum() + 1e-9) - 1)
    return wape + rbias, wape, rbias


# ═══════════════════════════════════════════════════════════════════
#  CONFIG
# ═══════════════════════════════════════════════════════════════════
TRAIN_PATH = "train_solo_track.parquet"   # поменяй на team если нужно
TEST_PATH  = "test_solo_track.parquet"
TARGET_COL = "target_1h"

STATUS_COLS = [f"status_{i}" for i in range(1, 7)]  # только 6 колонок

# Лаги таргета (в шагах, 1 шаг = 30 мин)
LAG_STEPS = [1, 2, 3, 4, 6, 8, 12, 16, 24, 48, 96, 168, 336]

# Лаги статус-фич: минимум 8, чтобы не пересекаться с тестом (8 шагов = 4 часа)
# lag_8 → status 4ч назад; lag_48 → вчера; lag_336 → неделю назад
STATUS_LAG_STEPS = [8, 48, 336]

ROLLING_WINDOWS = [4, 8, 24, 48, 96]

# Размер буфера для рекурсивного предсказания
MAX_LAG    = max(LAG_STEPS) + max(ROLLING_WINDOWS)   # 336 + 96 = 432
BUFFER_LEN = MAX_LAG + 1                              # 433

VAL_DAYS     = 14
RANDOM_STATE = 42

# Tweedie p=1.5: compound Poisson–Gamma, ideal для правоскошенных
# объёмов (count + continuous). Лучше MSE/MAE для лог-нормального таргета.
# Если хочешь прямо MAE → поменяй на "regression_l1", убери tweedie_variance_power
LGB_PARAMS = {
    "objective":              "tweedie",
    "tweedie_variance_power": 1.5,
    "metric":                 "tweedie",
    "boosting_type":          "gbdt",
    "n_estimators":           3000,
    "learning_rate":          0.03,
    "num_leaves":             127,
    "max_depth":              -1,
    "min_child_samples":      20,
    "subsample":              0.8,
    "subsample_freq":         1,
    "colsample_bytree":       0.7,
    "reg_alpha":              0.05,
    "reg_lambda":             1.0,
    "n_jobs":                 -1,
    "random_state":           RANDOM_STATE,
    "verbose":                -1,
}
EARLY_STOPPING = 150


# ═══════════════════════════════════════════════════════════════════
#  LOAD
# ═══════════════════════════════════════════════════════════════════
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)

train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])

train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

print(f"Train: {train_df.shape}  {train_df['timestamp'].min()} → {train_df['timestamp'].max()}")
print(f"Test:  {test_df.shape}   {test_df['timestamp'].min()} → {test_df['timestamp'].max()}")
print(f"Routes in test: {test_df['route_id'].nunique()}")


# ═══════════════════════════════════════════════════════════════════
#  FEATURE ENGINEERING
# ═══════════════════════════════════════════════════════════════════
def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    ts  = df["timestamp"]
    out = df.copy()
    out["hour"]         = ts.dt.hour.astype(np.int8)
    out["halfhour_idx"] = (ts.dt.hour * 2 + ts.dt.minute // 30).astype(np.int8)  # 0..47
    out["dow"]          = ts.dt.dayofweek.astype(np.int8)                         # 0=пн
    out["month"]        = ts.dt.month.astype(np.int8)
    out["is_weekend"]   = (ts.dt.dayofweek >= 5).astype(np.int8)
    # Циклическое кодирование (нет разрыва в 23→0, вс→пн)
    out["hh_sin"]  = np.sin(2 * np.pi * out["halfhour_idx"] / 48)
    out["hh_cos"]  = np.cos(2 * np.pi * out["halfhour_idx"] / 48)
    out["dow_sin"] = np.sin(2 * np.pi * out["dow"] / 7)
    out["dow_cos"] = np.cos(2 * np.pi * out["dow"] / 7)
    return out


def add_lag_rolling_features(grp: pd.DataFrame) -> pd.DataFrame:
    """Применяется к per-route DataFrame, отсортированному по timestamp."""
    grp = grp.copy()
    tgt = grp[TARGET_COL]

    # ── Лаги таргета ──────────────────────────────────────────────
    for lag in LAG_STEPS:
        grp[f"lag_{lag}"] = tgt.shift(lag)

    # ── Rolling stats (shift(1) — нет утечки текущего значения) ───
    shifted = tgt.shift(1)
    for w in ROLLING_WINDOWS:
        mp = max(1, w // 4)
        grp[f"rmean_{w}"]  = shifted.rolling(w, min_periods=mp).mean()
        grp[f"rstd_{w}"]   = shifted.rolling(w, min_periods=mp).std()
    grp["rmax_48"] = shifted.rolling(48, min_periods=24).max()
    grp["rmin_48"] = shifted.rolling(48, min_periods=24).min()

    # ── Трендовые фичи ────────────────────────────────────────────
    grp["trend_4v8"]   = grp["rmean_4"]  - grp["rmean_8"]    # краткосрочный
    grp["trend_8v48"]  = grp["rmean_8"]  - grp["rmean_48"]   # среднесрочный
    grp["trend_48v96"] = grp["rmean_48"] - grp["rmean_96"]   # долгосрочный
    grp["ratio_yest"]  = tgt / (tgt.shift(48) + 1e-6)        # к тому же часу вчера
    grp["ratio_week"]  = tgt / (tgt.shift(336) + 1e-6)       # к тому же часу недели

    # ── Статус-фичи: только безопасные лаги (≥8, не пересекаются с тестом) ──
    # status_1..3 = исходящие статусы текущего склада  → предвестники отгрузки
    # status_4..6 = входящие статусы с предыдущего склада → будущий объём
    for col in STATUS_COLS:
        if col in grp.columns:
            for lag in STATUS_LAG_STEPS:
                grp[f"{col}_lag{lag}"] = grp[col].shift(lag)
            # Суммарный поток (статусы 4+5+6 = incoming flow) с лагом 8
            if col in ("status_4", "status_5", "status_6"):
                grp[f"{col}_rmean48"] = grp[col].shift(8).rolling(48, min_periods=12).mean()

    return grp


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    df = add_time_features(df)
    df = df.groupby("route_id", group_keys=False).apply(add_lag_rolling_features)
    return df


# ═══════════════════════════════════════════════════════════════════
#  BUILD TRAIN FEATURES
# ═══════════════════════════════════════════════════════════════════
print("\nBuilding features...")
train_feat = build_features(train_df)

# Убираем строки, где lag_336 = NaN (первые 336 строк каждого маршрута)
train_feat = train_feat.dropna(subset=[f"lag_{LAG_STEPS[-1]}"]).copy()
print(f"Train after lag-dropna: {train_feat.shape}")

# Список колонок для модели
FEATURE_COLS = [
    c for c in train_feat.columns
    if c not in (["timestamp", TARGET_COL] + STATUS_COLS)
]
print(f"Feature count: {len(FEATURE_COLS)}")


# ═══════════════════════════════════════════════════════════════════
#  TRAIN / VAL SPLIT (time-based)
# ═══════════════════════════════════════════════════════════════════
val_cutoff = train_feat["timestamp"].max() - pd.Timedelta(days=VAL_DAYS)
mask_tr    = train_feat["timestamp"] <= val_cutoff
mask_val   = train_feat["timestamp"] >  val_cutoff

X_tr   = train_feat.loc[mask_tr,  FEATURE_COLS]
y_tr   = train_feat.loc[mask_tr,  TARGET_COL].clip(lower=0)
X_val  = train_feat.loc[mask_val, FEATURE_COLS]
y_val  = train_feat.loc[mask_val, TARGET_COL].clip(lower=0)

print(f"\nTrain set : {X_tr.shape}")
print(f"Val   set : {X_val.shape}  ({train_feat.loc[mask_val,'timestamp'].min()} → {train_feat.loc[mask_val,'timestamp'].max()})")


# ═══════════════════════════════════════════════════════════════════
#  TRAIN LIGHTGBM
# ═══════════════════════════════════════════════════════════════════
cat_feats = ["route_id", "dow", "month"]

dtrain = lgb.Dataset(X_tr,  label=y_tr,  categorical_feature=cat_feats, free_raw_data=False)
dval   = lgb.Dataset(X_val, label=y_val, categorical_feature=cat_feats, free_raw_data=False, reference=dtrain)

print("\nTraining LightGBM (Tweedie p=1.5)...")
callbacks = [
    lgb.early_stopping(EARLY_STOPPING, verbose=True),
    lgb.log_evaluation(period=200),
]
model = lgb.train(
    LGB_PARAMS,
    dtrain,
    num_boost_round=LGB_PARAMS["n_estimators"],
    valid_sets=[dval],
    callbacks=callbacks,
)


# ═══════════════════════════════════════════════════════════════════
#  VALIDATION + BIAS CALIBRATION
# ═══════════════════════════════════════════════════════════════════
val_pred_raw = model.predict(X_val).clip(0)

total_raw, wape_raw, rb_raw = wape_rbias(y_val, val_pred_raw)
print(f"\n── Val (raw) ──────────────────────────────────")
print(f"  WAPE={wape_raw:.4f}  |RBias|={rb_raw:.4f}  total={total_raw:.4f}")

# Bias calibration: scale = sum(y_true) / sum(y_pred)
# Устраняет систематическое смещение → |Relative Bias| → 0
CALIB_SCALE = float(y_val.sum() / val_pred_raw.sum())
val_pred_cal = (val_pred_raw * CALIB_SCALE).clip(0)

total_cal, wape_cal, rb_cal = wape_rbias(y_val, val_pred_cal)
print(f"── Val (calibrated, scale={CALIB_SCALE:.4f}) ───────")
print(f"  WAPE={wape_cal:.4f}  |RBias|={rb_cal:.4f}  total={total_cal:.4f}")
print(f"  Δ improvement: {total_raw - total_cal:.4f}")

# Feature importance
feat_imp = pd.Series(
    model.feature_importance(importance_type="gain"),
    index=FEATURE_COLS
).sort_values(ascending=False)
print(f"\nTop-15 features:\n{feat_imp.head(15).to_string()}")


# ═══════════════════════════════════════════════════════════════════
#  RECURSIVE TEST PREDICTION
# ═══════════════════════════════════════════════════════════════════
# Порядок: предсказываем тестовые шаги поочерёдно.
# Предыдущие предсказания используются как лаги для последующих шагов.
# Статус-фичи с lag≥8: для всех тестовых строк они уходят в трейн → OK.
print("\nRecursive test prediction...")
predictions: dict[int, float] = {}

for route_id in tqdm(sorted(test_df["route_id"].unique())):
    route_train = (
        train_df[train_df["route_id"] == route_id]
        .tail(BUFFER_LEN)
        .copy()
    )
    route_test = test_df[test_df["route_id"] == route_id].sort_values("timestamp")

    buffer = route_train.copy()

    for _, test_row in route_test.iterrows():
        new_row = {
            "route_id":  route_id,
            "timestamp": test_row["timestamp"],
            TARGET_COL:  np.nan,
        }
        # Status-колонки NaN для тестовых строк (лаги ≥8 → уйдут в трейн)
        for col in STATUS_COLS:
            new_row[col] = np.nan

        buffer = pd.concat([buffer, pd.DataFrame([new_row])], ignore_index=True)

        # Считаем фичи только для буфера нужного размера
        sub = buffer.tail(BUFFER_LEN)
        sub = add_time_features(sub)
        sub = add_lag_rolling_features(sub)

        feat_row = sub.iloc[[-1]][FEATURE_COLS]
        pred     = float(max(0.0, model.predict(feat_row)[0]))

        # Применяем bias calibration к каждому предсказанию
        pred_calibrated = pred * CALIB_SCALE

        predictions[test_row["id"]] = pred_calibrated
        buffer.at[buffer.index[-1], TARGET_COL] = pred  # сырое значение в буфер!


# ═══════════════════════════════════════════════════════════════════
#  SUBMISSION
# ═══════════════════════════════════════════════════════════════════
submission = (
    pd.DataFrame(list(predictions.items()), columns=["id", "y_pred"])
    .sort_values("id")
    .reset_index(drop=True)
)
submission.to_csv("submission_lgb.csv", index=False)
print(f"\n✅ Saved: submission_lgb.csv  ({len(submission)} rows)")
print(submission.head(10).to_string())
print(f"\ny_pred stats:\n{submission['y_pred'].describe().round(2)}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║    LightGBM v2 — log1p + MAE, concat features, stable          ║
# ╚══════════════════════════════════════════════════════════════════╝
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None


# ═══════════════════════════════════════════════════════════════════
#  METRIC
# ═══════════════════════════════════════════════════════════════════
def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias


# ═══════════════════════════════════════════════════════════════════
#  CONFIG
# ═══════════════════════════════════════════════════════════════════
TRAIN_PATH  = "train_solo_track.parquet"
TEST_PATH   = "test_solo_track.parquet"
TARGET_COL  = "target_1h"
STATUS_COLS = [f"status_{i}" for i in range(1, 7)]

# Лаги таргета (шаги по 30 мин).
# Для теста (8 шагов) lags 1–7 будут NaN для последних тестовых строк.
# LightGBM обрабатывает NaN — учит оптимальное направление сплита.
LAG_STEPS = [1, 2, 3, 4, 6, 8, 12, 16, 24, 48, 96, 168, 336]

# Лаги статус-фич: минимум 8, чтобы всегда попадать в трейн для любой тест-строки
STATUS_LAG_STEPS = [8, 48, 336]

ROLLING_WINDOWS = [4, 8, 24, 48, 96]

VAL_DAYS     = 14
RANDOM_STATE = 42

# log1p + regression_l1 (MAE) — стабильный выбор:
# • log1p нормализует лог-нормальный таргет
# • MAE в log-пространстве ≈ MALE, устойчив к выбросам
# • model.predict() возвращает log1p(y), нужен np.expm1() на выходе
LGB_PARAMS = {
    "objective":         "regression_l1",   # MAE — напрямую минимизирует числитель WAPE
    "metric":            "mae",
    "boosting_type":     "gbdt",
    "n_estimators":      3000,
    "learning_rate":     0.03,
    "num_leaves":        127,
    "max_depth":         -1,
    "min_child_samples": 20,
    "subsample":         0.8,
    "subsample_freq":    1,
    "colsample_bytree":  0.7,
    "reg_alpha":         0.05,
    "reg_lambda":        1.0,
    "n_jobs":            -1,
    "random_state":      RANDOM_STATE,
    "verbose":           -1,
}
EARLY_STOPPING = 150


# ═══════════════════════════════════════════════════════════════════
#  LOAD
# ═══════════════════════════════════════════════════════════════════
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)

train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])

train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

print(f"Train: {train_df.shape}  |  {train_df['timestamp'].min()} → {train_df['timestamp'].max()}")
print(f"Test:  {test_df.shape}   |  {test_df['timestamp'].min()} → {test_df['timestamp'].max()}")
print(f"Target stats: mean={train_df[TARGET_COL].mean():.1f}  "
      f"median={train_df[TARGET_COL].median():.1f}  max={train_df[TARGET_COL].max():.1f}")


# ═══════════════════════════════════════════════════════════════════
#  CONCAT TRAIN + TEST (NaN target/status для тестовых строк)
# ═══════════════════════════════════════════════════════════════════
# Это позволяет вычислить все лаги одним проходом vectorized.
# Для тест-строк lag_1..7 частично NaN — LightGBM справляется.

test_ext = test_df.copy()
test_ext[TARGET_COL] = np.nan
for col in STATUS_COLS:
    test_ext[col] = np.nan
test_ext["is_test"] = True
train_df["is_test"] = False

full_df = pd.concat([train_df, test_ext], ignore_index=True)
full_df = full_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
print(f"\nFull df (train+test): {full_df.shape}")


# ═══════════════════════════════════════════════════════════════════
#  FEATURE ENGINEERING
# ═══════════════════════════════════════════════════════════════════
def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    ts  = df["timestamp"]
    out = df.copy()
    out["halfhour_idx"] = (ts.dt.hour * 2 + ts.dt.minute // 30).astype(np.int8)
    out["hour"]         = ts.dt.hour.astype(np.int8)
    out["dow"]          = ts.dt.dayofweek.astype(np.int8)
    out["month"]        = ts.dt.month.astype(np.int8)
    out["is_weekend"]   = (ts.dt.dayofweek >= 5).astype(np.int8)
    out["hh_sin"]  = np.sin(2 * np.pi * out["halfhour_idx"] / 48)
    out["hh_cos"]  = np.cos(2 * np.pi * out["halfhour_idx"] / 48)
    out["dow_sin"] = np.sin(2 * np.pi * out["dow"] / 7)
    out["dow_cos"] = np.cos(2 * np.pi * out["dow"] / 7)
    return out


def add_lag_rolling_features(grp: pd.DataFrame) -> pd.DataFrame:
    grp = grp.copy()
    tgt = grp[TARGET_COL]

    for lag in LAG_STEPS:
        grp[f"lag_{lag}"] = tgt.shift(lag)

    shifted = tgt.shift(1)
    for w in ROLLING_WINDOWS:
        mp = max(1, w // 4)
        grp[f"rmean_{w}"]  = shifted.rolling(w, min_periods=mp).mean()
        grp[f"rstd_{w}"]   = shifted.rolling(w, min_periods=mp).std()
    grp["rmax_48"] = shifted.rolling(48, min_periods=24).max()
    grp["rmin_48"] = shifted.rolling(48, min_periods=24).min()

    grp["trend_4v8"]   = grp["rmean_4"]  - grp["rmean_8"]
    grp["trend_8v48"]  = grp["rmean_8"]  - grp["rmean_48"]
    grp["trend_48v96"] = grp["rmean_48"] - grp["rmean_96"]

    # Безопасные статус-лаги (min 8 → всегда в трейне для любой тест-строки)
    for col in STATUS_COLS:
        if col in grp.columns:
            for lag in STATUS_LAG_STEPS:
                grp[f"{col}_lag{lag}"] = grp[col].shift(lag)

    # Incoming flow (status_4+5+6 предвещают будущую отгрузку)
    incoming = (grp.get("status_4", 0) + grp.get("status_5", 0) + grp.get("status_6", 0))
    grp["incoming_flow_lag8"]  = incoming.shift(8)
    grp["incoming_flow_lag48"] = incoming.shift(48)

    return grp


print("Building features (groupby apply)...")
full_feat = add_time_features(full_df)
full_feat = (
    full_feat
    .groupby("route_id", group_keys=False)
    .apply(add_lag_rolling_features)
)
print(f"Features built: {full_feat.shape}")


# ═══════════════════════════════════════════════════════════════════
#  SPLIT BACK TO TRAIN / TEST
# ═══════════════════════════════════════════════════════════════════
FEATURE_COLS = [
    c for c in full_feat.columns
    if c not in (["timestamp", TARGET_COL, "is_test"] + STATUS_COLS)
]

# Тренировочный набор: только не-тест строки, с известным таргетом, с lag_336
train_feat = full_feat[
    (~full_feat["is_test"]) &
    (full_feat[f"lag_{LAG_STEPS[-1]}"].notna())
].copy()

test_feat = full_feat[full_feat["is_test"]].copy()

print(f"\nTrain feat: {train_feat.shape}")
print(f"Test  feat: {test_feat.shape}")
print(f"Feature count: {len(FEATURE_COLS)}")

# Санитарная проверка: NaN-процент в тест-фичах
nan_pct = test_feat[FEATURE_COLS].isna().mean()
print(f"\nTop NaN features in test:\n{nan_pct[nan_pct > 0].sort_values(ascending=False).head(10)}")


# ═══════════════════════════════════════════════════════════════════
#  TRAIN / VAL SPLIT
# ═══════════════════════════════════════════════════════════════════
val_cutoff = train_feat["timestamp"].max() - pd.Timedelta(days=VAL_DAYS)
mask_tr    = train_feat["timestamp"] <= val_cutoff
mask_val   = train_feat["timestamp"] >  val_cutoff

X_tr  = train_feat.loc[mask_tr,  FEATURE_COLS]
y_tr  = np.log1p(train_feat.loc[mask_tr,  TARGET_COL].clip(lower=0))
X_val = train_feat.loc[mask_val, FEATURE_COLS]
y_val_log  = np.log1p(train_feat.loc[mask_val, TARGET_COL].clip(lower=0))
y_val_orig = train_feat.loc[mask_val, TARGET_COL].clip(lower=0)

print(f"\nTrain rows: {X_tr.shape[0]}  |  Val rows: {X_val.shape[0]}")
print(f"Target log1p: mean={y_tr.mean():.3f}  std={y_tr.std():.3f}  max={y_tr.max():.3f}")


# ═══════════════════════════════════════════════════════════════════
#  TRAIN
# ═══════════════════════════════════════════════════════════════════
cat_feats = ["route_id", "dow", "month"]

dtrain = lgb.Dataset(X_tr,  label=y_tr,       categorical_feature=cat_feats, free_raw_data=False)
dval   = lgb.Dataset(X_val, label=y_val_log,  categorical_feature=cat_feats,
                     free_raw_data=False, reference=dtrain)

print("\nTraining LightGBM (log1p + MAE)...")
callbacks = [
    lgb.early_stopping(EARLY_STOPPING, verbose=True),
    lgb.log_evaluation(period=200),
]
model = lgb.train(
    LGB_PARAMS,
    dtrain,
    num_boost_round=LGB_PARAMS["n_estimators"],
    valid_sets=[dval],
    callbacks=callbacks,
)
print(f"\nBest iteration: {model.best_iteration}")


# ═══════════════════════════════════════════════════════════════════
#  VALIDATE + CALIBRATION
# ═══════════════════════════════════════════════════════════════════
val_pred_log  = model.predict(X_val)
val_pred_orig = np.expm1(val_pred_log).clip(0)

print(f"\n── Val predictions sanity check ──")
print(f"  y_val  : mean={y_val_orig.mean():.2f}  std={y_val_orig.std():.2f}  max={y_val_orig.max():.2f}")
print(f"  y_pred : mean={val_pred_orig.mean():.2f}  std={val_pred_orig.std():.2f}  max={val_pred_orig.max():.2f}")

total_raw, wape_raw, rb_raw = wape_rbias(y_val_orig, val_pred_orig)
print(f"\n── Val (raw) ──")
print(f"  WAPE={wape_raw:.4f}  |RBias|={rb_raw:.4f}  total={total_raw:.4f}")

# Bias calibration
CALIB_SCALE = float(y_val_orig.sum() / val_pred_orig.sum())
val_pred_cal = (val_pred_orig * CALIB_SCALE).clip(0)
total_cal, wape_cal, rb_cal = wape_rbias(y_val_orig, val_pred_cal)
print(f"── Val (calibrated scale={CALIB_SCALE:.4f}) ──")
print(f"  WAPE={wape_cal:.4f}  |RBias|={rb_cal:.4f}  total={total_cal:.4f}")

# Feature importance
feat_imp = pd.Series(
    model.feature_importance(importance_type="gain"),
    index=FEATURE_COLS
).sort_values(ascending=False)
print(f"\nTop-15 features:\n{feat_imp.head(15).to_string()}")


# ═══════════════════════════════════════════════════════════════════
#  PREDICT TEST
# ═══════════════════════════════════════════════════════════════════
X_test = test_feat[FEATURE_COLS]
test_pred_log  = model.predict(X_test)
test_pred_orig = np.expm1(test_pred_log).clip(0)

print(f"\n── Test predictions sanity check ──")
print(f"  mean={test_pred_orig.mean():.2f}  std={test_pred_orig.std():.2f}  "
      f"min={test_pred_orig.min():.2f}  max={test_pred_orig.max():.2f}")

# Применяем calibration
test_pred_calibrated = (test_pred_orig * CALIB_SCALE).clip(0)
print(f"  After calibration (scale={CALIB_SCALE:.4f}):")
print(f"  mean={test_pred_calibrated.mean():.2f}  max={test_pred_calibrated.max():.2f}")


# ═══════════════════════════════════════════════════════════════════
#  SUBMISSION
# ═══════════════════════════════════════════════════════════════════
submission = test_feat[["id"]].copy() if "id" in test_feat.columns else test_df[["id"]].copy()
submission["y_pred"] = test_pred_calibrated.values

submission = submission.sort_values("id").reset_index(drop=True)
submission.to_csv("submission_lgb_v2.csv", index=False)

print(f"\n✅ Saved: submission_lgb_v2.csv ({len(submission)} rows)")
print(submission.head(10).to_string())
print(f"\ny_pred stats:\n{submission['y_pred'].describe().round(2)}")

In [ ]:
submission["y_pred"] = test_pred_calibrated

submission = submission.sort_values("id").reset_index(drop=True)
submission.to_csv("submission_lgb_v2.csv", index=False)

print(f"\n✅ Saved: submission_lgb_v2.csv ({len(submission)} rows)")
print(submission.head(10).to_string())
print(f"\ny_pred stats:\n{submission['y_pred'].describe().round(2)}")

In [ ]:
# ============================================================
# SARIMA per route — local validation + full submission
# Solo track: target_1h, horizon = 8 half-hour points
# ============================================================
import warnings
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from tqdm import tqdm

warnings.filterwarnings("ignore")


# -------------------- Metric --------------------
def wape_rbias(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None)
    denom = y_true.sum() + 1e-9
    wape = np.abs(y_pred - y_true).sum() / denom
    rbias = np.abs(y_pred.sum() / denom - 1)
    return wape + rbias, wape, rbias


# -------------------- Config --------------------
TRACK = "solo"
TRAIN_DAYS = 14

SARIMA_ORDER = (1, 1, 1)
SARIMA_SEASONAL_ORDER = (1, 1, 0, 24)

TRACK_CONFIG = {
    "solo": {
        "train_path": "train_solo_track.parquet",
        "test_path": "test_solo_track.parquet",
        "target_col": "target_1h",
        "forecast_points": 8,
    },
    "team": {
        "train_path": "train_team_track.parquet",
        "test_path": "test_team_track.parquet",
        "target_col": "target_2h",
        "forecast_points": 10,
    },
}

cfg = TRACK_CONFIG[TRACK]
TARGET_COL = cfg["target_col"]
FORECAST_POINTS = cfg["forecast_points"]


# -------------------- Data --------------------
train_df = pd.read_parquet(cfg["train_path"])
test_df = pd.read_parquet(cfg["test_path"])

train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"] = pd.to_datetime(test_df["timestamp"])

train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")
print(f"Routes:      {train_df['route_id'].nunique()}")
print(f"Target col:  {TARGET_COL}")
print(f"Horizon:     {FORECAST_POINTS}")


# -------------------- Forecast helper --------------------
def sarima_forecast_for_timestamps(route_history_df, future_timestamps):
    """
    route_history_df: DataFrame only with past known rows for one route
    future_timestamps: iterable of timestamps to predict
    returns: pd.Series(index=future_timestamps, values=preds)
    """
    future_timestamps = pd.to_datetime(pd.Series(future_timestamps)).sort_values()
    future_index = pd.DatetimeIndex(future_timestamps)

    route_train_30 = (
        route_history_df
        .set_index("timestamp")[TARGET_COL]
        .sort_index()
        .asfreq("30min")
        .interpolate(method="time")
        .bfill()
        .ffill()
    )

    if len(route_train_30) == 0:
        return pd.Series(0.0, index=future_index)

    # берем только последние TRAIN_DAYS истории
    cutoff = route_train_30.index.max() - pd.Timedelta(days=TRAIN_DAYS)
    route_train_30 = route_train_30[route_train_30.index > cutoff]

    # fallback на очень короткий ряд
    if len(route_train_30) < 96:
        mean_val = float(route_train_30.mean())
        return pd.Series(mean_val, index=future_index)

    route_train_1h = route_train_30.resample("1h").mean().interpolate(method="time")

    last_train_ts = route_train_1h.index[-1]
    max_future_ts = future_index.max().ceil("1h")
    n_hours = int((max_future_ts - last_train_ts) / pd.Timedelta("1h")) + 2

    try:
        model = SARIMAX(
            route_train_1h,
            order=SARIMA_ORDER,
            seasonal_order=SARIMA_SEASONAL_ORDER,
            enforce_stationarity=False,
            enforce_invertibility=False,
        )
        result = model.fit(disp=False, maxiter=200, method="lbfgs")

        hourly_idx = pd.date_range(
            start=last_train_ts + pd.Timedelta("1h"),
            periods=n_hours,
            freq="1h",
        )
        forecast_1h = pd.Series(result.forecast(steps=n_hours).values, index=hourly_idx)

        forecast_30min = (
            forecast_1h
            .resample("30min")
            .interpolate(method="linear")
        )

        preds = []
        for ts in future_index:
            pred = float(forecast_30min.get(ts, forecast_30min.iloc[-1]))
            preds.append(max(0.0, pred))
        return pd.Series(preds, index=future_index)

    except Exception:
        preds = []
        for ts in future_index:
            h, m = ts.hour, ts.minute
            mask = (route_train_30.index.hour == h) & (route_train_30.index.minute == m)
            fb = route_train_30[mask].mean() if mask.any() else route_train_30.mean()
            preds.append(max(0.0, float(fb)))
        return pd.Series(preds, index=future_index)


# -------------------- Local validation --------------------
# Последние 8 точек каждого route_id используем как локальный "test"
val_rows = []
route_ids = train_df["route_id"].unique()

for route_id in tqdm(route_ids, desc="Collect local val"):
    route_df = train_df[train_df["route_id"] == route_id].sort_values("timestamp").reset_index(drop=True)

    if len(route_df) <= FORECAST_POINTS + 100:
        continue

    val_part = route_df.tail(FORECAST_POINTS).copy()
    hist_part = route_df.iloc[:-FORECAST_POINTS].copy()

    val_pred = sarima_forecast_for_timestamps(
        route_history_df=hist_part,
        future_timestamps=val_part["timestamp"].values,
    )

    tmp = val_part[["route_id", "timestamp", TARGET_COL]].copy()
    tmp["y_pred"] = val_pred.values
    val_rows.append(tmp)

val_pred_df = pd.concat(val_rows, ignore_index=True).sort_values(["route_id", "timestamp"]).reset_index(drop=True)

y_val_true = val_pred_df[TARGET_COL].values
y_val_pred = val_pred_df["y_pred"].values

print("\n── Val predictions sanity check ──")
print(f"  y_val  : mean={y_val_true.mean():.2f}  std={y_val_true.std():.2f}  max={y_val_true.max():.2f}")
print(f"  y_pred : mean={y_val_pred.mean():.2f}  std={y_val_pred.std():.2f}  max={y_val_pred.max():.2f}")

total_raw, wape_raw, rbias_raw = wape_rbias(y_val_true, y_val_pred)
print("\n── Val (raw) ──")
print(f"  WAPE={wape_raw:.4f}  |RBias|={rbias_raw:.4f}  total={total_raw:.4f}")

# calibration по сумме
calib_scale = float(y_val_true.sum() / (y_val_pred.sum() + 1e-9))
y_val_pred_cal = np.clip(y_val_pred * calib_scale, 0, None)

total_cal, wape_cal, rbias_cal = wape_rbias(y_val_true, y_val_pred_cal)
print(f"── Val (calibrated scale={calib_scale:.4f}) ──")
print(f"  WAPE={wape_cal:.4f}  |RBias|={rbias_cal:.4f}  total={total_cal:.4f}")

val_pred_df["y_pred_cal"] = y_val_pred_cal
val_pred_df.to_csv("sarima_local_validation.csv", index=False)
print("\nSaved: sarima_local_validation.csv")


# -------------------- Full train -> test prediction --------------------
predictions_raw = {}
test_route_ids = test_df["route_id"].unique()

for route_id in tqdm(test_route_ids, desc="SARIMA full train"):
    route_train = train_df[train_df["route_id"] == route_id].sort_values("timestamp").copy()
    route_test = test_df[test_df["route_id"] == route_id].sort_values("timestamp").copy()

    pred_series = sarima_forecast_for_timestamps(
        route_history_df=route_train,
        future_timestamps=route_test["timestamp"].values,
    )

    for (_, row), pred in zip(route_test.iterrows(), pred_series.values):
        predictions_raw[row["id"]] = float(max(0.0, pred))

submission_raw = (
    pd.DataFrame(list(predictions_raw.items()), columns=["id", "y_pred"])
    .sort_values("id")
    .reset_index(drop=True)
)
submission_raw.to_csv("submission_sarima_raw.csv", index=False)

submission_cal = submission_raw.copy()
submission_cal["y_pred"] = np.clip(submission_cal["y_pred"] * calib_scale, 0, None)
submission_cal.to_csv("submission_sarima_calibrated.csv", index=False)

print("\n✅ Saved:")
print("  submission_sarima_raw.csv")
print("  submission_sarima_calibrated.csv")

print("\nRaw submission stats:")
print(submission_raw["y_pred"].describe().round(2))

print("\nCalibrated submission stats:")
print(submission_cal["y_pred"].describe().round(2))

In [ ]:
# ============================================================
# ARIMAX per route — status lags as seasonal proxies
# Быстро: (1,1,1)(0,0,0,0) + exog из status_lag48/lag336
# ============================================================
import warnings
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from tqdm import tqdm

warnings.filterwarnings("ignore")


# ───────────────────────── Metric ────────────────────────────
def wape_rbias(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None)
    denom  = y_true.sum() + 1e-9
    wape   = np.abs(y_pred - y_true).sum() / denom
    rbias  = np.abs(y_pred.sum() / denom - 1)
    return wape + rbias, wape, rbias


# ───────────────────────── Config ────────────────────────────
TRACK = "solo"
TRAIN_DAYS = 21

# Без сезонных членов ARIMA — они заменены exog-лагами
ARIMAX_ORDER = (1, 1, 1)

TRACK_CONFIG = {
    "solo": {
        "train_path":      "train_solo_track.parquet",
        "test_path":       "test_solo_track.parquet",
        "target_col":      "target_1h",
        "forecast_points": 8,
        "status_cols":     [f"status_{i}" for i in range(1, 7)],
    },
    "team": {
        "train_path":      "train_team_track.parquet",
        "test_path":       "test_team_track.parquet",
        "target_col":      "target_2h",
        "forecast_points": 10,
        "status_cols":     [f"status_{i}" for i in range(1, 9)],
    },
}

cfg             = TRACK_CONFIG[TRACK]
TARGET_COL      = cfg["target_col"]
FORECAST_POINTS = cfg["forecast_points"]
STATUS_COLS     = cfg["status_cols"]

# lag48  = то же время вчера (суточная сезонность)
# lag336 = то же время неделю назад (недельная сезонность)
# Оба лага всегда в истории: горизонт теста = 8 шагов << 48
STATUS_LAGS = [48, 336]


# ──────────────────── Build exog ─────────────────────────────
def build_exog_df(series_30: pd.DataFrame) -> pd.DataFrame:
    """
    series_30: DataFrame с DatetimeIndex 30min, колонки STATUS_COLS
    Возвращает DataFrame с DatetimeIndex и exog-колонками.
    Все операции на DatetimeIndex — никаких reset_index.
    """
    assert isinstance(series_30.index, pd.DatetimeIndex)

    out_cols_present = [c for c in ["status_1","status_2","status_3"] if c in series_30.columns]
    in_cols_present  = [c for c in ["status_4","status_5","status_6"] if c in series_30.columns]

    out_flow = series_30[out_cols_present].sum(axis=1) if out_cols_present else pd.Series(0.0, index=series_30.index)
    in_flow  = series_30[in_cols_present].sum(axis=1)  if in_cols_present  else pd.Series(0.0, index=series_30.index)

    result = pd.DataFrame(index=series_30.index)
    for lag in STATUS_LAGS:
        result[f"out_flow_lag{lag}"] = out_flow.shift(lag)
        result[f"in_flow_lag{lag}"]  = in_flow.shift(lag)
        for col in STATUS_COLS:
            if col in series_30.columns:
                result[f"{col}_lag{lag}"] = series_30[col].shift(lag)

    return result


EXOG_COLS = (
    [f"out_flow_lag{l}" for l in STATUS_LAGS] +
    [f"in_flow_lag{l}"  for l in STATUS_LAGS] +
    [f"{c}_lag{l}" for c in STATUS_COLS for l in STATUS_LAGS]
)


# ────────────────── Forecast function ────────────────────────
def arimax_forecast_for_route(route_history_df, future_timestamps):
    """
    route_history_df: полная история маршрута (TARGET_COL + STATUS_COLS)
    future_timestamps: timestamp-ы для предсказания
    """
    future_index = pd.DatetimeIndex(sorted(pd.to_datetime(list(future_timestamps))))

    # ── 1. Плотный 30-мин ряд ─────────────────────────────────
    route_sorted = route_history_df.sort_values("timestamp")
    base = (
        route_sorted
        .set_index("timestamp")
        [[TARGET_COL] + STATUS_COLS]
        .asfreq("30min")
    )
    # interpolate пока DatetimeIndex ✓
    base = base.interpolate(method="time").bfill().ffill()

    # ── 2. Обрезаем до TRAIN_DAYS ─────────────────────────────
    cutoff = base.index.max() - pd.Timedelta(days=TRAIN_DAYS)
    base   = base[base.index > cutoff]

    if len(base) < 96:
        fb = float(base[TARGET_COL].mean()) if len(base) > 0 else 0.0
        return pd.Series(fb, index=future_index)

    # ── 3. Расширяем 30-мин сетку до конца тестового окна ────
    # Будущие строки: STATUS = NaN (но shift(48/336) уйдёт в историю)
    max_ext = max(future_index.max(), base.index.max())
    full_30_idx = pd.date_range(start=base.index.min(), end=max_ext, freq="30min")
    base_ext = base.reindex(full_30_idx)          # DatetimeIndex ✓
    # заполняем статусы только внутри исторической части
    hist_mask = base_ext.index <= base.index.max()
    base_ext.loc[hist_mask, STATUS_COLS] = (
        base_ext.loc[hist_mask, STATUS_COLS].interpolate(method="time").bfill().ffill()
    )

    # ── 4. Считаем exog на расширенном 30-мин ряду ───────────
    exog_30 = build_exog_df(base_ext[STATUS_COLS])  # DatetimeIndex ✓

    # ── 5. Ресэмпл в 1H ──────────────────────────────────────
    # Таргет 1H
    target_1h = (
        base[TARGET_COL]
        .resample("1h").mean()
        .interpolate(method="time")
        .bfill().ffill()
    )
    # Exog 1H — весь расширенный период
    exog_1h = (
        exog_30
        .resample("1h").mean()
        .ffill().bfill()
    )

    X_train  = exog_1h.reindex(target_1h.index).ffill().bfill()[EXOG_COLS]

    # ── 6. Future exog ────────────────────────────────────────
    last_train_ts = target_1h.index[-1]
    n_hours = int((future_index.max().ceil("1h") - last_train_ts) / pd.Timedelta("1h")) + 2
    future_1h_idx = pd.date_range(
        start=last_train_ts + pd.Timedelta("1h"),
        periods=n_hours,
        freq="1h",
    )
    X_future = exog_1h.reindex(future_1h_idx).ffill().bfill()[EXOG_COLS]

    # ── 7. Fit ARIMAX(1,1,1) — без seasonal членов ────────────
    try:
        model = SARIMAX(
            endog=target_1h,
            exog=X_train,
            order=ARIMAX_ORDER,
            seasonal_order=(0, 0, 0, 0),
            enforce_stationarity=False,
            enforce_invertibility=False,
        )
        result = model.fit(disp=False, maxiter=100, method="lbfgs")

        fc_vals = result.forecast(steps=n_hours, exog=X_future)
        fc_1h   = pd.Series(fc_vals.values, index=future_1h_idx)
        fc_30   = fc_1h.resample("30min").interpolate(method="linear")

        preds = []
        for ts in future_index:
            val = fc_30.get(ts, None)
            if val is None or np.isnan(float(val)):
                val = fc_30.iloc[-1]
            preds.append(max(0.0, float(val)))

        return pd.Series(preds, index=future_index)

    except Exception:
        # Fallback: same-time median из истории
        y30   = base[TARGET_COL]
        preds = []
        for ts in future_index:
            mask = (y30.index.hour == ts.hour) & (y30.index.minute == ts.minute)
            fb   = float(y30[mask].mean()) if mask.any() else float(y30.mean())
            preds.append(max(0.0, fb))
        return pd.Series(preds, index=future_index)


# ─────────────────── Local validation ────────────────────────
print("Running local validation...")
val_rows  = []
route_ids = train_df["route_id"].unique()

for route_id in tqdm(route_ids, desc="Val ARIMAX"):
    route_df = (
        train_df[train_df["route_id"] == route_id]
        .sort_values("timestamp")
        .reset_index(drop=True)
    )
    min_len = FORECAST_POINTS + max(STATUS_LAGS) + 50
    if len(route_df) <= min_len:
        continue

    val_part  = route_df.tail(FORECAST_POINTS).copy()
    hist_part = route_df.iloc[:-FORECAST_POINTS].copy()

    preds = arimax_forecast_for_route(hist_part, val_part["timestamp"].values)

    tmp           = val_part[["route_id", "timestamp", TARGET_COL]].copy()
    tmp["y_pred"] = preds.values
    val_rows.append(tmp)

val_df     = pd.concat(val_rows, ignore_index=True)
y_true     = val_df[TARGET_COL].values
y_pred_raw = val_df["y_pred"].values

print("\n── Val sanity check ──")
print(f"  y_val  : mean={y_true.mean():.2f}  std={y_true.std():.2f}  max={y_true.max():.2f}")
print(f"  y_pred : mean={y_pred_raw.mean():.2f}  std={y_pred_raw.std():.2f}  max={y_pred_raw.max():.2f}")

total_raw, wape_raw, rb_raw = wape_rbias(y_true, y_pred_raw)
print(f"\n── Val (raw) ──")
print(f"  WAPE={wape_raw:.4f}  |RBias|={rb_raw:.4f}  total={total_raw:.4f}")

calib_scale = float(y_true.sum() / (y_pred_raw.sum() + 1e-9))
y_pred_cal  = np.clip(y_pred_raw * calib_scale, 0, None)
total_cal, wape_cal, rb_cal = wape_rbias(y_true, y_pred_cal)
print(f"── Val (calibrated scale={calib_scale:.4f}) ──")
print(f"  WAPE={wape_cal:.4f}  |RBias|={rb_cal:.4f}  total={total_cal:.4f}")

val_df["y_pred_cal"] = y_pred_cal
val_df.to_csv("arimax_val.csv", index=False)
print("Saved: arimax_val.csv")


# ──────────────────── Full test prediction ────────────────────
print("\nPredicting test...")
predictions = {}

for route_id in tqdm(test_df["route_id"].unique(), desc="Test ARIMAX"):
    route_train = train_df[train_df["route_id"] == route_id].sort_values("timestamp").copy()
    route_test  = test_df[test_df["route_id"] == route_id].sort_values("timestamp").copy()

    preds = arimax_forecast_for_route(route_train, route_test["timestamp"].values)
    for (_, row), pred in zip(route_test.iterrows(), preds.values):
        predictions[row["id"]] = float(max(0.0, pred))

sub_raw = (
    pd.DataFrame(list(predictions.items()), columns=["id", "y_pred"])
    .sort_values("id").reset_index(drop=True)
)
sub_raw.to_csv("submission_arimax_raw.csv", index=False)

sub_cal = sub_raw.copy()
sub_cal["y_pred"] = np.clip(sub_cal["y_pred"] * calib_scale, 0, None)
sub_cal.to_csv("submission_arimax_calibrated.csv", index=False)

print("\n✅ Saved: submission_arimax_raw.csv / submission_arimax_calibrated.csv")
print(sub_cal["y_pred"].describe().round(2))

In [ ]:
# ============================================================
# SARIMAX per route — SARIMA(1,1,1)(1,1,0,24) + slim exog
# Баг simple_differencing убран, exog сокращён до 2-4 колонок
# Ожидаемое время: ~50-80 мин на 1000 маршрутов
# ============================================================
import warnings
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from tqdm import tqdm

warnings.filterwarnings("ignore")


def wape_rbias(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None)
    denom  = y_true.sum() + 1e-9
    wape   = np.abs(y_pred - y_true).sum() / denom
    rbias  = np.abs(y_pred.sum() / denom - 1)
    return wape + rbias, wape, rbias


# ───────────────────────── Config ─────────────────────────────
TRACK = "solo"
TRAIN_DAYS = 14  # 14 дней = 336 hourly точек — достаточно и быстро

SARIMA_ORDER          = (1, 1, 1)
SARIMA_SEASONAL_ORDER = (1, 1, 0, 24)  # суточная сезонность в часовом ряду

TRACK_CONFIG = {
    "solo": {
        "train_path":      "train_solo_track.parquet",
        "test_path":       "test_solo_track.parquet",
        "target_col":      "target_1h",
        "forecast_points": 8,
        "status_cols":     [f"status_{i}" for i in range(1, 7)],
    },
    "team": {
        "train_path":      "train_team_track.parquet",
        "test_path":       "test_team_track.parquet",
        "target_col":      "target_2h",
        "forecast_points": 10,
        "status_cols":     [f"status_{i}" for i in range(1, 9)],
    },
}

cfg             = TRACK_CONFIG[TRACK]
TARGET_COL      = cfg["target_col"]
FORECAST_POINTS = cfg["forecast_points"]
STATUS_COLS     = cfg["status_cols"]

# Только 2 ключевых exog: суммарный входящий и исходящий поток вчера.
# Минимум параметров → быстрый fit, при этом статусный сигнал есть.
# lag48 = 24ч назад в часовом ряду (48 получасовых шагов).
STATUS_LAG = 48   # одиночный безопасный лаг


# ─────────────────── Build slim exog ──────────────────────────
def build_slim_exog(df_30: pd.DataFrame) -> pd.DataFrame:
    """
    df_30: DatetimeIndex, 30min, колонки STATUS_COLS.
    Возвращает DataFrame с 2 колонками: out_flow_lag48, in_flow_lag48.
    Только 2 колонки → fit в разы быстрее.
    """
    assert isinstance(df_30.index, pd.DatetimeIndex)

    out_cols = [c for c in ["status_1","status_2","status_3"] if c in df_30.columns]
    in_cols  = [c for c in ["status_4","status_5","status_6"] if c in df_30.columns]

    out_flow = df_30[out_cols].sum(axis=1) if out_cols else pd.Series(0.0, index=df_30.index)
    in_flow  = df_30[in_cols].sum(axis=1)  if in_cols  else pd.Series(0.0, index=df_30.index)

    result = pd.DataFrame(index=df_30.index)
    result[f"out_flow_lag{STATUS_LAG}"] = out_flow.shift(STATUS_LAG)
    result[f"in_flow_lag{STATUS_LAG}"]  = in_flow.shift(STATUS_LAG)
    return result


EXOG_COLS = [f"out_flow_lag{STATUS_LAG}", f"in_flow_lag{STATUS_LAG}"]


# ─────────────────── Forecast per route ───────────────────────
def sarimax_forecast_for_route(route_history_df, future_timestamps):
    future_index = pd.DatetimeIndex(sorted(pd.to_datetime(list(future_timestamps))))

    # 1. Плотный 30-мин ряд (DatetimeIndex)
    base = (
        route_history_df
        .sort_values("timestamp")
        .set_index("timestamp")
        [[TARGET_COL] + STATUS_COLS]
        .asfreq("30min")
    )
    base = base.interpolate(method="time").bfill().ffill()

    # 2. Обрезаем до TRAIN_DAYS
    cutoff = base.index.max() - pd.Timedelta(days=TRAIN_DAYS)
    base   = base[base.index > cutoff]

    if len(base) < 96:
        fb = float(base[TARGET_COL].mean()) if len(base) > 0 else 0.0
        return pd.Series(fb, index=future_index)

    # 3. Расширяем сетку до конца тестового окна для корректных lag48
    max_ext     = max(future_index.max(), base.index.max())
    full_30_idx = pd.date_range(start=base.index.min(), end=max_ext, freq="30min")
    base_ext    = base.reindex(full_30_idx)
    hist_mask   = base_ext.index <= base.index.max()
    base_ext.loc[hist_mask, STATUS_COLS] = (
        base_ext.loc[hist_mask, STATUS_COLS].interpolate(method="time").bfill().ffill()
    )

    # 4. Slim exog на 30-мин DatetimeIndex → ресэмпл до 1H
    exog_30 = build_slim_exog(base_ext[STATUS_COLS])

    target_1h = (
        base[TARGET_COL]
        .resample("1h").mean()
        .interpolate(method="time")
        .bfill().ffill()
    )
    exog_1h = exog_30.resample("1h").mean().ffill().bfill()

    X_train = exog_1h.reindex(target_1h.index).ffill().bfill()[EXOG_COLS]

    # 5. Future exog
    last_ts = target_1h.index[-1]
    n_hours = int((future_index.max().ceil("1h") - last_ts) / pd.Timedelta("1h")) + 2
    fut_1h  = pd.date_range(start=last_ts + pd.Timedelta("1h"), periods=n_hours, freq="1h")
    X_future = exog_1h.reindex(fut_1h).ffill().bfill()[EXOG_COLS]

    # 6. Fit SARIMAX — сезонность через ARIMA-члены (надёжно),
    #    exog несёт статусный сигнал. БЕЗ simple_differencing!
    try:
        model = SARIMAX(
            endog=target_1h,
            exog=X_train,
            order=SARIMA_ORDER,
            seasonal_order=SARIMA_SEASONAL_ORDER,
            enforce_stationarity=False,
            enforce_invertibility=False,
            # simple_differencing=False  ← дефолт, явно НЕ включаем
        )
        result = model.fit(
            disp=False,
            maxiter=100,
            method="lbfgs",
        )

        fc     = result.forecast(steps=n_hours, exog=X_future)
        fc_1h  = pd.Series(fc.values, index=fut_1h)
        fc_30  = fc_1h.resample("30min").interpolate(method="linear")

        preds = []
        for ts in future_index:
            val = fc_30.get(ts, None)
            if val is None or np.isnan(float(val)):
                val = fc_30.iloc[-1]
            preds.append(max(0.0, float(val)))
        return pd.Series(preds, index=future_index)

    except Exception:
        y30   = base[TARGET_COL]
        preds = []
        for ts in future_index:
            mask = (y30.index.hour == ts.hour) & (y30.index.minute == ts.minute)
            fb   = float(y30[mask].mean()) if mask.any() else float(y30.mean())
            preds.append(max(0.0, fb))
        return pd.Series(preds, index=future_index)


# ─────────────────── Local validation ─────────────────────────
print("Running local validation...")
val_rows = []

for route_id in tqdm(train_df["route_id"].unique(), desc="Val"):
    route_df = (
        train_df[train_df["route_id"] == route_id]
        .sort_values("timestamp").reset_index(drop=True)
    )
    if len(route_df) <= FORECAST_POINTS + STATUS_LAG + 50:
        continue

    val_part  = route_df.tail(FORECAST_POINTS).copy()
    hist_part = route_df.iloc[:-FORECAST_POINTS].copy()
    preds     = sarimax_forecast_for_route(hist_part, val_part["timestamp"].values)

    tmp           = val_part[["route_id","timestamp",TARGET_COL]].copy()
    tmp["y_pred"] = preds.values
    val_rows.append(tmp)

val_df     = pd.concat(val_rows, ignore_index=True)
y_true     = val_df[TARGET_COL].values
y_pred_raw = val_df["y_pred"].values

print("\n── Val sanity check ──")
print(f"  y_val  : mean={y_true.mean():.2f}  std={y_true.std():.2f}  max={y_true.max():.2f}")
print(f"  y_pred : mean={y_pred_raw.mean():.2f}  std={y_pred_raw.std():.2f}  max={y_pred_raw.max():.2f}")

total_raw, wape_raw, rb_raw = wape_rbias(y_true, y_pred_raw)
print(f"\n── Val (raw) ──")
print(f"  WAPE={wape_raw:.4f}  |RBias|={rb_raw:.4f}  total={total_raw:.4f}")

calib_scale = float(y_true.sum() / (y_pred_raw.sum() + 1e-9))
y_pred_cal  = np.clip(y_pred_raw * calib_scale, 0, None)
total_cal, wape_cal, rb_cal = wape_rbias(y_true, y_pred_cal)
print(f"── Val (calibrated scale={calib_scale:.4f}) ──")
print(f"  WAPE={wape_cal:.4f}  |RBias|={rb_cal:.4f}  total={total_cal:.4f}")

val_df["y_pred_cal"] = y_pred_cal
val_df.to_csv("sarimax_slim_val.csv", index=False)
print("Saved: sarimax_slim_val.csv")


# ─────────────────── Full test prediction ─────────────────────
print("\nPredicting test...")
predictions = {}

for route_id in tqdm(test_df["route_id"].unique(), desc="Test"):
    route_train = train_df[train_df["route_id"] == route_id].sort_values("timestamp").copy()
    route_test  = test_df[test_df["route_id"] == route_id].sort_values("timestamp").copy()

    preds = sarimax_forecast_for_route(route_train, route_test["timestamp"].values)
    for (_, row), pred in zip(route_test.iterrows(), preds.values):
        predictions[row["id"]] = float(max(0.0, pred))

sub_raw = (
    pd.DataFrame(list(predictions.items()), columns=["id","y_pred"])
    .sort_values("id").reset_index(drop=True)
)
sub_raw.to_csv("submission_sarimax_slim_raw.csv", index=False)

sub_cal = sub_raw.copy()
sub_cal["y_pred"] = np.clip(sub_cal["y_pred"] * calib_scale, 0, None)
sub_cal.to_csv("submission_sarimax_slim_calibrated.csv", index=False)

print("\n✅ Saved: submission_sarimax_slim_raw.csv / _calibrated.csv")
print(sub_raw["y_pred"].describe().round(2))

In [ ]:
# ============================================================
# LightGBM Direct Multi-Step — solo track
# ============================================================
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias

# -------------------- Config --------------------
TRAIN_PATH    = "train_solo_track.parquet"
TEST_PATH     = "test_solo_track.parquet"
TARGET_COL    = "target_1h"
FORECAST_STEPS = 8   # горизонт
VAL_DAYS      = 14

# Все возможные лаги — для каждого шага h будем брать только lag >= h+1
ALL_LAGS     = [1, 2, 3, 4, 6, 8, 12, 16, 24, 48, 96, 168, 336]
ROLLING_WINS = [4, 8, 24, 48, 96]

LGB_PARAMS = {
    "objective":         "regression_l1",
    "metric":            "mae",
    "boosting_type":     "gbdt",
    "n_estimators":      800,
    "learning_rate":     0.03,
    "num_leaves":        127,
    "min_child_samples": 20,
    "subsample":         0.8,
    "subsample_freq":    1,
    "colsample_bytree":  0.7,
    "reg_alpha":         0.05,
    "reg_lambda":        1.0,
    "n_jobs":            -1,
    "random_state":      42,
    "verbose":           -1,
}
EARLY_STOPPING = 100

# -------------------- Load --------------------
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

print(f"Train: {train_df.shape}")
print(f"Test:  {test_df.shape}")

# -------------------- Feature engineering --------------------
def time_features(df):
    ts = df["timestamp"]
    df = df.copy()
    df["hh_idx"]    = (ts.dt.hour * 2 + ts.dt.minute // 30).astype(np.int8)
    df["hour"]      = ts.dt.hour.astype(np.int8)
    df["dow"]       = ts.dt.dayofweek.astype(np.int8)
    df["month"]     = ts.dt.month.astype(np.int8)
    df["is_weekend"]= (ts.dt.dayofweek >= 5).astype(np.int8)
    df["hh_sin"]    = np.sin(2 * np.pi * df["hh_idx"] / 48)
    df["hh_cos"]    = np.cos(2 * np.pi * df["hh_idx"] / 48)
    df["dow_sin"]   = np.sin(2 * np.pi * df["dow"] / 7)
    df["dow_cos"]   = np.cos(2 * np.pi * df["dow"] / 7)
    return df

def lag_rolling_features(grp):
    grp = grp.copy()
    tgt = grp[TARGET_COL]
    # Лаги: shift(1) = lag_1 означает "значение на шаг назад относительно текущей строки"
    for lag in ALL_LAGS:
        grp[f"lag_{lag}"] = tgt.shift(lag)
    # Rolling по shift(1) — никогда не смотрит в будущее
    s = tgt.shift(1)
    for w in ROLLING_WINS:
        mp = max(1, w // 4)
        grp[f"rmean_{w}"] = s.rolling(w, min_periods=mp).mean()
        grp[f"rstd_{w}"]  = s.rolling(w, min_periods=mp).std()
    grp["rmax_48"] = s.rolling(48, min_periods=24).max()
    grp["rmin_48"] = s.rolling(48, min_periods=24).min()
    grp["trend_4v8"]   = grp["rmean_4"]  - grp["rmean_8"]
    grp["trend_8v48"]  = grp["rmean_8"]  - grp["rmean_48"]
    grp["trend_48v96"] = grp["rmean_48"] - grp["rmean_96"]
    return grp

print("Building features on train...")
train_feat = time_features(train_df)
train_feat = train_feat.groupby("route_id", group_keys=False).apply(lag_rolling_features)

print("Building features on test...")
# Важно: для теста конкатенируем с трейном, чтобы lag_1 первой тест-строки
# взял последнее значение из трейна
full_df = pd.concat([
    train_df.assign(is_test=False),
    test_df.assign(**{TARGET_COL: np.nan, "is_test": True}),
], ignore_index=True).sort_values(["route_id", "timestamp"]).reset_index(drop=True)

full_feat = time_features(full_df)
full_feat = full_feat.groupby("route_id", group_keys=False).apply(lag_rolling_features)
test_feat = full_feat[full_feat["is_test"]].copy()

# -------------------- Признаки не зависящие от горизонта --------------------
CALENDAR_FEATS = ["hh_idx", "hour", "dow", "month", "is_weekend",
                  "hh_sin", "hh_cos", "dow_sin", "dow_cos", "route_id"]

ROLLING_FEATS  = [c for c in train_feat.columns
                  if c.startswith("rmean_") or c.startswith("rstd_")
                  or c in ("rmax_48", "rmin_48",
                            "trend_4v8", "trend_8v48", "trend_48v96")]

# rolling строились с shift(1) — lag_1-эквивалент, то есть они безопасны только для h>=2
# но реально rmean_96, rmean_48 почти не меняются за 8 шагов → используем для всех h
# (консервативно — если хочешь строго, убери их для h=1)

# -------------------- Val cutoff --------------------
val_cutoff = train_feat["timestamp"].max() - pd.Timedelta(days=VAL_DAYS)
cat_feats  = ["route_id", "dow", "month"]

# -------------------- Direct multi-step loop --------------------
models     = {}
# Используем dict вместо numpy-array с хрупкой индексацией
test_preds_dict = {}  # {(route_id, step): pred}
val_results = []      # для финальной метрики

# Сбрасываем индекс test_feat один раз — избегаем IndexError
test_feat = test_feat.reset_index(drop=True)
test_feat["step_in_route"] = test_feat.groupby("route_id").cumcount() + 1

for h in range(1, FORECAST_STEPS + 1):
    print(f"\n── Step h={h} ──")

    safe_lags = [f"lag_{l}" for l in ALL_LAGS if l >= h]
    FEAT_COLS = CALENDAR_FEATS + ROLLING_FEATS + safe_lags

    # Таргет для горизонта h
    tmp = train_feat.copy()
    tmp[f"target_h{h}"] = tmp.groupby("route_id")[TARGET_COL].shift(-h)

    mask_valid = (
        tmp[f"target_h{h}"].notna() &
        tmp[f"lag_{ALL_LAGS[-1]}"].notna()
    )
    tmp = tmp[mask_valid].copy()

    X_all = tmp[FEAT_COLS]
    y_all = np.log1p(tmp[f"target_h{h}"].clip(lower=0))

    mask_tr  = tmp["timestamp"] <= val_cutoff
    mask_val = tmp["timestamp"] >  val_cutoff

    X_tr, y_tr   = X_all[mask_tr],  y_all[mask_tr]
    X_vl, y_vl   = X_all[mask_val], y_all[mask_val]
    y_vl_orig    = tmp.loc[mask_val, f"target_h{h}"].clip(lower=0)

    print(f"  train={X_tr.shape[0]}  val={X_vl.shape[0]}  feats={len(FEAT_COLS)}")

    dtrain = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_feats, free_raw_data=False)
    dval   = lgb.Dataset(X_vl, label=y_vl, categorical_feature=cat_feats,
                         free_raw_data=False, reference=dtrain)

    model_h = lgb.train(
        LGB_PARAMS,
        dtrain,
        num_boost_round=LGB_PARAMS["n_estimators"],
        valid_sets=[dval],
        callbacks=[
            lgb.early_stopping(EARLY_STOPPING, verbose=False),
            lgb.log_evaluation(period=500),
        ],
    )
    models[h] = (model_h, FEAT_COLS)

    # --- Val метрика ---
    vl_pred = np.expm1(model_h.predict(X_vl)).clip(0)
    tot, wape, rb = wape_rbias(y_vl_orig.values, vl_pred)
    print(f"  h={h}: WAPE={wape:.4f}  |RBias|={rb:.4f}  total={tot:.4f}")
    val_results.append({"h": h, "wape": wape, "rbias": rb, "total": tot})

    # --- Test predictions для шага h (векторно) ---
    mask_step_h = test_feat["step_in_route"] == h
    X_test_h    = test_feat.loc[mask_step_h, FEAT_COLS]

    if len(X_test_h) > 0:
        test_pred_h = np.expm1(model_h.predict(X_test_h)).clip(0)
        for route_id, pred in zip(
            test_feat.loc[mask_step_h, "route_id"].values,
            test_pred_h
        ):
            test_preds_dict[(int(route_id), h)] = float(pred)

# -------------------- Сводная метрика по всем горизонтам --------------------
print("\n── Per-horizon val metrics ──")
val_df = pd.DataFrame(val_results)
print(val_df.to_string(index=False))
print(f"\nMean WAPE: {val_df['wape'].mean():.4f}")
print(f"Mean total: {val_df['total'].mean():.4f}")

# -------------------- Calibration (по h=1) --------------------
model_h1, feat_h1 = models[1]
tmp_h1 = train_feat.copy()
tmp_h1["target_h1"] = tmp_h1.groupby("route_id")[TARGET_COL].shift(-1)
mask_v1 = (tmp_h1["timestamp"] > val_cutoff) & tmp_h1["target_h1"].notna()
val_pred_h1 = np.expm1(model_h1.predict(tmp_h1.loc[mask_v1, feat_h1])).clip(0)
y_true_h1   = tmp_h1.loc[mask_v1, "target_h1"].clip(lower=0).values
CALIB_SCALE = float(y_true_h1.sum() / (val_pred_h1.sum() + 1e-9))
print(f"\nCalibration scale (h=1): {CALIB_SCALE:.4f}")

# -------------------- Submission --------------------
test_out = test_feat.copy()
test_out["step_in_route"] = test_out.groupby("route_id").cumcount() + 1

final_preds = []
for _, row in test_out.iterrows():
    route_id = int(row["route_id"])
    h        = int(row["step_in_route"])
    pred     = test_preds_dict.get((route_id, h), 0.0)
    final_preds.append(max(0.0, pred * CALIB_SCALE))

# id берём из оригинального test_df (отсортированного так же)
submission = test_df[["id"]].copy().reset_index(drop=True)
submission["y_pred"] = final_preds
submission = submission.sort_values("id").reset_index(drop=True)
submission.to_csv("submission_lgb_direct.csv", index=False)

print(f"\n✅ Saved: submission_lgb_direct.csv ({len(submission)} rows)")
print(submission["y_pred"].describe().round(2))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

train = pd.read_parquet("train_solo_track.parquet")
train["timestamp"] = pd.to_datetime(train["timestamp"])
train = train.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

val_cutoff = train["timestamp"].max() - pd.Timedelta(days=14)

def wape(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    return np.abs(yp - yt).sum() / (yt.sum() + 1e-9)

conv = (
    train.groupby("route_id")
    .apply(lambda g: g["target_1h"].sum() / (g["status_3"].sum() + 1e-9))
    .rename("conversion_factor").reset_index()
)
train = train.merge(conv, on="route_id")

# ── Тест 1: длинные лаги до 7 дней ──
print("── Длинные лаги status_3 ──")
results = []
for lag in [48, 96, 144, 192, 240, 288, 336, 480, 672]:
    train[f"pred_lag{lag}"] = (
        train.groupby("route_id")["status_3"]
        .shift(lag) * train["conversion_factor"]
    )
    val = train[train["timestamp"] > val_cutoff].dropna(subset=[f"pred_lag{lag}"])
    w = wape(val["target_1h"].values, val[f"pred_lag{lag}"].values)
    print(f"  lag={lag:>4} ({lag/2:>5.1f}h / {lag/48:>4.1f}d):  WAPE={w:.4f}")
    results.append((lag, w))

best_lag, best_wape = min(results, key=lambda x: x[1])
print(f"\nЛучший лаг: {best_lag} ({best_lag/2:.0f}h / {best_lag/48:.1f}d)  WAPE={best_wape:.4f}")

# ── Тест 2: накопленный буфер (rolling sum) ──
# Идея: товары накапливаются на складе, потом отгружаются партией
# rolling_sum(status_3, window) = накопленный pipeline
print("\n── Накопленный буфер (rolling sum status_3) ──")
for window in [4, 8, 16, 48, 96, 336]:
    col = f"pred_cum{window}"
    train[col] = (
        train.groupby("route_id")["status_3"]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=window//2).sum())
        * train["conversion_factor"] / (window / 2)  # нормируем обратно к 30мин единице
    )
    val = train[train["timestamp"] > val_cutoff].dropna(subset=[col])
    w = wape(val["target_1h"].values, val[col].values)
    print(f"  rolling_sum window={window:>4} ({window/2:>5.1f}h):  WAPE={w:.4f}")

# ── Тест 3: автокорреляция status_3 с target на маршруте 410 ──
r = train[train["route_id"] == 410].copy()
target = r.set_index("timestamp")["target_1h"]
s3     = r.set_index("timestamp")["status_3"]

lags   = list(range(0, 673, 4))
ccf    = [target.corr(s3.shift(lag)) for lag in lags]
best_i = int(np.argmax(np.abs(ccf)))

fig, ax = plt.subplots(figsize=(16, 4))
ax.bar(lags, ccf, width=3, alpha=0.7, color="steelblue")
ax.axvline(lags[best_i], color="red", linewidth=2, linestyle="--",
           label=f"best lag={lags[best_i]} ({lags[best_i]/2:.0f}h), corr={ccf[best_i]:.3f}")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Lag (шагов по 30 мин)")
ax.set_ylabel("Pearson corr")
ax.set_title("Route 410: cross-corr status_3 → target_1h (до 14 дней)")
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("status3_long_lag_ccf.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nRoute 410: best lag = {lags[best_i]} ({lags[best_i]/2:.0f}h), corr = {ccf[best_i]:.4f}")

In [ ]:
# ============================================================
# LightGBM Direct Multi-Step + status features — solo track
# ============================================================
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias

# -------------------- Config --------------------
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8
VAL_DAYS       = 14

ALL_LAGS     = [1, 2, 3, 4, 6, 8, 12, 16, 24, 48, 96, 168, 336]
ROLLING_WINS = [4, 8, 24, 48, 96]

# ── ИЗМЕНЕНИЕ 1: лаги статусов ──
STATUS_LAGS  = [1, 2, 3, 4, 6, 8, 12, 16, 24, 48]

LGB_PARAMS = {
    "objective":         "regression_l1",
    "metric":            "mae",
    "boosting_type":     "gbdt",
    "n_estimators":      800,
    "learning_rate":     0.03,
    "num_leaves":        127,
    "min_child_samples": 20,
    "subsample":         0.8,
    "subsample_freq":    1,
    "colsample_bytree":  0.7,
    "reg_alpha":         0.05,
    "reg_lambda":        1.0,
    "n_jobs":            -1,
    "random_state":      42,
    "verbose":           -1,
}
EARLY_STOPPING = 100

# -------------------- Load --------------------
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

# ── ИЗМЕНЕНИЕ 2: определяем status_cols из трейна ──
status_cols = sorted([c for c in train_df.columns if c.startswith("status_")])
print(f"Train: {train_df.shape}")
print(f"Test:  {test_df.shape}")
print(f"Status cols: {status_cols}")

# -------------------- Feature engineering --------------------
def time_features(df):
    ts = df["timestamp"]
    df = df.copy()
    df["hh_idx"]     = (ts.dt.hour * 2 + ts.dt.minute // 30).astype(np.int8)
    df["hour"]       = ts.dt.hour.astype(np.int8)
    df["dow"]        = ts.dt.dayofweek.astype(np.int8)
    df["month"]      = ts.dt.month.astype(np.int8)
    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(np.int8)
    df["hh_sin"]     = np.sin(2 * np.pi * df["hh_idx"] / 48)
    df["hh_cos"]     = np.cos(2 * np.pi * df["hh_idx"] / 48)
    df["dow_sin"]    = np.sin(2 * np.pi * df["dow"] / 7)
    df["dow_cos"]    = np.cos(2 * np.pi * df["dow"] / 7)
    return df

# ── ИЗМЕНЕНИЕ 3: добавляем лаги статусов и within-route нормализацию ──
def lag_rolling_features(grp):
    grp = grp.copy()
    tgt = grp[TARGET_COL]

    # --- Лаги таргета ---
    for lag in ALL_LAGS:
        grp[f"lag_{lag}"] = tgt.shift(lag)

    # --- Rolling таргета ---
    s = tgt.shift(1)
    for w in ROLLING_WINS:
        mp = max(1, w // 4)
        grp[f"rmean_{w}"] = s.rolling(w, min_periods=mp).mean()
        grp[f"rstd_{w}"]  = s.rolling(w, min_periods=mp).std()
    grp["rmax_48"] = s.rolling(48, min_periods=24).max()
    grp["rmin_48"] = s.rolling(48, min_periods=24).min()
    grp["trend_4v8"]   = grp["rmean_4"]  - grp["rmean_8"]
    grp["trend_8v48"]  = grp["rmean_8"]  - grp["rmean_48"]
    grp["trend_48v96"] = grp["rmean_48"] - grp["rmean_96"]

    # --- Лаги статусов ---
    # shift(lag): при lag=1 для test-строки берёт последнее значение из трейна ✓
    for col in status_cols:
        if col not in grp.columns:
            continue
        for lag in STATUS_LAGS:
            grp[f"{col}_lag{lag}"] = grp[col].shift(lag)

        # Within-route нормализация: убираем кросс-секционную компоненту
        # rolling mean по 7 дням = "норма" для этого маршрута
        norm_base = grp[col].shift(1).rolling(336, min_periods=48).mean()
        grp[f"{col}_norm"] = grp[col].shift(1) / (norm_base + 1e-3)

    return grp

print("Building features on train...")
train_feat = time_features(train_df)
train_feat = train_feat.groupby("route_id", group_keys=False).apply(lag_rolling_features)

print("Building features on test (concat with train for correct lags)...")
# test_df не имеет статусов — при concat они будут NaN для тест-строк,
# но shift(lag) подтянет значения из трейна для первых lag тест-строк ✓
full_df = pd.concat([
    train_df.assign(is_test=False),
    test_df.assign(**{TARGET_COL: np.nan, "is_test": True,
                      **{col: np.nan for col in status_cols
                         if col not in test_df.columns}}),
], ignore_index=True).sort_values(["route_id", "timestamp"]).reset_index(drop=True)

full_feat = time_features(full_df)
full_feat = full_feat.groupby("route_id", group_keys=False).apply(lag_rolling_features)
test_feat = full_feat[full_feat["is_test"]].copy()

# -------------------- Определяем группы признаков --------------------
CALENDAR_FEATS = ["hh_idx", "hour", "dow", "month", "is_weekend",
                  "hh_sin", "hh_cos", "dow_sin", "dow_cos", "route_id"]

ROLLING_FEATS  = [c for c in train_feat.columns
                  if c.startswith("rmean_") or c.startswith("rstd_")
                  or c in ("rmax_48", "rmin_48",
                            "trend_4v8", "trend_8v48", "trend_48v96")]

# Нормализованные статусы безопасны для всех h (shift(1) внутри)
NORM_STATUS_FEATS = [f"{col}_norm" for col in status_cols
                     if f"{col}_norm" in train_feat.columns]

val_cutoff = train_feat["timestamp"].max() - pd.Timedelta(days=VAL_DAYS)
cat_feats  = ["route_id", "dow", "month"]

# -------------------- Direct multi-step loop --------------------
models          = {}
test_preds_dict = {}
val_results     = []

test_feat = test_feat.reset_index(drop=True)
test_feat["step_in_route"] = test_feat.groupby("route_id").cumcount() + 1

for h in range(1, FORECAST_STEPS + 1):
    print(f"\n── Step h={h} ──")

    # Безопасные лаги таргета для горизонта h
    safe_lags = [f"lag_{l}" for l in ALL_LAGS if l >= h]

    # Безопасные лаги статусов для горизонта h
    safe_status_lags = [
        f"{col}_lag{lag}"
        for col in status_cols
        for lag in STATUS_LAGS
        if lag >= h and f"{col}_lag{lag}" in train_feat.columns
    ]

    FEAT_COLS = CALENDAR_FEATS + ROLLING_FEATS + NORM_STATUS_FEATS + safe_lags + safe_status_lags

    # Таргет для горизонта h
    tmp = train_feat.copy()
    tmp[f"target_h{h}"] = tmp.groupby("route_id")[TARGET_COL].shift(-h)

    mask_valid = (
        tmp[f"target_h{h}"].notna() &
        tmp[f"lag_{ALL_LAGS[-1]}"].notna()
    )
    tmp = tmp[mask_valid].copy()

    X_all = tmp[FEAT_COLS]
    y_all = np.log1p(tmp[f"target_h{h}"].clip(lower=0))

    mask_tr  = tmp["timestamp"] <= val_cutoff
    mask_val = tmp["timestamp"] >  val_cutoff

    X_tr, y_tr   = X_all[mask_tr],  y_all[mask_tr]
    X_vl, y_vl   = X_all[mask_val], y_all[mask_val]
    y_vl_orig    = tmp.loc[mask_val, f"target_h{h}"].clip(lower=0)

    print(f"  train={X_tr.shape[0]}  val={X_vl.shape[0]}  feats={len(FEAT_COLS)}")
    print(f"  status feats: {len(safe_status_lags)} lags + {len(NORM_STATUS_FEATS)} norm")

    dtrain = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_feats, free_raw_data=False)
    dval   = lgb.Dataset(X_vl, label=y_vl, categorical_feature=cat_feats,
                         free_raw_data=False, reference=dtrain)

    model_h = lgb.train(
        LGB_PARAMS,
        dtrain,
        num_boost_round=LGB_PARAMS["n_estimators"],
        valid_sets=[dval],
        callbacks=[
            lgb.early_stopping(EARLY_STOPPING, verbose=False),
            lgb.log_evaluation(period=200),
        ],
    )
    models[h] = (model_h, FEAT_COLS)

    # Val метрика
    vl_pred = np.expm1(model_h.predict(X_vl)).clip(0)
    tot, wape, rb = wape_rbias(y_vl_orig.values, vl_pred)
    print(f"  h={h}: WAPE={wape:.4f}  |RBias|={rb:.4f}  total={tot:.4f}")
    val_results.append({"h": h, "wape": wape, "rbias": rb, "total": tot})

    # Feature importance (топ-10 для понимания что модель использует)
    imp = pd.Series(
        model_h.feature_importance(importance_type="gain"),
        index=FEAT_COLS
    ).sort_values(ascending=False)
    print(f"  Top-5 features: {list(imp.head(5).index)}")

    # Test predictions
    mask_step_h = test_feat["step_in_route"] == h
    X_test_h    = test_feat.loc[mask_step_h, FEAT_COLS]
    if len(X_test_h) > 0:
        test_pred_h = np.expm1(model_h.predict(X_test_h)).clip(0)
        for route_id, pred in zip(
            test_feat.loc[mask_step_h, "route_id"].values,
            test_pred_h
        ):
            test_preds_dict[(int(route_id), h)] = float(pred)

# -------------------- Сводная метрика --------------------
print("\n── Per-horizon val metrics ──")
val_df = pd.DataFrame(val_results)
print(val_df.to_string(index=False))
print(f"\nMean WAPE:  {val_df['wape'].mean():.4f}")
print(f"Mean total: {val_df['total'].mean():.4f}")

# -------------------- Calibration --------------------
model_h1, feat_h1 = models[1]
tmp_h1 = train_feat.copy()
tmp_h1["target_h1"] = tmp_h1.groupby("route_id")[TARGET_COL].shift(-1)
mask_v1 = (tmp_h1["timestamp"] > val_cutoff) & tmp_h1["target_h1"].notna()
val_pred_h1 = np.expm1(model_h1.predict(tmp_h1.loc[mask_v1, feat_h1])).clip(0)
y_true_h1   = tmp_h1.loc[mask_v1, "target_h1"].clip(lower=0).values
CALIB_SCALE = float(y_true_h1.sum() / (val_pred_h1.sum() + 1e-9))
print(f"\nCalibration scale (h=1): {CALIB_SCALE:.4f}")

# -------------------- Submission --------------------
test_out = test_feat.copy()
test_out["step_in_route"] = test_out.groupby("route_id").cumcount() + 1

final_preds = []
for _, row in test_out.iterrows():
    route_id = int(row["route_id"])
    h        = int(row["step_in_route"])
    pred     = test_preds_dict.get((route_id, h), 0.0)
    final_preds.append(max(0.0, pred * CALIB_SCALE))

submission = test_df[["id"]].copy().reset_index(drop=True)
submission["y_pred"] = final_preds
submission = submission.sort_values("id").reset_index(drop=True)
submission.to_csv("submission_lgb_direct_v2.csv", index=False)

print(f"\n✅ Saved: submission_lgb_direct_v2.csv ({len(submission)} rows)")
print(submission["y_pred"].describe().round(2))

In [ ]:
# ============================================================
# LightGBM Direct Multi-Step v3
#   - status lags с корректным norm8 (shift(8))
#   - весь трейн (без lag_336.notna() фильтра)
#   - per-horizon calibration
#   - + глобальная модель с фичей h для сравнения
# ============================================================
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias

# -------------------- Config --------------------
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8
VAL_DAYS       = 14

ALL_LAGS     = [1, 2, 3, 4, 6, 8, 12, 16, 24, 48, 96, 168, 336]
ROLLING_WINS = [4, 8, 24, 48, 96]
STATUS_LAGS  = [1, 2, 3, 4, 6, 8, 12, 16, 24, 48]
MAX_H        = FORECAST_STEPS   # для norm8 используем shift(MAX_H)

LGB_PARAMS = {
    "objective":         "regression_l1",
    "metric":            "mae",
    "boosting_type":     "gbdt",
    "n_estimators":      800,
    "learning_rate":     0.03,
    "num_leaves":        127,
    "min_child_samples": 20,
    "subsample":         0.8,
    "subsample_freq":    1,
    "colsample_bytree":  0.7,
    "reg_alpha":         0.05,
    "reg_lambda":        1.0,
    "n_jobs":            -1,
    "random_state":      42,
    "verbose":           -1,
}
EARLY_STOPPING = 100

# -------------------- Load --------------------
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

status_cols = sorted([c for c in train_df.columns if c.startswith("status_")])
print(f"Train: {train_df.shape}  Test: {test_df.shape}")
print(f"Status cols: {status_cols}")

# -------------------- Feature engineering --------------------
def time_features(df):
    ts = df["timestamp"]
    df = df.copy()
    df["hh_idx"]     = (ts.dt.hour * 2 + ts.dt.minute // 30).astype(np.int8)
    df["hour"]       = ts.dt.hour.astype(np.int8)
    df["dow"]        = ts.dt.dayofweek.astype(np.int8)
    df["month"]      = ts.dt.month.astype(np.int8)
    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(np.int8)
    df["hh_sin"]     = np.sin(2 * np.pi * df["hh_idx"] / 48)
    df["hh_cos"]     = np.cos(2 * np.pi * df["hh_idx"] / 48)
    df["dow_sin"]    = np.sin(2 * np.pi * df["dow"] / 7)
    df["dow_cos"]    = np.cos(2 * np.pi * df["dow"] / 7)
    return df

def lag_rolling_features(grp):
    grp = grp.copy()
    tgt = grp[TARGET_COL]

    # --- Лаги таргета ---
    for lag in ALL_LAGS:
        grp[f"lag_{lag}"] = tgt.shift(lag)

    # --- Rolling таргета (shift(1) → безопасны для h>=1) ---
    s = tgt.shift(1)
    for w in ROLLING_WINS:
        mp = max(1, w // 4)
        grp[f"rmean_{w}"] = s.rolling(w, min_periods=mp).mean()
        grp[f"rstd_{w}"]  = s.rolling(w, min_periods=mp).std()
    grp["rmax_48"] = s.rolling(48, min_periods=24).max()
    grp["rmin_48"] = s.rolling(48, min_periods=24).min()
    grp["trend_4v8"]   = grp["rmean_4"]  - grp["rmean_8"]
    grp["trend_8v48"]  = grp["rmean_8"]  - grp["rmean_48"]
    grp["trend_48v96"] = grp["rmean_48"] - grp["rmean_96"]

    # --- Лаги статусов ---
    for col in status_cols:
        if col not in grp.columns:
            continue
        for lag in STATUS_LAGS:
            grp[f"{col}_lag{lag}"] = grp[col].shift(lag)

        # norm8: shift(MAX_H=8) → для любой тест-строки h<=8
        # берёт значение из трейна, NaN никогда не будет
        norm_base = grp[col].shift(MAX_H).rolling(336, min_periods=48).mean()
        grp[f"{col}_norm8"] = grp[col].shift(MAX_H) / (norm_base + 1e-3)

    return grp

print("Building features on train...")
train_feat = time_features(train_df)
train_feat = train_feat.groupby("route_id", group_keys=False).apply(lag_rolling_features)

# Конкатенируем трейн + тест чтобы shift подтянул статусы из трейна в первые 8 строк теста
print("Building features on test...")
full_df = pd.concat([
    train_df.assign(is_test=False),
    test_df.assign(**{
        TARGET_COL: np.nan,
        "is_test": True,
        **{col: np.nan for col in status_cols if col not in test_df.columns}
    }),
], ignore_index=True).sort_values(["route_id", "timestamp"]).reset_index(drop=True)

full_feat = time_features(full_df)
full_feat = full_feat.groupby("route_id", group_keys=False).apply(lag_rolling_features)
test_feat = full_feat[full_feat["is_test"]].copy().reset_index(drop=True)
test_feat["step_in_route"] = test_feat.groupby("route_id").cumcount() + 1

# -------------------- Определяем группы признаков --------------------
CALENDAR_FEATS = ["hh_idx", "hour", "dow", "month", "is_weekend",
                  "hh_sin", "hh_cos", "dow_sin", "dow_cos", "route_id"]

ROLLING_FEATS = [c for c in train_feat.columns
                 if c.startswith("rmean_") or c.startswith("rstd_")
                 or c in ("rmax_48", "rmin_48",
                           "trend_4v8", "trend_8v48", "trend_48v96")]

# norm8 безопасна для ВСЕХ h <= 8 (внутри shift(8))
NORM_STATUS_FEATS = [f"{col}_norm8" for col in status_cols
                     if f"{col}_norm8" in train_feat.columns]

val_cutoff = train_feat["timestamp"].max() - pd.Timedelta(days=VAL_DAYS)
cat_feats  = ["route_id", "dow", "month"]

# ============================================================
# ЧАСТЬ 1: Direct multi-step (отдельная модель на каждый h)
# ============================================================
models          = {}
test_preds_dict = {}
val_results     = []
calib_scales    = {}

for h in range(1, FORECAST_STEPS + 1):
    print(f"\n── Step h={h} ──")

    safe_lags = [f"lag_{l}" for l in ALL_LAGS if l >= h]
    safe_status_lags = [
        f"{col}_lag{lag}"
        for col in status_cols
        for lag in STATUS_LAGS
        if lag >= h and f"{col}_lag{lag}" in train_feat.columns
    ]
    FEAT_COLS = CALENDAR_FEATS + ROLLING_FEATS + NORM_STATUS_FEATS + safe_lags + safe_status_lags

    # Таргет h шагов вперёд
    tmp = train_feat.copy()
    tmp[f"target_h{h}"] = tmp.groupby("route_id")[TARGET_COL].shift(-h)

    # ── ИСПОЛЬЗУЕМ ВЕСЬ ТРЕЙН: убираем lag_336.notna() фильтр ──
    # LightGBM сам обрабатывает NaN в лагах
    tmp = tmp[tmp[f"target_h{h}"].notna()].copy()

    X_all = tmp[FEAT_COLS]
    y_all = np.log1p(tmp[f"target_h{h}"].clip(lower=0))

    mask_tr  = tmp["timestamp"] <= val_cutoff
    mask_val = tmp["timestamp"] >  val_cutoff

    X_tr, y_tr = X_all[mask_tr],  y_all[mask_tr]
    X_vl, y_vl = X_all[mask_val], y_all[mask_val]
    y_vl_orig  = tmp.loc[mask_val, f"target_h{h}"].clip(lower=0)

    print(f"  train={X_tr.shape[0]}  val={X_vl.shape[0]}  feats={len(FEAT_COLS)}")

    dtrain = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_feats, free_raw_data=False)
    dval   = lgb.Dataset(X_vl, label=y_vl, categorical_feature=cat_feats,
                         free_raw_data=False, reference=dtrain)

    model_h = lgb.train(
        LGB_PARAMS,
        dtrain,
        num_boost_round=LGB_PARAMS["n_estimators"],
        valid_sets=[dval],
        callbacks=[
            lgb.early_stopping(EARLY_STOPPING, verbose=False),
            lgb.log_evaluation(period=200),
        ],
    )
    models[h] = (model_h, FEAT_COLS)

    vl_pred = np.expm1(model_h.predict(X_vl)).clip(0)
    tot, wape, rb = wape_rbias(y_vl_orig.values, vl_pred)
    print(f"  h={h}: WAPE={wape:.4f}  |RBias|={rb:.4f}  total={tot:.4f}")
    val_results.append({"h": h, "wape": wape, "rbias": rb, "total": tot})

    # ── Per-horizon calibration ──
    calib_scales[h] = float(y_vl_orig.values.sum() / (vl_pred.sum() + 1e-9))
    print(f"  calib_scale(h={h}): {calib_scales[h]:.4f}")

    imp = pd.Series(model_h.feature_importance(importance_type="gain"),
                    index=FEAT_COLS).sort_values(ascending=False)
    print(f"  Top-5: {list(imp.head(5).index)}")

    # Test predictions
    mask_step_h = test_feat["step_in_route"] == h
    X_test_h    = test_feat.loc[mask_step_h, FEAT_COLS]
    if len(X_test_h) > 0:
        preds_h = np.expm1(model_h.predict(X_test_h)).clip(0)
        for rid, pred in zip(test_feat.loc[mask_step_h, "route_id"].values, preds_h):
            test_preds_dict[(int(rid), h)] = float(pred)

print("\n── Per-horizon val metrics ──")
val_df = pd.DataFrame(val_results)
print(val_df.to_string(index=False))
print(f"\nMean WAPE:  {val_df['wape'].mean():.4f}")
print(f"Mean total: {val_df['total'].mean():.4f}")

# Submission v3 (per-horizon calib)
test_out = test_feat.copy()
final_preds = []
for _, row in test_out.iterrows():
    rid  = int(row["route_id"])
    h    = int(row["step_in_route"])
    pred = test_preds_dict.get((rid, h), 0.0)
    final_preds.append(max(0.0, pred * calib_scales[h]))

submission_v3 = test_df[["id"]].copy().reset_index(drop=True)
submission_v3["y_pred"] = final_preds
submission_v3 = submission_v3.sort_values("id").reset_index(drop=True)
submission_v3.to_csv("submission_lgb_direct_v3.csv", index=False)
print(f"\n✅ Saved: submission_lgb_direct_v3.csv")
print(submission_v3["y_pred"].describe().round(2))

In [ ]:
# ============================================================
# LightGBM Single Global Model — все горизонты + Optuna тюнинг
# Solo track: target_1h, horizon = 8 steps
# ============================================================
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None
optuna.logging.set_verbosity(optuna.logging.WARNING)


def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias


# ─────────────────── Config ───────────────────────────────────
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8
N_TRIALS       = 50   # Optuna trials (увеличь до 100 если есть время)

# Val: последний тест-подобный холдаут (8 строк per route)
VAL_ROWS_PER_ROUTE = 8

ALL_LAGS     = [1, 2, 3, 4, 6, 8, 12, 16, 24, 48, 96, 168, 336]
ROLLING_WINS = [4, 8, 24, 48, 96]

STATUS_COLS = [f"status_{i}" for i in range(1, 7)]


# ─────────────────── Load ─────────────────────────────────────
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

print(f"Train: {train_df.shape}")
print(f"Test:  {test_df.shape}")


# ─────────────────── Feature engineering ──────────────────────
def time_features(df):
    ts   = df["timestamp"]
    df   = df.copy()
    hh   = ts.dt.hour * 2 + ts.dt.minute // 30
    df["hh_idx"]     = hh.astype(np.int8)
    df["hour"]       = ts.dt.hour.astype(np.int8)
    df["dow"]        = ts.dt.dayofweek.astype(np.int8)
    df["month"]      = ts.dt.month.astype(np.int8)
    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(np.int8)
    df["hh_sin"]     = np.sin(2 * np.pi * hh / 48)
    df["hh_cos"]     = np.cos(2 * np.pi * hh / 48)
    df["dow_sin"]    = np.sin(2 * np.pi * df["dow"] / 7)
    df["dow_cos"]    = np.cos(2 * np.pi * df["dow"] / 7)
    return df


def lag_rolling_features(grp):
    grp = grp.copy()
    tgt = grp[TARGET_COL]
    for lag in ALL_LAGS:
        grp[f"lag_{lag}"] = tgt.shift(lag)
    s = tgt.shift(1)
    for w in ROLLING_WINS:
        mp = max(1, w // 4)
        grp[f"rmean_{w}"]  = s.rolling(w, min_periods=mp).mean()
        grp[f"rstd_{w}"]   = s.rolling(w, min_periods=mp).std()
    grp["rmax_48"]       = s.rolling(48, min_periods=24).max()
    grp["rmin_48"]       = s.rolling(48, min_periods=24).min()
    grp["trend_4v8"]     = grp["rmean_4"]  - grp["rmean_8"]
    grp["trend_8v48"]    = grp["rmean_8"]  - grp["rmean_48"]
    grp["trend_48v96"]   = grp["rmean_48"] - grp["rmean_96"]
    # Статусы с безопасным лагом 8 (минимальный горизонт)
    for col in STATUS_COLS:
        if col in grp.columns:
            grp[f"{col}_lag8"]   = grp[col].shift(8)
            grp[f"{col}_lag48"]  = grp[col].shift(48)
    return grp


print("Building features on train...")
train_feat = time_features(train_df)
train_feat = train_feat.groupby("route_id", group_keys=False).apply(lag_rolling_features)

print("Building features on test (concat with train for proper lags)...")
full_df = pd.concat([
    train_df.assign(is_test=False),
    test_df.assign(**{TARGET_COL: np.nan, "is_test": True}),
], ignore_index=True).sort_values(["route_id", "timestamp"]).reset_index(drop=True)
full_feat = time_features(full_df)
full_feat = full_feat.groupby("route_id", group_keys=False).apply(lag_rolling_features)
test_feat = full_feat[full_feat["is_test"]].copy().reset_index(drop=True)
test_feat["step_in_route"] = test_feat.groupby("route_id").cumcount() + 1


# ─────────────────── Build multi-step dataset ─────────────────
# Разворачиваем: для каждой строки создаём FORECAST_STEPS записей,
# каждая с целевым значением target_h и горизонтом h как признак.
# Лаги фильтруются: оставляем только lag >= h (нет leakage).

BASE_LAG_FEATS  = [f"lag_{l}" for l in ALL_LAGS]
ROLLING_FEATS   = [c for c in train_feat.columns
                   if c.startswith("rmean_") or c.startswith("rstd_")
                   or c in ("rmax_48", "rmin_48",
                             "trend_4v8", "trend_8v48", "trend_48v96")]
STATUS_FEATS    = [c for c in train_feat.columns
                   if any(c.startswith(f"{s}_lag") for s in STATUS_COLS)]
CALENDAR_FEATS  = ["hh_idx", "hour", "dow", "month", "is_weekend",
                   "hh_sin", "hh_cos", "dow_sin", "dow_cos", "route_id"]

# Все признаки кроме лагов (они фильтруются per-h)
BASE_FEATS = CALENDAR_FEATS + ROLLING_FEATS + STATUS_FEATS


def build_multistep_dataset(feat_df, max_lag_needed=ALL_LAGS[-1]):
    """
    Разворачивает DataFrame в multi-step формат.
    Добавляет колонки: target_shifted, h, safe_lag_*
    """
    chunks = []
    for h in range(1, FORECAST_STEPS + 1):
        chunk = feat_df.copy()
        # Таргет для горизонта h
        chunk["target_shifted"] = chunk.groupby("route_id")[TARGET_COL].shift(-h)
        chunk["h"] = np.int8(h)
        # Обнуляем лаги < h (они бы смотрели в будущее)
        for lag in ALL_LAGS:
            if lag < h:
                chunk[f"lag_{lag}"] = np.nan
        chunks.append(chunk)
    return pd.concat(chunks, ignore_index=True)


print("Building multi-step dataset...")
ms_train = build_multistep_dataset(train_feat)
ms_train = ms_train[
    ms_train["target_shifted"].notna() &
    ms_train[f"lag_{ALL_LAGS[-1]}"].notna()
].copy()

# Логарифм таргета
ms_train["y"] = np.log1p(ms_train["target_shifted"].clip(lower=0))

ALL_FEATS = BASE_FEATS + BASE_LAG_FEATS + ["h"]
CAT_FEATS = ["route_id", "dow", "month"]

print(f"Multi-step dataset: {ms_train.shape}, feats={len(ALL_FEATS)}")


# ─────────────────── Holodaut = тест-подобный ─────────────────
# Для каждого route_id: последние VAL_ROWS_PER_ROUTE строк train
# в разных горизонтах h — точно такой же профиль, как тест
val_anchor_idx = (
    train_feat
    .groupby("route_id")
    .apply(lambda g: g.index[-(VAL_ROWS_PER_ROUTE):])
    .explode()
    .values
)

mask_val  = ms_train.index.isin(val_anchor_idx)
# Нет — нам нужны строки, у которых anchor_row (до shift) в val_anchor_idx
# Делаем через original timestamp
val_ts_set = set(
    train_feat.loc[train_feat.groupby("route_id")
                   .apply(lambda g: g.index[-(VAL_ROWS_PER_ROUTE):])
                   .explode().values, "timestamp"]
)

mask_val  = ms_train["timestamp"].isin(val_ts_set)
mask_train = ~mask_val

X_tr  = ms_train.loc[mask_train, ALL_FEATS]
y_tr  = ms_train.loc[mask_train, "y"]
X_vl  = ms_train.loc[mask_val,   ALL_FEATS]
y_vl  = ms_train.loc[mask_val,   "y"]
y_vl_orig = ms_train.loc[mask_val, "target_shifted"].clip(lower=0).values

print(f"Train rows: {len(X_tr)}  Val rows: {len(X_vl)}")




In [ ]:
# ─────────────────── Optuna objective ─────────────────────────
def objective(trial):
    params = {
        "objective":         "regression_l1",
        "metric":            "mae",         # своя метрика
        "boosting_type":     "gbdt",
        "n_estimators":      trial.suggest_int("n_estimators", 400, 1500),
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "num_leaves":        trial.suggest_int("num_leaves", 31, 255),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
        "subsample_freq":    1,
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "max_depth":         trial.suggest_int("max_depth", 4, 12),
        "n_jobs":            -1,
        "random_state":      42,
        "verbose":           -1,
    }

    dtrain = lgb.Dataset(X_tr, label=y_tr, categorical_feature=CAT_FEATS, free_raw_data=False)
    dval   = lgb.Dataset(X_vl, label=y_vl, categorical_feature=CAT_FEATS,
                         free_raw_data=False, reference=dtrain)

    model = lgb.train(
        params,
        dtrain,
        num_boost_round=params["n_estimators"],
        valid_sets=[dval],
        callbacks=[
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(period=9999),
        ],
    )

    pred = np.expm1(model.predict(X_vl)).clip(0)
    total, _, _ = wape_rbias(y_vl_orig, pred)
    return total


print(f"\nRunning Optuna ({N_TRIALS} trials)...")
study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best = study.best_params
print(f"\nBest trial: {study.best_value:.4f}")
print(f"Best params: {best}")


# ─────────────────── Final model on best params ───────────────
FINAL_PARAMS = {
    "objective":         "regression_l1",
    "metric":            "mae",
    "boosting_type":     "gbdt",
    "n_estimators":      best["n_estimators"],
    "learning_rate":     best["learning_rate"],
    "num_leaves":        best["num_leaves"],
    "min_child_samples": best["min_child_samples"],
    "subsample":         best["subsample"],
    "subsample_freq":    1,
    "colsample_bytree":  best["colsample_bytree"],
    "reg_alpha":         best["reg_alpha"],
    "reg_lambda":        best["reg_lambda"],
    "max_depth":         best["max_depth"],
    "n_jobs":            -1,
    "random_state":      42,
    "verbose":           -1,
}

print("\nTraining final model on full train...")
dtrain_full = lgb.Dataset(
    ms_train[ALL_FEATS], label=ms_train["y"],
    categorical_feature=CAT_FEATS, free_raw_data=False
)
final_model = lgb.train(
    FINAL_PARAMS,
    dtrain_full,
    num_boost_round=int(best["n_estimators"] * 1.05),  # +5% без early stopping
    callbacks=[lgb.log_evaluation(period=200)],
)
final_model.save_model("lgb_global_model.txt")
print("Saved: lgb_global_model.txt")


# ─────────────────── Validation metric (итоговая) ─────────────
print("\n── Final val metrics per horizon ──")
for h in range(1, FORECAST_STEPS + 1):
    mask_h = mask_val & (ms_train["h"] == h)
    if mask_h.sum() == 0:
        continue
    pred_h = np.expm1(final_model.predict(ms_train.loc[mask_h, ALL_FEATS])).clip(0)
    true_h = ms_train.loc[mask_h, "target_shifted"].clip(lower=0).values
    tot, wape, rb = wape_rbias(true_h, pred_h)
    print(f"  h={h}: WAPE={wape:.4f}  |RBias|={rb:.4f}  total={tot:.4f}")

pred_all = np.expm1(final_model.predict(X_vl)).clip(0)
tot_all, wape_all, rb_all = wape_rbias(y_vl_orig, pred_all)
print(f"\n── Overall val ──")
print(f"  WAPE={wape_all:.4f}  |RBias|={rb_all:.4f}  total={tot_all:.4f}")

# Calibration
calib_scale = float(y_vl_orig.sum() / (pred_all.sum() + 1e-9))
pred_cal = np.clip(pred_all * calib_scale, 0, None)
tot_cal, wape_cal, rb_cal = wape_rbias(y_vl_orig, pred_cal)
print(f"── Val (calibrated scale={calib_scale:.4f}) ──")
print(f"  WAPE={wape_cal:.4f}  |RBias|={rb_cal:.4f}  total={tot_cal:.4f}")


# ─────────────────── Test predictions ─────────────────────────
print("\nPredicting test...")
test_rows = []
for h in range(1, FORECAST_STEPS + 1):
    mask_h = test_feat["step_in_route"] == h
    chunk  = test_feat[mask_h].copy()
    # Обнуляем лаги < h для консистентности с трейном
    for lag in ALL_LAGS:
        if lag < h and f"lag_{lag}" in chunk.columns:
            chunk[f"lag_{lag}"] = np.nan
    chunk["h"] = np.int8(h)
    chunk["pred"] = np.expm1(final_model.predict(chunk[ALL_FEATS])).clip(0)
    test_rows.append(chunk[["route_id", "step_in_route", "pred"]])

test_pred_df = pd.concat(test_rows).sort_values(["route_id", "step_in_route"])

# Склеиваем с оригинальным test_df по route_id + порядку
test_out = test_feat.copy()
test_out["pred_raw"] = 0.0
for h in range(1, FORECAST_STEPS + 1):
    m = test_pred_df["step_in_route"] == h
    test_out.loc[test_out["step_in_route"] == h, "pred_raw"] = test_pred_df.loc[m, "pred"].values

# Submission raw
sub_raw = test_df[["id"]].copy().reset_index(drop=True)
sub_raw["y_pred"] = test_out.sort_values(["route_id", "step_in_route"])["pred_raw"].values
sub_raw = sub_raw.sort_values("id").reset_index(drop=True)
sub_raw.to_csv("submission_lgb_global_raw.csv", index=False)

# Submission calibrated
sub_cal = sub_raw.copy()
sub_cal["y_pred"] = np.clip(sub_cal["y_pred"] * calib_scale, 0, None)
sub_cal.to_csv("submission_lgb_global_calibrated.csv", index=False)

print("\n✅ Saved:")
print("  submission_lgb_global_raw.csv")
print("  submission_lgb_global_calibrated.csv")
print(sub_raw["y_pred"].describe().round(2))

In [ ]:


# ─────────────────── Final model on best params ───────────────
FINAL_PARAMS = {
    "objective":         "regression_l1",
    "metric":            "mae",
    "boosting_type":     "gbdt",
    "n_estimators":      2000,
    "learning_rate":     0.03,
    "num_leaves":        127,
    "min_child_samples": 20,
    "subsample":         0.8,
    "subsample_freq":    1,
    "colsample_bytree":  0.7,
    "reg_alpha":         0.05,
    "reg_lambda":        1.0,
    "n_jobs":            -1,
    "random_state":      42,
    "verbose":           -1,
}

print("\nTraining final model on full train...")
dtrain_full = lgb.Dataset(
    ms_train[ALL_FEATS], label=ms_train["y"],
    categorical_feature=CAT_FEATS, free_raw_data=False
)
final_model = lgb.train(
    FINAL_PARAMS,
    dtrain_full,
    num_boost_round=3000,  # +5% без early stopping
    callbacks=[lgb.log_evaluation(period=200)],
)
final_model.save_model("lgb_global_model.txt")
print("Saved: lgb_global_model.txt")


# ─────────────────── Validation metric (итоговая) ─────────────
print("\n── Final val metrics per horizon ──")
for h in range(1, FORECAST_STEPS + 1):
    mask_h = mask_val & (ms_train["h"] == h)
    if mask_h.sum() == 0:
        continue
    pred_h = np.expm1(final_model.predict(ms_train.loc[mask_h, ALL_FEATS])).clip(0)
    true_h = ms_train.loc[mask_h, "target_shifted"].clip(lower=0).values
    tot, wape, rb = wape_rbias(true_h, pred_h)
    print(f"  h={h}: WAPE={wape:.4f}  |RBias|={rb:.4f}  total={tot:.4f}")

pred_all = np.expm1(final_model.predict(X_vl)).clip(0)
tot_all, wape_all, rb_all = wape_rbias(y_vl_orig, pred_all)
print(f"\n── Overall val ──")
print(f"  WAPE={wape_all:.4f}  |RBias|={rb_all:.4f}  total={tot_all:.4f}")

# Calibration
calib_scale = float(y_vl_orig.sum() / (pred_all.sum() + 1e-9))
pred_cal = np.clip(pred_all * calib_scale, 0, None)
tot_cal, wape_cal, rb_cal = wape_rbias(y_vl_orig, pred_cal)
print(f"── Val (calibrated scale={calib_scale:.4f}) ──")
print(f"  WAPE={wape_cal:.4f}  |RBias|={rb_cal:.4f}  total={tot_cal:.4f}")


# ─────────────────── Test predictions ─────────────────────────
print("\nPredicting test...")
test_rows = []
for h in range(1, FORECAST_STEPS + 1):
    mask_h = test_feat["step_in_route"] == h
    chunk  = test_feat[mask_h].copy()
    # Обнуляем лаги < h для консистентности с трейном
    for lag in ALL_LAGS:
        if lag < h and f"lag_{lag}" in chunk.columns:
            chunk[f"lag_{lag}"] = np.nan
    chunk["h"] = np.int8(h)
    chunk["pred"] = np.expm1(final_model.predict(chunk[ALL_FEATS])).clip(0)
    test_rows.append(chunk[["route_id", "step_in_route", "pred"]])

test_pred_df = pd.concat(test_rows).sort_values(["route_id", "step_in_route"])

# Склеиваем с оригинальным test_df по route_id + порядку
test_out = test_feat.copy()
test_out["pred_raw"] = 0.0
for h in range(1, FORECAST_STEPS + 1):
    m = test_pred_df["step_in_route"] == h
    test_out.loc[test_out["step_in_route"] == h, "pred_raw"] = test_pred_df.loc[m, "pred"].values

# Submission raw
sub_raw = test_df[["id"]].copy().reset_index(drop=True)
sub_raw["y_pred"] = test_out.sort_values(["route_id", "step_in_route"])["pred_raw"].values
sub_raw = sub_raw.sort_values("id").reset_index(drop=True)
sub_raw.to_csv("submission_lgb_global_raw.csv", index=False)

# Submission calibrated
sub_cal = sub_raw.copy()
sub_cal["y_pred"] = np.clip(sub_cal["y_pred"] * calib_scale, 0, None)
sub_cal.to_csv("submission_lgb_global_calibrated.csv", index=False)

print("\n✅ Saved:")
print("  submission_lgb_global_raw.csv")
print("  submission_lgb_global_calibrated.csv")
print(sub_raw["y_pred"].describe().round(2))

In [ ]:
# ============================================================
# TimesFM zero-shot inference on Mac (Apple Silicon / Intel)
# Task: Solo track, target1h, horizon=8 (30-min steps)
# ============================================================

import warnings
import numpy as np
import pandas as pd
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ---- Install if needed ----
# pip install transformers torch accelerate

import torch
from transformers import TimesFmConfig, TimesFmModelForPrediction

# ============================================================
# CONFIG
# ============================================================
TRACK = "solo"
TRACKCONFIG = {
    "solo": {
        "trainpath": "trainsolotrack.parquet",
        "testpath":  "testsolotrack.parquet",
        "targetcol": "target1h",
        "forecastpoints": 8,
    },
    "team": {
        "trainpath": "trainteamtrack.parquet",
        "testpath":  "testteamtrack.parquet",
        "targetcol": "target2h",
        "forecastpoints": 10,
    },
}
cfg = TRACKCONFIG[TRACK]
TARGETCOL = cfg["targetcol"]
FORECAST_POINTS = cfg["forecastpoints"]  # 8 шагов по 30 мин

# Сколько истории давать модели (контекстное окно)
# TimesFM принимает от 32 до 512 точек оптимально
CONTEXT_LEN = 336  # 7 суток по 30 мин

# ============================================================
# DEVICE — MPS для Apple Silicon, иначе CPU
# ============================================================
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(f"Using device: {DEVICE}")

# ============================================================
# LOAD MODEL
# ============================================================
# timesfm-2.0-500m-pytorch ~1GB в fp32, ~500MB в bfloat16
MODEL_ID = "google/timesfm-2.0-500m-pytorch"

print(f"Loading {MODEL_ID}...")
model = TimesFmModelForPrediction.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map=DEVICE,
)
model.eval()
print("Model loaded!")

# ============================================================
# DATA
# ============================================================
traindf = pd.read_parquet(cfg["trainpath"])
testdf  = pd.read_parquet(cfg["testpath"])
traindf["timestamp"] = pd.to_datetime(traindf["timestamp"])
testdf["timestamp"]  = pd.to_datetime(testdf["timestamp"])
traindf = traindf.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
testdf  = testdf.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
print(f"Train: {traindf.shape}, {traindf.timestamp.min()} → {traindf.timestamp.max()}")
print(f"Test:  {testdf.shape},  {testdf.timestamp.min()} → {testdf.timestamp.max()}")

# ============================================================
# METRIC
# ============================================================
def wape_rbias(ytrue: np.ndarray, ypred: np.ndarray):
    ytrue = np.asarray(ytrue, dtype=float)
    ypred = np.clip(np.asarray(ypred, dtype=float), 0, None)
    denom = ytrue.sum() + 1e-9
    wape  = np.abs(ypred - ytrue).sum() / denom
    rbias = abs(ypred.sum() / denom - 1)
    return wape + rbias, wape, rbias

# ============================================================
# INFERENCE HELPER
# ============================================================
@torch.no_grad()
def timesfm_forecast(history_values: np.ndarray, horizon: int) -> np.ndarray:
    """
    history_values: 1D float array (30-min granularity, последние CONTEXT_LEN точек)
    horizon: сколько точек предсказать
    Возвращает: 1D array длиной horizon (point forecast)
    """
    # TimesFM ожидает batch: List[List[float]]
    ctx = history_values[-CONTEXT_LEN:].tolist()

    inputs = {
        "past_values": torch.tensor([ctx], dtype=torch.bfloat16).to(DEVICE),
    }

    # frequency_input: 0=high (sub-daily), 1=daily, 2=weekly
    # 30-минутные данные → 0
    outputs = model(
        **inputs,
        prediction_length=horizon,
        frequency_input=torch.tensor([0]).to(DEVICE),
    )

    # outputs.mean_predictions shape: (batch, horizon)
    forecast = outputs.mean_predictions[0].float().cpu().numpy()
    return np.clip(forecast, 0, None)


# ============================================================
# LOCAL VALIDATION (last FORECAST_POINTS rows per route)
# ============================================================
print("\n=== Local Validation ===")
val_rows = []
route_ids = traindf["route_id"].unique()

for route_id in tqdm(route_ids, desc="Val TimesFM"):
    routedf = (
        traindf[traindf["route_id"] == route_id]
        .sort_values("timestamp")
        .reset_index(drop=True)
    )
    if len(routedf) < FORECAST_POINTS + 32:
        continue

    histpart = routedf.iloc[:-FORECAST_POINTS].copy()
    valpart  = routedf.tail(FORECAST_POINTS).copy()

    # Заполняем 30-мин ряд без пропусков
    hist_series = (
        histpart.set_index("timestamp")[TARGETCOL]
        .asfreq("30min")
        .interpolate(method="time")
        .bfill().ffill()
    )

    preds = timesfm_forecast(hist_series.values, horizon=FORECAST_POINTS)

    tmp = valpart[["route_id", "timestamp", TARGETCOL]].copy()
    tmp["ypred"] = preds[:len(tmp)]
    val_rows.append(tmp)

valpreddf = (
    pd.concat(val_rows, ignore_index=True)
    .sort_values(["route_id", "timestamp"])
    .reset_index(drop=True)
)

ytrue = valpreddf[TARGETCOL].values
ypred = valpreddf["ypred"].values
total, wape, rbias = wape_rbias(ytrue, ypred)
print(f"\nVal (raw):  WAPE={wape:.4f}  RBias={rbias:.4f}  Total={total:.4f}")

# Post-hoc calibration
calib_scale = float(ytrue.sum() / (ypred.sum() + 1e-9))
ypred_cal = np.clip(ypred * calib_scale, 0, None)
total_cal, wape_cal, rbias_cal = wape_rbias(ytrue, ypred_cal)
print(f"Val (calib scale={calib_scale:.4f}): WAPE={wape_cal:.4f}  RBias={rbias_cal:.4f}  Total={total_cal:.4f}")

valpreddf.to_csv("timesfm_local_validation.csv", index=False)
print("Saved timesfm_local_validation.csv")


# ============================================================
# FULL TEST PREDICTION
# ============================================================
print("\n=== Test Prediction ===")
predictions_raw = {}

for route_id in tqdm(sorted(testdf["route_id"].unique()), desc="Test TimesFM"):
    routetrain = (
        traindf[traindf["route_id"] == route_id]
        .sort_values("timestamp")
        .copy()
    )
    routetest = (
        testdf[testdf["route_id"] == route_id]
        .sort_values("timestamp")
        .copy()
    )

    # 30-мин ряд из train
    hist_series = (
        routetrain.set_index("timestamp")[TARGETCOL]
        .asfreq("30min")
        .interpolate(method="time")
        .bfill().ffill()
    )

    n_test = len(routetest)
    if len(hist_series) < 32 or n_test == 0:
        # Fallback: seasonal mean
        for _, row in routetest.iterrows():
            predictions_raw[row["id"]] = float(
                hist_series[
                    (hist_series.index.hour == row["timestamp"].hour) &
                    (hist_series.index.minute == row["timestamp"].minute)
                ].mean() if len(hist_series) > 0 else 0.0
            )
        continue

    preds = timesfm_forecast(hist_series.values, horizon=n_test)

    for i, (_, row) in enumerate(routetest.iterrows()):
        predictions_raw[row["id"]] = float(max(0.0, preds[i]))

# Apply calibration from validation
submission_raw = (
    pd.DataFrame(list(predictions_raw.items()), columns=["id", "ypred"])
    .sort_values("id")
    .reset_index(drop=True)
)

submission_cal = submission_raw.copy()
submission_cal["ypred"] = np.clip(submission_cal["ypred"] * calib_scale, 0, None)

submission_raw.to_csv("submission_timesfm_raw.csv", index=False)
submission_cal.to_csv("submission_timesfm_calibrated.csv", index=False)

print(f"\nSaved submission_timesfm_raw.csv & submission_timesfm_calibrated.csv")
print(submission_raw["ypred"].describe().round(2))

In [ ]:
import os, ssl
os.environ["HF_HUB_DISABLE_SSL_VERIFICATION"] = "1"
ssl._create_default_https_context = ssl._create_unverified_context

# Читай данные ДО импорта timesfm — это решает ArrowKeyError
import pandas as pd
import numpy as np
from tqdm import tqdm
# Только потом timesfm
import timesfm

In [ ]:
import os
import ssl

# Отключить проверку SSL для huggingface_hub
os.environ["HF_HUB_DISABLE_SSL_VERIFICATION"] = "1"

# Отключить для httpx (который использует huggingface_hub)
import httpx
original_init = httpx.Client.__init__

def patched_init(self, *args, **kwargs):
    kwargs["verify"] = False
    original_init(self, *args, **kwargs)

httpx.Client.__init__ = patched_init

# Глобально для ssl
ssl._create_default_https_context = ssl._create_unverified_context

In [ ]:
# ============================================================
# TimesFM 2.0 zero-shot — Solo track
# - context_len=2048 (максимальная история ~42 суток)
# - валидация raw vs calibrated
# - сохраняем оба сабмита
# ============================================================
import warnings
import numpy as np
import pandas as pd
import timesfm
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None


def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias


# -------------------- Config --------------------
TRAIN_PATH      = "train_solo_track.parquet"
TEST_PATH       = "test_solo_track.parquet"
TARGET_COL      = "target_1h"
FORECAST_STEPS  = 8
CONTEXT_LEN     = 2048   # ~42 суток истории, максимум модели
VAL_DAYS        = 14


# -------------------- Load model --------------------
print("Loading TimesFM 2.0 (context_len=2048)...")
tfm = timesfm.TimesFm(
    hparams=timesfm.TimesFmHparams(
        backend="cpu",
        per_core_batch_size=32,
        horizon_len=FORECAST_STEPS,
        input_patch_len=32,
        output_patch_len=128,
        num_layers=50,
        model_dims=1280,
        use_positional_embedding=False,
        context_len=2048,             # ← максимум
    ),
    checkpoint=timesfm.TimesFmCheckpoint(
        huggingface_repo_id="google/timesfm-2.0-500m-pytorch"
    ),
)
print("Model loaded!\n")


# -------------------- Data --------------------
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
print(f"Train: {train_df.shape}  {train_df.timestamp.min()} → {train_df.timestamp.max()}")
print(f"Test:  {test_df.shape}   {test_df.timestamp.min()} → {test_df.timestamp.max()}")
print(f"Routes in test: {test_df.route_id.nunique()}\n")


# -------------------- Helper --------------------
def get_context(route_df: pd.DataFrame) -> np.ndarray:
    """30-мин ряд без пропусков → последние CONTEXT_LEN точек."""
    series = (
        route_df.set_index("timestamp")[TARGET_COL]
        .asfreq("30min")
        .interpolate(method="time")
        .bfill().ffill()
    )
    return series.values[-CONTEXT_LEN:].astype(float)


# ============================================================
# LOCAL VALIDATION
# Прогоняем val через ту же логику что и test:
#   история до val_cutoff → предсказываем следующие 8 точек
# Дополнительно смотрим калибровку per-step
# ============================================================
print("=" * 60)
print("LOCAL VALIDATION")
print("=" * 60)

val_cutoff = train_df["timestamp"].max() - pd.Timedelta(days=VAL_DAYS)
print(f"Val cutoff: {val_cutoff}  (last {VAL_DAYS} days)\n")

histories_val   = []
y_trues_val     = []
route_ids_val   = []

for route_id, grp in tqdm(train_df.groupby("route_id"), desc="Collect val"):
    grp       = grp.sort_values("timestamp").reset_index(drop=True)
    hist_part = grp[grp["timestamp"] <= val_cutoff]
    val_part  = grp[grp["timestamp"] >  val_cutoff].head(FORECAST_STEPS)

    if len(hist_part) < 32 or len(val_part) == 0:
        continue

    histories_val.append(get_context(hist_part))
    y_trues_val.append(val_part[TARGET_COL].values)
    route_ids_val.append(route_id)

print(f"Routes in val: {len(histories_val)}")
print(f"Avg context length: {np.mean([len(h) for h in histories_val]):.0f} points\n")

print("Running val inference (batch)...")
val_preds, _ = tfm.forecast(
    inputs=histories_val,
    freq=[0] * len(histories_val),
)
val_preds = np.clip(val_preds, 0, None)   # shape: (N_routes, 8)

# ── Глобальные метрики raw ──
y_true_flat = np.concatenate([y_trues_val[i] for i in range(len(y_trues_val))])
y_pred_flat = np.concatenate([
    val_preds[i, :len(y_trues_val[i])] for i in range(len(y_trues_val))
])

total_raw, wape_raw, rbias_raw = wape_rbias(y_true_flat, y_pred_flat)
print(f"{'Val RAW':30s}  WAPE={wape_raw:.4f}  |RBias|={rbias_raw:.4f}  Total={total_raw:.4f}")

# ── Калибровочный коэффициент (глобальный) ──
calib_scale = float(y_true_flat.sum() / (y_pred_flat.sum() + 1e-9))
y_pred_cal  = np.clip(y_pred_flat * calib_scale, 0, None)

total_cal, wape_cal, rbias_cal = wape_rbias(y_true_flat, y_pred_cal)
print(f"{'Val CALIBRATED (global)':30s}  WAPE={wape_cal:.4f}  |RBias|={rbias_cal:.4f}  Total={total_cal:.4f}")
print(f"  calib_scale = {calib_scale:.4f}  ({'under' if calib_scale > 1 else 'over'}-predicted raw)\n")

# ── Метрики по шагам h=1..8 (видно, помогает ли калибровка на каждом) ──
print(f"{'Step':>5} | {'WAPE raw':>10} {'RBias raw':>10} {'Total raw':>10} │ {'WAPE cal':>10} {'RBias cal':>10} {'Total cal':>10}")
print("-" * 75)

step_rows = []
for h in range(1, FORECAST_STEPS + 1):
    yt_h = np.array([
        y_trues_val[i][h - 1]
        for i in range(len(y_trues_val))
        if len(y_trues_val[i]) >= h
    ])
    yp_h_raw = np.array([
        val_preds[i, h - 1]
        for i in range(len(y_trues_val))
        if len(y_trues_val[i]) >= h
    ])
    yp_h_cal = np.clip(yp_h_raw * calib_scale, 0, None)

    tot_r, w_r, rb_r = wape_rbias(yt_h, yp_h_raw)
    tot_c, w_c, rb_c = wape_rbias(yt_h, yp_h_cal)

    print(f"  h={h:>2} | {w_r:>10.4f} {rb_r:>10.4f} {tot_r:>10.4f} │ {w_c:>10.4f} {rb_c:>10.4f} {tot_c:>10.4f}")
    step_rows.append({
        "h": h,
        "wape_raw": w_r, "rbias_raw": rb_r, "total_raw": tot_r,
        "wape_cal": w_c, "rbias_cal": rb_c, "total_cal": tot_c,
    })

step_df = pd.DataFrame(step_rows)
step_df.to_csv("timesfm_val_per_step.csv", index=False)
print(f"\nSaved: timesfm_val_per_step.csv")
print(f"\nCalibration {'HELPS ✅' if total_cal < total_raw else 'HURTS ❌'} "
      f"(Δ = {total_cal - total_raw:+.4f})")


# ============================================================
# TEST PREDICTION
# ============================================================
print("\n" + "=" * 60)
print("TEST PREDICTION")
print("=" * 60)

histories_test  = []
test_route_ids  = []
test_n_pts      = []

for route_id, grp in tqdm(test_df.groupby("route_id"), desc="Collect test"):
    route_train = train_df[train_df["route_id"] == route_id].sort_values("timestamp")
    route_test  = grp.sort_values("timestamp")
    if len(route_train) < 32:
        continue
    histories_test.append(get_context(route_train))
    test_route_ids.append(route_id)
    test_n_pts.append(len(route_test))

print(f"Routes in test: {len(test_route_ids)}")
print("Running test inference (batch)...")

test_preds, _ = tfm.forecast(
    inputs=histories_test,
    freq=[0] * len(test_route_ids),
)
test_preds = np.clip(test_preds, 0, None)   # shape: (N_routes, 8)

# ── Собираем submission ──
predictions_raw = {}
for i, route_id in enumerate(test_route_ids):
    route_test = test_df[test_df["route_id"] == route_id].sort_values("timestamp")
    for j, (_, row) in enumerate(route_test.iterrows()):
        pred = float(test_preds[i, j]) if j < test_preds.shape[1] else float(test_preds[i, -1])
        predictions_raw[row["id"]] = max(0.0, pred)

submission_raw = (
    pd.DataFrame(list(predictions_raw.items()), columns=["id", "y_pred"])
    .sort_values("id").reset_index(drop=True)
)
submission_cal = submission_raw.copy()
submission_cal["y_pred"] = np.clip(submission_cal["y_pred"] * calib_scale, 0, None)

submission_raw.to_csv("submission_timesfm_raw.csv", index=False)
submission_cal.to_csv("submission_timesfm_calibrated.csv", index=False)

print(f"\n✅ Saved: submission_timesfm_raw.csv")
print(f"✅ Saved: submission_timesfm_calibrated.csv")
print(f"\n── Raw stats ──\n{submission_raw['y_pred'].describe().round(2)}")
print(f"\n── Calibrated stats (scale={calib_scale:.4f}) ──\n{submission_cal['y_pred'].describe().round(2)}")

In [ ]:
# ============================================================
# Chronos-2 zero-shot — Solo track
# ============================================================
import warnings
import numpy as np
import pandas as pd
import torch
from chronos import Chronos2Pipeline
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None


def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias


# -------------------- Config --------------------
TRAIN_PATH      = "train_solo_track.parquet"
TEST_PATH       = "test_solo_track.parquet"
TARGET_COL      = "target_1h"
FORECAST_STEPS  = 8
CONTEXT_LEN     = 2048
VAL_DAYS        = 14

# -------------------- Device --------------------
if torch.backends.mps.is_available():
    DEVICE = "mps"
    print("Apple Silicon MPS ✅")
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(f"Device: {DEVICE}")

# -------------------- Load model --------------------
MODEL_ID = "amazon/chronos-2"
print(f"Loading {MODEL_ID}...")
pipeline = Chronos2Pipeline.from_pretrained(
    MODEL_ID,
    device_map=DEVICE,
    torch_dtype=torch.bfloat16,
)
print("Model loaded!\n")

# -------------------- Data --------------------
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

status_cols = sorted([c for c in train_df.columns if c.startswith("status_")])
print(f"Train: {train_df.shape}  Test: {test_df.shape}")
print(f"Status cols: {status_cols}\n")


# -------------------- Helper --------------------
def prepare_context_df(route_df: pd.DataFrame, route_id) -> pd.DataFrame:
    df = (
        route_df.set_index("timestamp")[[TARGET_COL] + status_cols]
        .asfreq("30min")
        .interpolate(method="time")
        .bfill().ffill()
        .tail(CONTEXT_LEN)
        .reset_index()
    )
    df["id"] = route_id
    df = df.rename(columns={TARGET_COL: "target"})
    return df


def run_inference(context_full: pd.DataFrame) -> pd.DataFrame:
    return pipeline.predict_df(
        context_full,
        prediction_length=FORECAST_STEPS,
        quantile_levels=[0.1, 0.5, 0.9],
        id_column="id",
        timestamp_column="timestamp",
        target="target",
    )


def get_pred_col(pred_df: pd.DataFrame) -> str:
    """Определяем колонку с point forecast — может быть 'mean' или '0.5'."""
    print(f"  pred_df columns: {list(pred_df.columns)}")
    for candidate in ["mean", "0.5", "q0.5", "median"]:
        if candidate in pred_df.columns:
            return candidate
    # если ничего из известных — берём первую числовую колонку кроме id/timestamp
    num_cols = [c for c in pred_df.columns
                if c not in ("id", "timestamp") and pd.api.types.is_numeric_dtype(pred_df[c])]
    print(f"  Using fallback col: {num_cols[0]}")
    return num_cols[0]


def build_preds_dict(pred_df: pd.DataFrame, pred_col: str) -> dict:
    result = {}
    for route_id, grp in pred_df.groupby("id"):
        result[route_id] = np.clip(
            grp.sort_values("timestamp")[pred_col].values, 0, None
        )
    return result


# ============================================================
# LOCAL VALIDATION
# ============================================================
print("=" * 60)
print("LOCAL VALIDATION")
print("=" * 60)

val_cutoff = train_df["timestamp"].max() - pd.Timedelta(days=VAL_DAYS)
print(f"Val cutoff: {val_cutoff}\n")

context_dfs   = []
y_trues_val   = []
route_ids_val = []

for route_id, grp in tqdm(train_df.groupby("route_id"), desc="Collect val"):
    grp       = grp.sort_values("timestamp").reset_index(drop=True)
    hist_part = grp[grp["timestamp"] <= val_cutoff]
    val_part  = grp[grp["timestamp"] >  val_cutoff].head(FORECAST_STEPS)
    if len(hist_part) < 32 or len(val_part) == 0:
        continue
    context_dfs.append(prepare_context_df(hist_part, route_id))
    y_trues_val.append(val_part[TARGET_COL].values)
    route_ids_val.append(route_id)

context_full = pd.concat(context_dfs, ignore_index=True)
print(f"Routes in val: {len(route_ids_val)}")
print("Running val inference...")

pred_df  = run_inference(context_full)
pred_col = get_pred_col(pred_df)           # ← определяем реальное имя колонки
print(f"  Using point forecast column: '{pred_col}'")

val_preds = build_preds_dict(pred_df, pred_col)

y_true_flat = np.concatenate(y_trues_val)
y_pred_flat = np.concatenate([
    val_preds.get(rid, np.zeros(len(yt)))[:len(yt)]
    for rid, yt in zip(route_ids_val, y_trues_val)
])

total_raw, wape_raw, rbias_raw = wape_rbias(y_true_flat, y_pred_flat)
print(f"\n{'Val RAW':30s}  WAPE={wape_raw:.4f}  |RBias|={rbias_raw:.4f}  Total={total_raw:.4f}")

calib_scale = float(y_true_flat.sum() / (y_pred_flat.sum() + 1e-9))
y_pred_cal  = np.clip(y_pred_flat * calib_scale, 0, None)
total_cal, wape_cal, rbias_cal = wape_rbias(y_true_flat, y_pred_cal)
print(f"{'Val CALIBRATED':30s}  WAPE={wape_cal:.4f}  |RBias|={rbias_cal:.4f}  Total={total_cal:.4f}")
print(f"  calib_scale = {calib_scale:.4f}")

# Per-step метрики
print(f"\n{'Step':>5} | {'WAPE raw':>10} {'RBias raw':>10} {'Total raw':>10} │ {'WAPE cal':>10} {'RBias cal':>10} {'Total cal':>10}")
print("-" * 75)
step_rows = []
for h in range(1, FORECAST_STEPS + 1):
    idxs  = [i for i in range(len(y_trues_val)) if len(y_trues_val[i]) >= h]
    yt_h  = np.array([y_trues_val[i][h-1] for i in idxs])
    yp_h  = np.array([val_preds.get(route_ids_val[i], np.zeros(FORECAST_STEPS))[h-1] for i in idxs])
    yp_hc = np.clip(yp_h * calib_scale, 0, None)
    tot_r, w_r, rb_r = wape_rbias(yt_h, yp_h)
    tot_c, w_c, rb_c = wape_rbias(yt_h, yp_hc)
    print(f"  h={h:>2} | {w_r:>10.4f} {rb_r:>10.4f} {tot_r:>10.4f} │ {w_c:>10.4f} {rb_c:>10.4f} {tot_c:>10.4f}")
    step_rows.append({"h": h,
                      "wape_raw": w_r, "rbias_raw": rb_r, "total_raw": tot_r,
                      "wape_cal": w_c, "rbias_cal": rb_c, "total_cal": tot_c})

pd.DataFrame(step_rows).to_csv("chronos2_val_per_step.csv", index=False)
print(f"\nCalibration {'HELPS ✅' if total_cal < total_raw else 'HURTS ❌'} "
      f"(Δ = {total_cal - total_raw:+.4f})")


# ============================================================
# TEST PREDICTION
# ============================================================
print("\n" + "=" * 60)
print("TEST PREDICTION")
print("=" * 60)

test_context_dfs = []
test_route_ids   = []

for route_id, grp in tqdm(test_df.groupby("route_id"), desc="Collect test"):
    route_train = train_df[train_df["route_id"] == route_id].sort_values("timestamp")
    if len(route_train) < 32:
        continue
    test_context_dfs.append(prepare_context_df(route_train, route_id))
    test_route_ids.append(route_id)

test_context_full = pd.concat(test_context_dfs, ignore_index=True)
print(f"Routes in test: {len(test_route_ids)}")
print("Running test inference...")

test_pred_df = run_inference(test_context_full)
test_preds   = build_preds_dict(test_pred_df, pred_col)

predictions_raw = {}
for route_id in test_route_ids:
    route_test = test_df[test_df["route_id"] == route_id].sort_values("timestamp")
    preds = test_preds.get(route_id, np.zeros(FORECAST_STEPS))
    for j, (_, row) in enumerate(route_test.iterrows()):
        pred = float(preds[j]) if j < len(preds) else float(preds[-1])
        predictions_raw[row["id"]] = max(0.0, pred)

submission_raw = (
    pd.DataFrame(list(predictions_raw.items()), columns=["id", "y_pred"])
    .sort_values("id").reset_index(drop=True)
)
submission_cal = submission_raw.copy()
submission_cal["y_pred"] = np.clip(submission_cal["y_pred"] * calib_scale, 0, None)

submission_raw.to_csv("submission_chronos2_raw.csv", index=False)
submission_cal.to_csv("submission_chronos2_calibrated.csv", index=False)

print(f"\n✅ submission_chronos2_raw.csv")
print(f"✅ submission_chronos2_calibrated.csv")
print(f"\n── Raw ──\n{submission_raw['y_pred'].describe().round(2)}")
print(f"\n── Calibrated (scale={calib_scale:.4f}) ──\n{submission_cal['y_pred'].describe().round(2)}")

In [ ]:
# ============================================================
# Chronos-2 — Rolling Cross-Validation (5 folds × 8 steps)
# Последние 40 точек трейна разбиваем на 5 окон по 8
# ============================================================
import warnings
import numpy as np
import pandas as pd
import torch

from tqdm import tqdm
import os
warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None


def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias


# -------------------- Config --------------------
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8
N_FOLDS        = 5                        # 5 × 8 = 40 последних точек
TOTAL_VAL_PTS  = N_FOLDS * FORECAST_STEPS  # 40
CONTEXT_LEN    = 8192

# -------------------- Device --------------------
if torch.backends.mps.is_available():
    DEVICE = "mps"
    print("Apple Silicon MPS ✅")
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(f"Device: {DEVICE}\n")
os.environ["TRANSFORMERS_OFFLINE"] = "1"   # трансформеры не лезут в сеть
os.environ["HF_DATASETS_OFFLINE"] = "1"    # датасеты тоже
os.environ["HF_HUB_OFFLINE"] = "1"         # huggingface hub тоже
# -------------------- Load model --------------------
from chronos import Chronos2Pipeline
MODEL_ID = "amazon/chronos-2"
print(f"Loading {MODEL_ID}...")
pipeline = Chronos2Pipeline.from_pretrained(
    MODEL_ID,
    device_map=DEVICE,
    torch_dtype=torch.bfloat16,
    local_files_only=True,
)
print("Model loaded!\n")

# -------------------- Data --------------------
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

status_cols = sorted([c for c in train_df.columns if c.startswith("status_")])
print(f"Train: {train_df.shape}  Test: {test_df.shape}")
print(f"Status cols: {status_cols}\n")


# -------------------- Helpers --------------------
def prepare_context_df(route_df: pd.DataFrame, route_id) -> pd.DataFrame:
    df = (
        route_df.set_index("timestamp")[[TARGET_COL] + status_cols]
        .asfreq("30min")
        .interpolate(method="time")
        .bfill().ffill()
        .tail(CONTEXT_LEN)
        .reset_index()
    )
    df["id"] = route_id
    df = df.rename(columns={TARGET_COL: "target"})
    return df


def run_inference(context_full: pd.DataFrame, cross_learning: bool = True) -> pd.DataFrame:
    return pipeline.predict_df(
        context_full,
        prediction_length=FORECAST_STEPS,
        quantile_levels=[0.1, 0.5, 0.9],
        id_column="id",
        timestamp_column="timestamp",
        target="target",
        cross_learning=cross_learning,
    )


def get_pred_col(pred_df: pd.DataFrame) -> str:
    for candidate in ["mean", "0.5", "q0.5", "median"]:
        if candidate in pred_df.columns:
            return candidate
    num_cols = [c for c in pred_df.columns
                if c not in ("id", "timestamp") and pd.api.types.is_numeric_dtype(pred_df[c])]
    return num_cols[0]


def build_preds_dict(pred_df: pd.DataFrame, pred_col: str) -> dict:
    result = {}
    for route_id, grp in pred_df.groupby("id"):
        result[route_id] = np.clip(grp.sort_values("timestamp")[pred_col].values, 0, None)
    return result


# ============================================================
# ROLLING CROSS-VALIDATION
#
# Схема (индексы от конца ряда):
#   fold 0: история=[:-40]       val=[-40:-32]  (самый старый)
#   fold 1: история=[:-32]       val=[-32:-24]
#   fold 2: история=[:-24]       val=[-24:-16]
#   fold 3: история=[:-16]       val=[-16: -8]
#   fold 4: история=[: -8]       val=[ -8:   ]  (самый свежий)
# ============================================================
print("=" * 65)
print(f"ROLLING CV  ({N_FOLDS} folds × {FORECAST_STEPS} steps = {TOTAL_VAL_PTS} points)")
print("=" * 65)

# Результаты по каждому фолду
fold_results  = []    # [{fold, wape_raw, rbias_raw, ..., wape_cal, ...}]
all_yt, all_yp_raw = [], []   # для глобальной метрики

# Калибровка считается из fold 0..3, применяется на fold 4
# (или считаем глобальную по всем фолдам — выбор за тобой)
# Здесь считаем глобальную по всем фолдам вместе

for fold in range(N_FOLDS):
    # Граница: val начинается с позиции -(TOTAL_VAL_PTS - fold*FORECAST_STEPS)
    val_offset_end   = TOTAL_VAL_PTS - fold * FORECAST_STEPS        # 40, 32, 24, 16, 8
    val_offset_start = val_offset_end - FORECAST_STEPS               # 32, 24, 16,  8, 0

    context_dfs   = []
    y_trues_fold  = []
    route_ids_fold = []

    for route_id, grp in train_df.groupby("route_id"):
        grp = grp.sort_values("timestamp").reset_index(drop=True)

        # нужно минимум TOTAL_VAL_PTS + 32 точек чтобы у нас была история
        if len(grp) < TOTAL_VAL_PTS + 32:
            continue

        if val_offset_start == 0:
            hist_part = grp.iloc[:-val_offset_end]          # fold 4: всё кроме последних 8
            val_part  = grp.iloc[-val_offset_end:]
        else:
            hist_part = grp.iloc[:-val_offset_end]
            val_part  = grp.iloc[-val_offset_end:-val_offset_start]

        if len(hist_part) < 32 or len(val_part) == 0:
            continue

        context_dfs.append(prepare_context_df(hist_part, route_id))
        y_trues_fold.append(val_part[TARGET_COL].values[:FORECAST_STEPS])
        route_ids_fold.append(route_id)

    context_full = pd.concat(context_dfs, ignore_index=True)

    print(f"\nFold {fold+1}/{N_FOLDS}  "
          f"(val offset: -{val_offset_end}..{'-'+str(val_offset_start) if val_offset_start else 'end'})  "
          f"routes={len(route_ids_fold)}")

    pred_df  = run_inference(context_full)
    pred_col = get_pred_col(pred_df)
    preds    = build_preds_dict(pred_df, pred_col)

    yt_fold = np.concatenate(y_trues_fold)
    yp_fold = np.concatenate([
        preds.get(rid, np.zeros(len(yt)))[:len(yt)]
        for rid, yt in zip(route_ids_fold, y_trues_fold)
    ])

    tot, wape, rb = wape_rbias(yt_fold, yp_fold)
    print(f"  RAW   WAPE={wape:.4f}  |RBias|={rb:.4f}  Total={tot:.4f}")

    fold_results.append({
        "fold": fold + 1,
        "val_start_offset": f"-{val_offset_end}",
        "val_end_offset":   f"-{val_offset_start}" if val_offset_start else "end",
        "n_routes": len(route_ids_fold),
        "wape_raw": wape, "rbias_raw": rb, "total_raw": tot,
    })

    all_yt.append(yt_fold)
    all_yp_raw.append(yp_fold)

# ── Глобальные метрики по всем фолдам ──
y_true_all = np.concatenate(all_yt)
y_pred_all = np.concatenate(all_yp_raw)

total_raw, wape_raw, rbias_raw = wape_rbias(y_true_all, y_pred_all)

calib_scale = float(y_true_all.sum() / (y_pred_all.sum() + 1e-9))
y_pred_cal  = np.clip(y_pred_all * calib_scale, 0, None)
total_cal, wape_cal, rbias_cal = wape_rbias(y_true_all, y_pred_cal)

print("\n" + "=" * 65)
print("SUMMARY ACROSS ALL FOLDS")
print("=" * 65)
cv_df = pd.DataFrame(fold_results)
print(cv_df[["fold", "val_start_offset", "val_end_offset",
             "wape_raw", "rbias_raw", "total_raw"]].to_string(index=False))
print(f"\n{'Mean raw':30s}  WAPE={cv_df['wape_raw'].mean():.4f}  "
      f"|RBias|={cv_df['rbias_raw'].mean():.4f}  Total={cv_df['total_raw'].mean():.4f}")
print(f"\n{'Global RAW (pooled)':30s}  WAPE={wape_raw:.4f}  |RBias|={rbias_raw:.4f}  Total={total_raw:.4f}")
print(f"{'Global CALIBRATED':30s}  WAPE={wape_cal:.4f}  |RBias|={rbias_cal:.4f}  Total={total_cal:.4f}")
print(f"  calib_scale = {calib_scale:.4f}")
print(f"\nCalibration {'HELPS ✅' if total_cal < total_raw else 'HURTS ❌'} "
      f"(Δ = {total_cal - total_raw:+.4f})")

cv_df.to_csv("chronos2_cv_results.csv", index=False)
print("\nSaved: chronos2_cv_results.csv")


# ============================================================
# TEST PREDICTION (с глобальным calib_scale из CV)
# ============================================================
print("\n" + "=" * 65)
print("TEST PREDICTION")
print("=" * 65)

test_context_dfs = []
test_route_ids   = []

for route_id, grp in tqdm(test_df.groupby("route_id"), desc="Collect test"):
    route_train = train_df[train_df["route_id"] == route_id].sort_values("timestamp")
    if len(route_train) < 32:
        continue
    test_context_dfs.append(prepare_context_df(route_train, route_id))
    test_route_ids.append(route_id)

test_context_full = pd.concat(test_context_dfs, ignore_index=True)
print(f"Routes: {len(test_route_ids)}  Running inference...")

test_pred_df = run_inference(test_context_full)
test_preds   = build_preds_dict(test_pred_df, pred_col)

predictions_raw = {}
for route_id in test_route_ids:
    route_test = test_df[test_df["route_id"] == route_id].sort_values("timestamp")
    preds = test_preds.get(route_id, np.zeros(FORECAST_STEPS))
    for j, (_, row) in enumerate(route_test.iterrows()):
        pred = float(preds[j]) if j < len(preds) else float(preds[-1])
        predictions_raw[row["id"]] = max(0.0, pred)

submission_raw = (
    pd.DataFrame(list(predictions_raw.items()), columns=["id", "y_pred"])
    .sort_values("id").reset_index(drop=True)
)
submission_cal = submission_raw.copy()
submission_cal["y_pred"] = np.clip(submission_cal["y_pred"] * calib_scale, 0, None)

submission_raw.to_csv("submission_chronos2_raw_cl_8192.csv", index=False)
submission_cal.to_csv("submission_chronos2_calibrated_cl_8192.csv", index=False)
print(f"\n✅ submission_chronos2_raw.csv")
print(f"✅ submission_chronos2_calibrated.csv")

In [ ]:
# ============================================================
# Chronos-2 — Rolling Cross-Validation (5 folds × 8 steps)
# Последние 40 точек трейна разбиваем на 5 окон по 8
# ============================================================
import warnings
import numpy as np
import pandas as pd
import torch
from chronos import Chronos2Pipeline
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None


def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias


# -------------------- Config --------------------
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8
N_FOLDS        = 5                        # 5 × 8 = 40 последних точек
TOTAL_VAL_PTS  = N_FOLDS * FORECAST_STEPS  # 40
CONTEXT_LEN    = 2048

# -------------------- Device --------------------
if torch.backends.mps.is_available():
    DEVICE = "mps"
    print("Apple Silicon MPS ✅")
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(f"Device: {DEVICE}\n")

# -------------------- Load model --------------------
MODEL_ID = "amazon/chronos-2"
print(f"Loading {MODEL_ID}...")
pipeline = Chronos2Pipeline.from_pretrained(
    MODEL_ID,
    device_map=DEVICE,
    torch_dtype=torch.bfloat16,
)
print("Model loaded!\n")

# -------------------- Data --------------------
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

status_cols = sorted([c for c in train_df.columns if c.startswith("status_")])
print(f"Train: {train_df.shape}  Test: {test_df.shape}")
print(f"Status cols: {status_cols}\n")


# -------------------- Helpers --------------------
def prepare_context_df(route_df: pd.DataFrame, route_id) -> pd.DataFrame:
    df = (
        route_df.set_index("timestamp")[[TARGET_COL] + status_cols]
        .asfreq("30min")
        .interpolate(method="time")
        .bfill().ffill()
        .tail(CONTEXT_LEN)
        .reset_index()
    )
    df["id"] = route_id
    df = df.rename(columns={TARGET_COL: "target"})
    return df


def run_inference(context_full: pd.DataFrame, cross_learning: bool = True) -> pd.DataFrame:
    return pipeline.predict_df(
        context_full,
        prediction_length=FORECAST_STEPS,
        quantile_levels=[0.1, 0.5, 0.9],
        id_column="id",
        timestamp_column="timestamp",
        target="target",
        cross_learning=cross_learning,
    )


def get_pred_col(pred_df: pd.DataFrame) -> str:
    for candidate in ["mean", "0.5", "q0.5", "median"]:
        if candidate in pred_df.columns:
            return candidate
    num_cols = [c for c in pred_df.columns
                if c not in ("id", "timestamp") and pd.api.types.is_numeric_dtype(pred_df[c])]
    return num_cols[0]


def build_preds_dict(pred_df: pd.DataFrame, pred_col: str) -> dict:
    result = {}
    for route_id, grp in pred_df.groupby("id"):
        result[route_id] = np.clip(grp.sort_values("timestamp")[pred_col].values, 0, None)
    return result


# ============================================================
# ROLLING CROSS-VALIDATION
#
# Схема (индексы от конца ряда):
#   fold 0: история=[:-40]       val=[-40:-32]  (самый старый)
#   fold 1: история=[:-32]       val=[-32:-24]
#   fold 2: история=[:-24]       val=[-24:-16]
#   fold 3: история=[:-16]       val=[-16: -8]
#   fold 4: история=[: -8]       val=[ -8:   ]  (самый свежий)
# ============================================================
print("=" * 65)
print(f"ROLLING CV  ({N_FOLDS} folds × {FORECAST_STEPS} steps = {TOTAL_VAL_PTS} points)")
print("=" * 65)

# Результаты по каждому фолду
fold_results  = []    # [{fold, wape_raw, rbias_raw, ..., wape_cal, ...}]
all_yt, all_yp_raw = [], []   # для глобальной метрики

# Калибровка считается из fold 0..3, применяется на fold 4
# (или считаем глобальную по всем фолдам — выбор за тобой)
# Здесь считаем глобальную по всем фолдам вместе

for fold in range(N_FOLDS):
    # Граница: val начинается с позиции -(TOTAL_VAL_PTS - fold*FORECAST_STEPS)
    val_offset_end   = TOTAL_VAL_PTS - fold * FORECAST_STEPS        # 40, 32, 24, 16, 8
    val_offset_start = val_offset_end - FORECAST_STEPS               # 32, 24, 16,  8, 0

    context_dfs   = []
    y_trues_fold  = []
    route_ids_fold = []

    for route_id, grp in train_df.groupby("route_id"):
        grp = grp.sort_values("timestamp").reset_index(drop=True)

        # нужно минимум TOTAL_VAL_PTS + 32 точек чтобы у нас была история
        if len(grp) < TOTAL_VAL_PTS + 32:
            continue

        if val_offset_start == 0:
            hist_part = grp.iloc[:-val_offset_end]          # fold 4: всё кроме последних 8
            val_part  = grp.iloc[-val_offset_end:]
        else:
            hist_part = grp.iloc[:-val_offset_end]
            val_part  = grp.iloc[-val_offset_end:-val_offset_start]

        if len(hist_part) < 32 or len(val_part) == 0:
            continue

        context_dfs.append(prepare_context_df(hist_part, route_id))
        y_trues_fold.append(val_part[TARGET_COL].values[:FORECAST_STEPS])
        route_ids_fold.append(route_id)

    context_full = pd.concat(context_dfs, ignore_index=True)

    print(f"\nFold {fold+1}/{N_FOLDS}  "
          f"(val offset: -{val_offset_end}..{'-'+str(val_offset_start) if val_offset_start else 'end'})  "
          f"routes={len(route_ids_fold)}")

    pred_df  = run_inference(context_full)
    pred_col = get_pred_col(pred_df)
    preds    = build_preds_dict(pred_df, pred_col)

    yt_fold = np.concatenate(y_trues_fold)
    yp_fold = np.concatenate([
        preds.get(rid, np.zeros(len(yt)))[:len(yt)]
        for rid, yt in zip(route_ids_fold, y_trues_fold)
    ])

    tot, wape, rb = wape_rbias(yt_fold, yp_fold)
    print(f"  RAW   WAPE={wape:.4f}  |RBias|={rb:.4f}  Total={tot:.4f}")

    fold_results.append({
        "fold": fold + 1,
        "val_start_offset": f"-{val_offset_end}",
        "val_end_offset":   f"-{val_offset_start}" if val_offset_start else "end",
        "n_routes": len(route_ids_fold),
        "wape_raw": wape, "rbias_raw": rb, "total_raw": tot,
    })

    all_yt.append(yt_fold)
    all_yp_raw.append(yp_fold)

# ── Глобальные метрики по всем фолдам ──
y_true_all = np.concatenate(all_yt)
y_pred_all = np.concatenate(all_yp_raw)

total_raw, wape_raw, rbias_raw = wape_rbias(y_true_all, y_pred_all)

calib_scale = float(y_true_all.sum() / (y_pred_all.sum() + 1e-9))
y_pred_cal  = np.clip(y_pred_all * calib_scale, 0, None)
total_cal, wape_cal, rbias_cal = wape_rbias(y_true_all, y_pred_cal)

print("\n" + "=" * 65)
print("SUMMARY ACROSS ALL FOLDS")
print("=" * 65)
cv_df = pd.DataFrame(fold_results)
print(cv_df[["fold", "val_start_offset", "val_end_offset",
             "wape_raw", "rbias_raw", "total_raw"]].to_string(index=False))
print(f"\n{'Mean raw':30s}  WAPE={cv_df['wape_raw'].mean():.4f}  "
      f"|RBias|={cv_df['rbias_raw'].mean():.4f}  Total={cv_df['total_raw'].mean():.4f}")
print(f"\n{'Global RAW (pooled)':30s}  WAPE={wape_raw:.4f}  |RBias|={rbias_raw:.4f}  Total={total_raw:.4f}")
print(f"{'Global CALIBRATED':30s}  WAPE={wape_cal:.4f}  |RBias|={rbias_cal:.4f}  Total={total_cal:.4f}")
print(f"  calib_scale = {calib_scale:.4f}")
print(f"\nCalibration {'HELPS ✅' if total_cal < total_raw else 'HURTS ❌'} "
      f"(Δ = {total_cal - total_raw:+.4f})")

cv_df.to_csv("chronos2_cv_results.csv", index=False)
print("\nSaved: chronos2_cv_results.csv")


# ============================================================
# TEST PREDICTION (с глобальным calib_scale из CV)
# ============================================================
print("\n" + "=" * 65)
print("TEST PREDICTION")
print("=" * 65)

test_context_dfs = []
test_route_ids   = []

for route_id, grp in tqdm(test_df.groupby("route_id"), desc="Collect test"):
    route_train = train_df[train_df["route_id"] == route_id].sort_values("timestamp")
    if len(route_train) < 32:
        continue
    test_context_dfs.append(prepare_context_df(route_train, route_id))
    test_route_ids.append(route_id)

test_context_full = pd.concat(test_context_dfs, ignore_index=True)
print(f"Routes: {len(test_route_ids)}  Running inference...")

test_pred_df = run_inference(test_context_full)
test_preds   = build_preds_dict(test_pred_df, pred_col)

predictions_raw = {}
for route_id in test_route_ids:
    route_test = test_df[test_df["route_id"] == route_id].sort_values("timestamp")
    preds = test_preds.get(route_id, np.zeros(FORECAST_STEPS))
    for j, (_, row) in enumerate(route_test.iterrows()):
        pred = float(preds[j]) if j < len(preds) else float(preds[-1])
        predictions_raw[row["id"]] = max(0.0, pred)

submission_raw = (
    pd.DataFrame(list(predictions_raw.items()), columns=["id", "y_pred"])
    .sort_values("id").reset_index(drop=True)
)
submission_cal = submission_raw.copy()
submission_cal["y_pred"] = np.clip(submission_cal["y_pred"] * calib_scale, 0, None)

submission_raw.to_csv("submission_chronos2_raw.csv", index=False)
submission_cal.to_csv("submission_chronos2_calibrated.csv", index=False)
print(f"\n✅ submission_chronos2_raw.csv")
print(f"✅ submission_chronos2_calibrated.csv")

In [ ]:
# ============================================================
# LightGBM Direct Multi-Step — solo track
# Rolling CV: 5 фолдов × 8 шагов, без лика
# Обучение на [:-40pts], val на последних 40 точках
# ============================================================
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None


def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias


# -------------------- Config --------------------
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8
N_FOLDS        = 5
TOTAL_VAL_PTS  = N_FOLDS * FORECAST_STEPS   # 40 точек = 20 часов

ALL_LAGS     = [1, 2, 3, 4, 6, 8, 12, 16, 24, 48, 96, 168, 336]
ROLLING_WINS = [4, 8, 24, 48, 96]

LGB_PARAMS = {
    "objective":         "regression_l1",
    "metric":            "mae",
    "boosting_type":     "gbdt",
    "n_estimators":      800,
    "learning_rate":     0.03,
    "num_leaves":        127,
    "min_child_samples": 20,
    "subsample":         0.8,
    "subsample_freq":    1,
    "colsample_bytree":  0.7,
    "reg_alpha":         0.05,
    "reg_lambda":        1.0,
    "n_jobs":            -1,
    "random_state":      42,
    "verbose":           -1,
}
EARLY_STOPPING = 100

# -------------------- Load --------------------
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
print(f"Train: {train_df.shape}  Test: {test_df.shape}")

# -------------------- Feature engineering --------------------
def time_features(df):
    ts = df["timestamp"]
    df = df.copy()
    df["hh_idx"]     = (ts.dt.hour * 2 + ts.dt.minute // 30).astype(np.int8)
    df["hour"]       = ts.dt.hour.astype(np.int8)
    df["dow"]        = ts.dt.dayofweek.astype(np.int8)
    df["month"]      = ts.dt.month.astype(np.int8)
    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(np.int8)
    df["hh_sin"]     = np.sin(2 * np.pi * df["hh_idx"] / 48)
    df["hh_cos"]     = np.cos(2 * np.pi * df["hh_idx"] / 48)
    df["dow_sin"]    = np.sin(2 * np.pi * df["dow"] / 7)
    df["dow_cos"]    = np.cos(2 * np.pi * df["dow"] / 7)
    return df


def lag_rolling_features(grp):
    grp = grp.copy()
    tgt = grp[TARGET_COL]
    for lag in ALL_LAGS:
        grp[f"lag_{lag}"] = tgt.shift(lag)
    s = tgt.shift(1)
    for w in ROLLING_WINS:
        mp = max(1, w // 4)
        grp[f"rmean_{w}"] = s.rolling(w, min_periods=mp).mean()
        grp[f"rstd_{w}"]  = s.rolling(w, min_periods=mp).std()
    grp["rmax_48"]     = s.rolling(48, min_periods=24).max()
    grp["rmin_48"]     = s.rolling(48, min_periods=24).min()
    grp["trend_4v8"]   = grp["rmean_4"]  - grp["rmean_8"]
    grp["trend_8v48"]  = grp["rmean_8"]  - grp["rmean_48"]
    grp["trend_48v96"] = grp["rmean_48"] - grp["rmean_96"]
    return grp


print("Building features on train...")
train_feat = time_features(train_df)
train_feat = train_feat.groupby("route_id", group_keys=False).apply(lag_rolling_features)

print("Building features on test...")
full_df = pd.concat([
    train_df.assign(is_test=False),
    test_df.assign(**{TARGET_COL: np.nan, "is_test": True}),
], ignore_index=True).sort_values(["route_id", "timestamp"]).reset_index(drop=True)
full_feat = time_features(full_df)
full_feat = full_feat.groupby("route_id", group_keys=False).apply(lag_rolling_features)
test_feat = full_feat[full_feat["is_test"]].copy().reset_index(drop=True)
test_feat["step_in_route"] = test_feat.groupby("route_id").cumcount() + 1

CALENDAR_FEATS = ["hh_idx", "hour", "dow", "month", "is_weekend",
                  "hh_sin", "hh_cos", "dow_sin", "dow_cos", "route_id"]
ROLLING_FEATS  = [c for c in train_feat.columns
                  if c.startswith("rmean_") or c.startswith("rstd_")
                  or c in ("rmax_48", "rmin_48", "trend_4v8", "trend_8v48", "trend_48v96")]
cat_feats = ["route_id", "dow", "month"]

# ============================================================
# CUTOFF: обучаем на всём КРОМЕ последних TOTAL_VAL_PTS точек
#
# val_cutoff = timestamp 40-й точки с конца по каждому маршруту
# Берём глобальный минимум из per-route cutoff'ов — самый консервативный
# ============================================================
# Для каждого маршрута находим timestamp точки [-TOTAL_VAL_PTS]
per_route_cutoff = (
    train_feat.groupby("route_id")["timestamp"]
    .apply(lambda ts: ts.sort_values().iloc[-TOTAL_VAL_PTS - 1])  # последняя точка ПЕРЕД val
)
# Глобальный cutoff — берём медиану (не минимум, чтобы не терять много данных)
VAL_CUTOFF = per_route_cutoff.median()
print(f"\nVAL_CUTOFF (медиана по маршрутам): {VAL_CUTOFF}")
print(f"  Это {TOTAL_VAL_PTS} точек × 30 мин = {TOTAL_VAL_PTS * 30 / 60:.0f} часов от конца трейна")
print(f"  Train max: {train_feat['timestamp'].max()}")

# ============================================================
# TRAIN MODELS (h=1..8) — только на данных до VAL_CUTOFF
# ============================================================
print("\n" + "=" * 65)
print("TRAINING MODELS (train=[:-40pts], early stop на следующих 8)")
print("=" * 65)

models          = {}
test_preds_dict = {}

# Early stopping валидируем на следующих 8 точках после cutoff (fold 5)
ES_CUTOFF = VAL_CUTOFF + pd.Timedelta(minutes=30 * FORECAST_STEPS)

for h in range(1, FORECAST_STEPS + 1):
    safe_lags = [f"lag_{l}" for l in ALL_LAGS if l >= h]
    FEAT_COLS = CALENDAR_FEATS + ROLLING_FEATS + safe_lags

    tmp = train_feat.copy()
    tmp[f"target_h{h}"] = tmp.groupby("route_id")[TARGET_COL].shift(-h)
    tmp = tmp[tmp[f"target_h{h}"].notna() & tmp[f"lag_{ALL_LAGS[-1]}"].notna()].copy()

    mask_tr  = tmp["timestamp"] <= VAL_CUTOFF
    mask_es  = (tmp["timestamp"] > VAL_CUTOFF) & (tmp["timestamp"] <= ES_CUTOFF)

    X_tr, y_tr = tmp.loc[mask_tr, FEAT_COLS], np.log1p(tmp.loc[mask_tr, f"target_h{h}"].clip(0))
    X_es, y_es = tmp.loc[mask_es, FEAT_COLS], np.log1p(tmp.loc[mask_es, f"target_h{h}"].clip(0))

    print(f"\n── h={h}  train={mask_tr.sum()}  early_stop_val={mask_es.sum()}  feats={len(FEAT_COLS)}")

    dtrain = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_feats, free_raw_data=False)
    des    = lgb.Dataset(X_es, label=y_es, categorical_feature=cat_feats,
                         free_raw_data=False, reference=dtrain)

    model_h = lgb.train(
        LGB_PARAMS, dtrain,
        num_boost_round=LGB_PARAMS["n_estimators"],
        valid_sets=[des],
        callbacks=[
            lgb.early_stopping(EARLY_STOPPING, verbose=False),
            lgb.log_evaluation(period=500),
        ],
    )
    models[h] = (model_h, FEAT_COLS)

    # Test predictions
    mask_step_h = test_feat["step_in_route"] == h
    X_test_h    = test_feat.loc[mask_step_h, FEAT_COLS]
    if len(X_test_h) > 0:
        for route_id, pred in zip(
            test_feat.loc[mask_step_h, "route_id"].values,
            np.expm1(model_h.predict(X_test_h)).clip(0)
        ):
            test_preds_dict[(int(route_id), h)] = float(pred)

# ============================================================
# ROLLING CV — 5 фолдов на последних 40 точках
# Модель их НЕ видела при обучении → чистый CV
# ============================================================
print("\n" + "=" * 65)
print(f"ROLLING CV  ({N_FOLDS} folds × {FORECAST_STEPS} steps)")
print("=" * 65)

raw_rows = []

for fold in range(N_FOLDS):
    val_offset_end   = TOTAL_VAL_PTS - fold * FORECAST_STEPS   # 40, 32, 24, 16, 8
    val_offset_start = val_offset_end - FORECAST_STEPS          # 32, 24, 16,  8, 0

    for route_id, grp in train_feat.groupby("route_id"):
        grp = grp.sort_values("timestamp").reset_index(drop=True)
        if len(grp) < TOTAL_VAL_PTS + max(ALL_LAGS) + 1:
            continue

        val_part = (grp.iloc[-val_offset_end:-val_offset_start]
                    if val_offset_start > 0
                    else grp.iloc[-val_offset_end:]).copy()

        if len(val_part) == 0:
            continue

        for step_idx in range(len(val_part)):
            h = step_idx + 1
            if h > FORECAST_STEPS or h not in models:
                continue

            model_h, FEAT_COLS = models[h]

            # Берём фичи из train_feat для точки на h шагов ДО val_part[step_idx]
            # то есть той строки, по которой model_h предсказывает h шагов вперёд
            target_ts = val_part.iloc[step_idx]["timestamp"]
            # строка в train_feat с timestamp = target_ts - h*30min
            src_ts = target_ts - pd.Timedelta(minutes=30 * h)
            mask_src = (train_feat["route_id"] == route_id) & (train_feat["timestamp"] == src_ts)
            feat_row = train_feat.loc[mask_src, FEAT_COLS]

            if len(feat_row) == 0:
                continue

            pred   = float(np.expm1(model_h.predict(feat_row)).clip(0)[0])
            y_true = float(val_part.iloc[step_idx][TARGET_COL])
            ae     = abs(pred - y_true)
            ape    = ae / (y_true + 1e-9)
            err    = pred - y_true

            raw_rows.append({
                "fold":      fold + 1,
                "route_id":  route_id,
                "timestamp": val_part.iloc[step_idx]["timestamp"],
                "h":         h,
                "y_true":    y_true,
                "y_pred":    pred,
                "ae":        ae,
                "ape":       ape,
                "err":       err,
            })

raw_cv_df = pd.DataFrame(raw_rows)

# ── Агрегаты по фолду ──
agg_fold = raw_cv_df.groupby("fold").apply(lambda df: pd.Series({
    "wape":   np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias":  abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":    df["ae"].mean(),
    "n":      len(df),
})).reset_index()

# ── Агрегаты по шагу h ──
agg_h = raw_cv_df.groupby("h").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
})).reset_index()

# ── Агрегаты по маршруту ──
agg_route = raw_cv_df.groupby("route_id").agg(
    n_points    = ("y_true",  "count"),
    mae         = ("ae",      "mean"),
    median_ae   = ("ae",      "median"),
    std_ae      = ("ae",      "std"),
    mape        = ("ape",     "mean"),
    median_ape  = ("ape",     "median"),
    mean_err    = ("err",     "mean"),
    median_err  = ("err",     "median"),
    mean_true   = ("y_true",  "mean"),
    mean_pred   = ("y_pred",  "mean"),
).reset_index()
agg_route["calib_scale"] = (
    raw_cv_df.groupby("route_id")["y_true"].sum().values /
    (raw_cv_df.groupby("route_id")["y_pred"].sum().values + 1e-9)
)
agg_route["wape"] = (
    raw_cv_df.groupby("route_id")["ae"].sum().values /
    (raw_cv_df.groupby("route_id")["y_true"].sum().values + 1e-9)
)

# ── Глобальные метрики ──
total_raw, wape_raw, rbias_raw = wape_rbias(raw_cv_df["y_true"], raw_cv_df["y_pred"])
calib_scale_cv = float(raw_cv_df["y_true"].sum() / (raw_cv_df["y_pred"].sum() + 1e-9))
y_pred_cal = np.clip(raw_cv_df["y_pred"] * calib_scale_cv, 0, None)
total_cal, wape_cal, rbias_cal = wape_rbias(raw_cv_df["y_true"], y_pred_cal)

print("\n── По фолдам ──")
print(agg_fold.to_string(index=False))
print("\n── По шагам h ──")
print(agg_h.to_string(index=False))
print(f"\n{'Global RAW':25s}  WAPE={wape_raw:.4f}  |RBias|={rbias_raw:.4f}  Total={total_raw:.4f}")
print(f"{'Global CALIBRATED':25s}  WAPE={wape_cal:.4f}  |RBias|={rbias_cal:.4f}  Total={total_cal:.4f}")
print(f"  calib_scale (CV) = {calib_scale_cv:.4f}")
print(f"\nCalibration {'HELPS ✅' if total_cal < total_raw else 'HURTS ❌'} "
      f"(Δ = {total_cal - total_raw:+.4f})")
print("\n── Топ-10 сложных маршрутов (по median_ape) ──")
print(agg_route.sort_values("median_ape", ascending=False)
      [["route_id", "mae", "median_ae", "mape", "median_ape", "mean_err", "calib_scale"]]
      .head(10).to_string(index=False))

raw_cv_df.to_csv("lgb_cv_raw.csv", index=False)
agg_route.to_csv("lgb_cv_agg_route.csv", index=False)
agg_fold.to_csv("lgb_cv_agg_fold.csv", index=False)
agg_h.to_csv("lgb_cv_agg_h.csv", index=False)
print("\n✅ lgb_cv_raw.csv")
print("✅ lgb_cv_agg_route.csv")
print("✅ lgb_cv_agg_fold.csv")
print("✅ lgb_cv_agg_h.csv")

# ============================================================
# SUBMISSION
# ============================================================
final_preds = []
for _, row in test_feat.iterrows():
    route_id = int(row["route_id"])
    h        = int(row["step_in_route"])
    pred     = test_preds_dict.get((route_id, h), 0.0)
    final_preds.append(max(0.0, pred * calib_scale_cv))

submission = test_df[["id"]].copy().reset_index(drop=True)
submission["y_pred"] = final_preds
submission = submission.sort_values("id").reset_index(drop=True)
submission.to_csv("submission_lgb_direct.csv", index=False)
print(f"\n✅ submission_lgb_direct.csv ({len(submission)} rows)")
print(submission["y_pred"].describe().round(2))

In [ ]:
raw_cv_df.head()

In [ ]:
# ============================================================
# Chronos-2 — Rolling CV (5 folds × 8 steps)
# + per-route статистика для блендинга (как в LGB)
# ============================================================
import warnings
import numpy as np
import pandas as pd
import torch
from chronos import Chronos2Pipeline
from tqdm import tqdm
import os

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None


def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias


# -------------------- Config --------------------
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8
N_FOLDS        = 5
TOTAL_VAL_PTS  = N_FOLDS * FORECAST_STEPS   # 40
CONTEXT_LEN    = 2048

os.environ["TRANSFORMERS_OFFLINE"] = "1"   # трансформеры не лезут в сеть
os.environ["HF_DATASETS_OFFLINE"] = "1"    # датасеты тоже
os.environ["HF_HUB_OFFLINE"] = "1"         # huggingface hub тоже

# -------------------- Device --------------------
if torch.backends.mps.is_available():
    DEVICE = "mps"
    print("Apple Silicon MPS ✅")
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(f"Device: {DEVICE}\n")

# -------------------- Load model --------------------
MODEL_ID = "amazon/chronos-2"
print(f"Loading {MODEL_ID}...")
pipeline = Chronos2Pipeline.from_pretrained(
    MODEL_ID,
    device_map=DEVICE,
    torch_dtype=torch.bfloat16,
    local_files_only=True,
)
print("Model loaded!\n")

# -------------------- Data --------------------
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

status_cols = sorted([c for c in train_df.columns if c.startswith("status_")])
print(f"Train: {train_df.shape}  Test: {test_df.shape}")
print(f"Status cols: {status_cols}\n")


# -------------------- Helpers --------------------
def prepare_context_df(route_df: pd.DataFrame, route_id) -> pd.DataFrame:
    df = (
        route_df.set_index("timestamp")[[TARGET_COL] + status_cols]
        .asfreq("30min")
        .interpolate(method="time")
        .bfill().ffill()
        .tail(CONTEXT_LEN)
        .reset_index()
    )
    df["id"] = route_id
    df = df.rename(columns={TARGET_COL: "target"})
    return df


def run_inference(context_full: pd.DataFrame, cross_learning: bool = False) -> pd.DataFrame:
    return pipeline.predict_df(
        context_full,
        prediction_length=FORECAST_STEPS,
        quantile_levels=[0.1, 0.5, 0.9],
        id_column="id",
        timestamp_column="timestamp",
        target="target",
        cross_learning=cross_learning,
    )


def get_pred_col(pred_df: pd.DataFrame) -> str:
    for candidate in ["mean", "0.5", "q0.5", "median"]:
        if candidate in pred_df.columns:
            return candidate
    num_cols = [c for c in pred_df.columns
                if c not in ("id", "timestamp") and pd.api.types.is_numeric_dtype(pred_df[c])]
    return num_cols[0]


def build_preds_dict(pred_df: pd.DataFrame, pred_col: str) -> dict:
    result = {}
    for route_id, grp in pred_df.groupby("id"):
        result[route_id] = np.clip(grp.sort_values("timestamp")[pred_col].values, 0, None)
    return result


# ============================================================
# ROLLING CROSS-VALIDATION
# fold 1: история=[:-40]  val=[-40:-32]
# fold 2: история=[:-32]  val=[-32:-24]
# fold 3: история=[:-24]  val=[-24:-16]
# fold 4: история=[:-16]  val=[-16: -8]
# fold 5: история=[: -8]  val=[ -8:end]
# ============================================================
print("=" * 65)
print(f"ROLLING CV  ({N_FOLDS} folds × {FORECAST_STEPS} steps = {TOTAL_VAL_PTS} points)")
print("=" * 65)

raw_rows   = []   # все точки всех фолдов — для per-route аналитики
pred_col   = None

for fold in range(N_FOLDS):
    val_offset_end   = TOTAL_VAL_PTS - fold * FORECAST_STEPS   # 40, 32, 24, 16, 8
    val_offset_start = val_offset_end - FORECAST_STEPS          # 32, 24, 16,  8, 0

    context_dfs    = []
    y_trues_fold   = []
    route_ids_fold = []
    val_timestamps = []   # чтобы потом разложить по строкам

    for route_id, grp in train_df.groupby("route_id"):
        grp = grp.sort_values("timestamp").reset_index(drop=True)
        if len(grp) < TOTAL_VAL_PTS + 32:
            continue

        hist_part = grp.iloc[:-val_offset_end]
        val_part  = (grp.iloc[-val_offset_end:-val_offset_start]
                     if val_offset_start > 0
                     else grp.iloc[-val_offset_end:])

        if len(hist_part) < 32 or len(val_part) == 0:
            continue

        context_dfs.append(prepare_context_df(hist_part, route_id))
        y_trues_fold.append(val_part[TARGET_COL].values[:FORECAST_STEPS])
        route_ids_fold.append(route_id)
        val_timestamps.append(val_part["timestamp"].values[:FORECAST_STEPS])

    context_full = pd.concat(context_dfs, ignore_index=True)

    print(f"\nFold {fold+1}/{N_FOLDS}  "
          f"(val: -{val_offset_end}..{'-'+str(val_offset_start) if val_offset_start else 'end'})  "
          f"routes={len(route_ids_fold)}")

    pred_df_fold = run_inference(context_full)

    if pred_col is None:
        pred_col = get_pred_col(pred_df_fold)
        print(f"  Point forecast column: '{pred_col}'")

    preds = build_preds_dict(pred_df_fold, pred_col)

    # ── Собираем raw_rows по каждой точке ──
    yt_fold, yp_fold = [], []
    for i, route_id in enumerate(route_ids_fold):
        yp = preds.get(route_id, np.zeros(len(y_trues_fold[i])))
        for h_idx in range(len(y_trues_fold[i])):
            y_true = float(y_trues_fold[i][h_idx])
            y_pred = float(yp[h_idx]) if h_idx < len(yp) else 0.0
            ae     = abs(y_pred - y_true)
            ape    = ae / (y_true + 1e-9)
            err    = y_pred - y_true
            raw_rows.append({
                "fold":      fold + 1,
                "route_id":  route_id,
                "timestamp": val_timestamps[i][h_idx],
                "h":         h_idx + 1,
                "y_true":    y_true,
                "y_pred":    y_pred,
                "ae":        ae,
                "ape":       ape,
                "err":       err,
            })
            yt_fold.append(y_true)
            yp_fold.append(y_pred)

    yt_fold = np.array(yt_fold)
    yp_fold = np.array(yp_fold)
    tot, wape, rb = wape_rbias(yt_fold, yp_fold)
    print(f"  RAW   WAPE={wape:.4f}  |RBias|={rb:.4f}  Total={tot:.4f}")

# ============================================================
# АГРЕГАТЫ (аналогично LGB для блендинга)
# ============================================================
raw_cv_df = pd.DataFrame(raw_rows)

# ── По фолду ──
agg_fold = raw_cv_df.groupby("fold").apply(lambda df: pd.Series({
    "wape":   np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias":  abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":    df["ae"].mean(),
    "n":      len(df),
})).reset_index()

# ── По шагу h ──
agg_h = raw_cv_df.groupby("h").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
})).reset_index()

# ── По маршруту ──
agg_route = raw_cv_df.groupby("route_id").agg(
    n_points   = ("y_true",  "count"),
    mae        = ("ae",      "mean"),
    median_ae  = ("ae",      "median"),
    std_ae     = ("ae",      "std"),
    mape       = ("ape",     "mean"),
    median_ape = ("ape",     "median"),
    mean_err   = ("err",     "mean"),
    median_err = ("err",     "median"),
    mean_true  = ("y_true",  "mean"),
    mean_pred  = ("y_pred",  "mean"),
).reset_index()
agg_route["calib_scale"] = (
    raw_cv_df.groupby("route_id")["y_true"].sum().values /
    (raw_cv_df.groupby("route_id")["y_pred"].sum().values + 1e-9)
)
agg_route["wape"] = (
    raw_cv_df.groupby("route_id")["ae"].sum().values /
    (raw_cv_df.groupby("route_id")["y_true"].sum().values + 1e-9)
)

# ── Глобальные метрики ──
y_true_all = raw_cv_df["y_true"].values
y_pred_all = raw_cv_df["y_pred"].values
total_raw, wape_raw, rbias_raw = wape_rbias(y_true_all, y_pred_all)

calib_scale = float(y_true_all.sum() / (y_pred_all.sum() + 1e-9))
y_pred_cal  = np.clip(y_pred_all * calib_scale, 0, None)
total_cal, wape_cal, rbias_cal = wape_rbias(y_true_all, y_pred_cal)

print("\n" + "=" * 65)
print("SUMMARY")
print("=" * 65)
print("\n── По фолдам ──")
print(agg_fold.to_string(index=False))
print("\n── По шагам h ──")
print(agg_h.to_string(index=False))
print(f"\n{'Global RAW':25s}  WAPE={wape_raw:.4f}  |RBias|={rbias_raw:.4f}  Total={total_raw:.4f}")
print(f"{'Global CALIBRATED':25s}  WAPE={wape_cal:.4f}  |RBias|={rbias_cal:.4f}  Total={total_cal:.4f}")
print(f"  calib_scale = {calib_scale:.4f}")
print(f"\nCalibration {'HELPS ✅' if total_cal < total_raw else 'HURTS ❌'} "
      f"(Δ = {total_cal - total_raw:+.4f})")
print("\n── Топ-10 сложных маршрутов (по median_ape) ──")
print(agg_route.sort_values("median_ape", ascending=False)
      [["route_id", "mae", "median_ae", "mape", "median_ape", "mean_err", "calib_scale"]]
      .head(10).to_string(index=False))

raw_cv_df.to_csv("chronos2_cv_raw.csv", index=False)
agg_route.to_csv("chronos2_cv_agg_route.csv", index=False)
agg_fold.to_csv("chronos2_cv_agg_fold.csv", index=False)
agg_h.to_csv("chronos2_cv_agg_h.csv", index=False)
print("\n✅ chronos2_cv_raw.csv")
print("✅ chronos2_cv_agg_route.csv")
print("✅ chronos2_cv_agg_fold.csv")
print("✅ chronos2_cv_agg_h.csv")

# ============================================================
# TEST PREDICTION
# ============================================================
print("\n" + "=" * 65)
print("TEST PREDICTION")
print("=" * 65)

test_context_dfs = []
test_route_ids   = []

for route_id, grp in tqdm(test_df.groupby("route_id"), desc="Collect test"):
    route_train = train_df[train_df["route_id"] == route_id].sort_values("timestamp")
    if len(route_train) < 32:
        continue
    test_context_dfs.append(prepare_context_df(route_train, route_id))
    test_route_ids.append(route_id)

test_context_full = pd.concat(test_context_dfs, ignore_index=True)
print(f"Routes: {len(test_route_ids)}  Running inference...")

test_pred_df = run_inference(test_context_full)
test_preds   = build_preds_dict(test_pred_df, pred_col)

predictions_raw = {}
for route_id in test_route_ids:
    route_test = test_df[test_df["route_id"] == route_id].sort_values("timestamp")
    preds = test_preds.get(route_id, np.zeros(FORECAST_STEPS))
    for j, (_, row) in enumerate(route_test.iterrows()):
        pred = float(preds[j]) if j < len(preds) else float(preds[-1])
        predictions_raw[row["id"]] = max(0.0, pred)

submission_raw = (
    pd.DataFrame(list(predictions_raw.items()), columns=["id", "y_pred"])
    .sort_values("id").reset_index(drop=True)
)
submission_cal = submission_raw.copy()
submission_cal["y_pred"] = np.clip(submission_cal["y_pred"] * calib_scale, 0, None)

submission_raw.to_csv("submission_chronos2_raw.csv", index=False)
submission_cal.to_csv("submission_chronos2_calibrated.csv", index=False)
print(f"\n✅ submission_chronos2_raw.csv")
print(f"✅ submission_chronos2_calibrated.csv")
print(f"\n── Raw ──\n{submission_raw['y_pred'].describe().round(2)}")
print(f"\n── Calibrated (scale={calib_scale:.4f}) ──\n{submission_cal['y_pred'].describe().round(2)}")

In [ ]:
import os
os.environ["TRANSFORMERS_OFFLINE"] = "1"   # трансформеры не лезут в сеть
os.environ["HF_DATASETS_OFFLINE"] = "1"    # датасеты тоже
os.environ["HF_HUB_OFFLINE"] = "1"         # huggingface hub тоже

# Ставим ДО любых импортов chronos/transformers
from chronos import Chronos2Pipeline
import torch
if torch.backends.mps.is_available():
    DEVICE = "mps"
    print("Apple Silicon MPS ✅")
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(f"Device: {DEVICE}\n")
pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map=DEVICE,
    torch_dtype=torch.bfloat16,
    local_files_only=True,
)

In [ ]:
# ============================================================
# Chronos-2 — Rolling CV (5 folds × 8 steps)
# + per-route статистика для блендинга (как в LGB)
# ============================================================
import warnings
import numpy as np
import pandas as pd
import torch
from chronos import Chronos2Pipeline
from tqdm import tqdm
import os

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None


def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias


# -------------------- Config --------------------
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8
N_FOLDS        = 5
TOTAL_VAL_PTS  = N_FOLDS * FORECAST_STEPS   # 40
CONTEXT_LEN    = 2048

os.environ["TRANSFORMERS_OFFLINE"] = "1"   # трансформеры не лезут в сеть
os.environ["HF_DATASETS_OFFLINE"] = "1"    # датасеты тоже
os.environ["HF_HUB_OFFLINE"] = "1"         # huggingface hub тоже

# -------------------- Device --------------------
if torch.backends.mps.is_available():
    DEVICE = "mps"
    print("Apple Silicon MPS ✅")
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(f"Device: {DEVICE}\n")

# -------------------- Load model --------------------
MODEL_ID = "amazon/chronos-2"
print(f"Loading {MODEL_ID}...")
pipeline = Chronos2Pipeline.from_pretrained(
    MODEL_ID,
    device_map=DEVICE,
    torch_dtype=torch.bfloat16,
    local_files_only=True,
)
print("Model loaded!\n")

# -------------------- Data --------------------
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

status_cols = sorted([c for c in train_df.columns if c.startswith("status_")])
print(f"Train: {train_df.shape}  Test: {test_df.shape}")
print(f"Status cols: {status_cols}\n")


# -------------------- Helpers --------------------
def prepare_context_df(route_df: pd.DataFrame, route_id) -> pd.DataFrame:
    df = (
        route_df.set_index("timestamp")[[TARGET_COL] + status_cols]
        .asfreq("30min")
        .interpolate(method="time")
        .bfill().ffill()
        .tail(CONTEXT_LEN)
        .reset_index()
    )
    df["id"] = route_id
    df = df.rename(columns={TARGET_COL: "target"})
    return df


def run_inference(context_full: pd.DataFrame, cross_learning: bool = ) -> pd.DataFrame:
    return pipeline.predict_df(
        context_full,
        prediction_length=FORECAST_STEPS,
        quantile_levels=[0.1, 0.5, 0.9],
        id_column="id",
        timestamp_column="timestamp",
        target="target",
        cross_learning=cross_learning,
    )


def get_pred_col(pred_df: pd.DataFrame) -> str:
    for candidate in ["mean", "0.5", "q0.5", "median"]:
        if candidate in pred_df.columns:
            return candidate
    num_cols = [c for c in pred_df.columns
                if c not in ("id", "timestamp") and pd.api.types.is_numeric_dtype(pred_df[c])]
    return num_cols[0]


def build_preds_dict(pred_df: pd.DataFrame, pred_col: str) -> dict:
    result = {}
    for route_id, grp in pred_df.groupby("id"):
        result[route_id] = np.clip(grp.sort_values("timestamp")[pred_col].values, 0, None)
    return result


# ============================================================
# ROLLING CROSS-VALIDATION
# fold 1: история=[:-40]  val=[-40:-32]
# fold 2: история=[:-32]  val=[-32:-24]
# fold 3: история=[:-24]  val=[-24:-16]
# fold 4: история=[:-16]  val=[-16: -8]
# fold 5: история=[: -8]  val=[ -8:end]
# ============================================================
print("=" * 65)
print(f"ROLLING CV  ({N_FOLDS} folds × {FORECAST_STEPS} steps = {TOTAL_VAL_PTS} points)")
print("=" * 65)

raw_rows   = []   # все точки всех фолдов — для per-route аналитики
pred_col   = None

for fold in range(N_FOLDS):
    val_offset_end   = TOTAL_VAL_PTS - fold * FORECAST_STEPS   # 40, 32, 24, 16, 8
    val_offset_start = val_offset_end - FORECAST_STEPS          # 32, 24, 16,  8, 0

    context_dfs    = []
    y_trues_fold   = []
    route_ids_fold = []
    val_timestamps = []   # чтобы потом разложить по строкам

    for route_id, grp in train_df.groupby("route_id"):
        grp = grp.sort_values("timestamp").reset_index(drop=True)
        if len(grp) < TOTAL_VAL_PTS + 32:
            continue

        hist_part = grp.iloc[:-val_offset_end]
        val_part  = (grp.iloc[-val_offset_end:-val_offset_start]
                     if val_offset_start > 0
                     else grp.iloc[-val_offset_end:])

        if len(hist_part) < 32 or len(val_part) == 0:
            continue

        context_dfs.append(prepare_context_df(hist_part, route_id))
        y_trues_fold.append(val_part[TARGET_COL].values[:FORECAST_STEPS])
        route_ids_fold.append(route_id)
        val_timestamps.append(val_part["timestamp"].values[:FORECAST_STEPS])

    context_full = pd.concat(context_dfs, ignore_index=True)

    print(f"\nFold {fold+1}/{N_FOLDS}  "
          f"(val: -{val_offset_end}..{'-'+str(val_offset_start) if val_offset_start else 'end'})  "
          f"routes={len(route_ids_fold)}")

    pred_df_fold = run_inference(context_full)

    if pred_col is None:
        pred_col = get_pred_col(pred_df_fold)
        print(f"  Point forecast column: '{pred_col}'")

    preds = build_preds_dict(pred_df_fold, pred_col)

    # ── Собираем raw_rows по каждой точке ──
    yt_fold, yp_fold = [], []
    for i, route_id in enumerate(route_ids_fold):
        yp = preds.get(route_id, np.zeros(len(y_trues_fold[i])))
        for h_idx in range(len(y_trues_fold[i])):
            y_true = float(y_trues_fold[i][h_idx])
            y_pred = float(yp[h_idx]) if h_idx < len(yp) else 0.0
            ae     = abs(y_pred - y_true)
            ape    = ae / (y_true + 1e-9)
            err    = y_pred - y_true
            raw_rows.append({
                "fold":      fold + 1,
                "route_id":  route_id,
                "timestamp": val_timestamps[i][h_idx],
                "h":         h_idx + 1,
                "y_true":    y_true,
                "y_pred":    y_pred,
                "ae":        ae,
                "ape":       ape,
                "err":       err,
            })
            yt_fold.append(y_true)
            yp_fold.append(y_pred)

    yt_fold = np.array(yt_fold)
    yp_fold = np.array(yp_fold)
    tot, wape, rb = wape_rbias(yt_fold, yp_fold)
    print(f"  RAW   WAPE={wape:.4f}  |RBias|={rb:.4f}  Total={tot:.4f}")

# ============================================================
# АГРЕГАТЫ (аналогично LGB для блендинга)
# ============================================================
raw_cv_df = pd.DataFrame(raw_rows)

# ── По фолду ──
agg_fold = raw_cv_df.groupby("fold").apply(lambda df: pd.Series({
    "wape":   np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias":  abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":    df["ae"].mean(),
    "n":      len(df),
})).reset_index()

# ── По шагу h ──
agg_h = raw_cv_df.groupby("h").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
})).reset_index()

# ── По маршруту ──
agg_route = raw_cv_df.groupby("route_id").agg(
    n_points   = ("y_true",  "count"),
    mae        = ("ae",      "mean"),
    median_ae  = ("ae",      "median"),
    std_ae     = ("ae",      "std"),
    mape       = ("ape",     "mean"),
    median_ape = ("ape",     "median"),
    mean_err   = ("err",     "mean"),
    median_err = ("err",     "median"),
    mean_true  = ("y_true",  "mean"),
    mean_pred  = ("y_pred",  "mean"),
).reset_index()
agg_route["calib_scale"] = (
    raw_cv_df.groupby("route_id")["y_true"].sum().values /
    (raw_cv_df.groupby("route_id")["y_pred"].sum().values + 1e-9)
)
agg_route["wape"] = (
    raw_cv_df.groupby("route_id")["ae"].sum().values /
    (raw_cv_df.groupby("route_id")["y_true"].sum().values + 1e-9)
)

# ── Глобальные метрики ──
y_true_all = raw_cv_df["y_true"].values
y_pred_all = raw_cv_df["y_pred"].values
total_raw, wape_raw, rbias_raw = wape_rbias(y_true_all, y_pred_all)

calib_scale = float(y_true_all.sum() / (y_pred_all.sum() + 1e-9))
y_pred_cal  = np.clip(y_pred_all * calib_scale, 0, None)
total_cal, wape_cal, rbias_cal = wape_rbias(y_true_all, y_pred_cal)

print("\n" + "=" * 65)
print("SUMMARY")
print("=" * 65)
print("\n── По фолдам ──")
print(agg_fold.to_string(index=False))
print("\n── По шагам h ──")
print(agg_h.to_string(index=False))
print(f"\n{'Global RAW':25s}  WAPE={wape_raw:.4f}  |RBias|={rbias_raw:.4f}  Total={total_raw:.4f}")
print(f"{'Global CALIBRATED':25s}  WAPE={wape_cal:.4f}  |RBias|={rbias_cal:.4f}  Total={total_cal:.4f}")
print(f"  calib_scale = {calib_scale:.4f}")
print(f"\nCalibration {'HELPS ✅' if total_cal < total_raw else 'HURTS ❌'} "
      f"(Δ = {total_cal - total_raw:+.4f})")
print("\n── Топ-10 сложных маршрутов (по median_ape) ──")
print(agg_route.sort_values("median_ape", ascending=False)
      [["route_id", "mae", "median_ae", "mape", "median_ape", "mean_err", "calib_scale"]]
      .head(10).to_string(index=False))

raw_cv_df.to_csv("chronos2_cv_raw.csv", index=False)
agg_route.to_csv("chronos2_cv_agg_route.csv", index=False)
agg_fold.to_csv("chronos2_cv_agg_fold.csv", index=False)
agg_h.to_csv("chronos2_cv_agg_h.csv", index=False)
print("\n✅ chronos2_cv_raw.csv")
print("✅ chronos2_cv_agg_route.csv")
print("✅ chronos2_cv_agg_fold.csv")
print("✅ chronos2_cv_agg_h.csv")

# ============================================================
# TEST PREDICTION
# ============================================================
print("\n" + "=" * 65)
print("TEST PREDICTION")
print("=" * 65)

test_context_dfs = []
test_route_ids   = []

for route_id, grp in tqdm(test_df.groupby("route_id"), desc="Collect test"):
    route_train = train_df[train_df["route_id"] == route_id].sort_values("timestamp")
    if len(route_train) < 32:
        continue
    test_context_dfs.append(prepare_context_df(route_train, route_id))
    test_route_ids.append(route_id)

test_context_full = pd.concat(test_context_dfs, ignore_index=True)
print(f"Routes: {len(test_route_ids)}  Running inference...")

test_pred_df = run_inference(test_context_full)
test_preds   = build_preds_dict(test_pred_df, pred_col)

predictions_raw = {}
for route_id in test_route_ids:
    route_test = test_df[test_df["route_id"] == route_id].sort_values("timestamp")
    preds = test_preds.get(route_id, np.zeros(FORECAST_STEPS))
    for j, (_, row) in enumerate(route_test.iterrows()):
        pred = float(preds[j]) if j < len(preds) else float(preds[-1])
        predictions_raw[row["id"]] = max(0.0, pred)

submission_raw = (
    pd.DataFrame(list(predictions_raw.items()), columns=["id", "y_pred"])
    .sort_values("id").reset_index(drop=True)
)
submission_cal = submission_raw.copy()
submission_cal["y_pred"] = np.clip(submission_cal["y_pred"] * calib_scale, 0, None)

submission_raw.to_csv("submission_chronos2_raw.csv", index=False)
submission_cal.to_csv("submission_chronos2_calibrated.csv", index=False)
print(f"\n✅ submission_chronos2_raw.csv")
print(f"✅ submission_chronos2_calibrated.csv")
print(f"\n── Raw ──\n{submission_raw['y_pred'].describe().round(2)}")
print(f"\n── Calibrated (scale={calib_scale:.4f}) ──\n{submission_cal['y_pred'].describe().round(2)}")

In [ ]:
# ============================================================
# SARIMA per route — Rolling CV (5 folds × 8 steps) + submission
# ============================================================
import warnings
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from tqdm import tqdm

warnings.filterwarnings("ignore")


def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias


# -------------------- Config --------------------
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8
N_FOLDS        = 5
TOTAL_VAL_PTS  = N_FOLDS * FORECAST_STEPS  # 40
TRAIN_DAYS     = 14

SARIMA_ORDER          = (1, 1, 1)
SARIMA_SEASONAL_ORDER = (1, 1, 0, 24)

# -------------------- Data --------------------
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
print(f"Train: {train_df.shape}  Test: {test_df.shape}")
route_ids = train_df["route_id"].unique()


# -------------------- Forecast helper (твоя рабочая версия) --------------------
def sarima_forecast_for_timestamps(route_history_df, future_timestamps):
    future_timestamps = pd.to_datetime(pd.Series(future_timestamps)).sort_values()
    future_index = pd.DatetimeIndex(future_timestamps)

    route_train_30 = (
        route_history_df
        .set_index("timestamp")[TARGET_COL]
        .sort_index()
        .asfreq("30min")
        .interpolate(method="time")
        .bfill().ffill()
    )

    if len(route_train_30) == 0:
        return pd.Series(0.0, index=future_index)

    cutoff = route_train_30.index.max() - pd.Timedelta(days=TRAIN_DAYS)
    route_train_30 = route_train_30[route_train_30.index > cutoff]

    if len(route_train_30) < 96:
        mean_val = float(route_train_30.mean())
        return pd.Series(mean_val, index=future_index)

    route_train_1h = route_train_30.resample("1h").mean().interpolate(method="time")

    last_train_ts = route_train_1h.index[-1]
    max_future_ts = future_index.max().ceil("1h")
    n_hours = int((max_future_ts - last_train_ts) / pd.Timedelta("1h")) + 2

    try:
        model = SARIMAX(
            route_train_1h,
            order=SARIMA_ORDER,
            seasonal_order=SARIMA_SEASONAL_ORDER,
            enforce_stationarity=False,
            enforce_invertibility=False,
        )
        result = model.fit(disp=False, maxiter=200, method="lbfgs")

        hourly_idx = pd.date_range(
            start=last_train_ts + pd.Timedelta("1h"),
            periods=n_hours,
            freq="1h",
        )
        forecast_1h = pd.Series(result.forecast(steps=n_hours).values, index=hourly_idx)
        forecast_30min = forecast_1h.resample("30min").interpolate(method="linear")

        preds = []
        for ts in future_index:
            pred = float(forecast_30min.get(ts, forecast_30min.iloc[-1]))
            preds.append(max(0.0, pred))
        return pd.Series(preds, index=future_index)

    except Exception:
        preds = []
        for ts in future_index:
            h, m = ts.hour, ts.minute
            mask = (route_train_30.index.hour == h) & (route_train_30.index.minute == m)
            fb = route_train_30[mask].mean() if mask.any() else route_train_30.mean()
            preds.append(max(0.0, float(fb)))
        return pd.Series(preds, index=future_index)


# ============================================================
# ROLLING CROSS-VALIDATION
# fold 1: история=[:-40]  val=[-40:-32]
# fold 2: история=[:-32]  val=[-32:-24]
# fold 3: история=[:-24]  val=[-24:-16]
# fold 4: история=[:-16]  val=[-16: -8]
# fold 5: история=[: -8]  val=[ -8:end]
# ============================================================
print("=" * 65)
print(f"SARIMA ROLLING CV  ({N_FOLDS} folds × {FORECAST_STEPS} steps = {TOTAL_VAL_PTS} points)")
print("=" * 65)

raw_rows = []

for fold in range(N_FOLDS):
    val_offset_end   = TOTAL_VAL_PTS - fold * FORECAST_STEPS
    val_offset_start = val_offset_end - FORECAST_STEPS

    fold_yt, fold_yp = [], []

    for route_id in tqdm(route_ids, desc=f"Fold {fold+1}/{N_FOLDS}", leave=False):
        grp = train_df[train_df["route_id"] == route_id]\
                .sort_values("timestamp").reset_index(drop=True)

        if len(grp) < TOTAL_VAL_PTS + 100:
            continue

        hist_part = grp.iloc[:-val_offset_end].copy()
        val_part  = (grp.iloc[-val_offset_end:-val_offset_start]
                     if val_offset_start > 0
                     else grp.iloc[-val_offset_end:]).copy()

        if len(hist_part) < 96 or len(val_part) == 0:
            continue

        forecast = sarima_forecast_for_timestamps(
            route_history_df=hist_part,
            future_timestamps=val_part["timestamp"].values,
        )

        y_true_v   = val_part[TARGET_COL].values[:FORECAST_STEPS]
        timestamps = val_part["timestamp"].values[:FORECAST_STEPS]
        yp_v       = forecast.values[:FORECAST_STEPS]

        for h_idx in range(len(y_true_v)):
            y_true = float(y_true_v[h_idx])
            y_pred = float(yp_v[h_idx])
            ae     = abs(y_pred - y_true)
            ape    = ae / (y_true + 1e-9)
            err    = y_pred - y_true
            raw_rows.append({
                "fold":      fold + 1,
                "route_id":  route_id,
                "timestamp": timestamps[h_idx],
                "h":         h_idx + 1,
                "y_true":    y_true,
                "y_pred":    y_pred,
                "ae":        ae,
                "ape":       ape,
                "err":       err,
            })
            fold_yt.append(y_true)
            fold_yp.append(y_pred)

    yt = np.array(fold_yt)
    yp = np.array(fold_yp)
    tot, wape, rb = wape_rbias(yt, yp)
    print(f"Fold {fold+1}  WAPE={wape:.4f}  |RBias|={rb:.4f}  Total={tot:.4f}")

# ============================================================
# АГРЕГАТЫ (идентичны LGB и Chronos)
# ============================================================
raw_cv_df = pd.DataFrame(raw_rows)

agg_fold = raw_cv_df.groupby("fold").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
    "n":     len(df),
})).reset_index()

agg_h = raw_cv_df.groupby("h").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
})).reset_index()

agg_route = raw_cv_df.groupby("route_id").agg(
    n_points   = ("y_true",  "count"),
    mae        = ("ae",      "mean"),
    median_ae  = ("ae",      "median"),
    std_ae     = ("ae",      "std"),
    mape       = ("ape",     "mean"),
    median_ape = ("ape",     "median"),
    mean_err   = ("err",     "mean"),
    median_err = ("err",     "median"),
    mean_true  = ("y_true",  "mean"),
    mean_pred  = ("y_pred",  "mean"),
).reset_index()
agg_route["calib_scale"] = (
    raw_cv_df.groupby("route_id")["y_true"].sum().values /
    (raw_cv_df.groupby("route_id")["y_pred"].sum().values + 1e-9)
)
agg_route["wape"] = (
    raw_cv_df.groupby("route_id")["ae"].sum().values /
    (raw_cv_df.groupby("route_id")["y_true"].sum().values + 1e-9)
)

# Глобальные метрики
y_true_all = raw_cv_df["y_true"].values
y_pred_all = raw_cv_df["y_pred"].values
total_raw, wape_raw, rbias_raw = wape_rbias(y_true_all, y_pred_all)

calib_scale = float(y_true_all.sum() / (y_pred_all.sum() + 1e-9))
y_pred_cal  = np.clip(y_pred_all * calib_scale, 0, None)
total_cal, wape_cal, rbias_cal = wape_rbias(y_true_all, y_pred_cal)

print("\n" + "=" * 65)
print("SUMMARY")
print("=" * 65)
print("\n── По фолдам ──")
print(agg_fold.to_string(index=False))
print("\n── По шагам h ──")
print(agg_h.to_string(index=False))
print(f"\n{'Global RAW':25s}  WAPE={wape_raw:.4f}  |RBias|={rbias_raw:.4f}  Total={total_raw:.4f}")
print(f"{'Global CALIBRATED':25s}  WAPE={wape_cal:.4f}  |RBias|={rbias_cal:.4f}  Total={total_cal:.4f}")
print(f"  calib_scale = {calib_scale:.4f}")
print(f"\nCalibration {'HELPS ✅' if total_cal < total_raw else 'HURTS ❌'} "
      f"(Δ = {total_cal - total_raw:+.4f})")
print("\n── Топ-10 сложных маршрутов ──")
print(agg_route.sort_values("median_ape", ascending=False)
      [["route_id", "mae", "median_ae", "mape", "calib_scale"]]
      .head(10).to_string(index=False))

raw_cv_df.to_csv("sarima_cv_raw.csv", index=False)
agg_route.to_csv("sarima_cv_agg_route.csv", index=False)
agg_fold.to_csv("sarima_cv_agg_fold.csv", index=False)
agg_h.to_csv("sarima_cv_agg_h.csv", index=False)
print("\n✅ sarima_cv_raw.csv")
print("✅ sarima_cv_agg_route.csv")
print("✅ sarima_cv_agg_fold.csv")
print("✅ sarima_cv_agg_h.csv")

# ============================================================
# TEST PREDICTION
# ============================================================
print("\n" + "=" * 65)
print("TEST PREDICTION")
print("=" * 65)

predictions_raw = {}
for route_id in tqdm(route_ids, desc="Test inference"):
    route_train = train_df[train_df["route_id"] == route_id].sort_values("timestamp").copy()
    route_test  = test_df[test_df["route_id"] == route_id].sort_values("timestamp").copy()
    if len(route_train) < 96 or len(route_test) == 0:
        continue
    forecast = sarima_forecast_for_timestamps(
        route_history_df=route_train,
        future_timestamps=route_test["timestamp"].values,
    )
    for (_, row), pred in zip(route_test.iterrows(), forecast.values):
        predictions_raw[row["id"]] = float(max(0.0, pred))

submission_raw = (
    pd.DataFrame(list(predictions_raw.items()), columns=["id", "y_pred"])
    .sort_values("id").reset_index(drop=True)
)
submission_cal = submission_raw.copy()
submission_cal["y_pred"] = np.clip(submission_cal["y_pred"] * calib_scale, 0, None)

submission_raw.to_csv("submission_sarima_raw.csv", index=False)
submission_cal.to_csv("submission_sarima_calibrated.csv", index=False)
print(f"\n✅ submission_sarima_raw.csv")
print(f"✅ submission_sarima_calibrated.csv")
print(f"\n── Raw ──\n{submission_raw['y_pred'].describe().round(2)}")
print(f"\n── Calibrated (scale={calib_scale:.4f}) ──\n{submission_cal['y_pred'].describe().round(2)}")

In [ ]:
import numpy as np
import pandas as pd

for model_name, raw_path, out_path in [
    ("SARIMA",   "sarima_cv_raw.csv",   "sarima_cv_agg_route.csv"),
    ("Chronos2", "chronos2_cv_raw.csv", "chronos2_cv_agg_route.csv"),
    ("LGB",      "lgb_cv_raw.csv",      "lgb_cv_agg_route.csv"),
]:
    raw = pd.read_csv(raw_path)

    agg = raw.groupby("route_id").agg(
        n_points   = ("y_true",  "count"),
        mae        = ("ae",      "mean"),
        median_ae  = ("ae",      "median"),
        std_ae     = ("ae",      "std"),
        mape       = ("ape",     "mean"),
        median_ape = ("ape",     "median"),
        mean_err   = ("err",     "mean"),
        median_err = ("err",     "median"),
        mean_true  = ("y_true",  "mean"),
        mean_pred  = ("y_pred",  "mean"),
    ).reset_index()

    # Считаем по группам через transform чтобы не слететь с порядком индексов
    grp = raw.groupby("route_id")
    agg["sum_true"] = grp["y_true"].sum().values
    agg["sum_pred"] = grp["y_pred"].sum().values
    agg["sum_ae"]   = grp["ae"].sum().values

    agg["wape"]        = agg["sum_ae"]  / (agg["sum_true"] + 1e-9)
    agg["rbias"]       = (agg["sum_pred"] / (agg["sum_true"] + 1e-9) - 1).abs()
    agg["total"]       = agg["wape"] + agg["rbias"]
    agg["calib_scale"] = agg["sum_true"] / (agg["sum_pred"] + 1e-9)

    agg = agg.drop(columns=["sum_true", "sum_pred", "sum_ae"])

    agg.to_csv(out_path, index=False)
    print(f"\n── {model_name} ──")
    print(f"  Маршрутов: {len(agg)}")
    print(f"  Global WAPE:  {agg['wape'].mean():.4f}")
    print(f"  Global RBias: {agg['rbias'].mean():.4f}")
    print(f"  Global Total: {agg['total'].mean():.4f}")
    print(f"✅ {out_path}")

In [ ]:
import numpy as np
import pandas as pd

def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    return np.abs(yp-yt).sum()/s + np.abs(yp.sum()/s - 1)

sar_raw = pd.read_csv("sarima_cv_raw.csv")[["route_id","fold","timestamp","h","y_true","y_pred"]]\
            .rename(columns={"y_pred": "y_pred_sar"})
chr_raw = pd.read_csv("chronos2_cv_raw.csv")[["route_id","fold","timestamp","h","y_true","y_pred"]]\
            .rename(columns={"y_pred": "y_pred_chr"})

cv = chr_raw.merge(sar_raw, on=["route_id","fold","timestamp","h","y_true"], how="inner")

TRAIN_FOLDS = [1, 2, 3, 4]
TEST_FOLD   = 5
cv_train = cv[cv["fold"].isin(TRAIN_FOLDS)].copy()
cv_test  = cv[cv["fold"] == TEST_FOLD].copy()

base_total = wape_rbias(cv_test["y_true"], cv_test["y_pred_chr"])
sar_total  = wape_rbias(cv_test["y_true"], cv_test["y_pred_sar"])
print(f"Baseline Chronos fold5: {base_total:.4f}")
print(f"Baseline SARIMA  fold5: {sar_total:.4f}")

# Преагрегируем по маршруту для скорости
def route_stats(df):
    g = df.groupby("route_id")
    return pd.DataFrame({
        "ae_chr":   g.apply(lambda x: x["y_pred_chr"].sub(x["y_true"]).abs().sum()),
        "ae_sar":   g.apply(lambda x: x["y_pred_sar"].sub(x["y_true"]).abs().sum()),
        "pred_chr": g["y_pred_chr"].sum(),
        "pred_sar": g["y_pred_sar"].sum(),
        "true_r":   g["y_true"].sum(),
    }).reset_index()

for HORIZON_FILTER, label in [
    ([1],             "h=1"),
    ([1,2,3,4],       "h=1..4"),
    ([1,2,3,4,5,6,7,8], "h=1..8"),
]:
    cv_tr_h = cv_train[cv_train["h"].isin(HORIZON_FILTER)]
    stats   = route_stats(cv_tr_h)
    SUM_TR  = cv_tr_h["y_true"].sum()

    # Честный жадный: на каждом шаге пересчитываем от ТЕКУЩЕГО состояния
    cur_ae   = stats["ae_chr"].sum()
    cur_pred = stats["pred_chr"].sum()
    swap_set = set()

    # Сортируем по первоначальному delta (как эвристика порядка)
    stats["delta_ae"]   = stats["ae_sar"]   - stats["ae_chr"]
    stats["delta_pred"] = stats["pred_sar"] - stats["pred_chr"]
    stats = stats.sort_values("delta_ae")   # сначала маршруты где SARIMA точнее по AE

    improved = True
    while improved:
        improved = False
        best_delta, best_rid = 0, None

        for _, row in stats[~stats["route_id"].isin(swap_set)].iterrows():
            # Маргинальный эффект от ТЕКУЩЕГО состояния
            new_ae   = cur_ae   + row["ae_sar"]   - row["ae_chr"]
            new_pred = cur_pred + row["pred_sar"] - row["pred_chr"]
            new_total = new_ae / SUM_TR + abs(new_pred / SUM_TR - 1)
            cur_total = cur_ae / SUM_TR + abs(cur_pred / SUM_TR - 1)
            delta = new_total - cur_total

            if delta < best_delta:
                best_delta = delta
                best_rid   = row["route_id"]
                best_row   = row

        if best_rid is not None:
            swap_set.add(best_rid)
            cur_ae   += best_row["ae_sar"]   - best_row["ae_chr"]
            cur_pred += best_row["pred_sar"] - best_row["pred_chr"]
            improved = True

    # Применяем swap на fold 5
    cv_test["y_pred_swap"] = np.where(
        cv_test["route_id"].isin(swap_set),
        cv_test["y_pred_sar"],
        cv_test["y_pred_chr"],
    )

    # Калибровка честно по train
    cv_train["y_pred_swap"] = np.where(
        cv_train["route_id"].isin(swap_set),
        cv_train["y_pred_sar"],
        cv_train["y_pred_chr"],
    )
    cs = float(cv_train["y_true"].sum() /
               (cv_train["y_pred_swap"].clip(lower=0).sum() + 1e-9))

    y_cal = np.clip(cv_test["y_pred_swap"] * cs, 0, None)
    tot   = wape_rbias(cv_test["y_true"].values, y_cal.values)
    delta = tot - base_total
    sign  = "✅ BETTER" if delta < 0 else "❌ WORSE"

    print(f"\n── {label:10s} │ swap={len(swap_set):>3} маршрутов │ calib={cs:.3f} ──")
    print(f"   Total={tot:.4f}  Δ={delta:+.4f}  {sign}")
    # print(f"   Маршруты: {sorted(swap_set)}")

In [ ]:
# Насколько стабильно "кто лучше" между фолдами?
cv_train = cv[cv["fold"].isin([1,2,3,4])]
cv_test  = cv[cv["fold"] == 5]

# На каждом фолде и маршруте: кто лучше?
winner = cv.groupby(["fold","route_id"]).apply(lambda df: pd.Series({
    "sar_better": (df["y_pred_sar"].sub(df["y_true"]).abs().sum() <
                   df["y_pred_chr"].sub(df["y_true"]).abs().sum())
})).reset_index()

# Стабильность: маршрут где SARIMA лучше на ВСЕХ фолдах 1-4
always_sar = winner[winner["fold"].isin([1,2,3,4])]\
    .groupby("route_id")["sar_better"].sum()

print("Распределение: на скольких из 4 фолдов SARIMA лучше")
print(always_sar.value_counts().sort_index())

# Проверяем: если SARIMA лучше на всех 4 фолдах → лучше ли на fold 5?
fold5_winner = winner[winner["fold"]==5].set_index("route_id")["sar_better"]

for n_folds_sar_wins in [4, 3, 2]:
    stable_routes = always_sar[always_sar >= n_folds_sar_wins].index
    fold5_correct = fold5_winner.loc[fold5_winner.index.isin(stable_routes)].sum()
    fold5_total   = fold5_winner.loc[fold5_winner.index.isin(stable_routes)].count()
    print(f"\nSARIMA лучше на ≥{n_folds_sar_wins}/4 фолдах: {len(stable_routes)} маршрутов")
    print(f"  Из них SARIMA лучше и на fold5: {fold5_correct}/{fold5_total} "
          f"({100*fold5_correct/(fold5_total+1e-9):.1f}%)")
    print(f"  (random baseline = 50%)")

In [ ]:
import numpy as np
import pandas as pd

def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    return np.abs(yp-yt).sum()/s + np.abs(yp.sum()/s - 1)

lgb_raw = pd.read_csv("lgb_cv_raw.csv")[["route_id","fold","timestamp","h","y_true","y_pred"]]\
            .rename(columns={"y_pred": "y_pred_lgb"})
chr_raw = pd.read_csv("chronos2_cv_raw.csv")[["route_id","fold","timestamp","h","y_true","y_pred"]]\
            .rename(columns={"y_pred": "y_pred_chr"})

cv = chr_raw.merge(lgb_raw, on=["route_id","fold","timestamp","h","y_true"], how="inner")
print(f"Точек: {len(cv)}  Маршрутов: {cv['route_id'].nunique()}")

TRAIN_FOLDS = [1, 2, 3, 4]
TEST_FOLD   = 5
cv_train = cv[cv["fold"].isin(TRAIN_FOLDS)].copy()
cv_test  = cv[cv["fold"] == TEST_FOLD].copy()

base_chr = wape_rbias(cv_test["y_true"], cv_test["y_pred_chr"])
base_lgb = wape_rbias(cv_test["y_true"], cv_test["y_pred_lgb"])
print(f"\nBaseline Chronos fold5: {base_chr:.4f}")
print(f"Baseline LGB     fold5: {base_lgb:.4f}")

# ============================================================
# 1. Стабильность сигнала: кто лучше по фолдам
# ============================================================
winner = cv.groupby(["fold","route_id"]).apply(lambda df: pd.Series({
    "lgb_better": (df["y_pred_lgb"].sub(df["y_true"]).abs().sum() <
                   df["y_pred_chr"].sub(df["y_true"]).abs().sum())
})).reset_index()

always_lgb  = winner[winner["fold"].isin(TRAIN_FOLDS)]\
    .groupby("route_id")["lgb_better"].sum()
fold5_winner = winner[winner["fold"]==TEST_FOLD].set_index("route_id")["lgb_better"]

print("\n── Распределение: на скольких из 4 фолдов LGB лучше Chronos ──")
print(always_lgb.value_counts().sort_index())

print("\n── Предсказательная сила сигнала на fold 5 ──")
for n in [4, 3, 2]:
    stable = always_lgb[always_lgb >= n].index
    correct = fold5_winner.loc[fold5_winner.index.isin(stable)].sum()
    total   = fold5_winner.loc[fold5_winner.index.isin(stable)].count()
    print(f"  LGB лучше на ≥{n}/4 фолдах: {len(stable):>4} маршрутов  "
          f"→ fold5 правильно {correct}/{total} ({100*correct/(total+1e-9):.1f}%)  "
          f"(random=50%)")

# ============================================================
# 2. Честный жадный swap на train → тест на fold 5
# ============================================================
def route_stats(df, col_new, col_old):
    g = df.groupby("route_id")
    return pd.DataFrame({
        "ae_new":  g.apply(lambda x: x[col_new].sub(x["y_true"]).abs().sum()),
        "ae_old":  g.apply(lambda x: x[col_old].sub(x["y_true"]).abs().sum()),
        "pred_new": g[col_new].sum(),
        "pred_old": g[col_old].sum(),
    }).reset_index()

print("\n" + "="*65)
print("ЖАДНЫЙ SWAP: Chronos → LGB")
print("="*65)

for HORIZON_FILTER, label in [
    ([1],               "h=1"),
    ([1,2,3,4],         "h=1..4"),
    ([1,2,3,4,5,6,7,8], "h=1..8"),
]:
    cv_tr_h = cv_train[cv_train["h"].isin(HORIZON_FILTER)]
    stats   = route_stats(cv_tr_h, "y_pred_lgb", "y_pred_chr")
    SUM_TR  = cv_tr_h["y_true"].sum()

    cur_ae   = stats["ae_old"].sum()
    cur_pred = stats["pred_old"].sum()
    swap_set = set()

    improved = True
    while improved:
        improved = False
        best_delta, best_rid, best_row = 0, None, None

        for _, row in stats[~stats["route_id"].isin(swap_set)].iterrows():
            new_ae   = cur_ae   + row["ae_new"]   - row["ae_old"]
            new_pred = cur_pred + row["pred_new"] - row["pred_old"]
            new_total = new_ae / SUM_TR + abs(new_pred / SUM_TR - 1)
            cur_total = cur_ae / SUM_TR + abs(cur_pred / SUM_TR - 1)
            delta = new_total - cur_total
            if delta < best_delta:
                best_delta, best_rid, best_row = delta, row["route_id"], row

        if best_rid is not None:
            swap_set.add(best_rid)
            cur_ae   += best_row["ae_new"] - best_row["ae_old"]
            cur_pred += best_row["pred_new"] - best_row["pred_old"]
            improved = True

    # Применяем на fold 5
    cv_test["y_pred_swap"] = np.where(
        cv_test["route_id"].isin(swap_set),
        cv_test["y_pred_lgb"],
        cv_test["y_pred_chr"],
    )

    # Калибровка по train (не смотрим в fold 5)
    cv_train["y_pred_swap"] = np.where(
        cv_train["route_id"].isin(swap_set),
        cv_train["y_pred_lgb"],
        cv_train["y_pred_chr"],
    )
    cs = float(cv_train["y_true"].sum() /
               (cv_train["y_pred_swap"].clip(lower=0).sum() + 1e-9))

    y_cal = np.clip(cv_test["y_pred_swap"] * cs, 0, None)
    tot   = wape_rbias(cv_test["y_true"].values, y_cal.values)
    delta = tot - base_chr
    sign  = "✅ BETTER" if delta < 0 else "❌ WORSE"

    print(f"\n── {label:10s} │ swap={len(swap_set):>3} маршрутов │ calib={cs:.3f} ──")
    print(f"   Total={tot:.4f}  Δ={delta:+.4f}  {sign}")

# ============================================================
# 3. Weighted blend LGB+Chronos: grid search на train, тест на fold 5
# ============================================================
print("\n" + "="*65)
print("WEIGHTED BLEND: grid search на folds 1-4, тест на fold 5")
print("="*65)

best_w, best_tot_tr = None, 1e9
for w_lgb in np.arange(0.0, 1.01, 0.05):
    y_bl = cv_train["y_pred_lgb"] * w_lgb + cv_train["y_pred_chr"] * (1 - w_lgb)
    cs   = float(cv_train["y_true"].sum() / (y_bl.clip(lower=0).sum() + 1e-9))
    tot  = wape_rbias(cv_train["y_true"].values, np.clip(y_bl * cs, 0, None).values)
    if tot < best_tot_tr:
        best_tot_tr, best_w = tot, w_lgb

# Применяем на fold 5
y_bl_test = cv_test["y_pred_lgb"] * best_w + cv_test["y_pred_chr"] * (1 - best_w)
y_bl_train = cv_train["y_pred_lgb"] * best_w + cv_train["y_pred_chr"] * (1 - best_w)
cs_best = float(cv_train["y_true"].sum() / (y_bl_train.clip(lower=0).sum() + 1e-9))
y_bl_cal = np.clip(y_bl_test * cs_best, 0, None)
tot_blend = wape_rbias(cv_test["y_true"].values, y_bl_cal.values)

print(f"Оптимальный w_lgb={best_w:.2f}  (найден на folds 1-4)")
print(f"На fold 5: Total={tot_blend:.4f}  Δ={tot_blend-base_chr:+.4f}  "
      f"{'✅ BETTER' if tot_blend < base_chr else '❌ WORSE'}")

In [ ]:
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.model_selection import cross_val_score

lgb_raw = pd.read_csv("lgb_cv_raw.csv")[["route_id","fold","h","y_true","y_pred"]]\
            .rename(columns={"y_pred": "y_pred_lgb"})
chr_raw = pd.read_csv("chronos2_cv_raw.csv")[["route_id","fold","h","y_true","y_pred"]]\
            .rename(columns={"y_pred": "y_pred_chr"})
cv = chr_raw.merge(lgb_raw, on=["route_id","fold","h","y_true"], how="inner")

train_df = pd.read_parquet("train_solo_track.parquet")
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
train_df = train_df.sort_values(["route_id","timestamp"]).reset_index(drop=True)
TARGET_COL     = "target_1h"
TOTAL_VAL_PTS  = 40

def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    return float(np.abs(yp-yt).sum()/s + np.abs(yp.sum()/s - 1))


# ============================================================
# Строим фичи маршрута — из истории ДО начала фолда
# Задача: предсказать на каком маршруте LGB лучше Chronos
# ============================================================
def route_features(series: np.ndarray) -> dict:
    """Характеристики ряда из истории маршрута."""
    if len(series) < 16:
        return None
    s   = series.astype(float)
    s8  = s[-8:]
    s16 = s[-16:]
    s32 = s[-32:] if len(s) >= 32 else s
    s96 = s[-96:] if len(s) >= 96 else s

    # Тренд
    x = np.arange(len(s32))
    trend_coef = np.polyfit(x, s32, 1)[0] / (np.mean(np.abs(s32)) + 1e-9)

    # Волатильность
    cv_global = np.std(s96) / (np.mean(s96) + 1e-9)
    cv_recent = np.std(s8)  / (np.mean(s8)  + 1e-9)
    vol_ratio  = cv_recent / (cv_global + 1e-9)  # >1 = недавно нестабильно

    # Автокорреляция (lag 1, lag 48 = суточная)
    def autocorr(x, lag):
        if len(x) <= lag:
            return 0.0
        return float(np.corrcoef(x[:-lag], x[lag:])[0, 1])

    ac1  = autocorr(s96, 1)
    ac48 = autocorr(s96, 48) if len(s96) > 48 else 0.0

    # Структурный сдвиг: изменилось ли среднее недавно
    mean_old = np.mean(s[-32:-8]) if len(s) >= 32 else np.mean(s)
    mean_new = np.mean(s8)
    level_shift = (mean_new - mean_old) / (mean_old + 1e-9)

    # Сезонность: насколько регулярен суточный паттерн
    if len(s96) >= 96:
        day1 = s96[-96:-48]
        day2 = s96[-48:]
        season_corr = float(np.corrcoef(day1, day2)[0, 1]) if len(day1) == len(day2) else 0.0
    else:
        season_corr = 0.0

    # Ненулевые значения (спарсность)
    sparsity = float((s96 == 0).mean())

    # Масштаб
    log_mean = float(np.log1p(np.mean(np.abs(s96))))

    return {
        "trend_coef":    trend_coef,
        "cv_global":     cv_global,
        "cv_recent":     cv_recent,
        "vol_ratio":     vol_ratio,
        "ac1":           ac1,
        "ac48":          ac48,
        "level_shift":   level_shift,
        "season_corr":   season_corr,
        "sparsity":      sparsity,
        "log_mean":      log_mean,
        "len_series":    len(s),
    }


# ============================================================
# Собираем обучающую выборку для мета-модели
# На каждом фолде × маршрут:
#   X = фичи истории ДО фолда
#   y = wape_lgb - wape_chr на этом фолде
#      (< 0 = LGB лучше, > 0 = Chronos лучше)
# ============================================================
print("Строим мета-датасет...")
meta_rows = []
TOTAL_VAL_PTS = 40

for fold in range(1, 6):
    val_offset_end   = TOTAL_VAL_PTS - (fold - 1) * 8
    val_offset_start = val_offset_end - 8

    fold_cv = cv[cv["fold"] == fold]

    for route_id, grp_cv in fold_cv.groupby("route_id"):
        grp_tr = train_df[train_df["route_id"] == route_id]\
                    .sort_values("timestamp").reset_index(drop=True)

        if len(grp_tr) < TOTAL_VAL_PTS + 32:
            continue

        # История ДО фолда — честно, без лика
        hist = grp_tr.iloc[:-val_offset_end][TARGET_COL].values

        feats = route_features(hist)
        if feats is None:
            continue

        # Таргет: wape_lgb - wape_chr на этом фолде для этого маршрута
        s = grp_cv["y_true"].sum() + 1e-9
        wape_lgb = grp_cv["y_pred_lgb"].sub(grp_cv["y_true"]).abs().sum() / s
        wape_chr = grp_cv["y_pred_chr"].sub(grp_cv["y_true"]).abs().sum() / s
        target   = wape_lgb - wape_chr   # <0: LGB лучше, >0: Chronos лучше

        feats["route_id"] = route_id
        feats["fold"]     = fold
        feats["target"]   = target
        feats["wape_lgb"] = wape_lgb
        feats["wape_chr"] = wape_chr
        meta_rows.append(feats)

meta_df = pd.DataFrame(meta_rows)
print(f"Мета-датасет: {meta_df.shape}")
print(f"target>0 (Chronos лучше): {(meta_df['target']>0).sum()}  "
      f"target<0 (LGB лучше): {(meta_df['target']<0).sum()}")

FEAT_COLS = ["trend_coef","cv_global","cv_recent","vol_ratio",
             "ac1","ac48","level_shift","season_corr","sparsity",
             "log_mean","len_series"]

# ============================================================
# Бэктест мета-модели:
# Train: фолды 1-4  →  Test: фолд 5
# ============================================================
meta_train = meta_df[meta_df["fold"].isin([1,2,3,4])]
meta_test  = meta_df[meta_df["fold"] == 5]

X_train = meta_train[FEAT_COLS].values
y_train = meta_train["target"].values
X_test  = meta_test[FEAT_COLS].values
y_test  = meta_test["target"].values

# Бинарная классификация: предсказываем кто лучше
from lightgbm import LGBMClassifier
clf = LGBMClassifier(n_estimators=100, learning_rate=0.05,
                     max_depth=4, min_child_samples=20,
                     verbose=-1, random_state=42)
clf.fit(X_train, (y_train > 0).astype(int))   # 1 = Chronos лучше, 0 = LGB лучше

proba = clf.predict_proba(X_test)[:, 1]       # P(Chronos лучше)
pred_winner = (proba > 0.5).astype(int)
true_winner = (y_test > 0).astype(int)

acc = (pred_winner == true_winner).mean()
print(f"\n── Мета-модель accuracy на fold 5: {acc:.3f}  (random=0.50) ──")

# Важность фичей
fi = pd.Series(clf.feature_importances_, index=FEAT_COLS).sort_values(ascending=False)
print("\nВажность фичей:")
print(fi.to_string())

# ============================================================
# Регрессионная мета-модель (не бинарная — непрерывный сигнал)
# ============================================================
from lightgbm import LGBMRegressor
reg = LGBMRegressor(n_estimators=100, learning_rate=0.05,
                    max_depth=4, min_child_samples=20,
                    verbose=-1, random_state=42)
reg.fit(X_train, y_train)
pred_diff = reg.predict(X_test)   # предсказанный wape_lgb - wape_chr

corr = np.corrcoef(pred_diff, y_test)[0,1]
print(f"\nКорреляция предсказанного diff с реальным: {corr:.3f}")

# ============================================================
# Применяем мета-модель для выбора модели на fold 5
# ============================================================
cv_test_merged = cv[cv["fold"] == 5].copy()
cv_train_all   = cv[cv["fold"].isin([1,2,3,4])].copy()

# Маршруты где мета-модель говорит "LGB лучше" (pred_diff < 0 → P < threshold)
THRESHOLDS = [0.3, 0.4, 0.5, 0.6]
print("\n── Swap на fold 5 по мета-модели (разные пороги) ──")

route_proba = meta_test[["route_id"]].copy()
route_proba["proba_chr_better"] = proba
route_proba["pred_diff"]        = pred_diff

for thr in THRESHOLDS:
    lgb_routes = set(route_proba[route_proba["proba_chr_better"] < thr]["route_id"])

    cv_test_merged["y_pred_meta"] = np.where(
        cv_test_merged["route_id"].isin(lgb_routes),
        cv_test_merged["y_pred_lgb"],
        cv_test_merged["y_pred_chr"],
    )
    cv_train_all["y_pred_meta"] = np.where(
        cv_train_all["route_id"].isin(lgb_routes),
        cv_train_all["y_pred_lgb"],
        cv_train_all["y_pred_chr"],
    )
    cs = float(cv_train_all["y_true"].sum() /
               (cv_train_all["y_pred_meta"].clip(lower=0).sum() + 1e-9))

    y_cal = np.clip(cv_test_merged["y_pred_meta"] * cs, 0, None)

    yt = cv_test_merged["y_true"].values
    yp = y_cal.values
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp-yt).sum() / s
    rbias = abs(yp.sum() / s - 1)
    tot   = wape + rbias

    base_chr = wape_rbias(cv_test_merged["y_true"], cv_test_merged["y_pred_chr"])
    delta = tot - base_chr
    print(f"  thr={thr}  swap={len(lgb_routes):>4}  calib={cs:.3f}  "
          f"Total={tot:.4f}  Δ={delta:+.4f}  "
          f"{'✅' if delta < 0 else '❌'}")

meta_df.to_csv("meta_model_dataset.csv", index=False)
route_proba.to_csv("meta_model_fold5_proba.csv", index=False)
print("\n✅ meta_model_dataset.csv")
print("✅ meta_model_fold5_proba.csv")

In [ ]:
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier, LGBMRegressor

# ============================================================
# 0. Утилиты
# ============================================================
def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    return float(np.abs(yp - yt).sum() / s + np.abs(yp.sum() / s - 1))

# ============================================================
# 1. Загружаем данные
# ============================================================
lgb_raw = pd.read_csv("lgb_cv_raw.csv")[["route_id","fold","h","y_true","y_pred"]]\
            .rename(columns={"y_pred": "y_pred_lgb"})
chr_raw = pd.read_csv("chronos2_cv_raw.csv")[["route_id","fold","h","y_true","y_pred"]]\
            .rename(columns={"y_pred": "y_pred_chr"})
cv = chr_raw.merge(lgb_raw, on=["route_id","fold","h","y_true"], how="inner")

train_df = pd.read_parquet("train_solo_track.parquet")
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
train_df = train_df.sort_values(["route_id","timestamp"]).reset_index(drop=True)
TARGET_COL    = "target_1h"
TOTAL_VAL_PTS = 40

print(f"cv: {cv.shape}  train_df: {train_df.shape}")

# ============================================================
# 2. Функция фичей маршрута из истории
# ============================================================
def route_features(series: np.ndarray) -> dict:
    if len(series) < 16:
        return None
    s   = series.astype(float)
    s8  = s[-8:]
    s32 = s[-32:] if len(s) >= 32 else s
    s96 = s[-96:] if len(s) >= 96 else s

    x = np.arange(len(s32))
    trend_coef = np.polyfit(x, s32, 1)[0] / (np.mean(np.abs(s32)) + 1e-9)

    cv_global  = np.std(s96) / (np.mean(s96) + 1e-9)
    cv_recent  = np.std(s8)  / (np.mean(s8)  + 1e-9)
    vol_ratio  = cv_recent   / (cv_global + 1e-9)

    def autocorr(x, lag):
        if len(x) <= lag: return 0.0
        return float(np.corrcoef(x[:-lag], x[lag:])[0, 1])

    ac1  = autocorr(s96, 1)
    ac48 = autocorr(s96, 48) if len(s96) > 48 else 0.0

    mean_old    = np.mean(s[-32:-8]) if len(s) >= 32 else np.mean(s)
    mean_new    = np.mean(s8)
    level_shift = (mean_new - mean_old) / (mean_old + 1e-9)

    if len(s96) >= 96:
        day1 = s96[-96:-48]
        day2 = s96[-48:]
        season_corr = float(np.corrcoef(day1, day2)[0, 1]) if len(day1) == len(day2) else 0.0
    else:
        season_corr = 0.0

    sparsity = float((s96 == 0).mean())
    log_mean = float(np.log1p(np.mean(np.abs(s96))))

    return {
        "trend_coef":  trend_coef,
        "cv_global":   cv_global,
        "cv_recent":   cv_recent,
        "vol_ratio":   vol_ratio,
        "ac1":         ac1,
        "ac48":        ac48,
        "level_shift": level_shift,
        "season_corr": season_corr,
        "sparsity":    sparsity,
        "log_mean":    log_mean,
        "len_series":  len(s),
    }

# ============================================================
# 3. Строим мета-датасет
# ============================================================
print("Строим мета-датасет...")
meta_rows = []

for fold in range(1, 6):
    val_offset_end   = TOTAL_VAL_PTS - (fold - 1) * 8
    val_offset_start = val_offset_end - 8
    fold_cv = cv[cv["fold"] == fold]

    for route_id, grp_cv in fold_cv.groupby("route_id"):
        grp_tr = train_df[train_df["route_id"] == route_id]\
                    .sort_values("timestamp").reset_index(drop=True)

        if len(grp_tr) < TOTAL_VAL_PTS + 32:
            continue

        hist  = grp_tr.iloc[:-val_offset_end][TARGET_COL].values
        feats = route_features(hist)
        if feats is None:
            continue

        s        = grp_cv["y_true"].sum() + 1e-9
        wape_lgb = grp_cv["y_pred_lgb"].sub(grp_cv["y_true"]).abs().sum() / s
        wape_chr = grp_cv["y_pred_chr"].sub(grp_cv["y_true"]).abs().sum() / s
        target   = wape_lgb - wape_chr   # <0: LGB лучше, >0: Chronos лучше

        # Объём маршрута на этом фолде
        feats["route_id"]  = route_id
        feats["fold"]      = fold
        feats["target"]    = target
        feats["wape_lgb"]  = wape_lgb
        feats["wape_chr"]  = wape_chr
        feats["vol"]       = float(grp_cv["y_true"].sum())
        meta_rows.append(feats)

meta_df = pd.DataFrame(meta_rows)
print(f"Мета-датасет: {meta_df.shape}")
print(f"Chronos лучше (target>0): {(meta_df['target']>0).sum()}  "
      f"LGB лучше (target<0): {(meta_df['target']<0).sum()}")

# ============================================================
# 4. Train/test split: фолды 1-4 → fold 5
# ============================================================
FEAT_COLS = ["trend_coef","cv_global","cv_recent","vol_ratio",
             "ac1","ac48","level_shift","season_corr",
             "sparsity","log_mean","len_series"]

meta_train = meta_df[meta_df["fold"].isin([1,2,3,4])]
meta_test  = meta_df[meta_df["fold"] == 5]

X_train = meta_train[FEAT_COLS].values
y_train = meta_train["target"].values
X_test  = meta_test[FEAT_COLS].values
y_test  = meta_test["target"].values

# ============================================================
# 5. Классификатор
# ============================================================
clf = LGBMClassifier(n_estimators=100, learning_rate=0.05,
                     max_depth=4, min_child_samples=20,
                     verbose=-1, random_state=42)
clf.fit(X_train, (y_train > 0).astype(int))

proba      = clf.predict_proba(X_test)[:, 1]   # P(Chronos лучше)
pred_class = (proba > 0.5).astype(int)
true_class = (y_test > 0).astype(int)
acc = (pred_class == true_class).mean()
print(f"\n── Классификатор accuracy на fold 5: {acc:.3f}  (random=0.50) ──")

fi = pd.Series(clf.feature_importances_, index=FEAT_COLS).sort_values(ascending=False)
print("\nВажность фичей:")
print(fi.to_string())

# ============================================================
# 6. Регрессор + честная корреляция
# ============================================================
reg = LGBMRegressor(n_estimators=100, learning_rate=0.05,
                    max_depth=4, min_child_samples=20,
                    verbose=-1, random_state=42)
reg.fit(X_train, y_train)

pred_diff_train = reg.predict(X_train)
pred_diff_test  = reg.predict(X_test)

corr_train = float(np.corrcoef(pred_diff_train, y_train)[0, 1])
corr_test  = float(np.corrcoef(pred_diff_test,  y_test )[0, 1])
print(f"\nКорреляция предсказанного diff — TRAIN: {corr_train:.3f}  TEST: {corr_test:.3f}")

# ============================================================
# 7. Volume-weighted swap на fold 5
# ============================================================
cv_test  = cv[cv["fold"] == 5].copy()
cv_train = cv[cv["fold"].isin([1,2,3,4])].copy()

# Базовые метрики
base_chr = wape_rbias(cv_test["y_true"].values, cv_test["y_pred_chr"].values)
base_lgb = wape_rbias(cv_test["y_true"].values, cv_test["y_pred_lgb"].values)
print(f"\nBaseline Chronos fold5: {base_chr:.4f}")
print(f"Baseline LGB     fold5: {base_lgb:.4f}")

# Объём каждого маршрута на fold 5
route_vol = cv_test.groupby("route_id")["y_true"].sum()

route_proba = meta_test[["route_id"]].copy()
route_proba["proba_chr_better"] = proba
route_proba["pred_diff"]        = pred_diff_test
route_proba["volume"]           = route_proba["route_id"].map(route_vol)

# expected_gain > 0 → ожидаем что LGB лучше, и это важный маршрут
route_proba["expected_gain"] = (0.5 - route_proba["proba_chr_better"]) \
                                * route_proba["volume"]

print("\n── Threshold-based swap ──")
for thr in [0.3, 0.4, 0.5, 0.6, 0.7]:
    lgb_routes = set(route_proba[route_proba["proba_chr_better"] < thr]["route_id"])

    cv_test["y_pred_swap"] = np.where(
        cv_test["route_id"].isin(lgb_routes),
        cv_test["y_pred_lgb"], cv_test["y_pred_chr"]
    )
    cv_train["y_pred_swap"] = np.where(
        cv_train["route_id"].isin(lgb_routes),
        cv_train["y_pred_lgb"], cv_train["y_pred_chr"]
    )
    cs    = float(cv_train["y_true"].sum() /
                  (cv_train["y_pred_swap"].clip(lower=0).sum() + 1e-9))
    y_cal = np.clip(cv_test["y_pred_swap"] * cs, 0, None)
    tot   = wape_rbias(cv_test["y_true"].values, y_cal.values)
    print(f"  thr={thr}  swap={len(lgb_routes):>4}  calib={cs:.3f}  "
          f"Total={tot:.4f}  Δ={tot-base_chr:+.4f}  "
          f"{'✅' if tot < base_chr else '❌'}")

print("\n── Volume-weighted top-N swap ──")
for top_n in [5, 10, 20, 30, 50, 100, 200]:
    lgb_routes = set(
        route_proba.nlargest(top_n, "expected_gain")["route_id"]
    )
    cv_test["y_pred_swap"] = np.where(
        cv_test["route_id"].isin(lgb_routes),
        cv_test["y_pred_lgb"], cv_test["y_pred_chr"]
    )
    cv_train["y_pred_swap"] = np.where(
        cv_train["route_id"].isin(lgb_routes),
        cv_train["y_pred_lgb"], cv_train["y_pred_chr"]
    )
    cs    = float(cv_train["y_true"].sum() /
                  (cv_train["y_pred_swap"].clip(lower=0).sum() + 1e-9))
    y_cal = np.clip(cv_test["y_pred_swap"] * cs, 0, None)
    tot   = wape_rbias(cv_test["y_true"].values, y_cal.values)
    print(f"  top_n={top_n:>4}  calib={cs:.3f}  "
          f"Total={tot:.4f}  Δ={tot-base_chr:+.4f}  "
          f"{'✅' if tot < base_chr else '❌'}")

# ============================================================
# 8. Weighted blend по предсказанному diff
# ============================================================
print("\n── Soft blend: вес LGB = sigmoid(-pred_diff * k) ──")
# pred_diff < 0 → LGB лучше → хотим больший вес на LGB
from scipy.special import expit  # sigmoid

for k in [1, 2, 5, 10]:
    w_lgb = expit(-pred_diff_test * k)   # shape: (n_routes,)

    # Маппим веса на строки cv_test
    route_w = pd.Series(w_lgb, index=meta_test["route_id"].values)
    cv_test["w_lgb"] = cv_test["route_id"].map(route_w).fillna(0.5)

    cv_test["y_pred_blend"] = (cv_test["w_lgb"] * cv_test["y_pred_lgb"] +
                               (1 - cv_test["w_lgb"]) * cv_test["y_pred_chr"])

    # Калибровка на train с теми же весами
    route_w_tr = pd.Series(expit(-pred_diff_train * k),
                            index=meta_train["route_id"].values)
    cv_train["w_lgb"] = cv_train["route_id"].map(route_w_tr).fillna(0.5)
    cv_train["y_pred_blend"] = (cv_train["w_lgb"] * cv_train["y_pred_lgb"] +
                                (1 - cv_train["w_lgb"]) * cv_train["y_pred_chr"])

    cs    = float(cv_train["y_true"].sum() /
                  (cv_train["y_pred_blend"].clip(lower=0).sum() + 1e-9))
    y_cal = np.clip(cv_test["y_pred_blend"] * cs, 0, None)
    tot   = wape_rbias(cv_test["y_true"].values, y_cal.values)
    print(f"  k={k:>2}  calib={cs:.3f}  "
          f"Total={tot:.4f}  Δ={tot-base_chr:+.4f}  "
          f"{'✅' if tot < base_chr else '❌'}")

# ============================================================
# 9. Сохраняем
# ============================================================
meta_df.to_csv("meta_model_dataset.csv", index=False)
route_proba.to_csv("meta_model_fold5_proba.csv", index=False)
print("\n✅ meta_model_dataset.csv")
print("✅ meta_model_fold5_proba.csv")

In [ ]:
# ============================================================
# Chronos-2 — LoRA Fine-tuning + Rolling CV (5 folds × 8 steps)
# ============================================================
import warnings
import numpy as np
import pandas as pd
import torch
from chronos import Chronos2Pipeline
from peft import LoraConfig
from tqdm import tqdm
import os

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None


def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias


# -------------------- Config --------------------
TRAIN_PATH       = "train_solo_track.parquet"
TEST_PATH        = "test_solo_track.parquet"
TARGET_COL       = "target_1h"
FORECAST_STEPS   = 8
N_FOLDS          = 5
TOTAL_VAL_PTS    = N_FOLDS * FORECAST_STEPS   # 40
CONTEXT_LEN      = 2048
FINETUNED_PATH   = "./chronos2_lora_finetuned"

NUM_STEPS        = 1500
BATCH_SIZE       = 32
LR               = 1e-4
LOGGING_STEPS    = 100

os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"]  = "1"
os.environ["HF_HUB_OFFLINE"]       = "1"


# -------------------- Device --------------------
if torch.backends.mps.is_available():
    DEVICE = "mps"
    print("Apple Silicon MPS ✅")
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(f"Device: {DEVICE}\n")


# -------------------- Data --------------------
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

status_cols = sorted([c for c in train_df.columns if c.startswith("status_")])
print(f"Train: {train_df.shape}  Test: {test_df.shape}")
print(f"Status cols: {status_cols}\n")


# ============================================================
# STEP 1 — BASE MODEL
# ============================================================
MODEL_ID = "amazon/chronos-2"
print(f"Loading {MODEL_ID}...")
pipeline = Chronos2Pipeline.from_pretrained(
    MODEL_ID,
    device_map=DEVICE,
    torch_dtype=torch.bfloat16,
    local_files_only=True,
)
print("Model loaded!\n")


# -------------------- Config (добавь) --------------------
N_RECENT_WINDOWS = 20   # окон с каждого маршрута
# Каждое окно = ровно CONTEXT_LEN точек контекста + FORECAST_STEPS предикта
# → Chronos не может взять "случайный" подокна внутри — окно единственное
WINDOW_SIZE = CONTEXT_LEN + FORECAST_STEPS   # 2048 + 8 = 2056


def prepare_finetune_inputs(train_df: pd.DataFrame) -> list:
    inputs = []
    for route_id, grp in tqdm(train_df.groupby("route_id"), desc="Prep FT data"):
        grp = grp.sort_values("timestamp").reset_index(drop=True)

        # Отрезаем CV-зону (40 последних точек)
        hist = grp.iloc[:-TOTAL_VAL_PTS] if len(grp) > TOTAL_VAL_PTS + WINDOW_SIZE else grp
        if len(hist) < WINDOW_SIZE:
            continue

        df = (
            hist.set_index("timestamp")[[TARGET_COL] + status_cols]
            .asfreq("30min")
            .interpolate(method="time")
            .bfill().ffill()
        )
        target_arr = df[TARGET_COL].values
        cov_arrs   = {col: df[col].values for col in status_cols}

        # Создаём N_RECENT_WINDOWS окон, шагая назад от конца истории
        for i in range(N_RECENT_WINDOWS):
            end_idx   = len(target_arr) - i * FORECAST_STEPS
            start_idx = end_idx - WINDOW_SIZE
            if start_idx < 0:
                break   # история слишком короткая для этого окна

            inputs.append({
                "target": target_arr[start_idx:end_idx],
                "past_covariates":   {col: cov_arrs[col][start_idx:end_idx]
                                      for col in status_cols},
                "future_covariates": {col: None for col in status_cols},
            })

    return inputs


def prepare_context_df(route_df: pd.DataFrame, route_id) -> pd.DataFrame:
    """Готовит DataFrame для predict_df()."""
    df = (
        route_df.set_index("timestamp")[[TARGET_COL] + status_cols]
        .asfreq("30min")
        .interpolate(method="time")
        .bfill().ffill()
        .tail(CONTEXT_LEN)
        .reset_index()
    )
    df["id"] = route_id
    df = df.rename(columns={TARGET_COL: "target"})
    return df


def run_inference(ctx_pipeline, context_full: pd.DataFrame) -> pd.DataFrame:
    return ctx_pipeline.predict_df(
        context_full,
        prediction_length=FORECAST_STEPS,
        quantile_levels=[0.1, 0.5, 0.9],
        id_column="id",
        timestamp_column="timestamp",
        target="target",
        cross_learning=False,
    )


def get_pred_col(pred_df: pd.DataFrame) -> str:
    for candidate in ["mean", "0.5", "q0.5", "median"]:
        if candidate in pred_df.columns:
            return candidate
    num_cols = [c for c in pred_df.columns
                if c not in ("id", "timestamp") and pd.api.types.is_numeric_dtype(pred_df[c])]
    return num_cols[0]


def build_preds_dict(pred_df: pd.DataFrame, pred_col: str) -> dict:
    result = {}
    for route_id, grp in pred_df.groupby("id"):
        result[route_id] = np.clip(grp.sort_values("timestamp")[pred_col].values, 0, None)
    return result


# ============================================================
# STEP 2 — LoRA FINE-TUNING
# Берём историю без последних TOTAL_VAL_PTS точек (зона CV)
# ============================================================
print("=" * 65)
print("STEP 2 — LoRA Fine-Tuning")
print("=" * 65)

train_inputs = prepare_finetune_inputs(train_df)
print(f"Fine-tune windows: {len(train_inputs)}")

lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q", "v", "wi_0", "wi_1"],
    bias="none",
    task_type="SEQ_2_SEQ_LM",
)

print(f"Starting LoRA fine-tuning: steps={NUM_STEPS}, batch={BATCH_SIZE}, lr={LR}\n")

# fit() возвращает НОВЫЙ pipeline!
finetuned_pipeline = pipeline.fit(
    inputs=train_inputs,
    prediction_length=FORECAST_STEPS,
    finetune_mode="lora",          # ← ключевой аргумент
    lora_config=lora_cfg,
    num_steps=NUM_STEPS,
    learning_rate=LR,
    batch_size=BATCH_SIZE,
    logging_steps=LOGGING_STEPS,
)

finetuned_pipeline.save_pretrained(FINETUNED_PATH)
print(f"\n✅ Fine-tuned model saved → {FINETUNED_PATH}")

In [ ]:
# ============================================================
# STEP 3 — ROLLING CROSS-VALIDATION (на дообученном pipeline)
# ============================================================
print("\n" + "=" * 65)
print(f"STEP 3 — ROLLING CV  ({N_FOLDS} folds × {FORECAST_STEPS} steps)")
print("=" * 65)

raw_rows = []
pred_col = None

for fold in range(N_FOLDS):
    val_offset_end   = TOTAL_VAL_PTS - fold * FORECAST_STEPS
    val_offset_start = val_offset_end - FORECAST_STEPS

    context_dfs    = []
    y_trues_fold   = []
    route_ids_fold = []
    val_timestamps = []

    for route_id, grp in train_df.groupby("route_id"):
        grp = grp.sort_values("timestamp").reset_index(drop=True)
        if len(grp) < TOTAL_VAL_PTS + 32:
            continue
        hist_part = grp.iloc[:-val_offset_end]
        val_part  = (grp.iloc[-val_offset_end:-val_offset_start]
                     if val_offset_start > 0
                     else grp.iloc[-val_offset_end:])
        if len(hist_part) < 32 or len(val_part) == 0:
            continue
        context_dfs.append(prepare_context_df(hist_part, route_id))
        y_trues_fold.append(val_part[TARGET_COL].values[:FORECAST_STEPS])
        route_ids_fold.append(route_id)
        val_timestamps.append(val_part["timestamp"].values[:FORECAST_STEPS])

    context_full = pd.concat(context_dfs, ignore_index=True)

    print(f"\nFold {fold+1}/{N_FOLDS}  "
          f"(val: -{val_offset_end}..{'-'+str(val_offset_start) if val_offset_start else 'end'})  "
          f"routes={len(route_ids_fold)}")

    # используем finetuned_pipeline
    pred_df_fold = run_inference(finetuned_pipeline, context_full)

    if pred_col is None:
        pred_col = get_pred_col(pred_df_fold)
        print(f"  Point forecast column: '{pred_col}'")

    preds = build_preds_dict(pred_df_fold, pred_col)

    yt_fold, yp_fold = [], []
    for i, route_id in enumerate(route_ids_fold):
        yp = preds.get(route_id, np.zeros(len(y_trues_fold[i])))
        for h_idx in range(len(y_trues_fold[i])):
            y_true = float(y_trues_fold[i][h_idx])
            y_pred = float(yp[h_idx]) if h_idx < len(yp) else 0.0
            ae  = abs(y_pred - y_true)
            ape = ae / (y_true + 1e-9)
            err = y_pred - y_true
            raw_rows.append({
                "fold": fold + 1, "route_id": route_id,
                "timestamp": val_timestamps[i][h_idx],
                "h": h_idx + 1, "y_true": y_true,
                "y_pred": y_pred, "ae": ae, "ape": ape, "err": err,
            })
            yt_fold.append(y_true)
            yp_fold.append(y_pred)

    yt_fold = np.array(yt_fold)
    yp_fold = np.array(yp_fold)
    tot, wape, rb = wape_rbias(yt_fold, yp_fold)
    print(f"  LoRA  WAPE={wape:.4f}  |RBias|={rb:.4f}  Total={tot:.4f}")


# ============================================================
# АГРЕГАТЫ
# ============================================================
raw_cv_df = pd.DataFrame(raw_rows)

agg_fold = raw_cv_df.groupby("fold").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
    "n":     len(df),
})).reset_index()

agg_h = raw_cv_df.groupby("h").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
})).reset_index()

agg_route = raw_cv_df.groupby("route_id").agg(
    n_points=("y_true","count"), mae=("ae","mean"),
    median_ae=("ae","median"), std_ae=("ae","std"),
    mape=("ape","mean"), median_ape=("ape","median"),
    mean_err=("err","mean"), median_err=("err","median"),
    mean_true=("y_true","mean"), mean_pred=("y_pred","mean"),
).reset_index()
agg_route["calib_scale"] = (
    raw_cv_df.groupby("route_id")["y_true"].sum().values /
    (raw_cv_df.groupby("route_id")["y_pred"].sum().values + 1e-9)
)
agg_route["wape"] = (
    raw_cv_df.groupby("route_id")["ae"].sum().values /
    (raw_cv_df.groupby("route_id")["y_true"].sum().values + 1e-9)
)

y_true_all = raw_cv_df["y_true"].values
y_pred_all = raw_cv_df["y_pred"].values
total_raw, wape_raw, rbias_raw = wape_rbias(y_true_all, y_pred_all)

calib_scale = float(y_true_all.sum() / (y_pred_all.sum() + 1e-9))
y_pred_cal  = np.clip(y_pred_all * calib_scale, 0, None)
total_cal, wape_cal, rbias_cal = wape_rbias(y_true_all, y_pred_cal)

print("\n" + "=" * 65)
print("SUMMARY (LoRA fine-tuned)")
print("=" * 65)
print("\n── По фолдам ──")
print(agg_fold.to_string(index=False))
print("\n── По шагам h ──")
print(agg_h.to_string(index=False))
print(f"\n{'Global RAW':25s}  WAPE={wape_raw:.4f}  |RBias|={rbias_raw:.4f}  Total={total_raw:.4f}")
print(f"{'Global CALIBRATED':25s}  WAPE={wape_cal:.4f}  |RBias|={rbias_cal:.4f}  Total={total_cal:.4f}")
print(f"  calib_scale = {calib_scale:.4f}")
print(f"\nCalibration {'HELPS ✅' if total_cal < total_raw else 'HURTS ❌'} "
      f"(Δ = {total_cal - total_raw:+.4f})")

raw_cv_df.to_csv("chronos2_lora_cv_raw.csv", index=False)
agg_route.to_csv("chronos2_lora_cv_agg_route.csv", index=False)
agg_fold.to_csv("chronos2_lora_cv_agg_fold.csv", index=False)
agg_h.to_csv("chronos2_lora_cv_agg_h.csv", index=False)
print("\n✅ chronos2_lora_cv_raw.csv  ✅ chronos2_lora_cv_agg_route.csv")


# ============================================================
# STEP 4 — TEST PREDICTION
# ============================================================
print("\n" + "=" * 65)
print("STEP 4 — TEST PREDICTION")
print("=" * 65)

test_context_dfs = []
test_route_ids   = []

for route_id, grp in tqdm(test_df.groupby("route_id"), desc="Collect test"):
    route_train = train_df[train_df["route_id"] == route_id].sort_values("timestamp")
    if len(route_train) < 32:
        continue
    test_context_dfs.append(prepare_context_df(route_train, route_id))
    test_route_ids.append(route_id)

test_context_full = pd.concat(test_context_dfs, ignore_index=True)
print(f"Routes: {len(test_route_ids)}  Running inference...")

test_pred_df = run_inference(finetuned_pipeline, test_context_full)
test_preds   = build_preds_dict(test_pred_df, pred_col)

predictions_raw = {}
for route_id in test_route_ids:
    route_test = test_df[test_df["route_id"] == route_id].sort_values("timestamp")
    preds = test_preds.get(route_id, np.zeros(FORECAST_STEPS))
    for j, (_, row) in enumerate(route_test.iterrows()):
        pred = float(preds[j]) if j < len(preds) else float(preds[-1])
        predictions_raw[row["id"]] = max(0.0, pred)

submission_raw = (
    pd.DataFrame(list(predictions_raw.items()), columns=["id", "y_pred"])
    .sort_values("id").reset_index(drop=True)
)
submission_cal = submission_raw.copy()
submission_cal["y_pred"] = np.clip(submission_cal["y_pred"] * calib_scale, 0, None)

submission_raw.to_csv("submission_chronos2_lora_raw.csv", index=False)
submission_cal.to_csv("submission_chronos2_lora_calibrated.csv", index=False)
print(f"\n✅ submission_chronos2_lora_raw.csv")
print(f"✅ submission_chronos2_lora_calibrated.csv")
print(f"\n── Raw ──\n{submission_raw['y_pred'].describe().round(2)}")
print(f"\n── Calibrated (scale={calib_scale:.4f}) ──\n{submission_cal['y_pred'].describe().round(2)}")

In [ ]:
# ============================================================
# Chronos-2 — Rolling CV (5 folds × 8 steps)
# + per-route статистика для блендинга (как в LGB)
# ============================================================
import warnings
import numpy as np
import pandas as pd
import torch
from chronos import Chronos2Pipeline
from tqdm import tqdm
import os

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None


def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias


# -------------------- Config --------------------
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8
N_FOLDS        = 5
TOTAL_VAL_PTS  = N_FOLDS * FORECAST_STEPS   # 40
CONTEXT_LEN    = 2048

os.environ["TRANSFORMERS_OFFLINE"] = "1"   # трансформеры не лезут в сеть
os.environ["HF_DATASETS_OFFLINE"] = "1"    # датасеты тоже
os.environ["HF_HUB_OFFLINE"] = "1"         # huggingface hub тоже

# -------------------- Device --------------------
if torch.backends.mps.is_available():
    DEVICE = "mps"
    print("Apple Silicon MPS ✅")
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(f"Device: {DEVICE}\n")

# -------------------- Load model --------------------
MODEL_ID = "amazon/chronos-2"
print(f"Loading {MODEL_ID}...")
pipeline = Chronos2Pipeline.from_pretrained(
    MODEL_ID,
    device_map=DEVICE,
    torch_dtype=torch.bfloat16,
    local_files_only=True,
)
print("Model loaded!\n")

# -------------------- Data --------------------
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

status_cols = sorted([c for c in train_df.columns if c.startswith("status_")])
print(f"Train: {train_df.shape}  Test: {test_df.shape}")
print(f"Status cols: {status_cols}\n")


# -------------------- Helpers --------------------
def prepare_context_df(route_df: pd.DataFrame, route_id) -> pd.DataFrame:
    df = (
        route_df.set_index("timestamp")[[TARGET_COL] + status_cols]
        .asfreq("30min")
        .interpolate(method="time")
        .bfill().ffill()
        .tail(CONTEXT_LEN)
        .reset_index()
    )
    df["id"] = route_id
    df = df.rename(columns={TARGET_COL: "target"})
    return df


def run_inference(context_full: pd.DataFrame, cross_learning: bool = False) -> pd.DataFrame:
    return pipeline.predict_df(
        context_full,
        prediction_length=FORECAST_STEPS,
        quantile_levels=[0.1, 0.5, 0.9],
        id_column="id",
        timestamp_column="timestamp",
        target="target",
        cross_learning=cross_learning,
    )


def get_pred_col(pred_df: pd.DataFrame) -> str:
    for candidate in ["mean", "0.5", "q0.5", "median"]:
        if candidate in pred_df.columns:
            return candidate
    num_cols = [c for c in pred_df.columns
                if c not in ("id", "timestamp") and pd.api.types.is_numeric_dtype(pred_df[c])]
    return num_cols[0]


# ── Изменение 1: build_preds_dict возвращает оба квантиля ──
def build_preds_dict(pred_df: pd.DataFrame) -> dict:
    # Находим колонки 0.5 и 0.1
    col_50 = next(c for c in ["0.5", "q0.5", "mean", "median"]
                  if c in pred_df.columns)
    col_10 = next((c for c in ["0.1", "q0.1"] if c in pred_df.columns), None)

    result = {}
    for route_id, grp in pred_df.groupby("id"):
        grp_s = grp.sort_values("timestamp")
        p50 = np.clip(grp_s[col_50].values, 0, None)
        p10 = np.clip(grp_s[col_10].values, 0, None) if col_10 else p50
        result[route_id] = {"p50": p50, "p10": p10}
    return result


# ============================================================
# ROLLING CROSS-VALIDATION
# fold 1: история=[:-40]  val=[-40:-32]
# fold 2: история=[:-32]  val=[-32:-24]
# fold 3: история=[:-24]  val=[-24:-16]
# fold 4: история=[:-16]  val=[-16: -8]
# fold 5: история=[: -8]  val=[ -8:end]
# ============================================================
print("=" * 65)
print(f"ROLLING CV  ({N_FOLDS} folds × {FORECAST_STEPS} steps = {TOTAL_VAL_PTS} points)")
print("=" * 65)

raw_rows   = []   # все точки всех фолдов — для per-route аналитики
pred_col   = None

for fold in range(N_FOLDS):
    val_offset_end   = TOTAL_VAL_PTS - fold * FORECAST_STEPS   # 40, 32, 24, 16, 8
    val_offset_start = val_offset_end - FORECAST_STEPS          # 32, 24, 16,  8, 0

    context_dfs    = []
    y_trues_fold   = []
    route_ids_fold = []
    val_timestamps = []   # чтобы потом разложить по строкам

    for route_id, grp in train_df.groupby("route_id"):
        grp = grp.sort_values("timestamp").reset_index(drop=True)
        if len(grp) < TOTAL_VAL_PTS + 32:
            continue

        hist_part = grp.iloc[:-val_offset_end]
        val_part  = (grp.iloc[-val_offset_end:-val_offset_start]
                     if val_offset_start > 0
                     else grp.iloc[-val_offset_end:])

        if len(hist_part) < 32 or len(val_part) == 0:
            continue

        context_dfs.append(prepare_context_df(hist_part, route_id))
        y_trues_fold.append(val_part[TARGET_COL].values[:FORECAST_STEPS])
        route_ids_fold.append(route_id)
        val_timestamps.append(val_part["timestamp"].values[:FORECAST_STEPS])

    context_full = pd.concat(context_dfs, ignore_index=True)

    print(f"\nFold {fold+1}/{N_FOLDS}  "
          f"(val: -{val_offset_end}..{'-'+str(val_offset_start) if val_offset_start else 'end'})  "
          f"routes={len(route_ids_fold)}")

    pred_df_fold = run_inference(context_full)

    if pred_col is None:
        pred_col = get_pred_col(pred_df_fold)
        print(f"  Point forecast column: '{pred_col}'")

    # ── Изменение 2: в цикле raw_rows добавляем y_pred_q10 ──
    # Заменяем блок сборки raw_rows:
    preds = build_preds_dict(pred_df_fold)  # теперь без pred_col

    yt_fold, yp_fold = [], []
    for i, route_id in enumerate(route_ids_fold):
        p = preds.get(route_id, {"p50": np.zeros(FORECAST_STEPS),
                                  "p10": np.zeros(FORECAST_STEPS)})
        for h_idx in range(len(y_trues_fold[i])):
            y_true  = float(y_trues_fold[i][h_idx])
            y_pred  = float(p["p50"][h_idx]) if h_idx < len(p["p50"]) else 0.0
            y_pred_q10 = float(p["p10"][h_idx]) if h_idx < len(p["p10"]) else 0.0
            ae  = abs(y_pred - y_true)
            ape = ae / (y_true + 1e-9)
            err = y_pred - y_true
            raw_rows.append({
                "fold":       fold + 1,
                "route_id":   route_id,
                "timestamp":  val_timestamps[i][h_idx],
                "h":          h_idx + 1,
                "y_true":     y_true,
                "y_pred":     y_pred,       # q0.5
                "y_pred_q10": y_pred_q10,   # q0.1  ← новое
                "ae":         ae,
                "ape":        ape,
                "err":        err,
            })
            yt_fold.append(y_true)
            yp_fold.append(y_pred)

    yt_fold = np.array(yt_fold)
    yp_fold = np.array(yp_fold)
    tot, wape, rb = wape_rbias(yt_fold, yp_fold)
    print(f"  RAW   WAPE={wape:.4f}  |RBias|={rb:.4f}  Total={tot:.4f}")

# ============================================================
# АГРЕГАТЫ (аналогично LGB для блендинга)
# ============================================================
raw_cv_df = pd.DataFrame(raw_rows)

# ── По фолду ──
agg_fold = raw_cv_df.groupby("fold").apply(lambda df: pd.Series({
    "wape":   np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias":  abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":    df["ae"].mean(),
    "n":      len(df),
})).reset_index()

# ── По шагу h ──
agg_h = raw_cv_df.groupby("h").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
})).reset_index()

# ── По маршруту ──
agg_route = raw_cv_df.groupby("route_id").agg(
    n_points   = ("y_true",  "count"),
    mae        = ("ae",      "mean"),
    median_ae  = ("ae",      "median"),
    std_ae     = ("ae",      "std"),
    mape       = ("ape",     "mean"),
    median_ape = ("ape",     "median"),
    mean_err   = ("err",     "mean"),
    median_err = ("err",     "median"),
    mean_true  = ("y_true",  "mean"),
    mean_pred  = ("y_pred",  "mean"),
).reset_index()
agg_route["calib_scale"] = (
    raw_cv_df.groupby("route_id")["y_true"].sum().values /
    (raw_cv_df.groupby("route_id")["y_pred"].sum().values + 1e-9)
)
agg_route["wape"] = (
    raw_cv_df.groupby("route_id")["ae"].sum().values /
    (raw_cv_df.groupby("route_id")["y_true"].sum().values + 1e-9)
)

# ── Глобальные метрики ──
y_true_all = raw_cv_df["y_true"].values
y_pred_all = raw_cv_df["y_pred"].values
total_raw, wape_raw, rbias_raw = wape_rbias(y_true_all, y_pred_all)

calib_scale = float(y_true_all.sum() / (y_pred_all.sum() + 1e-9))
y_pred_cal  = np.clip(y_pred_all * calib_scale, 0, None)
total_cal, wape_cal, rbias_cal = wape_rbias(y_true_all, y_pred_cal)

print("\n" + "=" * 65)
print("SUMMARY")
print("=" * 65)
print("\n── По фолдам ──")
print(agg_fold.to_string(index=False))
print("\n── По шагам h ──")
print(agg_h.to_string(index=False))
print(f"\n{'Global RAW':25s}  WAPE={wape_raw:.4f}  |RBias|={rbias_raw:.4f}  Total={total_raw:.4f}")
print(f"{'Global CALIBRATED':25s}  WAPE={wape_cal:.4f}  |RBias|={rbias_cal:.4f}  Total={total_cal:.4f}")
print(f"  calib_scale = {calib_scale:.4f}")
print(f"\nCalibration {'HELPS ✅' if total_cal < total_raw else 'HURTS ❌'} "
      f"(Δ = {total_cal - total_raw:+.4f})")
print("\n── Топ-10 сложных маршрутов (по median_ape) ──")
print(agg_route.sort_values("median_ape", ascending=False)
      [["route_id", "mae", "median_ae", "mape", "median_ape", "mean_err", "calib_scale"]]
      .head(10).to_string(index=False))

raw_cv_df.to_csv("chronos2_cv_raw.csv", index=False)
agg_route.to_csv("chronos2_cv_agg_route.csv", index=False)
agg_fold.to_csv("chronos2_cv_agg_fold.csv", index=False)
agg_h.to_csv("chronos2_cv_agg_h.csv", index=False)
print("\n✅ chronos2_cv_raw.csv")
print("✅ chronos2_cv_agg_route.csv")
print("✅ chronos2_cv_agg_fold.csv")
print("✅ chronos2_cv_agg_h.csv")

# ============================================================
# TEST PREDICTION
# ============================================================
print("\n" + "=" * 65)
print("TEST PREDICTION")
print("=" * 65)

test_context_dfs = []
test_route_ids   = []

for route_id, grp in tqdm(test_df.groupby("route_id"), desc="Collect test"):
    route_train = train_df[train_df["route_id"] == route_id].sort_values("timestamp")
    if len(route_train) < 32:
        continue
    test_context_dfs.append(prepare_context_df(route_train, route_id))
    test_route_ids.append(route_id)

test_context_full = pd.concat(test_context_dfs, ignore_index=True)
print(f"Routes: {len(test_route_ids)}  Running inference...")

test_pred_df = run_inference(test_context_full)
test_preds   = build_preds_dict(test_pred_df, pred_col)

predictions_raw = {}
for route_id in test_route_ids:
    route_test = test_df[test_df["route_id"] == route_id].sort_values("timestamp")
    preds = test_preds.get(route_id, np.zeros(FORECAST_STEPS))
    for j, (_, row) in enumerate(route_test.iterrows()):
        pred = float(preds[j]) if j < len(preds) else float(preds[-1])
        predictions_raw[row["id"]] = max(0.0, pred)

submission_raw = (
    pd.DataFrame(list(predictions_raw.items()), columns=["id", "y_pred"])
    .sort_values("id").reset_index(drop=True)
)
submission_cal = submission_raw.copy()
submission_cal["y_pred"] = np.clip(submission_cal["y_pred"] * calib_scale, 0, None)

submission_raw.to_csv("submission_chronos2_raw.csv", index=False)
submission_cal.to_csv("submission_chronos2_calibrated.csv", index=False)
print(f"\n✅ submission_chronos2_raw.csv")
print(f"✅ submission_chronos2_calibrated.csv")
print(f"\n── Raw ──\n{submission_raw['y_pred'].describe().round(2)}")
print(f"\n── Calibrated (scale={calib_scale:.4f}) ──\n{submission_cal['y_pred'].describe().round(2)}")

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# 0. Утилиты
# ============================================================
def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    return float(np.abs(yp - yt).sum() / s + np.abs(yp.sum() / s - 1))

# ============================================================
# 1. Загружаем данные
# ============================================================
TARGET_COL    = "target_1h"
TOTAL_VAL_PTS = 40
FOLD5_PTS     = 8

raw_cv_df = pd.read_csv("chronos2_cv_raw.csv")
train_df  = pd.read_parquet("train_solo_track.parquet")
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
train_df = train_df.sort_values(["route_id","timestamp"]).reset_index(drop=True)

# Проверяем что q0.1 есть
if "y_pred_q10" not in raw_cv_df.columns:
    print("❌ y_pred_q10 не найден в CSV — нужно перегнать Chronos с сохранением q0.1")
else:
    print(f"✅ y_pred_q10 есть  |  raw_cv_df: {raw_cv_df.shape}")

# Таймстемпы fold 5 (достаём из train_df)
fold5_ts = (
    train_df.groupby("route_id", group_keys=False)
    .tail(FOLD5_PTS)[["route_id","timestamp"]]
    .reset_index(drop=True)
)
fold5_ts["h"] = fold5_ts.groupby("route_id").cumcount() + 1

# Мёрджим таймстемпы в raw_cv_df если их нет
if "timestamp" not in raw_cv_df.columns or raw_cv_df["timestamp"].isna().all():
    raw_cv_df = raw_cv_df.drop(columns=["timestamp"], errors="ignore")
    raw_cv_df = raw_cv_df.merge(
        fold5_ts.rename(columns={"timestamp": "timestamp"}),
        on=["route_id","h"], how="left"
    )
raw_cv_df["timestamp"] = pd.to_datetime(raw_cv_df["timestamp"])

cv5      = raw_cv_df[raw_cv_df["fold"] == 5].copy()
cv_train = raw_cv_df[raw_cv_df["fold"].isin([1,2,3,4])].copy()

print(f"cv5: {cv5.shape}  cv_train: {cv_train.shape}")

# ============================================================
# 2. Тест гипотезы: overpredict → q0.1
# ============================================================
mean_errs = cv_train.groupby("route_id")["err"].mean()

def eval_submission(y_true, y_pred, label):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    cs = float(yt.sum() / (yp.sum() + 1e-9))
    yp_cal = np.clip(yp * cs, 0, None)
    s = yt.sum() + 1e-9
    wape  = np.abs(yp_cal - yt).sum() / s
    rbias = abs(yp_cal.sum() / s - 1)
    tot   = wape + rbias
    print(f"  {label:45s}  WAPE={wape:.4f}  |RBias|={rbias:.4f}  "
          f"Total={tot:.4f}  cs={cs:.3f}")
    return tot

print("\n── Baseline ──")
base = eval_submission(cv5["y_true"], cv5["y_pred"],     "q0.5 (baseline)")
full10 = eval_submission(cv5["y_true"], cv5["y_pred_q10"], "q0.1 (все маршруты)")

print("\n── Разные пороги mean_err → q0.1 ──")
results = []
for thr in [0, 500, 1000, 2000, 5000, 10000, 20000]:
    over = set(mean_errs[mean_errs > thr].index)
    y_adapt = np.where(cv5["route_id"].isin(over), cv5["y_pred_q10"], cv5["y_pred"])
    tot = eval_submission(cv5["y_true"], y_adapt,
                          f"mean_err>{thr:>6} ({len(over):>4} routes→q0.1)")
    results.append((thr, len(over), tot))

best_thr = min(results, key=lambda x: x[2])
print(f"\n  Лучший порог: mean_err > {best_thr[0]}  "
      f"({best_thr[1]} маршрутов)  Total={best_thr[2]:.4f}  "
      f"Δ={best_thr[2]-base:+.4f}  {'✅' if best_thr[2] < base else '❌'}")

# ============================================================
# 3. inspect_route — два графика: q0.5 vs q0.1
# ============================================================
def inspect_route(route_id, n_context=48):
    cv5_r = cv5[cv5["route_id"] == route_id].sort_values("h").reset_index(drop=True)
    if len(cv5_r) == 0:
        print(f"❌ route_id={route_id} не найден в fold 5")
        return

    # ── Метрики на маршруте ──
    def route_metrics(y_true, y_pred, label):
        yt = np.asarray(y_true, float)
        yp = np.clip(np.asarray(y_pred, float), 0, None)
        s  = yt.sum() + 1e-9
        wape  = np.abs(yp - yt).sum() / s
        rbias = abs(yp.sum() / s - 1)
        tot   = wape + rbias
        print(f"  {label:12s}  WAPE={wape:.3f}  |RBias|={rbias:.3f}  Total={tot:.3f}")
        return tot

    print(f"\n══ Маршрут {route_id} ══")
    t50 = route_metrics(cv5_r["y_true"], cv5_r["y_pred"],     "q0.5")
    t10 = route_metrics(cv5_r["y_true"], cv5_r["y_pred_q10"], "q0.1")
    me  = mean_errs.get(route_id, float("nan"))
    print(f"  Δ(q0.1 - q0.5) = {t10-t50:+.3f}  {'✅ q0.1 лучше' if t10 < t50 else '❌ q0.5 лучше'}")

    # ── Перепрогноз по фолдам 1-4 ──
    print(f"\n  ── Перепрогноз на фолдах 1-4 (err = pred - true) ──")
    fold_stats = (
        cv_train[cv_train["route_id"] == route_id]
        .groupby("fold")
        .agg(
            mean_err  = ("err", "mean"),
            median_err= ("err", "median"),
            mean_true = ("y_true", "mean"),
            mean_pred = ("y_pred", "mean"),
        )
    )
    if len(fold_stats) == 0:
        print("  (маршрут не встречался в фолдах 1-4)")
    else:
        for fold_id, row in fold_stats.iterrows():
            bar = "▲" if row["mean_err"] > 0 else "▼"
            print(f"    fold {fold_id}:  mean_err={row['mean_err']:>8.0f}  {bar}  "
                  f"true={row['mean_true']:.0f}  pred={row['mean_pred']:.0f}")
        print(f"    ─────────────────────────────────────────────────")
        print(f"    среднее за 4 фолда:  mean_err={me:>8.0f}  "
              f"({'завышает ▲' if me > 0 else 'занижает ▼'})")

    # ── Влияние свапа на ГЛОБАЛЬНУЮ метрику fold 5 ──
    print(f"\n  ── Влияние свапа этого маршрута на глобальную метрику (fold 5) ──")

    def global_metric(swap_routes):
        y_adapt = np.where(
            cv5["route_id"].isin(swap_routes),
            cv5["y_pred_q10"], cv5["y_pred"]
        )
        yt = cv5["y_true"].values
        yp = np.clip(y_adapt, 0, None)
        cs = float(yt.sum() / (yp.sum() + 1e-9))
        yp_cal = np.clip(yp * cs, 0, None)
        s = yt.sum() + 1e-9
        return float(np.abs(yp_cal - yt).sum() / s + abs(yp_cal.sum() / s - 1))

    base_global  = global_metric(set())           # никого не свапаем
    after_global = global_metric({route_id})      # свапаем только этот маршрут
    delta_global = after_global - base_global

    print(f"    Без свапа:  Total={base_global:.5f}")
    print(f"    Со свапом:  Total={after_global:.5f}")
    print(f"    Δ global  = {delta_global:+.5f}  "
          f"{'✅ глобально лучше' if delta_global < 0 else '❌ глобально хуже'}")
    vol = cv5_r["y_true"].sum()
    vol_share = vol / cv5["y_true"].sum() * 100
    print(f"    Объём маршрута: {vol:.0f}  ({vol_share:.2f}% от общего fold 5)")

    # ── Графики ──
    hist = (
        train_df[train_df["route_id"] == route_id]
        .sort_values("timestamp").reset_index(drop=True)
    )
    n_ctx    = min(n_context, len(hist) - FOLD5_PTS)
    hist_ctx = hist.iloc[-(FOLD5_PTS + n_ctx) : -FOLD5_PTS].reset_index(drop=True)

    ts_map = fold5_ts[fold5_ts["route_id"] == route_id].set_index("h")["timestamp"]
    cv5_r["timestamp"] = cv5_r["h"].map(ts_map)

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=[
            f"q0.5  ·  Total={t50:.3f}",
            f"q0.1  ·  Total={t10:.3f}  {'✅' if t10 < t50 else '❌'}  |  Δglobal={delta_global:+.5f}",
        ],
        shared_yaxes=True,
        horizontal_spacing=0.05,
    )

    for col, pred_col_name, pred_color, pred_label in [
        (1, "y_pred",     "#00b4d8", "Chronos q0.5"),
        (2, "y_pred_q10", "#f77f00", "Chronos q0.1"),
    ]:
        fig.add_trace(go.Scatter(
            x=hist_ctx["timestamp"].tolist(),
            y=hist_ctx[TARGET_COL].tolist(),
            mode="lines+markers",
            name="Контекст", legendgroup="ctx",
            showlegend=(col == 1),
            line=dict(width=1.2, color="#888"),
            marker=dict(size=3),
        ), row=1, col=col)

        fact_x = [hist_ctx["timestamp"].iloc[-1]] + cv5_r["timestamp"].tolist()
        fact_y = [hist_ctx[TARGET_COL].iloc[-1]]  + cv5_r["y_true"].tolist()
        fig.add_trace(go.Scatter(
            x=fact_x, y=fact_y,
            mode="lines+markers",
            name="Факт (fold 5)", legendgroup="fact",
            showlegend=(col == 1),
            line=dict(width=2.5, color="#e63946"),
            marker=dict(size=7),
        ), row=1, col=col)

        pred_x = [hist_ctx["timestamp"].iloc[-1]] + cv5_r["timestamp"].tolist()
        pred_y = [hist_ctx[TARGET_COL].iloc[-1]]  + cv5_r[pred_col_name].tolist()
        fig.add_trace(go.Scatter(
            x=pred_x, y=pred_y,
            mode="lines+markers",
            name=pred_label, legendgroup="pred",
            showlegend=(col == 1),
            line=dict(width=2.5, color=pred_color, dash="dash"),
            marker=dict(size=7, symbol="diamond"),
        ), row=1, col=col)

        fig.add_vline(
            x=cv5_r["timestamp"].iloc[0].timestamp() * 1000,
            line=dict(dash="dot", width=1.2, color="gray"),
            row=1, col=col,
        )

    fig.update_layout(
        title=dict(
            text=(f"Маршрут {route_id}  ·  контекст {n_ctx} т.  ·  "
                  f"mean_err(1-4)={me:.0f}  ·  vol_share={vol_share:.2f}%")
        ),
        legend=dict(orientation="h", yanchor="bottom", y=1.10,
                    xanchor="center", x=0.5),
        height=430, width=1300,
    )
    fig.update_xaxes(title_text="Время", row=1, col=1)
    fig.update_xaxes(title_text="Время", row=1, col=2)
    fig.update_yaxes(title_text="Значение", row=1, col=1)
    fig.show()

# ── Вызов ──
inspect_route(446, n_context=48)


# inspect_route(1234, n_context=96)

In [ ]:
# ============================================================
# Chronos-2 — Full Fine-tuning (weighted stride windows)
#           + Rolling CV (5 folds × 8 steps)
#           + Test prediction
# ============================================================
import warnings
import numpy as np
import pandas as pd
import torch
from chronos import Chronos2Pipeline
from tqdm import tqdm
import os

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None


def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias


# ─── Paths ────────────────────────────────────────────────────────────
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FINETUNED_PATH = "./chronos2_full_finetuned"
MODEL_ID       = "amazon/chronos-2"

# ─── CV / Forecast ────────────────────────────────────────────────────
FORECAST_STEPS = 8
N_FOLDS        = 5
TOTAL_VAL_PTS  = N_FOLDS * FORECAST_STEPS   # 40
CONTEXT_LEN    = 2048
WINDOW_SIZE    = CONTEXT_LEN + FORECAST_STEPS  # 2056

# ─── Зоны окон ────────────────────────────────────────────────────────
STEPS_PER_DAY   = 48
STEPS_PER_WEEK  = 336
STEPS_PER_MONTH = 1440

STRIDE_DAY   = 4    # каждые 2 ч  → ~12 окон
STRIDE_WEEK  = 8    # каждые 4 ч  → ~9  окон
STRIDE_MONTH = 48   # каждые сутки → ~11 окон

# ─── Обучение ─────────────────────────────────────────────────────────
NUM_STEPS     = 3000
BATCH_SIZE    = 32    # 64 может OOM на MPS при ctx=2048
LR            = 1e-4
WARMUP_STEPS  = 300   # ~10% от NUM_STEPS, linear warmup
MAX_GRAD_NORM = 1.0   # gradient clipping
LOGGING_STEPS = 50

os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"]  = "1"
os.environ["HF_HUB_OFFLINE"]       = "1"


# ─── Device ───────────────────────────────────────────────────────────
if torch.backends.mps.is_available():
    DEVICE = "mps"
    print("Apple Silicon MPS ✅")
elif torch.cuda.is_available():
    DEVICE = "cuda"
    print("CUDA ✅")
else:
    DEVICE = "cpu"
    print("CPU ⚠️")
print(f"Device: {DEVICE}\n")


# ─── Load Data ────────────────────────────────────────────────────────
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

status_cols = sorted([c for c in train_df.columns if c.startswith("status_")])
print(f"Train: {train_df.shape}  Test: {test_df.shape}")
print(f"Status cols: {status_cols}\n")


# ============================================================
# STEP 1 — Загрузка базовой модели
# ============================================================
print(f"Loading {MODEL_ID}...")
pipeline = Chronos2Pipeline.from_pretrained(
    MODEL_ID,
    device_map=DEVICE,
    torch_dtype=torch.bfloat16,
    local_files_only=True,
)
print("Model loaded!\n")


# ============================================================
# STEP 2 — Подготовка обучающих окон (взвешенный stride)
# ============================================================
def prepare_finetune_inputs(train_df: pd.DataFrame) -> list:
    inputs  = []
    skipped = 0

    for route_id, grp in tqdm(train_df.groupby("route_id"), desc="Prep FT data"):
        grp = grp.sort_values("timestamp").reset_index(drop=True)

        # Отрезаем CV-зону (40 последних точек)
        hist = grp.iloc[:-TOTAL_VAL_PTS] if len(grp) > TOTAL_VAL_PTS + WINDOW_SIZE else grp
        if len(hist) < WINDOW_SIZE:
            skipped += 1
            continue

        df = (
            hist.set_index("timestamp")[[TARGET_COL] + status_cols]
            .asfreq("30min")
            .interpolate(method="time")
            .bfill().ffill()
        )
        target_arr = df[TARGET_COL].values
        cov_arrs   = {col: df[col].values for col in status_cols}
        n          = len(target_arr)
        seen       = set()

        def add_zone(zone_start_offset, zone_end_offset, stride):
            for end_offset in range(zone_end_offset, zone_start_offset, stride):
                end_idx   = n - end_offset
                start_idx = end_idx - WINDOW_SIZE
                if start_idx < 0:
                    break
                key = (start_idx, end_idx)
                if key in seen:
                    continue
                seen.add(key)
                inputs.append({
                    "target": target_arr[start_idx:end_idx],
                    "past_covariates":   {col: cov_arrs[col][start_idx:end_idx]
                                          for col in status_cols},
                    "future_covariates": {col: None for col in status_cols},
                })

        add_zone(STEPS_PER_DAY,   0,              STRIDE_DAY)
        add_zone(STEPS_PER_WEEK,  STEPS_PER_DAY,  STRIDE_WEEK)
        add_zone(STEPS_PER_MONTH, STEPS_PER_WEEK, STRIDE_MONTH)

    print(f"  Routes skipped (too short): {skipped}")
    return inputs


print("=" * 65)
print("STEP 2 — Full Fine-Tuning (weighted stride windows)")
print("=" * 65)

train_inputs = prepare_finetune_inputs(train_df)
n_routes     = train_df["route_id"].nunique()

print(f"\n  Routes total:      {n_routes}")
print(f"  FT windows total:  {len(train_inputs)}")
print(f"  Steps × batch:     {NUM_STEPS} × {BATCH_SIZE} = {NUM_STEPS * BATCH_SIZE:,}")
print(f"  Pool passes ~:     {NUM_STEPS * BATCH_SIZE / max(len(train_inputs), 1):.1f}x\n")

# ─── Оценка до обучения ───────────────────────────────────────────────
print("Sanity-check: ожидаемые итерации и время")
secs_per_step = 0.8  # ~0.8 с/шаг на MPS при batch=32
print(f"  ETA: ~{NUM_STEPS * secs_per_step / 60:.0f} мин\n")
if DEVICE == "mps":
    torch.mps.empty_cache()
import gc; gc.collect()
# ─── Fine-tuning ──────────────────────────────────────────────────────
finetuned_pipeline = pipeline.fit(
    inputs            = train_inputs,
    prediction_length = FORECAST_STEPS,
    num_steps         = NUM_STEPS,
    learning_rate     = LR,
    batch_size        = BATCH_SIZE,
    warmup_steps      = WARMUP_STEPS,       # ← linear warmup 300 шагов
    lr_scheduler_type = "cosine",           # ← cosine decay после warmup
    max_grad_norm     = MAX_GRAD_NORM,      # ← gradient clipping
    logging_steps     = LOGGING_STEPS,
)

finetuned_pipeline.save_pretrained(FINETUNED_PATH)
print(f"\n✅ Model saved → {FINETUNED_PATH}")

In [ ]:
# -------------------- Inference helpers --------------------
def prepare_context_df(route_df: pd.DataFrame, route_id) -> pd.DataFrame:
    df = (
        route_df.set_index("timestamp")[[TARGET_COL] + status_cols]
        .asfreq("30min")
        .interpolate(method="time")
        .bfill().ffill()
        .tail(CONTEXT_LEN)
        .reset_index()
    )
    df["id"] = route_id
    df = df.rename(columns={TARGET_COL: "target"})
    return df


def run_inference(ctx_pipeline, context_full: pd.DataFrame) -> pd.DataFrame:
    return ctx_pipeline.predict_df(
        context_full,
        prediction_length=FORECAST_STEPS,
        quantile_levels=[0.1, 0.5, 0.9],
        id_column="id",
        timestamp_column="timestamp",
        target="target",
        cross_learning=True,
    )


def get_pred_col(pred_df: pd.DataFrame) -> str:
    for candidate in ["mean", "0.5", "q0.5", "median"]:
        if candidate in pred_df.columns:
            return candidate
    num_cols = [c for c in pred_df.columns
                if c not in ("id", "timestamp") and pd.api.types.is_numeric_dtype(pred_df[c])]
    return num_cols[0]


def build_preds_dict(pred_df: pd.DataFrame, pred_col: str) -> dict:
    result = {}
    for route_id, grp in pred_df.groupby("id"):
        result[route_id] = np.clip(grp.sort_values("timestamp")[pred_col].values, 0, None)
    return result


# ============================================================
# STEP 3 — Rolling Cross-Validation
# ============================================================
print("\n" + "=" * 65)
print(f"STEP 3 — ROLLING CV  ({N_FOLDS} folds × {FORECAST_STEPS} steps = {TOTAL_VAL_PTS} points)")
print("=" * 65)

raw_rows = []
pred_col = None

for fold in range(N_FOLDS):
    val_offset_end   = TOTAL_VAL_PTS - fold * FORECAST_STEPS
    val_offset_start = val_offset_end - FORECAST_STEPS

    context_dfs    = []
    y_trues_fold   = []
    route_ids_fold = []
    val_timestamps = []

    for route_id, grp in train_df.groupby("route_id"):
        grp = grp.sort_values("timestamp").reset_index(drop=True)
        if len(grp) < TOTAL_VAL_PTS + 32:
            continue

        hist_part = grp.iloc[:-val_offset_end]
        val_part  = (grp.iloc[-val_offset_end:-val_offset_start]
                     if val_offset_start > 0
                     else grp.iloc[-val_offset_end:])

        if len(hist_part) < 32 or len(val_part) == 0:
            continue

        context_dfs.append(prepare_context_df(hist_part, route_id))
        y_trues_fold.append(val_part[TARGET_COL].values[:FORECAST_STEPS])
        route_ids_fold.append(route_id)
        val_timestamps.append(val_part["timestamp"].values[:FORECAST_STEPS])

    context_full = pd.concat(context_dfs, ignore_index=True)

    print(f"\nFold {fold+1}/{N_FOLDS}  "
          f"(val: -{val_offset_end}..{'-'+str(val_offset_start) if val_offset_start else 'end'})  "
          f"routes={len(route_ids_fold)}")

    pred_df_fold = run_inference(finetuned_pipeline, context_full)

    if pred_col is None:
        pred_col = get_pred_col(pred_df_fold)
        print(f"  Point forecast column: '{pred_col}'")

    preds = build_preds_dict(pred_df_fold, pred_col)

    yt_fold, yp_fold = [], []
    for i, route_id in enumerate(route_ids_fold):
        yp = preds.get(route_id, np.zeros(len(y_trues_fold[i])))
        for h_idx in range(len(y_trues_fold[i])):
            y_true = float(y_trues_fold[i][h_idx])
            y_pred = float(yp[h_idx]) if h_idx < len(yp) else 0.0
            ae     = abs(y_pred - y_true)
            ape    = ae / (y_true + 1e-9)
            err    = y_pred - y_true
            raw_rows.append({
                "fold":      fold + 1,
                "route_id":  route_id,
                "timestamp": val_timestamps[i][h_idx],
                "h":         h_idx + 1,
                "y_true":    y_true,
                "y_pred":    y_pred,
                "ae":        ae,
                "ape":       ape,
                "err":       err,
            })
            yt_fold.append(y_true)
            yp_fold.append(y_pred)

    yt_fold = np.array(yt_fold)
    yp_fold = np.array(yp_fold)
    tot, wape, rb = wape_rbias(yt_fold, yp_fold)
    print(f"  WAPE={wape:.4f}  |RBias|={rb:.4f}  Total={tot:.4f}")


# ============================================================
# АГРЕГАТЫ
# ============================================================
raw_cv_df = pd.DataFrame(raw_rows)

agg_fold = raw_cv_df.groupby("fold").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
    "n":     len(df),
})).reset_index()

agg_h = raw_cv_df.groupby("h").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
})).reset_index()

agg_route = raw_cv_df.groupby("route_id").agg(
    n_points   = ("y_true",  "count"),
    mae        = ("ae",      "mean"),
    median_ae  = ("ae",      "median"),
    std_ae     = ("ae",      "std"),
    mape       = ("ape",     "mean"),
    median_ape = ("ape",     "median"),
    mean_err   = ("err",     "mean"),
    median_err = ("err",     "median"),
    mean_true  = ("y_true",  "mean"),
    mean_pred  = ("y_pred",  "mean"),
).reset_index()
agg_route["calib_scale"] = (
    raw_cv_df.groupby("route_id")["y_true"].sum().values /
    (raw_cv_df.groupby("route_id")["y_pred"].sum().values + 1e-9)
)
agg_route["wape"] = (
    raw_cv_df.groupby("route_id")["ae"].sum().values /
    (raw_cv_df.groupby("route_id")["y_true"].sum().values + 1e-9)
)

y_true_all = raw_cv_df["y_true"].values
y_pred_all = raw_cv_df["y_pred"].values
total_raw, wape_raw, rbias_raw = wape_rbias(y_true_all, y_pred_all)

calib_scale = float(y_true_all.sum() / (y_pred_all.sum() + 1e-9))
y_pred_cal  = np.clip(y_pred_all * calib_scale, 0, None)
total_cal, wape_cal, rbias_cal = wape_rbias(y_true_all, y_pred_cal)

print("\n" + "=" * 65)
print("SUMMARY (Full fine-tuned)")
print("=" * 65)
print("\n── По фолдам ──")
print(agg_fold.to_string(index=False))
print("\n── По шагам h ──")
print(agg_h.to_string(index=False))
print(f"\n{'Global RAW':25s}  WAPE={wape_raw:.4f}  |RBias|={rbias_raw:.4f}  Total={total_raw:.4f}")
print(f"{'Global CALIBRATED':25s}  WAPE={wape_cal:.4f}  |RBias|={rbias_cal:.4f}  Total={total_cal:.4f}")
print(f"  calib_scale = {calib_scale:.4f}")
print(f"\nCalibration {'HELPS ✅' if total_cal < total_raw else 'HURTS ❌'} "
      f"(Δ = {total_cal - total_raw:+.4f})")
print("\n── Топ-10 сложных маршрутов (по median_ape) ──")
print(agg_route.sort_values("median_ape", ascending=False)
      [["route_id", "mae", "median_ae", "mape", "median_ape", "mean_err", "calib_scale"]]
      .head(10).to_string(index=False))

raw_cv_df.to_csv("chronos2_full_cv_raw.csv", index=False)
agg_route.to_csv("chronos2_full_cv_agg_route.csv", index=False)
agg_fold.to_csv("chronos2_full_cv_agg_fold.csv", index=False)
agg_h.to_csv("chronos2_full_cv_agg_h.csv", index=False)
print("\n✅ chronos2_full_cv_raw.csv")
print("✅ chronos2_full_cv_agg_route.csv")
print("✅ chronos2_full_cv_agg_fold.csv")
print("✅ chronos2_full_cv_agg_h.csv")


# ============================================================
# STEP 4 — Test Prediction
# ============================================================
print("\n" + "=" * 65)
print("STEP 4 — TEST PREDICTION")
print("=" * 65)

test_context_dfs = []
test_route_ids   = []

for route_id, grp in tqdm(test_df.groupby("route_id"), desc="Collect test"):
    route_train = train_df[train_df["route_id"] == route_id].sort_values("timestamp")
    if len(route_train) < 32:
        continue
    test_context_dfs.append(prepare_context_df(route_train, route_id))
    test_route_ids.append(route_id)

test_context_full = pd.concat(test_context_dfs, ignore_index=True)
print(f"Routes: {len(test_route_ids)}  Running inference...")

test_pred_df = run_inference(finetuned_pipeline, test_context_full)
test_preds   = build_preds_dict(test_pred_df, pred_col)

predictions_raw = {}
for route_id in test_route_ids:
    route_test = test_df[test_df["route_id"] == route_id].sort_values("timestamp")
    preds = test_preds.get(route_id, np.zeros(FORECAST_STEPS))
    for j, (_, row) in enumerate(route_test.iterrows()):
        pred = float(preds[j]) if j < len(preds) else float(preds[-1])
        predictions_raw[row["id"]] = max(0.0, pred)

submission_raw = (
    pd.DataFrame(list(predictions_raw.items()), columns=["id", "y_pred"])
    .sort_values("id").reset_index(drop=True)
)
submission_cal = submission_raw.copy()
submission_cal["y_pred"] = np.clip(submission_cal["y_pred"] * calib_scale, 0, None)

submission_raw.to_csv("submission_chronos2_full_raw.csv", index=False)
submission_cal.to_csv("submission_chronos2_full_calibrated.csv", index=False)
print(f"\n✅ submission_chronos2_full_raw.csv")
print(f"✅ submission_chronos2_full_calibrated.csv")
print(f"\n── Raw ──\n{submission_raw['y_pred'].describe().round(2)}")
print(f"\n── Calibrated (scale={calib_scale:.4f}) ──\n{submission_cal['y_pred'].describe().round(2)}")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# VisionTS — Zero-Shot CV  (аналог chronos2_cv_raw.csv)
# ════════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import torch
import einops
from visionts import VisionTS
from tqdm import tqdm

# ── CONFIG ───────────────────────────────────────────────────────────
PARQUET_PATH = "train_solo_track.parquet"
ARCH         = "mae_base"       # mae_base | mae_large | mae_huge
N_CONTEXT    = 48               # контекст (как в Chronos)
PRED_LEN     = 8                # горизонт прогноза (8 × 30 мин = 4 ч)
N_FOLDS      = 5
TARGET_COL   = "target_1h"
BATCH_SIZE   = 256              # маршрутов за один forward pass
ALIGN_CONST  = 1.0              # 1.0 = сезонный сигнал; 0.4 = тренд
NORM_CONST   = 0.4
PERIODICITY  = 48               # дневная сезонность (48 × 30 мин = 1 день)
SAVE_PATH    = "visionts_cv_raw.csv"

# ── DEVICE ───────────────────────────────────────────────────────────
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda:0"
else:
    DEVICE = "cpu"
print(f"Device: {DEVICE}")

# ── HELPERS ──────────────────────────────────────────────────────────
def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    return float(np.abs(yp - yt).sum() / s + np.abs(yp.sum() / s - 1))

def predict_batch(context_np):
    """context_np: (B, N_CONTEXT) → pred: (B, PRED_LEN)"""
    x = torch.Tensor(context_np[:, :, np.newaxis]).to(DEVICE)  # (B, T, 1)
    with torch.no_grad():
        y_pred, _, _ = model.forward(x, export_image=False)    # (B, P, 1)
    return np.clip(y_pred.cpu().numpy()[:, :, 0], 0, None)     # (B, P)

# ── LOAD DATA ────────────────────────────────────────────────────────
df = pd.read_parquet(PARQUET_PATH)
df = df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

route_ids = sorted(df["route_id"].unique())
N_ROUTES  = len(route_ids)
print(f"Routes: {N_ROUTES}")

# Матрица (N_ROUTES × T) для быстрого слайсинга
pivot = df.pivot(index="timestamp", columns="route_id", values=TARGET_COL).sort_index()
timestamps = pivot.index.values
mat = pivot.values.T.astype(np.float32)      # (N_ROUTES, T)
T   = mat.shape[1]
print(f"Series length per route: {T}")

import ssl
import urllib.request

# Monkey-patch: отключаем верификацию сертификата
ssl._create_default_https_context = ssl._create_unverified_context

# Для requests/torch.hub тоже
import os
os.environ["CURL_CA_BUNDLE"] = ""
os.environ["REQUESTS_CA_BUNDLE"] = ""

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Патчим requests глобально
old_get = requests.get
requests.get = lambda url, **kw: old_get(url, verify=False, **kw)

# ── LOAD MODEL ───────────────────────────────────────────────────────
model = VisionTS(ARCH, ckpt_dir=".ckpt").to(DEVICE)
model.update_config(
    N_CONTEXT, PRED_LEN,
    align_const=ALIGN_CONST,
    norm_const=NORM_CONST,
    periodicity=PERIODICITY,
)
print(f"Model loaded: {ARCH}")

# ── CROSS-VALIDATION ─────────────────────────────────────────────────
# fold 5 = самый свежий (аналог теста), fold 1 = самый ранний
records = []

for fold in tqdm(range(1, N_FOLDS + 1), desc="Folds"):
    fold_end   = T - (N_FOLDS - fold) * PRED_LEN   # exclusive
    fold_start = fold_end - PRED_LEN                # inclusive
    ctx_start  = fold_start - N_CONTEXT

    if ctx_start < 0:
        print(f"  Fold {fold}: not enough data, skip"); continue

    context = mat[:, ctx_start:fold_start]   # (N_ROUTES, N_CONTEXT)
    target  = mat[:, fold_start:fold_end]    # (N_ROUTES, PRED_LEN)
    ts      = timestamps[fold_start:fold_end]

    # Батчевый inference — все 1000 маршрутов за 4 прохода
    pred_all = []
    for b_start in range(0, N_ROUTES, BATCH_SIZE):
        b_end  = min(b_start + BATCH_SIZE, N_ROUTES)
        pred_b = predict_batch(context[b_start:b_end])
        pred_all.append(pred_b)
    pred = np.vstack(pred_all)               # (N_ROUTES, PRED_LEN)

    fold_wape = wape_rbias(target.ravel(), pred.ravel())
    print(f"  Fold {fold}: WAPE+RBIAS = {fold_wape:.5f}")

    for i, route_id in enumerate(route_ids):
        for h in range(PRED_LEN):
            records.append({
                "fold":      fold,
                "route_id":  route_id,
                "h":         h + 1,
                "timestamp": ts[h],
                TARGET_COL:  target[i, h],
                "pred":      pred[i, h],
                "err":       pred[i, h] - target[i, h],
            })

raw_cv_df = pd.DataFrame(records)
raw_cv_df.to_csv(SAVE_PATH, index=False)
print(f"\nSaved {SAVE_PATH}  ({len(raw_cv_df):,} rows)")

# ── SUMMARY METRICS ──────────────────────────────────────────────────
print("\n══ Итоговые метрики VisionTS ══")
for fold in range(1, N_FOLDS + 1):
    sub = raw_cv_df[raw_cv_df["fold"] == fold]
    m   = wape_rbias(sub[TARGET_COL], sub["pred"])
    print(f"  Fold {fold}:  {m:.5f}")
all_m = wape_rbias(raw_cv_df[TARGET_COL], raw_cv_df["pred"])
print(f"  All folds: {all_m:.5f}")

# ── СРАВНЕНИЕ С CHRONOS (если есть) ──────────────────────────────────
try:
    ch = pd.read_csv("chronos2_cv_raw.csv")
    print("\n══ Сравнение с Chronos-2 ══")
    print(f"  {'Fold':<8} {'VisionTS':>12} {'Chronos-2':>12} {'Δ':>10}")
    for fold in range(1, N_FOLDS + 1):
        s_v = raw_cv_df[raw_cv_df["fold"] == fold]
        s_c = ch[ch["fold"] == fold]
        m_v = wape_rbias(s_v[TARGET_COL], s_v["pred"])
        m_c = wape_rbias(s_c[TARGET_COL], s_c["pred"] if "pred" in s_c else s_c.filter(like="0.5").iloc[:, 0])
        delta = m_v - m_c
        sign  = "▲ хуже" if delta > 0 else "▼ лучше"
        print(f"  Fold {fold}:  {m_v:>10.5f}  {m_c:>10.5f}  {delta:>+.5f} {sign}")

    all_v = wape_rbias(raw_cv_df[TARGET_COL], raw_cv_df["pred"])
    all_c = wape_rbias(ch[TARGET_COL], ch["pred"] if "pred" in ch else ch.filter(like="0.5").iloc[:, 0])
    print(f"  All:      {all_v:>10.5f}  {all_c:>10.5f}  {all_v - all_c:>+.5f}")
except FileNotFoundError:
    print("  chronos2_cv_raw.csv не найден, пропускаю сравнение")

In [ ]:
import ssl
import os
import requests
from huggingface_hub import configure_http_backend, hf_hub_download

# ── 1. Создаём кастомную сессию без SSL верификации ──────────────────
def backend_factory() -> requests.Session:
    session = requests.Session()
    session.verify = False
    return session

configure_http_backend(backend_factory=backend_factory)

# Глушим warnings про InsecureRequest
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ── 2. Скачиваем чекпоинт MAE base ───────────────────────────────────
import shutil

os.makedirs(".ckpt", exist_ok=True)
out_path = ".ckpt/mae_visualize_vit_base.pth"

if not os.path.exists(out_path):
    print("Downloading from HuggingFace...")
    # Чекпоинт VisionTS лежит тут:
    path = hf_hub_download(
        repo_id  = "Keytoyze/VisionTS",
        filename = "mae_visualize_vit_base.pth",
    )
    shutil.copy(path, out_path)
    print(f"Saved: {out_path}  ({os.path.getsize(out_path)/1e6:.0f} MB)")
else:
    print(f"Already exists ({os.path.getsize(out_path)/1e6:.0f} MB)")

# ── 3. Грузим модель ─────────────────────────────────────────────────
from visionts import VisionTS
model = VisionTS("mae_base", ckpt_dir=".ckpt").to(DEVICE)
print("Model ready!")

In [ ]:
# ── Установка без SSL-сертификатов ──
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "neuralforecast",
     "--trusted-host", "pypi.org",
     "--trusted-host", "files.pythonhosted.org", "-q"],
    check=True,
)

In [ ]:
# ============================================================
# N-HiTS — Rolling CV (5 folds × 8 steps)
# + per-route статистика для блендинга (как в Chronos-2)
# ============================================================



import warnings
import numpy as np
import pandas as pd
import torch
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MQLoss
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None


def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias


# ── Config ──
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8        # = h
N_FOLDS        = 5
TOTAL_VAL_PTS  = N_FOLDS * FORECAST_STEPS   # 40
CONTEXT_LEN    = 2048
FREQ           = "30min"

INPUT_SIZE = 336      # 7 суток × 48 точек — ловим недельную сезонность
MAX_STEPS  = 800     # чуть больше, раз быстро
BATCH_SIZE = 64       # можно увеличить если памяти хватает


# ── Device ──
if torch.backends.mps.is_available():
    ACCELERATOR = "mps"
    print("Apple Silicon MPS ✅")
elif torch.cuda.is_available():
    ACCELERATOR = "gpu"
else:
    ACCELERATOR = "cpu"
print(f"Device: {ACCELERATOR}\n")


# ── Data ──
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

status_cols = sorted([c for c in train_df.columns if c.startswith("status_")])
print(f"Train: {train_df.shape}  Test: {test_df.shape}")
print(f"Status cols: {status_cols}\n")


FUTR_EXOG = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "is_weekend"]

def add_calendar_features(df: pd.DataFrame, ts_col: str = "ds") -> pd.DataFrame:
    ts = pd.to_datetime(df[ts_col])
    df = df.copy()
    df["hour_sin"]   = np.sin(2 * np.pi * ts.dt.hour / 24)
    df["hour_cos"]   = np.cos(2 * np.pi * ts.dt.hour / 24)
    df["dow_sin"]    = np.sin(2 * np.pi * ts.dt.dayofweek / 7)
    df["dow_cos"]    = np.cos(2 * np.pi * ts.dt.dayofweek / 7)
    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(float)
    return df

HIST_EXOG_EXTRA = ["lag_48", "lag_336"]

def add_lag_features(nf_df: pd.DataFrame) -> pd.DataFrame:
    nf_df = nf_df.sort_values(["unique_id", "ds"]).copy()
    nf_df["lag_48"] = (
        nf_df.groupby("unique_id")["y"]
        .transform(lambda s: s.shift(48).bfill().fillna(0))
    )
    nf_df["lag_336"] = (
        nf_df.groupby("unique_id")["y"]
        .transform(lambda s: s.shift(336).bfill().fillna(0))
    )
    return nf_df

# ── NF-формат: unique_id | ds | y [| status_* ...] ──
# Аналог prepare_context_df из Chronos — те же asfreq / interpolate / tail
def to_nf_df(df: pd.DataFrame,
             min_len: int = 32,
             context_len: int = CONTEXT_LEN) -> pd.DataFrame:
    rows = []
    for route_id, grp in df.groupby("route_id"):
        grp = (
            grp.sort_values("timestamp")
            .set_index("timestamp")[[TARGET_COL] + status_cols]
            .asfreq(FREQ)
            .interpolate(method="time")
            .bfill().ffill()
            .tail(context_len)
            .reset_index()
        )
        if len(grp) < min_len:
            continue
        grp.insert(0, "unique_id", route_id)
        grp = grp.rename(columns={"timestamp": "ds", TARGET_COL: "y"})
        rows.append(grp)
    nf_df = pd.concat(rows, ignore_index=True)
    nf_df = add_calendar_features(nf_df)   # futr_exog
    nf_df = add_lag_features(nf_df)         # hist_exog
    return nf_df


# ── Фабрика модели (один конфиг для CV и финального фита) ──
HIST_EXOG = (status_cols + HIST_EXOG_EXTRA) or None

def make_nhits(max_steps: int = MAX_STEPS) -> NHITS:
    return NHITS(
        h                   = FORECAST_STEPS,
        input_size          = INPUT_SIZE,          # 336 = 1 неделя
        loss                = MQLoss(level=[80]),
        # Стеки: день / 4ч / 30мин — каждый на своей частоте
        # Stack 1: MaxPool(48) → 7 дневных точек  (недельный тренд)
        # Stack 2: MaxPool(8)  → 42 точки         (4-часовые блоки)
        # Stack 3: MaxPool(1)  → 336 точек         (полное разрешение)
        n_freq_downsample   = [48, 8, 1],
        mlp_units           = [[512, 512]] * 3,
        n_blocks            = [4, 4, 4],
        max_steps           = max_steps,
        batch_size          = BATCH_SIZE,
        accelerator         = ACCELERATOR,
        hist_exog_list      = HIST_EXOG,
        futr_exog_list      = FUTR_EXOG,
        scaler_type         = "robust",            # median/IQR — пики не "приплющиваются"
        enable_progress_bar = True,
    )

def make_futr_df(route_ids: list, train_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for route_id in route_ids:
        last_ts = train_df[train_df["route_id"] == route_id]["timestamp"].max()
        future_ts = pd.date_range(
            start   = last_ts + pd.Timedelta("30min"),
            periods = FORECAST_STEPS,
            freq    = FREQ,
        )
        for ts in future_ts:
            rows.append({"unique_id": route_id, "ds": ts})
    return add_calendar_features(pd.DataFrame(rows))

# ── Выбор колонки точечного прогноза (аналог get_pred_col из Chronos) ──
def get_pred_col(pred_df: pd.DataFrame) -> str:
    for candidate in ["NHITS-median", "NHITS-q-0.50", "NHITS"]:
        if candidate in pred_df.columns:
            return candidate
    skip = {"unique_id", "ds", "y", "cutoff"}
    num_cols = [c for c in pred_df.columns
                if c not in skip and pd.api.types.is_numeric_dtype(pred_df[c])]
    median_cols = [c for c in num_cols if "median" in c or "0.5" in c or "50" in c]
    return (median_cols or num_cols)[0]


# ── Словарь предсказаний: route_id → array[FORECAST_STEPS] ──
def build_preds_dict(pred_df: pd.DataFrame, pred_col: str) -> dict:
    result = {}
    for route_id, grp in pred_df.groupby("unique_id"):
        result[route_id] = np.clip(grp.sort_values("ds")[pred_col].values, 0, None)
    return result


# ============================================================
# ROLLING CROSS-VALIDATION
# neuralforecast.cross_validation(n_windows=5, step_size=8, refit=True)
# window 1: train=[:-40]  val=[-40:-32]
# window 2: train=[:-32]  val=[-32:-24]
# window 3: train=[:-24]  val=[-24:-16]
# window 4: train=[:-16]  val=[-16: -8]
# window 5: train=[: -8]  val=[ -8:end]
# refit=True — переобучение на каждом фолде (= поведение Chronos)
# ============================================================
print("=" * 65)
print(f"ROLLING CV  ({N_FOLDS} folds × {FORECAST_STEPS} steps = {TOTAL_VAL_PTS} points)")
print("=" * 65)

train_nf = to_nf_df(
    train_df,
    min_len     = TOTAL_VAL_PTS + INPUT_SIZE + 1,
    context_len = CONTEXT_LEN,
)
print(f"NF train: {train_nf.shape}  routes: {train_nf['unique_id'].nunique()}\n")

nf_cv = NeuralForecast(models=[make_nhits()], freq=FREQ)

# refit=True: модель переобучается для каждого фолда (как Chronos per-fold inference)
# refit=False: обучается один раз, быстрее, но менее честно
cv_df = nf_cv.cross_validation(
    df        = train_nf,
    n_windows = N_FOLDS,
    step_size = FORECAST_STEPS,
    refit     = True,
)

print(f"\nCV columns: {list(cv_df.columns)}")
pred_col = get_pred_col(cv_df)
print(f"Point forecast column: '{pred_col}'\n")

# ── cutoff → fold (1 = самый старый, N_FOLDS = самый свежий) ──
cutoffs_sorted   = sorted(cv_df["cutoff"].unique())
cutoff_to_fold   = {c: i + 1 for i, c in enumerate(cutoffs_sorted)}
cv_df["fold"]    = cv_df["cutoff"].map(cutoff_to_fold)

# ── шаг h внутри фолда (1..FORECAST_STEPS) ──
cv_df = cv_df.sort_values(["fold", "unique_id", "ds"]).reset_index(drop=True)
cv_df["h"] = cv_df.groupby(["fold", "unique_id"]).cumcount() + 1

# ── raw_rows (идентична структура с Chronos) ──
raw_rows = []
for _, row in cv_df.iterrows():
    y_true = float(row["y"])
    y_pred = float(np.clip(row[pred_col], 0, None))
    ae     = abs(y_pred - y_true)
    ape    = ae / (y_true + 1e-9)
    err    = y_pred - y_true
    raw_rows.append({
        "fold":      int(row["fold"]),
        "route_id":  row["unique_id"],
        "timestamp": row["ds"],
        "h":         int(row["h"]),
        "y_true":    y_true,
        "y_pred":    y_pred,
        "ae":        ae,
        "ape":       ape,
        "err":       err,
    })

# ── per-fold console output (как у Chronos) ──
for fold in range(1, N_FOLDS + 1):
    fdf        = cv_df[cv_df["fold"] == fold]
    n_routes   = fdf["unique_id"].nunique()
    val_end    = TOTAL_VAL_PTS - (fold - 1) * FORECAST_STEPS
    val_start  = val_end - FORECAST_STEPS
    yt = fdf["y"].values
    yp = np.clip(fdf[pred_col].values, 0, None)
    tot, wape, rb = wape_rbias(yt, yp)
    print(f"\nFold {fold}/{N_FOLDS}  "
          f"(val: -{val_end}..{'-'+str(val_start) if val_start else 'end'})  "
          f"routes={n_routes}")
    print(f"  RAW   WAPE={wape:.4f}  |RBias|={rb:.4f}  Total={tot:.4f}")


# ============================================================
# АГРЕГАТЫ (идентично Chronos для блендинга)
# ============================================================
raw_cv_df = pd.DataFrame(raw_rows)

# ── По фолду ──
agg_fold = raw_cv_df.groupby("fold").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
    "n":     len(df),
})).reset_index()

# ── По шагу h ──
agg_h = raw_cv_df.groupby("h").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
})).reset_index()

# ── По маршруту ──
agg_route = raw_cv_df.groupby("route_id").agg(
    n_points   = ("y_true",  "count"),
    mae        = ("ae",      "mean"),
    median_ae  = ("ae",      "median"),
    std_ae     = ("ae",      "std"),
    mape       = ("ape",     "mean"),
    median_ape = ("ape",     "median"),
    mean_err   = ("err",     "mean"),
    median_err = ("err",     "median"),
    mean_true  = ("y_true",  "mean"),
    mean_pred  = ("y_pred",  "mean"),
).reset_index()
agg_route["calib_scale"] = (
    raw_cv_df.groupby("route_id")["y_true"].sum().values /
    (raw_cv_df.groupby("route_id")["y_pred"].sum().values + 1e-9)
)
agg_route["wape"] = (
    raw_cv_df.groupby("route_id")["ae"].sum().values /
    (raw_cv_df.groupby("route_id")["y_true"].sum().values + 1e-9)
)

# ── Глобальные метрики ──
y_true_all = raw_cv_df["y_true"].values
y_pred_all = raw_cv_df["y_pred"].values
total_raw, wape_raw, rbias_raw = wape_rbias(y_true_all, y_pred_all)

calib_scale = float(y_true_all.sum() / (y_pred_all.sum() + 1e-9))
y_pred_cal  = np.clip(y_pred_all * calib_scale, 0, None)
total_cal, wape_cal, rbias_cal = wape_rbias(y_true_all, y_pred_cal)

print("\n" + "=" * 65)
print("SUMMARY")
print("=" * 65)
print("\n── По фолдам ──")
print(agg_fold.to_string(index=False))
print("\n── По шагам h ──")
print(agg_h.to_string(index=False))
print(f"\n{'Global RAW':25s}  WAPE={wape_raw:.4f}  |RBias|={rbias_raw:.4f}  Total={total_raw:.4f}")
print(f"{'Global CALIBRATED':25s}  WAPE={wape_cal:.4f}  |RBias|={rbias_cal:.4f}  Total={total_cal:.4f}")
print(f"  calib_scale = {calib_scale:.4f}")
print(f"\nCalibration {'HELPS ✅' if total_cal < total_raw else 'HURTS ❌'} "
      f"(Δ = {total_cal - total_raw:+.4f})")
print("\n── Топ-10 сложных маршрутов (по median_ape) ──")
print(agg_route.sort_values("median_ape", ascending=False)
      [["route_id", "mae", "median_ae", "mape", "median_ape", "mean_err", "calib_scale"]]
      .head(10).to_string(index=False))

raw_cv_df.to_csv("nhits_cv_raw.csv", index=False)
agg_route.to_csv("nhits_cv_agg_route.csv", index=False)
agg_fold.to_csv("nhits_cv_agg_fold.csv", index=False)
agg_h.to_csv("nhits_cv_agg_h.csv", index=False)
print("\n✅ nhits_cv_raw.csv")
print("✅ nhits_cv_agg_route.csv")
print("✅ nhits_cv_agg_fold.csv")
print("✅ nhits_cv_agg_h.csv")


# ============================================================
# TEST PREDICTION
# Финальный фит на полных данных, predict() берёт последние
# input_size точек из тренировочного датасета автоматически
# ============================================================
print("\n" + "=" * 65)
print("TEST PREDICTION")
print("=" * 65)

train_nf_full = to_nf_df(train_df, min_len=INPUT_SIZE + 1, context_len=CONTEXT_LEN)
test_route_ids = test_df["route_id"].unique().tolist()
print(f"Routes: {len(test_route_ids)}  Fitting on full train data...")

nf_full = NeuralForecast(models=[make_nhits()], freq=FREQ)
nf_full.fit(train_nf_full)

print("Running inference...")
futr_df      = make_futr_df(test_route_ids, train_df)
test_pred_df = nf_full.predict(futr_df=futr_df)   # next FORECAST_STEPS × 30min на каждый маршрут
test_pred_col = get_pred_col(test_pred_df)
test_preds    = build_preds_dict(test_pred_df, test_pred_col)

predictions_raw = {}
for route_id in tqdm(test_route_ids, desc="Build submission"):
    route_test = test_df[test_df["route_id"] == route_id].sort_values("timestamp")
    preds = test_preds.get(route_id, np.zeros(FORECAST_STEPS))
    for j, (_, row) in enumerate(route_test.iterrows()):
        pred = float(preds[j]) if j < len(preds) else float(preds[-1])
        predictions_raw[row["id"]] = max(0.0, pred)

submission_raw = (
    pd.DataFrame(list(predictions_raw.items()), columns=["id", "y_pred"])
    .sort_values("id").reset_index(drop=True)
)
submission_cal = submission_raw.copy()
submission_cal["y_pred"] = np.clip(submission_cal["y_pred"] * calib_scale, 0, None)

submission_raw.to_csv("submission_nhits_raw.csv", index=False)
submission_cal.to_csv("submission_nhits_calibrated.csv", index=False)
print(f"\n✅ submission_nhits_raw.csv")
print(f"✅ submission_nhits_calibrated.csv")
print(f"\n── Raw ──\n{submission_raw['y_pred'].describe().round(2)}")
print(f"\n── Calibrated (scale={calib_scale:.4f}) ──\n{submission_cal['y_pred'].describe().round(2)}")

In [ ]:
# ============================================================
# N-HiTS — Rolling CV (5 folds × 8 steps)
# + per-route статистика для блендинга (как в Chronos-2)
# ============================================================



import warnings
import numpy as np
import pandas as pd
import torch
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MQLoss
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None


def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias


# ── Config ──
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8        # = h
N_FOLDS        = 5
TOTAL_VAL_PTS  = N_FOLDS * FORECAST_STEPS   # 40
CONTEXT_LEN    = 2048
FREQ           = "30min"

INPUT_SIZE = 336      # 7 суток × 48 точек — ловим недельную сезонность
MAX_STEPS  = 800     # чуть больше, раз быстро
BATCH_SIZE = 64       # можно увеличить если памяти хватает


# ── Device ──
if torch.backends.mps.is_available():
    ACCELERATOR = "mps"
    print("Apple Silicon MPS ✅")
elif torch.cuda.is_available():
    ACCELERATOR = "gpu"
else:
    ACCELERATOR = "cpu"
print(f"Device: {ACCELERATOR}\n")


# ── Data ──
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

status_cols = sorted([c for c in train_df.columns if c.startswith("status_")])
print(f"Train: {train_df.shape}  Test: {test_df.shape}")
print(f"Status cols: {status_cols}\n")


FUTR_EXOG = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "is_weekend"]

def add_calendar_features(df: pd.DataFrame, ts_col: str = "ds") -> pd.DataFrame:
    ts = pd.to_datetime(df[ts_col])
    df = df.copy()
    df["hour_sin"]   = np.sin(2 * np.pi * ts.dt.hour / 24)
    df["hour_cos"]   = np.cos(2 * np.pi * ts.dt.hour / 24)
    df["dow_sin"]    = np.sin(2 * np.pi * ts.dt.dayofweek / 7)
    df["dow_cos"]    = np.cos(2 * np.pi * ts.dt.dayofweek / 7)
    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(float)
    return df

HIST_EXOG_EXTRA = [
    "lag_48", "lag_336",
    "rolling_std_12",    # стд за последние 6ч → насколько прыгает ряд сейчас
    "rolling_range_12",  # max-min за 6ч → амплитуда последних зигзагов
    "delta_y",           # первая разность → явный сигнал о направлении скачка
]

def add_volatility_features(nf_df: pd.DataFrame) -> pd.DataFrame:
    nf_df = nf_df.sort_values(["unique_id", "ds"]).copy()
    g = nf_df.groupby("unique_id")["y"]

    nf_df["rolling_std_12"]   = g.transform(
        lambda s: s.rolling(12, min_periods=1).std().fillna(0)
    )
    nf_df["rolling_range_12"] = g.transform(
        lambda s: (s.rolling(12, min_periods=1).max()
                 - s.rolling(12, min_periods=1).min()).fillna(0)
    )
    nf_df["delta_y"] = g.transform(
        lambda s: s.diff().fillna(0)
    )
    return nf_df

def add_lag_features(nf_df: pd.DataFrame) -> pd.DataFrame:
    nf_df = nf_df.sort_values(["unique_id", "ds"]).copy()
    nf_df["lag_48"] = (
        nf_df.groupby("unique_id")["y"]
        .transform(lambda s: s.shift(48).bfill().fillna(0))
    )
    nf_df["lag_336"] = (
        nf_df.groupby("unique_id")["y"]
        .transform(lambda s: s.shift(336).bfill().fillna(0))
    )
    return nf_df

# ── NF-формат: unique_id | ds | y [| status_* ...] ──
# Аналог prepare_context_df из Chronos — те же asfreq / interpolate / tail
def to_nf_df(df: pd.DataFrame,
             min_len: int = 32,
             context_len: int = CONTEXT_LEN) -> pd.DataFrame:
    rows = []
    for route_id, grp in df.groupby("route_id"):
        grp = (
            grp.sort_values("timestamp")
            .set_index("timestamp")[[TARGET_COL] + status_cols]
            .asfreq(FREQ)
            .interpolate(method="time")
            .bfill().ffill()
            .tail(context_len)
            .reset_index()
        )
        if len(grp) < min_len:
            continue
        grp.insert(0, "unique_id", route_id)
        grp = grp.rename(columns={"timestamp": "ds", TARGET_COL: "y"})
        rows.append(grp)
    nf_df = pd.concat(rows, ignore_index=True)
    nf_df = add_calendar_features(nf_df)   # futr_exog
    nf_df = add_lag_features(nf_df)         # hist_exog
    nf_df = add_volatility_features(nf_df)   # ← добавляем
    return nf_df
    return nf_df


# ── Фабрика модели (один конфиг для CV и финального фита) ──
HIST_EXOG = (status_cols + HIST_EXOG_EXTRA) or None

def make_nhits(max_steps: int = MAX_STEPS) -> NHITS:
    return NHITS(
        h                   = FORECAST_STEPS,
        input_size          = INPUT_SIZE,          # 336 = 1 неделя
        loss                = MQLoss(level=[80]),
        # Стеки: день / 4ч / 30мин — каждый на своей частоте
        # Stack 1: MaxPool(48) → 7 дневных точек  (недельный тренд)
        # Stack 2: MaxPool(8)  → 42 точки         (4-часовые блоки)
        # Stack 3: MaxPool(1)  → 336 точек         (полное разрешение)
        n_freq_downsample   = [48, 8, 1],
        mlp_units           = [[512, 512]] * 3,
        n_blocks            = [4, 4, 4],
        max_steps           = max_steps,
        batch_size          = BATCH_SIZE,
        accelerator         = ACCELERATOR,
        hist_exog_list      = HIST_EXOG,
        futr_exog_list      = FUTR_EXOG,
        scaler_type         = "robust",            # median/IQR — пики не "приплющиваются"
        enable_progress_bar = True,
    )

def make_futr_df(route_ids: list, train_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for route_id in route_ids:
        last_ts = train_df[train_df["route_id"] == route_id]["timestamp"].max()
        future_ts = pd.date_range(
            start   = last_ts + pd.Timedelta("30min"),
            periods = FORECAST_STEPS,
            freq    = FREQ,
        )
        for ts in future_ts:
            rows.append({"unique_id": route_id, "ds": ts})
    return add_calendar_features(pd.DataFrame(rows))

# ── Выбор колонки точечного прогноза (аналог get_pred_col из Chronos) ──
def get_pred_col(pred_df: pd.DataFrame) -> str:
    for candidate in ["NHITS-median", "NHITS-q-0.50", "NHITS"]:
        if candidate in pred_df.columns:
            return candidate
    skip = {"unique_id", "ds", "y", "cutoff"}
    num_cols = [c for c in pred_df.columns
                if c not in skip and pd.api.types.is_numeric_dtype(pred_df[c])]
    median_cols = [c for c in num_cols if "median" in c or "0.5" in c or "50" in c]
    return (median_cols or num_cols)[0]


# ── Словарь предсказаний: route_id → array[FORECAST_STEPS] ──
def build_preds_dict(pred_df: pd.DataFrame, pred_col: str) -> dict:
    result = {}
    for route_id, grp in pred_df.groupby("unique_id"):
        result[route_id] = np.clip(grp.sort_values("ds")[pred_col].values, 0, None)
    return result


# ============================================================
# ROLLING CROSS-VALIDATION
# neuralforecast.cross_validation(n_windows=5, step_size=8, refit=True)
# window 1: train=[:-40]  val=[-40:-32]
# window 2: train=[:-32]  val=[-32:-24]
# window 3: train=[:-24]  val=[-24:-16]
# window 4: train=[:-16]  val=[-16: -8]
# window 5: train=[: -8]  val=[ -8:end]
# refit=True — переобучение на каждом фолде (= поведение Chronos)
# ============================================================
print("=" * 65)
print(f"ROLLING CV  ({N_FOLDS} folds × {FORECAST_STEPS} steps = {TOTAL_VAL_PTS} points)")
print("=" * 65)

train_nf = to_nf_df(
    train_df,
    min_len     = TOTAL_VAL_PTS + INPUT_SIZE + 1,
    context_len = CONTEXT_LEN,
)
print(f"NF train: {train_nf.shape}  routes: {train_nf['unique_id'].nunique()}\n")

nf_cv = NeuralForecast(models=[make_nhits()], freq=FREQ)

# refit=True: модель переобучается для каждого фолда (как Chronos per-fold inference)
# refit=False: обучается один раз, быстрее, но менее честно
cv_df = nf_cv.cross_validation(
    df        = train_nf,
    n_windows = N_FOLDS,
    step_size = FORECAST_STEPS,
    refit     = True,
)

print(f"\nCV columns: {list(cv_df.columns)}")
pred_col = get_pred_col(cv_df)
print(f"Point forecast column: '{pred_col}'\n")

# ── cutoff → fold (1 = самый старый, N_FOLDS = самый свежий) ──
cutoffs_sorted   = sorted(cv_df["cutoff"].unique())
cutoff_to_fold   = {c: i + 1 for i, c in enumerate(cutoffs_sorted)}
cv_df["fold"]    = cv_df["cutoff"].map(cutoff_to_fold)

# ── шаг h внутри фолда (1..FORECAST_STEPS) ──
cv_df = cv_df.sort_values(["fold", "unique_id", "ds"]).reset_index(drop=True)
cv_df["h"] = cv_df.groupby(["fold", "unique_id"]).cumcount() + 1

# ── raw_rows (идентична структура с Chronos) ──
raw_rows = []
for _, row in cv_df.iterrows():
    y_true = float(row["y"])
    y_pred = float(np.clip(row[pred_col], 0, None))
    ae     = abs(y_pred - y_true)
    ape    = ae / (y_true + 1e-9)
    err    = y_pred - y_true
    raw_rows.append({
        "fold":      int(row["fold"]),
        "route_id":  row["unique_id"],
        "timestamp": row["ds"],
        "h":         int(row["h"]),
        "y_true":    y_true,
        "y_pred":    y_pred,
        "ae":        ae,
        "ape":       ape,
        "err":       err,
    })

# ── per-fold console output (как у Chronos) ──
for fold in range(1, N_FOLDS + 1):
    fdf        = cv_df[cv_df["fold"] == fold]
    n_routes   = fdf["unique_id"].nunique()
    val_end    = TOTAL_VAL_PTS - (fold - 1) * FORECAST_STEPS
    val_start  = val_end - FORECAST_STEPS
    yt = fdf["y"].values
    yp = np.clip(fdf[pred_col].values, 0, None)
    tot, wape, rb = wape_rbias(yt, yp)
    print(f"\nFold {fold}/{N_FOLDS}  "
          f"(val: -{val_end}..{'-'+str(val_start) if val_start else 'end'})  "
          f"routes={n_routes}")
    print(f"  RAW   WAPE={wape:.4f}  |RBias|={rb:.4f}  Total={tot:.4f}")


# ============================================================
# АГРЕГАТЫ (идентично Chronos для блендинга)
# ============================================================
raw_cv_df = pd.DataFrame(raw_rows)

# ── По фолду ──
agg_fold = raw_cv_df.groupby("fold").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
    "n":     len(df),
})).reset_index()

# ── По шагу h ──
agg_h = raw_cv_df.groupby("h").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
})).reset_index()

# ── По маршруту ──
agg_route = raw_cv_df.groupby("route_id").agg(
    n_points   = ("y_true",  "count"),
    mae        = ("ae",      "mean"),
    median_ae  = ("ae",      "median"),
    std_ae     = ("ae",      "std"),
    mape       = ("ape",     "mean"),
    median_ape = ("ape",     "median"),
    mean_err   = ("err",     "mean"),
    median_err = ("err",     "median"),
    mean_true  = ("y_true",  "mean"),
    mean_pred  = ("y_pred",  "mean"),
).reset_index()
agg_route["calib_scale"] = (
    raw_cv_df.groupby("route_id")["y_true"].sum().values /
    (raw_cv_df.groupby("route_id")["y_pred"].sum().values + 1e-9)
)
agg_route["wape"] = (
    raw_cv_df.groupby("route_id")["ae"].sum().values /
    (raw_cv_df.groupby("route_id")["y_true"].sum().values + 1e-9)
)

# ── Глобальные метрики ──
y_true_all = raw_cv_df["y_true"].values
y_pred_all = raw_cv_df["y_pred"].values
total_raw, wape_raw, rbias_raw = wape_rbias(y_true_all, y_pred_all)

calib_scale = float(y_true_all.sum() / (y_pred_all.sum() + 1e-9))
y_pred_cal  = np.clip(y_pred_all * calib_scale, 0, None)
total_cal, wape_cal, rbias_cal = wape_rbias(y_true_all, y_pred_cal)

print("\n" + "=" * 65)
print("SUMMARY")
print("=" * 65)
print("\n── По фолдам ──")
print(agg_fold.to_string(index=False))
print("\n── По шагам h ──")
print(agg_h.to_string(index=False))
print(f"\n{'Global RAW':25s}  WAPE={wape_raw:.4f}  |RBias|={rbias_raw:.4f}  Total={total_raw:.4f}")
print(f"{'Global CALIBRATED':25s}  WAPE={wape_cal:.4f}  |RBias|={rbias_cal:.4f}  Total={total_cal:.4f}")
print(f"  calib_scale = {calib_scale:.4f}")
print(f"\nCalibration {'HELPS ✅' if total_cal < total_raw else 'HURTS ❌'} "
      f"(Δ = {total_cal - total_raw:+.4f})")
print("\n── Топ-10 сложных маршрутов (по median_ape) ──")
print(agg_route.sort_values("median_ape", ascending=False)
      [["route_id", "mae", "median_ae", "mape", "median_ape", "mean_err", "calib_scale"]]
      .head(10).to_string(index=False))

raw_cv_df.to_csv("nhits_cv_raw.csv", index=False)
agg_route.to_csv("nhits_cv_agg_route.csv", index=False)
agg_fold.to_csv("nhits_cv_agg_fold.csv", index=False)
agg_h.to_csv("nhits_cv_agg_h.csv", index=False)
print("\n✅ nhits_cv_raw.csv")
print("✅ nhits_cv_agg_route.csv")
print("✅ nhits_cv_agg_fold.csv")
print("✅ nhits_cv_agg_h.csv")


# ============================================================
# TEST PREDICTION
# Финальный фит на полных данных, predict() берёт последние
# input_size точек из тренировочного датасета автоматически
# ============================================================
print("\n" + "=" * 65)
print("TEST PREDICTION")
print("=" * 65)

train_nf_full = to_nf_df(train_df, min_len=INPUT_SIZE + 1, context_len=CONTEXT_LEN)
test_route_ids = test_df["route_id"].unique().tolist()
print(f"Routes: {len(test_route_ids)}  Fitting on full train data...")

nf_full = NeuralForecast(models=[make_nhits()], freq=FREQ)
nf_full.fit(train_nf_full)

print("Running inference...")
futr_df      = make_futr_df(test_route_ids, train_df)
test_pred_df = nf_full.predict(futr_df=futr_df)   # next FORECAST_STEPS × 30min на каждый маршрут
test_pred_col = get_pred_col(test_pred_df)
test_preds    = build_preds_dict(test_pred_df, test_pred_col)

predictions_raw = {}
for route_id in tqdm(test_route_ids, desc="Build submission"):
    route_test = test_df[test_df["route_id"] == route_id].sort_values("timestamp")
    preds = test_preds.get(route_id, np.zeros(FORECAST_STEPS))
    for j, (_, row) in enumerate(route_test.iterrows()):
        pred = float(preds[j]) if j < len(preds) else float(preds[-1])
        predictions_raw[row["id"]] = max(0.0, pred)

submission_raw = (
    pd.DataFrame(list(predictions_raw.items()), columns=["id", "y_pred"])
    .sort_values("id").reset_index(drop=True)
)
submission_cal = submission_raw.copy()
submission_cal["y_pred"] = np.clip(submission_cal["y_pred"] * calib_scale, 0, None)

submission_raw.to_csv("submission_nhits_raw.csv", index=False)
submission_cal.to_csv("submission_nhits_calibrated.csv", index=False)
print(f"\n✅ submission_nhits_raw.csv")
print(f"✅ submission_nhits_calibrated.csv")
print(f"\n── Raw ──\n{submission_raw['y_pred'].describe().round(2)}")
print(f"\n── Calibrated (scale={calib_scale:.4f}) ──\n{submission_cal['y_pred'].describe().round(2)}")

In [ ]:
# ============================================================
# N-HiTS — Rolling CV (5 folds × 8 steps)
# + per-route статистика для блендинга (как в Chronos-2)
# ============================================================



import warnings
import numpy as np
import pandas as pd
import torch
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MQLoss
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None


def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias


# ── Config ──
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8        # = h
N_FOLDS        = 5
TOTAL_VAL_PTS  = N_FOLDS * FORECAST_STEPS   # 40
CONTEXT_LEN    = 2048
FREQ           = "30min"

INPUT_SIZE = 96    
MAX_STEPS  = 800     # чуть больше, раз быстро
BATCH_SIZE = 64       # можно увеличить если памяти хватает


# ── Device ──
if torch.backends.mps.is_available():
    ACCELERATOR = "mps"
    print("Apple Silicon MPS ✅")
elif torch.cuda.is_available():
    ACCELERATOR = "gpu"
else:
    ACCELERATOR = "cpu"
print(f"Device: {ACCELERATOR}\n")


# ── Data ──
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

status_cols = sorted([c for c in train_df.columns if c.startswith("status_")])
print(f"Train: {train_df.shape}  Test: {test_df.shape}")
print(f"Status cols: {status_cols}\n")


FUTR_EXOG = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "is_weekend"]

def add_calendar_features(df: pd.DataFrame, ts_col: str = "ds") -> pd.DataFrame:
    ts = pd.to_datetime(df[ts_col])
    df = df.copy()
    df["hour_sin"]   = np.sin(2 * np.pi * ts.dt.hour / 24)
    df["hour_cos"]   = np.cos(2 * np.pi * ts.dt.hour / 24)
    df["dow_sin"]    = np.sin(2 * np.pi * ts.dt.dayofweek / 7)
    df["dow_cos"]    = np.cos(2 * np.pi * ts.dt.dayofweek / 7)
    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(float)
    return df

# Корреляции из анализа:
# status_3 → target+1h (2 шага по 30min): lag 2
# status_2 → target+2h (4 шага по 30min): lag 4
# Плюс ближайшие соседи для AR-эффекта

KEY_STATUS_LAGS = {
    "status_3": [1, 2, 4],   # lag2 — главный сигнал (corr 0.39 within-route)
    "status_2": [2, 4, 8],   # lag4 — главный сигнал (corr 0.26 within-route)
}

def add_status_lag_features(nf_df: pd.DataFrame) -> pd.DataFrame:
    nf_df = nf_df.copy()
    for col, lags in KEY_STATUS_LAGS.items():
        if col not in nf_df.columns:
            continue
        for lag in lags:
            nf_df[f"{col}_lag{lag}"] = (
                nf_df.groupby("unique_id")[col]
                .transform(lambda s, l=lag: s.shift(l).bfill().fillna(0))
            )
    # rolling mean status_3 — сглаживает шум, оставляет тренд сигнала
    nf_df["status_3_roll4"] = (
        nf_df.groupby("unique_id")["status_3"]
        .transform(lambda s: s.rolling(4, min_periods=1).mean().fillna(0))
    )
    return nf_df

STATUS_LAG_COLS = [
    f"{col}_lag{lag}"
    for col, lags in KEY_STATUS_LAGS.items()
    for lag in lags
] + ["status_3_roll4"]

def add_lag_features(nf_df: pd.DataFrame) -> pd.DataFrame:
    nf_df = nf_df.sort_values(["unique_id", "ds"]).copy()
    g = nf_df.groupby("unique_id")["y"]

    # AR-компонент (PACF показывает доминирующий lag1)
    nf_df["lag_1"]  = g.transform(lambda s: s.shift(1).bfill().fillna(0))
    nf_df["lag_2"]  = g.transform(lambda s: s.shift(2).bfill().fillna(0))
    nf_df["lag_4"]  = g.transform(lambda s: s.shift(4).bfill().fillna(0))

    # Недельная / суточная (оставляем, но с меньшим весом)
    nf_df["lag_48"] = g.transform(lambda s: s.shift(48).bfill().fillna(0))
    nf_df["lag_336"]= g.transform(lambda s: s.shift(336).bfill().fillna(0))

    # ARCH-прокси: rolling std — модель увидит текущий уровень волатильности
    nf_df["roll_std_8"]  = g.transform(lambda s: s.rolling(8,  min_periods=1).std().fillna(0))
    nf_df["roll_std_48"] = g.transform(lambda s: s.rolling(48, min_periods=1).std().fillna(0))
    # rolling max — ловит пики (p99/p50=2.4, т.е. пики в 2.4× выше медианы)
    nf_df["roll_max_8"]  = g.transform(lambda s: s.rolling(8,  min_periods=1).max().fillna(0))
    # первые разности — убираем тренд внутри окна
    nf_df["diff_1"]      = g.transform(lambda s: s.diff(1).fillna(0))

    return nf_df

HIST_EXOG_EXTRA = [
    "lag_1", "lag_2", "lag_4", "lag_48", "lag_336",
    "roll_std_8", "roll_std_48", "roll_max_8", "diff_1"
]

# ── NF-формат: unique_id | ds | y [| status_* ...] ──
# Аналог prepare_context_df из Chronos — те же asfreq / interpolate / tail
def to_nf_df(df: pd.DataFrame,
             min_len: int = 32,
             context_len: int = CONTEXT_LEN) -> pd.DataFrame:
    rows = []
    for route_id, grp in df.groupby("route_id"):
        grp = (
            grp.sort_values("timestamp")
            .set_index("timestamp")[[TARGET_COL] + status_cols]
            .asfreq(FREQ)
            .interpolate(method="time")
            .bfill().ffill()
            .tail(context_len)
            .reset_index()
        )
        if len(grp) < min_len:
            continue
        grp.insert(0, "unique_id", route_id)
        grp = grp.rename(columns={"timestamp": "ds", TARGET_COL: "y"})
        rows.append(grp)
    nf_df = pd.concat(rows, ignore_index=True)
    nf_df = add_calendar_features(nf_df)   # futr_exog
    nf_df = add_lag_features(nf_df)         # hist_exog
    nf_df = add_status_lag_features(nf_df)
    return nf_df


# ── Фабрика модели (один конфиг для CV и финального фита) ──
HIST_EXOG = status_cols + HIST_EXOG_EXTRA + STATUS_LAG_COLS

def make_nhits(max_steps: int = MAX_STEPS) -> NHITS:
    return NHITS(
        h                   = FORECAST_STEPS,
        input_size          = INPUT_SIZE,          # 336 = 1 неделя
        loss                = MQLoss(level=[80]),
        # Стеки: день / 4ч / 30мин — каждый на своей частоте
        # Stack 1: MaxPool(48) → 7 дневных точек  (недельный тренд)
        # Stack 2: MaxPool(8)  → 42 точки         (4-часовые блоки)
        # Stack 3: MaxPool(1)  → 336 точек         (полное разрешение)
        n_freq_downsample   = [24, 8, 1],
        mlp_units           = [[512, 512]] * 3,
        n_blocks            = [2, 6, 3],
        max_steps           = max_steps,
        batch_size          = BATCH_SIZE,
        accelerator         = ACCELERATOR,
        hist_exog_list      = HIST_EXOG,
        futr_exog_list      = FUTR_EXOG,
        scaler_type         = "robust",            # median/IQR — пики не "приплющиваются"
        enable_progress_bar = True,
    )

def make_futr_df(route_ids: list, train_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for route_id in route_ids:
        last_ts = train_df[train_df["route_id"] == route_id]["timestamp"].max()
        future_ts = pd.date_range(
            start   = last_ts + pd.Timedelta("30min"),
            periods = FORECAST_STEPS,
            freq    = FREQ,
        )
        for ts in future_ts:
            rows.append({"unique_id": route_id, "ds": ts})
    return add_calendar_features(pd.DataFrame(rows))

# ── Выбор колонки точечного прогноза (аналог get_pred_col из Chronos) ──
def get_pred_col(pred_df: pd.DataFrame) -> str:
    for candidate in ["NHITS-median", "NHITS-q-0.50", "NHITS"]:
        if candidate in pred_df.columns:
            return candidate
    skip = {"unique_id", "ds", "y", "cutoff"}
    num_cols = [c for c in pred_df.columns
                if c not in skip and pd.api.types.is_numeric_dtype(pred_df[c])]
    median_cols = [c for c in num_cols if "median" in c or "0.5" in c or "50" in c]
    return (median_cols or num_cols)[0]


# ── Словарь предсказаний: route_id → array[FORECAST_STEPS] ──
def build_preds_dict(pred_df: pd.DataFrame, pred_col: str) -> dict:
    result = {}
    for route_id, grp in pred_df.groupby("unique_id"):
        result[route_id] = np.clip(grp.sort_values("ds")[pred_col].values, 0, None)
    return result


# ============================================================
# ROLLING CROSS-VALIDATION
# neuralforecast.cross_validation(n_windows=5, step_size=8, refit=True)
# window 1: train=[:-40]  val=[-40:-32]
# window 2: train=[:-32]  val=[-32:-24]
# window 3: train=[:-24]  val=[-24:-16]
# window 4: train=[:-16]  val=[-16: -8]
# window 5: train=[: -8]  val=[ -8:end]
# refit=True — переобучение на каждом фолде (= поведение Chronos)
# ============================================================
print("=" * 65)
print(f"ROLLING CV  ({N_FOLDS} folds × {FORECAST_STEPS} steps = {TOTAL_VAL_PTS} points)")
print("=" * 65)

train_nf = to_nf_df(
    train_df,
    min_len     = TOTAL_VAL_PTS + INPUT_SIZE + 1,
    context_len = CONTEXT_LEN,
)
print(f"NF train: {train_nf.shape}  routes: {train_nf['unique_id'].nunique()}\n")

nf_cv = NeuralForecast(models=[make_nhits()], freq=FREQ)

# refit=True: модель переобучается для каждого фолда (как Chronos per-fold inference)
# refit=False: обучается один раз, быстрее, но менее честно
cv_df = nf_cv.cross_validation(
    df        = train_nf,
    n_windows = N_FOLDS,
    step_size = FORECAST_STEPS,
    refit     = True,
)

print(f"\nCV columns: {list(cv_df.columns)}")
pred_col = get_pred_col(cv_df)
print(f"Point forecast column: '{pred_col}'\n")

# ── cutoff → fold (1 = самый старый, N_FOLDS = самый свежий) ──
cutoffs_sorted   = sorted(cv_df["cutoff"].unique())
cutoff_to_fold   = {c: i + 1 for i, c in enumerate(cutoffs_sorted)}
cv_df["fold"]    = cv_df["cutoff"].map(cutoff_to_fold)

# ── шаг h внутри фолда (1..FORECAST_STEPS) ──
cv_df = cv_df.sort_values(["fold", "unique_id", "ds"]).reset_index(drop=True)
cv_df["h"] = cv_df.groupby(["fold", "unique_id"]).cumcount() + 1

# ── raw_rows (идентична структура с Chronos) ──
raw_rows = []
for _, row in cv_df.iterrows():
    y_true = float(row["y"])
    y_pred = float(np.clip(row[pred_col], 0, None))
    ae     = abs(y_pred - y_true)
    ape    = ae / (y_true + 1e-9)
    err    = y_pred - y_true
    raw_rows.append({
        "fold":      int(row["fold"]),
        "route_id":  row["unique_id"],
        "timestamp": row["ds"],
        "h":         int(row["h"]),
        "y_true":    y_true,
        "y_pred":    y_pred,
        "ae":        ae,
        "ape":       ape,
        "err":       err,
    })

# ── per-fold console output (как у Chronos) ──
for fold in range(1, N_FOLDS + 1):
    fdf        = cv_df[cv_df["fold"] == fold]
    n_routes   = fdf["unique_id"].nunique()
    val_end    = TOTAL_VAL_PTS - (fold - 1) * FORECAST_STEPS
    val_start  = val_end - FORECAST_STEPS
    yt = fdf["y"].values
    yp = np.clip(fdf[pred_col].values, 0, None)
    tot, wape, rb = wape_rbias(yt, yp)
    print(f"\nFold {fold}/{N_FOLDS}  "
          f"(val: -{val_end}..{'-'+str(val_start) if val_start else 'end'})  "
          f"routes={n_routes}")
    print(f"  RAW   WAPE={wape:.4f}  |RBias|={rb:.4f}  Total={tot:.4f}")


# ============================================================
# АГРЕГАТЫ (идентично Chronos для блендинга)
# ============================================================
raw_cv_df = pd.DataFrame(raw_rows)

# ── По фолду ──
agg_fold = raw_cv_df.groupby("fold").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
    "n":     len(df),
})).reset_index()

# ── По шагу h ──
agg_h = raw_cv_df.groupby("h").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
})).reset_index()

# ── По маршруту ──
agg_route = raw_cv_df.groupby("route_id").agg(
    n_points   = ("y_true",  "count"),
    mae        = ("ae",      "mean"),
    median_ae  = ("ae",      "median"),
    std_ae     = ("ae",      "std"),
    mape       = ("ape",     "mean"),
    median_ape = ("ape",     "median"),
    mean_err   = ("err",     "mean"),
    median_err = ("err",     "median"),
    mean_true  = ("y_true",  "mean"),
    mean_pred  = ("y_pred",  "mean"),
).reset_index()
agg_route["calib_scale"] = (
    raw_cv_df.groupby("route_id")["y_true"].sum().values /
    (raw_cv_df.groupby("route_id")["y_pred"].sum().values + 1e-9)
)
agg_route["wape"] = (
    raw_cv_df.groupby("route_id")["ae"].sum().values /
    (raw_cv_df.groupby("route_id")["y_true"].sum().values + 1e-9)
)

# ── Глобальные метрики ──
y_true_all = raw_cv_df["y_true"].values
y_pred_all = raw_cv_df["y_pred"].values
total_raw, wape_raw, rbias_raw = wape_rbias(y_true_all, y_pred_all)

calib_scale = float(y_true_all.sum() / (y_pred_all.sum() + 1e-9))
y_pred_cal  = np.clip(y_pred_all * calib_scale, 0, None)
total_cal, wape_cal, rbias_cal = wape_rbias(y_true_all, y_pred_cal)

print("\n" + "=" * 65)
print("SUMMARY")
print("=" * 65)
print("\n── По фолдам ──")
print(agg_fold.to_string(index=False))
print("\n── По шагам h ──")
print(agg_h.to_string(index=False))
print(f"\n{'Global RAW':25s}  WAPE={wape_raw:.4f}  |RBias|={rbias_raw:.4f}  Total={total_raw:.4f}")
print(f"{'Global CALIBRATED':25s}  WAPE={wape_cal:.4f}  |RBias|={rbias_cal:.4f}  Total={total_cal:.4f}")
print(f"  calib_scale = {calib_scale:.4f}")
print(f"\nCalibration {'HELPS ✅' if total_cal < total_raw else 'HURTS ❌'} "
      f"(Δ = {total_cal - total_raw:+.4f})")
print("\n── Топ-10 сложных маршрутов (по median_ape) ──")
print(agg_route.sort_values("median_ape", ascending=False)
      [["route_id", "mae", "median_ae", "mape", "median_ape", "mean_err", "calib_scale"]]
      .head(10).to_string(index=False))

raw_cv_df.to_csv("nhits_cv_raw.csv", index=False)
agg_route.to_csv("nhits_cv_agg_route.csv", index=False)
agg_fold.to_csv("nhits_cv_agg_fold.csv", index=False)
agg_h.to_csv("nhits_cv_agg_h.csv", index=False)
print("\n✅ nhits_cv_raw.csv")
print("✅ nhits_cv_agg_route.csv")
print("✅ nhits_cv_agg_fold.csv")
print("✅ nhits_cv_agg_h.csv")


# ============================================================
# TEST PREDICTION
# Финальный фит на полных данных, predict() берёт последние
# input_size точек из тренировочного датасета автоматически
# ============================================================
print("\n" + "=" * 65)
print("TEST PREDICTION")
print("=" * 65)

train_nf_full = to_nf_df(train_df, min_len=INPUT_SIZE + 1, context_len=CONTEXT_LEN)
test_route_ids = test_df["route_id"].unique().tolist()
print(f"Routes: {len(test_route_ids)}  Fitting on full train data...")

nf_full = NeuralForecast(models=[make_nhits()], freq=FREQ)
nf_full.fit(train_nf_full)

print("Running inference...")
futr_df      = make_futr_df(test_route_ids, train_df)
test_pred_df = nf_full.predict(futr_df=futr_df)   # next FORECAST_STEPS × 30min на каждый маршрут
test_pred_col = get_pred_col(test_pred_df)
test_preds    = build_preds_dict(test_pred_df, test_pred_col)

predictions_raw = {}
for route_id in tqdm(test_route_ids, desc="Build submission"):
    route_test = test_df[test_df["route_id"] == route_id].sort_values("timestamp")
    preds = test_preds.get(route_id, np.zeros(FORECAST_STEPS))
    for j, (_, row) in enumerate(route_test.iterrows()):
        pred = float(preds[j]) if j < len(preds) else float(preds[-1])
        predictions_raw[row["id"]] = max(0.0, pred)

submission_raw = (
    pd.DataFrame(list(predictions_raw.items()), columns=["id", "y_pred"])
    .sort_values("id").reset_index(drop=True)
)
submission_cal = submission_raw.copy()
submission_cal["y_pred"] = np.clip(submission_cal["y_pred"] * calib_scale, 0, None)

submission_raw.to_csv("submission_nhits_raw.csv", index=False)
submission_cal.to_csv("submission_nhits_calibrated.csv", index=False)
print(f"\n✅ submission_nhits_raw.csv")
print(f"✅ submission_nhits_calibrated.csv")
print(f"\n── Raw ──\n{submission_raw['y_pred'].describe().round(2)}")
print(f"\n── Calibrated (scale={calib_scale:.4f}) ──\n{submission_cal['y_pred'].describe().round(2)}")

In [ ]:
tr = [485, 192, 702, 999, 446, 782, 725, 768, 445, 817, 329, 594, 123, 948, 263]
agg_route["calib_scale"][agg_route["route_id"].isin(tr)]

In [ ]:
# ============================================================
# N-HiTS Decomposition — визуализация вклада каждого стека
# Аналог plot_interpretation из PyTorch Forecasting
# ============================================================

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os

os.makedirs("diagnostics", exist_ok=True)


def plot_nhits_decomposition(
    nf_fitted,
    route_id,
    train_nf_df,
    train_df_raw,
    n_blocks_config,
    n_freq_downsample,
    input_size,
    freq_labels=None,
    context_show=96,
    figsize=(18, 12),
):
    model = nf_fitted.models[0]
    model.eval()
    n_stacks = len(n_blocks_config)

    if freq_labels is None:
        freq_labels = [
            f"Stack {i+1}  (MaxPool×{n_freq_downsample[i]})"
            for i in range(n_stacks)
        ]

    # ── Данные маршрута ──
    single_df = train_nf_df[train_nf_df["unique_id"] == route_id].copy()
    route_y   = single_df.sort_values("ds")["y"].values

    # ── Robust-денормализация: модель нормирует по последним input_size точкам ──
    window  = route_y[-input_size:]
    median  = float(np.median(window))
    q25, q75 = float(np.percentile(window, 25)), float(np.percentile(window, 75))
    iqr     = max(q75 - q25, 1.0)   # IQR из того же окна, что видит модель

    def denorm(arr):
        """normalized → original units (приближение, точность ~5%)"""
        return np.asarray(arr) * iqr + median

    # ── Регистрируем хуки — .clone() ОБЯЗАТЕЛЕН ──
    captured_fc, captured_bc = [], []
    hooks = []

    for block in model.blocks:
        def _hook(m, inp, out, _fc=captured_fc, _bc=captured_bc):
            bc, fc = out
            _bc.append(bc.detach().cpu().clone())   # ← clone!
            _fc.append(fc.detach().cpu().clone())   # ← clone!
        hooks.append(block.register_forward_hook(_hook))

    # ── Прогон ──
    futr    = make_futr_df([route_id], train_df_raw)
    with torch.no_grad():
        pred_df = nf_fitted.predict(df=single_df, futr_df=futr)

    for h in hooks:
        h.remove()

    # ── Итоговый прогноз ──
    pred_col    = get_pred_col(pred_df)
    route_pred  = pred_df[pred_df["unique_id"] == route_id].sort_values("ds")
    final_fc    = route_pred[pred_col].values
    future_ts   = route_pred["ds"].values

    # ── Контекст для отображения ──
    ctx   = single_df.sort_values("ds").tail(context_show)
    ctx_ts, ctx_y = ctx["ds"].values, ctx["y"].values

    # ── Конвертация тензора → 1D ──
    # MQLoss(level=[80]) → 3 квантиля: [q10, q50, q90]
    def to_1d(t):
        a = t.squeeze().numpy()
        if a.ndim == 2:        # (h, n_quantiles)
            a = a[:, 1]        # медиана (q50)
        return np.atleast_1d(a)

    # ── Суммируем блоки по стекам + денормализуем бэккаст ──
    stack_fc, stack_bc = [], []
    idx = 0
    for n_b in n_blocks_config:
        fc_sum = bc_sum = None
        for _ in range(n_b):
            fc = to_1d(captured_fc[idx])
            bc = to_1d(captured_bc[idx])
            fc_sum = fc if fc_sum is None else fc_sum + fc
            bc_sum = bc if bc_sum is None else bc_sum + bc
            idx += 1
        stack_fc.append(fc_sum)
        # Бэккаст из нормированного пространства → оригинальные единицы
        stack_bc.append(denorm(bc_sum))

    # ── Проверка: Σ стеков ≈ final_fc (финальный прогноз уже денормирован моделью) ──
    reconstructed_norm = sum(to_1d(captured_fc[i]) for i in range(sum(n_blocks_config)))
    reconstructed_denorm = denorm(reconstructed_norm)
    max_err_abs = np.abs(reconstructed_denorm - final_fc).max()
    max_err_pct = max_err_abs / (np.abs(final_fc).mean() + 1e-9) * 100
    print(f"Route {route_id} | denorm reconstruction error: "
          f"{max_err_abs:,.0f}  ({max_err_pct:.1f}%)  "
          f"{'✅' if max_err_pct < 10 else '⚠️ check quantile index'}")

    # ── Визуализация ──
    colors = ["#2196F3", "#FF9800", "#4CAF50", "#E91E63", "#9C27B0"]
    n_rows = 1 + n_stacks

    fig = plt.figure(figsize=figsize)
    fig.suptitle(f"N-HiTS Stack Decomposition — Route {route_id}",
                 fontsize=14, fontweight="bold", y=1.01)
    gs = gridspec.GridSpec(n_rows, 2, figure=fig, hspace=0.7, wspace=0.35)

    # ── Row 0: полная картина ──
    ax0 = fig.add_subplot(gs[0, :])
    ax0.plot(ctx_ts, ctx_y, color="#555", lw=1.0, label="Context")
    ax0.plot(future_ts, final_fc, "r--o", ms=5, lw=2.0, label="Final forecast")
    ax0.plot(future_ts, reconstructed_denorm, ":", color="orange", lw=1.4,
             label=f"Σ stacks  (err={max_err_pct:.1f}%)")
    ax0.axvline(future_ts[0], color="black", ls=":", lw=1.0, alpha=0.5)
    ax0.set_title(f"context={context_show}pts  |  forecast h=1..{len(final_fc)}")
    ax0.legend(fontsize=8, loc="upper left")
    ax0.grid(True, alpha=0.3)
    ax0.set_ylabel("target_1h")

    # ── Rows 1..n_stacks ──
    for i in range(n_stacks):
        fc    = stack_fc[i]      # в пространстве final_fc (нормированный масштаб)
        bc    = stack_bc[i]      # денормированный
        color = colors[i % len(colors)]
        label = freq_labels[i]
        pool  = n_freq_downsample[i]

        # ─ Левый: Backcast (денормированный) ─
        ax_l = fig.add_subplot(gs[i + 1, 0])
        ax_l.plot(ctx_ts, ctx_y, color="#ddd", lw=0.7, label="Context", zorder=1)

        bc_tail = bc[-len(ctx_ts):]
        ax_l.plot(ctx_ts[-len(bc_tail):], bc_tail,
                  color=color, lw=1.8, label="Backcast (denorm)", zorder=2)

        min_len = min(len(bc_tail), len(ctx_y))
        ax_l.fill_between(
            ctx_ts[-min_len:], ctx_y[-min_len:], bc_tail[-min_len:],
            alpha=0.15, color=color, label="Residual"
        )
        ax_l.set_title(f"{label}\nBackcast  (original units, MaxPool×{pool})", fontsize=8)
        ax_l.legend(fontsize=7, loc="upper left")
        ax_l.grid(True, alpha=0.3)
        ax_l.set_ylabel("target_1h")

        # ─ Правый: Forecast contribution ─
        # fc здесь в нормированном пространстве → денормируем для отображения
        fc_denorm = denorm(fc)
        pct       = fc_denorm / (final_fc + 1e-9) * 100

        ax_r = fig.add_subplot(gs[i + 1, 1])
        h_steps = np.arange(1, len(fc_denorm) + 1)
        bars = ax_r.bar(h_steps, fc_denorm, color=color, alpha=0.8)
        ax_r.axhline(0, color="black", lw=0.6)
        ax_r.set_xlabel("h (шаг × 30min)")
        ax_r.set_ylabel("contribution (original units)")
        ax_r.grid(True, alpha=0.3, axis="y")

        ax_r2 = ax_r.twinx()
        ax_r2.plot(h_steps, pct, "k--o", ms=3, lw=0.8, label="% итога")
        ax_r2.set_ylabel("% of total forecast", fontsize=7)
        ax_r2.tick_params(labelsize=7)
        ax_r2.axhline(100, color="red", ls=":", lw=0.6, alpha=0.4)

        ax_r.set_title(f"{label}\nForecast contribution  (денормированный)", fontsize=8)

        fc_range = max(np.abs(fc_denorm).max() * 0.05, 1.0)
        for bar, v in zip(bars, fc_denorm):
            ax_r.text(
                bar.get_x() + bar.get_width() / 2,
                v + fc_range * np.sign(v) if v != 0 else fc_range,
                f"{v:,.0f}", ha="center",
                va="bottom" if v >= 0 else "top", fontsize=6.5,
            )

    plt.tight_layout()
    out_path = f"diagnostics/nhits_decomp_route{route_id}.png"
    plt.savefig(out_path, dpi=130, bbox_inches="tight")
    plt.close()
    print(f"✅  {out_path}")

    return {"final_fc": final_fc, "stack_fc": stack_fc, "stack_bc": stack_bc}


# ============================================================
# ИСПОЛЬЗОВАНИЕ
# ============================================================

# Конфиг должен совпадать с тем, что был при обучении
N_BLOCKS_CONFIG   = [2, 6, 3]
N_FREQ_DOWN       = [24, 8, 1]
FREQ_LABELS       = [
    "Stack 1 — Macro/trend  (MaxPool×24, ~12h grain)",
    "Stack 2 — 4h cycle     (MaxPool×8,  доминирующий по диагностике)",
    "Stack 3 — 30min detail (MaxPool×1,  полное разрешение)",
]


# plot_nhits_decomposition(
#     nf_fitted         = nf_full,
#     route_id          = 702,
#     train_nf_df       = train_nf_full,
#     train_df_raw      = train_df,
#     n_blocks_config   = N_BLOCKS_CONFIG,    # [2, 6, 3]
#     n_freq_downsample = N_FREQ_DOWN,        # [24, 8, 1]
#     input_size        = INPUT_SIZE,         # 96 — должен совпадать с тем, что при обучении
#     freq_labels       = FREQ_LABELS,
#     context_show      = 96,
# )

# Пакетный прогон по проблемным маршрутам из диагностики
PROBLEM_ROUTES = [485, 192, 702, 999, 446, 782, 725, 768, 445, 817, 329, 594, 123, 948, 263]
for rid in PROBLEM_ROUTES:
    plot_nhits_decomposition(
        nf_fitted         = nf_full,
        route_id          = rid,
        train_nf_df       = train_nf_full,
        train_df_raw      = train_df,
        n_blocks_config   = N_BLOCKS_CONFIG,
        n_freq_downsample = N_FREQ_DOWN,
        freq_labels       = FREQ_LABELS,
        input_size        = INPUT_SIZE, 
        context_show      = 96,
    )

In [ ]:
# ============================================================
# N-HiTS — Bracket Hyperparameter Search
# Phase 1: 1-fold  (25 configs  × ~2 min  = ~50 min)
# Phase 2: 3-fold  (top-6       × ~6 min  = ~36 min)
# Phase 3: 5-fold  (top-2       × ~10 min = ~20 min)
# Final:   full fit + test predictions         (~15 min)
# Total: ~2 hours
# ============================================================

import json, time, warnings
import numpy as np
import pandas as pd
import torch
from datetime import datetime
from tqdm import tqdm
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MQLoss

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

# ── Paths & constants ──
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8
N_FOLDS        = 5
TOTAL_VAL_PTS  = N_FOLDS * FORECAST_STEPS
CONTEXT_LEN    = 2048
FREQ           = "30min"
BATCH_SIZE     = 64
RESULTS_PATH   = "nhits_search_results.csv"

# Текущий бейзлайн — по нему режем в Phase 1
BASELINE_TOTAL = 0.3676   # global raw total твоей лучшей модели
PRUNE_FACTOR   = 1.08     # отсекаем если >baseline × 1.08

# ── Device ──
if torch.backends.mps.is_available():
    ACCELERATOR = "mps"; print("MPS ✅")
elif torch.cuda.is_available():
    ACCELERATOR = "gpu"; print("GPU ✅")
else:
    ACCELERATOR = "cpu"; print("CPU")


# ============================================================
# ДАННЫЕ И ФИЧИ
# ============================================================
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id","timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id","timestamp"]).reset_index(drop=True)
status_cols = sorted([c for c in train_df.columns if c.startswith("status_")])
print(f"Train: {train_df.shape}  Test: {test_df.shape}  status_cols: {status_cols}")

FUTR_EXOG = ["hour_sin","hour_cos","dow_sin","dow_cos","is_weekend"]

def add_calendar_features(df, ts_col="ds"):
    ts = pd.to_datetime(df[ts_col]); df = df.copy()
    df["hour_sin"]   = np.sin(2*np.pi*ts.dt.hour/24)
    df["hour_cos"]   = np.cos(2*np.pi*ts.dt.hour/24)
    df["dow_sin"]    = np.sin(2*np.pi*ts.dt.dayofweek/7)
    df["dow_cos"]    = np.cos(2*np.pi*ts.dt.dayofweek/7)
    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(float)
    return df

# Лаги статусов (из корреляционного анализа)
KEY_STATUS_LAGS = {"status_3": [1,2,4], "status_2": [2,4,8]}

def add_status_lag_features(nf_df):
    nf_df = nf_df.copy()
    for col, lags in KEY_STATUS_LAGS.items():
        if col not in nf_df.columns: continue
        for lag in lags:
            nf_df[f"{col}_lag{lag}"] = (
                nf_df.groupby("unique_id")[col]
                .transform(lambda s, l=lag: s.shift(l).bfill().fillna(0))
            )
    nf_df["status_3_roll4"] = (
        nf_df.groupby("unique_id")["status_3"]
        .transform(lambda s: s.rolling(4, min_periods=1).mean().fillna(0))
    )
    return nf_df

STATUS_LAG_COLS = [f"{c}_lag{l}" for c,lags in KEY_STATUS_LAGS.items()
                   for l in lags] + ["status_3_roll4"]

def add_lag_features(nf_df):
    nf_df = nf_df.sort_values(["unique_id","ds"]).copy()
    g = nf_df.groupby("unique_id")["y"]
    nf_df["lag_1"]       = g.transform(lambda s: s.shift(1).bfill().fillna(0))
    nf_df["lag_2"]       = g.transform(lambda s: s.shift(2).bfill().fillna(0))
    nf_df["lag_4"]       = g.transform(lambda s: s.shift(4).bfill().fillna(0))
    nf_df["lag_48"]      = g.transform(lambda s: s.shift(48).bfill().fillna(0))
    nf_df["lag_336"]     = g.transform(lambda s: s.shift(336).bfill().fillna(0))
    nf_df["roll_std_8"]  = g.transform(lambda s: s.rolling(8,  min_periods=1).std().fillna(0))
    nf_df["roll_std_48"] = g.transform(lambda s: s.rolling(48, min_periods=1).std().fillna(0))
    nf_df["roll_max_8"]  = g.transform(lambda s: s.rolling(8,  min_periods=1).max().fillna(0))
    nf_df["diff_1"]      = g.transform(lambda s: s.diff(1).fillna(0))
    return nf_df

HIST_EXOG_EXTRA = ["lag_1","lag_2","lag_4","lag_48","lag_336",
                   "roll_std_8","roll_std_48","roll_max_8","diff_1"]
HIST_EXOG = status_cols + HIST_EXOG_EXTRA + STATUS_LAG_COLS

def to_nf_df(df, min_len=32, context_len=CONTEXT_LEN):
    rows = []
    for route_id, grp in df.groupby("route_id"):
        grp = (grp.sort_values("timestamp")
               .set_index("timestamp")[[TARGET_COL]+status_cols]
               .asfreq(FREQ).interpolate(method="time").bfill().ffill()
               .tail(context_len).reset_index())
        if len(grp) < min_len: continue
        grp.insert(0,"unique_id",route_id)
        grp = grp.rename(columns={"timestamp":"ds", TARGET_COL:"y"})
        rows.append(grp)
    nf_df = pd.concat(rows, ignore_index=True)
    nf_df = add_calendar_features(nf_df)
    nf_df = add_lag_features(nf_df)
    nf_df = add_status_lag_features(nf_df)
    return nf_df

def make_futr_df(route_ids, src_df):
    rows = []
    for rid in route_ids:
        last_ts = src_df[src_df["route_id"]==rid]["timestamp"].max()
        for ts in pd.date_range(last_ts+pd.Timedelta("30min"), periods=FORECAST_STEPS, freq=FREQ):
            rows.append({"unique_id":rid, "ds":ts})
    return add_calendar_features(pd.DataFrame(rows))

def get_pred_col(pred_df):
    for c in ["NHITS-median","NHITS-q-0.50","NHITS"]:
        if c in pred_df.columns: return c
    skip = {"unique_id","ds","y","cutoff"}
    nums = [c for c in pred_df.columns if c not in skip
            and pd.api.types.is_numeric_dtype(pred_df[c])]
    meds = [c for c in nums if any(x in c for x in ["median","0.5","50"])]
    return (meds or nums)[0]

def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp-yt).sum() / s
    rbias = np.abs(yp.sum()/s - 1)
    return float(wape+rbias), float(wape), float(rbias)

def json_safe(obj):
    if isinstance(obj, (np.bool_,)):   return bool(obj)
    if isinstance(obj, np.integer):    return int(obj)
    if isinstance(obj, np.floating):   return float(obj)
    if isinstance(obj, np.ndarray):    return obj.tolist()
    if isinstance(obj, list):          return [json_safe(x) for x in obj]
    if isinstance(obj, dict):          return {k: json_safe(v) for k,v in obj.items()}
    return obj


# ============================================================
# ПОИСКОВОЕ ПРОСТРАНСТВО — 25 точечных конфигов
# Каждый проверяет конкретную гипотезу из диагностики
# ============================================================
#
# Что мы ищем:
# [A] input_size: 96/144/192/288/336 — насколько длинный контекст нужен
# [B] n_freq_downsample: как разложить по частотам под 4h-доминирование
# [C] n_blocks: распределение ёмкости между стеками
# [D] mlp_units: ширина сети
# [E] max_steps: длина обучения
#
SEARCH_CONFIGS = [
    # ── Baseline (без новых фич, для сравнения) ──
    {"id": "A0_baseline",
     "input_size":336, "n_freq_downsample":[48,8,1],
     "n_blocks":[4,4,4], "mlp_units":[[512,512]]*3, "max_steps":800},

    # ── [A] Короткий контекст: 96 = 2 суток ──
    {"id": "A1_96_24-8-1_2-6-3",
     "input_size":96, "n_freq_downsample":[24,8,1],
     "n_blocks":[2,6,3], "mlp_units":[[512,512]]*3, "max_steps":800},

    {"id": "A2_96_16-8-1_2-6-3",
     "input_size":96, "n_freq_downsample":[16,8,1],
     "n_blocks":[2,6,3], "mlp_units":[[512,512]]*3, "max_steps":800},

    {"id": "A3_96_32-8-1_2-6-3",
     "input_size":96, "n_freq_downsample":[32,8,1],
     "n_blocks":[2,6,3], "mlp_units":[[512,512]]*3, "max_steps":800},

    # ── [A] Средний контекст: 144 = 3 суток ──
    {"id": "B1_144_24-8-1_2-6-3",
     "input_size":144, "n_freq_downsample":[24,8,1],
     "n_blocks":[2,6,3], "mlp_units":[[512,512]]*3, "max_steps":800},

    {"id": "B2_144_48-8-1_2-6-3",
     "input_size":144, "n_freq_downsample":[48,8,1],
     "n_blocks":[2,6,3], "mlp_units":[[512,512]]*3, "max_steps":800},

    {"id": "B3_144_36-4-1_2-6-3",
     "input_size":144, "n_freq_downsample":[36,4,1],
     "n_blocks":[2,6,3], "mlp_units":[[512,512]]*3, "max_steps":800},

    # ── [A] 4 суток: 192 ──
    {"id": "C1_192_48-8-1_2-6-3",
     "input_size":192, "n_freq_downsample":[48,8,1],
     "n_blocks":[2,6,3], "mlp_units":[[512,512]]*3, "max_steps":800},

    {"id": "C2_192_24-8-1_2-6-3",
     "input_size":192, "n_freq_downsample":[24,8,1],
     "n_blocks":[2,6,3], "mlp_units":[[512,512]]*3, "max_steps":800},

    {"id": "C3_192_32-8-1_2-6-3",
     "input_size":192, "n_freq_downsample":[32,8,1],
     "n_blocks":[2,6,3], "mlp_units":[[512,512]]*3, "max_steps":800},

    # ── [A] 6 суток: 288 ──
    {"id": "D1_288_48-8-1_2-6-3",
     "input_size":288, "n_freq_downsample":[48,8,1],
     "n_blocks":[2,6,3], "mlp_units":[[512,512]]*3, "max_steps":800},

    {"id": "D2_288_96-8-1_2-6-3",
     "input_size":288, "n_freq_downsample":[96,8,1],
     "n_blocks":[2,6,3], "mlp_units":[[512,512]]*3, "max_steps":800},

    # ── [A] Неделя: 336 с новыми блоками ──
    {"id": "E1_336_48-8-1_2-6-3",
     "input_size":336, "n_freq_downsample":[48,8,1],
     "n_blocks":[2,6,3], "mlp_units":[[512,512]]*3, "max_steps":800},

    # ── [C] n_blocks: акцент на разных стеках ──
    {"id": "F1_192_48-8-1_2-8-4",
     "input_size":192, "n_freq_downsample":[48,8,1],
     "n_blocks":[2,8,4], "mlp_units":[[512,512]]*3, "max_steps":800},

    {"id": "F2_192_48-8-1_3-6-3",
     "input_size":192, "n_freq_downsample":[48,8,1],
     "n_blocks":[3,6,3], "mlp_units":[[512,512]]*3, "max_steps":800},

    {"id": "F3_144_24-8-1_2-8-4",
     "input_size":144, "n_freq_downsample":[24,8,1],
     "n_blocks":[2,8,4], "mlp_units":[[512,512]]*3, "max_steps":800},

    # ── [D] mlp_units ──
    {"id": "G1_192_48-8-1_256",
     "input_size":192, "n_freq_downsample":[48,8,1],
     "n_blocks":[2,6,3], "mlp_units":[[256,256]]*3, "max_steps":800},

    {"id": "G2_192_48-8-1_1024",
     "input_size":192, "n_freq_downsample":[48,8,1],
     "n_blocks":[2,6,3], "mlp_units":[[1024,512]]*3, "max_steps":800},

    {"id": "G3_144_24-8-1_256",
     "input_size":144, "n_freq_downsample":[24,8,1],
     "n_blocks":[2,6,3], "mlp_units":[[256,256]]*3, "max_steps":800},

    # ── [E] max_steps ──
    {"id": "H1_192_48-8-1_1200steps",
     "input_size":192, "n_freq_downsample":[48,8,1],
     "n_blocks":[2,6,3], "mlp_units":[[512,512]]*3, "max_steps":1200},

    {"id": "H2_144_24-8-1_1200steps",
     "input_size":144, "n_freq_downsample":[24,8,1],
     "n_blocks":[2,6,3], "mlp_units":[[512,512]]*3, "max_steps":1200},

    # ── 4-стековые варианты ──
    {"id": "I1_192_48-16-4-1_4stack",
     "input_size":192, "n_freq_downsample":[48,16,4,1],
     "n_blocks":[2,4,4,2], "mlp_units":[[512,512]]*4, "max_steps":800},

    {"id": "I2_288_48-16-4-1_4stack",
     "input_size":288, "n_freq_downsample":[48,16,4,1],
     "n_blocks":[2,4,4,2], "mlp_units":[[512,512]]*4, "max_steps":800},

    # ── Комбинированные лучшие гипотезы ──
    {"id": "J1_144_48-8-1_2-8-4_1024",
     "input_size":144, "n_freq_downsample":[48,8,1],
     "n_blocks":[2,8,4], "mlp_units":[[1024,512]]*3, "max_steps":800},

    {"id": "J2_192_48-8-1_2-8-4_1200",
     "input_size":192, "n_freq_downsample":[48,8,1],
     "n_blocks":[2,8,4], "mlp_units":[[512,512]]*3, "max_steps":1200},
]

print(f"\nВсего конфигов: {len(SEARCH_CONFIGS)}")
print(f"Оценка времени:")
print(f"  Phase 1 (1-fold):  {len(SEARCH_CONFIGS)} × 2 мин = ~{len(SEARCH_CONFIGS)*2} мин")
print(f"  Phase 2 (3-fold):  6 × 6 мин  = ~36 мин")
print(f"  Phase 3 (5-fold):  2 × 10 мин = ~20 мин")
print(f"  Final fit+predict: ~15 мин")
print(f"  ИТОГО: ~{len(SEARCH_CONFIGS)*2 + 36 + 20 + 15} мин ≈ {(len(SEARCH_CONFIGS)*2+71)/60:.1f} ч")


# ============================================================
# ПОДГОТОВКА ДАННЫХ (один раз per input_size)
# ============================================================
print("\nПодготовка датафреймов...")
unique_input_sizes = sorted(set(c["input_size"] for c in SEARCH_CONFIGS))
nf_dfs = {}
for inp in unique_input_sizes:
    min_len = TOTAL_VAL_PTS + inp + 1
    df_inp  = to_nf_df(train_df, min_len=min_len, context_len=CONTEXT_LEN)
    nf_dfs[inp] = df_inp
    print(f"  input_size={inp:>4}: {df_inp['unique_id'].nunique()} routes, shape={df_inp.shape}")


# ============================================================
# CORE: запуск одного конфига
# ============================================================
def run_one(cfg, n_windows, train_nf_df):
    t0 = time.time()
    n_stacks = len(cfg["n_freq_downsample"])
    model = NHITS(
        h                   = FORECAST_STEPS,
        input_size          = cfg["input_size"],
        loss                = MQLoss(level=[80]),
        n_freq_downsample   = cfg["n_freq_downsample"],
        mlp_units           = cfg["mlp_units"],
        n_blocks            = cfg["n_blocks"],
        max_steps           = cfg["max_steps"],
        batch_size          = BATCH_SIZE,
        accelerator         = ACCELERATOR,
        hist_exog_list      = HIST_EXOG,
        futr_exog_list      = FUTR_EXOG,
        scaler_type         = "robust",
        enable_progress_bar = False,
    )
    nf = NeuralForecast(models=[model], freq=FREQ)
    cv = nf.cross_validation(df=train_nf_df, n_windows=n_windows,
                              step_size=FORECAST_STEPS, refit=True)

    pred_col = get_pred_col(cv)
    total, wape, rbias = wape_rbias(cv["y"].values,
                                    np.clip(cv[pred_col].values, 0, None))
    # per-fold
    fold_rows = []
    for i, cut in enumerate(sorted(cv["cutoff"].unique())):
        sub = cv[cv["cutoff"]==cut]
        t,w,r = wape_rbias(sub["y"].values, np.clip(sub[pred_col].values,0,None))
        fold_rows.append(f"fold{i+1}={t:.4f}")

    return {
        "total":      total,
        "wape":       wape,
        "rbias":      rbias,
        "fold_str":   " | ".join(fold_rows),
        "elapsed":    time.time()-t0,
        "n_windows":  n_windows,
    }


def save_result(cfg, res, phase):
    row = {
        "phase": phase,
        "id": cfg["id"],
        "total":      res["total"],
        "wape":       res["wape"],
        "rbias":      res["rbias"],
        "n_windows":  res["n_windows"],
        "elapsed_s":  round(res["elapsed"],1),
        "input_size": cfg["input_size"],
        "freq_down":  str(cfg["n_freq_downsample"]),
        "n_blocks":   str(cfg["n_blocks"]),
        "mlp":        str(cfg["mlp_units"][0]),
        "max_steps":  cfg["max_steps"],
    }
    try:
        existing = pd.read_csv(RESULTS_PATH)
        pd.concat([existing, pd.DataFrame([row])], ignore_index=True).to_csv(RESULTS_PATH, index=False)
    except FileNotFoundError:
        pd.DataFrame([row]).to_csv(RESULTS_PATH, index=False)


def print_banner(phase, done, total, best_id, best_total, phase_start):
    elapsed  = time.time() - phase_start
    avg      = elapsed / max(done, 1)
    left     = (total - done) * avg
    bar_done = "█" * done
    bar_left = "░" * (total - done)
    print(f"\n  ┌─ Phase {phase} Progress ──────────────────────────────────")
    print(f"  │  [{bar_done}{bar_left}] {done}/{total}  "
          f"elapsed={elapsed/60:.1f}m  left≈{left/60:.1f}m")
    if best_id:
        print(f"  │  🏆 Лучший: {best_id}  total={best_total:.4f}")
    print(f"  └────────────────────────────────────────────────────────\n")


# ============================================================
# PHASE 1 — 1-fold скрининг
# ============================================================
print("\n" + "="*65)
print(f"PHASE 1 — 1-fold скрининг  ({len(SEARCH_CONFIGS)} конфигов)")
print(f"Прунинг: total > {BASELINE_TOTAL:.4f} × {PRUNE_FACTOR} = {BASELINE_TOTAL*PRUNE_FACTOR:.4f}")
print("="*65)

p1_results  = []
best1_total = float("inf")
best1_id    = None
pruned_cnt  = 0
P1_START    = time.time()

for i, cfg in enumerate(SEARCH_CONFIGS):
    print(f"\n[Phase1 {i+1}/{len(SEARCH_CONFIGS)}]  {cfg['id']}")
    print(f"  input={cfg['input_size']}  freq={cfg['n_freq_downsample']}  "
          f"blocks={cfg['n_blocks']}  mlp={cfg['mlp_units'][0]}  steps={cfg['max_steps']}")

    try:
        res    = run_one(cfg, n_windows=1, train_nf_df=nf_dfs[cfg["input_size"]])
        pruned = res["total"] > BASELINE_TOTAL * PRUNE_FACTOR

        if not pruned and res["total"] < best1_total:
            best1_total = res["total"]
            best1_id    = cfg["id"]
            mark = "  ← 🏆 НОВЫЙ ЛУЧШИЙ"
        else:
            mark = "  ✂️ PRUNED" if pruned else ""

        pruned_cnt += pruned
        p1_results.append({"cfg": cfg, "res": res, "pruned": pruned})
        save_result(cfg, res, phase=1)

        tag = "✂️  PRUNED" if pruned else "✅"
        print(f"  {tag}  total={res['total']:.4f}  wape={res['wape']:.4f}  "
              f"rbias={res['rbias']:.4f}  time={res['elapsed']:.0f}s{mark}")
        print(f"  {res['fold_str']}")

    except Exception as e:
        print(f"  ❌ ERROR: {e}")
        p1_results.append({"cfg": cfg, "res": {"total": float("nan")}, "pruned": True})
        pruned_cnt += 1

    print_banner(1, i+1, len(SEARCH_CONFIGS), best1_id, best1_total, P1_START)


# Отбор топ-6 для Phase 2
survived1 = sorted(
    [(r["cfg"], r["res"]["total"]) for r in p1_results
     if not r["pruned"] and not np.isnan(r["res"]["total"])],
    key=lambda x: x[1]
)
top6 = [cfg for cfg, _ in survived1[:6]]

print(f"\n{'='*65}")
print(f"Phase 1 итог: выжило {len(survived1)}/{len(SEARCH_CONFIGS)}, "
      f"pruned={pruned_cnt}")
print(f"Топ-6 → Phase 2:")
for i, (cfg, tot) in enumerate(survived1[:6]):
    print(f"  #{i+1}  total={tot:.4f}  {cfg['id']}")


# ============================================================
# PHASE 2 — 3-fold
# ============================================================
print("\n" + "="*65)
print(f"PHASE 2 — 3-fold  ({len(top6)} конфигов)")
print("="*65)

p2_results  = []
best2_total = float("inf")
best2_id    = None
P2_START    = time.time()

for i, cfg in enumerate(top6):
    print(f"\n[Phase2 {i+1}/{len(top6)}]  {cfg['id']}")
    try:
        res = run_one(cfg, n_windows=3, train_nf_df=nf_dfs[cfg["input_size"]])

        if res["total"] < best2_total:
            best2_total = res["total"]
            best2_id    = cfg["id"]
            mark = "  ← 🏆 НОВЫЙ ЛУЧШИЙ"
        else:
            mark = ""

        p2_results.append({"cfg": cfg, "res": res})
        save_result(cfg, res, phase=2)

        print(f"  ✅  total={res['total']:.4f}  wape={res['wape']:.4f}  "
              f"rbias={res['rbias']:.4f}  time={res['elapsed']:.0f}s{mark}")
        print(f"  {res['fold_str']}")

    except Exception as e:
        print(f"  ❌ ERROR: {e}")
        p2_results.append({"cfg": cfg, "res": {"total": float("nan")}})

    print_banner(2, i+1, len(top6), best2_id, best2_total, P2_START)


survived2 = sorted(
    [(r["cfg"], r["res"]["total"]) for r in p2_results
     if not np.isnan(r["res"]["total"])],
    key=lambda x: x[1]
)
top2 = [cfg for cfg, _ in survived2[:2]]

print(f"\nPhase 2 итог. Топ-2 → Phase 3:")
for i, (cfg, tot) in enumerate(survived2[:2]):
    print(f"  #{i+1}  total={tot:.4f}  {cfg['id']}")


# ============================================================
# PHASE 3 — полный 5-fold CV
# ============================================================
print("\n" + "="*65)
print(f"PHASE 3 — full 5-fold CV  ({len(top2)} конфигов)")
print("="*65)

p3_results  = []
best3_total = float("inf")
best_cfg    = None
P3_START    = time.time()

for i, cfg in enumerate(top2):
    print(f"\n[Phase3 {i+1}/{len(top2)}]  {cfg['id']}")
    try:
        res = run_one(cfg, n_windows=N_FOLDS, train_nf_df=nf_dfs[cfg["input_size"]])

        if res["total"] < best3_total:
            best3_total = res["total"]
            best_cfg    = cfg
            mark = "  ← 🏆 ПОБЕДИТЕЛЬ"
        else:
            mark = ""

        p3_results.append({"cfg": cfg, "res": res})
        save_result(cfg, res, phase=3)

        print(f"  ✅  total={res['total']:.4f}  wape={res['wape']:.4f}  "
              f"rbias={res['rbias']:.4f}  time={res['elapsed']:.0f}s{mark}")
        print(f"  {res['fold_str']}")

    except Exception as e:
        print(f"  ❌ ERROR: {e}")

# Если phase3 пустая — берём лучший из phase2
if best_cfg is None:
    best_cfg    = top2[0]
    best3_total = survived2[0][1]

print(f"\n{'='*65}")
print(f"🏆 ЛУЧШИЙ КОНФИГ: {best_cfg['id']}  total={best3_total:.4f}")
print(f"  input_size        = {best_cfg['input_size']}")
print(f"  n_freq_downsample = {best_cfg['n_freq_downsample']}")
print(f"  n_blocks          = {best_cfg['n_blocks']}")
print(f"  mlp_units         = {best_cfg['mlp_units']}")
print(f"  max_steps         = {best_cfg['max_steps']}")
print(f"  vs baseline:        {BASELINE_TOTAL:.4f}  Δ={best3_total-BASELINE_TOTAL:+.4f}")


# ============================================================
# ФИНАЛЬНЫЙ ФИТ + САБМИТ
# ============================================================
print("\n" + "="*65)
print("FINAL FIT + TEST PREDICTION")
print("="*65)

train_nf_full = to_nf_df(train_df, min_len=best_cfg["input_size"]+1, context_len=CONTEXT_LEN)
test_route_ids = test_df["route_id"].unique().tolist()

n_stacks_final = len(best_cfg["n_freq_downsample"])
final_model = NHITS(
    h                   = FORECAST_STEPS,
    input_size          = best_cfg["input_size"],
    loss                = MQLoss(level=[80]),
    n_freq_downsample   = best_cfg["n_freq_downsample"],
    mlp_units           = best_cfg["mlp_units"],
    n_blocks            = best_cfg["n_blocks"],
    max_steps           = best_cfg["max_steps"],
    batch_size          = BATCH_SIZE,
    accelerator         = ACCELERATOR,
    hist_exog_list      = HIST_EXOG,
    futr_exog_list      = FUTR_EXOG,
    scaler_type         = "robust",
    enable_progress_bar = True,
)

print(f"Fitting on {train_nf_full['unique_id'].nunique()} routes...")
nf_full = NeuralForecast(models=[final_model], freq=FREQ)
nf_full.fit(train_nf_full)

print("Predicting...")
futr_df      = make_futr_df(test_route_ids, train_df)
test_pred_df = nf_full.predict(futr_df=futr_df)
test_pred_col = get_pred_col(test_pred_df)
test_preds   = {rid: np.clip(g.sort_values("ds")[test_pred_col].values, 0, None)
                for rid, g in test_pred_df.groupby("unique_id")}

# CV для calib_scale (по финальному конфигу из phase3)
print("CV для calib_scale...")
cv_final = nf_full.cross_validation(
    df=nf_dfs[best_cfg["input_size"]], n_windows=N_FOLDS,
    step_size=FORECAST_STEPS, refit=True
)
pc = get_pred_col(cv_final)
y_true_cv = cv_final["y"].values
y_pred_cv = np.clip(cv_final[pc].values, 0, None)
calib_scale = float(y_true_cv.sum() / (y_pred_cv.sum() + 1e-9))
total_raw, wape_raw, rbias_raw = wape_rbias(y_true_cv, y_pred_cv)
print(f"CV  total={total_raw:.4f}  wape={wape_raw:.4f}  rbias={rbias_raw:.4f}")
print(f"calib_scale = {calib_scale:.4f}")

# Сабмиты
predictions_raw = {}
for rid in tqdm(test_route_ids, desc="Build submission"):
    preds = test_preds.get(rid, np.zeros(FORECAST_STEPS))
    for j, (_, row) in enumerate(
        test_df[test_df["route_id"]==rid].sort_values("timestamp").iterrows()
    ):
        predictions_raw[row["id"]] = float(max(0.0, preds[j] if j < len(preds) else preds[-1]))

sub_raw = (pd.DataFrame(list(predictions_raw.items()), columns=["id","y_pred"])
           .sort_values("id").reset_index(drop=True))
sub_cal = sub_raw.copy()
sub_cal["y_pred"] = np.clip(sub_cal["y_pred"] * calib_scale, 0, None)

sub_raw.to_csv("submission_nhits_search_raw.csv", index=False)
sub_cal.to_csv("submission_nhits_search_cal.csv", index=False)

print(f"\n✅ submission_nhits_search_raw.csv")
print(f"✅ submission_nhits_search_cal.csv")
print(f"✅ {RESULTS_PATH}")
print(f"\n── Raw ──\n{sub_raw['y_pred'].describe().round(0)}")
print(f"\n── Calibrated (scale={calib_scale:.4f}) ──\n{sub_cal['y_pred'].describe().round(0)}")

total_time = (time.time() - P1_START) / 60
print(f"\n⏱ Общее время: {total_time:.1f} мин")

In [ ]:
# ============================================================
# N-HiTS Bracket Search — ПОЛНЫЙ САМОДОСТАТОЧНЫЙ СКРИПТ
# Запускается с нуля после рестарта ядра
# ============================================================

import gc, json, time, warnings
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MQLoss

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

# ── Config ──
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8
N_FOLDS        = 5
TOTAL_VAL_PTS  = N_FOLDS * FORECAST_STEPS
CONTEXT_LEN    = 2048
FREQ           = "30min"
BATCH_SIZE     = 64
RESULTS_PATH   = "nhits_search_results.csv"
BASELINE_TOTAL = 0.3676
PRUNE_FACTOR   = 1.08   # порог = 0.3970

# ── Device ──
if torch.backends.mps.is_available():
    ACCELERATOR = "mps"; print("MPS ✅")
elif torch.cuda.is_available():
    ACCELERATOR = "gpu"; print("GPU ✅")
else:
    ACCELERATOR = "cpu"; print("CPU")


# ============================================================
# ДАННЫЕ И ФИЧИ
# ============================================================
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id","timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id","timestamp"]).reset_index(drop=True)
status_cols = sorted([c for c in train_df.columns if c.startswith("status_")])
print(f"Train: {train_df.shape}  Test: {test_df.shape}")

FUTR_EXOG = ["hour_sin","hour_cos","dow_sin","dow_cos","is_weekend"]

def add_calendar_features(df, ts_col="ds"):
    ts = pd.to_datetime(df[ts_col]); df = df.copy()
    df["hour_sin"]   = np.sin(2*np.pi*ts.dt.hour/24)
    df["hour_cos"]   = np.cos(2*np.pi*ts.dt.hour/24)
    df["dow_sin"]    = np.sin(2*np.pi*ts.dt.dayofweek/7)
    df["dow_cos"]    = np.cos(2*np.pi*ts.dt.dayofweek/7)
    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(float)
    return df

KEY_STATUS_LAGS = {"status_3": [1,2,4], "status_2": [2,4,8]}

def add_status_lag_features(nf_df):
    nf_df = nf_df.copy()
    for col, lags in KEY_STATUS_LAGS.items():
        if col not in nf_df.columns: continue
        for lag in lags:
            nf_df[f"{col}_lag{lag}"] = (
                nf_df.groupby("unique_id")[col]
                .transform(lambda s, l=lag: s.shift(l).bfill().fillna(0))
            )
    nf_df["status_3_roll4"] = (
        nf_df.groupby("unique_id")["status_3"]
        .transform(lambda s: s.rolling(4, min_periods=1).mean().fillna(0))
    )
    return nf_df

STATUS_LAG_COLS = [f"{c}_lag{l}" for c,lags in KEY_STATUS_LAGS.items()
                   for l in lags] + ["status_3_roll4"]

def add_lag_features(nf_df):
    nf_df = nf_df.sort_values(["unique_id","ds"]).copy()
    g = nf_df.groupby("unique_id")["y"]
    nf_df["lag_1"]       = g.transform(lambda s: s.shift(1).bfill().fillna(0))
    nf_df["lag_2"]       = g.transform(lambda s: s.shift(2).bfill().fillna(0))
    nf_df["lag_4"]       = g.transform(lambda s: s.shift(4).bfill().fillna(0))
    nf_df["lag_48"]      = g.transform(lambda s: s.shift(48).bfill().fillna(0))
    nf_df["lag_336"]     = g.transform(lambda s: s.shift(336).bfill().fillna(0))
    nf_df["roll_std_8"]  = g.transform(lambda s: s.rolling(8,  min_periods=1).std().fillna(0))
    nf_df["roll_std_48"] = g.transform(lambda s: s.rolling(48, min_periods=1).std().fillna(0))
    nf_df["roll_max_8"]  = g.transform(lambda s: s.rolling(8,  min_periods=1).max().fillna(0))
    nf_df["diff_1"]      = g.transform(lambda s: s.diff(1).fillna(0))
    return nf_df

HIST_EXOG_EXTRA = ["lag_1","lag_2","lag_4","lag_48","lag_336",
                   "roll_std_8","roll_std_48","roll_max_8","diff_1"]
HIST_EXOG = status_cols + HIST_EXOG_EXTRA + STATUS_LAG_COLS

def to_nf_df(df, min_len=32, context_len=CONTEXT_LEN):
    rows = []
    for route_id, grp in df.groupby("route_id"):
        grp = (grp.sort_values("timestamp")
               .set_index("timestamp")[[TARGET_COL]+status_cols]
               .asfreq(FREQ).interpolate(method="time").bfill().ffill()
               .tail(context_len).reset_index())
        if len(grp) < min_len: continue
        grp.insert(0, "unique_id", route_id)
        grp = grp.rename(columns={"timestamp":"ds", TARGET_COL:"y"})
        rows.append(grp)
    nf_df = pd.concat(rows, ignore_index=True)
    nf_df = add_calendar_features(nf_df)
    nf_df = add_lag_features(nf_df)
    nf_df = add_status_lag_features(nf_df)
    return nf_df

def make_futr_df(route_ids, src_df):
    rows = []
    for rid in route_ids:
        last_ts = src_df[src_df["route_id"]==rid]["timestamp"].max()
        for ts in pd.date_range(last_ts+pd.Timedelta("30min"),
                                periods=FORECAST_STEPS, freq=FREQ):
            rows.append({"unique_id": rid, "ds": ts})
    return add_calendar_features(pd.DataFrame(rows))

def get_pred_col(pred_df):
    for c in ["NHITS-median","NHITS-q-0.50","NHITS"]:
        if c in pred_df.columns: return c
    skip = {"unique_id","ds","y","cutoff"}
    nums = [c for c in pred_df.columns
            if c not in skip and pd.api.types.is_numeric_dtype(pred_df[c])]
    meds = [c for c in nums if any(x in c for x in ["median","0.5","50"])]
    return (meds or nums)[0]

def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    return float((np.abs(yp-yt).sum()/s) + abs(yp.sum()/s - 1)), \
           float(np.abs(yp-yt).sum()/s), \
           float(abs(yp.sum()/s - 1))

def free_memory():
    gc.collect()
    if torch.backends.mps.is_available():   torch.mps.empty_cache()
    elif torch.cuda.is_available():          torch.cuda.empty_cache()

def save_result(cfg, res, phase):
    row = {
        "phase": phase, "id": cfg["id"],
        "total": res.get("total", float("nan")),
        "wape":  res.get("wape",  float("nan")),
        "rbias": res.get("rbias", float("nan")),
        "n_windows": res.get("n_windows", 0),
        "elapsed_s": round(res.get("elapsed", 0), 1),
        "input_size": cfg["input_size"],
        "freq_down":  str(cfg["n_freq_downsample"]),
        "n_blocks":   str(cfg["n_blocks"]),
        "mlp":        str(cfg["mlp_units"][0]),
        "max_steps":  cfg["max_steps"],
    }
    try:
        pd.concat([pd.read_csv(RESULTS_PATH), pd.DataFrame([row])],
                  ignore_index=True).to_csv(RESULTS_PATH, index=False)
    except FileNotFoundError:
        pd.DataFrame([row]).to_csv(RESULTS_PATH, index=False)

def print_banner(done, total, best_id, best_total, t_start, phase):
    elapsed = time.time() - t_start
    left    = (total - done) * (elapsed / max(done, 1))
    bar     = "█"*done + "░"*(total-done)
    print(f"\n  ┌─ Phase {phase} [{bar}] {done}/{total}")
    print(f"  │  elapsed={elapsed/60:.1f}m  left≈{left/60:.1f}m")
    if best_id: print(f"  │  🏆 {best_id}  total={best_total:.4f}")
    print(f"  └{'─'*54}\n")


# ============================================================
# КОНФИГИ К ПРОГОНУ (только те, что ещё не запускались)
# ============================================================
# Из Phase1 уже известны 1–12. Пропускаем input≥192 (доказано хуже).
# Оставляем только input=96 и один 144 как cross-check.
TODO_CONFIGS = [
    # ── Оставшиеся из исходного плана (input=144, один 336) ──
    {"id":"E1_336_48-8-1_2-6-3",     "input_size":336, "n_freq_downsample":[48,8,1],  "n_blocks":[2,6,3],  "mlp_units":[[512,512]]*3,  "max_steps":800},
    {"id":"F3_144_24-8-1_2-8-4",     "input_size":144, "n_freq_downsample":[24,8,1],  "n_blocks":[2,8,4],  "mlp_units":[[512,512]]*3,  "max_steps":800},
    {"id":"G3_144_24-8-1_256",        "input_size":144, "n_freq_downsample":[24,8,1],  "n_blocks":[2,6,3],  "mlp_units":[[256,256]]*3,  "max_steps":800},
    {"id":"H2_144_24-8-1_1200steps",  "input_size":144, "n_freq_downsample":[24,8,1],  "n_blocks":[2,6,3],  "mlp_units":[[512,512]]*3,  "max_steps":1200},
    {"id":"J1_144_48-8-1_2-8-4_1024","input_size":144, "n_freq_downsample":[48,8,1],  "n_blocks":[2,8,4],  "mlp_units":[[1024,512]]*3, "max_steps":800},

    # ── НОВЫЕ: углубляемся в input=96 (победитель Phase1) ──
    {"id":"NEW_96_24-8-1_2-8-4",     "input_size":96,  "n_freq_downsample":[24,8,1],  "n_blocks":[2,8,4],  "mlp_units":[[512,512]]*3,  "max_steps":800},
    {"id":"NEW_96_24-8-1_1-8-3",     "input_size":96,  "n_freq_downsample":[24,8,1],  "n_blocks":[1,8,3],  "mlp_units":[[512,512]]*3,  "max_steps":800},
    {"id":"NEW_96_24-8-1_3-8-3",     "input_size":96,  "n_freq_downsample":[24,8,1],  "n_blocks":[3,8,3],  "mlp_units":[[512,512]]*3,  "max_steps":800},
    {"id":"NEW_96_24-8-1_2-10-4",    "input_size":96,  "n_freq_downsample":[24,8,1],  "n_blocks":[2,10,4], "mlp_units":[[512,512]]*3,  "max_steps":800},
    {"id":"NEW_96_24-8-1_256",        "input_size":96,  "n_freq_downsample":[24,8,1],  "n_blocks":[2,6,3],  "mlp_units":[[256,256]]*3,  "max_steps":800},
    {"id":"NEW_96_24-8-1_1024",       "input_size":96,  "n_freq_downsample":[24,8,1],  "n_blocks":[2,6,3],  "mlp_units":[[1024,512]]*3, "max_steps":800},
    {"id":"NEW_96_24-8-1_2-8-4_1024","input_size":96,  "n_freq_downsample":[24,8,1],  "n_blocks":[2,8,4],  "mlp_units":[[1024,512]]*3, "max_steps":800},
    {"id":"NEW_96_24-8-1_1200steps",  "input_size":96,  "n_freq_downsample":[24,8,1],  "n_blocks":[2,6,3],  "mlp_units":[[512,512]]*3,  "max_steps":1200},
    {"id":"NEW_96_24-8-1_2-8-4_1200","input_size":96,  "n_freq_downsample":[24,8,1],  "n_blocks":[2,8,4],  "mlp_units":[[512,512]]*3,  "max_steps":1200},
]

# ── Уже известные результаты Phase1 (1–12) ──
KNOWN_RESULTS = [
    {"id":"A0_baseline",         "input_size":336,"n_freq_downsample":[48,8,1],"n_blocks":[4,4,4],  "mlp_units":[[512,512]]*3,"max_steps":800, "total":0.3606,"pruned":False},
    {"id":"A1_96_24-8-1_2-6-3",  "input_size":96, "n_freq_downsample":[24,8,1],"n_blocks":[2,6,3],  "mlp_units":[[512,512]]*3,"max_steps":800, "total":0.3528,"pruned":False},
    {"id":"A2_96_16-8-1_2-6-3",  "input_size":96, "n_freq_downsample":[16,8,1],"n_blocks":[2,6,3],  "mlp_units":[[512,512]]*3,"max_steps":800, "total":0.3528,"pruned":False},
    {"id":"A3_96_32-8-1_2-6-3",  "input_size":96, "n_freq_downsample":[32,8,1],"n_blocks":[2,6,3],  "mlp_units":[[512,512]]*3,"max_steps":800, "total":0.3528,"pruned":False},
    {"id":"B1_144_24-8-1_2-6-3", "input_size":144,"n_freq_downsample":[24,8,1],"n_blocks":[2,6,3],  "mlp_units":[[512,512]]*3,"max_steps":800, "total":0.3648,"pruned":False},
    {"id":"B2_144_48-8-1_2-6-3", "input_size":144,"n_freq_downsample":[48,8,1],"n_blocks":[2,6,3],  "mlp_units":[[512,512]]*3,"max_steps":800, "total":0.3648,"pruned":False},
    {"id":"B3_144_36-4-1_2-6-3", "input_size":144,"n_freq_downsample":[36,4,1],"n_blocks":[2,6,3],  "mlp_units":[[512,512]]*3,"max_steps":800, "total":0.3820,"pruned":False},
    {"id":"C1_192_48-8-1_2-6-3", "input_size":192,"n_freq_downsample":[48,8,1],"n_blocks":[2,6,3],  "mlp_units":[[512,512]]*3,"max_steps":800, "total":0.3922,"pruned":False},
    {"id":"C2_192_24-8-1_2-6-3", "input_size":192,"n_freq_downsample":[24,8,1],"n_blocks":[2,6,3],  "mlp_units":[[512,512]]*3,"max_steps":800, "total":0.3922,"pruned":False},
    {"id":"C3_192_32-8-1_2-6-3", "input_size":192,"n_freq_downsample":[32,8,1],"n_blocks":[2,6,3],  "mlp_units":[[512,512]]*3,"max_steps":800, "total":0.3922,"pruned":False},
    {"id":"D1_288_48-8-1_2-6-3", "input_size":288,"n_freq_downsample":[48,8,1],"n_blocks":[2,6,3],  "mlp_units":[[512,512]]*3,"max_steps":800, "total":0.4075,"pruned":True},
    {"id":"D2_288_96-8-1_2-6-3", "input_size":288,"n_freq_downsample":[96,8,1],"n_blocks":[2,6,3],  "mlp_units":[[512,512]]*3,"max_steps":800, "total":0.4075,"pruned":True},
]

print(f"\nКонфигов к прогону: {len(TODO_CONFIGS)}")
print(f"Оценка: ~{len(TODO_CONFIGS)*2 + 4*6 + 2*10 + 15} мин ≈ "
      f"{(len(TODO_CONFIGS)*2 + 4*6 + 2*10 + 15)/60:.1f} ч")


# ============================================================
# ПОДГОТОВКА ДАТАФРЕЙМОВ (только нужные input_size)
# ============================================================
unique_inputs = sorted(set(c["input_size"] for c in TODO_CONFIGS))
print(f"\nГотовим датафреймы для input_size: {unique_inputs}")
nf_dfs = {}
for inp in unique_inputs:
    min_len = TOTAL_VAL_PTS + inp + 1
    print(f"  input={inp}...", end=" ", flush=True)
    nf_dfs[inp] = to_nf_df(train_df, min_len=min_len, context_len=CONTEXT_LEN)
    print(f"{nf_dfs[inp]['unique_id'].nunique()} routes")
    free_memory()

print("Датафреймы готовы ✅")


# ============================================================
# CORE: запуск одного конфига с очисткой памяти
# ============================================================
def run_one(cfg, n_windows, train_nf_df):
    t0 = time.time()
    model = NHITS(
        h                   = FORECAST_STEPS,
        input_size          = cfg["input_size"],
        loss                = MQLoss(level=[80]),
        n_freq_downsample   = cfg["n_freq_downsample"],
        mlp_units           = cfg["mlp_units"],
        n_blocks            = cfg["n_blocks"],
        max_steps           = cfg["max_steps"],
        batch_size          = BATCH_SIZE,
        accelerator         = ACCELERATOR,
        hist_exog_list      = HIST_EXOG,
        futr_exog_list      = FUTR_EXOG,
        scaler_type         = "robust",
        enable_progress_bar = False,
    )
    nf = NeuralForecast(models=[model], freq=FREQ)
    try:
        cv = nf.cross_validation(df=train_nf_df, n_windows=n_windows,
                                  step_size=FORECAST_STEPS, refit=True)
        pc = get_pred_col(cv)
        total, wape, rbias = wape_rbias(cv["y"].values,
                                        np.clip(cv[pc].values, 0, None))
        fold_rows = []
        for i, cut in enumerate(sorted(cv["cutoff"].unique())):
            sub = cv[cv["cutoff"]==cut]
            t,w,r = wape_rbias(sub["y"].values, np.clip(sub[pc].values,0,None))
            fold_rows.append(f"fold{i+1}={t:.4f}")
        return {"total": float(total), "wape": float(wape), "rbias": float(rbias),
                "fold_str": " | ".join(fold_rows),
                "elapsed": float(time.time()-t0), "n_windows": n_windows}
    finally:
        del model, nf
        free_memory()


# ============================================================
# PHASE 1 — продолжение (конфиги 13–26)
# ============================================================
print("\n" + "="*65)
print(f"PHASE 1 (продолжение) — {len(TODO_CONFIGS)} конфигов")
print(f"Порог pruning: {BASELINE_TOTAL*PRUNE_FACTOR:.4f}")
print("="*65)

p1_all     = list(KNOWN_RESULTS)
best1_total = min(r["total"] for r in p1_all if not r["pruned"])
best1_id    = next(r["id"] for r in p1_all if r["total"] == best1_total)
pruned_cnt  = sum(1 for r in p1_all if r["pruned"])
P1_START    = time.time()

for i, cfg in enumerate(TODO_CONFIGS):
    global_idx = len(KNOWN_RESULTS) + i + 1
    total_all  = len(KNOWN_RESULTS) + len(TODO_CONFIGS)
    print(f"\n[Phase1 {global_idx}/{total_all}]  {cfg['id']}")
    print(f"  input={cfg['input_size']}  freq={cfg['n_freq_downsample']}  "
          f"blocks={cfg['n_blocks']}  mlp={cfg['mlp_units'][0]}  steps={cfg['max_steps']}")

    # Skip input≥192 — доказано хуже (≥0.39 vs лучший 0.3528)
    if cfg["input_size"] >= 192:
        print(f"  ⏭  SKIP (input≥192, доказано хуже)")
        p1_all.append({**cfg, "total": float("nan"), "pruned": True})
        pruned_cnt += 1
        continue

    try:
        res    = run_one(cfg, n_windows=1, train_nf_df=nf_dfs[cfg["input_size"]])
        pruned = res["total"] > BASELINE_TOTAL * PRUNE_FACTOR
        mark   = ""
        if not pruned and res["total"] < best1_total:
            best1_total = res["total"]
            best1_id    = cfg["id"]
            mark = "  ← 🏆 НОВЫЙ ЛУЧШИЙ"
        pruned_cnt += pruned
        p1_all.append({**cfg, "total": res["total"], "pruned": pruned})
        save_result(cfg, res, phase=1)
        tag = "✂️  PRUNED" if pruned else "✅"
        print(f"  {tag}  total={res['total']:.4f}  wape={res['wape']:.4f}  "
              f"rbias={res['rbias']:.4f}  time={res['elapsed']:.0f}s{mark}")
        print(f"  {res['fold_str']}")
    except Exception as e:
        print(f"  ❌ ERROR: {e}")
        p1_all.append({**cfg, "total": float("nan"), "pruned": True})
        pruned_cnt += 1
        free_memory()

    print_banner(i+1, len(TODO_CONFIGS), best1_id, best1_total, P1_START, phase=1)

# ── Топ-4 для Phase 2 ──
survived1 = sorted(
    [(r, r["total"]) for r in p1_all
     if not r.get("pruned", False) and not np.isnan(r["total"])],
    key=lambda x: x[1]
)
top4 = [r for r, _ in survived1[:4]]
print(f"\nPhase 1 ИТОГ  |  Выжило: {len(survived1)}  |  Pruned: {pruned_cnt}")
print(f"Топ-10:")
for i, (r, tot) in enumerate(survived1[:10]):
    print(f"  #{i+1:2d}  total={tot:.4f}  {r['id']}")
print(f"\nТоп-4 → Phase 2: {[r['id'] for r in top4]}")


# ============================================================
# PHASE 2 — 3-fold для топ-4
# ============================================================
print("\n" + "="*65)
print(f"PHASE 2 — 3-fold  ({len(top4)} конфигов)")
print("="*65)

p2_results  = []
best2_total = float("inf")
best2_id    = None
P2_START    = time.time()

for i, cfg in enumerate(top4):
    print(f"\n[Phase2 {i+1}/{len(top4)}]  {cfg['id']}")
    try:
        res  = run_one(cfg, n_windows=3, train_nf_df=nf_dfs[cfg["input_size"]])
        mark = ""
        if res["total"] < best2_total:
            best2_total = res["total"]
            best2_id    = cfg["id"]
            mark = "  ← 🏆 НОВЫЙ ЛУЧШИЙ"
        p2_results.append({"cfg": cfg, "res": res})
        save_result(cfg, res, phase=2)
        print(f"  ✅  total={res['total']:.4f}  wape={res['wape']:.4f}  "
              f"rbias={res['rbias']:.4f}  time={res['elapsed']:.0f}s{mark}")
        print(f"  {res['fold_str']}")
    except Exception as e:
        print(f"  ❌ ERROR: {e}")
        p2_results.append({"cfg": cfg, "res": {"total": float("nan")}})
        free_memory()
    print_banner(i+1, len(top4), best2_id, best2_total, P2_START, phase=2)

survived2 = sorted(
    [(r["cfg"], r["res"]["total"]) for r in p2_results
     if not np.isnan(r["res"]["total"])],
    key=lambda x: x[1]
)
top2 = [cfg for cfg, _ in survived2[:2]]
print(f"\nPhase 2 итог. Топ-2 → Phase 3: {[c['id'] for c in top2]}")


# ============================================================
# PHASE 3 — полный 5-fold для топ-2
# ============================================================
print("\n" + "="*65)
print(f"PHASE 3 — full 5-fold  ({len(top2)} конфигов)")
print("="*65)

p3_results  = []
best_cfg    = None
best3_total = float("inf")
P3_START    = time.time()

for i, cfg in enumerate(top2):
    print(f"\n[Phase3 {i+1}/{len(top2)}]  {cfg['id']}")
    try:
        res  = run_one(cfg, n_windows=N_FOLDS, train_nf_df=nf_dfs[cfg["input_size"]])
        mark = ""
        if res["total"] < best3_total:
            best3_total = res["total"]
            best_cfg    = cfg
            mark = "  ← 🏆 ПОБЕДИТЕЛЬ"
        p3_results.append({"cfg": cfg, "res": res})
        save_result(cfg, res, phase=3)
        print(f"  ✅  total={res['total']:.4f}  wape={res['wape']:.4f}  "
              f"rbias={res['rbias']:.4f}  time={res['elapsed']:.0f}s{mark}")
        print(f"  {res['fold_str']}")
    except Exception as e:
        print(f"  ❌ ERROR: {e}")
        free_memory()
    print_banner(i+1, len(top2), best_cfg["id"] if best_cfg else None,
                 best3_total, P3_START, phase=3)

if best_cfg is None:
    best_cfg    = top2[0]
    best3_total = survived2[0][1]

print(f"\n{'='*65}")
print(f"🏆  ПОБЕДИТЕЛЬ: {best_cfg['id']}  total={best3_total:.4f}")
print(f"    vs baseline: {BASELINE_TOTAL:.4f}  Δ={best3_total-BASELINE_TOTAL:+.4f}")
for k,v in [("input_size",best_cfg["input_size"]),
            ("n_freq_downsample",best_cfg["n_freq_downsample"]),
            ("n_blocks",best_cfg["n_blocks"]),
            ("mlp_units",best_cfg["mlp_units"]),
            ("max_steps",best_cfg["max_steps"])]:
    print(f"    {k:<22} = {v}")


# ============================================================
# FINAL FIT + SUBMISSION
# ============================================================
print("\n" + "="*65)
print("FINAL FIT + TEST PREDICTION")
print("="*65)

# Финальный датафрейм — все данные, min_len мягкий
train_nf_full  = to_nf_df(train_df, min_len=best_cfg["input_size"]+1,
                            context_len=CONTEXT_LEN)
test_route_ids = test_df["route_id"].unique().tolist()
print(f"Full train routes: {train_nf_full['unique_id'].nunique()}")

final_model = NHITS(
    h                   = FORECAST_STEPS,
    input_size          = best_cfg["input_size"],
    loss                = MQLoss(level=[80]),
    n_freq_downsample   = best_cfg["n_freq_downsample"],
    mlp_units           = best_cfg["mlp_units"],
    n_blocks            = best_cfg["n_blocks"],
    max_steps           = best_cfg["max_steps"],
    batch_size          = BATCH_SIZE,
    accelerator         = ACCELERATOR,
    hist_exog_list      = HIST_EXOG,
    futr_exog_list      = FUTR_EXOG,
    scaler_type         = "robust",
    enable_progress_bar = True,
)
nf_full = NeuralForecast(models=[final_model], freq=FREQ)
nf_full.fit(train_nf_full)

# calib_scale через 5-fold CV
print("\nCV для calib_scale...")
nf_dfs_full = nf_dfs.get(best_cfg["input_size"])
if nf_dfs_full is None:
    nf_dfs_full = to_nf_df(train_df,
                            min_len=TOTAL_VAL_PTS+best_cfg["input_size"]+1,
                            context_len=CONTEXT_LEN)
cv_final  = nf_full.cross_validation(
    df=nf_dfs_full, n_windows=N_FOLDS, step_size=FORECAST_STEPS, refit=True
)
pc        = get_pred_col(cv_final)
y_true_cv = cv_final["y"].values
y_pred_cv = np.clip(cv_final[pc].values, 0, None)
total_cv, wape_cv, rbias_cv = wape_rbias(y_true_cv, y_pred_cv)
calib_scale = float(y_true_cv.sum() / (y_pred_cv.sum() + 1e-9))
print(f"CV  total={total_cv:.4f}  wape={wape_cv:.4f}  rbias={rbias_cv:.4f}")
print(f"calib_scale = {calib_scale:.4f}")

# Предикты на тест
print("\nPredicting...")
futr_df      = make_futr_df(test_route_ids, train_df)
test_pred_df = nf_full.predict(futr_df=futr_df)
tpc          = get_pred_col(test_pred_df)
test_preds   = {rid: np.clip(g.sort_values("ds")[tpc].values, 0, None)
                for rid, g in test_pred_df.groupby("unique_id")}

predictions_raw = {}
for rid in tqdm(test_route_ids, desc="Build submission"):
    preds = test_preds.get(rid, np.zeros(FORECAST_STEPS))
    for j, (_, row) in enumerate(
        test_df[test_df["route_id"]==rid].sort_values("timestamp").iterrows()
    ):
        predictions_raw[row["id"]] = float(
            max(0.0, preds[j] if j < len(preds) else preds[-1])
        )

sub_raw = (pd.DataFrame(list(predictions_raw.items()), columns=["id","y_pred"])
           .sort_values("id").reset_index(drop=True))
sub_cal = sub_raw.copy()
sub_cal["y_pred"] = np.clip(sub_cal["y_pred"] * calib_scale, 0, None)

sub_raw.to_csv("submission_nhits_best_raw.csv",  index=False)
sub_cal.to_csv("submission_nhits_best_cal.csv",  index=False)

print(f"\n✅ submission_nhits_best_raw.csv")
print(f"✅ submission_nhits_best_cal.csv")
print(f"✅ {RESULTS_PATH}")
print(f"\n── Raw ──\n{sub_raw['y_pred'].describe().round(0)}")
print(f"\n── Calibrated (scale={calib_scale:.4f}) ──\n{sub_cal['y_pred'].describe().round(0)}")
print(f"\n⏱  Общее время: {(time.time()-P1_START)/60:.1f} мин")

In [ ]:
# ============================================================
# Прямой 5-fold CV для 5 лучших кандидатов из Phase1
# Без многофазного отбора — он оказался слишком шумным
# ============================================================

DIRECT_CANDIDATES = [
    # Три победителя Phase1 по 1-fold (0.3528) + baseline + лучший 144
    {"id":"A1_96_24-8-1_2-6-3",   "input_size":96,  "n_freq_downsample":[24,8,1], "n_blocks":[2,6,3],  "mlp_units":[[512,512]]*3,  "max_steps":800},
    {"id":"A2_96_16-8-1_2-6-3",   "input_size":96,  "n_freq_downsample":[16,8,1], "n_blocks":[2,6,3],  "mlp_units":[[512,512]]*3,  "max_steps":800},
    {"id":"A3_96_32-8-1_2-6-3",   "input_size":96,  "n_freq_downsample":[32,8,1], "n_blocks":[2,6,3],  "mlp_units":[[512,512]]*3,  "max_steps":800},
    # Вариации которые реально могут дать прирост
    {"id":"NEW_96_24-8-1_2-8-4",  "input_size":96,  "n_freq_downsample":[24,8,1], "n_blocks":[2,8,4],  "mlp_units":[[512,512]]*3,  "max_steps":800},
    {"id":"NEW_96_24-8-1_2-10-4", "input_size":96,  "n_freq_downsample":[24,8,1], "n_blocks":[2,10,4], "mlp_units":[[512,512]]*3,  "max_steps":800},
]

print(f"Запускаем честный 5-fold CV для {len(DIRECT_CANDIDATES)} конфигов")
print(f"Оценка: ~{len(DIRECT_CANDIDATES)*10} мин")
print(f"Baseline для сравнения: {BASELINE_TOTAL:.4f}\n")

direct_results = []
best_direct_total = float("inf")
best_direct_cfg   = None
D_START = time.time()

# Датафрейм для input=96
if 96 not in nf_dfs:
    nf_dfs[96] = to_nf_df(train_df,
                           min_len=TOTAL_VAL_PTS + 96 + 1,
                           context_len=CONTEXT_LEN)
    print(f"Создали датафрейм input=96: {nf_dfs[96]['unique_id'].nunique()} routes")

for i, cfg in enumerate(DIRECT_CANDIDATES):
    print(f"\n[{i+1}/{len(DIRECT_CANDIDATES)}]  {cfg['id']}")
    print(f"  input={cfg['input_size']}  freq={cfg['n_freq_downsample']}  "
          f"blocks={cfg['n_blocks']}  mlp={cfg['mlp_units'][0]}  steps={cfg['max_steps']}")
    try:
        res  = run_one(cfg, n_windows=N_FOLDS, train_nf_df=nf_dfs[cfg["input_size"]])
        mark = ""
        if res["total"] < best_direct_total:
            best_direct_total = res["total"]
            best_direct_cfg   = cfg
            mark = "  ← 🏆 ЛУЧШИЙ"
        direct_results.append({"cfg": cfg, "res": res})
        save_result(cfg, res, phase=99)

        vs = res["total"] - BASELINE_TOTAL
        vs_str = f"{'🟢' if vs < 0 else '🔴'} {vs:+.4f} vs baseline"
        print(f"  ✅  total={res['total']:.4f}  wape={res['wape']:.4f}  "
              f"rbias={res['rbias']:.4f}  time={res['elapsed']:.0f}s  {vs_str}{mark}")
        print(f"  {res['fold_str']}")
    except Exception as e:
        print(f"  ❌ ERROR: {e}")
        free_memory()

    # Прогресс
    elapsed = time.time() - D_START
    left    = (len(DIRECT_CANDIDATES) - i - 1) * (elapsed / (i+1))
    print(f"\n  elapsed={elapsed/60:.1f}m  left≈{left/60:.1f}m  "
          f"🏆 пока лучший: {best_direct_cfg['id'] if best_direct_cfg else '—'}  "
          f"total={best_direct_total:.4f}")

# ── Итог ──
direct_results.sort(key=lambda x: x["res"]["total"])
print(f"\n{'='*65}")
print(f"ИТОГ прямого 5-fold поиска:")
print(f"{'#':<3} {'total':>7} {'wape':>7} {'rbias':>7}  id")
print("-"*55)
for i, r in enumerate(direct_results):
    vs  = r["res"]["total"] - BASELINE_TOTAL
    tag = "🟢" if vs < 0 else "🔴"
    print(f"{i+1:<3} {r['res']['total']:>7.4f} {r['res']['wape']:>7.4f} "
          f"{r['res']['rbias']:>7.4f}  {r['cfg']['id']}  {tag}{vs:+.4f}")

print(f"\nBaseline: {BASELINE_TOTAL:.4f}")
if best_direct_total < BASELINE_TOTAL:
    print(f"🟢 Улучшение: {best_direct_cfg['id']}  Δ={best_direct_total-BASELINE_TOTAL:+.4f}")
else:
    print(f"🔴 Ни один конфиг не побил baseline на 5-fold")
    print(f"   → Отправляй submission с оригинальным конфигом (input=336, blocks=[4,4,4])")

In [ ]:
# ============================================================
# TiRex — Rolling CV (5 folds × 8 steps)
# + per-route статистика для блендинга (как в N-HiTS)
# Zero-shot: нет фазы обучения, только rolling inference
# ============================================================

# ── Установка (без SSL) ──
# pip install tirex-ts -q --trusted-host pypi.org --trusted-host files.pythonhosted.org

import os
import ssl
import warnings
import numpy as np
import pandas as pd

from tqdm import tqdm

# Отключаем SSL для загрузки модели с HuggingFace
ssl._create_default_https_context = ssl._create_unverified_context
os.environ["CURL_CA_BUNDLE"]     = ""
os.environ["REQUESTS_CA_BUNDLE"] = ""
os.environ["HF_HUB_DISABLE_SSL_VERIFICATION"] = "1"

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None


# ── Метрика (идентично N-HiTS) ──
def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias


# ── Config ──
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8
N_FOLDS        = 5
TOTAL_VAL_PTS  = N_FOLDS * FORECAST_STEPS   # 40
CONTEXT_LEN    = 2048
FREQ           = "30min"
BATCH_SIZE     = 32   # маршрутов за один forward pass


# ── Data ──
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
print(f"Train: {train_df.shape}  Test: {test_df.shape}")


# ── Загрузка TiRex ──

from tirex import load_model
model = load_model("NX-AI/TiRex")
model.eval()
print("TiRex loaded ✅\n")


# ── Вспомогательная: подготовить ряд для одного маршрута ──
def prepare_series(df, route_id, context_len=CONTEXT_LEN):
    """
    Возвращает (numpy_values, pd.DatetimeIndex) с asfreq/interpolate,
    обрезанные до context_len точек.
    """
    grp = (
        df[df["route_id"] == route_id]
        .sort_values("timestamp")
        .set_index("timestamp")[[TARGET_COL]]
        .asfreq(FREQ)
        .interpolate(method="time")
        .bfill().ffill()
    )
    values = grp[TARGET_COL].values.astype(float)
    idx    = grp.index
    if context_len and len(values) > context_len:
        values = values[-context_len:]
        idx    = idx[-context_len:]
    return values, idx


# ── Батчевый инференс TiRex ──
# quantiles: [batch, h, 9] — уровни [0.1, 0.2, ..., 0.9], медиана = индекс 4
# means:     [batch, h]    — аналитическое среднее
def batch_forecast(contexts: list, h: int = FORECAST_STEPS) -> np.ndarray:
    """Возвращает means[batch, h] (numpy)."""
    all_means = []
    for i in range(0, len(contexts), BATCH_SIZE):
        batch = contexts[i : i + BATCH_SIZE]
        with torch.no_grad():
            _, means = model.forecast(batch, prediction_length=h)
        # means может быть Tensor или ndarray в зависимости от версии
        if isinstance(means, torch.Tensor):
            means = means.cpu().numpy()
        all_means.append(means)
    return np.concatenate(all_means, axis=0)  # [n_routes, h]


# ── Все маршруты ──
route_ids = sorted(train_df["route_id"].unique().tolist())
print(f"Routes total: {len(route_ids)}")


# ============================================================
# ROLLING CROSS-VALIDATION
# fold k: контекст = все точки до [-val_end_offset]
#         val      = points[-val_end_offset : -val_start_offset]
# fold 1: train=[:-40]  val=[-40:-32]
# fold 2: train=[:-32]  val=[-32:-24]
# fold 3: train=[:-24]  val=[-24:-16]
# fold 4: train=[:-16]  val=[-16: -8]
# fold 5: train=[: -8]  val=[ -8:end]
# Zero-shot: никакого refit — TiRex не обучается
# ============================================================
print("=" * 65)
print(f"ROLLING CV  ({N_FOLDS} folds × {FORECAST_STEPS} steps = {TOTAL_VAL_PTS} points)")
print("=" * 65)

raw_rows = []

for fold in range(1, N_FOLDS + 1):
    val_end_offset   = TOTAL_VAL_PTS - (fold - 1) * FORECAST_STEPS  # число точек с конца до края val
    val_start_offset = val_end_offset - FORECAST_STEPS               # число точек с конца до начала val (= cutoff)

    batch_contexts = []
    batch_meta     = []   # (route_id, val_ts, val_values)

    for route_id in route_ids:
        values, idx = prepare_series(train_df, route_id, context_len=None)  # без обрезки — сделаем сами

        if len(values) <= val_end_offset:
            continue  # ряд слишком короткий для этого фолда

        # Контекст — всё ДО валидационного окна
        ctx        = values[:-val_end_offset]
        val_values = values[-val_end_offset:] if val_start_offset == 0 else values[-val_end_offset:-val_start_offset]
        val_ts     = idx[-val_end_offset:]    if val_start_offset == 0 else idx[-val_end_offset:-val_start_offset]

        # Обрезаем контекст до CONTEXT_LEN
        if len(ctx) > CONTEXT_LEN:
            ctx = ctx[-CONTEXT_LEN:]

        if len(ctx) < 1:
            continue

        batch_contexts.append(ctx)
        batch_meta.append((route_id, val_ts, val_values))

    # Батчевый инференс для фолда
    all_means = batch_forecast(batch_contexts)   # [n_routes_fold, FORECAST_STEPS]

    fold_y_true, fold_y_pred = [], []
    for idx_r, (route_id, val_ts, val_values) in enumerate(batch_meta):
        preds = np.clip(all_means[idx_r], 0, None)
        for h_idx in range(FORECAST_STEPS):
            y_true = float(val_values[h_idx])
            y_pred = float(preds[h_idx])
            ae     = abs(y_pred - y_true)
            ape    = ae / (y_true + 1e-9)
            err    = y_pred - y_true
            raw_rows.append({
                "fold":      fold,
                "route_id":  route_id,
                "timestamp": val_ts[h_idx],
                "h":         h_idx + 1,
                "y_true":    y_true,
                "y_pred":    y_pred,
                "ae":        ae,
                "ape":       ape,
                "err":       err,
            })
            fold_y_true.append(y_true)
            fold_y_pred.append(y_pred)

    n_routes_fold = len(batch_meta)
    yt  = np.array(fold_y_true)
    yp  = np.array(fold_y_pred)
    tot, wape, rb = wape_rbias(yt, yp)

    val_end   = val_end_offset
    val_start = val_start_offset
    print(f"\nFold {fold}/{N_FOLDS}  "
          f"(val: -{val_end}..{'-'+str(val_start) if val_start else 'end'})  "
          f"routes={n_routes_fold}")
    print(f"  RAW   WAPE={wape:.4f}  |RBias|={rb:.4f}  Total={tot:.4f}")


# ============================================================
# АГРЕГАТЫ (идентично N-HiTS)
# ============================================================
raw_cv_df = pd.DataFrame(raw_rows)

# ── По фолду ──
agg_fold = raw_cv_df.groupby("fold").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
    "n":     len(df),
})).reset_index()

# ── По шагу h ──
agg_h = raw_cv_df.groupby("h").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
})).reset_index()

# ── По маршруту ──
agg_route = raw_cv_df.groupby("route_id").agg(
    n_points   = ("y_true",  "count"),
    mae        = ("ae",      "mean"),
    median_ae  = ("ae",      "median"),
    std_ae     = ("ae",      "std"),
    mape       = ("ape",     "mean"),
    median_ape = ("ape",     "median"),
    mean_err   = ("err",     "mean"),
    median_err = ("err",     "median"),
    mean_true  = ("y_true",  "mean"),
    mean_pred  = ("y_pred",  "mean"),
).reset_index()

# calib_scale & wape по маршруту — пересчитываем через merge для надёжности
route_sums = raw_cv_df.groupby("route_id").agg(
    sum_true = ("y_true", "sum"),
    sum_pred = ("y_pred", "sum"),
    sum_ae   = ("ae",     "sum"),
).reset_index()
agg_route = agg_route.merge(route_sums, on="route_id", how="left")
agg_route["calib_scale"] = agg_route["sum_true"] / (agg_route["sum_pred"] + 1e-9)
agg_route["wape"]        = agg_route["sum_ae"]   / (agg_route["sum_true"] + 1e-9)
agg_route.drop(columns=["sum_true", "sum_pred", "sum_ae"], inplace=True)

# ── Глобальные метрики ──
y_true_all = raw_cv_df["y_true"].values
y_pred_all = raw_cv_df["y_pred"].values
total_raw, wape_raw, rbias_raw = wape_rbias(y_true_all, y_pred_all)

calib_scale = float(y_true_all.sum() / (y_pred_all.sum() + 1e-9))
y_pred_cal  = np.clip(y_pred_all * calib_scale, 0, None)
total_cal, wape_cal, rbias_cal = wape_rbias(y_true_all, y_pred_cal)

print("\n" + "=" * 65)
print("SUMMARY")
print("=" * 65)
print("\n── По фолдам ──")
print(agg_fold.to_string(index=False))
print("\n── По шагам h ──")
print(agg_h.to_string(index=False))
print(f"\n{'Global RAW':25s}  WAPE={wape_raw:.4f}  |RBias|={rbias_raw:.4f}  Total={total_raw:.4f}")
print(f"{'Global CALIBRATED':25s}  WAPE={wape_cal:.4f}  |RBias|={rbias_cal:.4f}  Total={total_cal:.4f}")
print(f"  calib_scale = {calib_scale:.4f}")
print(f"\nCalibration {'HELPS ✅' if total_cal < total_raw else 'HURTS ❌'} "
      f"(Δ = {total_cal - total_raw:+.4f})")
print("\n── Топ-10 сложных маршрутов (по median_ape) ──")
print(agg_route.sort_values("median_ape", ascending=False)
      [["route_id", "mae", "median_ae", "mape", "median_ape", "mean_err", "calib_scale"]]
      .head(10).to_string(index=False))

raw_cv_df.to_csv("tirex_cv_raw.csv",        index=False)
agg_route.to_csv("tirex_cv_agg_route.csv",  index=False)
agg_fold.to_csv("tirex_cv_agg_fold.csv",    index=False)
agg_h.to_csv("tirex_cv_agg_h.csv",          index=False)
print("\n✅ tirex_cv_raw.csv")
print("✅ tirex_cv_agg_route.csv")
print("✅ tirex_cv_agg_fold.csv")
print("✅ tirex_cv_agg_h.csv")


# ============================================================
# TEST PREDICTION
# Контекст = последние CONTEXT_LEN точек тренировки на маршрут
# ============================================================
print("\n" + "=" * 65)
print("TEST PREDICTION")
print("=" * 65)

test_route_ids = test_df["route_id"].unique().tolist()
print(f"Routes: {len(test_route_ids)}  Running TiRex inference...")

test_contexts = []
test_meta     = []
for route_id in test_route_ids:
    ctx, _ = prepare_series(train_df, route_id, context_len=CONTEXT_LEN)
    route_test = test_df[test_df["route_id"] == route_id].sort_values("timestamp")
    test_contexts.append(ctx)
    test_meta.append((route_id, route_test))

test_means = batch_forecast(test_contexts)   # [n_routes, FORECAST_STEPS]

predictions_raw = {}
for idx_r, (route_id, route_test) in enumerate(tqdm(test_meta, desc="Build submission")):
    preds = np.clip(test_means[idx_r], 0, None)
    for j, (_, row) in enumerate(route_test.iterrows()):
        pred = float(preds[j]) if j < len(preds) else float(preds[-1])
        predictions_raw[row["id"]] = max(0.0, pred)

submission_raw = (
    pd.DataFrame(list(predictions_raw.items()), columns=["id", "y_pred"])
    .sort_values("id").reset_index(drop=True)
)
submission_cal = submission_raw.copy()
submission_cal["y_pred"] = np.clip(submission_cal["y_pred"] * calib_scale, 0, None)

submission_raw.to_csv("submission_tirex_raw.csv",        index=False)
submission_cal.to_csv("submission_tirex_calibrated.csv", index=False)
print(f"\n✅ submission_tirex_raw.csv")
print(f"✅ submission_tirex_calibrated.csv")
print(f"\n── Raw ──\n{submission_raw['y_pred'].describe().round(2)}")
print(f"\n── Calibrated (scale={calib_scale:.4f}) ──\n{submission_cal['y_pred'].describe().round(2)}")

In [ ]:
# ============================================================
# Sundial — Rolling CV (5 folds × 8 steps)
# + per-route статистика для блендинга
# Zero-shot generative model: sampling → mean = point forecast
# ============================================================

import ssl
import os
import warnings
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

# ── SSL Kill Switch ──
ssl._create_default_https_context = ssl._create_unverified_context
os.environ["CURL_CA_BUNDLE"]                  = ""
os.environ["REQUESTS_CA_BUNDLE"]              = ""
os.environ["HF_HUB_DISABLE_SSL_VERIFICATION"] = "1"

import requests, urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
_orig_req = requests.Session.request
def _patched_req(self, method, url, **kwargs):
    kwargs.setdefault("verify", False)
    return _orig_req(self, method, url, **kwargs)
requests.Session.request = _patched_req

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None


# ── Метрика ──
def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias


# ── Config ──
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8
N_FOLDS        = 5
TOTAL_VAL_PTS  = N_FOLDS * FORECAST_STEPS   # 40
CONTEXT_LEN    = 1024   # рекомендация из notebook Sundial
FREQ           = "30min"
BATCH_SIZE     = 16     # меньше чем TiRex — модель тяжелее
NUM_SAMPLES    = 20     # сэмплы из генеративного распределения


# ── Device ──
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(f"Device: {DEVICE}")


# ── Data ──
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
print(f"Train: {train_df.shape}  Test: {test_df.shape}")


# ── Load Sundial ──
# pip install transformers==4.40.1  (требование из notebook)
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained(
    "thuml/sundial-base-128m",
    trust_remote_code=True,
)
model = model.to(DEVICE)
model.eval()
print("Sundial loaded ✅\n")


# ── Подготовить ряд маршрута ──
def prepare_series(df, route_id, context_len=CONTEXT_LEN):
    grp = (
        df[df["route_id"] == route_id]
        .sort_values("timestamp")
        .set_index("timestamp")[[TARGET_COL]]
        .asfreq(FREQ)
        .interpolate(method="time")
        .bfill().ffill()
    )
    values = grp[TARGET_COL].values.astype(float)
    idx    = grp.index
    if context_len and len(values) > context_len:
        values = values[-context_len:]
        idx    = idx[-context_len:]
    return values, idx


# ── Left-pad батча к одной длине (паддинг = среднее ряда) ──
def pad_to_tensor(arrays: list, device: str) -> torch.Tensor:
    """
    Sundial принимает [batch, context_len] — все ряды одной длины.
    Паддируем слева средним значения ряда, чтобы не вносить bias нулями.
    """
    max_len = max(len(a) for a in arrays)
    padded  = []
    for a in arrays:
        if len(a) < max_len:
            fill = float(np.mean(a)) if len(a) > 0 else 0.0
            a = np.concatenate([np.full(max_len - len(a), fill), a])
        padded.append(a[-max_len:])
    return torch.tensor(np.stack(padded), dtype=torch.float32).to(device)


# ── Батчевый инференс ──
def batch_forecast(contexts: list, h: int = FORECAST_STEPS) -> np.ndarray:
    """
    Возвращает point forecast [n_routes, h].
    model.generate → [batch, NUM_SAMPLES, h]
    Берём среднее по сэмплам как точечный прогноз.
    """
    all_preds = []
    for i in range(0, len(contexts), BATCH_SIZE):
        batch = contexts[i : i + BATCH_SIZE]
        inp   = pad_to_tensor(batch, DEVICE)          # [b, ctx_len]
        with torch.no_grad():
            forecast = model.generate(
                inp,
                max_new_tokens=h,
                num_samples=NUM_SAMPLES,
            )                                          # [b, NUM_SAMPLES, h]
        point = forecast.mean(dim=1).cpu().numpy()    # [b, h]
        all_preds.append(point)
    return np.concatenate(all_preds, axis=0)          # [n_routes, h]


route_ids = sorted(train_df["route_id"].unique().tolist())
print(f"Routes total: {len(route_ids)}")


# ============================================================
# ROLLING CROSS-VALIDATION
# fold 1: train=[:-40]  val=[-40:-32]
# fold 2: train=[:-32]  val=[-32:-24]
# fold 3: train=[:-24]  val=[-24:-16]
# fold 4: train=[:-16]  val=[-16: -8]
# fold 5: train=[: -8]  val=[ -8:end]
# ============================================================
print("=" * 65)
print(f"ROLLING CV  ({N_FOLDS} folds × {FORECAST_STEPS} steps = {TOTAL_VAL_PTS} points)")
print("=" * 65)

raw_rows = []

for fold in range(1, N_FOLDS + 1):
    val_end_offset   = TOTAL_VAL_PTS - (fold - 1) * FORECAST_STEPS
    val_start_offset = val_end_offset - FORECAST_STEPS

    batch_contexts, batch_meta = [], []

    for route_id in route_ids:
        values, idx = prepare_series(train_df, route_id, context_len=None)

        if len(values) <= val_end_offset:
            continue

        ctx        = values[:-val_end_offset]
        val_values = (values[-val_end_offset:]
                      if val_start_offset == 0
                      else values[-val_end_offset:-val_start_offset])
        val_ts     = (idx[-val_end_offset:]
                      if val_start_offset == 0
                      else idx[-val_end_offset:-val_start_offset])

        if len(ctx) > CONTEXT_LEN:
            ctx = ctx[-CONTEXT_LEN:]
        if len(ctx) < 1:
            continue

        batch_contexts.append(ctx)
        batch_meta.append((route_id, val_ts, val_values))

    all_means = batch_forecast(batch_contexts)

    fold_y_true, fold_y_pred = [], []
    for idx_r, (route_id, val_ts, val_values) in enumerate(batch_meta):
        preds = np.clip(all_means[idx_r], 0, None)
        for h_idx in range(FORECAST_STEPS):
            y_true = float(val_values[h_idx])
            y_pred = float(preds[h_idx])
            ae     = abs(y_pred - y_true)
            ape    = ae / (y_true + 1e-9)
            err    = y_pred - y_true
            raw_rows.append({
                "fold":      fold,
                "route_id":  route_id,
                "timestamp": val_ts[h_idx],
                "h":         h_idx + 1,
                "y_true":    y_true,
                "y_pred":    y_pred,
                "ae":        ae,
                "ape":       ape,
                "err":       err,
            })
            fold_y_true.append(y_true)
            fold_y_pred.append(y_pred)

    yt  = np.array(fold_y_true)
    yp  = np.array(fold_y_pred)
    tot, wape, rb = wape_rbias(yt, yp)
    print(f"\nFold {fold}/{N_FOLDS}  "
          f"(val: -{val_end_offset}..{'-'+str(val_start_offset) if val_start_offset else 'end'})  "
          f"routes={len(batch_meta)}")
    print(f"  RAW   WAPE={wape:.4f}  |RBias|={rb:.4f}  Total={tot:.4f}")


# ============================================================
# АГРЕГАТЫ
# ============================================================
raw_cv_df = pd.DataFrame(raw_rows)

agg_fold = raw_cv_df.groupby("fold").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
    "n":     len(df),
})).reset_index()

agg_h = raw_cv_df.groupby("h").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
})).reset_index()

agg_route = raw_cv_df.groupby("route_id").agg(
    n_points   = ("y_true",  "count"),
    mae        = ("ae",      "mean"),
    median_ae  = ("ae",      "median"),
    std_ae     = ("ae",      "std"),
    mape       = ("ape",     "mean"),
    median_ape = ("ape",     "median"),
    mean_err   = ("err",     "mean"),
    median_err = ("err",     "median"),
    mean_true  = ("y_true",  "mean"),
    mean_pred  = ("y_pred",  "mean"),
).reset_index()

route_sums = raw_cv_df.groupby("route_id").agg(
    sum_true = ("y_true", "sum"),
    sum_pred = ("y_pred", "sum"),
    sum_ae   = ("ae",     "sum"),
).reset_index()
agg_route = agg_route.merge(route_sums, on="route_id", how="left")
agg_route["calib_scale"] = agg_route["sum_true"] / (agg_route["sum_pred"] + 1e-9)
agg_route["wape"]        = agg_route["sum_ae"]   / (agg_route["sum_true"] + 1e-9)
agg_route.drop(columns=["sum_true", "sum_pred", "sum_ae"], inplace=True)

y_true_all = raw_cv_df["y_true"].values
y_pred_all = raw_cv_df["y_pred"].values
total_raw, wape_raw, rbias_raw = wape_rbias(y_true_all, y_pred_all)

calib_scale = float(y_true_all.sum() / (y_pred_all.sum() + 1e-9))
y_pred_cal  = np.clip(y_pred_all * calib_scale, 0, None)
total_cal, wape_cal, rbias_cal = wape_rbias(y_true_all, y_pred_cal)

print("\n" + "=" * 65)
print("SUMMARY")
print("=" * 65)
print("\n── По фолдам ──")
print(agg_fold.to_string(index=False))
print("\n── По шагам h ──")
print(agg_h.to_string(index=False))
print(f"\n{'Global RAW':25s}  WAPE={wape_raw:.4f}  |RBias|={rbias_raw:.4f}  Total={total_raw:.4f}")
print(f"{'Global CALIBRATED':25s}  WAPE={wape_cal:.4f}  |RBias|={rbias_cal:.4f}  Total={total_cal:.4f}")
print(f"  calib_scale = {calib_scale:.4f}")
print(f"\nCalibration {'HELPS ✅' if total_cal < total_raw else 'HURTS ❌'} "
      f"(Δ = {total_cal - total_raw:+.4f})")
print("\n── Топ-10 сложных маршрутов (по median_ape) ──")
print(agg_route.sort_values("median_ape", ascending=False)
      [["route_id", "mae", "median_ae", "mape", "median_ape", "mean_err", "calib_scale"]]
      .head(10).to_string(index=False))

raw_cv_df.to_csv("sundial_cv_raw.csv",        index=False)
agg_route.to_csv("sundial_cv_agg_route.csv",  index=False)
agg_fold.to_csv("sundial_cv_agg_fold.csv",    index=False)
agg_h.to_csv("sundial_cv_agg_h.csv",          index=False)
print("\n✅ sundial_cv_raw.csv / agg_route / agg_fold / agg_h")


# ============================================================
# TEST PREDICTION
# ============================================================
print("\n" + "=" * 65)
print("TEST PREDICTION")
print("=" * 65)

test_route_ids = test_df["route_id"].unique().tolist()
print(f"Routes: {len(test_route_ids)}  Running Sundial inference...")

test_contexts, test_meta = [], []
for route_id in test_route_ids:
    ctx, _ = prepare_series(train_df, route_id, context_len=CONTEXT_LEN)
    route_test = test_df[test_df["route_id"] == route_id].sort_values("timestamp")
    test_contexts.append(ctx)
    test_meta.append((route_id, route_test))

test_means = batch_forecast(test_contexts)

predictions_raw = {}
for idx_r, (route_id, route_test) in enumerate(tqdm(test_meta, desc="Build submission")):
    preds = np.clip(test_means[idx_r], 0, None)
    for j, (_, row) in enumerate(route_test.iterrows()):
        pred = float(preds[j]) if j < len(preds) else float(preds[-1])
        predictions_raw[row["id"]] = max(0.0, pred)

submission_raw = (
    pd.DataFrame(list(predictions_raw.items()), columns=["id", "y_pred"])
    .sort_values("id").reset_index(drop=True)
)
submission_cal = submission_raw.copy()
submission_cal["y_pred"] = np.clip(submission_cal["y_pred"] * calib_scale, 0, None)

submission_raw.to_csv("submission_sundial_raw.csv",        index=False)
submission_cal.to_csv("submission_sundial_calibrated.csv", index=False)
print(f"\n✅ submission_sundial_raw.csv")
print(f"✅ submission_sundial_calibrated.csv")
print(f"\n── Raw ──\n{submission_raw['y_pred'].describe().round(2)}")
print(f"\n── Calibrated (scale={calib_scale:.4f}) ──\n{submission_cal['y_pred'].describe().round(2)}")

In [ ]:
# ============================================================
# Lag-Llama — Rolling CV (5 folds × 8 steps) + Test prediction
# API: GluonTS PandasDataset + LagLlamaEstimator
# ============================================================

import os, sys, warnings
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

# ── Пути ──
LAG_LLAMA_DIR = "./lag-llama"
CKPT_PATH     = os.path.join(LAG_LLAMA_DIR, "lag-llama.ckpt")
sys.path.insert(0, LAG_LLAMA_DIR)

# ── Импорты GluonTS и Lag-Llama ──
from gluonts.evaluation import make_evaluation_predictions
from gluonts.dataset.pandas import PandasDataset
from lag_llama.gluon.estimator import LagLlamaEstimator

# ── Патч только если реально нет класса (после настоящего импорта) ──
try:
    from gluonts.torch.modules.loss import DistributionLoss, NegativeLogLikelihood
except ImportError:
    import gluonts.torch.modules.loss as _loss_mod
    class _DL:
        def __init__(self, *a, **kw): pass
        def __call__(self, *a, **kw): return 0.0
        def __getattr__(self, n): return lambda *a, **kw: None
    _loss_mod.DistributionLoss      = _DL
    _loss_mod.NegativeLogLikelihood = _DL
    print("DistributionLoss patched ✅")


# ── Config ──
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8
N_FOLDS        = 5
TOTAL_VAL_PTS  = N_FOLDS * FORECAST_STEPS   # 40
CONTEXT_LENGTH = 32
NUM_SAMPLES    = 100
FREQ           = "30min"

# ── Device ──
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")
print(f"Device: {DEVICE}")


# ── Метрика ──
def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias


# ── Data ──
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
print(f"Train: {train_df.shape}  Test: {test_df.shape}")
route_ids = sorted(train_df["route_id"].unique().tolist())
print(f"Routes: {len(route_ids)}")


# ── Подготовить ряд маршрута ──
def prepare_series(df, route_id):
    return (
        df[df["route_id"] == route_id]
        .sort_values("timestamp")
        .set_index("timestamp")[TARGET_COL]
        .asfreq(FREQ)
        .interpolate(method="time")
        .bfill().ffill()
    )


# ── Собрать PandasDataset ──
def build_gluonts_dataset(route_series: dict) -> PandasDataset:
    return PandasDataset(
        {str(rid): s.rename("target") for rid, s in route_series.items()},
        target="target",
    )

# ── Патч torch.load для совместимости с PyTorch >= 2.6 ──
import torch as _torch
_original_torch_load = _torch.load

def _patched_torch_load(f, map_location=None, pickle_module=None, weights_only=True, **kwargs):
    return _original_torch_load(f, map_location=map_location, weights_only=False, **kwargs)

_torch.load = _patched_torch_load
print("torch.load patched (weights_only=False) ✅")
# ── Создать predictor из чекпоинта ──
def build_predictor(prediction_length=FORECAST_STEPS):
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
    ea   = ckpt["hyper_parameters"]["model_kwargs"]
    estimator = LagLlamaEstimator(
        ckpt_path=CKPT_PATH,
        prediction_length=prediction_length,
        context_length=CONTEXT_LENGTH,
        input_size=ea["input_size"],
        n_layer=ea["n_layer"],
        n_embd_per_head=ea["n_embd_per_head"],
        n_head=ea["n_head"],
        scaling=ea["scaling"],
        time_feat=ea["time_feat"],
        rope_scaling={
            "type": "linear",
            "factor": max(1.0, (CONTEXT_LENGTH + prediction_length) / ea["context_length"]),
        },
        batch_size=64,
        num_parallel_samples=NUM_SAMPLES,
        device=DEVICE,
    )
    lm   = estimator.create_lightning_module()
    tr   = estimator.create_transformation()
    return estimator.create_predictor(tr, lm)


# ============================================================
# ROLLING CROSS-VALIDATION
# fold 1: series[:-32]  →  val = series[-40:-32]
# fold 2: series[:-24]  →  val = series[-32:-24]
# fold 3: series[:-16]  →  val = series[-24:-16]
# fold 4: series[: -8]  →  val = series[-16: -8]
# fold 5: series        →  val = series[ -8:end]
# ============================================================
print("=" * 65)
print(f"ROLLING CV  ({N_FOLDS} folds × {FORECAST_STEPS} steps = {TOTAL_VAL_PTS} pts)")
print("=" * 65)

raw_rows = []

for fold in range(1, N_FOLDS + 1):
    val_end_offset   = TOTAL_VAL_PTS - (fold - 1) * FORECAST_STEPS
    val_start_offset = val_end_offset - FORECAST_STEPS

    route_series = {}
    ground_truth = {}
    valid_routes = []

    for route_id in route_ids:
        s = prepare_series(train_df, route_id)
        if len(s) <= val_end_offset:
            continue

        ctx_series = s.iloc[:-val_start_offset] if val_start_offset > 0 else s
        gt_vals    = (s.iloc[-val_end_offset:-val_start_offset]
                      if val_start_offset > 0 else s.iloc[-val_end_offset:])

        route_series[route_id] = ctx_series
        ground_truth[route_id] = gt_vals
        valid_routes.append(route_id)

    print(f"\nFold {fold}/{N_FOLDS} — routes={len(valid_routes)}, building predictor…")
    dataset   = build_gluonts_dataset(route_series)
    predictor = build_predictor()

    fc_it, _ = make_evaluation_predictions(
        dataset=dataset, predictor=predictor, num_samples=NUM_SAMPLES
    )
    forecasts = list(fc_it)

    fold_y_true, fold_y_pred = [], []
    for fc, route_id in zip(forecasts, valid_routes):
        point = np.clip(fc.samples.mean(axis=0), 0, None)
        gt    = ground_truth[route_id]

        for h_idx in range(FORECAST_STEPS):
            y_true = float(gt.values[h_idx])
            y_pred = float(point[h_idx])
            ae     = abs(y_pred - y_true)
            raw_rows.append({
                "fold":      fold,
                "route_id":  route_id,
                "timestamp": gt.index[h_idx],
                "h":         h_idx + 1,
                "y_true":    y_true,
                "y_pred":    y_pred,
                "ae":        ae,
                "ape":       ae / (y_true + 1e-9),
                "err":       y_pred - y_true,
            })
            fold_y_true.append(y_true)
            fold_y_pred.append(y_pred)

    tot, wape, rb = wape_rbias(fold_y_true, fold_y_pred)
    print(f"  RAW   WAPE={wape:.4f}  |RBias|={rb:.4f}  Total={tot:.4f}")


# ── Агрегаты ──
raw_cv_df = pd.DataFrame(raw_rows)

agg_fold = raw_cv_df.groupby("fold").apply(lambda d: pd.Series({
    "wape":  np.abs(d["err"]).sum() / (d["y_true"].sum() + 1e-9),
    "rbias": abs(d["y_pred"].sum() / (d["y_true"].sum() + 1e-9) - 1),
    "mae":   d["ae"].mean(),
    "n":     len(d),
})).reset_index()

agg_h = raw_cv_df.groupby("h").apply(lambda d: pd.Series({
    "wape":  np.abs(d["err"]).sum() / (d["y_true"].sum() + 1e-9),
    "rbias": abs(d["y_pred"].sum() / (d["y_true"].sum() + 1e-9) - 1),
    "mae":   d["ae"].mean(),
})).reset_index()

agg_route = raw_cv_df.groupby("route_id").agg(
    mae=("ae","mean"), mape=("ape","mean"),
    mean_err=("err","mean"), mean_true=("y_true","mean"),
).reset_index()
rs = raw_cv_df.groupby("route_id").agg(
    sum_true=("y_true","sum"), sum_pred=("y_pred","sum"), sum_ae=("ae","sum"),
).reset_index()
agg_route = agg_route.merge(rs, on="route_id")
agg_route["wape"]        = agg_route["sum_ae"]   / (agg_route["sum_true"] + 1e-9)
agg_route["calib_scale"] = agg_route["sum_true"] / (agg_route["sum_pred"] + 1e-9)
agg_route.drop(columns=["sum_true","sum_pred","sum_ae"], inplace=True)

yt_all = raw_cv_df["y_true"].values
yp_all = raw_cv_df["y_pred"].values
tot_r, wape_r, rb_r = wape_rbias(yt_all, yp_all)
calib = float(yt_all.sum() / (yp_all.sum() + 1e-9))
yp_c  = np.clip(yp_all * calib, 0, None)
tot_c, wape_c, rb_c = wape_rbias(yt_all, yp_c)

print("\n" + "=" * 65)
print("SUMMARY")
print("=" * 65)
print("\n── По фолдам ──")
print(agg_fold.to_string(index=False))
print("\n── По шагам h ──")
print(agg_h.to_string(index=False))
print(f"\n{'Global RAW':25s}  WAPE={wape_r:.4f}  |RBias|={rb_r:.4f}  Total={tot_r:.4f}")
print(f"{'Global CALIBRATED':25s}  WAPE={wape_c:.4f}  |RBias|={rb_c:.4f}  Total={tot_c:.4f}")
print(f"  calib_scale = {calib:.4f}")
print(f"\nCalibration {'HELPS ✅' if tot_c < tot_r else 'HURTS ❌'}  (Δ = {tot_c-tot_r:+.4f})")

raw_cv_df.to_csv("lagllama_cv_raw.csv",       index=False)
agg_route.to_csv("lagllama_cv_agg_route.csv", index=False)
agg_fold.to_csv("lagllama_cv_agg_fold.csv",   index=False)
agg_h.to_csv("lagllama_cv_agg_h.csv",         index=False)
print("\n✅ lagllama_cv_raw / agg_route / agg_fold / agg_h saved")


# ============================================================
# TEST PREDICTION
# ============================================================
print("\n" + "=" * 65)
print("TEST PREDICTION")
print("=" * 65)

test_route_ids = test_df["route_id"].unique().tolist()
test_series    = {}
test_meta      = {}

for rid in test_route_ids:
    s = prepare_series(train_df, rid)
    test_series[rid] = s
    test_meta[rid]   = test_df[test_df["route_id"] == rid].sort_values("timestamp")

print(f"Routes: {len(test_route_ids)}  Running Lag-Llama inference…")
test_dataset  = build_gluonts_dataset(test_series)
test_predictor = build_predictor()
test_forecasts = list(test_predictor.predict(test_dataset))

predictions_raw = {}
for fc, rid in tqdm(zip(test_forecasts, test_route_ids), total=len(test_route_ids)):
    point      = np.clip(fc.samples.mean(axis=0), 0, None)
    route_test = test_meta[rid]
    for j, (_, row) in enumerate(route_test.iterrows()):
        pred_val = float(point[j]) if j < len(point) else float(point[-1])
        predictions_raw[row["id"]] = max(0.0, pred_val)

sub_raw = (pd.DataFrame(list(predictions_raw.items()), columns=["id","y_pred"])
           .sort_values("id").reset_index(drop=True))
sub_cal = sub_raw.copy()
sub_cal["y_pred"] = np.clip(sub_cal["y_pred"] * calib, 0, None)

sub_raw.to_csv("submission_lagllama_raw.csv",        index=False)
sub_cal.to_csv("submission_lagllama_calibrated.csv", index=False)
print(f"\n✅ submission_lagllama_raw.csv")
print(f"✅ submission_lagllama_calibrated.csv  (scale={calib:.4f})")
print(f"\n── Raw ──\n{sub_raw['y_pred'].describe().round(2)}")
print(f"\n── Calibrated ──\n{sub_cal['y_pred'].describe().round(2)}")

In [ ]:
# ── SSL KILL SWITCH — вставить самым первым блоком ──
import ssl
import os

# 1. stdlib ssl
ssl._create_default_https_context = ssl._create_unverified_context

# 2. env vars для curl / requests
os.environ["CURL_CA_BUNDLE"]                = ""
os.environ["REQUESTS_CA_BUNDLE"]            = ""
os.environ["SSL_CERT_FILE"]                 = ""
os.environ["HF_HUB_DISABLE_SSL_VERIFICATION"] = "1"

# 3. Патчим requests до импорта huggingface
import requests
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Monkey-patch: все requests.get/post/session → verify=False
_original_request = requests.Session.request
def _patched_request(self, method, url, **kwargs):
    kwargs.setdefault("verify", False)
    return _original_request(self, method, url, **kwargs)
requests.Session.request = _patched_request

# 4. Патчим huggingface_hub если уже установлен
try:
    import huggingface_hub.file_download as _hf_fd
    _hf_fd._CACHED_NO_EXIST  # просто проверяем что модуль есть
    import huggingface_hub.utils._http as _hf_http
    _orig_get_session = _hf_http.get_session

    def _patched_get_session():
        s = _orig_get_session()
        s.verify = False
        return s

    _hf_http.get_session = _patched_get_session
except Exception:
    pass

print("SSL disabled ✅")


In [ ]:
from huggingface_hub import hf_hub_download

hf_hub_download(
    repo_id="time-series-foundation-models/Lag-Llama",
    filename="lag-llama.ckpt",
    local_dir="./lag-llama",
)
print("✅ lag-llama.ckpt downloaded")

### Уменьшил историю до 21 дня

In [ ]:
# ============================================================
# N-HiTS — Rolling CV (5 folds × 8 steps)
# + per-route статистика для блендинга (как в Chronos-2)
# ============================================================



import warnings
import numpy as np
import pandas as pd
import torch
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MQLoss
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None


def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias


# ── Config ──
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8        # = h
N_FOLDS        = 5
TOTAL_VAL_PTS  = N_FOLDS * FORECAST_STEPS   # 40
CONTEXT_LEN    = 2048
FREQ           = "30min"

INPUT_SIZE = 336      # 7 суток × 48 точек — ловим недельную сезонность
MAX_STEPS  = 800     # чуть больше, раз быстро
BATCH_SIZE = 64       # можно увеличить если памяти хватает


# ── Device ──
if torch.backends.mps.is_available():
    ACCELERATOR = "mps"
    print("Apple Silicon MPS ✅")
elif torch.cuda.is_available():
    ACCELERATOR = "gpu"
else:
    ACCELERATOR = "cpu"
print(f"Device: {ACCELERATOR}\n")


# ── Data ──
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

status_cols = sorted([c for c in train_df.columns if c.startswith("status_")])
print(f"Train: {train_df.shape}  Test: {test_df.shape}")
print(f"Status cols: {status_cols}\n")


FUTR_EXOG = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "is_weekend"]

def add_calendar_features(df: pd.DataFrame, ts_col: str = "ds") -> pd.DataFrame:
    ts = pd.to_datetime(df[ts_col])
    df = df.copy()
    df["hour_sin"]   = np.sin(2 * np.pi * ts.dt.hour / 24)
    df["hour_cos"]   = np.cos(2 * np.pi * ts.dt.hour / 24)
    df["dow_sin"]    = np.sin(2 * np.pi * ts.dt.dayofweek / 7)
    df["dow_cos"]    = np.cos(2 * np.pi * ts.dt.dayofweek / 7)
    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(float)
    return df

HIST_EXOG_EXTRA = ["lag_48", "lag_336"]

def add_lag_features(nf_df: pd.DataFrame) -> pd.DataFrame:
    nf_df = nf_df.sort_values(["unique_id", "ds"]).copy()
    nf_df["lag_48"] = (
        nf_df.groupby("unique_id")["y"]
        .transform(lambda s: s.shift(48).bfill().fillna(0))
    )
    nf_df["lag_336"] = (
        nf_df.groupby("unique_id")["y"]
        .transform(lambda s: s.shift(336).bfill().fillna(0))
    )
    return nf_df

# ── NF-формат: unique_id | ds | y [| status_* ...] ──
# Аналог prepare_context_df из Chronos — те же asfreq / interpolate / tail
def to_nf_df(df: pd.DataFrame,
             min_len: int = 32,
             context_len: int = CONTEXT_LEN) -> pd.DataFrame:
    rows = []
    for route_id, grp in df.groupby("route_id"):
        grp = (
            grp.sort_values("timestamp")
            .set_index("timestamp")[[TARGET_COL] + status_cols]
            .asfreq(FREQ)
            .interpolate(method="time")
            .bfill().ffill()
            .tail(context_len)
            .reset_index()
        )
        if len(grp) < min_len:
            continue
        grp.insert(0, "unique_id", route_id)
        grp = grp.rename(columns={"timestamp": "ds", TARGET_COL: "y"})
        rows.append(grp)
    nf_df = pd.concat(rows, ignore_index=True)
    nf_df = add_calendar_features(nf_df)   # futr_exog
    nf_df = add_lag_features(nf_df)         # hist_exog
    return nf_df


# ── Фабрика модели (один конфиг для CV и финального фита) ──
HIST_EXOG = (status_cols + HIST_EXOG_EXTRA) or None

def make_nhits(max_steps: int = MAX_STEPS) -> NHITS:
    return NHITS(
        h                   = FORECAST_STEPS,
        input_size          = INPUT_SIZE,          # 336 = 1 неделя
        loss                = MQLoss(level=[80]),
        # Стеки: день / 4ч / 30мин — каждый на своей частоте
        # Stack 1: MaxPool(48) → 7 дневных точек  (недельный тренд)
        # Stack 2: MaxPool(8)  → 42 точки         (4-часовые блоки)
        # Stack 3: MaxPool(1)  → 336 точек         (полное разрешение)
        n_freq_downsample   = [48, 8, 1],
        mlp_units           = [[512, 512]] * 3,
        n_blocks            = [4, 4, 4],
        max_steps           = max_steps,
        batch_size          = BATCH_SIZE,
        accelerator         = ACCELERATOR,
        hist_exog_list      = HIST_EXOG,
        futr_exog_list      = FUTR_EXOG,
        scaler_type         = "robust",            # median/IQR — пики не "приплющиваются"
        enable_progress_bar = True,
    )

def make_futr_df(route_ids: list, train_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for route_id in route_ids:
        last_ts = train_df[train_df["route_id"] == route_id]["timestamp"].max()
        future_ts = pd.date_range(
            start   = last_ts + pd.Timedelta("30min"),
            periods = FORECAST_STEPS,
            freq    = FREQ,
        )
        for ts in future_ts:
            rows.append({"unique_id": route_id, "ds": ts})
    return add_calendar_features(pd.DataFrame(rows))

# ── Выбор колонки точечного прогноза (аналог get_pred_col из Chronos) ──
def get_pred_col(pred_df: pd.DataFrame) -> str:
    for candidate in ["NHITS-median", "NHITS-q-0.50", "NHITS"]:
        if candidate in pred_df.columns:
            return candidate
    skip = {"unique_id", "ds", "y", "cutoff"}
    num_cols = [c for c in pred_df.columns
                if c not in skip and pd.api.types.is_numeric_dtype(pred_df[c])]
    median_cols = [c for c in num_cols if "median" in c or "0.5" in c or "50" in c]
    return (median_cols or num_cols)[0]


# ── Словарь предсказаний: route_id → array[FORECAST_STEPS] ──
def build_preds_dict(pred_df: pd.DataFrame, pred_col: str) -> dict:
    result = {}
    for route_id, grp in pred_df.groupby("unique_id"):
        result[route_id] = np.clip(grp.sort_values("ds")[pred_col].values, 0, None)
    return result


# ============================================================
# ROLLING CROSS-VALIDATION
# neuralforecast.cross_validation(n_windows=5, step_size=8, refit=True)
# window 1: train=[:-40]  val=[-40:-32]
# window 2: train=[:-32]  val=[-32:-24]
# window 3: train=[:-24]  val=[-24:-16]
# window 4: train=[:-16]  val=[-16: -8]
# window 5: train=[: -8]  val=[ -8:end]
# refit=True — переобучение на каждом фолде (= поведение Chronos)
# ============================================================
print("=" * 65)
print(f"ROLLING CV  ({N_FOLDS} folds × {FORECAST_STEPS} steps = {TOTAL_VAL_PTS} points)")
print("=" * 65)

train_nf = to_nf_df(
    train_df,
    min_len     = TOTAL_VAL_PTS + INPUT_SIZE + 1,
    context_len = CONTEXT_LEN,
)
print(f"NF train: {train_nf.shape}  routes: {train_nf['unique_id'].nunique()}\n")

nf_cv = NeuralForecast(models=[make_nhits()], freq=FREQ)

# refit=True: модель переобучается для каждого фолда (как Chronos per-fold inference)
# refit=False: обучается один раз, быстрее, но менее честно
cv_df = nf_cv.cross_validation(
    df        = train_nf,
    n_windows = N_FOLDS,
    step_size = FORECAST_STEPS,
    refit     = True,
)

print(f"\nCV columns: {list(cv_df.columns)}")
pred_col = get_pred_col(cv_df)
print(f"Point forecast column: '{pred_col}'\n")

# ── cutoff → fold (1 = самый старый, N_FOLDS = самый свежий) ──
cutoffs_sorted   = sorted(cv_df["cutoff"].unique())
cutoff_to_fold   = {c: i + 1 for i, c in enumerate(cutoffs_sorted)}
cv_df["fold"]    = cv_df["cutoff"].map(cutoff_to_fold)

# ── шаг h внутри фолда (1..FORECAST_STEPS) ──
cv_df = cv_df.sort_values(["fold", "unique_id", "ds"]).reset_index(drop=True)
cv_df["h"] = cv_df.groupby(["fold", "unique_id"]).cumcount() + 1

# ── raw_rows (идентична структура с Chronos) ──
raw_rows = []
for _, row in cv_df.iterrows():
    y_true = float(row["y"])
    y_pred = float(np.clip(row[pred_col], 0, None))
    ae     = abs(y_pred - y_true)
    ape    = ae / (y_true + 1e-9)
    err    = y_pred - y_true
    raw_rows.append({
        "fold":      int(row["fold"]),
        "route_id":  row["unique_id"],
        "timestamp": row["ds"],
        "h":         int(row["h"]),
        "y_true":    y_true,
        "y_pred":    y_pred,
        "ae":        ae,
        "ape":       ape,
        "err":       err,
    })

# ── per-fold console output (как у Chronos) ──
for fold in range(1, N_FOLDS + 1):
    fdf        = cv_df[cv_df["fold"] == fold]
    n_routes   = fdf["unique_id"].nunique()
    val_end    = TOTAL_VAL_PTS - (fold - 1) * FORECAST_STEPS
    val_start  = val_end - FORECAST_STEPS
    yt = fdf["y"].values
    yp = np.clip(fdf[pred_col].values, 0, None)
    tot, wape, rb = wape_rbias(yt, yp)
    print(f"\nFold {fold}/{N_FOLDS}  "
          f"(val: -{val_end}..{'-'+str(val_start) if val_start else 'end'})  "
          f"routes={n_routes}")
    print(f"  RAW   WAPE={wape:.4f}  |RBias|={rb:.4f}  Total={tot:.4f}")


# ============================================================
# АГРЕГАТЫ (идентично Chronos для блендинга)
# ============================================================
raw_cv_df = pd.DataFrame(raw_rows)

# ── По фолду ──
agg_fold = raw_cv_df.groupby("fold").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
    "n":     len(df),
})).reset_index()

# ── По шагу h ──
agg_h = raw_cv_df.groupby("h").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
})).reset_index()

# ── По маршруту ──
agg_route = raw_cv_df.groupby("route_id").agg(
    n_points   = ("y_true",  "count"),
    mae        = ("ae",      "mean"),
    median_ae  = ("ae",      "median"),
    std_ae     = ("ae",      "std"),
    mape       = ("ape",     "mean"),
    median_ape = ("ape",     "median"),
    mean_err   = ("err",     "mean"),
    median_err = ("err",     "median"),
    mean_true  = ("y_true",  "mean"),
    mean_pred  = ("y_pred",  "mean"),
).reset_index()
agg_route["calib_scale"] = (
    raw_cv_df.groupby("route_id")["y_true"].sum().values /
    (raw_cv_df.groupby("route_id")["y_pred"].sum().values + 1e-9)
)
agg_route["wape"] = (
    raw_cv_df.groupby("route_id")["ae"].sum().values /
    (raw_cv_df.groupby("route_id")["y_true"].sum().values + 1e-9)
)

# ── Глобальные метрики ──
y_true_all = raw_cv_df["y_true"].values
y_pred_all = raw_cv_df["y_pred"].values
total_raw, wape_raw, rbias_raw = wape_rbias(y_true_all, y_pred_all)

calib_scale = float(y_true_all.sum() / (y_pred_all.sum() + 1e-9))
y_pred_cal  = np.clip(y_pred_all * calib_scale, 0, None)
total_cal, wape_cal, rbias_cal = wape_rbias(y_true_all, y_pred_cal)

print("\n" + "=" * 65)
print("SUMMARY")
print("=" * 65)
print("\n── По фолдам ──")
print(agg_fold.to_string(index=False))
print("\n── По шагам h ──")
print(agg_h.to_string(index=False))
print(f"\n{'Global RAW':25s}  WAPE={wape_raw:.4f}  |RBias|={rbias_raw:.4f}  Total={total_raw:.4f}")
print(f"{'Global CALIBRATED':25s}  WAPE={wape_cal:.4f}  |RBias|={rbias_cal:.4f}  Total={total_cal:.4f}")
print(f"  calib_scale = {calib_scale:.4f}")
print(f"\nCalibration {'HELPS ✅' if total_cal < total_raw else 'HURTS ❌'} "
      f"(Δ = {total_cal - total_raw:+.4f})")
print("\n── Топ-10 сложных маршрутов (по median_ape) ──")
print(agg_route.sort_values("median_ape", ascending=False)
      [["route_id", "mae", "median_ae", "mape", "median_ape", "mean_err", "calib_scale"]]
      .head(10).to_string(index=False))

raw_cv_df.to_csv("nhits_cv_raw.csv", index=False)
agg_route.to_csv("nhits_cv_agg_route.csv", index=False)
agg_fold.to_csv("nhits_cv_agg_fold.csv", index=False)
agg_h.to_csv("nhits_cv_agg_h.csv", index=False)
print("\n✅ nhits_cv_raw.csv")
print("✅ nhits_cv_agg_route.csv")
print("✅ nhits_cv_agg_fold.csv")
print("✅ nhits_cv_agg_h.csv")


# ============================================================
# TEST PREDICTION
# Финальный фит на полных данных, predict() берёт последние
# input_size точек из тренировочного датасета автоматически
# ============================================================
print("\n" + "=" * 65)
print("TEST PREDICTION")
print("=" * 65)

train_nf_full = to_nf_df(train_df, min_len=INPUT_SIZE + 1, context_len=CONTEXT_LEN)
test_route_ids = test_df["route_id"].unique().tolist()
print(f"Routes: {len(test_route_ids)}  Fitting on full train data...")

nf_full = NeuralForecast(models=[make_nhits()], freq=FREQ)
nf_full.fit(train_nf_full)

print("Running inference...")
futr_df      = make_futr_df(test_route_ids, train_df)
test_pred_df = nf_full.predict(futr_df=futr_df)   # next FORECAST_STEPS × 30min на каждый маршрут
test_pred_col = get_pred_col(test_pred_df)
test_preds    = build_preds_dict(test_pred_df, test_pred_col)

predictions_raw = {}
for route_id in tqdm(test_route_ids, desc="Build submission"):
    route_test = test_df[test_df["route_id"] == route_id].sort_values("timestamp")
    preds = test_preds.get(route_id, np.zeros(FORECAST_STEPS))
    for j, (_, row) in enumerate(route_test.iterrows()):
        pred = float(preds[j]) if j < len(preds) else float(preds[-1])
        predictions_raw[row["id"]] = max(0.0, pred)

submission_raw = (
    pd.DataFrame(list(predictions_raw.items()), columns=["id", "y_pred"])
    .sort_values("id").reset_index(drop=True)
)
submission_cal = submission_raw.copy()
submission_cal["y_pred"] = np.clip(submission_cal["y_pred"] * calib_scale, 0, None)

submission_raw.to_csv("submission_nhits_raw.csv", index=False)
submission_cal.to_csv("submission_nhits_calibrated.csv", index=False)
print(f"\n✅ submission_nhits_raw.csv")
print(f"✅ submission_nhits_calibrated.csv")
print(f"\n── Raw ──\n{submission_raw['y_pred'].describe().round(2)}")
print(f"\n── Calibrated (scale={calib_scale:.4f}) ──\n{submission_cal['y_pred'].describe().round(2)}")

### Вытащил лаги в futr exog

In [ ]:
# ============================================================
# N-HiTS — Rolling CV (5 folds × 8 steps)
# + per-route статистика для блендинга (как в Chronos-2)
# ============================================================



import warnings
import numpy as np
import pandas as pd
import torch
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MQLoss
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None


def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias


# ── Config ──
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8        # = h
N_FOLDS        = 5
TOTAL_VAL_PTS  = N_FOLDS * FORECAST_STEPS   # 40
CONTEXT_LEN    = 2048
FREQ           = "30min"

INPUT_SIZE = 336      # 7 суток × 48 точек — ловим недельную сезонность
MAX_STEPS  = 800     # чуть больше, раз быстро
BATCH_SIZE = 64       # можно увеличить если памяти хватает


# ── Device ──
if torch.backends.mps.is_available():
    ACCELERATOR = "mps"
    print("Apple Silicon MPS ✅")
elif torch.cuda.is_available():
    ACCELERATOR = "gpu"
else:
    ACCELERATOR = "cpu"
print(f"Device: {ACCELERATOR}\n")


# ── Data ──
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

status_cols = sorted([c for c in train_df.columns if c.startswith("status_")])
print(f"Train: {train_df.shape}  Test: {test_df.shape}")
print(f"Status cols: {status_cols}\n")


FUTR_EXOG = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "is_weekend"]

def add_calendar_features(df: pd.DataFrame, ts_col: str = "ds") -> pd.DataFrame:
    ts = pd.to_datetime(df[ts_col])
    df = df.copy()
    df["hour_sin"]   = np.sin(2 * np.pi * ts.dt.hour / 24)
    df["hour_cos"]   = np.cos(2 * np.pi * ts.dt.hour / 24)
    df["dow_sin"]    = np.sin(2 * np.pi * ts.dt.dayofweek / 7)
    df["dow_cos"]    = np.cos(2 * np.pi * ts.dt.dayofweek / 7)
    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(float)
    return df


# ── NF-формат: unique_id | ds | y [| status_* ...] ──
# Аналог prepare_context_df из Chronos — те же asfreq / interpolate / tail
def to_nf_df(df: pd.DataFrame,
             min_len: int = 32,
             context_len: int = CONTEXT_LEN) -> pd.DataFrame:
    rows = []
    for route_id, grp in df.groupby("route_id"):
        grp = (
            grp.sort_values("timestamp")
            .set_index("timestamp")[[TARGET_COL] + status_cols]
            .asfreq(FREQ)
            .interpolate(method="time")
            .bfill().ffill()
            .tail(context_len)
            .reset_index()
        )
        if len(grp) < min_len:
            continue
        grp.insert(0, "unique_id", route_id)
        grp = grp.rename(columns={"timestamp": "ds", TARGET_COL: "y"})
        rows.append(grp)
    nf_df = pd.concat(rows, ignore_index=True)
    nf_df = add_calendar_features(nf_df)   # futr_exog
    return nf_df


# ── Фабрика модели (один конфиг для CV и финального фита) ──
HIST_EXOG = (status_cols) or None

def make_nhits(max_steps: int = MAX_STEPS) -> NHITS:
    return NHITS(
        h                   = FORECAST_STEPS,
        input_size          = INPUT_SIZE,          # 336 = 1 неделя
        loss                = MQLoss(level=[80]),
        # Стеки: день / 4ч / 30мин — каждый на своей частоте
        # Stack 1: MaxPool(48) → 7 дневных точек  (недельный тренд)
        # Stack 2: MaxPool(8)  → 42 точки         (4-часовые блоки)
        # Stack 3: MaxPool(1)  → 336 точек         (полное разрешение)
        n_freq_downsample   = [48, 8, 1],
        mlp_units           = [[512, 512]] * 3,
        n_blocks            = [4, 4, 4],
        max_steps           = max_steps,
        batch_size          = BATCH_SIZE,
        accelerator         = ACCELERATOR,
        hist_exog_list      = HIST_EXOG,
        futr_exog_list      = FUTR_EXOG,
        scaler_type         = "robust",            # median/IQR — пики не "приплющиваются"
        enable_progress_bar = True,
    )

def make_futr_df(route_ids: list, train_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for route_id in route_ids:
        last_ts = train_df[train_df["route_id"] == route_id]["timestamp"].max()
        future_ts = pd.date_range(
            start   = last_ts + pd.Timedelta("30min"),
            periods = FORECAST_STEPS,
            freq    = FREQ,
        )
        for ts in future_ts:
            rows.append({"unique_id": route_id, "ds": ts})
    return add_calendar_features(pd.DataFrame(rows))

# ── Выбор колонки точечного прогноза (аналог get_pred_col из Chronos) ──
def get_pred_col(pred_df: pd.DataFrame) -> str:
    for candidate in ["NHITS-median", "NHITS-q-0.50", "NHITS"]:
        if candidate in pred_df.columns:
            return candidate
    skip = {"unique_id", "ds", "y", "cutoff"}
    num_cols = [c for c in pred_df.columns
                if c not in skip and pd.api.types.is_numeric_dtype(pred_df[c])]
    median_cols = [c for c in num_cols if "median" in c or "0.5" in c or "50" in c]
    return (median_cols or num_cols)[0]


# ── Словарь предсказаний: route_id → array[FORECAST_STEPS] ──
def build_preds_dict(pred_df: pd.DataFrame, pred_col: str) -> dict:
    result = {}
    for route_id, grp in pred_df.groupby("unique_id"):
        result[route_id] = np.clip(grp.sort_values("ds")[pred_col].values, 0, None)
    return result


# ============================================================
# ROLLING CROSS-VALIDATION
# neuralforecast.cross_validation(n_windows=5, step_size=8, refit=True)
# window 1: train=[:-40]  val=[-40:-32]
# window 2: train=[:-32]  val=[-32:-24]
# window 3: train=[:-24]  val=[-24:-16]
# window 4: train=[:-16]  val=[-16: -8]
# window 5: train=[: -8]  val=[ -8:end]
# refit=True — переобучение на каждом фолде (= поведение Chronos)
# ============================================================
print("=" * 65)
print(f"ROLLING CV  ({N_FOLDS} folds × {FORECAST_STEPS} steps = {TOTAL_VAL_PTS} points)")
print("=" * 65)

train_nf = to_nf_df(
    train_df,
    min_len     = TOTAL_VAL_PTS + INPUT_SIZE + 1,
    context_len = CONTEXT_LEN,
)
print(f"NF train: {train_nf.shape}  routes: {train_nf['unique_id'].nunique()}\n")

nf_cv = NeuralForecast(models=[make_nhits()], freq=FREQ)

# refit=True: модель переобучается для каждого фолда (как Chronos per-fold inference)
# refit=False: обучается один раз, быстрее, но менее честно
cv_df = nf_cv.cross_validation(
    df        = train_nf,
    n_windows = N_FOLDS,
    step_size = FORECAST_STEPS,
    refit     = True,
)

print(f"\nCV columns: {list(cv_df.columns)}")
pred_col = get_pred_col(cv_df)
print(f"Point forecast column: '{pred_col}'\n")

# ── cutoff → fold (1 = самый старый, N_FOLDS = самый свежий) ──
cutoffs_sorted   = sorted(cv_df["cutoff"].unique())
cutoff_to_fold   = {c: i + 1 for i, c in enumerate(cutoffs_sorted)}
cv_df["fold"]    = cv_df["cutoff"].map(cutoff_to_fold)

# ── шаг h внутри фолда (1..FORECAST_STEPS) ──
cv_df = cv_df.sort_values(["fold", "unique_id", "ds"]).reset_index(drop=True)
cv_df["h"] = cv_df.groupby(["fold", "unique_id"]).cumcount() + 1

# ── raw_rows (идентична структура с Chronos) ──
raw_rows = []
for _, row in cv_df.iterrows():
    y_true = float(row["y"])
    y_pred = float(np.clip(row[pred_col], 0, None))
    ae     = abs(y_pred - y_true)
    ape    = ae / (y_true + 1e-9)
    err    = y_pred - y_true
    raw_rows.append({
        "fold":      int(row["fold"]),
        "route_id":  row["unique_id"],
        "timestamp": row["ds"],
        "h":         int(row["h"]),
        "y_true":    y_true,
        "y_pred":    y_pred,
        "ae":        ae,
        "ape":       ape,
        "err":       err,
    })

# ── per-fold console output (как у Chronos) ──
for fold in range(1, N_FOLDS + 1):
    fdf        = cv_df[cv_df["fold"] == fold]
    n_routes   = fdf["unique_id"].nunique()
    val_end    = TOTAL_VAL_PTS - (fold - 1) * FORECAST_STEPS
    val_start  = val_end - FORECAST_STEPS
    yt = fdf["y"].values
    yp = np.clip(fdf[pred_col].values, 0, None)
    tot, wape, rb = wape_rbias(yt, yp)
    print(f"\nFold {fold}/{N_FOLDS}  "
          f"(val: -{val_end}..{'-'+str(val_start) if val_start else 'end'})  "
          f"routes={n_routes}")
    print(f"  RAW   WAPE={wape:.4f}  |RBias|={rb:.4f}  Total={tot:.4f}")


# ============================================================
# АГРЕГАТЫ (идентично Chronos для блендинга)
# ============================================================
raw_cv_df = pd.DataFrame(raw_rows)

# ── По фолду ──
agg_fold = raw_cv_df.groupby("fold").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
    "n":     len(df),
})).reset_index()

# ── По шагу h ──
agg_h = raw_cv_df.groupby("h").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
})).reset_index()

# ── По маршруту ──
agg_route = raw_cv_df.groupby("route_id").agg(
    n_points   = ("y_true",  "count"),
    mae        = ("ae",      "mean"),
    median_ae  = ("ae",      "median"),
    std_ae     = ("ae",      "std"),
    mape       = ("ape",     "mean"),
    median_ape = ("ape",     "median"),
    mean_err   = ("err",     "mean"),
    median_err = ("err",     "median"),
    mean_true  = ("y_true",  "mean"),
    mean_pred  = ("y_pred",  "mean"),
).reset_index()
agg_route["calib_scale"] = (
    raw_cv_df.groupby("route_id")["y_true"].sum().values /
    (raw_cv_df.groupby("route_id")["y_pred"].sum().values + 1e-9)
)
agg_route["wape"] = (
    raw_cv_df.groupby("route_id")["ae"].sum().values /
    (raw_cv_df.groupby("route_id")["y_true"].sum().values + 1e-9)
)

# ── Глобальные метрики ──
y_true_all = raw_cv_df["y_true"].values
y_pred_all = raw_cv_df["y_pred"].values
total_raw, wape_raw, rbias_raw = wape_rbias(y_true_all, y_pred_all)

calib_scale = float(y_true_all.sum() / (y_pred_all.sum() + 1e-9))
y_pred_cal  = np.clip(y_pred_all * calib_scale, 0, None)
total_cal, wape_cal, rbias_cal = wape_rbias(y_true_all, y_pred_cal)

print("\n" + "=" * 65)
print("SUMMARY")
print("=" * 65)
print("\n── По фолдам ──")
print(agg_fold.to_string(index=False))
print("\n── По шагам h ──")
print(agg_h.to_string(index=False))
print(f"\n{'Global RAW':25s}  WAPE={wape_raw:.4f}  |RBias|={rbias_raw:.4f}  Total={total_raw:.4f}")
print(f"{'Global CALIBRATED':25s}  WAPE={wape_cal:.4f}  |RBias|={rbias_cal:.4f}  Total={total_cal:.4f}")
print(f"  calib_scale = {calib_scale:.4f}")
print(f"\nCalibration {'HELPS ✅' if total_cal < total_raw else 'HURTS ❌'} "
      f"(Δ = {total_cal - total_raw:+.4f})")
print("\n── Топ-10 сложных маршрутов (по median_ape) ──")
print(agg_route.sort_values("median_ape", ascending=False)
      [["route_id", "mae", "median_ae", "mape", "median_ape", "mean_err", "calib_scale"]]
      .head(10).to_string(index=False))

raw_cv_df.to_csv("nhits_cv_raw.csv", index=False)
agg_route.to_csv("nhits_cv_agg_route.csv", index=False)
agg_fold.to_csv("nhits_cv_agg_fold.csv", index=False)
agg_h.to_csv("nhits_cv_agg_h.csv", index=False)
print("\n✅ nhits_cv_raw.csv")
print("✅ nhits_cv_agg_route.csv")
print("✅ nhits_cv_agg_fold.csv")
print("✅ nhits_cv_agg_h.csv")


# ============================================================
# TEST PREDICTION
# Финальный фит на полных данных, predict() берёт последние
# input_size точек из тренировочного датасета автоматически
# ============================================================
print("\n" + "=" * 65)
print("TEST PREDICTION")
print("=" * 65)

train_nf_full = to_nf_df(train_df, min_len=INPUT_SIZE + 1, context_len=CONTEXT_LEN)
test_route_ids = test_df["route_id"].unique().tolist()
print(f"Routes: {len(test_route_ids)}  Fitting on full train data...")

nf_full = NeuralForecast(models=[make_nhits()], freq=FREQ)
nf_full.fit(train_nf_full)

print("Running inference...")
futr_df      = make_futr_df(test_route_ids, train_df)
test_pred_df = nf_full.predict(futr_df=futr_df)   # next FORECAST_STEPS × 30min на каждый маршрут
test_pred_col = get_pred_col(test_pred_df)
test_preds    = build_preds_dict(test_pred_df, test_pred_col)

predictions_raw = {}
for route_id in tqdm(test_route_ids, desc="Build submission"):
    route_test = test_df[test_df["route_id"] == route_id].sort_values("timestamp")
    preds = test_preds.get(route_id, np.zeros(FORECAST_STEPS))
    for j, (_, row) in enumerate(route_test.iterrows()):
        pred = float(preds[j]) if j < len(preds) else float(preds[-1])
        predictions_raw[row["id"]] = max(0.0, pred)

submission_raw = (
    pd.DataFrame(list(predictions_raw.items()), columns=["id", "y_pred"])
    .sort_values("id").reset_index(drop=True)
)
submission_cal = submission_raw.copy()
submission_cal["y_pred"] = np.clip(submission_cal["y_pred"] * calib_scale, 0, None)

submission_raw.to_csv("submission_nhits_raw.csv", index=False)
submission_cal.to_csv("submission_nhits_calibrated.csv", index=False)
print(f"\n✅ submission_nhits_raw.csv")
print(f"✅ submission_nhits_calibrated.csv")
print(f"\n── Raw ──\n{submission_raw['y_pred'].describe().round(2)}")
print(f"\n── Calibrated (scale={calib_scale:.4f}) ──\n{submission_cal['y_pred'].describe().round(2)}")

### Убрал статусы, которые дают только шум и убрал лаговые ряды

In [ ]:
# ============================================================
# N-HiTS — Rolling CV (5 folds × 8 steps)
# + per-route статистика для блендинга (как в Chronos-2)
# ============================================================



import warnings
import numpy as np
import pandas as pd
import torch
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MQLoss
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None


def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias


# ── Config ──
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8        # = h
N_FOLDS        = 5
TOTAL_VAL_PTS  = N_FOLDS * FORECAST_STEPS   # 40
CONTEXT_LEN    = 2048
FREQ           = "30min"

INPUT_SIZE = 336      # 7 суток × 48 точек — ловим недельную сезонность
MAX_STEPS  = 800     # чуть больше, раз быстро
BATCH_SIZE = 64       # можно увеличить если памяти хватает


# ── Device ──
if torch.backends.mps.is_available():
    ACCELERATOR = "mps"
    print("Apple Silicon MPS ✅")
elif torch.cuda.is_available():
    ACCELERATOR = "gpu"
else:
    ACCELERATOR = "cpu"
print(f"Device: {ACCELERATOR}\n")


# ── Data ──
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

status_cols = sorted([c for c in train_df.columns if c in ["status_3", "status_2", "status_5"]])
print(f"Train: {train_df.shape}  Test: {test_df.shape}")
print(f"Status cols: {status_cols}\n")


FUTR_EXOG = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "is_weekend"]

def add_calendar_features(df: pd.DataFrame, ts_col: str = "ds") -> pd.DataFrame:
    ts = pd.to_datetime(df[ts_col])
    df = df.copy()
    df["hour_sin"]   = np.sin(2 * np.pi * ts.dt.hour / 24)
    df["hour_cos"]   = np.cos(2 * np.pi * ts.dt.hour / 24)
    df["dow_sin"]    = np.sin(2 * np.pi * ts.dt.dayofweek / 7)
    df["dow_cos"]    = np.cos(2 * np.pi * ts.dt.dayofweek / 7)
    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(float)
    return df


# ── NF-формат: unique_id | ds | y [| status_* ...] ──
# Аналог prepare_context_df из Chronos — те же asfreq / interpolate / tail
def to_nf_df(df: pd.DataFrame,
             min_len: int = 32,
             context_len: int = CONTEXT_LEN) -> pd.DataFrame:
    rows = []
    for route_id, grp in df.groupby("route_id"):
        grp = (
            grp.sort_values("timestamp")
            .set_index("timestamp")[[TARGET_COL] + status_cols]
            .asfreq(FREQ)
            .interpolate(method="time")
            .bfill().ffill()
            .tail(context_len)
            .reset_index()
        )
        if len(grp) < min_len:
            continue
        grp.insert(0, "unique_id", route_id)
        grp = grp.rename(columns={"timestamp": "ds", TARGET_COL: "y"})
        rows.append(grp)
    nf_df = pd.concat(rows, ignore_index=True)
    nf_df = add_calendar_features(nf_df)   # futr_exog
    return nf_df


# ── Фабрика модели (один конфиг для CV и финального фита) ──
HIST_EXOG = (status_cols) or None

def make_nhits(max_steps: int = MAX_STEPS) -> NHITS:
    return NHITS(
        h                   = FORECAST_STEPS,
        input_size          = INPUT_SIZE,          # 336 = 1 неделя
        loss                = MQLoss(level=[80]),
        # Стеки: день / 4ч / 30мин — каждый на своей частоте
        # Stack 1: MaxPool(48) → 7 дневных точек  (недельный тренд)
        # Stack 2: MaxPool(8)  → 42 точки         (4-часовые блоки)
        # Stack 3: MaxPool(1)  → 336 точек         (полное разрешение)
        n_freq_downsample   = [48, 8, 1],
        mlp_units           = [[512, 512]] * 3,
        n_blocks            = [4, 4, 4],
        max_steps           = max_steps,
        batch_size          = BATCH_SIZE,
        accelerator         = ACCELERATOR,
        hist_exog_list      = HIST_EXOG,
        futr_exog_list      = FUTR_EXOG,
        scaler_type         = "robust",            # median/IQR — пики не "приплющиваются"
        enable_progress_bar = True,
    )

def make_futr_df(route_ids: list, train_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for route_id in route_ids:
        last_ts = train_df[train_df["route_id"] == route_id]["timestamp"].max()
        future_ts = pd.date_range(
            start   = last_ts + pd.Timedelta("30min"),
            periods = FORECAST_STEPS,
            freq    = FREQ,
        )
        for ts in future_ts:
            rows.append({"unique_id": route_id, "ds": ts})
    return add_calendar_features(pd.DataFrame(rows))

# ── Выбор колонки точечного прогноза (аналог get_pred_col из Chronos) ──
def get_pred_col(pred_df: pd.DataFrame) -> str:
    for candidate in ["NHITS-median", "NHITS-q-0.50", "NHITS"]:
        if candidate in pred_df.columns:
            return candidate
    skip = {"unique_id", "ds", "y", "cutoff"}
    num_cols = [c for c in pred_df.columns
                if c not in skip and pd.api.types.is_numeric_dtype(pred_df[c])]
    median_cols = [c for c in num_cols if "median" in c or "0.5" in c or "50" in c]
    return (median_cols or num_cols)[0]


# ── Словарь предсказаний: route_id → array[FORECAST_STEPS] ──
def build_preds_dict(pred_df: pd.DataFrame, pred_col: str) -> dict:
    result = {}
    for route_id, grp in pred_df.groupby("unique_id"):
        result[route_id] = np.clip(grp.sort_values("ds")[pred_col].values, 0, None)
    return result


# ============================================================
# ROLLING CROSS-VALIDATION
# neuralforecast.cross_validation(n_windows=5, step_size=8, refit=True)
# window 1: train=[:-40]  val=[-40:-32]
# window 2: train=[:-32]  val=[-32:-24]
# window 3: train=[:-24]  val=[-24:-16]
# window 4: train=[:-16]  val=[-16: -8]
# window 5: train=[: -8]  val=[ -8:end]
# refit=True — переобучение на каждом фолде (= поведение Chronos)
# ============================================================
print("=" * 65)
print(f"ROLLING CV  ({N_FOLDS} folds × {FORECAST_STEPS} steps = {TOTAL_VAL_PTS} points)")
print("=" * 65)

train_nf = to_nf_df(
    train_df,
    min_len     = TOTAL_VAL_PTS + INPUT_SIZE + 1,
    context_len = CONTEXT_LEN,
)
print(f"NF train: {train_nf.shape}  routes: {train_nf['unique_id'].nunique()}\n")

nf_cv = NeuralForecast(models=[make_nhits()], freq=FREQ)

# refit=True: модель переобучается для каждого фолда (как Chronos per-fold inference)
# refit=False: обучается один раз, быстрее, но менее честно
cv_df = nf_cv.cross_validation(
    df        = train_nf,
    n_windows = N_FOLDS,
    step_size = FORECAST_STEPS,
    refit     = True,
)

print(f"\nCV columns: {list(cv_df.columns)}")
pred_col = get_pred_col(cv_df)
print(f"Point forecast column: '{pred_col}'\n")

# ── cutoff → fold (1 = самый старый, N_FOLDS = самый свежий) ──
cutoffs_sorted   = sorted(cv_df["cutoff"].unique())
cutoff_to_fold   = {c: i + 1 for i, c in enumerate(cutoffs_sorted)}
cv_df["fold"]    = cv_df["cutoff"].map(cutoff_to_fold)

# ── шаг h внутри фолда (1..FORECAST_STEPS) ──
cv_df = cv_df.sort_values(["fold", "unique_id", "ds"]).reset_index(drop=True)
cv_df["h"] = cv_df.groupby(["fold", "unique_id"]).cumcount() + 1

# ── raw_rows (идентична структура с Chronos) ──
raw_rows = []
for _, row in cv_df.iterrows():
    y_true = float(row["y"])
    y_pred = float(np.clip(row[pred_col], 0, None))
    ae     = abs(y_pred - y_true)
    ape    = ae / (y_true + 1e-9)
    err    = y_pred - y_true
    raw_rows.append({
        "fold":      int(row["fold"]),
        "route_id":  row["unique_id"],
        "timestamp": row["ds"],
        "h":         int(row["h"]),
        "y_true":    y_true,
        "y_pred":    y_pred,
        "ae":        ae,
        "ape":       ape,
        "err":       err,
    })

# ── per-fold console output (как у Chronos) ──
for fold in range(1, N_FOLDS + 1):
    fdf        = cv_df[cv_df["fold"] == fold]
    n_routes   = fdf["unique_id"].nunique()
    val_end    = TOTAL_VAL_PTS - (fold - 1) * FORECAST_STEPS
    val_start  = val_end - FORECAST_STEPS
    yt = fdf["y"].values
    yp = np.clip(fdf[pred_col].values, 0, None)
    tot, wape, rb = wape_rbias(yt, yp)
    print(f"\nFold {fold}/{N_FOLDS}  "
          f"(val: -{val_end}..{'-'+str(val_start) if val_start else 'end'})  "
          f"routes={n_routes}")
    print(f"  RAW   WAPE={wape:.4f}  |RBias|={rb:.4f}  Total={tot:.4f}")


# ============================================================
# АГРЕГАТЫ (идентично Chronos для блендинга)
# ============================================================
raw_cv_df = pd.DataFrame(raw_rows)

# ── По фолду ──
agg_fold = raw_cv_df.groupby("fold").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
    "n":     len(df),
})).reset_index()

# ── По шагу h ──
agg_h = raw_cv_df.groupby("h").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
})).reset_index()

# ── По маршруту ──
agg_route = raw_cv_df.groupby("route_id").agg(
    n_points   = ("y_true",  "count"),
    mae        = ("ae",      "mean"),
    median_ae  = ("ae",      "median"),
    std_ae     = ("ae",      "std"),
    mape       = ("ape",     "mean"),
    median_ape = ("ape",     "median"),
    mean_err   = ("err",     "mean"),
    median_err = ("err",     "median"),
    mean_true  = ("y_true",  "mean"),
    mean_pred  = ("y_pred",  "mean"),
).reset_index()
agg_route["calib_scale"] = (
    raw_cv_df.groupby("route_id")["y_true"].sum().values /
    (raw_cv_df.groupby("route_id")["y_pred"].sum().values + 1e-9)
)
agg_route["wape"] = (
    raw_cv_df.groupby("route_id")["ae"].sum().values /
    (raw_cv_df.groupby("route_id")["y_true"].sum().values + 1e-9)
)

# ── Глобальные метрики ──
y_true_all = raw_cv_df["y_true"].values
y_pred_all = raw_cv_df["y_pred"].values
total_raw, wape_raw, rbias_raw = wape_rbias(y_true_all, y_pred_all)

calib_scale = float(y_true_all.sum() / (y_pred_all.sum() + 1e-9))
y_pred_cal  = np.clip(y_pred_all * calib_scale, 0, None)
total_cal, wape_cal, rbias_cal = wape_rbias(y_true_all, y_pred_cal)

print("\n" + "=" * 65)
print("SUMMARY")
print("=" * 65)
print("\n── По фолдам ──")
print(agg_fold.to_string(index=False))
print("\n── По шагам h ──")
print(agg_h.to_string(index=False))
print(f"\n{'Global RAW':25s}  WAPE={wape_raw:.4f}  |RBias|={rbias_raw:.4f}  Total={total_raw:.4f}")
print(f"{'Global CALIBRATED':25s}  WAPE={wape_cal:.4f}  |RBias|={rbias_cal:.4f}  Total={total_cal:.4f}")
print(f"  calib_scale = {calib_scale:.4f}")
print(f"\nCalibration {'HELPS ✅' if total_cal < total_raw else 'HURTS ❌'} "
      f"(Δ = {total_cal - total_raw:+.4f})")
print("\n── Топ-10 сложных маршрутов (по median_ape) ──")
print(agg_route.sort_values("median_ape", ascending=False)
      [["route_id", "mae", "median_ae", "mape", "median_ape", "mean_err", "calib_scale"]]
      .head(10).to_string(index=False))

raw_cv_df.to_csv("nhits_cv_raw.csv", index=False)
agg_route.to_csv("nhits_cv_agg_route.csv", index=False)
agg_fold.to_csv("nhits_cv_agg_fold.csv", index=False)
agg_h.to_csv("nhits_cv_agg_h.csv", index=False)
print("\n✅ nhits_cv_raw.csv")
print("✅ nhits_cv_agg_route.csv")
print("✅ nhits_cv_agg_fold.csv")
print("✅ nhits_cv_agg_h.csv")


# ============================================================
# TEST PREDICTION
# Финальный фит на полных данных, predict() берёт последние
# input_size точек из тренировочного датасета автоматически
# ============================================================
print("\n" + "=" * 65)
print("TEST PREDICTION")
print("=" * 65)

train_nf_full = to_nf_df(train_df, min_len=INPUT_SIZE + 1, context_len=CONTEXT_LEN)
test_route_ids = test_df["route_id"].unique().tolist()
print(f"Routes: {len(test_route_ids)}  Fitting on full train data...")

nf_full = NeuralForecast(models=[make_nhits()], freq=FREQ)
nf_full.fit(train_nf_full)

print("Running inference...")
futr_df      = make_futr_df(test_route_ids, train_df)
test_pred_df = nf_full.predict(futr_df=futr_df)   # next FORECAST_STEPS × 30min на каждый маршрут
test_pred_col = get_pred_col(test_pred_df)
test_preds    = build_preds_dict(test_pred_df, test_pred_col)

predictions_raw = {}
for route_id in tqdm(test_route_ids, desc="Build submission"):
    route_test = test_df[test_df["route_id"] == route_id].sort_values("timestamp")
    preds = test_preds.get(route_id, np.zeros(FORECAST_STEPS))
    for j, (_, row) in enumerate(route_test.iterrows()):
        pred = float(preds[j]) if j < len(preds) else float(preds[-1])
        predictions_raw[row["id"]] = max(0.0, pred)

submission_raw = (
    pd.DataFrame(list(predictions_raw.items()), columns=["id", "y_pred"])
    .sort_values("id").reset_index(drop=True)
)
submission_cal = submission_raw.copy()
submission_cal["y_pred"] = np.clip(submission_cal["y_pred"] * calib_scale, 0, None)

submission_raw.to_csv("submission_nhits_raw.csv", index=False)
submission_cal.to_csv("submission_nhits_calibrated.csv", index=False)
print(f"\n✅ submission_nhits_raw.csv")
print(f"✅ submission_nhits_calibrated.csv")
print(f"\n── Raw ──\n{submission_raw['y_pred'].describe().round(2)}")
print(f"\n── Calibrated (scale={calib_scale:.4f}) ──\n{submission_cal['y_pred'].describe().round(2)}")

### Убрал стремные статусы, оставил лаги (ЛУЧШЕЕ)

In [ ]:
# ============================================================
# N-HiTS — Rolling CV (5 folds × 8 steps)
# + per-route статистика для блендинга (как в Chronos-2)
# ============================================================



import warnings
import numpy as np
import pandas as pd
import torch
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MQLoss
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None


def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias


# ── Config ──
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8        # = h
N_FOLDS        = 5
TOTAL_VAL_PTS  = N_FOLDS * FORECAST_STEPS   # 40
CONTEXT_LEN    = 2048
FREQ           = "30min"

INPUT_SIZE = 336      # 7 суток × 48 точек — ловим недельную сезонность
MAX_STEPS  = 800     # чуть больше, раз быстро
BATCH_SIZE = 64       # можно увеличить если памяти хватает


# ── Device ──
if torch.backends.mps.is_available():
    ACCELERATOR = "mps"
    print("Apple Silicon MPS ✅")
elif torch.cuda.is_available():
    ACCELERATOR = "gpu"
else:
    ACCELERATOR = "cpu"
print(f"Device: {ACCELERATOR}\n")


# ── Data ──
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

status_cols = sorted([c for c in train_df.columns if c in ["status_3", "status_2", "status_5"]])
print(f"Train: {train_df.shape}  Test: {test_df.shape}")
print(f"Status cols: {status_cols}\n")


FUTR_EXOG = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "is_weekend"]

def add_calendar_features(df: pd.DataFrame, ts_col: str = "ds") -> pd.DataFrame:
    ts = pd.to_datetime(df[ts_col])
    df = df.copy()
    df["hour_sin"]   = np.sin(2 * np.pi * ts.dt.hour / 24)
    df["hour_cos"]   = np.cos(2 * np.pi * ts.dt.hour / 24)
    df["dow_sin"]    = np.sin(2 * np.pi * ts.dt.dayofweek / 7)
    df["dow_cos"]    = np.cos(2 * np.pi * ts.dt.dayofweek / 7)
    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(float)
    return df

HIST_EXOG_EXTRA = ["lag_48", "lag_336"]

def add_lag_features(nf_df: pd.DataFrame) -> pd.DataFrame:
    nf_df = nf_df.sort_values(["unique_id", "ds"]).copy()
    nf_df["lag_48"] = (
        nf_df.groupby("unique_id")["y"]
        .transform(lambda s: s.shift(48).bfill().fillna(0))
    )
    nf_df["lag_336"] = (
        nf_df.groupby("unique_id")["y"]
        .transform(lambda s: s.shift(336).bfill().fillna(0))
    )
    return nf_df

# ── NF-формат: unique_id | ds | y [| status_* ...] ──
# Аналог prepare_context_df из Chronos — те же asfreq / interpolate / tail
def to_nf_df(df: pd.DataFrame,
             min_len: int = 32,
             context_len: int = CONTEXT_LEN) -> pd.DataFrame:
    rows = []
    for route_id, grp in df.groupby("route_id"):
        grp = (
            grp.sort_values("timestamp")
            .set_index("timestamp")[[TARGET_COL] + status_cols]
            .asfreq(FREQ)
            .interpolate(method="time")
            .bfill().ffill()
            .tail(context_len)
            .reset_index()
        )
        if len(grp) < min_len:
            continue
        grp.insert(0, "unique_id", route_id)
        grp = grp.rename(columns={"timestamp": "ds", TARGET_COL: "y"})
        rows.append(grp)
    nf_df = pd.concat(rows, ignore_index=True)
    nf_df = add_calendar_features(nf_df)   # futr_exog
    nf_df = add_lag_features(nf_df)         # hist_exog
    return nf_df


# ── Фабрика модели (один конфиг для CV и финального фита) ──
HIST_EXOG = (status_cols + HIST_EXOG_EXTRA) or None

def make_nhits(max_steps: int = MAX_STEPS) -> NHITS:
    return NHITS(
        h                   = FORECAST_STEPS,
        input_size          = INPUT_SIZE,          # 336 = 1 неделя
        loss                = MQLoss(level=[80]),
        # Стеки: день / 4ч / 30мин — каждый на своей частоте
        # Stack 1: MaxPool(48) → 7 дневных точек  (недельный тренд)
        # Stack 2: MaxPool(8)  → 42 точки         (4-часовые блоки)
        # Stack 3: MaxPool(1)  → 336 точек         (полное разрешение)
        n_freq_downsample   = [48, 8, 1],
        mlp_units           = [[512, 512]] * 3,
        n_blocks            = [4, 4, 4],
        max_steps           = max_steps,
        batch_size          = BATCH_SIZE,
        accelerator         = ACCELERATOR,
        hist_exog_list      = HIST_EXOG,
        futr_exog_list      = FUTR_EXOG,
        scaler_type         = "robust",            # median/IQR — пики не "приплющиваются"
        enable_progress_bar = True,
    )

def make_futr_df(route_ids: list, train_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for route_id in route_ids:
        last_ts = train_df[train_df["route_id"] == route_id]["timestamp"].max()
        future_ts = pd.date_range(
            start   = last_ts + pd.Timedelta("30min"),
            periods = FORECAST_STEPS,
            freq    = FREQ,
        )
        for ts in future_ts:
            rows.append({"unique_id": route_id, "ds": ts})
    return add_calendar_features(pd.DataFrame(rows))

# ── Выбор колонки точечного прогноза (аналог get_pred_col из Chronos) ──
def get_pred_col(pred_df: pd.DataFrame) -> str:
    for candidate in ["NHITS-median", "NHITS-q-0.50", "NHITS"]:
        if candidate in pred_df.columns:
            return candidate
    skip = {"unique_id", "ds", "y", "cutoff"}
    num_cols = [c for c in pred_df.columns
                if c not in skip and pd.api.types.is_numeric_dtype(pred_df[c])]
    median_cols = [c for c in num_cols if "median" in c or "0.5" in c or "50" in c]
    return (median_cols or num_cols)[0]


# ── Словарь предсказаний: route_id → array[FORECAST_STEPS] ──
def build_preds_dict(pred_df: pd.DataFrame, pred_col: str) -> dict:
    result = {}
    for route_id, grp in pred_df.groupby("unique_id"):
        result[route_id] = np.clip(grp.sort_values("ds")[pred_col].values, 0, None)
    return result


# ============================================================
# ROLLING CROSS-VALIDATION
# neuralforecast.cross_validation(n_windows=5, step_size=8, refit=True)
# window 1: train=[:-40]  val=[-40:-32]
# window 2: train=[:-32]  val=[-32:-24]
# window 3: train=[:-24]  val=[-24:-16]
# window 4: train=[:-16]  val=[-16: -8]
# window 5: train=[: -8]  val=[ -8:end]
# refit=True — переобучение на каждом фолде (= поведение Chronos)
# ============================================================
print("=" * 65)
print(f"ROLLING CV  ({N_FOLDS} folds × {FORECAST_STEPS} steps = {TOTAL_VAL_PTS} points)")
print("=" * 65)

train_nf = to_nf_df(
    train_df,
    min_len     = TOTAL_VAL_PTS + INPUT_SIZE + 1,
    context_len = CONTEXT_LEN,
)
print(f"NF train: {train_nf.shape}  routes: {train_nf['unique_id'].nunique()}\n")

nf_cv = NeuralForecast(models=[make_nhits()], freq=FREQ)

# refit=True: модель переобучается для каждого фолда (как Chronos per-fold inference)
# refit=False: обучается один раз, быстрее, но менее честно
cv_df = nf_cv.cross_validation(
    df        = train_nf,
    n_windows = N_FOLDS,
    step_size = FORECAST_STEPS,
    refit     = True,
)

print(f"\nCV columns: {list(cv_df.columns)}")
pred_col = get_pred_col(cv_df)
print(f"Point forecast column: '{pred_col}'\n")

# ── cutoff → fold (1 = самый старый, N_FOLDS = самый свежий) ──
cutoffs_sorted   = sorted(cv_df["cutoff"].unique())
cutoff_to_fold   = {c: i + 1 for i, c in enumerate(cutoffs_sorted)}
cv_df["fold"]    = cv_df["cutoff"].map(cutoff_to_fold)

# ── шаг h внутри фолда (1..FORECAST_STEPS) ──
cv_df = cv_df.sort_values(["fold", "unique_id", "ds"]).reset_index(drop=True)
cv_df["h"] = cv_df.groupby(["fold", "unique_id"]).cumcount() + 1

# ── raw_rows (идентична структура с Chronos) ──
raw_rows = []
for _, row in cv_df.iterrows():
    y_true = float(row["y"])
    y_pred = float(np.clip(row[pred_col], 0, None))
    ae     = abs(y_pred - y_true)
    ape    = ae / (y_true + 1e-9)
    err    = y_pred - y_true
    raw_rows.append({
        "fold":      int(row["fold"]),
        "route_id":  row["unique_id"],
        "timestamp": row["ds"],
        "h":         int(row["h"]),
        "y_true":    y_true,
        "y_pred":    y_pred,
        "ae":        ae,
        "ape":       ape,
        "err":       err,
    })

# ── per-fold console output (как у Chronos) ──
for fold in range(1, N_FOLDS + 1):
    fdf        = cv_df[cv_df["fold"] == fold]
    n_routes   = fdf["unique_id"].nunique()
    val_end    = TOTAL_VAL_PTS - (fold - 1) * FORECAST_STEPS
    val_start  = val_end - FORECAST_STEPS
    yt = fdf["y"].values
    yp = np.clip(fdf[pred_col].values, 0, None)
    tot, wape, rb = wape_rbias(yt, yp)
    print(f"\nFold {fold}/{N_FOLDS}  "
          f"(val: -{val_end}..{'-'+str(val_start) if val_start else 'end'})  "
          f"routes={n_routes}")
    print(f"  RAW   WAPE={wape:.4f}  |RBias|={rb:.4f}  Total={tot:.4f}")


# ============================================================
# АГРЕГАТЫ (идентично Chronos для блендинга)
# ============================================================
raw_cv_df = pd.DataFrame(raw_rows)

# ── По фолду ──
agg_fold = raw_cv_df.groupby("fold").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
    "n":     len(df),
})).reset_index()

# ── По шагу h ──
agg_h = raw_cv_df.groupby("h").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
})).reset_index()

# ── По маршруту ──
agg_route = raw_cv_df.groupby("route_id").agg(
    n_points   = ("y_true",  "count"),
    mae        = ("ae",      "mean"),
    median_ae  = ("ae",      "median"),
    std_ae     = ("ae",      "std"),
    mape       = ("ape",     "mean"),
    median_ape = ("ape",     "median"),
    mean_err   = ("err",     "mean"),
    median_err = ("err",     "median"),
    mean_true  = ("y_true",  "mean"),
    mean_pred  = ("y_pred",  "mean"),
).reset_index()
agg_route["calib_scale"] = (
    raw_cv_df.groupby("route_id")["y_true"].sum().values /
    (raw_cv_df.groupby("route_id")["y_pred"].sum().values + 1e-9)
)
agg_route["wape"] = (
    raw_cv_df.groupby("route_id")["ae"].sum().values /
    (raw_cv_df.groupby("route_id")["y_true"].sum().values + 1e-9)
)

# ── Глобальные метрики ──
y_true_all = raw_cv_df["y_true"].values
y_pred_all = raw_cv_df["y_pred"].values
total_raw, wape_raw, rbias_raw = wape_rbias(y_true_all, y_pred_all)

calib_scale = float(y_true_all.sum() / (y_pred_all.sum() + 1e-9))
y_pred_cal  = np.clip(y_pred_all * calib_scale, 0, None)
total_cal, wape_cal, rbias_cal = wape_rbias(y_true_all, y_pred_cal)

print("\n" + "=" * 65)
print("SUMMARY")
print("=" * 65)
print("\n── По фолдам ──")
print(agg_fold.to_string(index=False))
print("\n── По шагам h ──")
print(agg_h.to_string(index=False))
print(f"\n{'Global RAW':25s}  WAPE={wape_raw:.4f}  |RBias|={rbias_raw:.4f}  Total={total_raw:.4f}")
print(f"{'Global CALIBRATED':25s}  WAPE={wape_cal:.4f}  |RBias|={rbias_cal:.4f}  Total={total_cal:.4f}")
print(f"  calib_scale = {calib_scale:.4f}")
print(f"\nCalibration {'HELPS ✅' if total_cal < total_raw else 'HURTS ❌'} "
      f"(Δ = {total_cal - total_raw:+.4f})")
print("\n── Топ-10 сложных маршрутов (по median_ape) ──")
print(agg_route.sort_values("median_ape", ascending=False)
      [["route_id", "mae", "median_ae", "mape", "median_ape", "mean_err", "calib_scale"]]
      .head(10).to_string(index=False))

raw_cv_df.to_csv("nhits_cv_raw.csv", index=False)
agg_route.to_csv("nhits_cv_agg_route.csv", index=False)
agg_fold.to_csv("nhits_cv_agg_fold.csv", index=False)
agg_h.to_csv("nhits_cv_agg_h.csv", index=False)
print("\n✅ nhits_cv_raw.csv")
print("✅ nhits_cv_agg_route.csv")
print("✅ nhits_cv_agg_fold.csv")
print("✅ nhits_cv_agg_h.csv")


# ============================================================
# TEST PREDICTION
# Финальный фит на полных данных, predict() берёт последние
# input_size точек из тренировочного датасета автоматически
# ============================================================
print("\n" + "=" * 65)
print("TEST PREDICTION")
print("=" * 65)

train_nf_full = to_nf_df(train_df, min_len=INPUT_SIZE + 1, context_len=CONTEXT_LEN)
test_route_ids = test_df["route_id"].unique().tolist()
print(f"Routes: {len(test_route_ids)}  Fitting on full train data...")

nf_full = NeuralForecast(models=[make_nhits()], freq=FREQ)
nf_full.fit(train_nf_full)

print("Running inference...")
futr_df      = make_futr_df(test_route_ids, train_df)
test_pred_df = nf_full.predict(futr_df=futr_df)   # next FORECAST_STEPS × 30min на каждый маршрут
test_pred_col = get_pred_col(test_pred_df)
test_preds    = build_preds_dict(test_pred_df, test_pred_col)

predictions_raw = {}
for route_id in tqdm(test_route_ids, desc="Build submission"):
    route_test = test_df[test_df["route_id"] == route_id].sort_values("timestamp")
    preds = test_preds.get(route_id, np.zeros(FORECAST_STEPS))
    for j, (_, row) in enumerate(route_test.iterrows()):
        pred = float(preds[j]) if j < len(preds) else float(preds[-1])
        predictions_raw[row["id"]] = max(0.0, pred)

submission_raw = (
    pd.DataFrame(list(predictions_raw.items()), columns=["id", "y_pred"])
    .sort_values("id").reset_index(drop=True)
)
submission_cal = submission_raw.copy()
submission_cal["y_pred"] = np.clip(submission_cal["y_pred"] * calib_scale, 0, None)

submission_raw.to_csv("submission_nhits_raw.csv", index=False)
submission_cal.to_csv("submission_nhits_calibrated.csv", index=False)
print(f"\n✅ submission_nhits_raw.csv")
print(f"✅ submission_nhits_calibrated.csv")
print(f"\n── Raw ──\n{submission_raw['y_pred'].describe().round(2)}")
print(f"\n── Calibrated (scale={calib_scale:.4f}) ──\n{submission_cal['y_pred'].describe().round(2)}")

In [ ]:
# ============================================================
# N-HiTS — Rolling CV (5 folds × 8 steps)
# + per-route статистика для блендинга (как в Chronos-2)
# + per-route best_lag для status_* (2 месяца до первого фолда)
# ============================================================

import warnings
import numpy as np
import pandas as pd
import torch
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MQLoss
from statsmodels.tsa.stattools import ccf as ts_ccf
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None


def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias


# ── Config ──
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8
N_FOLDS        = 5
TOTAL_VAL_PTS  = N_FOLDS * FORECAST_STEPS   # 40
CONTEXT_LEN    = 2048
FREQ           = "30min"

INPUT_SIZE = 336
MAX_STEPS  = 800
BATCH_SIZE = 64

# ── Best-lag config ──
LAG_WINDOW_DAYS = 60                    # сколько дней истории для расчёта CCF
LAG_WINDOW      = LAG_WINDOW_DAYS * 48  # в точках 30мин = 2880
MAX_CCF_LAG     = 48                    # максимальный искомый лаг (24ч)
MIN_CORR        = 0.05                  # минимальная CCF-пиковая корреляция
DEFAULT_LAG     = 24                    # дефолт если CCF слабый / мало данных


# ── Device ──
if torch.backends.mps.is_available():
    ACCELERATOR = "mps"
    print("Apple Silicon MPS ✅")
elif torch.cuda.is_available():
    ACCELERATOR = "gpu"
else:
    ACCELERATOR = "cpu"
print(f"Device: {ACCELERATOR}\n")


# ── Data ──
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

status_cols = sorted([c for c in train_df.columns if c in ["status_3", "status_2", "status_5"]])
print(f"Train: {train_df.shape}  Test: {test_df.shape}")
print(f"Status cols: {status_cols}\n")


FUTR_EXOG      = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "is_weekend"]
HIST_EXOG_BASE = status_cols + ["lag_48", "lag_336"]   # без blag — добавятся позже


def add_calendar_features(df: pd.DataFrame, ts_col: str = "ds") -> pd.DataFrame:
    ts = pd.to_datetime(df[ts_col])
    df = df.copy()
    df["hour_sin"]   = np.sin(2 * np.pi * ts.dt.hour / 24)
    df["hour_cos"]   = np.cos(2 * np.pi * ts.dt.hour / 24)
    df["dow_sin"]    = np.sin(2 * np.pi * ts.dt.dayofweek / 7)
    df["dow_cos"]    = np.cos(2 * np.pi * ts.dt.dayofweek / 7)
    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(float)
    return df


def add_lag_features(nf_df: pd.DataFrame) -> pd.DataFrame:
    nf_df = nf_df.sort_values(["unique_id", "ds"]).copy()
    nf_df["lag_48"] = (
        nf_df.groupby("unique_id")["y"]
        .transform(lambda s: s.shift(48).bfill().fillna(0))
    )
    nf_df["lag_336"] = (
        nf_df.groupby("unique_id")["y"]
        .transform(lambda s: s.shift(336).bfill().fillna(0))
    )
    return nf_df


def to_nf_df(df: pd.DataFrame,
             min_len: int = 32,
             context_len: int = CONTEXT_LEN) -> pd.DataFrame:
    rows = []
    for route_id, grp in df.groupby("route_id"):
        grp = (
            grp.sort_values("timestamp")
            .set_index("timestamp")[[TARGET_COL] + status_cols]
            .asfreq(FREQ)
            .interpolate(method="time")
            .bfill().ffill()
            .tail(context_len)
            .reset_index()
        )
        if len(grp) < min_len:
            continue
        grp.insert(0, "unique_id", route_id)
        grp = grp.rename(columns={"timestamp": "ds", TARGET_COL: "y"})
        rows.append(grp)
    nf_df = pd.concat(rows, ignore_index=True)
    nf_df = add_calendar_features(nf_df)
    nf_df = add_lag_features(nf_df)
    return nf_df


# ============================================================
# BEST-LAG: CCF-based per-route feature engineering
# val_offset_pts:
#   CV   → TOTAL_VAL_PTS (40) — окно до начала первого фолда
#   Test → 0               — окно до конца трейна
# ============================================================

def compute_best_lags(nf_df: pd.DataFrame,
                      cols: list,
                      val_offset_pts: int = 0,
                      lag_window: int = LAG_WINDOW,
                      max_lag: int = MAX_CCF_LAG,
                      min_corr: float = MIN_CORR) -> dict:
    """
    Для каждого маршрута и каждой колонки cols считает CCF(status, y)
    на окне [end - lag_window : end], где end = max_ds - val_offset_pts.
    Возвращает {route_id: {col: best_lag_int или None}}.
    """
    result = {}
    for uid, grp in nf_df.groupby("unique_id"):
        grp = grp.sort_values("ds")

        end_ts   = grp["ds"].max() - pd.Timedelta(minutes=30 * val_offset_pts)
        start_ts = end_ts - pd.Timedelta(minutes=30 * lag_window)
        window   = grp[(grp["ds"] > start_ts) & (grp["ds"] <= end_ts)]

        # Если окно слишком маленькое — берём всё что есть до end_ts
        if len(window) < max_lag * 2:
            window = grp[grp["ds"] <= end_ts]

        route_lags = {}
        y = window["y"].values

        for col in cols:
            if col not in window.columns or len(window) < max_lag * 2:
                route_lags[col] = None
                continue
            x = window[col].fillna(0).values
            try:
                cc = ts_ccf(x, y, nlags=max_lag, adjusted=False)
                # cc[0]=лаг0, cc[k]=лаг k → ищем max среди лагов 1..max_lag
                best_lag = int(np.argmax(np.abs(cc[1:])) + 1)
                peak     = float(np.abs(cc[best_lag]))
                route_lags[col] = best_lag if peak >= min_corr else None
            except Exception:
                route_lags[col] = None

        result[uid] = route_lags
    return result


def add_best_lag_features(nf_df: pd.DataFrame,
                           best_lags: dict,
                           cols: list,
                           default_lag: int = DEFAULT_LAG) -> tuple[pd.DataFrame, list]:
    """
    Добавляет колонки {col}_blag с route-специфичным сдвигом.
    Возвращает (обновлённый nf_df, список новых колонок).
    """
    parts = []
    for uid, grp in nf_df.groupby("unique_id", sort=False):
        grp = grp.sort_values("ds").copy()
        for col in cols:
            feat = f"{col}_blag"
            lag  = best_lags.get(uid, {}).get(col) or default_lag
            grp[feat] = grp[col].shift(lag).bfill().fillna(0)
        parts.append(grp)
    result = pd.concat(parts, ignore_index=True)
    added  = [f"{col}_blag" for col in cols]
    return result, added


def print_lag_stats(best_lags: dict, cols: list, label: str = ""):
    """Краткая статистика по найденным лагам."""
    print(f"\n── Best-lag stats [{label}] ──")
    for col in cols:
        lags = [v[col] for v in best_lags.values() if v.get(col) is not None]
        none_cnt = sum(1 for v in best_lags.values() if v.get(col) is None)
        if lags:
            print(f"  {col}: median={np.median(lags):.0f}  "
                  f"min={min(lags)}  max={max(lags)}  "
                  f"nulls(corr<{MIN_CORR})={none_cnt}")
        else:
            print(f"  {col}: все маршруты имеют корреляцию < {MIN_CORR}")


# ── Фабрика модели — принимает итоговый hist_exog ──
def make_nhits(max_steps: int = MAX_STEPS,
               hist_exog: list = None) -> NHITS:
    return NHITS(
        h                   = FORECAST_STEPS,
        input_size          = INPUT_SIZE,
        loss                = MQLoss(level=[80]),
        n_freq_downsample   = [48, 8, 1],
        mlp_units           = [[512, 512]] * 3,
        n_blocks            = [4, 4, 4],
        max_steps           = max_steps,
        batch_size          = BATCH_SIZE,
        accelerator         = ACCELERATOR,
        hist_exog_list      = hist_exog or HIST_EXOG_BASE,
        futr_exog_list      = FUTR_EXOG,
        scaler_type         = "robust",
        enable_progress_bar = True,
    )


def make_futr_df(route_ids: list, train_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for route_id in route_ids:
        last_ts = train_df[train_df["route_id"] == route_id]["timestamp"].max()
        future_ts = pd.date_range(
            start   = last_ts + pd.Timedelta("30min"),
            periods = FORECAST_STEPS,
            freq    = FREQ,
        )
        for ts in future_ts:
            rows.append({"unique_id": route_id, "ds": ts})
    return add_calendar_features(pd.DataFrame(rows))


def get_pred_col(pred_df: pd.DataFrame) -> str:
    for candidate in ["NHITS-median", "NHITS-q-0.50", "NHITS"]:
        if candidate in pred_df.columns:
            return candidate
    skip = {"unique_id", "ds", "y", "cutoff"}
    num_cols = [c for c in pred_df.columns
                if c not in skip and pd.api.types.is_numeric_dtype(pred_df[c])]
    median_cols = [c for c in num_cols if "median" in c or "0.5" in c or "50" in c]
    return (median_cols or num_cols)[0]


def build_preds_dict(pred_df: pd.DataFrame, pred_col: str) -> dict:
    result = {}
    for route_id, grp in pred_df.groupby("unique_id"):
        result[route_id] = np.clip(grp.sort_values("ds")[pred_col].values, 0, None)
    return result


# ============================================================
# ROLLING CROSS-VALIDATION
# ============================================================
print("=" * 65)
print(f"ROLLING CV  ({N_FOLDS} folds × {FORECAST_STEPS} steps = {TOTAL_VAL_PTS} points)")
print("=" * 65)

train_nf = to_nf_df(
    train_df,
    min_len     = TOTAL_VAL_PTS + INPUT_SIZE + 1,
    context_len = CONTEXT_LEN,
)
print(f"NF train: {train_nf.shape}  routes: {train_nf['unique_id'].nunique()}\n")

# ── Считаем лаги на 2 месяцах до начала первого фолда ──
print("Computing per-route best lags for CV (2 months before fold 1)...")
best_lags_cv = compute_best_lags(
    train_nf,
    cols           = status_cols,
    val_offset_pts = TOTAL_VAL_PTS,   # отступаем на 40 точек от конца
    lag_window     = LAG_WINDOW,
)
print_lag_stats(best_lags_cv, status_cols, label="CV")

train_nf, blag_cols = add_best_lag_features(train_nf, best_lags_cv, status_cols)
HIST_EXOG_CV = HIST_EXOG_BASE + blag_cols
print(f"hist_exog for CV: {HIST_EXOG_CV}\n")

nf_cv = NeuralForecast(models=[make_nhits(hist_exog=HIST_EXOG_CV)], freq=FREQ)

cv_df = nf_cv.cross_validation(
    df        = train_nf,
    n_windows = N_FOLDS,
    step_size = FORECAST_STEPS,
    refit     = True,
)

print(f"\nCV columns: {list(cv_df.columns)}")
pred_col = get_pred_col(cv_df)
print(f"Point forecast column: '{pred_col}'\n")

cutoffs_sorted = sorted(cv_df["cutoff"].unique())
cutoff_to_fold = {c: i + 1 for i, c in enumerate(cutoffs_sorted)}
cv_df["fold"]  = cv_df["cutoff"].map(cutoff_to_fold)

cv_df = cv_df.sort_values(["fold", "unique_id", "ds"]).reset_index(drop=True)
cv_df["h"] = cv_df.groupby(["fold", "unique_id"]).cumcount() + 1

raw_rows = []
for _, row in cv_df.iterrows():
    y_true = float(row["y"])
    y_pred = float(np.clip(row[pred_col], 0, None))
    ae     = abs(y_pred - y_true)
    ape    = ae / (y_true + 1e-9)
    err    = y_pred - y_true
    raw_rows.append({
        "fold":      int(row["fold"]),
        "route_id":  row["unique_id"],
        "timestamp": row["ds"],
        "h":         int(row["h"]),
        "y_true":    y_true,
        "y_pred":    y_pred,
        "ae":        ae,
        "ape":       ape,
        "err":       err,
    })

for fold in range(1, N_FOLDS + 1):
    fdf      = cv_df[cv_df["fold"] == fold]
    n_routes = fdf["unique_id"].nunique()
    val_end  = TOTAL_VAL_PTS - (fold - 1) * FORECAST_STEPS
    val_start = val_end - FORECAST_STEPS
    yt = fdf["y"].values
    yp = np.clip(fdf[pred_col].values, 0, None)
    tot, wape, rb = wape_rbias(yt, yp)
    print(f"\nFold {fold}/{N_FOLDS}  "
          f"(val: -{val_end}..{'-'+str(val_start) if val_start else 'end'})  "
          f"routes={n_routes}")
    print(f"  RAW   WAPE={wape:.4f}  |RBias|={rb:.4f}  Total={tot:.4f}")


# ============================================================
# АГРЕГАТЫ
# ============================================================
raw_cv_df = pd.DataFrame(raw_rows)

agg_fold = raw_cv_df.groupby("fold").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
    "n":     len(df),
})).reset_index()

agg_h = raw_cv_df.groupby("h").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
})).reset_index()

agg_route = raw_cv_df.groupby("route_id").agg(
    n_points   = ("y_true",  "count"),
    mae        = ("ae",      "mean"),
    median_ae  = ("ae",      "median"),
    std_ae     = ("ae",      "std"),
    mape       = ("ape",     "mean"),
    median_ape = ("ape",     "median"),
    mean_err   = ("err",     "mean"),
    median_err = ("err",     "median"),
    mean_true  = ("y_true",  "mean"),
    mean_pred  = ("y_pred",  "mean"),
).reset_index()
agg_route["calib_scale"] = (
    raw_cv_df.groupby("route_id")["y_true"].sum().values /
    (raw_cv_df.groupby("route_id")["y_pred"].sum().values + 1e-9)
)
agg_route["wape"] = (
    raw_cv_df.groupby("route_id")["ae"].sum().values /
    (raw_cv_df.groupby("route_id")["y_true"].sum().values + 1e-9)
)

y_true_all = raw_cv_df["y_true"].values
y_pred_all = raw_cv_df["y_pred"].values
total_raw, wape_raw, rbias_raw = wape_rbias(y_true_all, y_pred_all)

calib_scale = float(y_true_all.sum() / (y_pred_all.sum() + 1e-9))
y_pred_cal  = np.clip(y_pred_all * calib_scale, 0, None)
total_cal, wape_cal, rbias_cal = wape_rbias(y_true_all, y_pred_cal)

print("\n" + "=" * 65)
print("SUMMARY")
print("=" * 65)
print("\n── По фолдам ──")
print(agg_fold.to_string(index=False))
print("\n── По шагам h ──")
print(agg_h.to_string(index=False))
print(f"\n{'Global RAW':25s}  WAPE={wape_raw:.4f}  |RBias|={rbias_raw:.4f}  Total={total_raw:.4f}")
print(f"{'Global CALIBRATED':25s}  WAPE={wape_cal:.4f}  |RBias|={rbias_cal:.4f}  Total={total_cal:.4f}")
print(f"  calib_scale = {calib_scale:.4f}")
print(f"\nCalibration {'HELPS ✅' if total_cal < total_raw else 'HURTS ❌'} "
      f"(Δ = {total_cal - total_raw:+.4f})")
print("\n── Топ-10 сложных маршрутов (по median_ape) ──")
print(agg_route.sort_values("median_ape", ascending=False)
      [["route_id", "mae", "median_ae", "mape", "median_ape", "mean_err", "calib_scale"]]
      .head(10).to_string(index=False))

raw_cv_df.to_csv("nhits_cv_raw.csv", index=False)
agg_route.to_csv("nhits_cv_agg_route.csv", index=False)
agg_fold.to_csv("nhits_cv_agg_fold.csv", index=False)
agg_h.to_csv("nhits_cv_agg_h.csv", index=False)
print("\n✅ nhits_cv_raw.csv / agg_route / agg_fold / agg_h")


# ============================================================
# TEST PREDICTION
# ============================================================
print("\n" + "=" * 65)
print("TEST PREDICTION")
print("=" * 65)

train_nf_full = to_nf_df(train_df, min_len=INPUT_SIZE + 1, context_len=CONTEXT_LEN)

# ── Считаем лаги на 2 месяцах в конце трейна ──
print("Computing per-route best lags for test (last 2 months of train)...")
best_lags_test = compute_best_lags(
    train_nf_full,
    cols           = status_cols,
    val_offset_pts = 0,             # берём до самого конца
    lag_window     = LAG_WINDOW,
)
print_lag_stats(best_lags_test, status_cols, label="Test")

train_nf_full, blag_cols_test = add_best_lag_features(
    train_nf_full, best_lags_test, status_cols
)
HIST_EXOG_TEST = HIST_EXOG_BASE + blag_cols_test
print(f"hist_exog for test: {HIST_EXOG_TEST}\n")

test_route_ids = test_df["route_id"].unique().tolist()
print(f"Routes: {len(test_route_ids)}  Fitting on full train data...")

nf_full = NeuralForecast(models=[make_nhits(hist_exog=HIST_EXOG_TEST)], freq=FREQ)
nf_full.fit(train_nf_full)

print("Running inference...")
futr_df       = make_futr_df(test_route_ids, train_df)
test_pred_df  = nf_full.predict(futr_df=futr_df)
test_pred_col = get_pred_col(test_pred_df)
test_preds    = build_preds_dict(test_pred_df, test_pred_col)

predictions_raw = {}
for route_id in tqdm(test_route_ids, desc="Build submission"):
    route_test = test_df[test_df["route_id"] == route_id].sort_values("timestamp")
    preds = test_preds.get(route_id, np.zeros(FORECAST_STEPS))
    for j, (_, row) in enumerate(route_test.iterrows()):
        pred = float(preds[j]) if j < len(preds) else float(preds[-1])
        predictions_raw[row["id"]] = max(0.0, pred)

submission_raw = (
    pd.DataFrame(list(predictions_raw.items()), columns=["id", "y_pred"])
    .sort_values("id").reset_index(drop=True)
)
submission_cal = submission_raw.copy()
submission_cal["y_pred"] = np.clip(submission_cal["y_pred"] * calib_scale, 0, None)

submission_raw.to_csv("submission_nhits_raw.csv", index=False)
submission_cal.to_csv("submission_nhits_calibrated.csv", index=False)
print(f"\n✅ submission_nhits_raw.csv")
print(f"✅ submission_nhits_calibrated.csv")
print(f"\n── Raw ──\n{submission_raw['y_pred'].describe().round(2)}")
print(f"\n── Calibrated (scale={calib_scale:.4f}) ──\n{submission_cal['y_pred'].describe().round(2)}")

In [ ]:
# ============================================================
# N-HiTS — Rolling CV (5 folds × 8 steps)
# + per-route статистика для блендинга (как в Chronos-2)
# ============================================================



import warnings
import numpy as np
import pandas as pd
import torch
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MQLoss
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None


def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias


# ── Config ──
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8        # = h
N_FOLDS        = 5
TOTAL_VAL_PTS  = N_FOLDS * FORECAST_STEPS   # 40
CONTEXT_LEN    = 2048
FREQ           = "30min"

INPUT_SIZE = 336      # 7 суток × 48 точек — ловим недельную сезонность
MAX_STEPS  = 800     # чуть больше, раз быстро
BATCH_SIZE = 64       # можно увеличить если памяти хватает


# ── Device ──
if torch.backends.mps.is_available():
    ACCELERATOR = "mps"
    print("Apple Silicon MPS ✅")
elif torch.cuda.is_available():
    ACCELERATOR = "gpu"
else:
    ACCELERATOR = "cpu"
print(f"Device: {ACCELERATOR}\n")


# ── Data ──
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

status_cols = sorted([c for c in train_df.columns if c in ["status_3", "status_2", "status_5"]])
print(f"Train: {train_df.shape}  Test: {test_df.shape}")
print(f"Status cols: {status_cols}\n")


FUTR_EXOG = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "is_weekend"]

def add_calendar_features(df: pd.DataFrame, ts_col: str = "ds") -> pd.DataFrame:
    ts = pd.to_datetime(df[ts_col])
    df = df.copy()
    df["hour_sin"]   = np.sin(2 * np.pi * ts.dt.hour / 24)
    df["hour_cos"]   = np.cos(2 * np.pi * ts.dt.hour / 24)
    df["dow_sin"]    = np.sin(2 * np.pi * ts.dt.dayofweek / 7)
    df["dow_cos"]    = np.cos(2 * np.pi * ts.dt.dayofweek / 7)
    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(float)
    return df

HIST_EXOG_EXTRA = ["lag_48", "lag_336", "in_transit"]

def add_lag_features(nf_df: pd.DataFrame) -> pd.DataFrame:
    nf_df = nf_df.sort_values(["unique_id", "ds"]).copy()
    nf_df["lag_48"] = (
        nf_df.groupby("unique_id")["y"]
        .transform(lambda s: s.shift(48).bfill().fillna(0))
    )
    nf_df["lag_336"] = (
        nf_df.groupby("unique_id")["y"]
        .transform(lambda s: s.shift(336).bfill().fillna(0))
    )
    return nf_df

def add_in_transit_feature(nf_df: pd.DataFrame,
                           status_col: str = "status_5",
                           window: int = 48) -> pd.DataFrame:
    """
    in_transit ≈ кумулятивный статус_5 минус кумулятивный target_1h за последние `window` точек.
    По сути backlog в пути в единицах status_5 (после скейлинга важен знак/динамика).
    """
    nf_df = nf_df.sort_values(["unique_id", "ds"]).copy()

    def _calc_group(g: pd.DataFrame) -> pd.Series:
        # rolling sum по status_5 и по y за окно
        s_status = g[status_col].fillna(0).rolling(window).sum()
        s_y      = g["y"].fillna(0).rolling(window).sum()
        return (s_status - s_y).fillna(0)

    nf_df["in_transit"] = (
        nf_df.groupby("unique_id", group_keys=False)
        .apply(_calc_group)
        .values
    )
    return nf_df

# ── NF-формат: unique_id | ds | y [| status_* ...] ──
# Аналог prepare_context_df из Chronos — те же asfreq / interpolate / tail
def to_nf_df(df: pd.DataFrame,
             min_len: int = 32,
             context_len: int = CONTEXT_LEN) -> pd.DataFrame:
    rows = []
    for route_id, grp in df.groupby("route_id"):
        grp = (
            grp.sort_values("timestamp")
            .set_index("timestamp")[[TARGET_COL] + status_cols]
            .asfreq(FREQ)
            .interpolate(method="time")
            .bfill().ffill()
            .tail(context_len)
            .reset_index()
        )
        if len(grp) < min_len:
            continue
        grp.insert(0, "unique_id", route_id)
        grp = grp.rename(columns={"timestamp": "ds", TARGET_COL: "y"})
        rows.append(grp)
    nf_df = pd.concat(rows, ignore_index=True)
    nf_df = add_calendar_features(nf_df)   # futr_exog
    nf_df = add_lag_features(nf_df)         # hist_exog
    nf_df = add_in_transit_feature(nf_df, status_col="status_5", window=48)
    return nf_df


# ── Фабрика модели (один конфиг для CV и финального фита) ──
HIST_EXOG = (status_cols + HIST_EXOG_EXTRA) or None

def make_nhits(max_steps: int = MAX_STEPS) -> NHITS:
    return NHITS(
        h                   = FORECAST_STEPS,
        input_size          = INPUT_SIZE,          # 336 = 1 неделя
        loss                = MQLoss(level=[80]),
        # Стеки: день / 4ч / 30мин — каждый на своей частоте
        # Stack 1: MaxPool(48) → 7 дневных точек  (недельный тренд)
        # Stack 2: MaxPool(8)  → 42 точки         (4-часовые блоки)
        # Stack 3: MaxPool(1)  → 336 точек         (полное разрешение)
        n_freq_downsample   = [48, 8, 1],
        mlp_units           = [[512, 512]] * 3,
        n_blocks            = [4, 4, 4],
        max_steps           = max_steps,
        batch_size          = BATCH_SIZE,
        accelerator         = ACCELERATOR,
        hist_exog_list      = HIST_EXOG,
        futr_exog_list      = FUTR_EXOG,
        scaler_type         = "robust",            # median/IQR — пики не "приплющиваются"
        enable_progress_bar = True,
    )

def make_futr_df(route_ids: list, train_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for route_id in route_ids:
        last_ts = train_df[train_df["route_id"] == route_id]["timestamp"].max()
        future_ts = pd.date_range(
            start   = last_ts + pd.Timedelta("30min"),
            periods = FORECAST_STEPS,
            freq    = FREQ,
        )
        for ts in future_ts:
            rows.append({"unique_id": route_id, "ds": ts})
    return add_calendar_features(pd.DataFrame(rows))

# ── Выбор колонки точечного прогноза (аналог get_pred_col из Chronos) ──
def get_pred_col(pred_df: pd.DataFrame) -> str:
    for candidate in ["NHITS-median", "NHITS-q-0.50", "NHITS"]:
        if candidate in pred_df.columns:
            return candidate
    skip = {"unique_id", "ds", "y", "cutoff"}
    num_cols = [c for c in pred_df.columns
                if c not in skip and pd.api.types.is_numeric_dtype(pred_df[c])]
    median_cols = [c for c in num_cols if "median" in c or "0.5" in c or "50" in c]
    return (median_cols or num_cols)[0]


# ── Словарь предсказаний: route_id → array[FORECAST_STEPS] ──
def build_preds_dict(pred_df: pd.DataFrame, pred_col: str) -> dict:
    result = {}
    for route_id, grp in pred_df.groupby("unique_id"):
        result[route_id] = np.clip(grp.sort_values("ds")[pred_col].values, 0, None)
    return result


# ============================================================
# ROLLING CROSS-VALIDATION
# neuralforecast.cross_validation(n_windows=5, step_size=8, refit=True)
# window 1: train=[:-40]  val=[-40:-32]
# window 2: train=[:-32]  val=[-32:-24]
# window 3: train=[:-24]  val=[-24:-16]
# window 4: train=[:-16]  val=[-16: -8]
# window 5: train=[: -8]  val=[ -8:end]
# refit=True — переобучение на каждом фолде (= поведение Chronos)
# ============================================================
print("=" * 65)
print(f"ROLLING CV  ({N_FOLDS} folds × {FORECAST_STEPS} steps = {TOTAL_VAL_PTS} points)")
print("=" * 65)

train_nf = to_nf_df(
    train_df,
    min_len     = TOTAL_VAL_PTS + INPUT_SIZE + 1,
    context_len = CONTEXT_LEN,
)
print(f"NF train: {train_nf.shape}  routes: {train_nf['unique_id'].nunique()}\n")

nf_cv = NeuralForecast(models=[make_nhits()], freq=FREQ)

# refit=True: модель переобучается для каждого фолда (как Chronos per-fold inference)
# refit=False: обучается один раз, быстрее, но менее честно
cv_df = nf_cv.cross_validation(
    df        = train_nf,
    n_windows = N_FOLDS,
    step_size = FORECAST_STEPS,
    refit     = True,
)

print(f"\nCV columns: {list(cv_df.columns)}")
pred_col = get_pred_col(cv_df)
print(f"Point forecast column: '{pred_col}'\n")

# ── cutoff → fold (1 = самый старый, N_FOLDS = самый свежий) ──
cutoffs_sorted   = sorted(cv_df["cutoff"].unique())
cutoff_to_fold   = {c: i + 1 for i, c in enumerate(cutoffs_sorted)}
cv_df["fold"]    = cv_df["cutoff"].map(cutoff_to_fold)

# ── шаг h внутри фолда (1..FORECAST_STEPS) ──
cv_df = cv_df.sort_values(["fold", "unique_id", "ds"]).reset_index(drop=True)
cv_df["h"] = cv_df.groupby(["fold", "unique_id"]).cumcount() + 1

# ── raw_rows (идентична структура с Chronos) ──
raw_rows = []
for _, row in cv_df.iterrows():
    y_true = float(row["y"])
    y_pred = float(np.clip(row[pred_col], 0, None))
    ae     = abs(y_pred - y_true)
    ape    = ae / (y_true + 1e-9)
    err    = y_pred - y_true
    raw_rows.append({
        "fold":      int(row["fold"]),
        "route_id":  row["unique_id"],
        "timestamp": row["ds"],
        "h":         int(row["h"]),
        "y_true":    y_true,
        "y_pred":    y_pred,
        "ae":        ae,
        "ape":       ape,
        "err":       err,
    })

# ── per-fold console output (как у Chronos) ──
for fold in range(1, N_FOLDS + 1):
    fdf        = cv_df[cv_df["fold"] == fold]
    n_routes   = fdf["unique_id"].nunique()
    val_end    = TOTAL_VAL_PTS - (fold - 1) * FORECAST_STEPS
    val_start  = val_end - FORECAST_STEPS
    yt = fdf["y"].values
    yp = np.clip(fdf[pred_col].values, 0, None)
    tot, wape, rb = wape_rbias(yt, yp)
    print(f"\nFold {fold}/{N_FOLDS}  "
          f"(val: -{val_end}..{'-'+str(val_start) if val_start else 'end'})  "
          f"routes={n_routes}")
    print(f"  RAW   WAPE={wape:.4f}  |RBias|={rb:.4f}  Total={tot:.4f}")


# ============================================================
# АГРЕГАТЫ (идентично Chronos для блендинга)
# ============================================================
raw_cv_df = pd.DataFrame(raw_rows)

# ── По фолду ──
agg_fold = raw_cv_df.groupby("fold").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
    "n":     len(df),
})).reset_index()

# ── По шагу h ──
agg_h = raw_cv_df.groupby("h").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
})).reset_index()

# ── По маршруту ──
agg_route = raw_cv_df.groupby("route_id").agg(
    n_points   = ("y_true",  "count"),
    mae        = ("ae",      "mean"),
    median_ae  = ("ae",      "median"),
    std_ae     = ("ae",      "std"),
    mape       = ("ape",     "mean"),
    median_ape = ("ape",     "median"),
    mean_err   = ("err",     "mean"),
    median_err = ("err",     "median"),
    mean_true  = ("y_true",  "mean"),
    mean_pred  = ("y_pred",  "mean"),
).reset_index()
agg_route["calib_scale"] = (
    raw_cv_df.groupby("route_id")["y_true"].sum().values /
    (raw_cv_df.groupby("route_id")["y_pred"].sum().values + 1e-9)
)
agg_route["wape"] = (
    raw_cv_df.groupby("route_id")["ae"].sum().values /
    (raw_cv_df.groupby("route_id")["y_true"].sum().values + 1e-9)
)

# ── Глобальные метрики ──
y_true_all = raw_cv_df["y_true"].values
y_pred_all = raw_cv_df["y_pred"].values
total_raw, wape_raw, rbias_raw = wape_rbias(y_true_all, y_pred_all)

calib_scale = float(y_true_all.sum() / (y_pred_all.sum() + 1e-9))
y_pred_cal  = np.clip(y_pred_all * calib_scale, 0, None)
total_cal, wape_cal, rbias_cal = wape_rbias(y_true_all, y_pred_cal)

print("\n" + "=" * 65)
print("SUMMARY")
print("=" * 65)
print("\n── По фолдам ──")
print(agg_fold.to_string(index=False))
print("\n── По шагам h ──")
print(agg_h.to_string(index=False))
print(f"\n{'Global RAW':25s}  WAPE={wape_raw:.4f}  |RBias|={rbias_raw:.4f}  Total={total_raw:.4f}")
print(f"{'Global CALIBRATED':25s}  WAPE={wape_cal:.4f}  |RBias|={rbias_cal:.4f}  Total={total_cal:.4f}")
print(f"  calib_scale = {calib_scale:.4f}")
print(f"\nCalibration {'HELPS ✅' if total_cal < total_raw else 'HURTS ❌'} "
      f"(Δ = {total_cal - total_raw:+.4f})")
print("\n── Топ-10 сложных маршрутов (по median_ape) ──")
print(agg_route.sort_values("median_ape", ascending=False)
      [["route_id", "mae", "median_ae", "mape", "median_ape", "mean_err", "calib_scale"]]
      .head(10).to_string(index=False))

raw_cv_df.to_csv("nhits_cv_raw.csv", index=False)
agg_route.to_csv("nhits_cv_agg_route.csv", index=False)
agg_fold.to_csv("nhits_cv_agg_fold.csv", index=False)
agg_h.to_csv("nhits_cv_agg_h.csv", index=False)
print("\n✅ nhits_cv_raw.csv")
print("✅ nhits_cv_agg_route.csv")
print("✅ nhits_cv_agg_fold.csv")
print("✅ nhits_cv_agg_h.csv")


# ============================================================
# TEST PREDICTION
# Финальный фит на полных данных, predict() берёт последние
# input_size точек из тренировочного датасета автоматически
# ============================================================
print("\n" + "=" * 65)
print("TEST PREDICTION")
print("=" * 65)

train_nf_full = to_nf_df(train_df, min_len=INPUT_SIZE + 1, context_len=CONTEXT_LEN)
test_route_ids = test_df["route_id"].unique().tolist()
print(f"Routes: {len(test_route_ids)}  Fitting on full train data...")

nf_full = NeuralForecast(models=[make_nhits()], freq=FREQ)
nf_full.fit(train_nf_full)

print("Running inference...")
futr_df      = make_futr_df(test_route_ids, train_df)
test_pred_df = nf_full.predict(futr_df=futr_df)   # next FORECAST_STEPS × 30min на каждый маршрут
test_pred_col = get_pred_col(test_pred_df)
test_preds    = build_preds_dict(test_pred_df, test_pred_col)

predictions_raw = {}
for route_id in tqdm(test_route_ids, desc="Build submission"):
    route_test = test_df[test_df["route_id"] == route_id].sort_values("timestamp")
    preds = test_preds.get(route_id, np.zeros(FORECAST_STEPS))
    for j, (_, row) in enumerate(route_test.iterrows()):
        pred = float(preds[j]) if j < len(preds) else float(preds[-1])
        predictions_raw[row["id"]] = max(0.0, pred)

submission_raw = (
    pd.DataFrame(list(predictions_raw.items()), columns=["id", "y_pred"])
    .sort_values("id").reset_index(drop=True)
)
submission_cal = submission_raw.copy()
submission_cal["y_pred"] = np.clip(submission_cal["y_pred"] * calib_scale, 0, None)

submission_raw.to_csv("submission_nhits_raw.csv", index=False)
submission_cal.to_csv("submission_nhits_calibrated.csv", index=False)
print(f"\n✅ submission_nhits_raw.csv")
print(f"✅ submission_nhits_calibrated.csv")
print(f"\n── Raw ──\n{submission_raw['y_pred'].describe().round(2)}")
print(f"\n── Calibrated (scale={calib_scale:.4f}) ──\n{submission_cal['y_pred'].describe().round(2)}")

In [ ]:
# ============================================================
# N-HiTS — эксперименты с гипотезами (по одной за раз)
# Rolling CV (5 folds × 8 steps) + Test prediction
# ============================================================

import warnings
import os
import json
from dataclasses import dataclass, asdict
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MQLoss
import gc

if torch.backends.mps.is_available():
    torch.mps.empty_cache()
elif torch.cuda.is_available():
    torch.cuda.empty_cache()

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

# ── Utility ──

def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias


# ── Base Config ──
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8
N_FOLDS        = 5
TOTAL_VAL_PTS  = N_FOLDS * FORECAST_STEPS   # 40
BASE_CONTEXT_LEN = 2048
FREQ           = "30min"

BASE_INPUT_SIZE = 336      # 7 суток × 48 точек
BASE_MAX_STEPS  = 800
BATCH_SIZE      = 64

EXPERIMENTS_DIR = "nhits_experiments"
os.makedirs(EXPERIMENTS_DIR, exist_ok=True)

# ── Device ──
if torch.backends.mps.is_available():
    ACCELERATOR = "mps"
    print("Apple Silicon MPS ✅")
elif torch.cuda.is_available():
    ACCELERATOR = "gpu"
else:
    ACCELERATOR = "cpu"
print(f"Device: {ACCELERATOR}\n")

# ── Data ──
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

status_cols = sorted([c for c in train_df.columns if c in ["status_3", "status_2", "status_5"]])
print(f"Train: {train_df.shape}  Test: {test_df.shape}")
print(f"Status cols: {status_cols}\n")

FUTR_EXOG = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "is_weekend"]


# ── Calendar features ──
def add_calendar_features(df: pd.DataFrame, ts_col: str = "ds") -> pd.DataFrame:
    ts = pd.to_datetime(df[ts_col])
    df = df.copy()
    df["hour_sin"]   = np.sin(2 * np.pi * ts.dt.hour / 24)
    df["hour_cos"]   = np.cos(2 * np.pi * ts.dt.hour / 24)
    df["dow_sin"]    = np.sin(2 * np.pi * ts.dt.dayofweek / 7)
    df["dow_cos"]    = np.cos(2 * np.pi * ts.dt.dayofweek / 7)
    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(float)
    return df


# ── Base lag features ──
def add_lag_features(nf_df: pd.DataFrame) -> pd.DataFrame:
    nf_df = nf_df.sort_values(["unique_id", "ds"]).copy()
    nf_df["lag_48"] = (
        nf_df.groupby("unique_id")["y"]
        .transform(lambda s: s.shift(48).bfill().fillna(0))
    )
    nf_df["lag_336"] = (
        nf_df.groupby("unique_id")["y"]
        .transform(lambda s: s.shift(336).bfill().fillna(0))
    )
    return nf_df


# ── In-transit proxy ──
def add_in_transit(nf_df: pd.DataFrame,
                   status_col: str = "status_5",
                   window: int = 48) -> pd.DataFrame:
    """
    in_transit ≈ rolling_sum(status_5) - rolling_sum(y) за окно `window`.
    """
    nf_df = nf_df.sort_values(["unique_id", "ds"]).copy()

    def _calc(g: pd.DataFrame) -> pd.Series:
        s_status = g[status_col].fillna(0).rolling(window).sum()
        s_y      = g["y"].fillna(0).rolling(window).sum()
        return (s_status - s_y).fillna(0)

    nf_df["in_transit"] = (
        nf_df.groupby("unique_id", group_keys=False)
        .apply(_calc)
        .values
    )
    return nf_df


# ── Rolling sums по статусам ──
def add_status_rolling(nf_df: pd.DataFrame,
                       windows: tuple = (24, 48)) -> pd.DataFrame:
    nf_df = nf_df.sort_values(["unique_id", "ds"]).copy()
    for col in status_cols:
        for w in windows:
            feat = f"{col}_rsum_{w}"
            nf_df[feat] = (
                nf_df.groupby("unique_id")[col]
                .transform(lambda s: s.shift(1).rolling(w).sum())
                .fillna(0)
            )
    return nf_df


# ── Trend features по y ──
def add_y_trend(nf_df: pd.DataFrame) -> pd.DataFrame:
    nf_df = nf_df.sort_values(["unique_id", "ds"]).copy()

    def _trend(g: pd.DataFrame) -> pd.DataFrame:
        g = g.copy()
        g["y_mean_24"] = g["y"].rolling(24).mean()
        g["y_mean_48"] = g["y"].rolling(48).mean()
        g["y_trend_24_48"] = (g["y_mean_24"] - g["y_mean_48"]).fillna(0)
        g["y_std_24"] = g["y"].rolling(24).std().fillna(0)
        return g

    return (
        nf_df.groupby("unique_id", group_keys=False)
        .apply(_trend)
        .reset_index(drop=True)
    )


# ── NF-format (без фич) ──
def to_nf_df(df: pd.DataFrame,
             min_len: int,
             context_len: int) -> pd.DataFrame:
    rows = []
    for route_id, grp in df.groupby("route_id"):
        grp = (
            grp.sort_values("timestamp")
            .set_index("timestamp")[[TARGET_COL] + status_cols]
            .asfreq(FREQ)
            .interpolate(method="time")
            .bfill().ffill()
            .tail(context_len)
            .reset_index()
        )
        if len(grp) < min_len:
            continue
        grp.insert(0, "unique_id", route_id)
        grp = grp.rename(columns={"timestamp": "ds", TARGET_COL: "y"})
        rows.append(grp)
    nf_df = pd.concat(rows, ignore_index=True)
    return nf_df


# ── Experiment config ──
@dataclass
class ExperimentConfig:
    name: str

    input_size: int = BASE_INPUT_SIZE
    context_len: int = BASE_CONTEXT_LEN
    use_in_transit: bool = False
    in_transit_window: int = 48
    use_status_rolling: bool = False
    status_rolling_windows: tuple = (24, 48)
    use_y_trend: bool = False
    max_steps: int = BASE_MAX_STEPS
    n_freq_downsample: tuple = (48, 8, 1)


BASE_CFG = ExperimentConfig(name="base")

EXPERIMENTS = [
    BASE_CFG,
    ExperimentConfig(name="input_672", input_size=672),
    ExperimentConfig(name="context_1008", context_len=1008),
    ExperimentConfig(name="status_rolling",
                     use_status_rolling=True,
                     status_rolling_windows=(24, 48)),
    ExperimentConfig(name="y_trend",
                     use_y_trend=True),
    ExperimentConfig(name="maxsteps_1200",
                     max_steps=1200),
    ExperimentConfig(name="downsample_96_8_1",
                     n_freq_downsample=(96, 8, 1)),
    ExperimentConfig(name="input_240", input_size=240),   # 5 суток
    ExperimentConfig(name="input_480", input_size=480),   # 10 суток
    ExperimentConfig(name="input_720", input_size=720),   # 15 суток
    ExperimentConfig(name="input_960", input_size=960),   # 20 суток
    ExperimentConfig(name="context_672",  context_len=672),   # 2 недели
    ExperimentConfig(name="context_1344", context_len=1344),  # 4 недели
    ExperimentConfig(name="context_4096", context_len=4096),  # длинный хвост
    ExperimentConfig(name="maxsteps_600",  max_steps=600),
    ExperimentConfig(name="maxsteps_1000", max_steps=1000),
    ExperimentConfig(name="maxsteps_1500", max_steps=1500),
    ExperimentConfig(name="downsample_24_8_1",
                    n_freq_downsample=(24, 8, 1)),

    ExperimentConfig(name="downsample_72_12_1",
                    n_freq_downsample=(72, 12, 1)),

    ExperimentConfig(name="downsample_168_24_1",
                    n_freq_downsample=(168, 24, 1)),   # weekly+daily+fine
    ExperimentConfig(name="in_transit_w24",
                    use_in_transit=True,
                    in_transit_window=24),

    ExperimentConfig(name="in_transit_w72",
                    use_in_transit=True,
                    in_transit_window=72),

    ExperimentConfig(name="in_transit_w96",
                    use_in_transit=True,
                    in_transit_window=96),
    ExperimentConfig(name="status_rolling_12_24",
                    use_status_rolling=True,
                    status_rolling_windows=(12, 24)),

    ExperimentConfig(name="status_rolling_24_72",
                    use_status_rolling=True,
                    status_rolling_windows=(24, 72)),

    ExperimentConfig(name="status_rolling_48_96",
                    use_status_rolling=True,
                    status_rolling_windows=(48, 96)),
    ExperimentConfig(name="input_480_context_1008",
                    input_size=480,
                    context_len=1008),

    ExperimentConfig(name="input_672_context_1344",
                    input_size=672,
                    context_len=1344),
]


# ── Build features according to config ──
def build_features(nf_df: pd.DataFrame, cfg: ExperimentConfig) -> tuple[pd.DataFrame, list]:
    """
    Возвращает (обновлённый nf_df, hist_exog_list)
    """
    nf_df = add_calendar_features(nf_df)
    nf_df = add_lag_features(nf_df)

    hist_exog = status_cols + ["lag_48", "lag_336"]

    if cfg.use_in_transit:
        nf_df = add_in_transit(nf_df, status_col="status_5",
                               window=cfg.in_transit_window)
        hist_exog.append("in_transit")

    if cfg.use_status_rolling:
        nf_df = add_status_rolling(nf_df, windows=cfg.status_rolling_windows)
        for col in status_cols:
            for w in cfg.status_rolling_windows:
                hist_exog.append(f"{col}_rsum_{w}")

    if cfg.use_y_trend:
        nf_df = add_y_trend(nf_df)
        hist_exog += ["y_mean_24", "y_mean_48", "y_trend_24_48", "y_std_24"]

    return nf_df, hist_exog


# ── Model factory ──
def make_nhits(cfg: ExperimentConfig, hist_exog: list) -> NHITS:
    n_stacks = len(cfg.n_freq_downsample)
    return NHITS(
        h                   = FORECAST_STEPS,
        input_size          = cfg.input_size,
        loss                = MQLoss(level=[80]),
        n_freq_downsample   = list(cfg.n_freq_downsample),
        mlp_units           = [[512, 512]] * n_stacks,
        n_blocks            = [4] * n_stacks,
        max_steps           = cfg.max_steps,
        batch_size          = BATCH_SIZE,
        accelerator         = ACCELERATOR,
        hist_exog_list      = hist_exog,
        futr_exog_list      = FUTR_EXOG,
        scaler_type         = "robust",
        enable_progress_bar = True,
    )


# ── Futr DF ──
def make_futr_df(route_ids: list, train_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for route_id in route_ids:
        last_ts = train_df[train_df["route_id"] == route_id]["timestamp"].max()
        future_ts = pd.date_range(
            start   = last_ts + pd.Timedelta("30min"),
            periods = FORECAST_STEPS,
            freq    = FREQ,
        )
        for ts in future_ts:
            rows.append({"unique_id": route_id, "ds": ts})
    return add_calendar_features(pd.DataFrame(rows))


# ── Helpers ──
def get_pred_col(pred_df: pd.DataFrame) -> str:
    for candidate in ["NHITS-median", "NHITS-q-0.50", "NHITS"]:
        if candidate in pred_df.columns:
            return candidate
    skip = {"unique_id", "ds", "y", "cutoff"}
    num_cols = [c for c in pred_df.columns
                if c not in skip and pd.api.types.is_numeric_dtype(pred_df[c])]
    median_cols = [c for c in num_cols if "median" in c or "0.5" in c or "50" in c]
    return (median_cols or num_cols)[0]


def build_preds_dict(pred_df: pd.DataFrame, pred_col: str) -> dict:
    result = {}
    for route_id, grp in pred_df.groupby("unique_id"):
        result[route_id] = np.clip(grp.sort_values("ds")[pred_col].values, 0, None)
    return result


# ── Experiment runner ──
def run_experiment(cfg: ExperimentConfig):
    print("\n" + "=" * 80)
    print(f"EXPERIMENT: {cfg.name}")
    print("=" * 80)
    print("Config:", asdict(cfg))

    # --- Train NF for CV ---
    train_nf = to_nf_df(
        train_df,
        min_len     = TOTAL_VAL_PTS + cfg.input_size + 1,
        context_len = cfg.context_len,
    )
    print(f"NF train: {train_nf.shape}  routes: {train_nf['unique_id'].nunique()}")

    train_nf, hist_exog = build_features(train_nf, cfg)
    print(f"hist_exog: {hist_exog}")

    # --- CV ---
    nf_cv = NeuralForecast(models=[make_nhits(cfg, hist_exog)], freq=FREQ)
    cv_df = nf_cv.cross_validation(
        df        = train_nf,
        n_windows = N_FOLDS,
        step_size = FORECAST_STEPS,
        refit     = True,
    )

    pred_col = get_pred_col(cv_df)
    cutoffs_sorted = sorted(cv_df["cutoff"].unique())
    cutoff_to_fold = {c: i + 1 for i, c in enumerate(cutoffs_sorted)}
    cv_df["fold"]  = cv_df["cutoff"].map(cutoff_to_fold)
    cv_df = cv_df.sort_values(["fold", "unique_id", "ds"]).reset_index(drop=True)
    cv_df["h"] = cv_df.groupby(["fold", "unique_id"]).cumcount() + 1

    raw_rows = []
    for _, row in cv_df.iterrows():
        y_true = float(row["y"])
        y_pred = float(np.clip(row[pred_col], 0, None))
        ae     = abs(y_pred - y_true)
        ape    = ae / (y_true + 1e-9)
        err    = y_pred - y_true
        raw_rows.append({
            "fold":      int(row["fold"]),
            "route_id":  row["unique_id"],
            "timestamp": row["ds"],
            "h":         int(row["h"]),
            "y_true":    y_true,
            "y_pred":    y_pred,
            "ae":        ae,
            "ape":       ape,
            "err":       err,
        })

    raw_cv_df = pd.DataFrame(raw_rows)

    agg_fold = raw_cv_df.groupby("fold").apply(lambda df: pd.Series({
        "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
        "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
        "mae":   df["ae"].mean(),
        "n":     len(df),
    })).reset_index()

    y_true_all = raw_cv_df["y_true"].values
    y_pred_all = raw_cv_df["y_pred"].values
    total_raw, wape_raw, rbias_raw = wape_rbias(y_true_all, y_pred_all)

    calib_scale = float(y_true_all.sum() / (y_pred_all.sum() + 1e-9))
    y_pred_cal  = np.clip(y_pred_all * calib_scale, 0, None)
    total_cal, wape_cal, rbias_cal = wape_rbias(y_true_all, y_pred_cal)

    print("\nCV summary:")
    print(agg_fold.to_string(index=False))
    print(f"\nGlobal RAW        WAPE={wape_raw:.4f}  |RBias|={rbias_raw:.4f}  Total={total_raw:.4f}")
    print(f"Global CALIBRATED WAPE={wape_cal:.4f}  |RBias|={rbias_cal:.4f}  Total={total_cal:.4f}")
    print(f"calib_scale = {calib_scale:.4f}")

    # --- Train NF on full train ---
    train_nf_full = to_nf_df(
        train_df,
        min_len     = cfg.input_size + 1,
        context_len = cfg.context_len,
    )
    train_nf_full, hist_exog_full = build_features(train_nf_full, cfg)

    nf_full = NeuralForecast(models=[make_nhits(cfg, hist_exog_full)], freq=FREQ)
    nf_full.fit(train_nf_full)

    # --- Predict on test ---
    test_route_ids = test_df["route_id"].unique().tolist()
    futr_df        = make_futr_df(test_route_ids, train_df)
    test_pred_df   = nf_full.predict(futr_df=futr_df)
    test_pred_col  = get_pred_col(test_pred_df)
    test_preds     = build_preds_dict(test_pred_df, test_pred_col)

    predictions_raw = {}
    for route_id in tqdm(test_route_ids, desc=f"Build submission [{cfg.name}]"):
        route_test = test_df[test_df["route_id"] == route_id].sort_values("timestamp")
        preds = test_preds.get(route_id, np.zeros(FORECAST_STEPS))
        for j, (_, row) in enumerate(route_test.iterrows()):
            pred = float(preds[j]) if j < len(preds) else float(preds[-1])
            predictions_raw[row["id"]] = max(0.0, pred)

    submission_raw = (
        pd.DataFrame(list(predictions_raw.items()), columns=["id", "y_pred"])
        .sort_values("id").reset_index(drop=True)
    )
    submission_cal = submission_raw.copy()
    submission_cal["y_pred"] = np.clip(submission_cal["y_pred"] * calib_scale, 0, None)

    # --- Save results ---
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    prefix = f"{EXPERIMENTS_DIR}/{cfg.name}_{ts}"

    submission_raw.to_csv(f"{prefix}_submission_raw.csv", index=False)
    submission_cal.to_csv(f"{prefix}_submission_calibrated.csv", index=False)
    agg_fold.to_csv(f"{prefix}_cv_agg_fold.csv", index=False)

    summary = {
        "config": asdict(cfg),
        "wape_raw": float(wape_raw),
        "rbias_raw": float(rbias_raw),
        "total_raw": float(total_raw),
        "wape_cal": float(wape_cal),
        "rbias_cal": float(rbias_cal),
        "total_cal": float(total_cal),
        "calib_scale": float(calib_scale),
        "cv_agg_fold_path": f"{prefix}_cv_agg_fold.csv",
        "submission_raw_path": f"{prefix}_submission_raw.csv",
        "submission_cal_path": f"{prefix}_submission_calibrated.csv",
    }
    with open(f"{prefix}_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    print(f"\nSaved experiment '{cfg.name}' to prefix: {prefix}")
    print(f"RAW WAPE={wape_raw:.4f}, CAL WAPE={wape_cal:.4f}")
        # ── Очистка памяти ──
    del nf_cv, nf_full, cv_df, train_nf, train_nf_full
    del raw_cv_df, test_pred_df, submission_raw, submission_cal
    import gc
    gc.collect()

    if ACCELERATOR == "mps":
        torch.mps.empty_cache()
    elif ACCELERATOR == "gpu":
        torch.cuda.empty_cache()


# ── Run all experiments ──
if __name__ == "__main__":
    for cfg in EXPERIMENTS:
        run_experiment(cfg)

In [ ]:
# ============================================================
# N-HiTS — Rolling CV (5 folds × 8 steps)
# + per-route статистика для блендинга (как в Chronos-2)
# ============================================================



import warnings
import numpy as np
import pandas as pd
import torch
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MQLoss
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None


def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias


# ── Config ──
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8        # = h
N_FOLDS        = 5
TOTAL_VAL_PTS  = N_FOLDS * FORECAST_STEPS   # 40
CONTEXT_LEN    = 2048
FREQ           = "30min"

INPUT_SIZE = 336      # 7 суток × 48 точек — ловим недельную сезонность
MAX_STEPS  = 800     # чуть больше, раз быстро
BATCH_SIZE = 64       # можно увеличить если памяти хватает


# ── Device ──
if torch.backends.mps.is_available():
    ACCELERATOR = "mps"
    print("Apple Silicon MPS ✅")
elif torch.cuda.is_available():
    ACCELERATOR = "gpu"
else:
    ACCELERATOR = "cpu"
print(f"Device: {ACCELERATOR}\n")


# ── Data ──
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

status_cols = sorted([c for c in train_df.columns if c in ["status_3", "status_2"]])
print(f"Train: {train_df.shape}  Test: {test_df.shape}")
print(f"Status cols: {status_cols}\n")


FUTR_EXOG = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "is_weekend"]

def add_calendar_features(df: pd.DataFrame, ts_col: str = "ds") -> pd.DataFrame:
    ts = pd.to_datetime(df[ts_col])
    df = df.copy()
    df["hour_sin"]   = np.sin(2 * np.pi * ts.dt.hour / 24)
    df["hour_cos"]   = np.cos(2 * np.pi * ts.dt.hour / 24)
    df["dow_sin"]    = np.sin(2 * np.pi * ts.dt.dayofweek / 7)
    df["dow_cos"]    = np.cos(2 * np.pi * ts.dt.dayofweek / 7)
    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(float)
    return df

HIST_EXOG_EXTRA = ["lag_48", "lag_336"]

def add_lag_features(nf_df: pd.DataFrame) -> pd.DataFrame:
    nf_df = nf_df.sort_values(["unique_id", "ds"]).copy()
    nf_df["lag_48"] = (
        nf_df.groupby("unique_id")["y"]
        .transform(lambda s: s.shift(48).bfill().fillna(0))
    )
    nf_df["lag_336"] = (
        nf_df.groupby("unique_id")["y"]
        .transform(lambda s: s.shift(336).bfill().fillna(0))
    )
    return nf_df

# ── NF-формат: unique_id | ds | y [| status_* ...] ──
# Аналог prepare_context_df из Chronos — те же asfreq / interpolate / tail
def to_nf_df(df: pd.DataFrame,
             min_len: int = 32,
             context_len: int = CONTEXT_LEN) -> pd.DataFrame:
    rows = []
    for route_id, grp in df.groupby("route_id"):
        grp = (
            grp.sort_values("timestamp")
            .set_index("timestamp")[[TARGET_COL] + status_cols]
            .asfreq(FREQ)
            .interpolate(method="time")
            .bfill().ffill()
            .tail(context_len)
            .reset_index()
        )
        if len(grp) < min_len:
            continue
        grp.insert(0, "unique_id", route_id)
        grp = grp.rename(columns={"timestamp": "ds", TARGET_COL: "y"})
        rows.append(grp)
    nf_df = pd.concat(rows, ignore_index=True)
    nf_df = add_calendar_features(nf_df)   # futr_exog
    nf_df = add_lag_features(nf_df)         # hist_exog
    return nf_df


# ── Фабрика модели (один конфиг для CV и финального фита) ──
HIST_EXOG = (status_cols + HIST_EXOG_EXTRA) or None

def make_nhits(max_steps: int = MAX_STEPS) -> NHITS:
    return NHITS(
        h                   = FORECAST_STEPS,
        input_size          = INPUT_SIZE,          # 336 = 1 неделя
        loss                = MQLoss(level=[80]),
        # Стеки: день / 4ч / 30мин — каждый на своей частоте
        # Stack 1: MaxPool(48) → 7 дневных точек  (недельный тренд)
        # Stack 2: MaxPool(8)  → 42 точки         (4-часовые блоки)
        # Stack 3: MaxPool(1)  → 336 точек         (полное разрешение)
        n_freq_downsample   = [48, 8, 1],
        mlp_units           = [[512, 512]] * 3,
        n_blocks            = [4, 4, 4],
        max_steps           = max_steps,
        batch_size          = BATCH_SIZE,
        accelerator         = ACCELERATOR,
        hist_exog_list      = HIST_EXOG,
        futr_exog_list      = FUTR_EXOG,
        scaler_type         = "robust",            # median/IQR — пики не "приплющиваются"
        enable_progress_bar = True,
    )

def make_futr_df(route_ids: list, train_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for route_id in route_ids:
        last_ts = train_df[train_df["route_id"] == route_id]["timestamp"].max()
        future_ts = pd.date_range(
            start   = last_ts + pd.Timedelta("30min"),
            periods = FORECAST_STEPS,
            freq    = FREQ,
        )
        for ts in future_ts:
            rows.append({"unique_id": route_id, "ds": ts})
    return add_calendar_features(pd.DataFrame(rows))

# ── Выбор колонки точечного прогноза (аналог get_pred_col из Chronos) ──
def get_pred_col(pred_df: pd.DataFrame) -> str:
    for candidate in ["NHITS-median", "NHITS-q-0.50", "NHITS"]:
        if candidate in pred_df.columns:
            return candidate
    skip = {"unique_id", "ds", "y", "cutoff"}
    num_cols = [c for c in pred_df.columns
                if c not in skip and pd.api.types.is_numeric_dtype(pred_df[c])]
    median_cols = [c for c in num_cols if "median" in c or "0.5" in c or "50" in c]
    return (median_cols or num_cols)[0]


# ── Словарь предсказаний: route_id → array[FORECAST_STEPS] ──
def build_preds_dict(pred_df: pd.DataFrame, pred_col: str) -> dict:
    result = {}
    for route_id, grp in pred_df.groupby("unique_id"):
        result[route_id] = np.clip(grp.sort_values("ds")[pred_col].values, 0, None)
    return result


# ============================================================
# ROLLING CROSS-VALIDATION
# neuralforecast.cross_validation(n_windows=5, step_size=8, refit=True)
# window 1: train=[:-40]  val=[-40:-32]
# window 2: train=[:-32]  val=[-32:-24]
# window 3: train=[:-24]  val=[-24:-16]
# window 4: train=[:-16]  val=[-16: -8]
# window 5: train=[: -8]  val=[ -8:end]
# refit=True — переобучение на каждом фолде (= поведение Chronos)
# ============================================================
print("=" * 65)
print(f"ROLLING CV  ({N_FOLDS} folds × {FORECAST_STEPS} steps = {TOTAL_VAL_PTS} points)")
print("=" * 65)

train_nf = to_nf_df(
    train_df,
    min_len     = TOTAL_VAL_PTS + INPUT_SIZE + 1,
    context_len = CONTEXT_LEN,
)
print(f"NF train: {train_nf.shape}  routes: {train_nf['unique_id'].nunique()}\n")

nf_cv = NeuralForecast(models=[make_nhits()], freq=FREQ)

# refit=True: модель переобучается для каждого фолда (как Chronos per-fold inference)
# refit=False: обучается один раз, быстрее, но менее честно
cv_df = nf_cv.cross_validation(
    df        = train_nf,
    n_windows = N_FOLDS,
    step_size = FORECAST_STEPS,
    refit     = True,
)

print(f"\nCV columns: {list(cv_df.columns)}")
pred_col = get_pred_col(cv_df)
print(f"Point forecast column: '{pred_col}'\n")

# ── cutoff → fold (1 = самый старый, N_FOLDS = самый свежий) ──
cutoffs_sorted   = sorted(cv_df["cutoff"].unique())
cutoff_to_fold   = {c: i + 1 for i, c in enumerate(cutoffs_sorted)}
cv_df["fold"]    = cv_df["cutoff"].map(cutoff_to_fold)

# ── шаг h внутри фолда (1..FORECAST_STEPS) ──
cv_df = cv_df.sort_values(["fold", "unique_id", "ds"]).reset_index(drop=True)
cv_df["h"] = cv_df.groupby(["fold", "unique_id"]).cumcount() + 1

# ── raw_rows (идентична структура с Chronos) ──
raw_rows = []
for _, row in cv_df.iterrows():
    y_true = float(row["y"])
    y_pred = float(np.clip(row[pred_col], 0, None))
    ae     = abs(y_pred - y_true)
    ape    = ae / (y_true + 1e-9)
    err    = y_pred - y_true
    raw_rows.append({
        "fold":      int(row["fold"]),
        "route_id":  row["unique_id"],
        "timestamp": row["ds"],
        "h":         int(row["h"]),
        "y_true":    y_true,
        "y_pred":    y_pred,
        "ae":        ae,
        "ape":       ape,
        "err":       err,
    })

# ── per-fold console output (как у Chronos) ──
for fold in range(1, N_FOLDS + 1):
    fdf        = cv_df[cv_df["fold"] == fold]
    n_routes   = fdf["unique_id"].nunique()
    val_end    = TOTAL_VAL_PTS - (fold - 1) * FORECAST_STEPS
    val_start  = val_end - FORECAST_STEPS
    yt = fdf["y"].values
    yp = np.clip(fdf[pred_col].values, 0, None)
    tot, wape, rb = wape_rbias(yt, yp)
    print(f"\nFold {fold}/{N_FOLDS}  "
          f"(val: -{val_end}..{'-'+str(val_start) if val_start else 'end'})  "
          f"routes={n_routes}")
    print(f"  RAW   WAPE={wape:.4f}  |RBias|={rb:.4f}  Total={tot:.4f}")


# ============================================================
# АГРЕГАТЫ (идентично Chronos для блендинга)
# ============================================================
raw_cv_df = pd.DataFrame(raw_rows)

# ── По фолду ──
agg_fold = raw_cv_df.groupby("fold").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
    "n":     len(df),
})).reset_index()

# ── По шагу h ──
agg_h = raw_cv_df.groupby("h").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
})).reset_index()

# ── По маршруту ──
agg_route = raw_cv_df.groupby("route_id").agg(
    n_points   = ("y_true",  "count"),
    mae        = ("ae",      "mean"),
    median_ae  = ("ae",      "median"),
    std_ae     = ("ae",      "std"),
    mape       = ("ape",     "mean"),
    median_ape = ("ape",     "median"),
    mean_err   = ("err",     "mean"),
    median_err = ("err",     "median"),
    mean_true  = ("y_true",  "mean"),
    mean_pred  = ("y_pred",  "mean"),
).reset_index()
agg_route["calib_scale"] = (
    raw_cv_df.groupby("route_id")["y_true"].sum().values /
    (raw_cv_df.groupby("route_id")["y_pred"].sum().values + 1e-9)
)
agg_route["wape"] = (
    raw_cv_df.groupby("route_id")["ae"].sum().values /
    (raw_cv_df.groupby("route_id")["y_true"].sum().values + 1e-9)
)

# ── Глобальные метрики ──
y_true_all = raw_cv_df["y_true"].values
y_pred_all = raw_cv_df["y_pred"].values
total_raw, wape_raw, rbias_raw = wape_rbias(y_true_all, y_pred_all)

calib_scale = float(y_true_all.sum() / (y_pred_all.sum() + 1e-9))
y_pred_cal  = np.clip(y_pred_all * calib_scale, 0, None)
total_cal, wape_cal, rbias_cal = wape_rbias(y_true_all, y_pred_cal)

print("\n" + "=" * 65)
print("SUMMARY")
print("=" * 65)
print("\n── По фолдам ──")
print(agg_fold.to_string(index=False))
print("\n── По шагам h ──")
print(agg_h.to_string(index=False))
print(f"\n{'Global RAW':25s}  WAPE={wape_raw:.4f}  |RBias|={rbias_raw:.4f}  Total={total_raw:.4f}")
print(f"{'Global CALIBRATED':25s}  WAPE={wape_cal:.4f}  |RBias|={rbias_cal:.4f}  Total={total_cal:.4f}")
print(f"  calib_scale = {calib_scale:.4f}")
print(f"\nCalibration {'HELPS ✅' if total_cal < total_raw else 'HURTS ❌'} "
      f"(Δ = {total_cal - total_raw:+.4f})")
print("\n── Топ-10 сложных маршрутов (по median_ape) ──")
print(agg_route.sort_values("median_ape", ascending=False)
      [["route_id", "mae", "median_ae", "mape", "median_ape", "mean_err", "calib_scale"]]
      .head(10).to_string(index=False))

raw_cv_df.to_csv("nhits_cv_raw.csv", index=False)
agg_route.to_csv("nhits_cv_agg_route.csv", index=False)
agg_fold.to_csv("nhits_cv_agg_fold.csv", index=False)
agg_h.to_csv("nhits_cv_agg_h.csv", index=False)
print("\n✅ nhits_cv_raw.csv")
print("✅ nhits_cv_agg_route.csv")
print("✅ nhits_cv_agg_fold.csv")
print("✅ nhits_cv_agg_h.csv")


# ============================================================
# TEST PREDICTION
# Финальный фит на полных данных, predict() берёт последние
# input_size точек из тренировочного датасета автоматически
# ============================================================
print("\n" + "=" * 65)
print("TEST PREDICTION")
print("=" * 65)

train_nf_full = to_nf_df(train_df, min_len=INPUT_SIZE + 1, context_len=CONTEXT_LEN)
test_route_ids = test_df["route_id"].unique().tolist()
print(f"Routes: {len(test_route_ids)}  Fitting on full train data...")

nf_full = NeuralForecast(models=[make_nhits()], freq=FREQ)
nf_full.fit(train_nf_full)

print("Running inference...")
futr_df      = make_futr_df(test_route_ids, train_df)
test_pred_df = nf_full.predict(futr_df=futr_df)   # next FORECAST_STEPS × 30min на каждый маршрут
test_pred_col = get_pred_col(test_pred_df)
test_preds    = build_preds_dict(test_pred_df, test_pred_col)

predictions_raw = {}
for route_id in tqdm(test_route_ids, desc="Build submission"):
    route_test = test_df[test_df["route_id"] == route_id].sort_values("timestamp")
    preds = test_preds.get(route_id, np.zeros(FORECAST_STEPS))
    for j, (_, row) in enumerate(route_test.iterrows()):
        pred = float(preds[j]) if j < len(preds) else float(preds[-1])
        predictions_raw[row["id"]] = max(0.0, pred)

submission_raw = (
    pd.DataFrame(list(predictions_raw.items()), columns=["id", "y_pred"])
    .sort_values("id").reset_index(drop=True)
)
submission_cal = submission_raw.copy()
submission_cal["y_pred"] = np.clip(submission_cal["y_pred"] * calib_scale, 0, None)

submission_raw.to_csv("submission_nhits_raw.csv", index=False)
submission_cal.to_csv("submission_nhits_calibrated.csv", index=False)
print(f"\n✅ submission_nhits_raw.csv")
print(f"✅ submission_nhits_calibrated.csv")
print(f"\n── Raw ──\n{submission_raw['y_pred'].describe().round(2)}")
print(f"\n── Calibrated (scale={calib_scale:.4f}) ──\n{submission_cal['y_pred'].describe().round(2)}")

In [ ]:
# ============================================================
# N-HiTS — Seed Ensemble (N_SEEDS runs) + Rolling CV (5×8)
# Честный ансамбль: CV усредняется по сидам до подсчёта метрик
# ============================================================

import warnings
import numpy as np
import pandas as pd
import torch
import pytorch_lightning as pl
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MQLoss
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None


def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias


# ── Config ──
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8
N_FOLDS        = 5
TOTAL_VAL_PTS  = N_FOLDS * FORECAST_STEPS   # 40
CONTEXT_LEN    = 2048
FREQ           = "30min"

INPUT_SIZE = 336
MAX_STEPS  = 800
BATCH_SIZE = 64

# ── Seed ensemble config ──
N_SEEDS = 5   # 5–10, чем больше — тем ниже variance, но дольше
SEEDS   = [1, 42, 123, 777, 2024, 31415, 99, 1337][:N_SEEDS]
print(f"Seeds ({N_SEEDS}): {SEEDS}\n")

# ── Device ──
if torch.backends.mps.is_available():
    ACCELERATOR = "mps"
    print("Apple Silicon MPS ✅")
elif torch.cuda.is_available():
    ACCELERATOR = "gpu"
else:
    ACCELERATOR = "cpu"
print(f"Device: {ACCELERATOR}\n")

# ── Data ──
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

status_cols = sorted([c for c in train_df.columns if c in ["status_3", "status_2", "status_5"]])
print(f"Train: {train_df.shape}  Test: {test_df.shape}")
print(f"Status cols: {status_cols}\n")

FUTR_EXOG = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "is_weekend"]


def add_calendar_features(df: pd.DataFrame, ts_col: str = "ds") -> pd.DataFrame:
    ts = pd.to_datetime(df[ts_col])
    df = df.copy()
    df["hour_sin"]   = np.sin(2 * np.pi * ts.dt.hour / 24)
    df["hour_cos"]   = np.cos(2 * np.pi * ts.dt.hour / 24)
    df["dow_sin"]    = np.sin(2 * np.pi * ts.dt.dayofweek / 7)
    df["dow_cos"]    = np.cos(2 * np.pi * ts.dt.dayofweek / 7)
    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(float)
    return df


HIST_EXOG_EXTRA = ["lag_48", "lag_336"]


def add_lag_features(nf_df: pd.DataFrame) -> pd.DataFrame:
    nf_df = nf_df.sort_values(["unique_id", "ds"]).copy()
    nf_df["lag_48"] = (
        nf_df.groupby("unique_id")["y"]
        .transform(lambda s: s.shift(48).bfill().fillna(0))
    )
    nf_df["lag_336"] = (
        nf_df.groupby("unique_id")["y"]
        .transform(lambda s: s.shift(336).bfill().fillna(0))
    )
    return nf_df


def to_nf_df(df: pd.DataFrame,
             min_len: int = 32,
             context_len: int = CONTEXT_LEN) -> pd.DataFrame:
    rows = []
    for route_id, grp in df.groupby("route_id"):
        grp = (
            grp.sort_values("timestamp")
            .set_index("timestamp")[[TARGET_COL] + status_cols]
            .asfreq(FREQ)
            .interpolate(method="time")
            .bfill().ffill()
            .tail(context_len)
            .reset_index()
        )
        if len(grp) < min_len:
            continue
        grp.insert(0, "unique_id", route_id)
        grp = grp.rename(columns={"timestamp": "ds", TARGET_COL: "y"})
        rows.append(grp)
    nf_df = pd.concat(rows, ignore_index=True)
    nf_df = add_calendar_features(nf_df)
    nf_df = add_lag_features(nf_df)
    return nf_df


HIST_EXOG = (status_cols + HIST_EXOG_EXTRA) or None


def set_all_seeds(seed: int):
    """Фиксируем все генераторы случайных чисел для воспроизводимости."""
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    pl.seed_everything(seed, workers=True)


def make_nhits(seed: int, max_steps: int = MAX_STEPS) -> NHITS:
    return NHITS(
        h                   = FORECAST_STEPS,
        input_size          = INPUT_SIZE,
        loss                = MQLoss(level=[80]),
        n_freq_downsample   = [48, 8, 1],
        mlp_units           = [[512, 512]] * 3,
        n_blocks            = [4, 4, 4],
        max_steps           = max_steps,
        batch_size          = BATCH_SIZE,
        accelerator         = ACCELERATOR,
        hist_exog_list      = HIST_EXOG,
        futr_exog_list      = FUTR_EXOG,
        scaler_type         = "robust",
        random_seed         = seed,       # ← ключевой параметр
        enable_progress_bar = True,
    )


def get_pred_col(pred_df: pd.DataFrame) -> str:
    for candidate in ["NHITS-median", "NHITS-q-0.50", "NHITS"]:
        if candidate in pred_df.columns:
            return candidate
    skip = {"unique_id", "ds", "y", "cutoff"}
    num_cols = [c for c in pred_df.columns
                if c not in skip and pd.api.types.is_numeric_dtype(pred_df[c])]
    median_cols = [c for c in num_cols if "median" in c or "0.5" in c or "50" in c]
    return (median_cols or num_cols)[0]


def make_futr_df(route_ids: list, train_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for route_id in route_ids:
        last_ts = train_df[train_df["route_id"] == route_id]["timestamp"].max()
        future_ts = pd.date_range(
            start   = last_ts + pd.Timedelta("30min"),
            periods = FORECAST_STEPS,
            freq    = FREQ,
        )
        for ts in future_ts:
            rows.append({"unique_id": route_id, "ds": ts})
    return add_calendar_features(pd.DataFrame(rows))


# ============================================================
# STEP 1: ROLLING CV ENSEMBLE
# Для каждого seed запускаем полный cross_validation с refit=True.
# Собираем pred-колонки в один датафрейм, затем усредняем.
# ============================================================
print("=" * 65)
print(f"CV SEED ENSEMBLE  ({N_SEEDS} seeds × {N_FOLDS} folds × {FORECAST_STEPS} steps)")
print("=" * 65)

train_nf = to_nf_df(
    train_df,
    min_len     = TOTAL_VAL_PTS + INPUT_SIZE + 1,
    context_len = CONTEXT_LEN,
)
print(f"NF train: {train_nf.shape}  routes: {train_nf['unique_id'].nunique()}\n")

# Сюда собираем предсказания каждого seed: key=(unique_id, ds, cutoff) → pred
# Используем merge по ключевым колонкам, чтобы не перепутать строки
cv_ensemble_cols = []   # список Series с pred каждого seed
cv_base_df       = None # unique_id / ds / cutoff / y (берём один раз)

for i, seed in enumerate(SEEDS):
    print(f"\n{'─'*50}")
    print(f"SEED {seed}  ({i+1}/{N_SEEDS})")
    print(f"{'─'*50}")
    set_all_seeds(seed)

    nf_cv = NeuralForecast(models=[make_nhits(seed=seed)], freq=FREQ)
    cv_df_seed = nf_cv.cross_validation(
        df        = train_nf,
        n_windows = N_FOLDS,
        step_size = FORECAST_STEPS,
        refit     = True,
    )

    pred_col = get_pred_col(cv_df_seed)

    if cv_base_df is None:
        # Сохраняем структуру (unique_id, ds, cutoff, y) из первого прогона
        cv_base_df = cv_df_seed[["unique_id", "ds", "cutoff", "y"]].copy()

    # Сохраняем точечный прогноз данного seed как отдельную колонку
    col_name = f"pred_seed_{seed}"
    cv_df_seed = cv_df_seed[["unique_id", "ds", "cutoff", pred_col]].rename(
        columns={pred_col: col_name}
    )
    cv_ensemble_cols.append(cv_df_seed)

    # Быстрый single-seed отчёт
    yt = cv_base_df["y"].values if cv_base_df is not None else cv_df_seed["y"].values
    yp = np.clip(cv_df_seed[col_name].values, 0, None)
    # нужен merge обратно на y
    tmp = cv_base_df.merge(cv_df_seed, on=["unique_id", "ds", "cutoff"])
    t, w, r = wape_rbias(tmp["y"].values, np.clip(tmp[col_name].values, 0, None))
    print(f"  Seed {seed}: WAPE={w:.4f}  |RBias|={r:.4f}  Total={t:.4f}")


# ── Merge всех seed-предсказаний ──
cv_merged = cv_base_df.copy()
for seed_df in cv_ensemble_cols:
    cv_merged = cv_merged.merge(seed_df, on=["unique_id", "ds", "cutoff"])

pred_seed_cols = [f"pred_seed_{s}" for s in SEEDS]
cv_merged["pred_ensemble"] = cv_merged[pred_seed_cols].clip(lower=0).mean(axis=1)

# ── cutoff → fold ──
cutoffs_sorted = sorted(cv_merged["cutoff"].unique())
cutoff_to_fold = {c: i + 1 for i, c in enumerate(cutoffs_sorted)}
cv_merged["fold"] = cv_merged["cutoff"].map(cutoff_to_fold)
cv_merged = cv_merged.sort_values(["fold", "unique_id", "ds"]).reset_index(drop=True)
cv_merged["h"] = cv_merged.groupby(["fold", "unique_id"]).cumcount() + 1


# ── Per-fold метрики: single-seed baseline vs ensemble ──
print("\n" + "=" * 65)
print("FOLD METRICS:  Single-seed (mean across seeds) vs Ensemble")
print("=" * 65)

fold_rows = []
for fold in range(1, N_FOLDS + 1):
    fdf = cv_merged[cv_merged["fold"] == fold]
    yt  = fdf["y"].values

    # среднее single-seed WAPE (по каждому seed отдельно)
    single_totals = []
    for sc in pred_seed_cols:
        t, w, r = wape_rbias(yt, fdf[sc].values)
        single_totals.append(t)

    yp_ens = fdf["pred_ensemble"].values
    t_ens, w_ens, r_ens = wape_rbias(yt, yp_ens)

    print(f"\nFold {fold}/{N_FOLDS}  routes={fdf['unique_id'].nunique()}")
    print(f"  Single-seed avg Total = {np.mean(single_totals):.4f}  "
          f"(min={np.min(single_totals):.4f} max={np.max(single_totals):.4f})")
    print(f"  Ensemble      Total   = {t_ens:.4f}  "
          f"WAPE={w_ens:.4f}  |RBias|={r_ens:.4f}")
    print(f"  Δ (ens − avg) = {t_ens - np.mean(single_totals):+.4f}  "
          f"{'✅ WINS' if t_ens < np.mean(single_totals) else '❌'}")
    fold_rows.append({
        "fold": fold,
        "single_avg": np.mean(single_totals),
        "single_min": np.min(single_totals),
        "ensemble":   t_ens,
        "delta":      t_ens - np.mean(single_totals),
    })


# ── Глобальные метрики ──
yt_all  = cv_merged["y"].values
yp_ens  = cv_merged["pred_ensemble"].values

single_global = []
for sc in pred_seed_cols:
    t, _, _ = wape_rbias(yt_all, cv_merged[sc].values)
    single_global.append(t)

t_raw, w_raw, r_raw = wape_rbias(yt_all, yp_ens)
calib_scale = float(yt_all.sum() / (yp_ens.sum() + 1e-9))
t_cal, w_cal, r_cal = wape_rbias(yt_all, np.clip(yp_ens * calib_scale, 0, None))

print("\n" + "=" * 65)
print("GLOBAL SUMMARY")
print("=" * 65)
print(f"{'Single-seed (avg)':30s}  Total = {np.mean(single_global):.4f}")
print(f"{'Single-seed (best)':30s}  Total = {np.min(single_global):.4f}")
print(f"{'Ensemble raw':30s}  Total = {t_raw:.4f}  WAPE={w_raw:.4f}  |RBias|={r_raw:.4f}")
print(f"{'Ensemble calibrated':30s}  Total = {t_cal:.4f}  WAPE={w_cal:.4f}  |RBias|={r_cal:.4f}")
print(f"  calib_scale = {calib_scale:.4f}")
print(f"\nEnsemble vs avg single-seed: {t_raw - np.mean(single_global):+.4f}  "
      f"{'✅ WINS' if t_raw < np.mean(single_global) else '❌'}")
print(f"Calibration: {'HELPS ✅' if t_cal < t_raw else 'HURTS ❌'}  "
      f"(Δ = {t_cal - t_raw:+.4f})")


# ── raw_rows для совместимости с agg_route / блендингом ──
raw_rows = []
for _, row in cv_merged.iterrows():
    y_true = float(row["y"])
    y_pred = float(max(0.0, row["pred_ensemble"]))
    ae = abs(y_pred - y_true)
    raw_rows.append({
        "fold":      int(row["fold"]),
        "route_id":  row["unique_id"],
        "timestamp": row["ds"],
        "h":         int(row["h"]),
        "y_true":    y_true,
        "y_pred":    y_pred,
        "ae":        ae,
        "ape":       ae / (y_true + 1e-9),
        "err":       y_pred - y_true,
    })
raw_cv_df = pd.DataFrame(raw_rows)

agg_fold = raw_cv_df.groupby("fold").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
    "n":     len(df),
})).reset_index()

agg_h = raw_cv_df.groupby("h").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
})).reset_index()

agg_route = raw_cv_df.groupby("route_id").agg(
    n_points   = ("y_true", "count"),
    mae        = ("ae",     "mean"),
    median_ae  = ("ae",     "median"),
    std_ae     = ("ae",     "std"),
    mape       = ("ape",    "mean"),
    median_ape = ("ape",    "median"),
    mean_err   = ("err",    "mean"),
    median_err = ("err",    "median"),
    mean_true  = ("y_true", "mean"),
    mean_pred  = ("y_pred", "mean"),
).reset_index()
agg_route["calib_scale"] = (
    raw_cv_df.groupby("route_id")["y_true"].sum().values /
    (raw_cv_df.groupby("route_id")["y_pred"].sum().values + 1e-9)
)
agg_route["wape"] = (
    raw_cv_df.groupby("route_id")["ae"].sum().values /
    (raw_cv_df.groupby("route_id")["y_true"].sum().values + 1e-9)
)

print("\n── По шагам h (ensemble) ──")
print(agg_h.to_string(index=False))
print("\n── Топ-10 сложных маршрутов ──")
print(agg_route.sort_values("median_ape", ascending=False)
      [["route_id", "mae", "median_ae", "mape", "median_ape", "mean_err", "calib_scale"]]
      .head(10).to_string(index=False))

raw_cv_df.to_csv("nhits_ensemble_cv_raw.csv", index=False)
agg_route.to_csv("nhits_ensemble_cv_agg_route.csv", index=False)
agg_fold.to_csv("nhits_ensemble_cv_agg_fold.csv", index=False)
agg_h.to_csv("nhits_ensemble_cv_agg_h.csv", index=False)
cv_merged.to_csv("nhits_ensemble_cv_all_seeds.csv", index=False)
print("\n✅ nhits_ensemble_cv_raw.csv")
print("✅ nhits_ensemble_cv_agg_route.csv")
print("✅ nhits_ensemble_cv_agg_fold.csv")
print("✅ nhits_ensemble_cv_agg_h.csv")
print("✅ nhits_ensemble_cv_all_seeds.csv  (предсказания всех seeds для анализа)")


# ============================================================
# STEP 2: TEST SEED ENSEMBLE
# Каждый seed: fit(full_train) → predict() → собираем предсказания
# ============================================================
print("\n" + "=" * 65)
print("TEST SEED ENSEMBLE")
print("=" * 65)

train_nf_full  = to_nf_df(train_df, min_len=INPUT_SIZE + 1, context_len=CONTEXT_LEN)
test_route_ids = test_df["route_id"].unique().tolist()
futr_df        = make_futr_df(test_route_ids, train_df)
print(f"Routes: {len(test_route_ids)}")

# Собираем предсказания каждого seed: route_id → array[FORECAST_STEPS]
all_test_preds = []   # list of dicts: route_id → np.array

for i, seed in enumerate(SEEDS):
    print(f"\n── Seed {seed}  ({i+1}/{N_SEEDS}) — fit + predict ──")
    set_all_seeds(seed)

    nf_full = NeuralForecast(models=[make_nhits(seed=seed)], freq=FREQ)
    nf_full.fit(train_nf_full)
    pred_df = nf_full.predict(futr_df=futr_df)

    pred_col = get_pred_col(pred_df)
    seed_preds = {}
    for route_id, grp in pred_df.groupby("unique_id"):
        seed_preds[route_id] = np.clip(grp.sort_values("ds")[pred_col].values, 0, None)

    all_test_preds.append(seed_preds)
    print(f"  Seed {seed} done ✅")


# ── Усреднение по seeds ──
print("\nAveraging predictions across seeds...")
ensemble_test_preds = {}
for route_id in test_route_ids:
    arrays = [sp[route_id] for sp in all_test_preds if route_id in sp]
    if arrays:
        ensemble_test_preds[route_id] = np.stack(arrays).mean(axis=0)
    else:
        ensemble_test_preds[route_id] = np.zeros(FORECAST_STEPS)


# ── Построение сабмитов ──
predictions_raw = {}
for route_id in tqdm(test_route_ids, desc="Build submission"):
    route_test = test_df[test_df["route_id"] == route_id].sort_values("timestamp")
    preds = ensemble_test_preds.get(route_id, np.zeros(FORECAST_STEPS))
    for j, (_, row) in enumerate(route_test.iterrows()):
        pred = float(preds[j]) if j < len(preds) else float(preds[-1])
        predictions_raw[row["id"]] = max(0.0, pred)

submission_raw = (
    pd.DataFrame(list(predictions_raw.items()), columns=["id", "y_pred"])
    .sort_values("id").reset_index(drop=True)
)
submission_cal = submission_raw.copy()
submission_cal["y_pred"] = np.clip(submission_cal["y_pred"] * calib_scale, 0, None)

submission_raw.to_csv("submission_nhits_ensemble_raw.csv", index=False)
submission_cal.to_csv("submission_nhits_ensemble_calibrated.csv", index=False)
print(f"\n✅ submission_nhits_ensemble_raw.csv")
print(f"✅ submission_nhits_ensemble_calibrated.csv")
print(f"\n── Raw ──\n{submission_raw['y_pred'].describe().round(2)}")
print(f"\n── Calibrated (scale={calib_scale:.4f}) ──\n{submission_cal['y_pred'].describe().round(2)}")

# ── Итоговое сравнение ──
print("\n" + "=" * 65)
print("FINAL COMPARISON  (на CV)")
print("=" * 65)
for row in fold_rows:
    print(f"  Fold {row['fold']}:  single_avg={row['single_avg']:.4f}  "
          f"ensemble={row['ensemble']:.4f}  Δ={row['delta']:+.4f}")
print(f"\n  Global single avg : {np.mean(single_global):.4f}")
print(f"  Global ensemble   : {t_raw:.4f}  (Δ={t_raw - np.mean(single_global):+.4f})")
print(f"  Global ens + calib: {t_cal:.4f}  (Δ={t_cal - np.mean(single_global):+.4f})")

In [ ]:
# ============================================================
# N-HiTS — Rolling CV (5 folds × 8 steps)
# + per-route статистика для блендинга (как в Chronos-2)
# ============================================================



import warnings
import numpy as np
import pandas as pd
import torch
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MQLoss
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None


def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias


# ── Config ──
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8        # = h
N_FOLDS        = 5
TOTAL_VAL_PTS  = N_FOLDS * FORECAST_STEPS   # 40
CONTEXT_LEN    = 2048
FREQ           = "30min"

INPUT_SIZE = 336      # 7 суток × 48 точек — ловим недельную сезонность
MAX_STEPS  = 800     # чуть больше, раз быстро
BATCH_SIZE = 64       # можно увеличить если памяти хватает


# ── Device ──
if torch.backends.mps.is_available():
    ACCELERATOR = "mps"
    print("Apple Silicon MPS ✅")
elif torch.cuda.is_available():
    ACCELERATOR = "gpu"
else:
    ACCELERATOR = "cpu"
print(f"Device: {ACCELERATOR}\n")


# ── Data ──
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

status_cols = sorted([c for c in train_df.columns if c in ["status_3", "status_2", "status_5"]])
print(f"Train: {train_df.shape}  Test: {test_df.shape}")
print(f"Status cols: {status_cols}\n")


FUTR_EXOG = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "is_weekend"]

def add_calendar_features(df: pd.DataFrame, ts_col: str = "ds") -> pd.DataFrame:
    ts = pd.to_datetime(df[ts_col])
    df = df.copy()
    df["hour_sin"]   = np.sin(2 * np.pi * ts.dt.hour / 24)
    df["hour_cos"]   = np.cos(2 * np.pi * ts.dt.hour / 24)
    df["dow_sin"]    = np.sin(2 * np.pi * ts.dt.dayofweek / 7)
    df["dow_cos"]    = np.cos(2 * np.pi * ts.dt.dayofweek / 7)
    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(float)
    return df

HIST_EXOG_EXTRA = ["lag_48", "lag_336"]

def add_lag_features(nf_df: pd.DataFrame) -> pd.DataFrame:
    nf_df = nf_df.sort_values(["unique_id", "ds"]).copy()
    nf_df["lag_48"] = (
        nf_df.groupby("unique_id")["y"]
        .transform(lambda s: s.shift(48).bfill().fillna(0))
    )
    nf_df["lag_336"] = (
        nf_df.groupby("unique_id")["y"]
        .transform(lambda s: s.shift(336).bfill().fillna(0))
    )
    return nf_df

# ── NF-формат: unique_id | ds | y [| status_* ...] ──
# Аналог prepare_context_df из Chronos — те же asfreq / interpolate / tail
def to_nf_df(df: pd.DataFrame,
             min_len: int = 32,
             context_len: int = CONTEXT_LEN) -> pd.DataFrame:
    rows = []
    for route_id, grp in df.groupby("route_id"):
        grp = (
            grp.sort_values("timestamp")
            .set_index("timestamp")[[TARGET_COL] + status_cols]
            .asfreq(FREQ)
            .interpolate(method="time")
            .bfill().ffill()
            .tail(context_len)
            .reset_index()
        )
        if len(grp) < min_len:
            continue
        grp.insert(0, "unique_id", route_id)
        grp = grp.rename(columns={"timestamp": "ds", TARGET_COL: "y"})
        rows.append(grp)
    nf_df = pd.concat(rows, ignore_index=True)
    nf_df = add_calendar_features(nf_df)   # futr_exog
    nf_df = add_lag_features(nf_df)         # hist_exog
    return nf_df


# ── Фабрика модели (один конфиг для CV и финального фита) ──
HIST_EXOG = (status_cols + HIST_EXOG_EXTRA) or None

def make_nhits(max_steps: int = MAX_STEPS) -> NHITS:
    return NHITS(
        h                   = FORECAST_STEPS,
        input_size          = INPUT_SIZE,          # 336 = 1 неделя
        loss                = MQLoss(level=[80]),
        # Стеки: день / 4ч / 30мин — каждый на своей частоте
        # Stack 1: MaxPool(48) → 7 дневных точек  (недельный тренд)
        # Stack 2: MaxPool(8)  → 42 точки         (4-часовые блоки)
        # Stack 3: MaxPool(1)  → 336 точек         (полное разрешение)
        n_freq_downsample   = [48, 8, 1],
        mlp_units           = [[512, 512]] * 3,
        n_blocks            = [4, 4, 4],
        max_steps           = max_steps,
        batch_size          = BATCH_SIZE,
        accelerator         = ACCELERATOR,
        hist_exog_list      = HIST_EXOG,
        futr_exog_list      = FUTR_EXOG,
        scaler_type         = "revin",
        enable_progress_bar = True,
    )

def make_futr_df(route_ids: list, train_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for route_id in route_ids:
        last_ts = train_df[train_df["route_id"] == route_id]["timestamp"].max()
        future_ts = pd.date_range(
            start   = last_ts + pd.Timedelta("30min"),
            periods = FORECAST_STEPS,
            freq    = FREQ,
        )
        for ts in future_ts:
            rows.append({"unique_id": route_id, "ds": ts})
    return add_calendar_features(pd.DataFrame(rows))

# ── Выбор колонки точечного прогноза (аналог get_pred_col из Chronos) ──
def get_pred_col(pred_df: pd.DataFrame) -> str:
    for candidate in ["NHITS-median", "NHITS-q-0.50", "NHITS"]:
        if candidate in pred_df.columns:
            return candidate
    skip = {"unique_id", "ds", "y", "cutoff"}
    num_cols = [c for c in pred_df.columns
                if c not in skip and pd.api.types.is_numeric_dtype(pred_df[c])]
    median_cols = [c for c in num_cols if "median" in c or "0.5" in c or "50" in c]
    return (median_cols or num_cols)[0]


# ── Словарь предсказаний: route_id → array[FORECAST_STEPS] ──
def build_preds_dict(pred_df: pd.DataFrame, pred_col: str) -> dict:
    result = {}
    for route_id, grp in pred_df.groupby("unique_id"):
        result[route_id] = np.clip(grp.sort_values("ds")[pred_col].values, 0, None)
    return result


# ============================================================
# ROLLING CROSS-VALIDATION
# neuralforecast.cross_validation(n_windows=5, step_size=8, refit=True)
# window 1: train=[:-40]  val=[-40:-32]
# window 2: train=[:-32]  val=[-32:-24]
# window 3: train=[:-24]  val=[-24:-16]
# window 4: train=[:-16]  val=[-16: -8]
# window 5: train=[: -8]  val=[ -8:end]
# refit=True — переобучение на каждом фолде (= поведение Chronos)
# ============================================================
print("=" * 65)
print(f"ROLLING CV  ({N_FOLDS} folds × {FORECAST_STEPS} steps = {TOTAL_VAL_PTS} points)")
print("=" * 65)

train_nf = to_nf_df(
    train_df,
    min_len     = TOTAL_VAL_PTS + INPUT_SIZE + 1,
    context_len = CONTEXT_LEN,
)
print(f"NF train: {train_nf.shape}  routes: {train_nf['unique_id'].nunique()}\n")

nf_cv = NeuralForecast(models=[make_nhits()], freq=FREQ)

# refit=True: модель переобучается для каждого фолда (как Chronos per-fold inference)
# refit=False: обучается один раз, быстрее, но менее честно
cv_df = nf_cv.cross_validation(
    df        = train_nf,
    n_windows = N_FOLDS,
    step_size = FORECAST_STEPS,
    refit     = True,
)

print(f"\nCV columns: {list(cv_df.columns)}")
pred_col = get_pred_col(cv_df)
print(f"Point forecast column: '{pred_col}'\n")

# ── cutoff → fold (1 = самый старый, N_FOLDS = самый свежий) ──
cutoffs_sorted   = sorted(cv_df["cutoff"].unique())
cutoff_to_fold   = {c: i + 1 for i, c in enumerate(cutoffs_sorted)}
cv_df["fold"]    = cv_df["cutoff"].map(cutoff_to_fold)

# ── шаг h внутри фолда (1..FORECAST_STEPS) ──
cv_df = cv_df.sort_values(["fold", "unique_id", "ds"]).reset_index(drop=True)
cv_df["h"] = cv_df.groupby(["fold", "unique_id"]).cumcount() + 1

# ── raw_rows (идентична структура с Chronos) ──
raw_rows = []
for _, row in cv_df.iterrows():
    y_true = float(row["y"])
    y_pred = float(np.clip(row[pred_col], 0, None))
    ae     = abs(y_pred - y_true)
    ape    = ae / (y_true + 1e-9)
    err    = y_pred - y_true
    raw_rows.append({
        "fold":      int(row["fold"]),
        "route_id":  row["unique_id"],
        "timestamp": row["ds"],
        "h":         int(row["h"]),
        "y_true":    y_true,
        "y_pred":    y_pred,
        "ae":        ae,
        "ape":       ape,
        "err":       err,
    })

# ── per-fold console output (как у Chronos) ──
for fold in range(1, N_FOLDS + 1):
    fdf        = cv_df[cv_df["fold"] == fold]
    n_routes   = fdf["unique_id"].nunique()
    val_end    = TOTAL_VAL_PTS - (fold - 1) * FORECAST_STEPS
    val_start  = val_end - FORECAST_STEPS
    yt = fdf["y"].values
    yp = np.clip(fdf[pred_col].values, 0, None)
    tot, wape, rb = wape_rbias(yt, yp)
    print(f"\nFold {fold}/{N_FOLDS}  "
          f"(val: -{val_end}..{'-'+str(val_start) if val_start else 'end'})  "
          f"routes={n_routes}")
    print(f"  RAW   WAPE={wape:.4f}  |RBias|={rb:.4f}  Total={tot:.4f}")


# ============================================================
# АГРЕГАТЫ (идентично Chronos для блендинга)
# ============================================================
raw_cv_df = pd.DataFrame(raw_rows)

# ── По фолду ──
agg_fold = raw_cv_df.groupby("fold").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
    "n":     len(df),
})).reset_index()

# ── По шагу h ──
agg_h = raw_cv_df.groupby("h").apply(lambda df: pd.Series({
    "wape":  np.abs(df["err"]).sum() / (df["y_true"].sum() + 1e-9),
    "rbias": abs(df["y_pred"].sum() / (df["y_true"].sum() + 1e-9) - 1),
    "mae":   df["ae"].mean(),
})).reset_index()

# ── По маршруту ──
agg_route = raw_cv_df.groupby("route_id").agg(
    n_points   = ("y_true",  "count"),
    mae        = ("ae",      "mean"),
    median_ae  = ("ae",      "median"),
    std_ae     = ("ae",      "std"),
    mape       = ("ape",     "mean"),
    median_ape = ("ape",     "median"),
    mean_err   = ("err",     "mean"),
    median_err = ("err",     "median"),
    mean_true  = ("y_true",  "mean"),
    mean_pred  = ("y_pred",  "mean"),
).reset_index()
agg_route["calib_scale"] = (
    raw_cv_df.groupby("route_id")["y_true"].sum().values /
    (raw_cv_df.groupby("route_id")["y_pred"].sum().values + 1e-9)
)
agg_route["wape"] = (
    raw_cv_df.groupby("route_id")["ae"].sum().values /
    (raw_cv_df.groupby("route_id")["y_true"].sum().values + 1e-9)
)

# ── Глобальные метрики ──
y_true_all = raw_cv_df["y_true"].values
y_pred_all = raw_cv_df["y_pred"].values
total_raw, wape_raw, rbias_raw = wape_rbias(y_true_all, y_pred_all)

calib_scale = float(y_true_all.sum() / (y_pred_all.sum() + 1e-9))
y_pred_cal  = np.clip(y_pred_all * calib_scale, 0, None)
total_cal, wape_cal, rbias_cal = wape_rbias(y_true_all, y_pred_cal)

print("\n" + "=" * 65)
print("SUMMARY")
print("=" * 65)
print("\n── По фолдам ──")
print(agg_fold.to_string(index=False))
print("\n── По шагам h ──")
print(agg_h.to_string(index=False))
print(f"\n{'Global RAW':25s}  WAPE={wape_raw:.4f}  |RBias|={rbias_raw:.4f}  Total={total_raw:.4f}")
print(f"{'Global CALIBRATED':25s}  WAPE={wape_cal:.4f}  |RBias|={rbias_cal:.4f}  Total={total_cal:.4f}")
print(f"  calib_scale = {calib_scale:.4f}")
print(f"\nCalibration {'HELPS ✅' if total_cal < total_raw else 'HURTS ❌'} "
      f"(Δ = {total_cal - total_raw:+.4f})")
print("\n── Топ-10 сложных маршрутов (по median_ape) ──")
print(agg_route.sort_values("median_ape", ascending=False)
      [["route_id", "mae", "median_ae", "mape", "median_ape", "mean_err", "calib_scale"]]
      .head(10).to_string(index=False))

raw_cv_df.to_csv("nhits_cv_raw.csv", index=False)
agg_route.to_csv("nhits_cv_agg_route.csv", index=False)
agg_fold.to_csv("nhits_cv_agg_fold.csv", index=False)
agg_h.to_csv("nhits_cv_agg_h.csv", index=False)
print("\n✅ nhits_cv_raw.csv")
print("✅ nhits_cv_agg_route.csv")
print("✅ nhits_cv_agg_fold.csv")
print("✅ nhits_cv_agg_h.csv")


# ============================================================
# TEST PREDICTION
# Финальный фит на полных данных, predict() берёт последние
# input_size точек из тренировочного датасета автоматически
# ============================================================
print("\n" + "=" * 65)
print("TEST PREDICTION")
print("=" * 65)

train_nf_full = to_nf_df(train_df, min_len=INPUT_SIZE + 1, context_len=CONTEXT_LEN)
test_route_ids = test_df["route_id"].unique().tolist()
print(f"Routes: {len(test_route_ids)}  Fitting on full train data...")

nf_full = NeuralForecast(models=[make_nhits()], freq=FREQ)
nf_full.fit(train_nf_full)

print("Running inference...")
futr_df      = make_futr_df(test_route_ids, train_df)
test_pred_df = nf_full.predict(futr_df=futr_df)   # next FORECAST_STEPS × 30min на каждый маршрут
test_pred_col = get_pred_col(test_pred_df)
test_preds    = build_preds_dict(test_pred_df, test_pred_col)

predictions_raw = {}
for route_id in tqdm(test_route_ids, desc="Build submission"):
    route_test = test_df[test_df["route_id"] == route_id].sort_values("timestamp")
    preds = test_preds.get(route_id, np.zeros(FORECAST_STEPS))
    for j, (_, row) in enumerate(route_test.iterrows()):
        pred = float(preds[j]) if j < len(preds) else float(preds[-1])
        predictions_raw[row["id"]] = max(0.0, pred)

submission_raw = (
    pd.DataFrame(list(predictions_raw.items()), columns=["id", "y_pred"])
    .sort_values("id").reset_index(drop=True)
)
submission_cal = submission_raw.copy()
submission_cal["y_pred"] = np.clip(submission_cal["y_pred"] * calib_scale, 0, None)

submission_raw.to_csv("submission_nhits_raw.csv", index=False)
submission_cal.to_csv("submission_nhits_calibrated.csv", index=False)
print(f"\n✅ submission_nhits_raw.csv")
print(f"✅ submission_nhits_calibrated.csv")
print(f"\n── Raw ──\n{submission_raw['y_pred'].describe().round(2)}")
print(f"\n── Calibrated (scale={calib_scale:.4f}) ──\n{submission_cal['y_pred'].describe().round(2)}")

In [ ]:
# ============================================================
# N-HiTS — Grid Search по n_pool_kernel_size / n_freq_downsample
# 6 конфигов × 5 фолдов × CV_STEPS (400) → выбор лучшего →
# финальный фит лучшего (FINAL_STEPS=800) → predict()
# ============================================================

import warnings, os
import numpy as np
import pandas as pd
import torch
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MQLoss
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

# ── Метрика ──
def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias

# ── Config ──
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8
N_FOLDS        = 5
TOTAL_VAL_PTS  = N_FOLDS * FORECAST_STEPS   # 40
CONTEXT_LEN    = 2048
FREQ           = "30min"
INPUT_SIZE     = 336
BATCH_SIZE     = 64

CV_STEPS    = 400   # ускоренный прогон для отбора
FINAL_STEPS = 800   # финальный фит победителя

# ── Device ──
if torch.backends.mps.is_available():
    ACCELERATOR = "mps"
elif torch.cuda.is_available():
    ACCELERATOR = "gpu"
else:
    ACCELERATOR = "cpu"
print(f"Device: {ACCELERATOR}")

# ── Data ──
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
status_cols = sorted([c for c in train_df.columns if c in ["status_3", "status_2", "status_5"]])

FUTR_EXOG       = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "is_weekend"]
HIST_EXOG_EXTRA = ["lag_48", "lag_336"]
HIST_EXOG       = (status_cols + HIST_EXOG_EXTRA) or None

# ── Feature engineering ──
def add_calendar_features(df, ts_col="ds"):
    ts = pd.to_datetime(df[ts_col])
    df = df.copy()
    df["hour_sin"]   = np.sin(2 * np.pi * ts.dt.hour / 24)
    df["hour_cos"]   = np.cos(2 * np.pi * ts.dt.hour / 24)
    df["dow_sin"]    = np.sin(2 * np.pi * ts.dt.dayofweek / 7)
    df["dow_cos"]    = np.cos(2 * np.pi * ts.dt.dayofweek / 7)
    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(float)
    return df

def add_lag_features(nf_df):
    nf_df = nf_df.sort_values(["unique_id", "ds"]).copy()
    nf_df["lag_48"]  = nf_df.groupby("unique_id")["y"].transform(lambda s: s.shift(48).bfill().fillna(0))
    nf_df["lag_336"] = nf_df.groupby("unique_id")["y"].transform(lambda s: s.shift(336).bfill().fillna(0))
    return nf_df

def to_nf_df(df, min_len=32, context_len=CONTEXT_LEN):
    rows = []
    for route_id, grp in df.groupby("route_id"):
        grp = (
            grp.sort_values("timestamp")
            .set_index("timestamp")[[TARGET_COL] + status_cols]
            .asfreq(FREQ).interpolate(method="time")
            .bfill().ffill().tail(context_len).reset_index()
        )
        if len(grp) < min_len:
            continue
        grp.insert(0, "unique_id", route_id)
        grp = grp.rename(columns={"timestamp": "ds", TARGET_COL: "y"})
        rows.append(grp)
    nf_df = pd.concat(rows, ignore_index=True)
    nf_df = add_calendar_features(nf_df)
    nf_df = add_lag_features(nf_df)
    return nf_df

def make_futr_df(route_ids, train_df):
    rows = []
    for route_id in route_ids:
        last_ts = train_df[train_df["route_id"] == route_id]["timestamp"].max()
        future_ts = pd.date_range(start=last_ts + pd.Timedelta("30min"),
                                  periods=FORECAST_STEPS, freq=FREQ)
        for ts in future_ts:
            rows.append({"unique_id": route_id, "ds": ts})
    return add_calendar_features(pd.DataFrame(rows))

def get_pred_col(pred_df):
    for c in ["NHITS-median", "NHITS-q-0.50", "NHITS"]:
        if c in pred_df.columns:
            return c
    skip = {"unique_id", "ds", "y", "cutoff"}
    num_cols = [c for c in pred_df.columns if c not in skip
                and pd.api.types.is_numeric_dtype(pred_df[c])]
    median_cols = [c for c in num_cols if any(x in c for x in ["median","0.5","50"])]
    return (median_cols or num_cols)[0]

def build_preds_dict(pred_df, pred_col):
    return {
        rid: np.clip(grp.sort_values("ds")[pred_col].values, 0, None)
        for rid, grp in pred_df.groupby("unique_id")
    }

# ── Фабрика модели (принимает dict конфига) ──
def make_nhits_from_cfg(cfg: dict, max_steps: int) -> NHITS:
    return NHITS(
        h                   = FORECAST_STEPS,
        input_size          = INPUT_SIZE,
        loss                = MQLoss(level=[80]),
        n_freq_downsample   = cfg["n_freq_downsample"],
        n_pool_kernel_size  = cfg["n_pool_kernel_size"],
        mlp_units           = [[512, 512]] * 3,
        n_blocks            = [4, 4, 4],
        max_steps           = max_steps,
        batch_size          = BATCH_SIZE,
        accelerator         = ACCELERATOR,
        hist_exog_list      = HIST_EXOG,
        futr_exog_list      = FUTR_EXOG,
        scaler_type         = "robust",
        enable_progress_bar = True,
    )

# ── 6 конфигов ──
CONFIGS = [
    {"name": "baseline",     "n_freq_downsample": [48, 8, 1], "n_pool_kernel_size": [4, 2, 1]},
    {"name": "soft_pool",    "n_freq_downsample": [48, 8, 1], "n_pool_kernel_size": [2, 2, 1]},
    {"name": "no_pool",      "n_freq_downsample": [48, 8, 1], "n_pool_kernel_size": [1, 1, 1]},
    {"name": "4h_aligned",   "n_freq_downsample": [48, 8, 1], "n_pool_kernel_size": [8, 2, 1]},
    {"name": "halfday_freq", "n_freq_downsample": [24, 4, 1], "n_pool_kernel_size": [2, 2, 1]},
    {"name": "short_focus",  "n_freq_downsample": [8,  2, 1], "n_pool_kernel_size": [2, 1, 1]},
]

# ── Данные для CV ──
train_nf = to_nf_df(
    train_df,
    min_len     = TOTAL_VAL_PTS + INPUT_SIZE + 1,
    context_len = CONTEXT_LEN,
)
print(f"NF train: {train_nf.shape}  routes: {train_nf['unique_id'].nunique()}\n")

# ============================================================
# GRID SEARCH LOOP
# ============================================================
grid_results = []

for cfg in CONFIGS:
    print(f"\n{'='*65}")
    print(f"CONFIG: {cfg['name']}  "
          f"freq={cfg['n_freq_downsample']}  pool={cfg['n_pool_kernel_size']}")
    print(f"{'='*65}")

    nf_cv = NeuralForecast(models=[make_nhits_from_cfg(cfg, CV_STEPS)], freq=FREQ)
    cv_df = nf_cv.cross_validation(
        df        = train_nf,
        n_windows = N_FOLDS,
        step_size = FORECAST_STEPS,
        refit     = True,
    )

    pred_col = get_pred_col(cv_df)

    # fold mapping
    cutoffs_sorted = sorted(cv_df["cutoff"].unique())
    cutoff_to_fold = {c: i + 1 for i, c in enumerate(cutoffs_sorted)}
    cv_df["fold"]  = cv_df["cutoff"].map(cutoff_to_fold)
    cv_df = cv_df.sort_values(["fold", "unique_id", "ds"]).reset_index(drop=True)
    cv_df["h"] = cv_df.groupby(["fold", "unique_id"]).cumcount() + 1

    # per-fold print
    for fold in range(1, N_FOLDS + 1):
        fdf = cv_df[cv_df["fold"] == fold]
        yt  = fdf["y"].values
        yp  = np.clip(fdf[pred_col].values, 0, None)
        tot, wape, rb = wape_rbias(yt, yp)
        print(f"  Fold {fold}  WAPE={wape:.4f}  |RBias|={rb:.4f}  Total={tot:.4f}")

    # global
    yt_all = cv_df["y"].values
    yp_all = np.clip(cv_df[pred_col].values, 0, None)
    tot_g, wape_g, rb_g = wape_rbias(yt_all, yp_all)
    calib  = float(yt_all.sum() / (yp_all.sum() + 1e-9))
    yp_cal = np.clip(yp_all * calib, 0, None)
    tot_c, wape_c, rb_c = wape_rbias(yt_all, yp_cal)

    print(f"\n  ── Global RAW       WAPE={wape_g:.4f}  |RBias|={rb_g:.4f}  Total={tot_g:.4f}")
    print(f"  ── Global CALIB     WAPE={wape_c:.4f}  |RBias|={rb_c:.4f}  Total={tot_c:.4f}  scale={calib:.4f}")

    grid_results.append({
        "name":        cfg["name"],
        "freq":        str(cfg["n_freq_downsample"]),
        "pool":        str(cfg["n_pool_kernel_size"]),
        "wape_raw":    round(wape_g, 4),
        "rbias_raw":   round(rb_g,   4),
        "total_raw":   round(tot_g,  4),
        "wape_cal":    round(wape_c, 4),
        "rbias_cal":   round(rb_c,   4),
        "total_cal":   round(tot_c,  4),
        "calib_scale": round(calib,  4),
    })

# ============================================================
# СВОДНАЯ ТАБЛИЦА
# ============================================================
results_df = pd.DataFrame(grid_results).sort_values("total_raw")
results_df.to_csv("nhits_grid_results.csv", index=False)

print("\n\n" + "=" * 65)
print("GRID SEARCH RESULTS (sorted by total_raw)")
print("=" * 65)
print(results_df.to_string(index=False))

best_name = results_df.iloc[0]["name"]
best_cfg  = next(c for c in CONFIGS if c["name"] == best_name)
print(f"\n🏆 Best config: {best_name}  "
      f"freq={best_cfg['n_freq_downsample']}  pool={best_cfg['n_pool_kernel_size']}")
print(f"   total_raw={results_df.iloc[0]['total_raw']:.4f}  "
      f"wape={results_df.iloc[0]['wape_raw']:.4f}  "
      f"rbias={results_df.iloc[0]['rbias_raw']:.4f}")

# ============================================================
# ФИНАЛЬНЫЙ ФИТ ЛУЧШЕГО КОНФИГА (FINAL_STEPS=800)
# ============================================================
print(f"\n{'='*65}")
print(f"FINAL FIT: {best_name}  (max_steps={FINAL_STEPS})")
print("=" * 65)

train_nf_full = to_nf_df(train_df, min_len=INPUT_SIZE + 1, context_len=CONTEXT_LEN)
test_route_ids = test_df["route_id"].unique().tolist()
print(f"Routes: {len(test_route_ids)}")

nf_full = NeuralForecast(models=[make_nhits_from_cfg(best_cfg, FINAL_STEPS)], freq=FREQ)
nf_full.fit(train_nf_full)

futr_df      = make_futr_df(test_route_ids, train_df)
test_pred_df = nf_full.predict(futr_df=futr_df)
test_pred_col = get_pred_col(test_pred_df)
test_preds    = build_preds_dict(test_pred_df, test_pred_col)

# calib_scale из CV лучшего конфига
best_row   = results_df.iloc[0]
calib_scale = float(best_row["calib_scale"])

predictions_raw = {}
for route_id in tqdm(test_route_ids, desc="Build submission"):
    route_test = test_df[test_df["route_id"] == route_id].sort_values("timestamp")
    preds = test_preds.get(route_id, np.zeros(FORECAST_STEPS))
    for j, (_, row) in enumerate(route_test.iterrows()):
        pred = float(preds[j]) if j < len(preds) else float(preds[-1])
        predictions_raw[row["id"]] = max(0.0, pred)

submission_raw = (
    pd.DataFrame(list(predictions_raw.items()), columns=["id", "y_pred"])
    .sort_values("id").reset_index(drop=True)
)
submission_cal = submission_raw.copy()
submission_cal["y_pred"] = np.clip(submission_cal["y_pred"] * calib_scale, 0, None)

submission_raw.to_csv(f"submission_nhits_{best_name}_raw.csv",        index=False)
submission_cal.to_csv(f"submission_nhits_{best_name}_calibrated.csv", index=False)

print(f"\n✅ nhits_grid_results.csv")
print(f"✅ submission_nhits_{best_name}_raw.csv")
print(f"✅ submission_nhits_{best_name}_calibrated.csv")
print(f"\n── Raw ──\n{submission_raw['y_pred'].describe().round(2)}")
print(f"\n── Calibrated (scale={calib_scale:.4f}) ──\n{submission_cal['y_pred'].describe().round(2)}")

 {"phase": 1, "name": "p1_freq_4_2_1",
     "n_freq_downsample": [4,2,1], "n_pool_kernel_size": [1,1,1], "n_blocks": [4,4,4],
     "note": "Честная иерархия для h=8: 2/4/8 контрол. точек"}
     
Fold 1  WAPE=0.3265  |RBias|=0.0939  Total=0.4204
Fold 2  WAPE=0.3387  |RBias|=0.0164  Total=0.3551
Fold 3  WAPE=0.3278  |RBias|=0.0024  Total=0.3301
Fold 4  WAPE=0.3235  |RBias|=0.0119  Total=0.3354
Fold 5  WAPE=0.3338  |RBias|=0.0290  Total=0.3628


── RAW    WAPE=0.3299  |RBias|=0.0316  Total=0.3616
── CALIB  WAPE=0.3314  |RBias|=0.0000  Total=0.3314  scale=1.0327

In [ ]:
# ============================================================
# N-HiTS — 10 систематических экспериментов
# Phase 1 (1-4): n_freq_downsample при pool=[1,1,1], blocks=[4,4,4]
# Phase 2 (5-8): n_blocks при pool=[1,1,1], freq=[48,8,1]  (изоляция!)
# Phase 3 (9-10): лучший freq × лучший blocks
#
# Каждый эксперимент: CV_STEPS=400
# Победитель в конце: полная CV на FINAL_STEPS=800 → predict()
# ============================================================

import warnings, gc
import numpy as np
import pandas as pd
import torch
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MQLoss
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

# ── Метрика ──
def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias

# ── Config ──
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8
N_FOLDS        = 5
TOTAL_VAL_PTS  = N_FOLDS * FORECAST_STEPS
CONTEXT_LEN    = 2048
FREQ           = "30min"
INPUT_SIZE     = 336
BATCH_SIZE     = 64

CV_STEPS    = 400   # все 10 экспериментов — быстрый прогон
FINAL_STEPS = 800   # победитель переобучается здесь для честного CV + predict

# Предыдущий лучший результат из grid search — точка отсчёта
PREV_BEST_TOTAL = 0.3563  # ← замени на число из прошлого прогона, напр. 0.3142

if torch.backends.mps.is_available():
    ACCELERATOR = "mps"
elif torch.cuda.is_available():
    ACCELERATOR = "gpu"
else:
    ACCELERATOR = "cpu"
print(f"Device: {ACCELERATOR}")

# ── Data ──
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id","timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id","timestamp"]).reset_index(drop=True)
status_cols = sorted([c for c in train_df.columns if c in ["status_2","status_3","status_5"]])

FUTR_EXOG = ["hour_sin","hour_cos","dow_sin","dow_cos","is_weekend"]
HIST_EXOG = (status_cols + ["lag_48","lag_336"]) or None

def add_calendar_features(df, ts_col="ds"):
    ts = pd.to_datetime(df[ts_col])
    df = df.copy()
    df["hour_sin"]   = np.sin(2*np.pi*ts.dt.hour/24)
    df["hour_cos"]   = np.cos(2*np.pi*ts.dt.hour/24)
    df["dow_sin"]    = np.sin(2*np.pi*ts.dt.dayofweek/7)
    df["dow_cos"]    = np.cos(2*np.pi*ts.dt.dayofweek/7)
    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(float)
    return df

def add_lag_features(nf_df):
    nf_df = nf_df.sort_values(["unique_id","ds"]).copy()
    for lag in [48, 336]:
        nf_df[f"lag_{lag}"] = (
            nf_df.groupby("unique_id")["y"]
            .transform(lambda s, l=lag: s.shift(l).bfill().fillna(0))
        )
    return nf_df

def to_nf_df(df, min_len=32, context_len=CONTEXT_LEN):
    rows = []
    for route_id, grp in df.groupby("route_id"):
        grp = (
            grp.sort_values("timestamp")
            .set_index("timestamp")[[TARGET_COL]+status_cols]
            .asfreq(FREQ).interpolate(method="time")
            .bfill().ffill().tail(context_len).reset_index()
        )
        if len(grp) < min_len:
            continue
        grp.insert(0, "unique_id", route_id)
        grp = grp.rename(columns={"timestamp":"ds", TARGET_COL:"y"})
        rows.append(grp)
    nf_df = pd.concat(rows, ignore_index=True)
    nf_df = add_calendar_features(nf_df)
    nf_df = add_lag_features(nf_df)
    return nf_df

def make_futr_df(route_ids, train_df):
    rows = []
    for rid in route_ids:
        last_ts = train_df[train_df["route_id"]==rid]["timestamp"].max()
        for ts in pd.date_range(start=last_ts+pd.Timedelta("30min"),
                                periods=FORECAST_STEPS, freq=FREQ):
            rows.append({"unique_id": rid, "ds": ts})
    return add_calendar_features(pd.DataFrame(rows))

def get_pred_col(pred_df):
    for c in ["NHITS-median","NHITS-q-0.50","NHITS"]:
        if c in pred_df.columns:
            return c
    skip = {"unique_id","ds","y","cutoff"}
    num = [c for c in pred_df.columns if c not in skip
           and pd.api.types.is_numeric_dtype(pred_df[c])]
    med = [c for c in num if any(x in c for x in ["median","0.5","50"])]
    return (med or num)[0]

def build_preds_dict(pred_df, pred_col):
    return {rid: np.clip(g.sort_values("ds")[pred_col].values, 0, None)
            for rid, g in pred_df.groupby("unique_id")}

def make_nhits(cfg, max_steps):
    return NHITS(
        h                  = FORECAST_STEPS,
        input_size         = INPUT_SIZE,
        loss               = MQLoss(level=[80]),
        n_freq_downsample  = cfg["n_freq_downsample"],
        n_pool_kernel_size = cfg["n_pool_kernel_size"],
        n_blocks           = cfg["n_blocks"],
        mlp_units          = [[512, 512]] * len(cfg["n_blocks"]),
        max_steps          = max_steps,
        batch_size         = BATCH_SIZE,
        accelerator        = ACCELERATOR,
        hist_exog_list     = HIST_EXOG,
        futr_exog_list     = FUTR_EXOG,
        scaler_type        = "robust",
        enable_progress_bar= True,
    )

def run_cv(cfg, train_nf, max_steps):
    """Запускает CV и возвращает dict с метриками."""
    nf = NeuralForecast(models=[make_nhits(cfg, max_steps)], freq=FREQ)
    cv_df = nf.cross_validation(
        df=train_nf, n_windows=N_FOLDS, step_size=FORECAST_STEPS, refit=True
    )
    pred_col = get_pred_col(cv_df)

    # fold mapping
    cuts = sorted(cv_df["cutoff"].unique())
    cv_df["fold"] = cv_df["cutoff"].map({c: i+1 for i, c in enumerate(cuts)})
    cv_df = cv_df.sort_values(["fold","unique_id","ds"]).reset_index(drop=True)
    cv_df["h"] = cv_df.groupby(["fold","unique_id"]).cumcount() + 1

    yt = cv_df["y"].values
    yp = np.clip(cv_df[pred_col].values, 0, None)
    tot, wape, rb = wape_rbias(yt, yp)
    calib = float(yt.sum() / (yp.sum() + 1e-9))
    yp_c  = np.clip(yp * calib, 0, None)
    tot_c, wape_c, rb_c = wape_rbias(yt, yp_c)

    # per-fold
    fold_rows = []
    for fold in range(1, N_FOLDS+1):
        fdf = cv_df[cv_df["fold"]==fold]
        t, w, r = wape_rbias(fdf["y"].values,
                              np.clip(fdf[pred_col].values, 0, None))
        fold_rows.append({"fold": fold, "wape": round(w,4),
                          "rbias": round(r,4), "total": round(t,4)})

    del nf, cv_df
    gc.collect()

    return {
        "wape_raw":    round(wape,  4),
        "rbias_raw":   round(rb,    4),
        "total_raw":   round(tot,   4),
        "wape_cal":    round(wape_c,4),
        "rbias_cal":   round(rb_c,  4),
        "total_cal":   round(tot_c, 4),
        "calib_scale": round(calib, 4),
        "fold_stats":  fold_rows,
    }

# ──────────────────────────────────────────────────────────────
# 10 ЭКСПЕРИМЕНТОВ
# ──────────────────────────────────────────────────────────────
# Phase 3 пре-планирована как freq_4_2_1 × best_blocks.
# После прогона фаз 1-2 можно скорректировать вручную.

EXPERIMENTS = [
    # ── Phase 1: freq (pool=[1,1,1], blocks=[4,4,4]) ──────────
    # {"phase": 1, "name": "p1_freq_4_2_1",
    #  "n_freq_downsample": [4,2,1], "n_pool_kernel_size": [1,1,1], "n_blocks": [4,4,4],
    #  "note": "Честная иерархия для h=8: 2/4/8 контрол. точек"},

    {"phase": 1, "name": "p1_freq_8_4_1",
     "n_freq_downsample": [8,4,1], "n_pool_kernel_size": [1,1,1], "n_blocks": [4,4,4],
     "note": "4ч/2ч/30мин — совпадает с сильнейшей сезонностью (4h_8)"},

    {"phase": 1, "name": "p1_freq_2_1_1",
     "n_freq_downsample": [2,1,1], "n_pool_kernel_size": [1,1,1], "n_blocks": [4,4,4],
     "note": "Два стека на полном разрешении, один — полугрубый"},

    {"phase": 1, "name": "p1_freq_1_1_1",
     "n_freq_downsample": [1,1,1], "n_pool_kernel_size": [1,1,1], "n_blocks": [4,4,4],
     "note": "Нет иерархии вообще — 3 параллельных MLP на одном сигнале"},

    # ── Phase 2: n_blocks (pool=[1,1,1], freq=[48,8,1]) ───────
    # freq намеренно остаётся baseline для изоляции эффекта блоков
    {"phase": 2, "name": "p2_blk_2_4_8",
     "n_freq_downsample": [48,8,1], "n_pool_kernel_size": [1,1,1], "n_blocks": [2,4,8],
     "note": "Больше параметров в short-term стеке (8 блоков)"},

    {"phase": 2, "name": "p2_blk_6_6_6",
     "n_freq_downsample": [48,8,1], "n_pool_kernel_size": [1,1,1], "n_blocks": [6,6,6],
     "note": "Просто больше ёмкости везде (+50% блоков)"},

    {"phase": 2, "name": "p2_blk_2_2_8",
     "n_freq_downsample": [48,8,1], "n_pool_kernel_size": [1,1,1], "n_blocks": [2,2,8],
     "note": "Экстремальный акцент на коротком стеке"},

    {"phase": 2, "name": "p2_blk_8_4_2",
     "n_freq_downsample": [48,8,1], "n_pool_kernel_size": [1,1,1], "n_blocks": [8,4,2],
     "note": "Инверсия: акцент на длинном стеке — проверка обратной гипотезы"},

    # ── Phase 3: combo (best_freq × best_blocks) ──────────────
    # Пре-планировано как наиболее перспективные пары
    {"phase": 3, "name": "p3_4_2_1_blk248",
     "n_freq_downsample": [4,2,1], "n_pool_kernel_size": [1,1,1], "n_blocks": [2,4,8],
     "note": "Честная иерархия + акцент блоков на коротком стеке"},

    {"phase": 3, "name": "p3_4_2_1_blk666",
     "n_freq_downsample": [4,2,1], "n_pool_kernel_size": [1,1,1], "n_blocks": [6,6,6],
     "note": "Честная иерархия + увеличенная ёмкость везде"},
]

# ── Данные ──
train_nf = to_nf_df(
    train_df,
    min_len     = TOTAL_VAL_PTS + INPUT_SIZE + 1,
    context_len = CONTEXT_LEN,
)
print(f"NF train: {train_nf.shape}  routes: {train_nf['unique_id'].nunique()}\n")

# ──────────────────────────────────────────────────────────────
# MAIN LOOP
# ──────────────────────────────────────────────────────────────
all_results   = []
best_so_far   = {"total_raw": PREV_BEST_TOTAL or float("inf"), "name": "prev_best"}
current_phase = None

for i, cfg in enumerate(EXPERIMENTS, 1):

    # Заголовок фазы
    if cfg["phase"] != current_phase:
        current_phase = cfg["phase"]
        phase_titles = {
            1: "PHASE 1 — n_freq_downsample  (pool=[1,1,1], blocks=[4,4,4] fixed)",
            2: "PHASE 2 — n_blocks  (pool=[1,1,1], freq=[48,8,1] fixed)",
            3: "PHASE 3 — Combo: best_freq × best_blocks",
        }
        print(f"\n{'='*65}")
        print(phase_titles[current_phase])
        print("="*65)

    print(f"\n[{i:02d}/10] {cfg['name']}")
    print(f"  freq={cfg['n_freq_downsample']}  "
          f"pool={cfg['n_pool_kernel_size']}  "
          f"blocks={cfg['n_blocks']}")
    print(f"  {cfg['note']}")

    metrics = run_cv(cfg, train_nf, CV_STEPS)

    # per-fold вывод
    for fr in metrics["fold_stats"]:
        print(f"  Fold {fr['fold']}  WAPE={fr['wape']:.4f}  "
              f"|RBias|={fr['rbias']:.4f}  Total={fr['total']:.4f}")

    print(f"\n  ── RAW    WAPE={metrics['wape_raw']:.4f}  "
          f"|RBias|={metrics['rbias_raw']:.4f}  "
          f"Total={metrics['total_raw']:.4f}")
    print(f"  ── CALIB  WAPE={metrics['wape_cal']:.4f}  "
          f"|RBias|={metrics['rbias_cal']:.4f}  "
          f"Total={metrics['total_cal']:.4f}  "
          f"scale={metrics['calib_scale']:.4f}")

    row = {"#": i, "name": cfg["name"], "phase": cfg["phase"],
           "freq": str(cfg["n_freq_downsample"]),
           "blocks": str(cfg["n_blocks"]), **metrics}
    del row["fold_stats"]
    all_results.append(row)

    # Обновляем победителя
    if metrics["total_raw"] < best_so_far["total_raw"]:
        best_so_far = {"name": cfg["name"], **metrics, "cfg": cfg}
        print(f"\n  🏆 NEW BEST!  Total={metrics['total_raw']:.4f}  "
              f"(prev: {all_results[-2]['total_raw'] if len(all_results)>1 else PREV_BEST_TOTAL:.4f})")
    else:
        delta = metrics["total_raw"] - best_so_far["total_raw"]
        print(f"\n  Current best: {best_so_far['name']}  "
              f"Total={best_so_far['total_raw']:.4f}  "
              f"(this: {metrics['total_raw']:.4f}, Δ={delta:+.4f})")

# ──────────────────────────────────────────────────────────────
# СВОДНАЯ ТАБЛИЦА
# ──────────────────────────────────────────────────────────────
results_df = (
    pd.DataFrame(all_results)
    .sort_values("total_raw")
    [["#","name","phase","freq","blocks","wape_raw","rbias_raw","total_raw","total_cal","calib_scale"]]
)
results_df.to_csv("nhits_exp10_results.csv", index=False)

print("\n\n" + "="*65)
print("LEADERBOARD — все 10 экспериментов (CV_STEPS=400)")
print("="*65)
print(results_df.to_string(index=False))

best_cfg = best_so_far["cfg"]
print(f"\n🏆 WINNER:  {best_so_far['name']}")
print(f"   freq={best_cfg['n_freq_downsample']}  "
      f"pool={best_cfg['n_pool_kernel_size']}  "
      f"blocks={best_cfg['n_blocks']}")
print(f"   total_raw={best_so_far['total_raw']:.4f}")

# ──────────────────────────────────────────────────────────────
# WINNER — ПОЛНАЯ CV (FINAL_STEPS=800)
# Эти метрики сравнимы с прошлым grid search (тоже FINAL_STEPS)
# ──────────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print(f"WINNER FULL CV  (FINAL_STEPS={FINAL_STEPS})")
print(f"Конфиг: {best_so_far['name']}")
print("="*65)

winner_metrics = run_cv(best_cfg, train_nf, FINAL_STEPS)

print(f"\nPer-fold:")
for fr in winner_metrics["fold_stats"]:
    print(f"  Fold {fr['fold']}  WAPE={fr['wape']:.4f}  "
          f"|RBias|={fr['rbias']:.4f}  Total={fr['total']:.4f}")

print(f"\n── RAW    WAPE={winner_metrics['wape_raw']:.4f}  "
      f"|RBias|={winner_metrics['rbias_raw']:.4f}  "
      f"Total={winner_metrics['total_raw']:.4f}")
print(f"── CALIB  WAPE={winner_metrics['wape_cal']:.4f}  "
      f"|RBias|={winner_metrics['rbias_cal']:.4f}  "
      f"Total={winner_metrics['total_cal']:.4f}  "
      f"scale={winner_metrics['calib_scale']:.4f}")

calib_scale = winner_metrics["calib_scale"]

# Сохраняем финальные метрики победителя
pd.DataFrame([{
    "name": best_so_far["name"],
    "steps": FINAL_STEPS,
    **{k: v for k, v in winner_metrics.items() if k != "fold_stats"},
}]).to_csv("nhits_winner_full_cv.csv", index=False)

# ──────────────────────────────────────────────────────────────
# ФИНАЛЬНЫЙ PREDICT
# ──────────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print(f"FINAL PREDICT  (FINAL_STEPS={FINAL_STEPS})")
print("="*65)

train_nf_full  = to_nf_df(train_df, min_len=INPUT_SIZE+1, context_len=CONTEXT_LEN)
test_route_ids = test_df["route_id"].unique().tolist()
print(f"Routes: {len(test_route_ids)}  Fitting on full train data...")

nf_full = NeuralForecast(models=[make_nhits(best_cfg, FINAL_STEPS)], freq=FREQ)
nf_full.fit(train_nf_full)

futr_df      = make_futr_df(test_route_ids, train_df)
test_pred_df = nf_full.predict(futr_df=futr_df)
test_pred_col = get_pred_col(test_pred_df)
test_preds    = build_preds_dict(test_pred_df, test_pred_col)

predictions_raw = {}
for route_id in tqdm(test_route_ids, desc="Build submission"):
    route_test = test_df[test_df["route_id"]==route_id].sort_values("timestamp")
    preds = test_preds.get(route_id, np.zeros(FORECAST_STEPS))
    for j, (_, row) in enumerate(route_test.iterrows()):
        pred = float(preds[j]) if j < len(preds) else float(preds[-1])
        predictions_raw[row["id"]] = max(0.0, pred)

name = best_so_far["name"]
submission_raw = (
    pd.DataFrame(list(predictions_raw.items()), columns=["id","y_pred"])
    .sort_values("id").reset_index(drop=True)
)
submission_cal = submission_raw.copy()
submission_cal["y_pred"] = np.clip(submission_cal["y_pred"] * calib_scale, 0, None)

submission_raw.to_csv(f"submission_{name}_raw.csv",        index=False)
submission_cal.to_csv(f"submission_{name}_calibrated.csv", index=False)

print(f"\n✅ nhits_exp10_results.csv")
print(f"✅ nhits_winner_full_cv.csv")
print(f"✅ submission_{name}_raw.csv")
print(f"✅ submission_{name}_calibrated.csv")
print(f"\n── Raw ──\n{submission_raw['y_pred'].describe().round(2)}")
print(f"\n── Calibrated (scale={calib_scale:.4f}) ──\n{submission_cal['y_pred'].describe().round(2)}")

In [ ]:
# ============================================================
# N-HiTS — Feature & Architecture experiments (10 runs)
# Все фичи предвычисляются один раз, потом subset по конфигу
# Best config: n_freq=[48,8,1], pool=[1,1,1], blocks=[8,4,2]
# ============================================================

import warnings, gc
import numpy as np
import pandas as pd
import torch
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MQLoss
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias

# ── Config ──
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8
N_FOLDS        = 5
TOTAL_VAL_PTS  = N_FOLDS * FORECAST_STEPS
CONTEXT_LEN    = 2048
FREQ           = "30min"
INPUT_SIZE     = 336
BATCH_SIZE     = 64
CV_STEPS       = 400
FINAL_STEPS    = 800

PREV_BEST_TOTAL = 0.3491  # ← best total_raw из прошлого прогона

if torch.backends.mps.is_available():   ACCELERATOR = "mps"
elif torch.cuda.is_available():         ACCELERATOR = "gpu"
else:                                   ACCELERATOR = "cpu"
print(f"Device: {ACCELERATOR}")

# ── Data ──
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id","timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id","timestamp"]).reset_index(drop=True)

# Все доступные статус-колонки
ALL_STATUS = sorted([c for c in train_df.columns
                     if c in ["status_1","status_2","status_3","status_5"]])
ALL_LAGS   = [1, 2, 4, 48, 336]
FUTR_EXOG  = ["hour_sin","hour_cos","dow_sin","dow_cos","is_weekend"]

# Baseline фичи (лучший конфиг из прошлого прогона)
BASE_HIST  = ["status_2","status_3","status_5","lag_48","lag_336"]

def add_calendar_features(df, ts_col="ds"):
    ts = pd.to_datetime(df[ts_col])
    df = df.copy()
    df["hour_sin"]   = np.sin(2*np.pi*ts.dt.hour/24)
    df["hour_cos"]   = np.cos(2*np.pi*ts.dt.hour/24)
    df["dow_sin"]    = np.sin(2*np.pi*ts.dt.dayofweek/7)
    df["dow_cos"]    = np.cos(2*np.pi*ts.dt.dayofweek/7)
    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(float)
    return df

def add_lag_features(nf_df, lags):
    nf_df = nf_df.sort_values(["unique_id","ds"]).copy()
    for lag in lags:
        col = f"lag_{lag}"
        if col not in nf_df.columns:
            nf_df[col] = (
                nf_df.groupby("unique_id")["y"]
                .transform(lambda s, l=lag: s.shift(l).bfill().fillna(0))
            )
    return nf_df

def to_nf_df_full(df, min_len=32, context_len=CONTEXT_LEN):
    """Строит NF датафрейм со ВСЕМИ возможными фичами — один раз на весь прогон."""
    rows = []
    for route_id, grp in df.groupby("route_id"):
        grp = (
            grp.sort_values("timestamp")
            .set_index("timestamp")[[TARGET_COL] + ALL_STATUS]
            .asfreq(FREQ).interpolate(method="time")
            .bfill().ffill().tail(context_len).reset_index()
        )
        if len(grp) < min_len:
            continue
        grp.insert(0, "unique_id", route_id)
        grp = grp.rename(columns={"timestamp":"ds", TARGET_COL:"y"})
        rows.append(grp)
    nf_df = pd.concat(rows, ignore_index=True)
    nf_df = add_calendar_features(nf_df)
    nf_df = add_lag_features(nf_df, ALL_LAGS)
    return nf_df

def subset_nf_df(nf_df_full, hist_exog):
    """Выбирает нужные колонки — без пересчёта."""
    keep = ["unique_id","ds","y"] + FUTR_EXOG + hist_exog
    return nf_df_full[keep].copy()

def make_futr_df(route_ids, train_df):
    rows = []
    for rid in route_ids:
        last_ts = train_df[train_df["route_id"]==rid]["timestamp"].max()
        for ts in pd.date_range(start=last_ts+pd.Timedelta("30min"),
                                periods=FORECAST_STEPS, freq=FREQ):
            rows.append({"unique_id": rid, "ds": ts})
    return add_calendar_features(pd.DataFrame(rows))

def get_pred_col(pred_df):
    for c in ["NHITS-median","NHITS-q-0.50","NHITS"]:
        if c in pred_df.columns: return c
    skip = {"unique_id","ds","y","cutoff"}
    num  = [c for c in pred_df.columns if c not in skip
            and pd.api.types.is_numeric_dtype(pred_df[c])]
    med  = [c for c in num if any(x in c for x in ["median","0.5","50"])]
    return (med or num)[0]

def build_preds_dict(pred_df, pred_col):
    return {rid: np.clip(g.sort_values("ds")[pred_col].values, 0, None)
            for rid, g in pred_df.groupby("unique_id")}

def make_nhits(cfg, max_steps):
    n_stacks = len(cfg["n_blocks"])
    return NHITS(
        h                  = FORECAST_STEPS,
        input_size         = INPUT_SIZE,
        loss               = MQLoss(level=[80]),
        n_freq_downsample  = cfg["n_freq_downsample"],
        n_pool_kernel_size = cfg["n_pool_kernel_size"],
        n_blocks           = cfg["n_blocks"],
        mlp_units          = cfg.get("mlp_units", [[512,512]] * n_stacks),
        max_steps          = max_steps,
        batch_size         = BATCH_SIZE,
        accelerator        = ACCELERATOR,
        hist_exog_list     = cfg["hist_exog"],
        futr_exog_list     = FUTR_EXOG,
        scaler_type        = "robust",
        enable_progress_bar= True,
    )

def run_cv(cfg, train_nf_full, max_steps):
    nf_df = subset_nf_df(train_nf_full, cfg["hist_exog"])
    nf    = NeuralForecast(models=[make_nhits(cfg, max_steps)], freq=FREQ)
    cv_df = nf.cross_validation(
        df=nf_df, n_windows=N_FOLDS, step_size=FORECAST_STEPS, refit=True
    )
    pred_col = get_pred_col(cv_df)

    cuts = sorted(cv_df["cutoff"].unique())
    cv_df["fold"] = cv_df["cutoff"].map({c: i+1 for i, c in enumerate(cuts)})
    cv_df = cv_df.sort_values(["fold","unique_id","ds"]).reset_index(drop=True)

    yt = cv_df["y"].values
    yp = np.clip(cv_df[pred_col].values, 0, None)
    tot, wape, rb = wape_rbias(yt, yp)
    calib = float(yt.sum() / (yp.sum() + 1e-9))
    yp_c  = np.clip(yp * calib, 0, None)
    tot_c, wape_c, rb_c = wape_rbias(yt, yp_c)

    fold_rows = []
    for fold in range(1, N_FOLDS+1):
        fdf = cv_df[cv_df["fold"]==fold]
        t, w, r = wape_rbias(fdf["y"].values,
                              np.clip(fdf[pred_col].values, 0, None))
        fold_rows.append({"fold":fold,"wape":round(w,4),
                          "rbias":round(r,4),"total":round(t,4)})
    del nf, cv_df, nf_df
    gc.collect()

    return {
        "wape_raw":    round(wape,  4),
        "rbias_raw":   round(rb,    4),
        "total_raw":   round(tot,   4),
        "wape_cal":    round(wape_c,4),
        "rbias_cal":   round(rb_c,  4),
        "total_cal":   round(tot_c, 4),
        "calib_scale": round(calib, 4),
        "fold_stats":  fold_rows,
    }

# ──────────────────────────────────────────────────────────────
# 10 ЭКСПЕРИМЕНТОВ
# ──────────────────────────────────────────────────────────────
# Фазы 3 пре-планированы как p1_+lag12_s1 × p2_blk_12_6_3/p2_mlp_1024_512
# После прогона фаз 1-2 можно скорректировать вручную перед запуском

EXPERIMENTS = [
    # ── Phase 1: Фичи (arch = best [8,4,2], pool=[1,1,1]) ────
    {
        "phase": 1, "name": "p1_+lag12",
        "n_freq_downsample": [48,8,1], "n_pool_kernel_size": [1,1,1],
        "n_blocks": [8,4,2], "mlp_units": [[512,512]]*3,
        "hist_exog": ["status_2","status_3","status_5",
                      "lag_1","lag_2","lag_48","lag_336"],
        "note": "+lag_1, lag_2 — прямое использование ACF lag1=0.44",
    },
    {
        "phase": 1, "name": "p1_+status1",
        "n_freq_downsample": [48,8,1], "n_pool_kernel_size": [1,1,1],
        "n_blocks": [8,4,2], "mlp_units": [[512,512]]*3,
        "hist_exog": ["status_1","status_2","status_3","status_5",
                      "lag_48","lag_336"],
        "note": "+status_1 (spearman=0.52, сейчас не используется)",
    },
    {
        "phase": 1, "name": "p1_+lag12_s1",
        "n_freq_downsample": [48,8,1], "n_pool_kernel_size": [1,1,1],
        "n_blocks": [8,4,2], "mlp_units": [[512,512]]*3,
        "hist_exog": ["status_1","status_2","status_3","status_5",
                      "lag_1","lag_2","lag_48","lag_336"],
        "note": "Комбо: +lag_1,2 + status_1",
    },
    {
        "phase": 1, "name": "p1_+lag124_s1",
        "n_freq_downsample": [48,8,1], "n_pool_kernel_size": [1,1,1],
        "n_blocks": [8,4,2], "mlp_units": [[512,512]]*3,
        "hist_exog": ["status_1","status_2","status_3","status_5",
                      "lag_1","lag_2","lag_4","lag_48","lag_336"],
        "note": "+lag_1,2,4 + status_1 — расширенный набор лагов",
    },

    # ── Phase 2: Архитектура (features = BASE_HIST) ──────────
    {
        "phase": 2, "name": "p2_blk_12_6_3",
        "n_freq_downsample": [48,8,1], "n_pool_kernel_size": [1,1,1],
        "n_blocks": [12,6,3], "mlp_units": [[512,512]]*3,
        "hist_exog": BASE_HIST,
        "note": "n_blocks=[12,6,3] — геометрия 2x от победившего [8,4,2]",
    },
    {
        "phase": 2, "name": "p2_blk_16_4_2",
        "n_freq_downsample": [48,8,1], "n_pool_kernel_size": [1,1,1],
        "n_blocks": [16,4,2], "mlp_units": [[512,512]]*3,
        "hist_exog": BASE_HIST,
        "note": "Экстремально глубокий длинный стек",
    },
    {
        "phase": 2, "name": "p2_mlp_1024_512",
        "n_freq_downsample": [48,8,1], "n_pool_kernel_size": [1,1,1],
        "n_blocks": [8,4,2], "mlp_units": [[1024,512]]*3,
        "hist_exog": BASE_HIST,
        "note": "Шире MLP (1024→512) при том же [8,4,2]",
    },
    {
        "phase": 2, "name": "p2_mlp_1024_1024",
        "n_freq_downsample": [48,8,1], "n_pool_kernel_size": [1,1,1],
        "n_blocks": [8,4,2], "mlp_units": [[1024,1024]]*3,
        "hist_exog": BASE_HIST,
        "note": "Максимально широкий MLP",
    },

    # ── Phase 3: Combo (best_feat × best_arch) ───────────────
    # Пре-планировано: p1_+lag12_s1 × p2_blk_12_6_3
    # Скорректируй вручную если фазы 1/2 дадут другого победителя
    {
        "phase": 3, "name": "p3_lag12_s1_x_12_6_3",
        "n_freq_downsample": [48,8,1], "n_pool_kernel_size": [1,1,1],
        "n_blocks": [12,6,3], "mlp_units": [[512,512]]*3,
        "hist_exog": ["status_1","status_2","status_3","status_5",
                      "lag_1","lag_2","lag_48","lag_336"],
        "note": "Combo: +lag12+s1 × blocks=[12,6,3]",
    },
    {
        "phase": 3, "name": "p3_lag12_s1_x_1024_512",
        "n_freq_downsample": [48,8,1], "n_pool_kernel_size": [1,1,1],
        "n_blocks": [12,6,3], "mlp_units": [[1024,512]]*3,
        "hist_exog": ["status_1","status_2","status_3","status_5",
                      "lag_1","lag_2","lag_48","lag_336"],
        "note": "Combo: +lag12+s1 × blocks=[12,6,3] × wider MLP",
    },
]

# ── Предвычисляем ВСЕ фичи один раз ──
print("Building full NF dataframe with all features...")
train_nf_full = to_nf_df_full(
    train_df,
    min_len     = TOTAL_VAL_PTS + INPUT_SIZE + 1,
    context_len = CONTEXT_LEN,
)
print(f"NF full: {train_nf_full.shape}  routes: {train_nf_full['unique_id'].nunique()}\n")

# ──────────────────────────────────────────────────────────────
# MAIN LOOP
# ──────────────────────────────────────────────────────────────
all_results   = []
best_so_far   = {"total_raw": PREV_BEST_TOTAL, "name": "prev_best [8,4,2] baseline"}
current_phase = None

for i, cfg in enumerate(EXPERIMENTS, 1):
    if cfg["phase"] != current_phase:
        current_phase = cfg["phase"]
        titles = {
            1: "PHASE 1 — FEATURES  (arch=[48,8,1]/[1,1,1]/[8,4,2] fixed)",
            2: "PHASE 2 — ARCHITECTURE  (features=BASE_HIST fixed)",
            3: "PHASE 3 — COMBO  (best feat × best arch)",
        }
        print(f"\n{'='*65}")
        print(titles[current_phase])
        print("="*65)

    print(f"\n[{i:02d}/10] {cfg['name']}")
    print(f"  freq={cfg['n_freq_downsample']}  "
          f"pool={cfg['n_pool_kernel_size']}  "
          f"blocks={cfg['n_blocks']}")
    print(f"  mlp_units={cfg.get('mlp_units', [[512,512]]*3)[0]}"
          f" × {len(cfg['n_blocks'])} stacks")
    print(f"  hist_exog={cfg['hist_exog']}")
    print(f"  {cfg['note']}")

    metrics = run_cv(cfg, train_nf_full, CV_STEPS)

    for fr in metrics["fold_stats"]:
        print(f"  Fold {fr['fold']}  WAPE={fr['wape']:.4f}  "
              f"|RBias|={fr['rbias']:.4f}  Total={fr['total']:.4f}")

    print(f"\n  ── RAW    WAPE={metrics['wape_raw']:.4f}  "
          f"|RBias|={metrics['rbias_raw']:.4f}  "
          f"Total={metrics['total_raw']:.4f}")
    print(f"  ── CALIB  WAPE={metrics['wape_cal']:.4f}  "
          f"|RBias|={metrics['rbias_cal']:.4f}  "
          f"Total={metrics['total_cal']:.4f}  "
          f"scale={metrics['calib_scale']:.4f}")

    row = {"#": i, "name": cfg["name"], "phase": cfg["phase"],
           "hist_n": len(cfg["hist_exog"]),
           "blocks": str(cfg["n_blocks"]),
           "mlp_w": cfg.get("mlp_units",[[512,512]]*3)[0][0],
           **{k: v for k, v in metrics.items() if k != "fold_stats"}}
    all_results.append(row)

    if metrics["total_raw"] < best_so_far["total_raw"]:
        delta = metrics["total_raw"] - best_so_far["total_raw"]
        print(f"\n  🏆 NEW BEST!  Δ={delta:+.4f}  "
              f"(prev: {best_so_far['name']}={best_so_far['total_raw']:.4f})")
        best_so_far = {"name": cfg["name"], **metrics, "cfg": cfg}
    else:
        delta = metrics["total_raw"] - best_so_far["total_raw"]
        print(f"\n  Best: {best_so_far['name']}={best_so_far['total_raw']:.4f}  "
              f"(this: {metrics['total_raw']:.4f}, Δ={delta:+.4f})")

# ── Leaderboard ──
results_df = (
    pd.DataFrame(all_results)
    .sort_values("total_raw")
    [["#","name","phase","hist_n","blocks","mlp_w",
      "wape_raw","rbias_raw","total_raw","total_cal","calib_scale"]]
)
results_df.to_csv("nhits_exp10_feat_arch.csv", index=False)

print("\n\n" + "="*65)
print("LEADERBOARD  (CV_STEPS=400)")
print("="*65)
print(results_df.to_string(index=False))
print(f"\n🏆 WINNER:  {best_so_far['name']}  total_raw={best_so_far['total_raw']:.4f}")

# ──────────────────────────────────────────────────────────────
# WINNER — FULL CV (FINAL_STEPS=800) — сравнимо с прошлыми прогонами
# ──────────────────────────────────────────────────────────────
best_cfg = best_so_far["cfg"]
print(f"\n{'='*65}")
print(f"WINNER FULL CV  (FINAL_STEPS={FINAL_STEPS})")
print(f"Config: {best_so_far['name']}")
print("="*65)

winner_metrics = run_cv(best_cfg, train_nf_full, FINAL_STEPS)

print(f"\nPer-fold:")
for fr in winner_metrics["fold_stats"]:
    print(f"  Fold {fr['fold']}  WAPE={fr['wape']:.4f}  "
          f"|RBias|={fr['rbias']:.4f}  Total={fr['total']:.4f}")
print(f"\n── RAW    WAPE={winner_metrics['wape_raw']:.4f}  "
      f"|RBias|={winner_metrics['rbias_raw']:.4f}  "
      f"Total={winner_metrics['total_raw']:.4f}")
print(f"── CALIB  WAPE={winner_metrics['wape_cal']:.4f}  "
      f"|RBias|={winner_metrics['rbias_cal']:.4f}  "
      f"Total={winner_metrics['total_cal']:.4f}  "
      f"scale={winner_metrics['calib_scale']:.4f}")

calib_scale = winner_metrics["calib_scale"]
pd.DataFrame([{"name": best_so_far["name"], "steps": FINAL_STEPS,
               **{k: v for k, v in winner_metrics.items() if k != "fold_stats"}}
              ]).to_csv("nhits_winner_full_cv.csv", index=False)

# ──────────────────────────────────────────────────────────────
# FINAL PREDICT
# ──────────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("FINAL PREDICT")
print("="*65)

train_nf_pred = to_nf_df_full(train_df, min_len=INPUT_SIZE+1, context_len=CONTEXT_LEN)
train_nf_pred = subset_nf_df(train_nf_pred, best_cfg["hist_exog"])

test_route_ids = test_df["route_id"].unique().tolist()
print(f"Routes: {len(test_route_ids)}")

nf_full = NeuralForecast(models=[make_nhits(best_cfg, FINAL_STEPS)], freq=FREQ)
nf_full.fit(train_nf_pred)

futr_df      = make_futr_df(test_route_ids, train_df)
test_pred_df = nf_full.predict(futr_df=futr_df)
test_pred_col = get_pred_col(test_pred_df)
test_preds    = build_preds_dict(test_pred_df, test_pred_col)

predictions_raw = {}
for route_id in tqdm(test_route_ids, desc="Build submission"):
    route_test = test_df[test_df["route_id"]==route_id].sort_values("timestamp")
    preds = test_preds.get(route_id, np.zeros(FORECAST_STEPS))
    for j, (_, row) in enumerate(route_test.iterrows()):
        pred = float(preds[j]) if j < len(preds) else float(preds[-1])
        predictions_raw[row["id"]] = max(0.0, pred)

name = best_so_far["name"]
submission_raw = (
    pd.DataFrame(list(predictions_raw.items()), columns=["id","y_pred"])
    .sort_values("id").reset_index(drop=True)
)
submission_cal = submission_raw.copy()
submission_cal["y_pred"] = np.clip(submission_cal["y_pred"] * calib_scale, 0, None)

submission_raw.to_csv(f"submission_{name}_raw.csv",        index=False)
submission_cal.to_csv(f"submission_{name}_calibrated.csv", index=False)

print(f"\n✅ nhits_exp10_feat_arch.csv")
print(f"✅ nhits_winner_full_cv.csv")
print(f"✅ submission_{name}_raw.csv  (calib_scale={calib_scale:.4f})")
print(f"✅ submission_{name}_calibrated.csv")
print(f"\n── Raw ──\n{submission_raw['y_pred'].describe().round(2)}")
print(f"\n── Calibrated ──\n{submission_cal['y_pred'].describe().round(2)}")

In [ ]:
[(i, j) for i, j in enumerate(EXPERIMENTS, 1)]

In [ ]:
EXPERIMENTS = [
    # ── Phase 1: Фичи (arch = best [8,4,2], pool=[1,1,1]) ────
    {
        "phase": 1, "name": "p1_+lag12",
        "n_freq_downsample": [48,8,1], "n_pool_kernel_size": [1,1,1],
        "n_blocks": [8,4,2], "mlp_units": [[512,512]]*3,
        "hist_exog": ["status_2","status_3","status_5",
                      "lag_1","lag_2","lag_48","lag_336"],
        "note": "+lag_1, lag_2 — прямое использование ACF lag1=0.44",
    },
    {
        "phase": 1, "name": "p1_+status1",
        "n_freq_downsample": [48,8,1], "n_pool_kernel_size": [1,1,1],
        "n_blocks": [8,4,2], "mlp_units": [[512,512]]*3,
        "hist_exog": ["status_1","status_2","status_3","status_5",
                      "lag_48","lag_336"],
        "note": "+status_1 (spearman=0.52, сейчас не используется)",
    },
    {
        "phase": 1, "name": "p1_+lag12_s1",
        "n_freq_downsample": [48,8,1], "n_pool_kernel_size": [1,1,1],
        "n_blocks": [8,4,2], "mlp_units": [[512,512]]*3,
        "hist_exog": ["status_1","status_2","status_3","status_5",
                      "lag_1","lag_2","lag_48","lag_336"],
        "note": "Комбо: +lag_1,2 + status_1",
    },
    {
        "phase": 1, "name": "p1_+lag124_s1",
        "n_freq_downsample": [48,8,1], "n_pool_kernel_size": [1,1,1],
        "n_blocks": [8,4,2], "mlp_units": [[512,512]]*3,
        "hist_exog": ["status_1","status_2","status_3","status_5",
                      "lag_1","lag_2","lag_4","lag_48","lag_336"],
        "note": "+lag_1,2,4 + status_1 — расширенный набор лагов",
    },

    # ── Phase 2: Архитектура (features = BASE_HIST) ──────────
    {
        "phase": 2, "name": "p2_blk_12_6_3",
        "n_freq_downsample": [48,8,1], "n_pool_kernel_size": [1,1,1],
        "n_blocks": [12,6,3], "mlp_units": [[512,512]]*3,
        "hist_exog": BASE_HIST,
        "note": "n_blocks=[12,6,3] — геометрия 2x от победившего [8,4,2]",
    },
    {
        "phase": 2, "name": "p2_blk_16_4_2",
        "n_freq_downsample": [48,8,1], "n_pool_kernel_size": [1,1,1],
        "n_blocks": [16,4,2], "mlp_units": [[512,512]]*3,
        "hist_exog": BASE_HIST,
        "note": "Экстремально глубокий длинный стек",
    },
    {
        "phase": 2, "name": "p2_mlp_1024_1024",
        "n_freq_downsample": [48,8,1], "n_pool_kernel_size": [1,1,1],
        "n_blocks": [8,4,2], "mlp_units": [[512, 512, 512]]*3,
        "hist_exog": BASE_HIST,
        "note": "Шире MLP (1024→512) при том же [8,4,2]",
    },
    {
        "phase": 2, "name": "p2_mlp_1024_1024",
        "n_freq_downsample": [48,8,1], "n_pool_kernel_size": [1,1,1],
        "n_blocks": [8,4,2], "mlp_units": [[1024,1024]]*3,
        "hist_exog": BASE_HIST,
        "note": "Максимально широкий MLP",
    },

    # ── Phase 3: Combo (best_feat × best_arch) ───────────────
    # Пре-планировано: p1_+lag12_s1 × p2_blk_12_6_3
    # Скорректируй вручную если фазы 1/2 дадут другого победителя
    {
        "phase": 3, "name": "p3_lag12_s1_x_12_6_3",
        "n_freq_downsample": [48,8,1], "n_pool_kernel_size": [1,1,1],
        "n_blocks": [12,6,3], "mlp_units": [[512,512]]*3,
        "hist_exog": ["status_1","status_2","status_3","status_5",
                      "lag_1","lag_2","lag_48","lag_336"],
        "note": "Combo: +lag12+s1 × blocks=[12,6,3]",
    },
    {
        "phase": 3, "name": "p3_lag12_s1_x_1024_512",
        "n_freq_downsample": [48,8,1], "n_pool_kernel_size": [1,1,1],
        "n_blocks": [12,6,3], "mlp_units": [[1024,512]]*3,
        "hist_exog": ["status_1","status_2","status_3","status_5",
                      "lag_1","lag_2","lag_48","lag_336"],
        "note": "Combo: +lag12+s1 × blocks=[12,6,3] × wider MLP",
    },
]

In [ ]:
for i, cfg in [(i, j) for i, j in enumerate(EXPERIMENTS, 1)][6:]:
    if cfg["phase"] != current_phase:
        current_phase = cfg["phase"]
        titles = {
            1: "PHASE 1 — FEATURES  (arch=[48,8,1]/[1,1,1]/[8,4,2] fixed)",
            2: "PHASE 2 — ARCHITECTURE  (features=BASE_HIST fixed)",
            3: "PHASE 3 — COMBO  (best feat × best arch)",
        }
        print(f"\n{'='*65}")
        print(titles[current_phase])
        print("="*65)

    print(f"\n[{i:02d}/10] {cfg['name']}")
    print(f"  freq={cfg['n_freq_downsample']}  "
          f"pool={cfg['n_pool_kernel_size']}  "
          f"blocks={cfg['n_blocks']}")
    print(f"  mlp_units={cfg.get('mlp_units', [[512,512]]*3)[0]}"
          f" × {len(cfg['n_blocks'])} stacks")
    print(f"  hist_exog={cfg['hist_exog']}")
    print(f"  {cfg['note']}")

    metrics = run_cv(cfg, train_nf_full, CV_STEPS)

    for fr in metrics["fold_stats"]:
        print(f"  Fold {fr['fold']}  WAPE={fr['wape']:.4f}  "
              f"|RBias|={fr['rbias']:.4f}  Total={fr['total']:.4f}")

    print(f"\n  ── RAW    WAPE={metrics['wape_raw']:.4f}  "
          f"|RBias|={metrics['rbias_raw']:.4f}  "
          f"Total={metrics['total_raw']:.4f}")
    print(f"  ── CALIB  WAPE={metrics['wape_cal']:.4f}  "
          f"|RBias|={metrics['rbias_cal']:.4f}  "
          f"Total={metrics['total_cal']:.4f}  "
          f"scale={metrics['calib_scale']:.4f}")

    row = {"#": i, "name": cfg["name"], "phase": cfg["phase"],
           "hist_n": len(cfg["hist_exog"]),
           "blocks": str(cfg["n_blocks"]),
           "mlp_w": cfg.get("mlp_units",[[512,512]]*3)[0][0],
           **{k: v for k, v in metrics.items() if k != "fold_stats"}}
    all_results.append(row)

    if metrics["total_raw"] < best_so_far["total_raw"]:
        delta = metrics["total_raw"] - best_so_far["total_raw"]
        print(f"\n  🏆 NEW BEST!  Δ={delta:+.4f}  "
              f"(prev: {best_so_far['name']}={best_so_far['total_raw']:.4f})")
        best_so_far = {"name": cfg["name"], **metrics, "cfg": cfg}
    else:
        delta = metrics["total_raw"] - best_so_far["total_raw"]
        print(f"\n  Best: {best_so_far['name']}={best_so_far['total_raw']:.4f}  "
              f"(this: {metrics['total_raw']:.4f}, Δ={delta:+.4f})")

# ── Leaderboard ──
results_df = (
    pd.DataFrame(all_results)
    .sort_values("total_raw")
    [["#","name","phase","hist_n","blocks","mlp_w",
      "wape_raw","rbias_raw","total_raw","total_cal","calib_scale"]]
)
results_df.to_csv("nhits_exp10_feat_arch.csv", index=False)

print("\n\n" + "="*65)
print("LEADERBOARD  (CV_STEPS=400)")
print("="*65)
print(results_df.to_string(index=False))
print(f"\n🏆 WINNER:  {best_so_far['name']}  total_raw={best_so_far['total_raw']:.4f}")

# ──────────────────────────────────────────────────────────────
# WINNER — FULL CV (FINAL_STEPS=800) — сравнимо с прошлыми прогонами
# ──────────────────────────────────────────────────────────────
best_cfg = best_so_far["cfg"]
print(f"\n{'='*65}")
print(f"WINNER FULL CV  (FINAL_STEPS={FINAL_STEPS})")
print(f"Config: {best_so_far['name']}")
print("="*65)

winner_metrics = run_cv(best_cfg, train_nf_full, FINAL_STEPS)

print(f"\nPer-fold:")
for fr in winner_metrics["fold_stats"]:
    print(f"  Fold {fr['fold']}  WAPE={fr['wape']:.4f}  "
          f"|RBias|={fr['rbias']:.4f}  Total={fr['total']:.4f}")
print(f"\n── RAW    WAPE={winner_metrics['wape_raw']:.4f}  "
      f"|RBias|={winner_metrics['rbias_raw']:.4f}  "
      f"Total={winner_metrics['total_raw']:.4f}")
print(f"── CALIB  WAPE={winner_metrics['wape_cal']:.4f}  "
      f"|RBias|={winner_metrics['rbias_cal']:.4f}  "
      f"Total={winner_metrics['total_cal']:.4f}  "
      f"scale={winner_metrics['calib_scale']:.4f}")

calib_scale = winner_metrics["calib_scale"]
pd.DataFrame([{"name": best_so_far["name"], "steps": FINAL_STEPS,
               **{k: v for k, v in winner_metrics.items() if k != "fold_stats"}}
              ]).to_csv("nhits_winner_full_cv.csv", index=False)

# ──────────────────────────────────────────────────────────────
# FINAL PREDICT
# ──────────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("FINAL PREDICT")
print("="*65)

train_nf_pred = to_nf_df_full(train_df, min_len=INPUT_SIZE+1, context_len=CONTEXT_LEN)
train_nf_pred = subset_nf_df(train_nf_pred, best_cfg["hist_exog"])

test_route_ids = test_df["route_id"].unique().tolist()
print(f"Routes: {len(test_route_ids)}")

nf_full = NeuralForecast(models=[make_nhits(best_cfg, FINAL_STEPS)], freq=FREQ)
nf_full.fit(train_nf_pred)

futr_df      = make_futr_df(test_route_ids, train_df)
test_pred_df = nf_full.predict(futr_df=futr_df)
test_pred_col = get_pred_col(test_pred_df)
test_preds    = build_preds_dict(test_pred_df, test_pred_col)

predictions_raw = {}
for route_id in tqdm(test_route_ids, desc="Build submission"):
    route_test = test_df[test_df["route_id"]==route_id].sort_values("timestamp")
    preds = test_preds.get(route_id, np.zeros(FORECAST_STEPS))
    for j, (_, row) in enumerate(route_test.iterrows()):
        pred = float(preds[j]) if j < len(preds) else float(preds[-1])
        predictions_raw[row["id"]] = max(0.0, pred)

name = best_so_far["name"]
submission_raw = (
    pd.DataFrame(list(predictions_raw.items()), columns=["id","y_pred"])
    .sort_values("id").reset_index(drop=True)
)
submission_cal = submission_raw.copy()
submission_cal["y_pred"] = np.clip(submission_cal["y_pred"] * calib_scale, 0, None)

submission_raw.to_csv(f"submission_{name}_raw.csv",        index=False)
submission_cal.to_csv(f"submission_{name}_calibrated.csv", index=False)

print(f"\n✅ nhits_exp10_feat_arch.csv")
print(f"✅ nhits_winner_full_cv.csv")
print(f"✅ submission_{name}_raw.csv  (calib_scale={calib_scale:.4f})")
print(f"✅ submission_{name}_calibrated.csv")
print(f"\n── Raw ──\n{submission_raw['y_pred'].describe().round(2)}")
print(f"\n── Calibrated ──\n{submission_cal['y_pred'].describe().round(2)}")

In [ ]:
esults_df = (
    pd.DataFrame(all_results)
    .sort_values("total_raw")
    [["#","name","phase","hist_n","blocks","mlp_w",
      "wape_raw","rbias_raw","total_raw","total_cal","calib_scale"]]
)
results_df.to_csv("nhits_exp10_feat_arch.csv", index=False)

print("\n\n" + "="*65)
print("LEADERBOARD  (CV_STEPS=400)")
print("="*65)
print(results_df.to_string(index=False))
print(f"\n🏆 WINNER:  {best_so_far['name']}  total_raw={best_so_far['total_raw']:.4f}")

# ──────────────────────────────────────────────────────────────
# WINNER — FULL CV (FINAL_STEPS=800) — сравнимо с прошлыми прогонами
# ──────────────────────────────────────────────────────────────
best_cfg = best_so_far["cfg"]
print(f"\n{'='*65}")
print(f"WINNER FULL CV  (FINAL_STEPS={FINAL_STEPS})")
print(f"Config: {best_so_far['name']}")
print("="*65)

winner_metrics = run_cv(best_cfg, train_nf_full, FINAL_STEPS)

print(f"\nPer-fold:")
for fr in winner_metrics["fold_stats"]:
    print(f"  Fold {fr['fold']}  WAPE={fr['wape']:.4f}  "
          f"|RBias|={fr['rbias']:.4f}  Total={fr['total']:.4f}")
print(f"\n── RAW    WAPE={winner_metrics['wape_raw']:.4f}  "
      f"|RBias|={winner_metrics['rbias_raw']:.4f}  "
      f"Total={winner_metrics['total_raw']:.4f}")
print(f"── CALIB  WAPE={winner_metrics['wape_cal']:.4f}  "
      f"|RBias|={winner_metrics['rbias_cal']:.4f}  "
      f"Total={winner_metrics['total_cal']:.4f}  "
      f"scale={winner_metrics['calib_scale']:.4f}")

calib_scale = winner_metrics["calib_scale"]
pd.DataFrame([{"name": best_so_far["name"], "steps": FINAL_STEPS,
               **{k: v for k, v in winner_metrics.items() if k != "fold_stats"}}
              ]).to_csv("nhits_winner_full_cv.csv", index=False)

# ──────────────────────────────────────────────────────────────
# FINAL PREDICT
# ──────────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("FINAL PREDICT")
print("="*65)

train_nf_pred = to_nf_df_full(train_df, min_len=INPUT_SIZE+1, context_len=CONTEXT_LEN)
train_nf_pred = subset_nf_df(train_nf_pred, best_cfg["hist_exog"])

test_route_ids = test_df["route_id"].unique().tolist()
print(f"Routes: {len(test_route_ids)}")

nf_full = NeuralForecast(models=[make_nhits(best_cfg, FINAL_STEPS)], freq=FREQ)
nf_full.fit(train_nf_pred)

futr_df      = make_futr_df(test_route_ids, train_df)
test_pred_df = nf_full.predict(futr_df=futr_df)
test_pred_col = get_pred_col(test_pred_df)
test_preds    = build_preds_dict(test_pred_df, test_pred_col)

predictions_raw = {}
for route_id in tqdm(test_route_ids, desc="Build submission"):
    route_test = test_df[test_df["route_id"]==route_id].sort_values("timestamp")
    preds = test_preds.get(route_id, np.zeros(FORECAST_STEPS))
    for j, (_, row) in enumerate(route_test.iterrows()):
        pred = float(preds[j]) if j < len(preds) else float(preds[-1])
        predictions_raw[row["id"]] = max(0.0, pred)

name = best_so_far["name"]
submission_raw = (
    pd.DataFrame(list(predictions_raw.items()), columns=["id","y_pred"])
    .sort_values("id").reset_index(drop=True)
)
submission_cal = submission_raw.copy()
submission_cal["y_pred"] = np.clip(submission_cal["y_pred"] * calib_scale, 0, None)

submission_raw.to_csv(f"submission_{name}_raw.csv",        index=False)
submission_cal.to_csv(f"submission_{name}_calibrated.csv", index=False)

print(f"\n✅ nhits_exp10_feat_arch.csv")
print(f"✅ nhits_winner_full_cv.csv")
print(f"✅ submission_{name}_raw.csv  (calib_scale={calib_scale:.4f})")
print(f"✅ submission_{name}_calibrated.csv")
print(f"\n── Raw ──\n{submission_raw['y_pred'].describe().round(2)}")
print(f"\n── Calibrated ──\n{submission_cal['y_pred'].describe().round(2)}")

In [ ]:
# ============================================================
# Per-Route Walk-Forward Calibration
# 1. Загружаем или перепрогоняем CV для лучшего конфига
# 2. Честная walk-forward калибровка по фолдам
# 3. Сравниваем global vs per-route calibration
# 4. Применяем к тесту
# ============================================================

import warnings, os, gc
import numpy as np
import pandas as pd
import torch
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MQLoss
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias

# ── Config ──
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8
N_FOLDS        = 5
TOTAL_VAL_PTS  = N_FOLDS * FORECAST_STEPS
CONTEXT_LEN    = 2048
FREQ           = "30min"
INPUT_SIZE     = 336
BATCH_SIZE     = 64
FINAL_STEPS    = 800

# ── Лучший конфиг из экспериментов ──
BEST_CFG = {
    "name":               "p2_blk_8_4_2",
    "n_freq_downsample":  [48, 8, 1],
    "n_pool_kernel_size": [1, 1, 1],
    "n_blocks":           [8, 4, 2],
    "mlp_units":          [[512, 512]] * 3,
    "hist_exog":          ["status_2", "status_3", "status_5", "lag_48", "lag_336"],
}

# Файл с raw CV — если уже есть, не перегоняем
CV_RAW_FILE = f"nhits_cv_raw_{BEST_CFG['name']}.csv"

if torch.backends.mps.is_available():   ACCELERATOR = "mps"
elif torch.cuda.is_available():         ACCELERATOR = "gpu"
else:                                   ACCELERATOR = "cpu"
print(f"Device: {ACCELERATOR}")

# ── Data ──
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id","timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id","timestamp"]).reset_index(drop=True)
status_cols = sorted([c for c in train_df.columns if c in ["status_2","status_3","status_5"]])

FUTR_EXOG = ["hour_sin","hour_cos","dow_sin","dow_cos","is_weekend"]

def add_calendar_features(df, ts_col="ds"):
    ts = pd.to_datetime(df[ts_col])
    df = df.copy()
    df["hour_sin"]   = np.sin(2*np.pi*ts.dt.hour/24)
    df["hour_cos"]   = np.cos(2*np.pi*ts.dt.hour/24)
    df["dow_sin"]    = np.sin(2*np.pi*ts.dt.dayofweek/7)
    df["dow_cos"]    = np.cos(2*np.pi*ts.dt.dayofweek/7)
    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(float)
    return df

def add_lag_features(nf_df):
    nf_df = nf_df.sort_values(["unique_id","ds"]).copy()
    for lag in [48, 336]:
        nf_df[f"lag_{lag}"] = (
            nf_df.groupby("unique_id")["y"]
            .transform(lambda s, l=lag: s.shift(l).bfill().fillna(0))
        )
    return nf_df

def to_nf_df(df, min_len=32, context_len=CONTEXT_LEN):
    rows = []
    for route_id, grp in df.groupby("route_id"):
        grp = (
            grp.sort_values("timestamp")
            .set_index("timestamp")[[TARGET_COL] + status_cols]
            .asfreq(FREQ).interpolate(method="time")
            .bfill().ffill().tail(context_len).reset_index()
        )
        if len(grp) < min_len:
            continue
        grp.insert(0, "unique_id", route_id)
        grp = grp.rename(columns={"timestamp":"ds", TARGET_COL:"y"})
        rows.append(grp)
    nf_df = pd.concat(rows, ignore_index=True)
    nf_df = add_calendar_features(nf_df)
    nf_df = add_lag_features(nf_df)
    return nf_df

def make_futr_df(route_ids, train_df):
    rows = []
    for rid in route_ids:
        last_ts = train_df[train_df["route_id"]==rid]["timestamp"].max()
        for ts in pd.date_range(start=last_ts+pd.Timedelta("30min"),
                                periods=FORECAST_STEPS, freq=FREQ):
            rows.append({"unique_id": rid, "ds": ts})
    return add_calendar_features(pd.DataFrame(rows))

def get_pred_col(pred_df):
    for c in ["NHITS-median","NHITS-q-0.50","NHITS"]:
        if c in pred_df.columns: return c
    skip = {"unique_id","ds","y","cutoff"}
    num  = [c for c in pred_df.columns if c not in skip
            and pd.api.types.is_numeric_dtype(pred_df[c])]
    med  = [c for c in num if any(x in c for x in ["median","0.5","50"])]
    return (med or num)[0]

def make_nhits(cfg, max_steps):
    return NHITS(
        h                  = FORECAST_STEPS,
        input_size         = INPUT_SIZE,
        loss               = MQLoss(level=[80]),
        n_freq_downsample  = cfg["n_freq_downsample"],
        n_pool_kernel_size = cfg["n_pool_kernel_size"],
        n_blocks           = cfg["n_blocks"],
        mlp_units          = cfg["mlp_units"],
        max_steps          = max_steps,
        batch_size         = BATCH_SIZE,
        accelerator        = ACCELERATOR,
        hist_exog_list     = cfg["hist_exog"],
        futr_exog_list     = FUTR_EXOG,
        scaler_type        = "robust",
        enable_progress_bar= True,
    )

# ============================================================
# ШАГ 1: Raw CV predictions — загружаем или пересчитываем
# ============================================================
if os.path.exists(CV_RAW_FILE):
    print(f"Loading existing CV raw: {CV_RAW_FILE}")
    raw_cv_df = pd.read_csv(CV_RAW_FILE, parse_dates=["timestamp"])
else:
    print(f"CV raw not found — running CV (FINAL_STEPS={FINAL_STEPS})...")
    train_nf = to_nf_df(
        train_df,
        min_len     = TOTAL_VAL_PTS + INPUT_SIZE + 1,
        context_len = CONTEXT_LEN,
    )
    print(f"NF train: {train_nf.shape}  routes: {train_nf['unique_id'].nunique()}")

    nf = NeuralForecast(models=[make_nhits(BEST_CFG, FINAL_STEPS)], freq=FREQ)
    cv_df = nf.cross_validation(
        df=train_nf, n_windows=N_FOLDS, step_size=FORECAST_STEPS, refit=True
    )
    pred_col = get_pred_col(cv_df)
    cuts = sorted(cv_df["cutoff"].unique())
    cv_df["fold"] = cv_df["cutoff"].map({c: i+1 for i, c in enumerate(cuts)})
    cv_df = cv_df.sort_values(["fold","unique_id","ds"]).reset_index(drop=True)
    cv_df["h"] = cv_df.groupby(["fold","unique_id"]).cumcount() + 1

    raw_rows = []
    for _, row in cv_df.iterrows():
        y_true = float(row["y"])
        y_pred = float(np.clip(row[pred_col], 0, None))
        raw_rows.append({
            "fold":      int(row["fold"]),
            "route_id":  row["unique_id"],
            "timestamp": row["ds"],
            "h":         int(row["h"]),
            "y_true":    y_true,
            "y_pred":    y_pred,
            "ae":        abs(y_pred - y_true),
            "err":       y_pred - y_true,
        })
    raw_cv_df = pd.DataFrame(raw_rows)
    raw_cv_df.to_csv(CV_RAW_FILE, index=False)
    print(f"✅ Saved: {CV_RAW_FILE}")
    del nf, cv_df
    gc.collect()

print(f"\nCV raw: {raw_cv_df.shape}  "
      f"folds: {raw_cv_df['fold'].nunique()}  "
      f"routes: {raw_cv_df['route_id'].nunique()}")

# ============================================================
# ШАГ 2: Честная walk-forward per-route calibration
#
# Fold k → scale_r computed from folds 1..k-1
# Fold 1 → scale_r = 1.0 (нет истории, не калибруем)
# Минимальный порог: если маршрут видели < MIN_OBS точек
#                    в прошлых фолдах → scale=global_scale
# ============================================================
MIN_OBS = 4  # минимум точек в истории для per-route scale

all_routes = raw_cv_df["route_id"].unique()

# Предвычислим глобальный scale для фолбэка
global_scale_all = (
    raw_cv_df["y_true"].sum() / (raw_cv_df["y_pred"].sum() + 1e-9)
)

calibrated_rows = []

for fold in range(1, N_FOLDS + 1):
    fold_df = raw_cv_df[raw_cv_df["fold"] == fold].copy()

    if fold == 1:
        # Нет истории — scale = 1.0 для всех маршрутов
        fold_df["scale_used"] = 1.0
        fold_df["calib_type"] = "none"
    else:
        # История = все предыдущие фолды
        history = raw_cv_df[raw_cv_df["fold"] < fold]

        # Per-route scale из истории
        route_stats = (
            history.groupby("route_id")
            .agg(sum_true=("y_true","sum"),
                 sum_pred=("y_pred","sum"),
                 n_obs   =("y_true","count"))
            .reset_index()
        )
        route_stats["per_route_scale"] = (
            route_stats["sum_true"] / (route_stats["sum_pred"] + 1e-9)
        )

        # Глобальный scale из истории (фолбэк для маршрутов с мало данных)
        global_scale_hist = (
            history["y_true"].sum() / (history["y_pred"].sum() + 1e-9)
        )

        # Мерджим в fold_df
        fold_df = fold_df.merge(
            route_stats[["route_id","per_route_scale","n_obs"]],
            on="route_id", how="left"
        )
        # Маршруты без истории или с малым количеством точек → global scale
        fold_df["scale_used"] = np.where(
            fold_df["n_obs"].fillna(0) >= MIN_OBS,
            fold_df["per_route_scale"],
            global_scale_hist,
        )
        fold_df["calib_type"] = np.where(
            fold_df["n_obs"].fillna(0) >= MIN_OBS, "per_route", "global_fallback"
        )
        fold_df.drop(columns=["per_route_scale","n_obs"], inplace=True)

    fold_df["y_pred_cal"] = np.clip(fold_df["y_pred"] * fold_df["scale_used"], 0, None)
    calibrated_rows.append(fold_df)

cal_cv_df = pd.concat(calibrated_rows, ignore_index=True)

# ============================================================
# ШАГ 3: Сравнение метрик
# ============================================================
print("\n" + "="*65)
print("CALIBRATION COMPARISON (walk-forward, честный)")
print("="*65)

# Raw (без калибровки)
tot_r, wape_r, rb_r = wape_rbias(cal_cv_df["y_true"], cal_cv_df["y_pred"])

# Global calibration (один scale на всё)
global_scale = cal_cv_df["y_true"].sum() / (cal_cv_df["y_pred"].sum() + 1e-9)
y_global_cal = np.clip(cal_cv_df["y_pred"] * global_scale, 0, None)
tot_g, wape_g, rb_g = wape_rbias(cal_cv_df["y_true"], y_global_cal)

# Per-route walk-forward calibration
tot_p, wape_p, rb_p = wape_rbias(cal_cv_df["y_true"], cal_cv_df["y_pred_cal"])

print(f"\n{'Method':<30} {'WAPE':>8} {'|RBias|':>9} {'Total':>8}")
print("-"*57)
print(f"{'Raw (no calibration)':<30} {wape_r:>8.4f} {rb_r:>9.4f} {tot_r:>8.4f}")
print(f"{'Global calibration':<30} {wape_g:>8.4f} {rb_g:>9.4f} {tot_g:>8.4f}")
print(f"{'Per-route walk-forward':<30} {wape_p:>8.4f} {rb_p:>9.4f} {tot_p:>8.4f}")
print()
print(f"  Global vs Raw:     Δtotal = {tot_g - tot_r:+.4f}")
print(f"  Per-route vs Raw:  Δtotal = {tot_p - tot_r:+.4f}")
print(f"  Per-route vs Global: Δtotal = {tot_p - tot_g:+.4f}")

# ── Per-fold разбивка ──
print(f"\n{'─'*65}")
print(f"{'Fold':<6} {'Raw':>8} {'Global':>9} {'PerRoute':>10} "
      f"{'Δ(pr-raw)':>11} {'calib_type dist':>5}")
print("─"*65)
for fold in range(1, N_FOLDS+1):
    fdf = cal_cv_df[cal_cv_df["fold"]==fold]
    t_r = wape_rbias(fdf["y_true"], fdf["y_pred"])[0]
    t_g = wape_rbias(fdf["y_true"],
                     np.clip(fdf["y_pred"]*global_scale,0,None))[0]
    t_p = wape_rbias(fdf["y_true"], fdf["y_pred_cal"])[0]
    types = fdf["calib_type"].value_counts().to_dict()
    print(f"  {fold:<4} {t_r:>8.4f} {t_g:>9.4f} {t_p:>10.4f} "
          f"{t_p-t_r:>+11.4f}   {types}")

# ── Распределение scale_used по маршрутам ──
route_scales = (
    cal_cv_df[cal_cv_df["fold"]==N_FOLDS]
    .groupby("route_id")["scale_used"].first()
)
print(f"\n── Scale distribution (fold {N_FOLDS}, per-route) ──")
print(route_scales.describe().round(4))
extreme = route_scales[(route_scales < 0.5) | (route_scales > 2.0)]
print(f"  Экстремальных scale (<0.5 или >2.0): {len(extreme)} маршрутов")

# ============================================================
# ШАГ 4: Per-route scale для теста = из ВСЕХ 5 фолдов
# ============================================================
route_stats_full = (
    raw_cv_df.groupby("route_id")
    .agg(sum_true=("y_true","sum"),
         sum_pred=("y_pred","sum"),
         n_obs   =("y_true","count"))
    .reset_index()
)
route_stats_full["per_route_scale"] = (
    route_stats_full["sum_true"] / (route_stats_full["sum_pred"] + 1e-9)
)
# Клипуем экстремальные значения scale — маршруты с аномальным drift
SCALE_MIN, SCALE_MAX = 0.5, 2.0
route_stats_full["per_route_scale_clipped"] = route_stats_full["per_route_scale"].clip(
    SCALE_MIN, SCALE_MAX
)
n_clipped = (route_stats_full["per_route_scale"] != route_stats_full["per_route_scale_clipped"]).sum()
print(f"\n── Test scale stats ──")
print(route_stats_full["per_route_scale"].describe().round(4))
print(f"  Clipped ({SCALE_MIN}–{SCALE_MAX}): {n_clipped} маршрутов")
route_stats_full.to_csv("nhits_per_route_scales.csv", index=False)

# ============================================================
# ШАГ 5: Применяем к тесту
# ============================================================
# Загружаем сырые предсказания лучшего конфига
RAW_SUBMISSION = f"submission_{BEST_CFG['name']}_raw.csv"

if os.path.exists(RAW_SUBMISSION):
    print(f"\nLoading: {RAW_SUBMISSION}")
    sub_raw = pd.read_csv(RAW_SUBMISSION)
else:
    print(f"\n{RAW_SUBMISSION} не найден — запускаем финальный predict...")

    train_nf_full = to_nf_df(train_df, min_len=INPUT_SIZE+1, context_len=CONTEXT_LEN)
    test_route_ids = test_df["route_id"].unique().tolist()
    nf_full = NeuralForecast(models=[make_nhits(BEST_CFG, FINAL_STEPS)], freq=FREQ)
    nf_full.fit(train_nf_full)

    futr_df      = make_futr_df(test_route_ids, train_df)
    test_pred_df = nf_full.predict(futr_df=futr_df)
    pred_col     = get_pred_col(test_pred_df)

    predictions_raw = {}
    for route_id in tqdm(test_route_ids, desc="Build submission"):
        route_test = test_df[test_df["route_id"]==route_id].sort_values("timestamp")
        grp = test_pred_df[test_pred_df.index == route_id] if hasattr(test_pred_df.index, '__iter__') else \
              test_pred_df[test_pred_df["unique_id"]==route_id]
        preds = np.clip(grp.sort_values("ds")[pred_col].values, 0, None)
        for j, (_, row) in enumerate(route_test.iterrows()):
            predictions_raw[row["id"]] = max(0.0, float(preds[j]) if j < len(preds) else 0.0)

    sub_raw = (
        pd.DataFrame(list(predictions_raw.items()), columns=["id","y_pred"])
        .sort_values("id").reset_index(drop=True)
    )
    sub_raw.to_csv(RAW_SUBMISSION, index=False)
    del nf_full
    gc.collect()

# Джойним route_id к submission (через test_df)
test_id_route = test_df[["id","route_id"]].copy()
sub_merged = sub_raw.merge(test_id_route, on="id", how="left")
sub_merged = sub_merged.merge(
    route_stats_full[["route_id","per_route_scale_clipped"]],
    on="route_id", how="left"
)
# Если маршрут не был в CV (новый) → глобальный scale
sub_merged["per_route_scale_clipped"] = sub_merged["per_route_scale_clipped"].fillna(global_scale)

# Три версии сабмита
sub_raw_out = sub_raw.copy()

sub_global = sub_raw.copy()
sub_global["y_pred"] = np.clip(sub_global["y_pred"] * global_scale, 0, None)

sub_per_route = sub_raw.copy()
sub_per_route["y_pred"] = np.clip(
    sub_merged["y_pred"] * sub_merged["per_route_scale_clipped"], 0, None
)

name = BEST_CFG["name"]
sub_raw_out.to_csv(   f"submission_{name}_raw.csv",             index=False)
sub_global.to_csv(    f"submission_{name}_global_cal.csv",      index=False)
sub_per_route.to_csv( f"submission_{name}_per_route_cal.csv",   index=False)

print(f"\n✅ nhits_per_route_scales.csv")
print(f"✅ {CV_RAW_FILE}")
print(f"✅ submission_{name}_raw.csv")
print(f"✅ submission_{name}_global_cal.csv          (scale={global_scale:.4f})")
print(f"✅ submission_{name}_per_route_cal.csv       (per-route, clip={SCALE_MIN}–{SCALE_MAX})")

print(f"\n── Дистрибуции ──")
print(f"{'':25} {'mean':>9} {'std':>9} {'min':>9} {'max':>9}")
for label, s in [("raw", sub_raw_out), ("global_cal", sub_global), ("per_route_cal", sub_per_route)]:
    d = s["y_pred"].describe()
    print(f"  {label:<23} {d['mean']:>9.0f} {d['std']:>9.0f} {d['min']:>9.0f} {d['max']:>9.0f}")

In [ ]:
# ============================================================
# Shrinkage calibration — подбор k на CV, применение к тесту
# ============================================================

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Загружаем CV raw (уже есть из прошлого запуска)
CV_RAW_FILE = "nhits_cv_raw_p2_blk_8_4_2.csv"
raw_cv_df   = pd.read_csv(CV_RAW_FILE, parse_dates=["timestamp"])

def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias

N_FOLDS        = 5
FORECAST_STEPS = 8
SCALE_MIN      = 0.5
SCALE_MAX      = 2.0

# ── Global scale (один на всё) ──
global_scale_all = (
    raw_cv_df["y_true"].sum() / (raw_cv_df["y_pred"].sum() + 1e-9)
)

# ============================================================
# Честный walk-forward с shrinkage
# fold k → scale из фолдов 1..k-1 со shrinkage к global
# ============================================================
def compute_shrinkage_cv(k_shrink: float) -> float:
    """
    Возвращает total метрику walk-forward CV при заданном k.
    k=0 → чистый per-route, k=inf → чистый global.
    """
    calibrated_rows = []
    for fold in range(1, N_FOLDS + 1):
        fold_df = raw_cv_df[raw_cv_df["fold"] == fold].copy()

        if fold == 1:
            fold_df["y_pred_cal"] = fold_df["y_pred"]
        else:
            history = raw_cv_df[raw_cv_df["fold"] < fold]
            global_scale_hist = (
                history["y_true"].sum() / (history["y_pred"].sum() + 1e-9)
            )
            route_stats = (
                history.groupby("route_id")
                .agg(sum_true=("y_true","sum"),
                     sum_pred=("y_pred","sum"),
                     n_obs   =("y_true","count"))
                .reset_index()
            )
            # Shrinkage formula
            route_stats["scale_raw"] = (
                route_stats["sum_true"] / (route_stats["sum_pred"] + 1e-9)
            )
            route_stats["scale_shrunk"] = (
                (route_stats["n_obs"] * route_stats["scale_raw"]
                 + k_shrink * global_scale_hist)
                / (route_stats["n_obs"] + k_shrink)
            ).clip(SCALE_MIN, SCALE_MAX)

            fold_df = fold_df.merge(
                route_stats[["route_id","scale_shrunk","n_obs"]],
                on="route_id", how="left"
            )
            fold_df["scale_shrunk"] = fold_df["scale_shrunk"].fillna(global_scale_hist)
            fold_df["y_pred_cal"] = np.clip(
                fold_df["y_pred"] * fold_df["scale_shrunk"], 0, None
            )
            fold_df.drop(columns=["scale_shrunk","n_obs"], inplace=True)

        calibrated_rows.append(fold_df)

    cal_df = pd.concat(calibrated_rows, ignore_index=True)
    tot, wape, rb = wape_rbias(cal_df["y_true"], cal_df["y_pred_cal"])
    return tot, wape, rb, cal_df

# ── Перебор k ──
k_values = [0, 1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 9999]
results_k = []
for k in k_values:
    tot, wape, rb, _ = compute_shrinkage_cv(k)
    results_k.append({"k": k, "total": round(tot,5),
                      "wape": round(wape,5), "rbias": round(rb,5)})

df_k = pd.DataFrame(results_k)
print("Shrinkage k sweep:")
print(df_k.to_string(index=False))

best_k_row = df_k.loc[df_k["total"].idxmin()]
best_k = float(best_k_row["k"])
print(f"\n🏆 Best k = {best_k}  total={best_k_row['total']:.5f}  "
      f"wape={best_k_row['wape']:.5f}  rbias={best_k_row['rbias']:.5f}")

# Reference: global и raw
tot_r, wape_r, rb_r = wape_rbias(raw_cv_df["y_true"], raw_cv_df["y_pred"])
y_global_cal = np.clip(raw_cv_df["y_pred"] * global_scale_all, 0, None)
tot_g, wape_g, rb_g = wape_rbias(raw_cv_df["y_true"], y_global_cal)

print(f"\n{'Method':<35} {'WAPE':>8} {'|RBias|':>9} {'Total':>8}")
print("-"*62)
print(f"{'Raw':<35} {wape_r:>8.4f} {rb_r:>9.4f} {tot_r:>8.4f}")
print(f"{'Global calibration':<35} {wape_g:>8.4f} {rb_g:>9.4f} {tot_g:>8.4f}")
print(f"{'Shrinkage k='+str(int(best_k)):<35} "
      f"{best_k_row['wape']:>8.4f} {best_k_row['rbias']:>9.4f} "
      f"{best_k_row['total']:>8.4f}")

# ── Plot k sweep ──
fig, ax = plt.subplots(figsize=(8, 4))
k_plot = df_k[df_k["k"] < 9999]
ax.plot(k_plot["k"], k_plot["total"], marker="o", color="#2563eb", label="Shrinkage CV")
ax.axhline(tot_g, color="#dc2626", linestyle="--", label=f"Global ({tot_g:.4f})")
ax.axhline(tot_r, color="#6b7280", linestyle=":",  label=f"Raw ({tot_r:.4f})")
best_row_plot = df_k[df_k["k"]==best_k].iloc[0]
ax.scatter([best_k], [best_row_plot["total"]], s=120, zorder=5,
           color="#16a34a", label=f"Best k={int(best_k)} ({best_row_plot['total']:.4f})")
ax.set_xlabel("Shrinkage parameter k")
ax.set_ylabel("Total (WAPE + |RBias|)")
ax.set_title("Per-route Shrinkage Calibration — CV sweep")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("shrinkage_k_sweep.png", dpi=150)
plt.close()
print("\n✅ shrinkage_k_sweep.png")

# ============================================================
# Финальный per-route shrinkage scale для теста
# (из ALL 5 фолдов, k = best_k)
# ============================================================
route_stats_full = (
    raw_cv_df.groupby("route_id")
    .agg(sum_true=("y_true","sum"),
         sum_pred=("y_pred","sum"),
         n_obs   =("y_true","count"))
    .reset_index()
)
route_stats_full["scale_raw"] = (
    route_stats_full["sum_true"] / (route_stats_full["sum_pred"] + 1e-9)
)
route_stats_full["scale_shrunk"] = (
    (route_stats_full["n_obs"] * route_stats_full["scale_raw"]
     + best_k * global_scale_all)
    / (route_stats_full["n_obs"] + best_k)
).clip(SCALE_MIN, SCALE_MAX)

route_stats_full.to_csv("nhits_per_route_scales_shrunk.csv", index=False)

print(f"\n── Test scale (shrinkage k={int(best_k)}) distribution ──")
print(route_stats_full["scale_shrunk"].describe().round(4))

# ============================================================
# Три submission: raw / global / shrinkage
# ============================================================
import os
RAW_FILE = "submission_p2_blk_8_4_2_raw.csv"
if not os.path.exists(RAW_FILE):
    RAW_FILE = "submission_nhits_raw.csv"   # fallback на первый прогон

test_df_sub = pd.read_parquet("test_solo_track.parquet")[["id","route_id"]]
sub_raw = pd.read_csv(RAW_FILE)
sub_merged = sub_raw.merge(test_df_sub, on="id", how="left")
sub_merged = sub_merged.merge(
    route_stats_full[["route_id","scale_shrunk"]],
    on="route_id", how="left"
)
sub_merged["scale_shrunk"] = sub_merged["scale_shrunk"].fillna(global_scale_all)

sub_global   = sub_raw.copy()
sub_global["y_pred"] = np.clip(sub_raw["y_pred"] * global_scale_all, 0, None)

sub_shrunk   = sub_raw.copy()
sub_shrunk["y_pred"] = np.clip(sub_merged["y_pred"] * sub_merged["scale_shrunk"], 0, None)

sub_global.to_csv("submission_p2_blk_8_4_2_global_cal.csv",   index=False)
sub_shrunk.to_csv("submission_p2_blk_8_4_2_shrunk_cal.csv",   index=False)

print(f"\n✅ nhits_per_route_scales_shrunk.csv")
print(f"✅ submission_p2_blk_8_4_2_global_cal.csv   (scale={global_scale_all:.4f})")
print(f"✅ submission_p2_blk_8_4_2_shrunk_cal.csv   (k={int(best_k)}, "
      f"mean_scale={route_stats_full['scale_shrunk'].mean():.4f})")

print(f"\n── Дистрибуции ──")
for label, s in [("raw", sub_raw), ("global", sub_global), ("shrunk", sub_shrunk)]:
    d = s["y_pred"].describe()
    print(f"  {label:<12} mean={d['mean']:>10.0f}  std={d['std']:>10.0f}  "
          f"min={d['min']:>8.0f}  max={d['max']:>12.0f}")

In [ ]:
# ============================================================
# Retraining experiments — Loss & Input size
# Phase 1 (1-4): Loss function (arch=best [8,4,2], input=336)
# Phase 2 (5-8): input_size (arch=best [8,4,2], loss=best)
# Phase 3 (9-10): combo best_loss × best_input
# ============================================================

import warnings, gc
import numpy as np
import pandas as pd
import torch
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import (
    MAE, MQLoss, HuberLoss, QuantileLoss, MAPE
)
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias

# ── Config ──
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8
N_FOLDS        = 5
TOTAL_VAL_PTS  = N_FOLDS * FORECAST_STEPS
CONTEXT_LEN    = 2048
FREQ           = "30min"
BATCH_SIZE     = 64
CV_STEPS       = 400
FINAL_STEPS    = 800

PREV_BEST_TOTAL = 0.32739  # global calib CV

if torch.backends.mps.is_available():   ACCELERATOR = "mps"
elif torch.cuda.is_available():         ACCELERATOR = "gpu"
else:                                   ACCELERATOR = "cpu"
print(f"Device: {ACCELERATOR}")

# ── Data ──
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id","timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id","timestamp"]).reset_index(drop=True)
status_cols = sorted([c for c in train_df.columns
                      if c in ["status_2","status_3","status_5"]])

FUTR_EXOG = ["hour_sin","hour_cos","dow_sin","dow_cos","is_weekend"]
BASE_HIST  = status_cols + ["lag_48","lag_336"]

def add_calendar_features(df, ts_col="ds"):
    ts = pd.to_datetime(df[ts_col])
    df = df.copy()
    df["hour_sin"]   = np.sin(2*np.pi*ts.dt.hour/24)
    df["hour_cos"]   = np.cos(2*np.pi*ts.dt.hour/24)
    df["dow_sin"]    = np.sin(2*np.pi*ts.dt.dayofweek/7)
    df["dow_cos"]    = np.cos(2*np.pi*ts.dt.dayofweek/7)
    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(float)
    return df

def add_lag_features(nf_df):
    nf_df = nf_df.sort_values(["unique_id","ds"]).copy()
    for lag in [48, 336]:
        nf_df[f"lag_{lag}"] = (
            nf_df.groupby("unique_id")["y"]
            .transform(lambda s, l=lag: s.shift(l).bfill().fillna(0))
        )
    return nf_df

def to_nf_df(df, min_len=32, context_len=CONTEXT_LEN):
    rows = []
    for route_id, grp in df.groupby("route_id"):
        grp = (
            grp.sort_values("timestamp")
            .set_index("timestamp")[[TARGET_COL]+status_cols]
            .asfreq(FREQ).interpolate(method="time")
            .bfill().ffill().tail(context_len).reset_index()
        )
        if len(grp) < min_len:
            continue
        grp.insert(0, "unique_id", route_id)
        grp = grp.rename(columns={"timestamp":"ds", TARGET_COL:"y"})
        rows.append(grp)
    nf_df = pd.concat(rows, ignore_index=True)
    nf_df = add_calendar_features(nf_df)
    nf_df = add_lag_features(nf_df)
    return nf_df

def make_futr_df(route_ids, train_df):
    rows = []
    for rid in route_ids:
        last_ts = train_df[train_df["route_id"]==rid]["timestamp"].max()
        for ts in pd.date_range(start=last_ts+pd.Timedelta("30min"),
                                periods=FORECAST_STEPS, freq=FREQ):
            rows.append({"unique_id": rid, "ds": ts})
    return add_calendar_features(pd.DataFrame(rows))

def get_pred_col(pred_df):
    for c in ["NHITS-median","NHITS-q-0.50","NHITS"]:
        if c in pred_df.columns: return c
    skip = {"unique_id","ds","y","cutoff"}
    num  = [c for c in pred_df.columns if c not in skip
            and pd.api.types.is_numeric_dtype(pred_df[c])]
    med  = [c for c in num if any(x in c for x in ["median","0.5","50"])]
    return (med or num)[0]

def build_preds_dict(pred_df, pred_col):
    return {rid: np.clip(g.sort_values("ds")[pred_col].values, 0, None)
            for rid, g in pred_df.groupby("unique_id")}

def make_nhits(cfg, max_steps):
    return NHITS(
        h                  = FORECAST_STEPS,
        input_size         = cfg["input_size"],
        loss               = cfg["loss"],
        n_freq_downsample  = [48, 8, 1],
        n_pool_kernel_size = [1, 1, 1],
        n_blocks           = [8, 4, 2],
        mlp_units          = [[512, 512]] * 3,
        max_steps          = max_steps,
        batch_size         = BATCH_SIZE,
        accelerator        = ACCELERATOR,
        hist_exog_list     = BASE_HIST,
        futr_exog_list     = FUTR_EXOG,
        scaler_type        = "robust",
        enable_progress_bar= True,
    )

def run_cv(cfg, train_nf, max_steps):
    nf    = NeuralForecast(models=[make_nhits(cfg, max_steps)], freq=FREQ)
    cv_df = nf.cross_validation(
        df=train_nf, n_windows=N_FOLDS, step_size=FORECAST_STEPS, refit=True
    )
    pred_col = get_pred_col(cv_df)
    cuts = sorted(cv_df["cutoff"].unique())
    cv_df["fold"] = cv_df["cutoff"].map({c: i+1 for i, c in enumerate(cuts)})
    cv_df = cv_df.sort_values(["fold","unique_id","ds"]).reset_index(drop=True)

    yt = cv_df["y"].values
    yp = np.clip(cv_df[pred_col].values, 0, None)
    tot_r, wape_r, rb_r = wape_rbias(yt, yp)

    # Global calibration (честная: весь CV, один scale)
    calib = float(yt.sum() / (yp.sum() + 1e-9))
    yp_c  = np.clip(yp * calib, 0, None)
    tot_c, wape_c, rb_c = wape_rbias(yt, yp_c)

    fold_rows = []
    for fold in range(1, N_FOLDS+1):
        fdf = cv_df[cv_df["fold"]==fold]
        t, w, r = wape_rbias(fdf["y"].values,
                              np.clip(fdf[pred_col].values, 0, None))
        fold_rows.append({"fold":fold,"wape":round(w,4),
                          "rbias":round(r,4),"total":round(t,4)})

    del nf, cv_df
    gc.collect()

    return {
        "wape_raw":    round(wape_r, 4),
        "rbias_raw":   round(rb_r,   4),
        "total_raw":   round(tot_r,  4),
        "wape_cal":    round(wape_c, 4),
        "rbias_cal":   round(rb_c,   4),
        "total_cal":   round(tot_c,  4),   # ← это и есть честная CV метрика
        "calib_scale": round(calib,  4),
        "fold_stats":  fold_rows,
    }

# ──────────────────────────────────────────────────────────────
# ЭКСПЕРИМЕНТЫ
# ──────────────────────────────────────────────────────────────
EXPERIMENTS = [

    # ── Phase 1: Loss function (input_size=336 fixed) ─────────
    {
        "phase": 1, "name": "p1_mae",
        "input_size": 336,
        "loss": MAE(),
        # Прямо оптимизирует то что мы меряем (WAPE = взвешенный MAE)
        "note": "MAE — прямое выравнивание с метрикой соревнования",
    },
    {
        "phase": 1, "name": "p1_huber_05",
        "input_size": 336,
        "loss": HuberLoss(delta=0.5),
        # delta в нормализованном пространстве (~0.5σ)
        # ARCH-эффект на 100% маршрутов → нужна устойчивость к выбросам
        "note": "Huber δ=0.5 — MAE с защитой от выбросов (ARCH effect)",
    },
    {
        "phase": 1, "name": "p1_huber_10",
        "input_size": 336,
        "loss": HuberLoss(delta=1.0),
        "note": "Huber δ=1.0 — менее агрессивная защита от выбросов",
    },
    {
        "phase": 1, "name": "p1_mq50",
        "input_size": 336,
        "loss": MQLoss(level=[50]),
        # level=[50] → оптимизируем только медиану (p=0.5)
        # Убираем p=0.1 и p=0.9 которые тянут в стороны
        "note": "MQLoss только медиана — убираем крайние квантили",
    },

    # ── Phase 2: input_size (loss = best из phase 1) ──────────
    # Заменить loss на победителя Phase 1 перед запуском Phase 2
    {
        "phase": 2, "name": "p2_input_96",
        "input_size": 96,        # 2 суток
        "loss": MAE(),           # ← замени на победителя Phase 1
        "note": "Только 2 дня истории — отсекаем дальний шум",
    },
    {
        "phase": 2, "name": "p2_input_168",
        "input_size": 168,       # 3.5 суток
        "loss": MAE(),
        "note": "3.5 дня — компромисс сезонность/шум",
    },
    {
        "phase": 2, "name": "p2_input_672",
        "input_size": 672,       # 2 недели
        "loss": MAE(),
        "note": "2 недели — ловим тренд длиннее",
    },
    {
        "phase": 2, "name": "p2_input_480",
        "input_size": 480,       # 10 дней
        "loss": MAE(),
        "note": "10 дней — между текущим (336) и 2 неделями",
    },

    # ── Phase 3: Combo (best_loss × best_input) ───────────────
    {
        "phase": 3, "name": "p3_combo_A",
        "input_size": 168,       # ← замени на победителя Phase 2
        "loss": MAE(),           # ← замени на победителя Phase 1
        "note": "Combo A: best_loss × best_input",
    },
    {
        "phase": 3, "name": "p3_combo_B",
        "input_size": 96,
        "loss": HuberLoss(delta=0.5),
        "note": "Combo B: альтернативная пара",
    },
]

# Датасет для CV
# Минимальная длина зависит от input_size конфига — берём максимальный
MAX_INPUT = max(cfg["input_size"] for cfg in EXPERIMENTS)
train_nf_cv = to_nf_df(
    train_df,
    min_len     = TOTAL_VAL_PTS + MAX_INPUT + 1,
    context_len = CONTEXT_LEN,
)
print(f"NF train: {train_nf_cv.shape}  routes: {train_nf_cv['unique_id'].nunique()}\n")

# ── Main loop ──
all_results   = []
best_so_far   = {"total_cal": PREV_BEST_TOTAL, "name": "prev_best (MQLoss80, global calib)"}
current_phase = None

for i, cfg in enumerate(EXPERIMENTS, 1):
    if cfg["phase"] != current_phase:
        current_phase = cfg["phase"]
        titles = {
            1: "PHASE 1 — LOSS FUNCTION  (input_size=336, arch=[8,4,2] fixed)",
            2: "PHASE 2 — INPUT SIZE  (loss=best_phase1, arch=[8,4,2] fixed)",
            3: "PHASE 3 — COMBO  (best_loss × best_input)",
        }
        print(f"\n{'='*65}")
        print(titles[current_phase])
        print("="*65)

    print(f"\n[{i:02d}/10] {cfg['name']}")
    print(f"  loss={cfg['loss'].__class__.__name__}  input_size={cfg['input_size']}")
    print(f"  {cfg['note']}")

    metrics = run_cv(cfg, train_nf_cv, CV_STEPS)

    for fr in metrics["fold_stats"]:
        print(f"  Fold {fr['fold']}  WAPE={fr['wape']:.4f}  "
              f"|RBias|={fr['rbias']:.4f}  Total={fr['total']:.4f}")

    # Основная метрика — total_cal (after global calib)
    print(f"\n  ── RAW   Total={metrics['total_raw']:.4f}  "
          f"WAPE={metrics['wape_raw']:.4f}  |RBias|={metrics['rbias_raw']:.4f}")
    print(f"  ── CAL   Total={metrics['total_cal']:.4f}  "
          f"WAPE={metrics['wape_cal']:.4f}  |RBias|={metrics['rbias_cal']:.4f}  "
          f"scale={metrics['calib_scale']:.4f}")

    row = {"#": i, "name": cfg["name"], "phase": cfg["phase"],
           "loss": cfg["loss"].__class__.__name__,
           "input_size": cfg["input_size"],
           **{k: v for k, v in metrics.items() if k != "fold_stats"}}
    all_results.append(row)

    if metrics["total_cal"] < best_so_far["total_cal"]:
        delta = metrics["total_cal"] - best_so_far["total_cal"]
        print(f"\n  🏆 NEW BEST!  Δ={delta:+.4f}  "
              f"(prev: {best_so_far['name']}={best_so_far['total_cal']:.4f})")
        best_so_far = {"name": cfg["name"], **metrics, "cfg": cfg}
    else:
        delta = metrics["total_cal"] - best_so_far["total_cal"]
        print(f"\n  Best: {best_so_far['name']}={best_so_far['total_cal']:.4f}  "
              f"(this={metrics['total_cal']:.4f}, Δ={delta:+.4f})")

# ── Leaderboard ──
results_df = (
    pd.DataFrame(all_results)
    .sort_values("total_cal")
    [["#","name","phase","loss","input_size",
      "wape_raw","rbias_raw","total_raw",
      "wape_cal","rbias_cal","total_cal","calib_scale"]]
)
results_df.to_csv("nhits_loss_input_results.csv", index=False)

print("\n\n" + "="*65)
print("LEADERBOARD  (CV_STEPS=400, total_cal)")
print("="*65)
print(results_df.to_string(index=False))
print(f"\n🏆 WINNER: {best_so_far['name']}  total_cal={best_so_far['total_cal']:.4f}")

# ──────────────────────────────────────────────────────────────
# WINNER — Full CV + Final predict
# ──────────────────────────────────────────────────────────────
best_cfg = best_so_far["cfg"]
print(f"\n{'='*65}")
print(f"WINNER FULL CV  (FINAL_STEPS={FINAL_STEPS})")
print("="*65)

winner_metrics = run_cv(best_cfg, train_nf_cv, FINAL_STEPS)
calib_scale = winner_metrics["calib_scale"]

for fr in winner_metrics["fold_stats"]:
    print(f"  Fold {fr['fold']}  WAPE={fr['wape']:.4f}  "
          f"|RBias|={fr['rbias']:.4f}  Total={fr['total']:.4f}")
print(f"\n── CAL  WAPE={winner_metrics['wape_cal']:.4f}  "
      f"|RBias|={winner_metrics['rbias_cal']:.4f}  "
      f"Total={winner_metrics['total_cal']:.4f}  "
      f"scale={calib_scale:.4f}")

pd.DataFrame([{"name": best_so_far["name"], "steps": FINAL_STEPS,
               **{k: v for k, v in winner_metrics.items() if k != "fold_stats"}}
              ]).to_csv("nhits_winner_loss_input_cv.csv", index=False)

# ── Final predict ──
print(f"\n{'='*65}")
print("FINAL PREDICT")
print("="*65)

train_nf_full  = to_nf_df(train_df, min_len=best_cfg["input_size"]+1, context_len=CONTEXT_LEN)
test_route_ids = test_df["route_id"].unique().tolist()
nf_full = NeuralForecast(models=[make_nhits(best_cfg, FINAL_STEPS)], freq=FREQ)
nf_full.fit(train_nf_full)

futr_df      = make_futr_df(test_route_ids, train_df)
test_pred_df = nf_full.predict(futr_df=futr_df)
test_pred_col = get_pred_col(test_pred_df)
test_preds    = build_preds_dict(test_pred_df, test_pred_col)

predictions_raw = {}
for route_id in tqdm(test_route_ids, desc="Build submission"):
    route_test = test_df[test_df["route_id"]==route_id].sort_values("timestamp")
    preds = test_preds.get(route_id, np.zeros(FORECAST_STEPS))
    for j, (_, row) in enumerate(route_test.iterrows()):
        predictions_raw[row["id"]] = max(0.0,
            float(preds[j]) if j < len(preds) else float(preds[-1]))

name     = best_so_far["name"]
sub_raw  = pd.DataFrame(list(predictions_raw.items()), columns=["id","y_pred"]) \
             .sort_values("id").reset_index(drop=True)
sub_cal  = sub_raw.copy()
sub_cal["y_pred"] = np.clip(sub_cal["y_pred"] * calib_scale, 0, None)

sub_raw.to_csv(f"submission_{name}_raw.csv",  index=False)
sub_cal.to_csv(f"submission_{name}_cal.csv",  index=False)

print(f"\n✅ nhits_loss_input_results.csv")
print(f"✅ nhits_winner_loss_input_cv.csv")
print(f"✅ submission_{name}_raw.csv")
print(f"✅ submission_{name}_cal.csv  (scale={calib_scale:.4f})")
print(f"\n── Raw ──\n{sub_raw['y_pred'].describe().round(2)}")
print(f"\n── Calibrated ──\n{sub_cal['y_pred'].describe().round(2)}")

In [ ]:
# ============================================================
# Оптимизированный routing grid search
# Всё тяжёлое — ONE-TIME precomputation
# Grid loop — чистый numpy, ~мс на итерацию
# ============================================================
import warnings, gc
import numpy as np
import pandas as pd
import torch
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MAE, MQLoss, HuberLoss
warnings.filterwarnings("ignore")

# ── Setup (те же константы) ──
TRAIN_PATH     = "train_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8
N_FOLDS        = 5
TOTAL_VAL_PTS  = N_FOLDS * FORECAST_STEPS
CONTEXT_LEN    = 2048
FREQ           = "30min"
BATCH_SIZE     = 64
CV_STEPS       = 400
FINAL_STEPS    = 800

if torch.backends.mps.is_available():   ACCELERATOR = "mps"
elif torch.cuda.is_available():         ACCELERATOR = "gpu"
else:                                   ACCELERATOR = "cpu"

train_df    = pd.read_parquet(TRAIN_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
train_df    = train_df.sort_values(["route_id","timestamp"]).reset_index(drop=True)
status_cols = sorted([c for c in train_df.columns
                      if c in ["status_2","status_3","status_5"]])
FUTR_EXOG   = ["hour_sin","hour_cos","dow_sin","dow_cos","is_weekend"]
BASE_HIST   = status_cols + ["lag_48","lag_336"]

def add_calendar_features(df, ts_col="ds"):
    ts = pd.to_datetime(df[ts_col]); df = df.copy()
    df["hour_sin"]   = np.sin(2*np.pi*ts.dt.hour/24)
    df["hour_cos"]   = np.cos(2*np.pi*ts.dt.hour/24)
    df["dow_sin"]    = np.sin(2*np.pi*ts.dt.dayofweek/7)
    df["dow_cos"]    = np.cos(2*np.pi*ts.dt.dayofweek/7)
    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(float)
    return df

def add_lag_features(nf_df):
    nf_df = nf_df.sort_values(["unique_id","ds"]).copy()
    for lag in [48, 336]:
        nf_df[f"lag_{lag}"] = (
            nf_df.groupby("unique_id")["y"]
            .transform(lambda s, l=lag: s.shift(l).bfill().fillna(0))
        )
    return nf_df

def to_nf_df(df, min_len=32, context_len=CONTEXT_LEN):
    rows = []
    for route_id, grp in df.groupby("route_id"):
        grp = (grp.sort_values("timestamp")
               .set_index("timestamp")[[TARGET_COL]+status_cols]
               .asfreq(FREQ).interpolate(method="time")
               .bfill().ffill().tail(context_len).reset_index())
        if len(grp) < min_len: continue
        grp.insert(0, "unique_id", route_id)
        grp = grp.rename(columns={"timestamp":"ds", TARGET_COL:"y"})
        rows.append(grp)
    nf_df = pd.concat(rows, ignore_index=True)
    nf_df = add_calendar_features(nf_df)
    nf_df = add_lag_features(nf_df)
    return nf_df

def get_pred_col(pred_df):
    for c in ["NHITS-median","NHITS-q-0.50","NHITS"]:
        if c in pred_df.columns: return c
    skip = {"unique_id","ds","y","cutoff"}
    num  = [c for c in pred_df.columns if c not in skip
            and pd.api.types.is_numeric_dtype(pred_df[c])]
    med  = [c for c in num if any(x in c for x in ["median","0.5","50"])]
    return (med or num)[0]

def make_nhits(loss, input_size, max_steps):
    return NHITS(
        h=FORECAST_STEPS, input_size=input_size, loss=loss,
        n_freq_downsample=[48,8,1], n_pool_kernel_size=[1,1,1],
        n_blocks=[8,4,2], mlp_units=[[512,512]]*3,
        max_steps=max_steps, batch_size=BATCH_SIZE,
        accelerator=ACCELERATOR, hist_exog_list=BASE_HIST,
        futr_exog_list=FUTR_EXOG, scaler_type="robust",
        enable_progress_bar=True,
    )

def run_cv_with_raw(model_name, loss, input_size, train_nf, max_steps):
    nf    = NeuralForecast(models=[make_nhits(loss, input_size, max_steps)], freq=FREQ)
    cv_df = nf.cross_validation(
        df=train_nf, n_windows=N_FOLDS, step_size=FORECAST_STEPS, refit=True
    )
    pred_col = get_pred_col(cv_df)
    cuts = sorted(cv_df["cutoff"].unique())
    cv_df["fold"] = cv_df["cutoff"].map({c: i+1 for i, c in enumerate(cuts)})
    raw = cv_df[["fold","unique_id","ds","y",pred_col]].copy()
    raw = raw.rename(columns={"unique_id":"route_id","ds":"timestamp",
                               "y":"y_true", pred_col:"y_pred"})
    raw["y_pred"] = np.clip(raw["y_pred"].values, 0, None)
    raw.to_csv(f"nhits_cv_raw_{model_name}.csv", index=False)
    del nf, cv_df; gc.collect()
    return raw

# ── Загрузка/обучение моделей ──
CANDIDATE_MODELS = [
    {"name": "mae_336",     "loss": MAE(),               "input_size": 336},
    {"name": "huber05_336", "loss": HuberLoss(delta=0.5), "input_size": 336},
    {"name": "mae_168",     "loss": MAE(),               "input_size": 168},
    {"name": "mae_96",      "loss": MAE(),               "input_size": 96},
]

all_raw = {"model_A": pd.read_csv("nhits_cv_raw_p2_blk_8_4_2.csv")}

MAX_INPUT = max(m["input_size"] for m in CANDIDATE_MODELS)
train_nf  = to_nf_df(train_df, min_len=TOTAL_VAL_PTS+MAX_INPUT+1,
                     context_len=CONTEXT_LEN)

for cfg in CANDIDATE_MODELS:
    fname = f"nhits_cv_raw_{cfg['name']}.csv"
    import os
    if os.path.exists(fname):
        print(f"Loading cached: {fname}")
        all_raw[cfg["name"]] = pd.read_csv(fname)
    else:
        print(f"\nTraining {cfg['name']}...")
        all_raw[cfg["name"]] = run_cv_with_raw(
            cfg["name"], cfg["loss"], cfg["input_size"], train_nf, CV_STEPS
        )



In [ ]:
  Fold 1  WAPE=0.3199  |RBias|=0.0438  Total=0.3637
  Fold 2  WAPE=0.3342  |RBias|=0.0357  Total=0.3700
  Fold 3  WAPE=0.3232  |RBias|=0.0270  Total=0.3502
  Fold 4  WAPE=0.3209  |RBias|=0.0078  Total=0.3287
  Fold 5  WAPE=0.3331  |RBias|=0.0359  Total=0.3690

── RAW    WAPE=0.3261  |RBias|=0.0302  Total=0.3563
── CALIB  WAPE=0.3274  |RBias|=0.0000  Total=0.3274  scale=1.0311

In [ ]:
# ============================================================
# Chronos-2 vs N-HiTS — per-route comparison
# Цель: найти маршруты где Chronos стабильно лучше
#
# Гипотезы:
#  H1: Короткая история (мало точек) → Chronos zero-shot лучше
#  H2: Sparse/intermittent (high zero_frac) → Chronos устойчивее
#  H3: Структурный сдвиг (growing/falling) → Chronos адаптивнее
#  H4: Низкая ACF (хаотичный ряд) → оба плохи, но разный bias
#  H5: Маленький маршрут (low mean_y) → мало сигнала для N-HiTS
#  H6: Высокий peak_ratio → разный способ обработки пиков
# ============================================================
import warnings, os
import numpy as np
import pandas as pd
import torch
from statsmodels.tsa.stattools import acf
from tqdm import tqdm
warnings.filterwarnings("ignore")

os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"]  = "1"
os.environ["HF_HUB_OFFLINE"]       = "1"

TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8
N_FOLDS        = 5
TOTAL_VAL_PTS  = N_FOLDS * FORECAST_STEPS
FREQ           = "30min"
CTX_LEN        = 2048   # победитель из sweep
SHRINKAGE_K    = 256
SCALE_MIN, SCALE_MAX = 0.5, 2.0

if torch.backends.mps.is_available(): DEVICE = "mps"
elif torch.cuda.is_available():       DEVICE = "cuda"
else:                                 DEVICE = "cpu"

# ── Load data ──
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])
train_df = train_df.sort_values(["route_id","timestamp"]).reset_index(drop=True)
status_cols = sorted([c for c in train_df.columns
                      if c in ["status_2","status_3","status_5"]])
EXTRA_COLS = status_cols + ["lag_48","lag_336",
                             "hour_sin","hour_cos","dow_sin","dow_cos","is_weekend"]

def add_all_features(df):
    df = df.copy().sort_values("timestamp")
    y = df["target"].values.astype(float)
    for lag in [48, 336]:
        lagged = np.full(len(y), np.nan)
        if lag < len(y): lagged[lag:] = y[:-lag]
        mean_y = np.nanmean(lagged) if not np.all(np.isnan(lagged)) else 0.0
        df[f"lag_{lag}"] = np.where(np.isnan(lagged), mean_y, lagged)
    ts = df["timestamp"]
    df["hour_sin"]   = np.sin(2*np.pi*ts.dt.hour/24)
    df["hour_cos"]   = np.cos(2*np.pi*ts.dt.hour/24)
    df["dow_sin"]    = np.sin(2*np.pi*ts.dt.dayofweek/7)
    df["dow_cos"]    = np.cos(2*np.pi*ts.dt.dayofweek/7)
    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(float)
    return df

def prepare_context(route_df, route_id, ctx_len):
    df = (route_df.set_index("timestamp")[[TARGET_COL]+status_cols]
          .asfreq(FREQ).interpolate(method="time").bfill().ffill()
          .tail(ctx_len).reset_index())
    df.insert(0, "id", route_id)
    df = df.rename(columns={TARGET_COL:"target"})
    df = add_all_features(df)
    return df[["id","timestamp","target"]+EXTRA_COLS]

def get_pred_col(pred_df):
    for c in ["0.5","mean","median","q0.5"]:
        if c in pred_df.columns: return c
    num = [c for c in pred_df.columns
           if c not in ("id","timestamp") and pd.api.types.is_numeric_dtype(pred_df[c])]
    return num[0]

# ============================================================
# STEP 1: Chronos CV с правильным per-route сохранением
# ============================================================
CHRONOS_RAW_FILE = "chronos2_cv_raw_2048.csv"

if os.path.exists(CHRONOS_RAW_FILE):
    print(f"Loading cached Chronos CV: {CHRONOS_RAW_FILE}")
    chronos_raw = pd.read_csv(CHRONOS_RAW_FILE)
else:
    from chronos import Chronos2Pipeline
    pipeline = Chronos2Pipeline.from_pretrained(
        "amazon/chronos-2", device_map=DEVICE,
        torch_dtype=torch.bfloat16, local_files_only=True,
    )
    print("Chronos-2 loaded\n")

    raw_rows = []
    for fold in range(N_FOLDS):
        val_end   = TOTAL_VAL_PTS - fold * FORECAST_STEPS
        val_start = val_end - FORECAST_STEPS

        context_dfs, y_trues_list, route_ids_fold = [], [], []
        for route_id, grp in train_df.groupby("route_id"):
            grp = grp.sort_values("timestamp").reset_index(drop=True)
            if len(grp) < TOTAL_VAL_PTS + 32: continue
            hist = grp.iloc[:-val_end]
            val  = grp.iloc[-val_end:-val_start] if val_start > 0 else grp.iloc[-val_end:]
            if len(hist) < 32 or len(val) == 0: continue
            context_dfs.append(prepare_context(hist, route_id, CTX_LEN))
            y_trues_list.append(val[TARGET_COL].values[:FORECAST_STEPS])
            route_ids_fold.append(route_id)

        ctx = pd.concat(context_dfs, ignore_index=True)
        print(f"Fold {fold+1}  routes={len(route_ids_fold)}", end="  ")

        pred_df  = pipeline.predict_df(
            ctx, prediction_length=FORECAST_STEPS,
            quantile_levels=[0.1,0.5,0.9],
            id_column="id", timestamp_column="timestamp",
            target="target", cross_learning=True,
        )
        pred_col = get_pred_col(pred_df)
        preds_fold = {rid: np.clip(g.sort_values("timestamp")[pred_col].values, 0, None)
                      for rid, g in pred_df.groupby("id")}

        yt_all = np.concatenate(y_trues_list)
        yp_all = np.concatenate([preds_fold.get(r, np.zeros(len(t)))[:len(t)]
                                  for r, t in zip(route_ids_fold, y_trues_list)])
        s = yt_all.sum() + 1e-9
        w = np.abs(yp_all-yt_all).sum()/s
        print(f"WAPE={w:.4f}")

        # ── Сохраняем per-route ──
        for rid, yt_arr in zip(route_ids_fold, y_trues_list):
            yp_arr = preds_fold.get(rid, np.zeros(len(yt_arr)))[:len(yt_arr)]
            for step, (yti, ypi) in enumerate(zip(yt_arr, yp_arr)):
                raw_rows.append({"fold": fold+1, "route_id": rid,
                                  "step": step, "y_true": float(yti),
                                  "y_pred": float(ypi)})

    chronos_raw = pd.DataFrame(raw_rows)
    chronos_raw.to_csv(CHRONOS_RAW_FILE, index=False)
    print(f"\n✅ {CHRONOS_RAW_FILE}")
    del pipeline; import gc; gc.collect()

# ============================================================
# STEP 2: Загружаем N-HiTS raw + global scale
# ============================================================
nhits_raw = pd.read_csv("nhits_cv_raw_p2_blk_8_4_2.csv")

nhits_gs = nhits_raw["y_true"].sum() / (nhits_raw["y_pred"].sum() + 1e-9)
nhits_raw["y_pred_cal"] = np.clip(nhits_raw["y_pred"] * nhits_gs, 0, None)

chronos_gs = chronos_raw["y_true"].sum() / (chronos_raw["y_pred"].sum() + 1e-9)
chronos_raw["y_pred_cal"] = np.clip(chronos_raw["y_pred"] * chronos_gs, 0, None)

def global_wape_rbias(df):
    yt = df["y_true"].values; yp = df["y_pred_cal"].values
    s = yt.sum()+1e-9
    return np.abs(yp-yt).sum()/s, np.abs(yp.sum()/s-1)

nh_w, nh_r = global_wape_rbias(nhits_raw)
ch_w, ch_r = global_wape_rbias(chronos_raw)
print(f"\nGlobal (calibrated):")
print(f"  N-HiTS:   WAPE={nh_w:.5f}  |RBias|={nh_r:.5f}  Total={nh_w+nh_r:.5f}")
print(f"  Chronos:  WAPE={ch_w:.5f}  |RBias|={ch_r:.5f}  Total={ch_w+ch_r:.5f}")

# ============================================================
# STEP 3: Per-route SAE сравнение
# ============================================================
def per_route_sae(df, model_name):
    return (df.groupby("route_id")
            .apply(lambda g: pd.Series({
                f"sae_{model_name}":  np.abs(g["y_pred_cal"]-g["y_true"]).sum(),
                f"mae_{model_name}":  np.abs(g["y_pred_cal"]-g["y_true"]).mean(),
                f"bias_{model_name}": (g["y_pred_cal"]-g["y_true"]).mean(),
                "sum_true":           g["y_true"].sum(),
                "n_obs":              len(g),
            }))
            .reset_index())

nh_stats  = per_route_sae(nhits_raw,   "nhits")
ch_stats  = per_route_sae(chronos_raw, "chron")
# sum_true одинаков — берём из nhits
comp = nh_stats.merge(
    ch_stats[["route_id","sae_chron","mae_chron","bias_chron"]],
    on="route_id", how="inner"
)
comp["sae_ratio"]  = comp["sae_chron"] / (comp["sae_nhits"] + 1e-9)
# < 1 = Chronos лучше, > 1 = N-HiTS лучше
comp["chron_wins"] = comp["sae_ratio"] < 1.0
comp["chron_wins_strong"] = comp["sae_ratio"] < 0.9   # >10% лучше

total_routes = len(comp)
n_chron_wins = comp["chron_wins"].sum()
n_chron_str  = comp["chron_wins_strong"].sum()
print(f"\nPer-route wins:")
print(f"  Chronos лучше N-HiTS:        {n_chron_wins}/{total_routes} "
      f"({n_chron_wins/total_routes*100:.1f}%)")
print(f"  Chronos лучше >10% (strong):  {n_chron_str}/{total_routes} "
      f"({n_chron_str/total_routes*100:.1f}%)")

# SAE вклад маршрутов где Chronos лучше
sae_chronos_wins = comp.loc[comp["chron_wins"], "sae_chron"].sum()
sae_nhits_wins   = comp.loc[comp["chron_wins"], "sae_nhits"].sum()
total_sae_nh     = comp["sae_nhits"].sum()
print(f"\n  Суммарный SAE на маршрутах где Chronos лучше:")
print(f"    N-HiTS SAE:   {sae_nhits_wins:.0f} ({sae_nhits_wins/total_sae_nh*100:.1f}% от общего)")
print(f"    Chronos SAE:  {sae_chronos_wins:.0f}")
print(f"    Выигрыш:      {sae_nhits_wins-sae_chronos_wins:.0f}")

# ============================================================
# STEP 4: Характеристики рядов для тестирования гипотез
# ============================================================
print("\nComputing route features for hypothesis testing...")
feat_rows = {}
for route_id, grp in train_df.groupby("route_id"):
    y = grp["target_1h"].values.astype(float)
    n = len(y)
    if n < 96: continue
    try:    acf1_val = acf(y, nlags=1, fft=True)[1]
    except: acf1_val = 0.0
    s = pd.Series(y)
    half = n // 2
    feat_rows[route_id] = {
        "n_history":     n,
        "mean_y":        y.mean(),
        "cv":            y.std() / (y.mean() + 1e-9),
        "zero_frac":     (y == 0).mean(),
        "acf1":          acf1_val,
        "roll_cv":       (s.rolling(48).std()/(s.rolling(48).mean()+1e-9)).mean(),
        "growth_recent": y[-48:].mean() / (y.mean() + 1e-9),
        "peak_ratio":    np.percentile(y,99) / (np.percentile(y,50) + 1e-9),
        # Нерегулярность: std скользящего среднего / глобальное среднее
        "level_instab":  s.rolling(96).mean().std() / (y.mean() + 1e-9),
    }

feat_df = pd.DataFrame(feat_rows).T.reset_index().rename(columns={"index":"route_id"})
feat_df["route_id"] = feat_df["route_id"].astype(int)

analysis = comp.merge(feat_df, on="route_id", how="left")



In [ ]:
import os
import ssl

# Отключить проверку SSL для huggingface_hub
os.environ["HF_HUB_DISABLE_SSL_VERIFICATION"] = "1"

# Отключить для httpx (который использует huggingface_hub)
import httpx
original_init = httpx.Client.__init__

def patched_init(self, *args, **kwargs):
    kwargs["verify"] = False
    original_init(self, *args, **kwargs)

httpx.Client.__init__ = patched_init

# Глобально для ssl
ssl._create_default_https_context = ssl._create_unverified_context

In [ ]:
# Заменить блок Step 5 (корреляции) на этот:

print("\n── Корреляция характеристик ряда с sae_ratio (Spearman) ──")
print("   sae_ratio < 1 = Chronos лучше, > 1 = N-HiTS лучше\n")

corr_rows = []
for col in feat_cols:
    sub = analysis[["sae_ratio", col]].dropna()
    if len(sub) < 10 or sub[col].nunique() < 3:
        r = float("nan")
    else:
        r = sub.corr(method="spearman").iloc[0, 1]
    corr_rows.append({"feature": col, "spearman_r": r})

    if np.isnan(r):
        print(f"  {col:<18}  NaN (недостаточно вариации)")
        continue

    bar = "█" * int(abs(r) * 40)
    sign = "-" if r < 0 else "+"
    direction = "→ Chronos лучше при высоком" if r < 0 else "→ N-HiTS лучше при высоком"
    print(f"  {col:<18} {sign}{abs(r):.3f}  {bar}  {direction}")

corr_df = pd.DataFrame(corr_rows).sort_values("spearman_r")

# Топ фичи (без NaN)
top_neg = (corr_df[corr_df["spearman_r"].notna() & (corr_df["spearman_r"] < 0)]
           .nsmallest(2, "spearman_r")["feature"].tolist())
top_pos = (corr_df[corr_df["spearman_r"].notna() & (corr_df["spearman_r"] > 0)]
           .nlargest(2, "spearman_r")["feature"].tolist())

print(f"\n── Квартильный разбор по ключевым фичам ──")
for feat in top_neg + top_pos:
    analysis[f"q_{feat}"] = pd.qcut(analysis[feat], q=4, labels=["Q1","Q2","Q3","Q4"],
                                      duplicates="drop")
    q_stats = analysis.groupby(f"q_{feat}").agg(
        n_chron_wins=("chron_wins","sum"),
        n_total=("chron_wins","count"),
        mean_sae_ratio=("sae_ratio","mean"),
        median_sae_ratio=("sae_ratio","median"),
    ).reset_index()
    q_stats["win_pct"] = q_stats["n_chron_wins"] / q_stats["n_total"] * 100
    direction = "(Chronos лучше при низком)" if feat in top_neg else "(Chronos лучше при высоком)"
    print(f"\n  {feat} {direction}:")
    print(q_stats[[f"q_{feat}","n_chron_wins","n_total","win_pct","median_sae_ratio"]]
          .round(3).to_string(index=False))

# ============================================================
# STEP 6: Профиль "Chronos stronghold" маршрутов
# ============================================================
# Маршруты где Chronos стабильно лучше — проверяем по фолдам
print("\n\n── Стабильность побед Chronos (по фолдам) ──")
fold_sae = {}
for model_df, mname in [(nhits_raw,"nhits"), (chronos_raw,"chron")]:
    fold_sae[mname] = (
        model_df.groupby(["route_id","fold"])
        .apply(lambda g: np.abs(g["y_pred_cal"]-g["y_true"]).sum())
        .reset_index().rename(columns={0: f"sae_{mname}"})
    )
fold_both = fold_sae["nhits"].merge(fold_sae["chron"], on=["route_id","fold"])
fold_both["chron_wins_fold"] = fold_both["sae_chron"] < fold_both["sae_nhits"]

route_stability = (fold_both.groupby("route_id")
                   .agg(n_folds_chron_wins=("chron_wins_fold","sum"),
                        n_folds=("chron_wins_fold","count"))
                   .reset_index())
route_stability["stability"] = (route_stability["n_folds_chron_wins"]
                                 / route_stability["n_folds"])

for min_stab in [1.0, 0.8, 0.6]:
    stable = route_stability[route_stability["stability"] >= min_stab]
    stable_analysis = stable.merge(analysis, on="route_id")
    sae_gain = (stable_analysis["sae_nhits"].sum() -
                stable_analysis["sae_chron"].sum())
    pct = sae_gain / total_sae_nh * 100
    print(f"  Chronos wins в {min_stab*100:.0f}%+ фолдов: "
          f"{len(stable)} маршрутов  "
          f"SAE gain={sae_gain:.0f}  ({pct:.2f}% глобального SAE)")

# Топ стабильных маршрутов Chronos
stable_100 = route_stability[route_stability["stability"]==1.0].merge(analysis, on="route_id")
print(f"\nТоп-20 маршрутов где Chronos стабильно лучше (все 5 фолдов):")
cols = ["route_id","sae_nhits","sae_chron","sae_ratio","n_history",
        "mean_y","cv","zero_frac","growth_recent","acf1"]
print(stable_100.sort_values("sae_ratio")[cols].head(20).round(3).to_string(index=False))

print(f"\n── Средний профиль: Chronos-always-win vs N-HiTS-always-win ──")
nh_stable = route_stability[route_stability["stability"]==0.0].merge(analysis, on="route_id")
feat_profile = feat_cols
print(f"\n{'Feature':<18} {'Chron-wins(all5)':>18} {'NHiTS-wins(all5)':>18}  {'Δ (C-N)':>10}")
print("─"*68)
for f in feat_profile:
    c_mean = stable_100[f].mean() if len(stable_100) > 0 else float("nan")
    n_mean = nh_stable[f].mean()  if len(nh_stable)  > 0 else float("nan")
    delta  = c_mean - n_mean
    print(f"  {f:<16} {c_mean:>18.3f} {n_mean:>18.3f}  {delta:>+10.3f}")

# ============================================================
# STEP 7: Сохраняем routing таблицу
# ============================================================
route_routing = route_stability.merge(analysis[["route_id","sae_ratio"]+feat_cols],
                                       on="route_id", how="left")
route_routing.to_csv("chronos_nhits_routing_table.csv", index=False)
print(f"\n✅ chronos_nhits_routing_table.csv")

# Потенциал блендинга: если брать Chronos только на stable_100 маршрутах
blend_sae = (comp["sae_nhits"].sum()
             - stable_100["sae_nhits"].sum()
             + stable_100["sae_chron"].sum())
blend_wape_approx = blend_sae / comp["sum_true"].sum()
print(f"\n── Теоретический потенциал роутинга ──")
print(f"  Текущий N-HiTS (global cal) WAPE: {nh_w:.5f}")
print(f"  Если Chronos на {len(stable_100)} стабильных маршрутах: "
      f"WAPE≈{blend_wape_approx:.5f}  "
      f"Δ={blend_wape_approx-nh_w:+.5f}")

In [ ]:
# ============================================================
# NHITS horizon sweep for winner config: p2_blk_8_4_2
# - sweep horizons: 2, 4, 8, 16, 32, 48, 336
# - 3-fold chained CV with step_size = horizon
# - reduced max_steps for fast comparison
# - optional predict() only when test horizon matches model horizon
# ============================================================

import warnings, gc
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MQLoss

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

# ── Metric ──
def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias

# ── Config ──
TRAIN_PATH      = "train_solo_track.parquet"
TEST_PATH       = "test_solo_track.parquet"
TARGET_COL      = "target_1h"
FREQ            = "30min"
CONTEXT_LEN     = 2048
BASE_INPUT_SIZE = 336
BATCH_SIZE      = 64

HORIZONS  = [2, 4, 8, 16, 32, 48, 48 * 7]
N_FOLDS   = 3
CV_STEPS  = 200          # быстрый прогон
FULL_STEPS_FOR_TEST = 400  # только если horizon совпадает с test horizon

BEST_CFG = {
    "name": "p2_blk_8_4_2",
    "n_freq_downsample": [48, 8, 1],
    "n_pool_kernel_size": [1, 1, 1],
    "n_blocks": [8, 4, 2],
}

if torch.backends.mps.is_available():
    ACCELERATOR = "mps"
elif torch.cuda.is_available():
    ACCELERATOR = "gpu"
else:
    ACCELERATOR = "cpu"
print(f"Device: {ACCELERATOR}")

# ── Data ──
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)

train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])

train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

status_cols = sorted([c for c in train_df.columns if c in ["status_2", "status_3", "status_5"]])

FUTR_EXOG = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "is_weekend"]
HIST_EXOG = (status_cols + ["lag_48", "lag_336"]) or None

def add_calendar_features(df, ts_col="ds"):
    ts = pd.to_datetime(df[ts_col])
    df = df.copy()
    df["hour_sin"]   = np.sin(2 * np.pi * ts.dt.hour / 24)
    df["hour_cos"]   = np.cos(2 * np.pi * ts.dt.hour / 24)
    df["dow_sin"]    = np.sin(2 * np.pi * ts.dt.dayofweek / 7)
    df["dow_cos"]    = np.cos(2 * np.pi * ts.dt.dayofweek / 7)
    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(float)
    return df

def add_lag_features(nf_df):
    nf_df = nf_df.sort_values(["unique_id", "ds"]).copy()
    for lag in [48, 336]:
        nf_df[f"lag_{lag}"] = (
            nf_df.groupby("unique_id")["y"]
            .transform(lambda s, l=lag: s.shift(l).bfill().fillna(0))
        )
    return nf_df

def to_nf_df(df, min_len=32, context_len=CONTEXT_LEN):
    rows = []
    for route_id, grp in df.groupby("route_id"):
        grp = (
            grp.sort_values("timestamp")
            .set_index("timestamp")[[TARGET_COL] + status_cols]
            .asfreq(FREQ).interpolate(method="time")
            .bfill().ffill().tail(context_len).reset_index()
        )
        if len(grp) < min_len:
            continue
        grp.insert(0, "unique_id", route_id)
        grp = grp.rename(columns={"timestamp": "ds", TARGET_COL: "y"})
        rows.append(grp)

    if not rows:
        return pd.DataFrame(columns=["unique_id", "ds", "y"] + status_cols)

    nf_df = pd.concat(rows, ignore_index=True)
    nf_df = add_calendar_features(nf_df)
    nf_df = add_lag_features(nf_df)
    return nf_df

def make_futr_df(route_ids, train_df, horizon):
    rows = []
    for rid in route_ids:
        last_ts = train_df.loc[train_df["route_id"] == rid, "timestamp"].max()
        for ts in pd.date_range(
            start=last_ts + pd.Timedelta("30min"),
            periods=horizon,
            freq=FREQ
        ):
            rows.append({"unique_id": rid, "ds": ts})
    return add_calendar_features(pd.DataFrame(rows))

def get_pred_col(pred_df):
    for c in ["NHITS-median", "NHITS-q-0.50", "NHITS"]:
        if c in pred_df.columns:
            return c
    skip = {"unique_id", "ds", "y", "cutoff"}
    num = [c for c in pred_df.columns if c not in skip and pd.api.types.is_numeric_dtype(pred_df[c])]
    med = [c for c in num if any(x in c for x in ["median", "0.5", "50"])]
    return (med or num)[0]

def build_preds_dict(pred_df, pred_col):
    return {
        rid: np.clip(g.sort_values("ds")[pred_col].values, 0, None)
        for rid, g in pred_df.groupby("unique_id")
    }

def resolve_input_size(horizon):
    # Чистый sweep по горизонту: input_size держим фиксированным.
    # Если захочешь "лучшую модель на каждом горизонте", попробуй:
    # return min(CONTEXT_LEN, max(BASE_INPUT_SIZE, 2 * horizon))
    return BASE_INPUT_SIZE

def make_nhits(cfg, horizon, input_size, max_steps):
    return NHITS(
        h=horizon,
        input_size=input_size,
        loss=MQLoss(level=[80]),
        n_freq_downsample=cfg["n_freq_downsample"],
        n_pool_kernel_size=cfg["n_pool_kernel_size"],
        n_blocks=cfg["n_blocks"],
        mlp_units=[[512, 512]] * len(cfg["n_blocks"]),
        max_steps=max_steps,
        batch_size=BATCH_SIZE,
        accelerator=ACCELERATOR,
        hist_exog_list=HIST_EXOG,
        futr_exog_list=FUTR_EXOG,
        scaler_type="robust",
        enable_progress_bar=False,
    )

def run_cv(cfg, train_nf, horizon, input_size, max_steps, n_folds):
    nf = NeuralForecast(
        models=[make_nhits(cfg, horizon, input_size, max_steps)],
        freq=FREQ
    )
    cv_df = nf.cross_validation(
        df=train_nf,
        n_windows=n_folds,
        step_size=horizon,
        refit=True
    )

    pred_col = get_pred_col(cv_df)

    cuts = sorted(cv_df["cutoff"].unique())
    cv_df["fold"] = cv_df["cutoff"].map({c: i + 1 for i, c in enumerate(cuts)})
    cv_df = cv_df.sort_values(["fold", "unique_id", "ds"]).reset_index(drop=True)
    cv_df["h_step"] = cv_df.groupby(["fold", "unique_id"]).cumcount() + 1

    yt = cv_df["y"].values
    yp = np.clip(cv_df[pred_col].values, 0, None)

    tot, wape, rb = wape_rbias(yt, yp)

    calib = float(yt.sum() / (yp.sum() + 1e-9))
    yp_c = np.clip(yp * calib, 0, None)
    tot_c, wape_c, rb_c = wape_rbias(yt, yp_c)

    fold_rows = []
    for fold in sorted(cv_df["fold"].unique()):
        fdf = cv_df[cv_df["fold"] == fold]
        t, w, r = wape_rbias(fdf["y"].values, np.clip(fdf[pred_col].values, 0, None))
        fold_rows.append({
            "fold": int(fold),
            "wape": round(w, 4),
            "rbias": round(r, 4),
            "total": round(t, 4),
        })

    del nf, cv_df
    gc.collect()

    return {
        "wape_raw": round(wape, 4),
        "rbias_raw": round(rb, 4),
        "total_raw": round(tot, 4),
        "wape_cal": round(wape_c, 4),
        "rbias_cal": round(rb_c, 4),
        "total_cal": round(tot_c, 4),
        "calib_scale": round(calib, 4),
        "fold_stats": fold_rows,
    }

def maybe_predict_test(cfg, horizon, input_size, max_steps):
    route_sizes = test_df.groupby("route_id").size()
    if route_sizes.nunique() != 1:
        print(f"Skip predict for h={horizon}: test has non-uniform route lengths")
        return None

    test_horizon = int(route_sizes.iloc[0])
    if test_horizon != horizon:
        print(f"Skip predict for h={horizon}: test horizon is {test_horizon}")
        return None

    print(f"Running predict() for h={horizon} ...")

    train_nf_full = to_nf_df(train_df, min_len=input_size + 1, context_len=CONTEXT_LEN)
    test_route_ids = test_df["route_id"].unique().tolist()

    nf_full = NeuralForecast(
        models=[make_nhits(cfg, horizon, input_size, max_steps)],
        freq=FREQ
    )
    nf_full.fit(train_nf_full)

    futr_df = make_futr_df(test_route_ids, train_df, horizon)
    test_pred_df = nf_full.predict(futr_df=futr_df)
    pred_col = get_pred_col(test_pred_df)
    test_preds = build_preds_dict(test_pred_df, pred_col)

    predictions_raw = {}
    for route_id in test_route_ids:
        route_test = test_df[test_df["route_id"] == route_id].sort_values("timestamp")
        preds = test_preds.get(route_id, np.zeros(horizon))
        for j, (_, row) in enumerate(route_test.iterrows()):
            pred = float(preds[j]) if j < len(preds) else float(preds[-1])
            predictions_raw[row["id"]] = max(0.0, pred)

    submission = (
        pd.DataFrame(list(predictions_raw.items()), columns=["id", "y_pred"])
        .sort_values("id")
        .reset_index(drop=True)
    )
    file_name = f"submission_{cfg['name']}_h{horizon}.csv"
    submission.to_csv(file_name, index=False)
    print(f"Saved: {file_name}")
    return file_name

# ── Main sweep ──
all_results = []
all_fold_rows = []

for horizon in HORIZONS:
    input_size = resolve_input_size(horizon)
    min_len = input_size + N_FOLDS * horizon + 1

    train_nf = to_nf_df(
        train_df,
        min_len=min_len,
        context_len=CONTEXT_LEN,
    )

    n_series = train_nf["unique_id"].nunique()
    n_rows = len(train_nf)

    print("\n" + "=" * 70)
    print(f"HORIZON = {horizon}")
    print(f"input_size={input_size} | min_len={min_len} | series={n_series} | rows={n_rows}")
    print("=" * 70)

    if n_series == 0:
        print("No eligible series, skipping.")
        continue

    metrics = run_cv(
        cfg=BEST_CFG,
        train_nf=train_nf,
        horizon=horizon,
        input_size=input_size,
        max_steps=CV_STEPS,
        n_folds=N_FOLDS,
    )

    for fr in metrics["fold_stats"]:
        print(f"Fold {fr['fold']}: WAPE={fr['wape']:.4f} |RBias|={fr['rbias']:.4f} Total={fr['total']:.4f}")
        all_fold_rows.append({
            "horizon": horizon,
            "fold": fr["fold"],
            "wape": fr["wape"],
            "rbias": fr["rbias"],
            "total": fr["total"],
        })

    print(f"RAW   : WAPE={metrics['wape_raw']:.4f} |RBias|={metrics['rbias_raw']:.4f} Total={metrics['total_raw']:.4f}")
    print(f"CALIB : WAPE={metrics['wape_cal']:.4f} |RBias|={metrics['rbias_cal']:.4f} Total={metrics['total_cal']:.4f} scale={metrics['calib_scale']:.4f}")

    submission_file = maybe_predict_test(
        cfg=BEST_CFG,
        horizon=horizon,
        input_size=input_size,
        max_steps=FULL_STEPS_FOR_TEST,
    )

    all_results.append({
        "model_name": BEST_CFG["name"],
        "horizon": horizon,
        "input_size": input_size,
        "n_folds": N_FOLDS,
        "cv_steps": CV_STEPS,
        "n_series": n_series,
        "n_rows": n_rows,
        "freq": str(BEST_CFG["n_freq_downsample"]),
        "blocks": str(BEST_CFG["n_blocks"]),
        "wape_raw": metrics["wape_raw"],
        "rbias_raw": metrics["rbias_raw"],
        "total_raw": metrics["total_raw"],
        "wape_cal": metrics["wape_cal"],
        "rbias_cal": metrics["rbias_cal"],
        "total_cal": metrics["total_cal"],
        "calib_scale": metrics["calib_scale"],
        "submission_file": submission_file,
    })

    del train_nf
    gc.collect()

# ── Save tables ──
results_df = pd.DataFrame(all_results).sort_values("horizon").reset_index(drop=True)
folds_df   = pd.DataFrame(all_fold_rows).sort_values(["horizon", "fold"]).reset_index(drop=True)

results_df.to_csv("nhits_horizon_sweep.csv", index=False)
folds_df.to_csv("nhits_horizon_sweep_folds.csv", index=False)

print("\nSaved: nhits_horizon_sweep.csv")
print("Saved: nhits_horizon_sweep_folds.csv")
print("\nLeaderboard by horizon:")
print(results_df[["horizon", "n_series", "wape_raw", "rbias_raw", "total_raw", "total_cal", "calib_scale"]].to_string(index=False))

# ── Plot ──
if len(results_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # total metric vs horizon
    ax = axes[0]
    ax.plot(results_df["horizon"], results_df["total_raw"], marker="o", linewidth=2, label="total_raw")
    ax.plot(results_df["horizon"], results_df["total_cal"], marker="o", linewidth=2, label="total_cal")
    ax.set_xscale("log", base=2)
    ax.set_xlabel("Forecast horizon")
    ax.set_ylabel("Metric value")
    ax.set_title("NHITS quality vs horizon")
    ax.grid(True, alpha=0.3)
    ax.legend()

    # number of series per horizon
    ax = axes[1]
    ax.bar(results_df["horizon"].astype(str), results_df["n_series"])
    ax.set_xlabel("Forecast horizon")
    ax.set_ylabel("Eligible series")
    ax.set_title("How many series remain in CV")
    ax.grid(True, axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig("nhits_horizon_sweep.png", dpi=220, bbox_inches="tight")
    plt.close()

    print("Saved: nhits_horizon_sweep.png")

In [ ]:
# ============================================================
# CLEAN SLIDE-STYLE PLOT:
# - без переобучения
# - только calibrated total
# - без лучшего горизонта
# - без нижних баров
# - больше воздуха сверху и по краям
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

# --- source ---
try:
    plot_df = results_df.copy()
except NameError:
    plot_df = pd.DataFrame(all_results).copy()

plot_df = plot_df[["horizon", "total_cal"]].copy()
plot_df = plot_df.sort_values("horizon").reset_index(drop=True)

# --- readable horizon labels ---
label_map = {
    2: "1ч",
    4: "2ч",
    8: "4ч",
    16: "8ч",
    32: "16ч",
    48: "24ч",
    336: "7д",
}
plot_df["label"] = plot_df["horizon"].map(label_map).fillna(plot_df["horizon"].astype(str))
x = np.arange(len(plot_df))
y = plot_df["total_cal"].values

# --- colors ---
BG = "#140022"        # background
PANEL = "#17142E"     # inner panel
TEXT = "#F6F4FF"
MUTED = "#B6A8DA"
GRID = "#6B5A9A"
CYAN = "#2EF3FF"

plt.rcParams["font.family"] = "DejaVu Sans"

fig = plt.figure(figsize=(14, 8), facecolor=BG)
ax = fig.add_axes([0.075, 0.19, 0.88, 0.60])  # left, bottom, width, height
ax.set_facecolor(PANEL)

# subtle glow line
ax.plot(
    x, y,
    color=CYAN,
    linewidth=3.2,
    marker="o",
    markersize=8,
    solid_capstyle="round",
    path_effects=[
        pe.Stroke(linewidth=10, foreground=(46/255, 243/255, 255/255, 0.10)),
        pe.Normal()
    ],
    zorder=3
)

# soft fill under line
ax.fill_between(x, y, np.min(y) - 0.02, color=CYAN, alpha=0.05, zorder=1)

# axes styling
for spine in ax.spines.values():
    spine.set_visible(False)

ax.grid(axis="y", color=GRID, alpha=0.18, linewidth=1)
ax.grid(axis="x", visible=False)

ax.set_xticks(x)
ax.set_xticklabels(plot_df["label"], fontsize=13, color=MUTED)
ax.tick_params(axis="y", colors=MUTED, labelsize=12, length=0)
ax.tick_params(axis="x", length=0)

# more vertical breathing room
ymin = float(np.min(y))
ymax = float(np.max(y))
pad = max((ymax - ymin) * 0.28, 0.015)
ax.set_ylim(ymin - pad * 0.35, ymax + pad)

# more horizontal breathing room
ax.set_xlim(-0.35, len(x) - 0.65)

# labels
ax.set_ylabel("Calibrated total", fontsize=13, color=TEXT, labelpad=14)

# title block kept clearly above plot


# subtle top divider line
fig.lines.append(plt.Line2D(
    [0.075, 0.955], [0.835, 0.835],
    transform=fig.transFigure,
    color=GRID, alpha=0.22, linewidth=1.0
))

# save
plt.savefig(
    "nhits_horizon_total_cal_clean.png",
    dpi=1200,
    facecolor=BG,
    bbox_inches="tight",
    pad_inches=0.35
)
plt.show()

In [ ]:
# ============================================================
# NHITS retrain staleness test
# - fixed horizon H=8
# - fixed validation folds
# - move only the last refit point backward
# - use CURRENT context at inference, but STALE weights
# ============================================================

import warnings, gc
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MQLoss

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

# ── Metric ──
def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias

# ── Config ──
TRAIN_PATH   = "train_solo_track.parquet"
TARGET_COL   = "target_1h"
FREQ         = "30min"
FREQ_DELTA   = pd.Timedelta(FREQ)

H            = 8
INPUT_SIZE   = 336
CONTEXT_LEN  = 2048
BATCH_SIZE   = 64
N_FOLDS      = 3
CV_STEPS     = 200

RETRAIN_LAGS = [
    0,          # baseline
    2, 4, 8, 16, 32, 48,
    48 * 7,     # 7 days
    48 * 14,    # 14 days
    48 * 30,    # 30 days
]

BEST_CFG = {
    "name": "p2_blk_8_4_2",
    "n_freq_downsample": [48, 8, 1],
    "n_pool_kernel_size": [1, 1, 1],
    "n_blocks": [8, 4, 2],
}

if torch.backends.mps.is_available():
    ACCELERATOR = "mps"
elif torch.cuda.is_available():
    ACCELERATOR = "gpu"
else:
    ACCELERATOR = "cpu"
print("Device:", ACCELERATOR)

# ── Load data ──
train_df = pd.read_parquet(TRAIN_PATH)
train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

status_cols = sorted([c for c in train_df.columns if c in ["status_2", "status_3", "status_5"]])

FUTR_EXOG = ["hour_sin","hour_cos","dow_sin","dow_cos","is_weekend"]
HIST_EXOG = (status_cols + ["lag_48","lag_336"]) or None

# ── Feature builders ──
def add_calendar_features(df, ts_col="ds"):
    ts = pd.to_datetime(df[ts_col])
    df = df.copy()
    df["hour_sin"]   = np.sin(2*np.pi*ts.dt.hour/24)
    df["hour_cos"]   = np.cos(2*np.pi*ts.dt.hour/24)
    df["dow_sin"]    = np.sin(2*np.pi*ts.dt.dayofweek/7)
    df["dow_cos"]    = np.cos(2*np.pi*ts.dt.dayofweek/7)
    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(float)
    return df

def add_lag_features(nf_df):
    nf_df = nf_df.sort_values(["unique_id","ds"]).copy()
    for lag in [48, 336]:
        nf_df[f"lag_{lag}"] = (
            nf_df.groupby("unique_id")["y"]
            .transform(lambda s, l=lag: s.shift(l).bfill().fillna(0))
        )
    return nf_df

def build_base_panel(df):
    rows = []
    for route_id, grp in df.groupby("route_id"):
        grp = (
            grp.sort_values("timestamp")
            .set_index("timestamp")[[TARGET_COL] + status_cols]
            .asfreq(FREQ).interpolate(method="time")
            .bfill().ffill()
            .reset_index()
        )
        grp.insert(0, "unique_id", route_id)
        grp = grp.rename(columns={"timestamp": "ds", TARGET_COL: "y"})
        rows.append(grp)
    base = pd.concat(rows, ignore_index=True)
    return base.sort_values(["unique_id", "ds"]).reset_index(drop=True)

def make_snapshot_nf(base_panel, max_ts, min_len=INPUT_SIZE+1, context_len=CONTEXT_LEN):
    snap = base_panel[base_panel["ds"] <= max_ts].copy()
    if snap.empty:
        return pd.DataFrame()

    snap = (
        snap.sort_values(["unique_id", "ds"])
            .groupby("unique_id", group_keys=False)
            .tail(context_len)
            .reset_index(drop=True)
    )

    sizes = snap.groupby("unique_id").size()
    ok_ids = sizes[sizes >= min_len].index
    snap = snap[snap["unique_id"].isin(ok_ids)].copy()

    if snap.empty:
        return pd.DataFrame()

    snap = add_calendar_features(snap)
    snap = add_lag_features(snap)
    return snap

def make_truth_df(base_panel, cutoff, horizon):
    end_ts = cutoff + horizon * FREQ_DELTA
    truth = base_panel[(base_panel["ds"] > cutoff) & (base_panel["ds"] <= end_ts)][["unique_id","ds","y"]].copy()
    if truth.empty:
        return pd.DataFrame(columns=["unique_id","ds","y"])

    cnt = truth.groupby("unique_id").size()
    ok_ids = cnt[cnt == horizon].index
    truth = truth[truth["unique_id"].isin(ok_ids)].copy()
    return truth.sort_values(["unique_id","ds"]).reset_index(drop=True)

def make_futr_df_from_snapshot(snapshot, horizon):
    last_ds = snapshot.groupby("unique_id", as_index=False)["ds"].max()
    rows = []
    for _, r in last_ds.iterrows():
        future_ds = pd.date_range(
            start=r["ds"] + FREQ_DELTA,
            periods=horizon,
            freq=FREQ
        )
        for ts in future_ds:
            rows.append({"unique_id": r["unique_id"], "ds": ts})
    futr_df = pd.DataFrame(rows)
    futr_df = add_calendar_features(futr_df)
    return futr_df

def get_pred_col(pred_df):
    for c in ["NHITS-median","NHITS-q-0.50","NHITS"]:
        if c in pred_df.columns:
            return c
    skip = {"unique_id","ds","y","cutoff"}
    num = [c for c in pred_df.columns if c not in skip and pd.api.types.is_numeric_dtype(pred_df[c])]
    med = [c for c in num if any(x in c for x in ["median","0.5","50"])]
    return (med or num)[0]

def make_nhits(cfg, max_steps):
    return NHITS(
        h=H,
        input_size=INPUT_SIZE,
        loss=MQLoss(level=[80]),
        n_freq_downsample=cfg["n_freq_downsample"],
        n_pool_kernel_size=cfg["n_pool_kernel_size"],
        n_blocks=cfg["n_blocks"],
        mlp_units=[[512, 512]] * len(cfg["n_blocks"]),
        max_steps=max_steps,
        batch_size=BATCH_SIZE,
        accelerator=ACCELERATOR,
        hist_exog_list=HIST_EXOG,
        futr_exog_list=FUTR_EXOG,
        scaler_type="robust",
        enable_progress_bar=False,
    )

def lag_label(steps):
    if steps == 0:
        return "без лага"
    if steps < 48:
        hours = steps / 2
        if hours.is_integer():
            h = int(hours)
            return f"{h} ч" if h != 1 else "1 ч"
        return f"{hours:.1f} ч"
    if steps % 48 == 0:
        days = steps // 48
        if days == 1:
            return "1 день"
        if days < 5:
            return f"{days} дня"
        return f"{days} дней"
    return str(steps)

# ── Build full aligned panel once ──
base_panel = build_base_panel(train_df)

# общий конец, который есть у всех рядов
common_end = base_panel.groupby("unique_id")["ds"].max().min()
fixed_cutoffs = [common_end - i * H * FREQ_DELTA for i in range(N_FOLDS, 0, -1)]

print("Common end:", common_end)
print("Fixed cutoffs:")
for i, c in enumerate(fixed_cutoffs, 1):
    print(f"  Fold {i}: cutoff={c}, target=({c + FREQ_DELTA} .. {c + H * FREQ_DELTA})")

def run_staleness_cv(retrain_lag_steps):
    fold_rows = []
    yt_all, yp_all = [], []

    for fold, cutoff in enumerate(fixed_cutoffs, 1):
        fit_end = cutoff - retrain_lag_steps * FREQ_DELTA

        fit_nf = make_snapshot_nf(
            base_panel=base_panel,
            max_ts=fit_end,
            min_len=INPUT_SIZE + 1,
            context_len=CONTEXT_LEN,
        )
        ctx_nf = make_snapshot_nf(
            base_panel=base_panel,
            max_ts=cutoff,
            min_len=INPUT_SIZE + 1,
            context_len=CONTEXT_LEN,
        )
        truth_df = make_truth_df(
            base_panel=base_panel,
            cutoff=cutoff,
            horizon=H,
        )

        if fit_nf.empty or ctx_nf.empty or truth_df.empty:
            fold_rows.append({
                "fold": fold,
                "status": "skip_empty",
                "n_series": 0,
                "cutoff": cutoff,
                "fit_end": fit_end,
            })
            continue

        fit_ids = set(fit_nf["unique_id"].unique())
        ctx_ids = set(ctx_nf["unique_id"].unique())
        truth_ids = set(truth_df["unique_id"].unique())
        common_ids = sorted(fit_ids & ctx_ids & truth_ids)

        if len(common_ids) == 0:
            fold_rows.append({
                "fold": fold,
                "status": "skip_no_common_ids",
                "n_series": 0,
                "cutoff": cutoff,
                "fit_end": fit_end,
            })
            continue

        fit_nf_f = fit_nf[fit_nf["unique_id"].isin(common_ids)].copy()
        ctx_nf_f = ctx_nf[ctx_nf["unique_id"].isin(common_ids)].copy()
        truth_f = truth_df[truth_df["unique_id"].isin(common_ids)].copy()

        try:
            nf = NeuralForecast(models=[make_nhits(BEST_CFG, CV_STEPS)], freq=FREQ)
            nf.fit(fit_nf_f)

            futr_df = make_futr_df_from_snapshot(ctx_nf_f, H)
            pred_df = nf.predict(df=ctx_nf_f, futr_df=futr_df)
            pred_col = get_pred_col(pred_df)

            pred_f = pred_df[["unique_id","ds",pred_col]].copy()
            pred_f = pred_f.sort_values(["unique_id","ds"]).reset_index(drop=True)

            merged = truth_f.merge(pred_f, on=["unique_id","ds"], how="inner")
            if merged.empty:
                fold_rows.append({
                    "fold": fold,
                    "status": "skip_empty_merge",
                    "n_series": len(common_ids),
                    "cutoff": cutoff,
                    "fit_end": fit_end,
                })
                del nf
                gc.collect()
                continue

            yt = merged["y"].values
            yp = np.clip(merged[pred_col].values, 0, None)

            t, w, r = wape_rbias(yt, yp)

            yt_all.append(yt)
            yp_all.append(yp)

            fold_rows.append({
                "fold": fold,
                "status": "ok",
                "n_series": merged["unique_id"].nunique(),
                "n_points": len(merged),
                "cutoff": cutoff,
                "fit_end": fit_end,
                "wape_raw": round(w, 4),
                "rbias_raw": round(r, 4),
                "total_raw": round(t, 4),
            })

            del nf, pred_df, pred_f, merged
            gc.collect()

        except Exception as e:
            fold_rows.append({
                "fold": fold,
                "status": f"error: {type(e).__name__}",
                "n_series": len(common_ids),
                "cutoff": cutoff,
                "fit_end": fit_end,
            })
            gc.collect()

    ok_rows = [r for r in fold_rows if r["status"] == "ok"]

    if len(ok_rows) == 0:
        return {
            "delay_steps": retrain_lag_steps,
            "delay_label": lag_label(retrain_lag_steps),
            "n_folds_ok": 0,
            "mean_n_series": 0,
            "wape_raw": np.nan,
            "rbias_raw": np.nan,
            "total_raw": np.nan,
            "wape_cal": np.nan,
            "rbias_cal": np.nan,
            "total_cal": np.nan,
            "calib_scale": np.nan,
            "fold_rows": fold_rows,
        }

    yt_all = np.concatenate(yt_all)
    yp_all = np.concatenate(yp_all)

    t_raw, w_raw, r_raw = wape_rbias(yt_all, yp_all)

    calib = float(yt_all.sum() / (yp_all.sum() + 1e-9))
    yp_cal = np.clip(yp_all * calib, 0, None)
    t_cal, w_cal, r_cal = wape_rbias(yt_all, yp_cal)

    return {
        "delay_steps": retrain_lag_steps,
        "delay_label": lag_label(retrain_lag_steps),
        "n_folds_ok": len(ok_rows),
        "mean_n_series": round(np.mean([r["n_series"] for r in ok_rows]), 1),
        "wape_raw": round(w_raw, 4),
        "rbias_raw": round(r_raw, 4),
        "total_raw": round(t_raw, 4),
        "wape_cal": round(w_cal, 4),
        "rbias_cal": round(r_cal, 4),
        "total_cal": round(t_cal, 4),
        "calib_scale": round(calib, 4),
        "fold_rows": fold_rows,
    }

# ── Main loop ──
all_rows = []
all_fold_rows = []

for delay in RETRAIN_LAGS:
    print("\n" + "=" * 72)
    print(f"Retrain lag: {delay} steps ({lag_label(delay)})")
    print("=" * 72)

    res = run_staleness_cv(delay)

    for fr in res["fold_rows"]:
        msg = f"Fold {fr['fold']} | status={fr['status']} | n_series={fr.get('n_series', 0)}"
        if fr["status"] == "ok":
            msg += f" | Total={fr['total_raw']:.4f}"
        print(msg)
        all_fold_rows.append({
            "delay_steps": delay,
            "delay_label": lag_label(delay),
            **fr
        })

    print(
        f"OK folds={res['n_folds_ok']} | mean_n_series={res['mean_n_series']} | "
        f"RAW total={res['total_raw']:.4f} | CAL total={res['total_cal']:.4f}"
        if res["n_folds_ok"] > 0 else
        "No valid folds"
    )

    row = {k: v for k, v in res.items() if k != "fold_rows"}
    all_rows.append(row)

# ── Save results ──
stale_df = pd.DataFrame(all_rows).sort_values("delay_steps").reset_index(drop=True)
stale_folds_df = pd.DataFrame(all_fold_rows).sort_values(["delay_steps", "fold"]).reset_index(drop=True)

stale_df.to_csv("nhits_retrain_staleness_h8.csv", index=False)
stale_folds_df.to_csv("nhits_retrain_staleness_h8_folds.csv", index=False)

print("\nSaved: nhits_retrain_staleness_h8.csv")
print("Saved: nhits_retrain_staleness_h8_folds.csv")
display(stale_df)

# ── Plot: clean slide-style ──
plot_df = stale_df[stale_df["n_folds_ok"] > 0].copy()

BG = "#140022"
PANEL = "#17142E"
TEXT = "#F6F4FF"
MUTED = "#B6A8DA"
GRID = "#6B5A9A"
CYAN = "#2EF3FF"

plt.rcParams["font.family"] = "DejaVu Sans"

fig = plt.figure(figsize=(14, 8), facecolor=BG)
ax = fig.add_axes([0.075, 0.19, 0.88, 0.60])
ax.set_facecolor(PANEL)

x = np.arange(len(plot_df))
y = plot_df["total_cal"].values

ax.plot(
    x, y,
    color=CYAN,
    linewidth=3.2,
    marker="o",
    markersize=8,
    solid_capstyle="round",
    path_effects=[
        pe.Stroke(linewidth=10, foreground=(46/255, 243/255, 255/255, 0.10)),
        pe.Normal()
    ],
    zorder=3
)

ax.fill_between(x, y, np.min(y) - 0.02, color=CYAN, alpha=0.05, zorder=1)

for spine in ax.spines.values():
    spine.set_visible(False)

ax.grid(axis="y", color=GRID, alpha=0.18, linewidth=1)
ax.grid(axis="x", visible=False)

ax.set_xticks(x)
ax.set_xticklabels(plot_df["delay_label"], fontsize=12, color=MUTED)
ax.tick_params(axis="y", colors=MUTED, labelsize=12, length=0)
ax.tick_params(axis="x", length=0)

ymin = float(np.min(y))
ymax = float(np.max(y))
pad = max((ymax - ymin) * 0.28, 0.015)
ax.set_ylim(ymin - pad * 0.35, ymax + pad)
ax.set_xlim(-0.35, len(x) - 0.65)

ax.set_ylabel("Calibrated total", fontsize=13, color=TEXT, labelpad=14)

fig.text(
    0.075, 0.905,
    "Качество при редком переобучении",
    color=TEXT, fontsize=28, fontweight="bold"
)
fig.text(
    0.075, 0.865,
    "NHITS p2_blk_8_4_2 · horizon = 8 · фиксированные CV-окна",
    color=MUTED, fontsize=13
)

fig.lines.append(plt.Line2D(
    [0.075, 0.955], [0.835, 0.835],
    transform=fig.transFigure,
    color=GRID, alpha=0.22, linewidth=1.0
))

plt.savefig(
    "nhits_retrain_staleness_h8.png",
    dpi=220,
    facecolor=BG,
    bbox_inches="tight",
    pad_inches=0.35
)
plt.show()

In [ ]:
plot_df = stale_df[stale_df["n_folds_ok"] > 0].copy()

BG = "#140022"
PANEL = "#17142E"
TEXT = "#F6F4FF"
MUTED = "#B6A8DA"
GRID = "#6B5A9A"
CYAN = "#2EF3FF"

plt.rcParams["font.family"] = "DejaVu Sans"

fig = plt.figure(figsize=(14, 8), facecolor=BG)
ax = fig.add_axes([0.075, 0.19, 0.88, 0.60])
ax.set_facecolor(PANEL)

x = np.arange(len(plot_df))
y = plot_df["total_cal"].values

ax.plot(
    x, y,
    color=CYAN,
    linewidth=3.2,
    marker="o",
    markersize=8,
    solid_capstyle="round",
    path_effects=[
        pe.Stroke(linewidth=10, foreground=(46/255, 243/255, 255/255, 0.10)),
        pe.Normal()
    ],
    zorder=3
)

ax.fill_between(x, y, np.min(y) - 0.02, color=CYAN, alpha=0.05, zorder=1)

for spine in ax.spines.values():
    spine.set_visible(False)

ax.grid(axis="y", color=GRID, alpha=0.18, linewidth=1)
ax.grid(axis="x", visible=False)

ax.set_xticks(x)
ax.set_xticklabels(plot_df["delay_label"], fontsize=12, color=MUTED)
ax.tick_params(axis="y", colors=MUTED, labelsize=12, length=0)
ax.tick_params(axis="x", length=0)

ymin = float(np.min(y))
ymax = float(np.max(y))
pad = max((ymax - ymin) * 0.28, 0.015)
ax.set_ylim(ymin - pad * 0.35, ymax + pad)
ax.set_xlim(-0.35, len(x) - 0.65)

ax.set_ylabel("Calibrated total", fontsize=13, color=TEXT, labelpad=14)

fig.text(
    0.075, 0.905,
    "Качество при редком переобучении",
    color=TEXT, fontsize=28, fontweight="bold"
)
fig.text(
    0.075, 0.865,
    "NHITS p2_blk_8_4_2 · horizon = 8 · фиксированные CV-окна",
    color=MUTED, fontsize=13
)

fig.lines.append(plt.Line2D(
    [0.075, 0.955], [0.835, 0.835],
    transform=fig.transFigure,
    color=GRID, alpha=0.22, linewidth=1.0
))

plt.savefig(
    "nhits_retrain_staleness_h8.png",
    dpi=1220,
    facecolor=BG,
    bbox_inches="tight",
    pad_inches=0.35
)
plt.show()

In [ ]:
# ============================================================
# NHITS ensemble over 10 seeds for fixed best config p2_blk_8_4_2
# Per-seed calibration in the same style as original code:
# calib_scale = sum(y_true) / sum(y_pred) on CV predictions
# Final submission = mean of calibrated predictions across seeds
# ============================================================

import warnings, gc, random, os
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MQLoss

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

# ── Metric ──
def wape_rbias(y_true, y_pred):
    yt = np.asarray(y_true, float)
    yp = np.clip(np.asarray(y_pred, float), 0, None)
    s  = yt.sum() + 1e-9
    wape  = np.abs(yp - yt).sum() / s
    rbias = np.abs(yp.sum() / s - 1)
    return wape + rbias, wape, rbias

# ── Config ──
TRAIN_PATH     = "train_solo_track.parquet"
TEST_PATH      = "test_solo_track.parquet"
TARGET_COL     = "target_1h"
FORECAST_STEPS = 8
CONTEXT_LEN    = 2048
FREQ           = "30min"
INPUT_SIZE     = 336
BATCH_SIZE     = 64

SEEDS              = list(range(10))   # можно заменить на свои 10 сидов
SEED_CV_STEPS      = 400               # быстрый прогон для calib каждого seed
FINAL_STEPS        = 800               # финальный fit каждого seed
CALIB_N_WINDOWS    = 1                 # быстро; если захочешь надежнее -> 3 или 5
STEP_SIZE          = FORECAST_STEPS

BEST_CFG = {
    "name": "p2_blk_8_4_2",
    "n_freq_downsample": [48, 8, 1],
    "n_pool_kernel_size": [1, 1, 1],
    "n_blocks": [8, 4, 2],
    "note": "Best from previous search",
}

if torch.backends.mps.is_available():
    ACCELERATOR = "mps"
elif torch.cuda.is_available():
    ACCELERATOR = "gpu"
else:
    ACCELERATOR = "cpu"
print(f"Device: {ACCELERATOR}")

# ── Data ──
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)

train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])

train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

status_cols = sorted([c for c in train_df.columns if c in ["status_2","status_3","status_5"]])

FUTR_EXOG = ["hour_sin","hour_cos","dow_sin","dow_cos","is_weekend"]
HIST_EXOG = (status_cols + ["lag_48","lag_336"]) or None

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def add_calendar_features(df, ts_col="ds"):
    ts = pd.to_datetime(df[ts_col])
    df = df.copy()
    df["hour_sin"]   = np.sin(2*np.pi*ts.dt.hour/24)
    df["hour_cos"]   = np.cos(2*np.pi*ts.dt.hour/24)
    df["dow_sin"]    = np.sin(2*np.pi*ts.dt.dayofweek/7)
    df["dow_cos"]    = np.cos(2*np.pi*ts.dt.dayofweek/7)
    df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(float)
    return df

def add_lag_features(nf_df):
    nf_df = nf_df.sort_values(["unique_id","ds"]).copy()
    for lag in [48, 336]:
        nf_df[f"lag_{lag}"] = (
            nf_df.groupby("unique_id")["y"]
            .transform(lambda s, l=lag: s.shift(l).bfill().fillna(0))
        )
    return nf_df

def to_nf_df(df, min_len=32, context_len=CONTEXT_LEN):
    rows = []
    for route_id, grp in df.groupby("route_id"):
        grp = (
            grp.sort_values("timestamp")
            .set_index("timestamp")[[TARGET_COL] + status_cols]
            .asfreq(FREQ).interpolate(method="time")
            .bfill().ffill().tail(context_len).reset_index()
        )
        if len(grp) < min_len:
            continue
        grp.insert(0, "unique_id", route_id)
        grp = grp.rename(columns={"timestamp":"ds", TARGET_COL:"y"})
        rows.append(grp)

    nf_df = pd.concat(rows, ignore_index=True)
    nf_df = add_calendar_features(nf_df)
    nf_df = add_lag_features(nf_df)
    return nf_df

def make_futr_df(route_ids, train_df):
    rows = []
    for rid in route_ids:
        last_ts = train_df[train_df["route_id"] == rid]["timestamp"].max()
        for ts in pd.date_range(
            start=last_ts + pd.Timedelta("30min"),
            periods=FORECAST_STEPS,
            freq=FREQ
        ):
            rows.append({"unique_id": rid, "ds": ts})
    return add_calendar_features(pd.DataFrame(rows))

def get_pred_col(pred_df):
    for c in ["NHITS-median", "NHITS-q-0.50", "NHITS"]:
        if c in pred_df.columns:
            return c
    skip = {"unique_id","ds","y","cutoff"}
    num = [c for c in pred_df.columns if c not in skip
           and pd.api.types.is_numeric_dtype(pred_df[c])]
    med = [c for c in num if any(x in c for x in ["median","0.5","50"])]
    return (med or num)[0]

def build_preds_dict(pred_df, pred_col):
    return {
        rid: np.clip(g.sort_values("ds")[pred_col].values, 0, None)
        for rid, g in pred_df.groupby("unique_id")
    }

def make_nhits(cfg, max_steps, seed):
    return NHITS(
        h                  = FORECAST_STEPS,
        input_size         = INPUT_SIZE,
        loss               = MQLoss(level=[80]),
        n_freq_downsample  = cfg["n_freq_downsample"],
        n_pool_kernel_size = cfg["n_pool_kernel_size"],
        n_blocks           = cfg["n_blocks"],
        mlp_units          = [[512, 512]] * len(cfg["n_blocks"]),
        max_steps          = max_steps,
        batch_size         = BATCH_SIZE,
        accelerator        = ACCELERATOR,
        hist_exog_list     = HIST_EXOG,
        futr_exog_list     = FUTR_EXOG,
        scaler_type        = "robust",
        random_seed        = seed,
        enable_progress_bar= True,
    )

def run_calibration_cv(cfg, train_nf, max_steps, seed, n_windows=1):
    nf = NeuralForecast(models=[make_nhits(cfg, max_steps, seed)], freq=FREQ)
    cv_df = nf.cross_validation(
        df=train_nf,
        n_windows=n_windows,
        step_size=STEP_SIZE,
        refit=True
    )
    pred_col = get_pred_col(cv_df)

    yt = cv_df["y"].values
    yp = np.clip(cv_df[pred_col].values, 0, None)

    total_raw, wape_raw, rbias_raw = wape_rbias(yt, yp)

    calib_scale = float(yt.sum() / (yp.sum() + 1e-9))
    yp_cal = np.clip(yp * calib_scale, 0, None)
    total_cal, wape_cal, rbias_cal = wape_rbias(yt, yp_cal)

    del nf, cv_df
    gc.collect()

    return {
        "seed": seed,
        "wape_raw": round(wape_raw, 4),
        "rbias_raw": round(rbias_raw, 4),
        "total_raw": round(total_raw, 4),
        "wape_cal": round(wape_cal, 4),
        "rbias_cal": round(rbias_cal, 4),
        "total_cal": round(total_cal, 4),
        "calib_scale": float(calib_scale),
    }

# ── Data for calibration CV ──
# Берем достаточно длинные ряды, чтобы был честный holdout под cv-window
train_nf_calib = to_nf_df(
    train_df,
    min_len=INPUT_SIZE + CALIB_N_WINDOWS * FORECAST_STEPS + 1,
    context_len=CONTEXT_LEN,
)
print(f"Calibration NF train: {train_nf_calib.shape}  routes: {train_nf_calib['unique_id'].nunique()}")

# ── Data for final full fit ──
train_nf_full = to_nf_df(
    train_df,
    min_len=INPUT_SIZE + 1,
    context_len=CONTEXT_LEN,
)
print(f"Full-fit NF train: {train_nf_full.shape}  routes: {train_nf_full['unique_id'].nunique()}")

test_route_ids = test_df["route_id"].unique().tolist()
futr_df = make_futr_df(test_route_ids, train_df)

# ──────────────────────────────────────────────────────────────
# STAGE 1: per-seed calibration
# ──────────────────────────────────────────────────────────────
seed_rows = []
for i, seed in enumerate(SEEDS, 1):
    print(f"\n{'='*70}")
    print(f"[CALIB {i:02d}/{len(SEEDS)}] seed={seed}")
    print("="*70)

    set_seed(seed)
    metrics = run_calibration_cv(
        cfg=BEST_CFG,
        train_nf=train_nf_calib,
        max_steps=SEED_CV_STEPS,
        seed=seed,
        n_windows=CALIB_N_WINDOWS,
    )
    seed_rows.append(metrics)

    print(f"RAW   WAPE={metrics['wape_raw']:.4f}  |RBias|={metrics['rbias_raw']:.4f}  Total={metrics['total_raw']:.4f}")
    print(f"CAL   WAPE={metrics['wape_cal']:.4f}  |RBias|={metrics['rbias_cal']:.4f}  Total={metrics['total_cal']:.4f}  scale={metrics['calib_scale']:.6f}")

seed_df = (
    pd.DataFrame(seed_rows)
    .sort_values(["total_cal", "total_raw", "seed"])
    .reset_index(drop=True)
)
seed_df.to_csv(f"nhits_{BEST_CFG['name']}_seed_calibration.csv", index=False)

print("\n" + "="*70)
print("SEED CALIBRATION LEADERBOARD")
print("="*70)
print(seed_df.to_string(index=False))

seed2scale = dict(zip(seed_df["seed"], seed_df["calib_scale"]))

# ──────────────────────────────────────────────────────────────
# STAGE 2: fit each seed on full train and predict
# ──────────────────────────────────────────────────────────────
all_seed_preds_raw = []
all_seed_preds_cal = []

for i, seed in enumerate(SEEDS, 1):
    calib_scale = float(seed2scale[seed])

    print(f"\n{'='*70}")
    print(f"[FIT/PRED {i:02d}/{len(SEEDS)}] seed={seed}  calib_scale={calib_scale:.6f}")
    print("="*70)

    set_seed(seed)
    nf_full = NeuralForecast(
        models=[make_nhits(BEST_CFG, FINAL_STEPS, seed)],
        freq=FREQ
    )
    nf_full.fit(train_nf_full)

    test_pred_df = nf_full.predict(futr_df=futr_df)
    pred_col = get_pred_col(test_pred_df)
    test_preds = build_preds_dict(test_pred_df, pred_col)

    predictions_raw = {}
    for route_id in tqdm(test_route_ids, desc=f"Build submission seed={seed}"):
        route_test = test_df[test_df["route_id"] == route_id].sort_values("timestamp")
        preds = test_preds.get(route_id, np.zeros(FORECAST_STEPS))
        for j, (_, row) in enumerate(route_test.iterrows()):
            pred = float(preds[j]) if j < len(preds) else float(preds[-1])
            predictions_raw[row["id"]] = max(0.0, pred)

    seed_sub_raw = (
        pd.DataFrame(list(predictions_raw.items()), columns=["id", f"y_pred_seed_{seed}"])
        .sort_values("id")
        .reset_index(drop=True)
    )

    seed_sub_cal = seed_sub_raw.copy()
    seed_sub_cal[f"y_pred_seed_{seed}"] = np.clip(
        seed_sub_cal[f"y_pred_seed_{seed}"] * calib_scale, 0, None
    )

    seed_sub_raw.to_csv(
        f"submission_{BEST_CFG['name']}_seed{seed}_raw.csv",
        index=False
    )
    seed_sub_cal.to_csv(
        f"submission_{BEST_CFG['name']}_seed{seed}_calibrated.csv",
        index=False
    )

    all_seed_preds_raw.append(seed_sub_raw)
    all_seed_preds_cal.append(seed_sub_cal)

    del nf_full, test_pred_df, test_preds
    gc.collect()

# ──────────────────────────────────────────────────────────────
# STAGE 3: ensemble
# ──────────────────────────────────────────────────────────────
ensemble_raw = all_seed_preds_raw[0][["id"]].copy()
for df_seed in all_seed_preds_raw:
    ensemble_raw = ensemble_raw.merge(df_seed, on="id", how="left")

raw_cols = [c for c in ensemble_raw.columns if c.startswith("y_pred_seed_")]
ensemble_raw["y_pred"] = ensemble_raw[raw_cols].mean(axis=1)
submission_ensemble_raw = ensemble_raw[["id", "y_pred"]].copy()

ensemble_cal = all_seed_preds_cal[0][["id"]].copy()
for df_seed in all_seed_preds_cal:
    ensemble_cal = ensemble_cal.merge(df_seed, on="id", how="left")

cal_cols = [c for c in ensemble_cal.columns if c.startswith("y_pred_seed_")]
ensemble_cal["y_pred"] = ensemble_cal[cal_cols].mean(axis=1)
submission_ensemble_cal = ensemble_cal[["id", "y_pred"]].copy()

submission_ensemble_raw.to_csv(
    f"submission_{BEST_CFG['name']}_ensemble10_raw_mean.csv",
    index=False
)
submission_ensemble_cal.to_csv(
    f"submission_{BEST_CFG['name']}_ensemble10_calibrated_mean.csv",
    index=False
)

# ── meta ──
meta_df = seed_df.copy()
meta_df["model_name"] = BEST_CFG["name"]
meta_df["seed_cv_steps"] = SEED_CV_STEPS
meta_df["final_steps"] = FINAL_STEPS
meta_df["calib_n_windows"] = CALIB_N_WINDOWS
meta_df.to_csv(f"nhits_{BEST_CFG['name']}_ensemble10_meta.csv", index=False)

print(f"\n✅ nhits_{BEST_CFG['name']}_seed_calibration.csv")
print(f"✅ nhits_{BEST_CFG['name']}_ensemble10_meta.csv")
for seed in SEEDS:
    print(f"✅ submission_{BEST_CFG['name']}_seed{seed}_raw.csv")
    print(f"✅ submission_{BEST_CFG['name']}_seed{seed}_calibrated.csv")
print(f"✅ submission_{BEST_CFG['name']}_ensemble10_raw_mean.csv")
print(f"✅ submission_{BEST_CFG['name']}_ensemble10_calibrated_mean.csv")

print(f"\n── Ensemble RAW ──\n{submission_ensemble_raw['y_pred'].describe().round(4)}")
print(f"\n── Ensemble CALIBRATED ──\n{submission_ensemble_cal['y_pred'].describe().round(4)}")

In [ ]:
from itertools import combinations
from pathlib import Path
import gc

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from catboost import CatBoostRegressor


# =========================
# CONFIG
# =========================
TRACK = "solo"          # "solo" or "team"
VALID_DAYS = 1
MAX_TRAIN_ROWS = 3_500_000
RANDOM_STATE = 42

LAGS_30M = [1, 2, 3, 6, 12, 24, 48]
ROLL_WINDOWS = [2, 4, 8, 16, 48]

# Для exact-повтора можно поставить 0.05, но будет заметно тяжелее.
BLEND_GRID_STEP = 0.1

TRACK_CONFIG = {
    "solo": {
        "train_path": "train_solo_track.parquet",
        "test_path": "test_solo_track.parquet",
        "target_col": "target_1h",
        "forecast_points": 8,
    },
    "team": {
        "train_path": "trainteamtrack.parquet",
        "test_path": "testteamtrack.parquet",
        "target_col": "target2h",
        "forecast_points": 10,
    },
}

CFG = TRACK_CONFIG[TRACK]
TARGET_COL = CFG["target_col"]
FORECAST_POINTS = CFG["forecast_points"]
FUTURE_TARGET_COLS = [f"targetstep{i}" for i in range(1, FORECAST_POINTS + 1)]

MODEL_SPECS = [
    dict(
        name="lgbpoisson7d",
        kind="lgbm",
        train_days=7,
        decay_days=3.0,
        params=dict(
            objective="poisson",
            n_estimators=1500,
            learning_rate=0.03,
            num_leaves=63,
            max_depth=-1,
            min_child_samples=200,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=1.0,
            reg_lambda=3.0,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=-1,
        ),
    ),
    dict(
        name="lgbpoisson9d",
        kind="lgbm",
        train_days=9,
        decay_days=4.0,
        params=dict(
            objective="poisson",
            n_estimators=1700,
            learning_rate=0.025,
            num_leaves=31,
            max_depth=-1,
            min_child_samples=250,
            subsample=0.85,
            colsample_bytree=0.8,
            reg_alpha=1.5,
            reg_lambda=4.0,
            random_state=RANDOM_STATE + 17,
            n_jobs=-1,
            verbosity=-1,
        ),
    ),
    dict(
        name="lgbmae7d",
        kind="lgbm",
        train_days=7,
        decay_days=3.0,
        params=dict(
            objective="mae",
            n_estimators=1500,
            learning_rate=0.03,
            num_leaves=63,
            max_depth=-1,
            min_child_samples=200,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=1.0,
            reg_lambda=3.0,
            random_state=RANDOM_STATE + 31,
            n_jobs=-1,
            verbosity=-1,
        ),
    ),
    dict(
        name="lgbpoisson5d",
        kind="lgbm",
        train_days=5,
        decay_days=2.0,
        params=dict(
            objective="poisson",
            n_estimators=1400,
            learning_rate=0.035,
            num_leaves=127,
            max_depth=-1,
            min_child_samples=120,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=1.0,
            reg_lambda=3.0,
            random_state=RANDOM_STATE + 71,
            n_jobs=-1,
            verbosity=-1,
        ),
    ),
    dict(
        name="lgbpoisson14d",
        kind="lgbm",
        train_days=14,
        decay_days=6.0,
        params=dict(
            objective="poisson",
            n_estimators=1900,
            learning_rate=0.025,
            num_leaves=255,
            max_depth=-1,
            min_child_samples=120,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=1.0,
            reg_lambda=3.0,
            random_state=RANDOM_STATE + 91,
            n_jobs=-1,
            verbosity=-1,
        ),
    ),
    dict(
        name="lgbmae14d",
        kind="lgbm",
        train_days=14,
        decay_days=6.0,
        params=dict(
            objective="mae",
            n_estimators=2900,
            learning_rate=0.025,
            num_leaves=255,
            max_depth=-1,
            min_child_samples=120,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=1.0,
            reg_lambda=3.0,
            random_state=RANDOM_STATE + 91,
            n_jobs=-1,
            verbosity=-1,
        ),
    ),
    dict(
        name="catpoisson14d",
        kind="catboost",
        train_days=14,
        decay_days=6.0,
        params=dict(
            loss_function="MAE",
            eval_metric="MAE",
            has_time=True,
            iterations=2000,
            learning_rate=0.03,
            depth=8,
            l2_leaf_reg=5.0,
            min_data_in_leaf=100,
            random_seed=RANDOM_STATE + 211,
            verbose=False,
            allow_writing_files=False,
        ),
    ),
    dict(
        name="ridge7d",
        kind="ridge",
        train_days=7,
        decay_days=None,
        alpha=4.0,
        max_train_rows=MAX_TRAIN_ROWS,
    ),
    dict(
        name="ridge7dalpha20",
        kind="ridge",
        train_days=7,
        decay_days=None,
        alpha=20.0,
        max_train_rows=MAX_TRAIN_ROWS,
    ),
]


# =========================
# METRIC / UTILS
# =========================
def wape_plus_rbias_score(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    denom = y_true.sum()
    if denom == 0:
        return np.nan
    wape = np.abs(y_pred - y_true).sum() / denom
    rbias = abs(y_pred.sum() / denom - 1.0)
    return float(wape + rbias)


def get_recent_subset(df, train_days):
    cutoff = df["source_timestamp"].max() - pd.Timedelta(days=train_days)
    return df.loc[df["source_timestamp"] >= cutoff].copy()


def get_time_decay_weights(df, decay_days):
    if decay_days is None:
        return None
    age_days = (df["source_timestamp"].max() - df["source_timestamp"]).dt.total_seconds() / (24 * 3600)
    return np.exp(-age_days / decay_days)


def build_ridge_model(alpha, numeric_features, categorical_features):
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value=0.0)),
        ("scaler", StandardScaler(with_mean=False)),
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocessor = ColumnTransformer([
        ("num", numeric_pipe, numeric_features),
        ("cat", categorical_pipe, categorical_features),
    ])
    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", Ridge(alpha=alpha)),
    ])


def prepare_catboost_frame(X):
    X = X.copy()
    if "route_id" in X.columns:
        X["route_id"] = X["route_id"].astype(str)
    return X


def generate_blend_weights(model_names, step=0.1):
    n = len(model_names)
    units = int(round(1.0 / step))
    for bars in combinations(range(units + n - 1), n - 1):
        parts = []
        prev = -1
        for b in bars:
            parts.append(b - prev - 1)
            prev = b
        parts.append(units + n - 1 - prev - 1)
        weights = np.array(parts, dtype=np.float32) / units
        yield dict(zip(model_names, weights))


def blend_single_col(pred_dict, weights, col):
    out = None
    for name, w in weights.items():
        if w == 0:
            continue
        cur = pred_dict[name][col].to_numpy(dtype=np.float32)
        out = cur * w if out is None else out + cur * w
    return pd.Series(out, index=pred_dict[next(iter(pred_dict))].index, dtype=np.float32)


def blend_multi_col(pred_dict, weights_per_horizon, cols):
    base_index = pred_dict[next(iter(pred_dict))].index
    out = pd.DataFrame(index=base_index, columns=cols, dtype=np.float32)
    for col in cols:
        out[col] = blend_single_col(pred_dict, weights_per_horizon[col], col).to_numpy(dtype=np.float32)
    return out


# =========================
# FEATURE ENGINEERING
# =========================
def add_time_features(df):
    df = df.copy()
    df["hour"] = df["timestamp"].dt.hour.astype("int8")
    df["minute"] = df["timestamp"].dt.minute.astype("int8")
    df["dayofweek"] = df["timestamp"].dt.dayofweek.astype("int8")
    df["isweekend"] = (df["dayofweek"] >= 5).astype("int8")
    df["halfhouridx"] = (df["hour"] * 2 + (df["minute"] == 30)).astype("int8")
    df["halfhour_sin"] = np.sin(2 * np.pi * df["halfhouridx"] / 48).astype("float32")
    df["halfhour_cos"] = np.cos(2 * np.pi * df["halfhouridx"] / 48).astype("float32")
    df["dow_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7).astype("float32")
    df["dow_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7).astype("float32")
    return df


def add_status_aggregates(df, status_cols):
    df = df.copy()
    current_cols = [c for c in ["status1", "status2", "status3"] if c in df.columns]
    prev_cols = [c for c in ["status4", "status5", "status6"] if c in df.columns]
    df["status_current_sum"] = df[current_cols].sum(axis=1).astype("float32")
    df["status_prev_sum"] = df[prev_cols].sum(axis=1).astype("float32")
    df["status_total_sum"] = (df[current_cols + prev_cols].sum(axis=1)).astype("float32")
    df["status_prev_to_current_ratio"] = (df["status_prev_sum"] / (df["status_current_sum"] + 1.0)).astype("float32")
    return df


def add_wb_promo_features(df):
    df = df.copy()
    date = df["timestamp"].dt.normalize()
    years = sorted(df["timestamp"].dt.year.unique())

    wb1111_starts = pd.to_datetime([f"{y}-10-27" for y in years])
    wb1111_ends = pd.to_datetime([f"{y}-11-13" for y in years])
    wb1111_days = pd.to_datetime([f"{y}-11-11" for y in years])

    sale_window = np.zeros(len(df), dtype=np.int8)
    for start, end in zip(wb1111_starts, wb1111_ends):
        sale_window |= ((date >= start) & (date <= end)).astype(np.int8)

    date_vals = date.values.astype("datetime64[D]")
    days_to_1111 = np.full(len(df), 9999, dtype=np.int16)
    for ev in wb1111_days.values.astype("datetime64[D]"):
        diff = (ev - date_vals).astype("timedelta64[D]").astype(np.int16)
        days_to_1111 = np.minimum(days_to_1111, diff)

    df["is1111_sale_window"] = sale_window.astype("int8")
    df["is_pre1111_14d"] = ((days_to_1111 >= 1) & (days_to_1111 <= 14)).astype("int8")
    df["days_to_1111_clip14"] = np.where((days_to_1111 >= 1) & (days_to_1111 <= 14), days_to_1111, 0).astype("int8")
    df["proximity_to_1111"] = np.where((days_to_1111 >= 1) & (days_to_1111 <= 14), 15 - days_to_1111, 0).astype("int8")

    df["is_wb_birthday_window_oct"] = (
        (df["timestamp"].dt.month == 10) &
        (df["timestamp"].dt.day >= 7) &
        (df["timestamp"].dt.day <= 20)
    ).astype("int8")

    df["is_back_to_school_season"] = (
        ((df["timestamp"].dt.month == 8) & (df["timestamp"].dt.day >= 15)) |
        ((df["timestamp"].dt.month == 9) & (df["timestamp"].dt.day <= 1))
    ).astype("int8")

    df["is_back_to_school_peak"] = (
        (df["timestamp"].dt.month == 8) &
        (df["timestamp"].dt.day >= 20) &
        (df["timestamp"].dt.day <= 31)
    ).astype("int8")

    df["is_promo_like_period"] = (
        (df["is1111_sale_window"] == 1) |
        (df["is_wb_birthday_window_oct"] == 1) |
        (df["is_back_to_school_season"] == 1)
    ).astype("int8")

    df["promo_score"] = (
        5 * df["is1111_sale_window"] +
        2 * df["is_pre1111_14d"] +
        2 * df["is_wb_birthday_window_oct"] +
        1 * df["is_back_to_school_peak"]
    ).astype("int8")
    return df


def add_route_seasonal_features(df, target_col):
    df = df.copy()

    route_hh_target_mean = (
        df.groupby(["route_id", "halfhouridx"], observed=False)[target_col]
        .mean().astype("float32").rename("route_hh_target_mean").reset_index()
    )
    route_dow_hh_target_mean = (
        df.groupby(["route_id", "dayofweek", "halfhouridx"], observed=False)[target_col]
        .mean().astype("float32").rename("route_dow_hh_target_mean").reset_index()
    )
    route_dow_hh_current_mean = (
        df.groupby(["route_id", "dayofweek", "halfhouridx"], observed=False)["status_current_sum"]
        .mean().astype("float32").rename("route_dow_hh_current_mean").reset_index()
    )
    route_dow_hh_prev_mean = (
        df.groupby(["route_id", "dayofweek", "halfhouridx"], observed=False)["status_prev_sum"]
        .mean().astype("float32").rename("route_dow_hh_prev_mean").reset_index()
    )

    df = df.merge(route_hh_target_mean, on=["route_id", "halfhouridx"], how="left")
    df = df.merge(route_dow_hh_target_mean, on=["route_id", "dayofweek", "halfhouridx"], how="left")
    df = df.merge(route_dow_hh_current_mean, on=["route_id", "dayofweek", "halfhouridx"], how="left")
    df = df.merge(route_dow_hh_prev_mean, on=["route_id", "dayofweek", "halfhouridx"], how="left")

    df["target_vs_route_hh_mean"] = (df[target_col] / (df["route_hh_target_mean"] + 1.0)).astype("float32")
    df["target_vs_route_dow_hh_mean"] = (df[target_col] / (df["route_dow_hh_target_mean"] + 1.0)).astype("float32")
    return df


def add_global_timestamp_features(df, target_col):
    df = df.copy()

    global_ts = (
        df.groupby("timestamp", observed=False)
        .agg(
            global_current_sum=("status_current_sum", "sum"),
            global_prev_sum=("status_prev_sum", "sum"),
            global_target_sum=(target_col, "sum"),
        )
        .astype("float32")
        .reset_index()
    )

    df = df.merge(global_ts, on="timestamp", how="left")
    df["route_share_current"] = (df["status_current_sum"] / (df["global_current_sum"] + 1.0)).astype("float32")
    df["route_share_prev"] = (df["status_prev_sum"] / (df["global_prev_sum"] + 1.0)).astype("float32")
    df["route_share_target"] = (df[target_col] / (df["global_target_sum"] + 1.0)).astype("float32")
    return df


def add_lag_features(df, base_signal_cols, group_col="route_id"):
    df = df.copy()
    grp = df.groupby(group_col, sort=False)

    for col in base_signal_cols:
        s = grp[col]
        for lag in LAGS_30M:
            df[f"{col}lag{lag}"] = s.shift(lag).astype("float32")

        prev = s.shift(1)
        for window in ROLL_WINDOWS:
            rolled = (
                prev.groupby(df[group_col], observed=False)
                .rolling(window=window, min_periods=1)
                .mean()
                .reset_index(level=0, drop=True)
                .astype("float32")
            )
            df[f"{col}rollmean{window}"] = rolled

        df[f"{col}diff1"] = s.diff(1).astype("float32")
        df[f"{col}diff2"] = s.diff(2).astype("float32")

    return df


# =========================
# DATA PREP
# =========================
def load_data():
    train_df = pd.read_parquet(CFG["train_path"])
    test_df = pd.read_parquet(CFG["test_path"])

    train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
    test_df["timestamp"] = pd.to_datetime(test_df["timestamp"])

    train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
    test_df = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
    return train_df, test_df


def make_future_targets(train_df):
    train_df = train_df.copy()
    grp = train_df.groupby("route_id", sort=False)
    for step in range(1, FORECAST_POINTS + 1):
        train_df[f"targetstep{step}"] = grp[TARGET_COL].shift(-step)
    return train_df


def build_feature_frame(train_df):
    status_cols = sorted([c for c in train_df.columns if c.startswith("status")])

    history_steps = max(max(LAGS_30M), max(ROLL_WINDOWS))
    buffer_days = int(np.ceil(history_steps / 48) + 1)
    max_model_train_days = max(s["train_days"] for s in MODEL_SPECS)
    raw_window_days = max_model_train_days + VALID_DAYS + buffer_days

    raw_start = train_df["timestamp"].max() - pd.Timedelta(days=raw_window_days)
    work_df = train_df.loc[train_df["timestamp"] >= raw_start].copy()

    for col in status_cols + [TARGET_COL] + FUTURE_TARGET_COLS:
        if col in work_df.columns:
            work_df[col] = work_df[col].astype("float32")

    work_df["route_id"] = work_df["route_id"].astype("category")

    work_df = add_time_features(work_df)
    work_df = add_wb_promo_features(work_df)
    work_df = add_status_aggregates(work_df, status_cols)
    work_df = add_route_seasonal_features(work_df, TARGET_COL)
    work_df = add_global_timestamp_features(work_df, TARGET_COL)

    base_signal_cols = status_cols + [TARGET_COL]
    work_df = add_lag_features(work_df, base_signal_cols)

    return work_df


def make_model_frames(work_df):
    feature_cols = [c for c in work_df.columns if c not in ["timestamp", "id"] + FUTURE_TARGET_COLS]

    target_ready_mask = work_df[FUTURE_TARGET_COLS].notna().all(axis=1)
    train_model_df = work_df.loc[target_ready_mask, feature_cols + ["timestamp"] + FUTURE_TARGET_COLS].copy()
    train_model_df = train_model_df.rename(columns={"timestamp": "source_timestamp"})

    max_model_train_days = max(s["train_days"] for s in MODEL_SPECS)
    train_window_start = train_model_df["source_timestamp"].max() - pd.Timedelta(days=max_model_train_days)
    train_model_df = train_model_df.loc[train_model_df["source_timestamp"] >= train_window_start].copy()

    infer_ts = work_df["timestamp"].max()
    test_model_df = work_df.loc[work_df["timestamp"] == infer_ts, feature_cols].copy()

    train_model_df = train_model_df.sort_values("source_timestamp").copy()

    valid_end = train_model_df["source_timestamp"].max()
    valid_start = valid_end - pd.Timedelta(days=VALID_DAYS)
    calib_start = valid_end - pd.Timedelta(hours=12)

    fit_df = train_model_df.loc[train_model_df["source_timestamp"] < valid_start].copy()
    valid_early_df = train_model_df.loc[
        (train_model_df["source_timestamp"] >= valid_start) &
        (train_model_df["source_timestamp"] < calib_start)
    ].copy()
    valid_calib_df = train_model_df.loc[train_model_df["source_timestamp"] >= calib_start].copy()

    return feature_cols, fit_df, valid_early_df, valid_calib_df, test_model_df


# =========================
# TRAIN / PREDICT
# =========================
def train_all_models(feature_cols, fit_df, valid_early_df, valid_calib_df, test_model_df, all_route_categories):
    categorical_features = ["route_id"]
    numeric_features = [c for c in feature_cols if c not in categorical_features]

    X_fit = fit_df[feature_cols].copy()
    X_valid = valid_early_df[feature_cols].copy()
    X_calib = valid_calib_df[feature_cols].copy()
    X_test = test_model_df[feature_cols].copy()

    for frame in [X_fit, X_valid, X_calib, X_test]:
        frame["route_id"] = pd.Categorical(frame["route_id"], categories=all_route_categories)

    y_fit = fit_df[FUTURE_TARGET_COLS].copy()
    y_valid = valid_early_df[FUTURE_TARGET_COLS].copy()
    y_calib = valid_calib_df[FUTURE_TARGET_COLS].copy()

    model_fit_preds = {}
    model_valid_preds = {}
    model_calib_preds = {}
    model_test_preds = {}
    score_rows = []

    for spec in MODEL_SPECS:
        fit_subset = get_recent_subset(fit_df, spec["train_days"])
        train_cap = spec.get("max_train_rows", MAX_TRAIN_ROWS)
        if len(fit_subset) > train_cap:
            fit_subset = fit_subset.sort_values("source_timestamp").tail(train_cap).copy()

        X_fit_m = fit_subset[feature_cols].copy()
        y_fit_m = fit_subset[FUTURE_TARGET_COLS].copy()
        X_fit_m["route_id"] = pd.Categorical(X_fit_m["route_id"], categories=all_route_categories)

        sample_weight = get_time_decay_weights(fit_subset, spec["decay_days"])

        fit_pred = pd.DataFrame(index=fit_df.index, columns=FUTURE_TARGET_COLS, dtype=np.float32)
        valid_pred = pd.DataFrame(index=valid_early_df.index, columns=FUTURE_TARGET_COLS, dtype=np.float32)
        calib_pred = pd.DataFrame(index=valid_calib_df.index, columns=FUTURE_TARGET_COLS, dtype=np.float32)
        test_pred = pd.DataFrame(index=test_model_df.index, columns=FUTURE_TARGET_COLS, dtype=np.float32)

        if spec["kind"] == "catboost":
            X_fit_cb = prepare_catboost_frame(X_fit_m)
            X_valid_cb = prepare_catboost_frame(X_valid)
            X_calib_cb = prepare_catboost_frame(X_calib)
            X_test_cb = prepare_catboost_frame(X_test)
            X_full_fit_cb = prepare_catboost_frame(X_fit)

        for target_col in FUTURE_TARGET_COLS:
            if spec["kind"] == "lgbm":
                model = LGBMRegressor(**spec["params"])
                model.fit(
                    X_fit_m,
                    y_fit_m[target_col],
                    sample_weight=sample_weight,
                    eval_set=[(X_valid, y_valid[target_col])],
                    eval_metric="l1",
                    categorical_feature=categorical_features,
                    callbacks=[early_stopping(100, verbose=False), log_evaluation(0)],
                )
                fit_pred[target_col] = np.clip(model.predict(X_fit), 0, None).astype(np.float32)
                valid_pred[target_col] = np.clip(model.predict(X_valid), 0, None).astype(np.float32)
                calib_pred[target_col] = np.clip(model.predict(X_calib), 0, None).astype(np.float32)
                test_pred[target_col] = np.clip(model.predict(X_test), 0, None).astype(np.float32)

            elif spec["kind"] == "catboost":
                model = CatBoostRegressor(**spec["params"])
                model.fit(
                    X_fit_cb,
                    y_fit_m[target_col],
                    sample_weight=sample_weight,
                    eval_set=(X_valid_cb, y_valid[target_col]),
                    cat_features=categorical_features,
                    use_best_model=True,
                    early_stopping_rounds=100,
                    verbose=False,
                )
                fit_pred[target_col] = np.clip(model.predict(X_full_fit_cb), 0, None).astype(np.float32)
                valid_pred[target_col] = np.clip(model.predict(X_valid_cb), 0, None).astype(np.float32)
                calib_pred[target_col] = np.clip(model.predict(X_calib_cb), 0, None).astype(np.float32)
                test_pred[target_col] = np.clip(model.predict(X_test_cb), 0, None).astype(np.float32)

            elif spec["kind"] == "ridge":
                model = build_ridge_model(spec["alpha"], numeric_features, categorical_features)
                if sample_weight is None:
                    model.fit(X_fit_m, y_fit_m[target_col])
                else:
                    model.fit(X_fit_m, y_fit_m[target_col], model__sample_weight=sample_weight)
                fit_pred[target_col] = np.clip(model.predict(X_fit), 0, None).astype(np.float32)
                valid_pred[target_col] = np.clip(model.predict(X_valid), 0, None).astype(np.float32)
                calib_pred[target_col] = np.clip(model.predict(X_calib), 0, None).astype(np.float32)
                test_pred[target_col] = np.clip(model.predict(X_test), 0, None).astype(np.float32)

            else:
                raise ValueError(f"Unknown model kind: {spec['kind']}")

        # horizon-wise calibration
        horizon_scales = {}
        for target_col in FUTURE_TARGET_COLS:
            pred_sum = calib_pred[target_col].sum()
            true_sum = y_calib[target_col].sum()
            scale = 1.0 if pred_sum == 0 else float(true_sum / pred_sum)
            horizon_scales[target_col] = scale
            fit_pred[target_col] *= scale
            valid_pred[target_col] *= scale
            calib_pred[target_col] *= scale
            test_pred[target_col] *= scale

        model_fit_preds[spec["name"]] = fit_pred
        model_valid_preds[spec["name"]] = valid_pred
        model_calib_preds[spec["name"]] = calib_pred
        model_test_preds[spec["name"]] = test_pred

        score_rows.append({
            "model": spec["name"],
            "kind": spec["kind"],
            "train_days": spec["train_days"],
            "calib_score": wape_plus_rbias_score(y_calib.to_numpy().ravel(), calib_pred.to_numpy().ravel()),
            "valid_score": wape_plus_rbias_score(y_valid.to_numpy().ravel(), valid_pred.to_numpy().ravel()),
            "mean_scale": np.mean(list(horizon_scales.values())),
        })

        gc.collect()

    score_table = pd.DataFrame(score_rows).sort_values("calib_score").reset_index(drop=True)
    return (
        score_table,
        y_valid,
        y_calib,
        model_fit_preds,
        model_valid_preds,
        model_calib_preds,
        model_test_preds,
    )


def search_best_per_horizon_blend(model_calib_preds, y_calib):
    model_names = list(model_calib_preds.keys())
    best_weights_per_horizon = {}
    best_score_per_horizon = {}

    for col in FUTURE_TARGET_COLS:
        if len(model_names) == 1:
            best_weights_per_horizon[col] = {model_names[0]: 1.0}
            best_score_per_horizon[col] = wape_plus_rbias_score(
                y_calib[col].to_numpy(),
                model_calib_preds[model_names[0]][col].to_numpy(),
            )
            continue

        best_score = None
        best_weights = None
        for weights in generate_blend_weights(model_names, step=BLEND_GRID_STEP):
            calib_blend = blend_single_col(model_calib_preds, weights, col)
            score = wape_plus_rbias_score(y_calib[col].to_numpy(), calib_blend.to_numpy())
            if (best_score is None) or (score < best_score):
                best_score = score
                best_weights = weights

        best_weights_per_horizon[col] = best_weights
        best_score_per_horizon[col] = best_score

    return best_weights_per_horizon, pd.Series(best_score_per_horizon)


def make_submission(test_df, test_model_df, pred_wide, out_path="submission.csv"):
    pred_wide = pred_wide.copy()
    pred_wide["route_id"] = test_model_df["route_id"].to_numpy()

    pred_long = pred_wide.melt(
        id_vars="route_id",
        value_vars=FUTURE_TARGET_COLS,
        var_name="step_col",
        value_name="target",
    )
    pred_long["step"] = pred_long["step_col"].str.extract(r"(\d+)").astype(int)
    pred_long = pred_long.sort_values(["route_id", "step"]).drop(columns="step_col")

    sub = test_df.sort_values(["route_id", "timestamp"]).copy()
    sub["step"] = sub.groupby("route_id").cumcount() + 1

    sub = sub.merge(pred_long, on=["route_id", "step"], how="left")
    submission = sub[["id", "target"]].copy()
    submission["target"] = submission["target"].clip(lower=0)

    submission.to_csv(out_path, index=False)
    return submission


# =========================
# MAIN
# =========================
def main():
    train_df, test_df = load_data()
    train_df = make_future_targets(train_df)
    work_df = build_feature_frame(train_df)

    feature_cols, fit_df, valid_early_df, valid_calib_df, test_model_df = make_model_frames(work_df)
    all_route_categories = work_df["route_id"].cat.categories

    (
        score_table,
        y_valid,
        y_calib,
        model_fit_preds,
        model_valid_preds,
        model_calib_preds,
        model_test_preds,
    ) = train_all_models(
        feature_cols=feature_cols,
        fit_df=fit_df,
        valid_early_df=valid_early_df,
        valid_calib_df=valid_calib_df,
        test_model_df=test_model_df,
        all_route_categories=all_route_categories,
    )

    print("\nSingle-model scores:")
    print(score_table)

    best_weights_per_horizon, best_horizon_scores = search_best_per_horizon_blend(model_calib_preds, y_calib)
    print("\nBest per-horizon calibration scores:")
    print(best_horizon_scores)

    valid_blend = blend_multi_col(model_valid_preds, best_weights_per_horizon, FUTURE_TARGET_COLS).clip(lower=0)
    test_blend = blend_multi_col(model_test_preds, best_weights_per_horizon, FUTURE_TARGET_COLS).clip(lower=0)

    overall_valid_score = wape_plus_rbias_score(y_valid.to_numpy().ravel(), valid_blend.to_numpy().ravel())
    overall_calib_score = wape_plus_rbias_score(y_calib.to_numpy().ravel(), blend_multi_col(
        model_calib_preds, best_weights_per_horizon, FUTURE_TARGET_COLS
    ).to_numpy().ravel())

    print(f"\nOverall per-horizon blend calib score: {overall_calib_score:.6f}")
    print(f"Overall per-horizon blend valid score: {overall_valid_score:.6f}")

    submission = make_submission(
        test_df=test_df,
        test_model_df=test_model_df,
        pred_wide=test_blend,
        out_path="submission.csv",
    )
    print("\nSubmission head:")
    print(submission.head())


if __name__ == "__main__":
    main()